# BLIP

## data

In [ ]:
# ===== COCO Karpathy Split 数据下载与验证 (最终修复版 v3 - 解决重复可视化) =====

# 1. 安装依赖
print("安装依赖...")
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pillow", "matplotlib", "requests"])
print("依赖安装完成。")

# 2. 导入模块
import os
import json
import urllib.request
import requests
import pickle
import random
from PIL import Image
import matplotlib.pyplot as plt
from io import BytesIO

# 3. 创建数据目录
os.makedirs('/content/coco_data', exist_ok=True)
os.chdir('/content/coco_data')
print("工作目录:", os.getcwd())

# 4. 下载 Karpathy Split 标注文件
print("=" * 60)
print("下载 Karpathy Split 标注文件...")
print("=" * 60)

urls = {
    'train': 'https://storage.googleapis.com/sfr-vision-language-research/datasets/coco_karpathy_train.json',
    'val': 'https://storage.googleapis.com/sfr-vision-language-research/datasets/coco_karpathy_val.json',
    'test': 'https://storage.googleapis.com/sfr-vision-language-research/datasets/coco_karpathy_test.json'
}

for split, url in urls.items():
    filename = f'coco_karpathy_{split}.json'
    if not os.path.exists(filename):
        print(f"下载 {filename}...")
        urllib.request.urlretrieve(url, filename)
    else:
        print(f"{filename} 已存在，跳过下载")

print("标注文件下载完成!\n")

# 5. 加载和验证数据
print("=" * 60)
print("加载数据...")
print("=" * 60)

train_data = json.load(open('coco_karpathy_train.json'))
val_data = json.load(open('coco_karpathy_val.json'))
test_data = json.load(open('coco_karpathy_test.json'))

def get_images_list(data):
    if isinstance(data, list):
        return data
    elif isinstance(data, dict) and 'images' in data:
        return data['images']
    return []

train_images = get_images_list(train_data)
val_images = get_images_list(val_data)
test_images = get_images_list(test_data)

print(f"Train样本数 (image-caption pairs): {len(train_images):,}")
print(f"Val样本数 (images): {len(val_images):,}")
print(f"Test样本数 (images): {len(test_images):,}")

# 6. ID 提取函数
def extract_numeric_id(image_id):
    """从任何格式的ID中提取唯一的数字ID"""
    if isinstance(image_id, int):
        return image_id
    elif isinstance(image_id, str):
        if '_' in image_id:
            id_part = image_id.split('_')[-1]
        else:
            id_part = image_id
        try:
            return int(id_part)
        except ValueError:
            return None
    return None

def get_image_id(img):
    """获取图像ID，兼容 BLIP 和 Karpathy 格式"""
    raw_id = None

    if 'image_id' in img:
        raw_id = img['image_id']
    elif 'cocoid' in img:
        raw_id = img['cocoid']
    elif 'id' in img:
        raw_id = img['id']
    elif 'image' in img and isinstance(img['image'], str):
        filename = img['image'].split('/')[-1]
        if '_' in filename and '.' in filename:
            try:
                id_str = filename.split('_')[-1].split('.')[0]
                raw_id = int(id_str)
            except:
                pass

    return raw_id

# 7. 验证无重叠
print("\n" + "=" * 60)
print("规范化和验证ID...")
print("=" * 60)

train_ids = {extract_numeric_id(get_image_id(img)) for img in train_images}
val_ids = {extract_numeric_id(get_image_id(img)) for img in val_images}
test_ids = {extract_numeric_id(get_image_id(img)) for img in test_images}

train_ids.discard(None)
val_ids.discard(None)
test_ids.discard(None)

print(f"\n数据集统计 (唯一图像ID):")
print(f"  Train (Member): {len(train_ids):,} 唯一ID")
print(f"  Val (Non-member): {len(val_ids):,} 唯一ID")
print(f"  Test (Non-member): {len(test_ids):,} 唯一ID")

overlap_train_val = len(train_ids & val_ids)
overlap_train_test = len(train_ids & test_ids)
overlap_val_test = len(val_ids & test_ids)

print(f"\n数据重叠检查:")
print(f"  Train ∩ Val: {overlap_train_val}")
print(f"  Train ∩ Test: {overlap_train_test}")
print(f"  Val ∩ Test: {overlap_val_test}")

if overlap_train_val == 0 and overlap_train_test == 0 and overlap_val_test == 0:
    print("  ✅ 无数据泄漏! Karpathy Split 验证成功。")
else:
    print("  ⚠️ 警告：存在数据重叠！")

# 8. Caption 提取函数
def get_caption(img):
    """提取单个caption，兼容 string 和 list 格式"""
    if 'caption' in img:
        caption_data = img['caption']
        if isinstance(caption_data, str):
            return caption_data
        elif isinstance(caption_data, list):
            if caption_data:
                return caption_data[0]
            else:
                return ""
    elif 'captions' in img:
        caps = img['captions']
        if isinstance(caps, list) and caps:
            return caps[0]
        elif isinstance(caps, str):
            return caps
    return ""

# 9. [关键修复] 数据去重 - 每个ID只保留一个样本
print("\n" + "=" * 60)
print("数据去重（按 Image ID）")
print("=" * 60)

def deduplicate_by_id(images, id_set):
    """按图像ID去重，每个ID只保留第一个样本"""
    id_to_image = {}
    for img in images:
        img_id = extract_numeric_id(get_image_id(img))
        if img_id and img_id in id_set:
            if img_id not in id_to_image:
                id_to_image[img_id] = img
    return id_to_image

# 去重
member_id_to_image = deduplicate_by_id(train_images, train_ids)
nonmember_id_to_image = deduplicate_by_id(val_images + test_images, val_ids | test_ids)

print(f"Train 去重: {len(train_images):,} 个样本 → {len(member_id_to_image):,} 个唯一图像")
print(f"Val+Test 去重: {len(val_images) + len(test_images):,} 个样本 → {len(nonmember_id_to_image):,} 个唯一图像")

# 10. 随机采样 1000 个
print("\n" + "=" * 60)
print("随机采样 1000 个唯一样本")
print("=" * 60)

random.seed(42)

sampled_member_ids = random.sample(
    list(member_id_to_image.keys()),
    min(1000, len(member_id_to_image))
)
sampled_member_images = [member_id_to_image[img_id] for img_id in sampled_member_ids]

sampled_nonmember_ids = random.sample(
    list(nonmember_id_to_image.keys()),
    min(1000, len(nonmember_id_to_image))
)
sampled_nonmember_images = [nonmember_id_to_image[img_id] for img_id in sampled_nonmember_ids]

print(f"✅ Member: {len(sampled_member_ids)} 个唯一图像")
print(f"✅ Non-member: {len(sampled_nonmember_ids)} 个唯一图像")

# 验证无重叠
overlap_final = set(sampled_member_ids) & set(sampled_nonmember_ids)
print(f"Member ∩ Non-member: {len(overlap_final)} (应为0)")

# 11. 可视化（使用去重后的数据）
print("\n" + "=" * 60)
print("样本可视化（去重后）")
print("=" * 60)

def download_coco_image(img):
    """根据image字段下载图像"""
    if 'image' in img:
        image_path = img['image']
        parts = image_path.split('/')
        if len(parts) == 2:
            split_dir, filename = parts
            url = f"http://images.cocodataset.org/{split_dir}/{filename}"
            try:
                response = requests.get(url, timeout=10)
                if response.status_code == 200:
                    return Image.open(BytesIO(response.content))
            except Exception as e:
                print(f"下载失败: {url}")

    # 备用方案
    raw_id = get_image_id(img)
    numeric_id = extract_numeric_id(raw_id)
    if numeric_id:
        for split in ['train2014', 'val2014']:
            url = f"http://images.cocodataset.org/{split}/COCO_{split}_{numeric_id:012d}.jpg"
            try:
                response = requests.get(url, timeout=10)
                if response.status_code == 200:
                    return Image.open(BytesIO(response.content))
            except:
                continue

    return None

def visualize_samples(images, split_name, num_samples=3):
    """可视化样本"""
    print(f"\n可视化 {split_name} 的 {num_samples} 个样本...")

    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    if num_samples == 1:
        axes = [axes]

    for i in range(min(num_samples, len(images))):
        img_data = images[i]
        img = download_coco_image(img_data)

        if img:
            axes[i].imshow(img)
            axes[i].axis('off')

            caption = get_caption(img_data)
            image_id = extract_numeric_id(get_image_id(img_data))

            title = f"ID: {image_id}\n{str(caption)[:50]}"
            if len(str(caption)) > 50:
                title += "..."
            axes[i].set_title(title, fontsize=9)
        else:
            image_id = extract_numeric_id(get_image_id(img_data))
            axes[i].text(0.5, 0.5, f'下载失败\nID: {image_id}',
                        ha='center', va='center', color='red')
            axes[i].axis('off')

    plt.suptitle(f'{split_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# [修复] 使用去重后的样本进行可视化
visualize_samples(sampled_member_images[:3], "Train (Member)", num_samples=3)
visualize_samples(sampled_nonmember_images[:3], "Val (Non-member)", num_samples=3)

# 12. 保存 MIA 数据
print("\n" + "=" * 60)
print("保存 MIA 数据")
print("=" * 60)

with open('member_ids_1k.pkl', 'wb') as f:
    pickle.dump(sampled_member_ids, f)

with open('nonmember_ids_1k.pkl', 'wb') as f:
    pickle.dump(sampled_nonmember_ids, f)

with open('member_images_1k.pkl', 'wb') as f:
    pickle.dump(sampled_member_images, f)

with open('nonmember_images_1k.pkl', 'wb') as f:
    pickle.dump(sampled_nonmember_images, f)

print("\n已保存:")
print("  - member_ids_1k.pkl (1000个唯一ID)")
print("  - nonmember_ids_1k.pkl (1000个唯一ID)")
print("  - member_images_1k.pkl (1000个唯一图像)")
print("  - nonmember_images_1k.pkl (1000个唯一图像)")

# 13. 数据质量检查
print("\n" + "=" * 60)
print("数据质量检查")
print("=" * 60)

member_captions = [get_caption(img) for img in sampled_member_images]
print(f"\nMember Caption 示例 (前3个):")
for i, cap in enumerate(member_captions[:3]):
    img_id = extract_numeric_id(get_image_id(sampled_member_images[i]))
    print(f"  ID {img_id}: {cap[:80]}...")

nonmember_captions = [get_caption(img) for img in sampled_nonmember_images]
print(f"\nNon-member Caption 示例 (前3个):")
for i, cap in enumerate(nonmember_captions[:3]):
    img_id = extract_numeric_id(get_image_id(sampled_nonmember_images[i]))
    print(f"  ID {img_id}: {cap[:80]}...")

# 14. 完成
print("\n" + "=" * 60)
print("✅ 完成!")
print("=" * 60)

print("\n文件列表:")
for f in sorted(os.listdir('.')):
    if os.path.isfile(f):
        size = os.path.getsize(f) / (1024*1024)
        print(f"  {f}: {size:.2f} MB")


## Gradaudit

In [ ]:
# ============================================================
# GradSafe-only MIA for BLIP - 最终修复版 (手动计算loss)
# ============================================================

import os, io, pickle, random, shutil, warnings, re, gc
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
from PIL import Image
from tqdm import tqdm
import requests
from io import BytesIO

import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

# ---------------------------
# 0) 路径与超参
# ---------------------------
MEMBER_PKL     = "/content/coco_data/member_images_1k.pkl"
NONMEMBER_PKL  = "/content/coco_data/nonmember_images_1k.pkl"

SEED = 42
NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

CALIB_MEM = 200
CALIB_NON = 200
PROBE_MEM = 800
PROBE_NON = 800

BATCH_K = 8
SENS_TAU = 0.10
VISION_LAST_N = 3
TEXT_LAST_M   = 3

# ---------------------------
# 1) 设备
# ---------------------------
assert torch.cuda.is_available(), "未检测到 CUDA"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

# ---------------------------
# 2) 加载 BLIP
# ---------------------------
try:
    from transformers import BlipProcessor, BlipForImageTextRetrieval
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.28.0"])
    from transformers import BlipProcessor, BlipForImageTextRetrieval

MODEL_NAME = 'Salesforce/blip-itm-base-coco'
print(f"⏳ 加载模型：{MODEL_NAME}")
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)

image_processor = processor.image_processor
tokenizer = processor.tokenizer
model.eval()
print("✅ BLIP (ITM) 加载完成")

# ---------------------------
# 3) 读取数据
# ---------------------------
def load_pickle_list(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

members_all    = load_pickle_list(MEMBER_PKL)
nonmembers_all = load_pickle_list(NONMEMBER_PKL)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)
members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 样本: members={len(members)} | non-members={len(nonmembers)}")

# ---------------------------
# 4) 图像读取
# ---------------------------
def load_coco_image(img_data: Dict[str,Any]) -> Optional[Image.Image]:
    image_path = img_data.get('image')
    if not image_path: return None
    try:
        parts = image_path.split('/')
        if len(parts) == 2:
            split_dir, filename = parts
            url = f"http://images.cocodataset.org/{split_dir}/{filename}"
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                return Image.open(BytesIO(response.content)).convert("RGB")
    except:
        return None
    return None

# ---------------------------
# 5) 选择性开梯度
# ---------------------------
def enable_grads_selectively_BLIP(m) -> List[str]:
    for _, p in m.named_parameters():
        p.requires_grad_(False)

    v_start = 12 - VISION_LAST_N
    t_start = 12 - TEXT_LAST_M

    print(f"\n🔧 开启梯度: Vision[{v_start},11], Text[{t_start},11]")

    selected = []
    for name, p in m.named_parameters():
        should_enable = False

        v_match = re.search(r'vision_model\.encoder\.layers\.(\d+)\.', name)
        if v_match:
            layer_idx = int(v_match.group(1))
            if layer_idx >= v_start and p.ndim >= 2:
                if any(kw in name for kw in ['self_attn', 'mlp']):
                    should_enable = True

        t_match = re.search(r'text_encoder\.encoder\.layer\.(\d+)\.', name)
        if t_match:
            layer_idx = int(t_match.group(1))
            if layer_idx >= t_start and p.ndim >= 2:
                if any(kw in name for kw in ['attention', 'query', 'key', 'value',
                                              'dense', 'intermediate', 'output']):
                    should_enable = True

        if should_enable:
            p.requires_grad_(True)
            selected.append(name)

    print(f"✅ 选中 {len(selected)} 个参数")
    return selected

collect_names = enable_grads_selectively_BLIP(model)

if len(collect_names) == 0:
    raise RuntimeError("❌ 没有参数开启梯度")

# ---------------------------
# 6) [关键修复] 手动计算对比损失
# ---------------------------
def compute_contrastive_loss(images: torch.Tensor, texts_input: Dict[str, torch.Tensor]) -> torch.Tensor:
    """
    手动计算 image-text 对比学习损失 (InfoNCE)
    images: [batch, 3, 384, 384]
    texts_input: {input_ids, attention_mask} each [batch, seq_len]
    """
    # 获取 image embeddings
    vision_outputs = model.vision_model(pixel_values=images)
    image_embeds = vision_outputs[1]  # pooler_output
    image_features = model.vision_proj(image_embeds)
    image_features = F.normalize(image_features, dim=-1)

    # 获取 text embeddings
    text_outputs = model.text_encoder(
        input_ids=texts_input.input_ids,
        attention_mask=texts_input.attention_mask,
        return_dict=True
    )
    text_embeds = text_outputs.last_hidden_state[:, 0, :]  # CLS token
    text_features = model.text_proj(text_embeds)
    text_features = F.normalize(text_features, dim=-1)

    # 计算相似度矩阵
    # logit_scale 在 BLIP ITM 中通常是可学习的参数
    logit_scale = model.logit_scale.exp() if hasattr(model, 'logit_scale') else torch.tensor(1.0).to(device)
    logits_per_image = logit_scale * image_features @ text_features.T  # [batch, batch]
    logits_per_text = logits_per_image.T

    # 对比学习损失 (InfoNCE)
    batch_size = images.shape[0]
    labels = torch.arange(batch_size, device=device)

    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)
    loss = (loss_i + loss_t) / 2

    return loss

def contrastive_loss_and_backward(images: torch.Tensor, toks: Dict[str, torch.Tensor]) -> None:
    """使用手动计算的对比损失"""
    model.train()
    model.zero_grad(set_to_none=True)

    with torch.cuda.amp.autocast():
        loss = compute_contrastive_loss(images, toks)

    loss.backward()

# ---------------------------
# 7) 构建批次
# ---------------------------
def make_batch_for_item(target: Dict[str, Any],
                        neg_pool: List[Dict[str, Any]],
                        kind: str,
                        K: int = BATCH_K) -> Optional[Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
    tgt_img = load_coco_image(target)
    if tgt_img is None:
        return None

    negs, tries = [], 0
    while len(negs) < K - 1 and tries < K * 40:
        cand = random.choice(neg_pool)
        if cand is target:
            tries += 1
            continue
        cim = load_coco_image(cand)
        if cim is None:
            tries += 1
            continue
        negs.append(cand)
        tries += 1

    batch_list = [target] + negs
    imgs, caps = [], []

    for obj in batch_list:
        im = load_coco_image(obj)
        if im is None:
            img_size = (image_processor.size['height'], image_processor.size['width'])
            im = Image.new("RGB", img_size, color="black")

        cap = ""
        if "caption" in obj:
            caption_data = obj["caption"]
            if isinstance(caption_data, str):
                cap = caption_data
            elif isinstance(caption_data, list) and caption_data:
                cap = caption_data[0]

        imgs.append(im)
        caps.append(cap.strip())

    images = torch.stack([
        image_processor(im, return_tensors="pt").pixel_values[0]
        for im in imgs
    ]).to(device)

    toks = tokenizer(
        caps,
        padding="max_length",
        truncation=True,
        max_length=77,
        return_tensors="pt"
    ).to(device)

    return images, toks

# ---------------------------
# 8) 梯度提取
# ---------------------------
def compute_grad_dict_for_item(target: Dict[str,Any],
                               neg_pool: List[Dict[str,Any]],
                               kind: str,
                               K: int = BATCH_K) -> Dict[str, torch.Tensor]:
    torch.cuda.empty_cache()

    made = make_batch_for_item(target, neg_pool, kind=kind, K=K)
    if made is None:
        return {}

    images, toks = made

    try:
        contrastive_loss_and_backward(images, toks)
    except Exception as e:
        model.zero_grad(set_to_none=True)
        return {}

    grad_dict = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        g = p.grad
        if g is None:
            continue
        if g.ndim >= 2:
            grad_dict[name] = g.detach().half().cpu()
        p.grad = None

    model.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    model.eval()

    return grad_dict

# ---------------------------
# 9) 行/列余弦
# ---------------------------
def row_col_cos(a: torch.Tensor, b: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if a.ndim < 2:
        flat = F.cosine_similarity(
            a.flatten().unsqueeze(0),
            b.flatten().unsqueeze(0),
            dim=1
        )
        return flat, flat

    r = F.cosine_similarity(a, b, dim=1)
    c = F.cosine_similarity(a.T, b.T, dim=1)

    return torch.nan_to_num(r, 0.0), torch.nan_to_num(c, 0.0)

# ---------------------------
# 10) 校准
# ---------------------------
def build_reference_grad(calib_members: List[Dict[str,Any]],
                         neg_pool: List[Dict[str,Any]],
                         take: int) -> Dict[str, torch.Tensor]:
    ref = {}
    ok = 0

    for i in tqdm(range(take), desc="参考梯度", miniters=10, dynamic_ncols=True):
        gd = compute_grad_dict_for_item(calib_members[i], neg_pool, kind="member", K=BATCH_K)
        if not gd:
            continue

        ok += 1
        if not ref:
            ref = {k: v.clone() for k, v in gd.items()}
        else:
            for k in ref:
                if k in gd and ref[k].shape == gd[k].shape:
                    ref[k] += gd[k]

    if ok == 0:
        raise RuntimeError("参考梯度构建失败")

    for k in ref:
        ref[k] /= ok

    print(f"  ✅ {ok}/{take} 样本, {len(ref)} 参数")
    return ref

def avg_rowcol_sims(samples: List[Dict[str,Any]],
                    ref: Dict[str,torch.Tensor],
                    neg_pool: List[Dict[str,Any]],
                    kind: str,
                    take: int) -> Tuple[Dict[str,torch.Tensor], Dict[str,torch.Tensor]]:
    acc_r, acc_c = {}, {}
    cnt = 0

    for i in tqdm(range(take), desc=f"相似度({kind})", miniters=10, dynamic_ncols=True):
        gd = compute_grad_dict_for_item(samples[i], neg_pool, kind=kind, K=BATCH_K)
        if not gd:
            continue

        cnt += 1
        for name, g in gd.items():
            if name not in ref or ref[name].shape != g.shape:
                continue

            rs, cs = row_col_cos(g, ref[name])

            if name not in acc_r:
                acc_r[name] = rs.clone()
                acc_c[name] = cs.clone()
            else:
                if acc_r[name].shape == rs.shape:
                    acc_r[name] += rs
                if acc_c[name].shape == cs.shape:
                    acc_c[name] += cs

    if cnt == 0:
        print(f"  ⚠️ {kind} 失败")
        return {}, {}

    for k in list(acc_r.keys()):
        acc_r[k] /= cnt
        acc_c[k] /= cnt

    print(f"  ✅ {cnt}/{take}")
    return acc_r, acc_c

def build_sensitive_masks(mem_rc, non_rc, tau: float = SENS_TAU, use_quantile: bool = True, q: float = 0.8):
    mr, mc = mem_rc
    nr, nc = non_rc
    row_masks, col_masks, total = {}, {}, 0

    if not mr or not mc:
        return {}, {}, 0

    for name in mr:
        if name not in nr or name not in mc or name not in nc:
            continue

        rgap = (mr[name] - nr[name]).detach().to(torch.float32).cpu()
        cgap = (mc[name] - nc[name]).detach().to(torch.float32).cpu()
        rgap = torch.nan_to_num(rgap, nan=0.0)
        cgap = torch.nan_to_num(cgap, nan=0.0)

        if use_quantile:
            rt = torch.quantile(rgap, q) if rgap.numel() > 0 else torch.tensor(float("inf"))
            ct = torch.quantile(cgap, q) if cgap.numel() > 0 else torch.tensor(float("inf"))
            rm = (rgap > rt)
            cm = (cgap > ct)
        else:
            rm = (rgap > float(tau))
            cm = (cgap > float(tau))

        row_masks[name] = rm
        col_masks[name] = cm
        total += int(rm.sum().item() + cm.sum().item())

    return row_masks, col_masks, total

# ---------------------------
# 11) 探测
# ---------------------------
def gradsafe_score_item(obj: Dict[str,Any],
                        ref: Dict[str,torch.Tensor],
                        row_masks: Dict[str,torch.Tensor],
                        col_masks: Dict[str,torch.Tensor],
                        neg_pool: List[Dict[str,Any]],
                        kind: str) -> float:
    gd = compute_grad_dict_for_item(obj, neg_pool, kind=kind, K=BATCH_K)
    if not gd:
        return 0.0

    sims = []
    for name, g in gd.items():
        if name in ref and name in row_masks and name in col_masks:
            if ref[name].shape != g.shape:
                continue

            rs, cs = row_col_cos(g, ref[name])
            rm, cm = row_masks[name], col_masks[name]

            if rm.any():
                sims.extend(rs[rm].cpu().tolist())
            if cm.any():
                sims.extend(cs[cm].cpu().tolist())

    return float(np.mean(sims)) if sims else 0.0

def score_probe_sets(probe_members, probe_nonmembers, ref, row_masks, col_masks,
                     neg_pool_mem, neg_pool_non):
    scores, labels = [], []

    for obj in tqdm(probe_members, desc="Probe-M", miniters=10, dynamic_ncols=True):
        s = gradsafe_score_item(obj, ref, row_masks, col_masks, neg_pool_mem, kind="member")
        scores.append(s)
        labels.append(1)

    for obj in tqdm(probe_nonmembers, desc="Probe-N", miniters=10, dynamic_ncols=True):
        s = gradsafe_score_item(obj, ref, row_masks, col_masks, neg_pool_non, kind="nonmember")
        scores.append(s)
        labels.append(0)

    return np.array(scores), np.array(labels)

# ---------------------------
# 12) 主流程
# ---------------------------
print("\n" + "=" * 60)
print("GradSafe MIA")
print("=" * 60)

calib_members = members[:CALIB_MEM]
calib_nonmems = nonmembers[:CALIB_NON]
probe_members = members[CALIB_MEM : CALIB_MEM + PROBE_MEM]
probe_nonmems = nonmembers[CALIB_NON : CALIB_NON + PROBE_NON]

print(f"\n📊 校准: M={len(calib_members)} N={len(calib_nonmems)}")
print(f"📊 探测: M={len(probe_members)} N={len(probe_nonmems)}\n")

neg_pool_for_member = members[CALIB_MEM + PROBE_MEM:] or members
neg_pool_for_nonmem = nonmembers[CALIB_NON + PROBE_NON:] or nonmembers

print("步骤1: 参考梯度")
ref_grads = build_reference_grad(calib_members, neg_pool_for_member, take=len(calib_members))

print("\n步骤2: 相似度")
mr, mc = avg_rowcol_sims(calib_members, ref_grads, neg_pool_for_member, kind="member", take=len(calib_members))
nr, nc = avg_rowcol_sims(calib_nonmems, ref_grads, neg_pool_for_nonmem, kind="nonmember", take=len(calib_nonmems))

print("\n步骤3: 敏感掩码")
row_masks, col_masks, total_crit = build_sensitive_masks((mr, mc), (nr, nc), tau=SENS_TAU)
print(f"  ✅ {total_crit} 敏感维度")

print("\n步骤4: 打分")
scores, labels = score_probe_sets(probe_members, probe_nonmems, ref_grads, row_masks, col_masks,
                                  neg_pool_for_member, neg_pool_for_nonmem)

# 评估
if np.all(scores == scores[0]):
    auc, best_thr, acc = 0.5, 0.0, 0.5
    cm = confusion_matrix(labels, (scores > np.median(scores)).astype(int))
else:
    auc = roc_auc_score(labels, scores)
    fpr, tpr, thr = roc_curve(labels, scores)
    best_idx = np.argmax(tpr - fpr)
    best_thr = thr[best_idx]
    pred = (scores >= best_thr).astype(int)
    acc = accuracy_score(labels, pred)
    cm = confusion_matrix(labels, pred)

print("\n" + "=" * 60)
print("结果")
print("=" * 60)
print(f"AUC = {auc:.4f}")
print(f"Acc = {acc:.4f}")
print(f"\n{cm}")

try:
    non_scores_np = scores[labels == 0]
    thr_5 = float(np.quantile(non_scores_np, 0.95))
    pred_5 = (scores >= thr_5).astype(int)
    cm_5 = confusion_matrix(labels, pred_5)
    tn, fp, fn, tp = cm_5.ravel()
    tpr_5 = tp / (tp + fn + 1e-12)

    print(f"\n@5% FPR:")
    print(f"TPR = {tpr_5*100:.2f}%")
    print(f"\n{cm_5}")
except:
    pass

print("\n✅ 完成!")

## GradNorm

In [ ]:
# ============================================================
# GradNorm MIA for BLIP (ITM/contrastive loss) - COCO PKL
# - Model: Salesforce/blip-itm-base-coco
# - Data : member_images_1k.pkl / nonmember_images_1k.pkl
# - Score: GradNorm = sqrt(sum ||grad||^2) over selected params only
# - Loss : manual InfoNCE (same as your fixed GradSafe version)
# - Eval : AUC, TPR@5%FPR on probe (800/800)
# ============================================================

import os, io, pickle, random, warnings, re, gc, time
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import requests
from io import BytesIO

import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Paths & hyperparams
# ---------------------------
MEMBER_PKL     = "/content/coco_data/member_images_1k.pkl"
NONMEMBER_PKL  = "/content/coco_data/nonmember_images_1k.pkl"

SEED = 42
NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

CALIB_MEM = 200
CALIB_NON = 200
PROBE_MEM = 800
PROBE_NON = 800

BATCH_K = 8  # build negatives in-batch (1 pos + K-1 neg)
VISION_LAST_N = 3
TEXT_LAST_M   = 3

# tokenizer config (keep as your version)
MAX_TXT_LEN = 77
PAD_MODE = "max_length"

# speed / stability
REQ_TIMEOUT = 10
PRINT_EVERY = 50
EPS = 1e-12

# ---------------------------
# 1) Device / seeds
# ---------------------------
assert torch.cuda.is_available(), "❌ 未检测到 CUDA"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print(f"✅ Device: {device}")

# ---------------------------
# 2) Load BLIP
# ---------------------------
try:
    from transformers import BlipProcessor, BlipForImageTextRetrieval
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.28.0"])
    from transformers import BlipProcessor, BlipForImageTextRetrieval

MODEL_NAME = "Salesforce/blip-itm-base-coco"
print(f"⏳ Loading model: {MODEL_NAME}")
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)

image_processor = processor.image_processor
tokenizer = processor.tokenizer
model.eval()
print("✅ BLIP loaded")

# ---------------------------
# 3) Load data (pkl)
# ---------------------------
def load_pickle_list(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

members_all    = load_pickle_list(MEMBER_PKL)
nonmembers_all = load_pickle_list(NONMEMBER_PKL)

rnd = random.Random(SEED)
rnd.shuffle(members_all)
rnd.shuffle(nonmembers_all)

members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]

print(f"📦 Samples: members={len(members)} | nonmembers={len(nonmembers)}")

# ---------------------------
# 4) COCO image loader
# ---------------------------
def load_coco_image(img_data: Dict[str,Any]) -> Optional[Image.Image]:
    image_path = img_data.get("image")
    if not image_path:
        return None
    try:
        parts = image_path.split("/")
        if len(parts) == 2:
            split_dir, filename = parts
            url = f"http://images.cocodataset.org/{split_dir}/{filename}"
            r = requests.get(url, timeout=REQ_TIMEOUT)
            if r.status_code == 200:
                return Image.open(BytesIO(r.content)).convert("RGB")
    except Exception:
        return None
    return None

def get_caption(obj: Dict[str,Any]) -> str:
    cap = ""
    if "caption" in obj:
        c = obj["caption"]
        if isinstance(c, str):
            cap = c
        elif isinstance(c, list) and len(c) > 0:
            cap = c[0]
    return str(cap).strip()

# ---------------------------
# 5) Enable grads selectively (same selection idea as your GradSafe)
# ---------------------------
def enable_grads_selectively_BLIP(m) -> List[str]:
    for _, p in m.named_parameters():
        p.requires_grad_(False)

    # BLIP base usually has 12 vision layers & 12 text layers
    v_start = 12 - VISION_LAST_N
    t_start = 12 - TEXT_LAST_M

    print(f"\n🔧 Trainable selection: Vision[{v_start}..11], Text[{t_start}..11]")

    selected = []
    for name, p in m.named_parameters():
        if p.ndim < 2:
            continue

        should_enable = False

        # vision: vision_model.encoder.layers.{i}.(self_attn|mlp)
        mv = re.search(r"vision_model\.encoder\.layers\.(\d+)\.", name)
        if mv:
            i = int(mv.group(1))
            if i >= v_start and any(kw in name for kw in ["self_attn", "mlp"]):
                should_enable = True

        # text: text_encoder.encoder.layer.{i}.*
        mt = re.search(r"text_encoder\.encoder\.layer\.(\d+)\.", name)
        if mt:
            i = int(mt.group(1))
            if i >= t_start and any(kw in name for kw in [
                "attention", "query", "key", "value",
                "dense", "intermediate", "output"
            ]):
                should_enable = True

        if should_enable:
            p.requires_grad_(True)
            selected.append(name)

    print(f"✅ Selected trainable params = {len(selected)}")
    return selected

selected_param_names = enable_grads_selectively_BLIP(model)
if len(selected_param_names) == 0:
    raise RuntimeError("❌ No parameters were selected for gradients (check regex patterns).")

# ---------------------------
# 6) Manual contrastive loss (your fixed version)
# ---------------------------
def compute_contrastive_loss(images: torch.Tensor, texts_input) -> torch.Tensor:
    """
    Manual InfoNCE loss using BLIP image/text projections.
    images: [B,3,H,W]
    texts_input: BatchEncoding with input_ids, attention_mask
    """
    # vision
    vision_outputs = model.vision_model(pixel_values=images)
    # depending on HF version, pooler_output might be at [1]
    image_embeds = vision_outputs[1]
    image_features = model.vision_proj(image_embeds)
    image_features = F.normalize(image_features, dim=-1)

    # text
    text_outputs = model.text_encoder(
        input_ids=texts_input.input_ids,
        attention_mask=texts_input.attention_mask,
        return_dict=True
    )
    text_embeds = text_outputs.last_hidden_state[:, 0, :]
    text_features = model.text_proj(text_embeds)
    text_features = F.normalize(text_features, dim=-1)

    # logits
    logit_scale = model.logit_scale.exp() if hasattr(model, "logit_scale") else torch.tensor(1.0, device=device)
    logits_per_image = logit_scale * (image_features @ text_features.T)
    logits_per_text  = logits_per_image.T

    B = images.shape[0]
    labels = torch.arange(B, device=device)

    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)
    return (loss_i + loss_t) / 2

# ---------------------------
# 7) Build one in-batch set (target + negatives)
# ---------------------------
def make_batch_for_item(target: Dict[str, Any],
                        neg_pool: List[Dict[str, Any]],
                        K: int = BATCH_K) -> Optional[Tuple[torch.Tensor, Any]]:
    tgt_img = load_coco_image(target)
    if tgt_img is None:
        return None

    # sample negatives (images + their own captions)
    negs = []
    tries = 0
    while len(negs) < K - 1 and tries < K * 40:
        cand = random.choice(neg_pool)
        if cand is target:
            tries += 1
            continue
        cim = load_coco_image(cand)
        if cim is None:
            tries += 1
            continue
        negs.append(cand)
        tries += 1

    batch_list = [target] + negs
    imgs, caps = [], []

    for obj in batch_list:
        im = load_coco_image(obj)
        if im is None:
            # keep shape consistent
            img_size = (image_processor.size["height"], image_processor.size["width"])
            im = Image.new("RGB", img_size, color="black")

        cap = get_caption(obj)
        imgs.append(im)
        caps.append(cap)

    images = torch.stack([
        image_processor(im, return_tensors="pt").pixel_values[0]
        for im in imgs
    ]).to(device, non_blocking=True)

    toks = tokenizer(
        caps,
        padding=PAD_MODE,
        truncation=True,
        max_length=MAX_TXT_LEN,
        return_tensors="pt"
    ).to(device)

    return images, toks

# ---------------------------
# 8) GradNorm score for one sample
# ---------------------------
def gradnorm_score_one(target: Dict[str,Any],
                       neg_pool: List[Dict[str,Any]],
                       K: int = BATCH_K) -> Optional[float]:
    made = make_batch_for_item(target, neg_pool, K=K)
    if made is None:
        return None

    images, toks = made

    model.train()
    model.zero_grad(set_to_none=True)

    try:
        # BF16 autocast for speed
        with torch.cuda.amp.autocast():
            loss = compute_contrastive_loss(images, toks)

        if (loss is None) or (not torch.isfinite(loss)):
            model.zero_grad(set_to_none=True)
            model.eval()
            torch.cuda.empty_cache()
            return None

        loss.backward()

        sq_sum = None
        for n, p in model.named_parameters():
            if not p.requires_grad:
                continue
            g = p.grad
            if g is None:
                continue
            gg = (g.detach().to(torch.float32) ** 2).sum()
            sq_sum = gg if sq_sum is None else (sq_sum + gg)

        model.zero_grad(set_to_none=True)
        model.eval()
        torch.cuda.empty_cache()

        if sq_sum is None:
            return None
        return float(torch.sqrt(sq_sum + EPS).item())

    except Exception:
        model.zero_grad(set_to_none=True)
        model.eval()
        torch.cuda.empty_cache()
        return None

# ---------------------------
# 9) Score dataset & Eval (AUC + TPR@5%FPR)
# ---------------------------
def score_set(objs: List[Dict[str,Any]],
              neg_pool: List[Dict[str,Any]],
              label: int,
              tag: str) -> Tuple[np.ndarray, np.ndarray]:
    scores = []
    labels = []

    ok = 0
    t0 = time.time()
    pbar = tqdm(objs, desc=f"GradNorm({tag})", dynamic_ncols=True)

    for i, obj in enumerate(pbar):
        s = gradnorm_score_one(obj, neg_pool, K=BATCH_K)
        if s is None:
            scores.append(np.nan)
        else:
            scores.append(float(s))
            ok += 1
        labels.append(label)

        if (i + 1) % max(1, PRINT_EVERY) == 0:
            elapsed = time.time() - t0
            pbar.set_postfix_str(f"ok={ok}/{i+1} elapsed={elapsed:.1f}s")

        if (i + 1) % 100 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    return np.array(scores, dtype=np.float32), np.array(labels, dtype=np.int32)

def drop_nan(scores: np.ndarray, labels: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    m = np.isfinite(scores)
    return scores[m], labels[m]

def tpr_at_fpr(scores: np.ndarray, labels: np.ndarray, target_fpr: float = 0.05) -> float:
    fpr, tpr, thr = roc_curve(labels, scores)
    idx = np.where(fpr <= target_fpr)[0]
    if len(idx) == 0:
        return 0.0
    return float(tpr[idx[-1]])

# ---------------------------
# 10) Main pipeline
# ---------------------------
print("\n" + "="*70)
print("GradNorm MIA for BLIP (member vs nonmember)")
print("="*70)

calib_members = members[:CALIB_MEM]
calib_nonmems = nonmembers[:CALIB_NON]
probe_members = members[CALIB_MEM:CALIB_MEM + PROBE_MEM]
probe_nonmems = nonmembers[CALIB_NON:CALIB_NON + PROBE_NON]

print(f"Calib: M={len(calib_members)} N={len(calib_nonmems)}")
print(f"Probe: M={len(probe_members)} N={len(probe_nonmems)}")

# negatives pool (same logic as your code)
neg_pool_for_member = members[CALIB_MEM + PROBE_MEM:] or members
neg_pool_for_nonmem = nonmembers[CALIB_NON + PROBE_NON:] or nonmembers

# NOTE: GradNorm does not need calib. We keep your split intact, but only probe used for eval.
# If you want: you can use calib just for warm-up / caching, not needed here.

print("\nStep) Score probe members")
m_scores, m_y = score_set(probe_members, neg_pool_for_member, label=1, tag="Probe-M")

print("\nStep) Score probe nonmembers")
n_scores, n_y = score_set(probe_nonmems, neg_pool_for_nonmem, label=0, tag="Probe-N")

scores = np.concatenate([m_scores, n_scores], axis=0)
labels = np.concatenate([m_y, n_y], axis=0)

scores, labels = drop_nan(scores, labels)
print(f"\n✅ Valid scored samples: {len(scores)}/{(PROBE_MEM+PROBE_NON)} (nan dropped)")
print(f"   n_valid_M={int((labels==1).sum())}  n_valid_N={int((labels==0).sum())}")

if len(np.unique(labels)) < 2:
    raise RuntimeError("Need both classes after dropping NaNs.")

auc_raw = float(roc_auc_score(labels, scores))

# auto direction fix
if auc_raw < 0.5:
    scores = -scores
    auc = 1.0 - auc_raw
    flipped = True
else:
    auc = auc_raw
    flipped = False

tpr5 = tpr_at_fpr(scores, labels, target_fpr=0.05)

print("\n" + "="*70)
print("📊 Results (GradNorm)")
print("="*70)
print(f"AUC        = {auc:.4f}  (raw={auc_raw:.4f}, flipped={flipped})")
print(f"TPR@5%FPR  = {tpr5*100:.2f}%")

m_sc = scores[labels==1]
n_sc = scores[labels==0]
print("\nScore stats (after direction fix if applied):")
print(f"  mean(M)={m_sc.mean():.6f}  std(M)={m_sc.std():.6f}")
print(f"  mean(N)={n_sc.mean():.6f}  std(N)={n_sc.std():.6f}")

print("\n✅ Done.")


## Gradaudit P/S/N

In [ ]:
# ==============================================================================
# COCO (Karpathy) P/S/N Experiment for Multi-Modal GradAudit-style MIA on BLIP-ITM
# - Data: member_images_1k.pkl, nonmember_images_1k.pkl
# - Model: Salesforce/blip-itm-base-coco
# - Attack: ref gradient + sensitive subspace mask + probe scoring
# - Experiments:
#   (1) AUC on P vs S
#   (2) FPR on S when threshold chosen to make TPR(P)=80%
# - Update:
#   ✅ Use Vision last3 + Text last3 gradients
#   ✅ Sensitive mask rule:
#        if tau-mask keeps <20% dims -> use top20% (q=0.8)
#        else -> use tau=0.10
# ==============================================================================

import os, io, re, gc, math, pickle, random, warnings
from typing import Dict, Any, Optional, List, Tuple
from io import BytesIO

import numpy as np
from PIL import Image
from tqdm import tqdm
import requests

import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

warnings.filterwarnings("ignore")

# ============================================================
# 0) Config
# ============================================================
MEMBER_PKL     = "/content/coco_data/member_images_1k.pkl"
NONMEMBER_PKL  = "/content/coco_data/nonmember_images_1k.pkl"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

CALIB_P = 200
CALIB_N = 200
PROBE_P = 800
PROBE_S = 800
PROBE_N = 800

BATCH_K = 8

# Sensitive mask (your requirement)
SENS_TAU = 0.10
Q_TOP = 0.80          # q=0.8 -> top 20%
MIN_KEEP_RATIO = 0.20 # guarantee at least 20% dims if tau selects too few

# Vision/Text last3
VISION_LAST_N = 3
TEXT_LAST_M   = 3

# Image cache
IMG_CACHE_DIR = "/content/coco_img_cache"
os.makedirs(IMG_CACHE_DIR, exist_ok=True)

assert torch.cuda.is_available(), "❌ CUDA not available"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True
print(f"✅ Device: {device}")

# ============================================================
# 1) Load BLIP-ITM model
# ============================================================
print("\n" + "="*80)
print("1) Load BLIP-ITM model")
print("="*80)

try:
    from transformers import BlipProcessor, BlipForImageTextRetrieval
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.28.0"])
    from transformers import BlipProcessor, BlipForImageTextRetrieval

MODEL_NAME = "Salesforce/blip-itm-base-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()

image_processor = processor.image_processor
tokenizer = processor.tokenizer
print("✅ BLIP-ITM ready")

# ============================================================
# 2) Load pickles and construct P / S / N
# ============================================================
print("\n" + "="*80)
print("2) Load data + construct P/S/N")
print("="*80)

def load_pickle_list(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

members_all    = load_pickle_list(MEMBER_PKL)
nonmembers_all = load_pickle_list(NONMEMBER_PKL)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)

members_all    = members_all[:NUM_MEMBERS]
nonmembers_all = nonmembers_all[:NUM_NONMEMBERS]

def get_caption_from_obj(obj: Dict[str,Any]) -> str:
    if "caption" in obj:
        c = obj["caption"]
        if isinstance(c, str): return c.strip()
        if isinstance(c, list) and len(c) > 0: return str(c[0]).strip()
        return ""
    if "captions" in obj:
        c = obj["captions"]
        if isinstance(c, str): return c.strip()
        if isinstance(c, list) and len(c) > 0: return str(c[0]).strip()
        return ""
    return ""

def set_caption(obj: Dict[str,Any], new_cap: str) -> Dict[str,Any]:
    x = dict(obj)
    x["caption"] = str(new_cap)
    return x

# P: paired
P_all = members_all

# S: shuffled captions within member pool, enforce j != i
caps = [get_caption_from_obj(x) for x in members_all]
perm = list(range(len(caps)))
random.Random(SEED+999).shuffle(perm)
for i in range(len(perm)):
    if perm[i] == i:
        j = (i + 1) % len(perm)
        perm[i], perm[j] = perm[j], perm[i]

S_all = [set_caption(members_all[i], caps[perm[i]]) for i in range(len(members_all))]

# N: nonmembers
N_all = nonmembers_all

print(f"✅ Built sets: P={len(P_all)} | S={len(S_all)} | N={len(N_all)}")
diff_cnt = sum(get_caption_from_obj(P_all[i]) != get_caption_from_obj(S_all[i]) for i in range(50))
print(f"Sanity (first 50): P vs S caption different count = {diff_cnt}/50")

# Split
P_calib = P_all[:CALIB_P]
N_calib = N_all[:CALIB_N]

P_probe = P_all[CALIB_P:CALIB_P+PROBE_P]
S_probe = S_all[CALIB_P:CALIB_P+PROBE_S]
N_probe = N_all[CALIB_N:CALIB_N+PROBE_N]

print(f"Calib: P={len(P_calib)}, N={len(N_calib)}")
print(f"Probe: P={len(P_probe)}, S={len(S_probe)}, N={len(N_probe)}")

neg_pool_member = P_all[CALIB_P+PROBE_P:] or P_all
neg_pool_nonmem = N_all[CALIB_N+PROBE_N:] or N_all

# ============================================================
# 3) Image loading (download with caching)
# ============================================================
print("\n" + "="*80)
print("3) Image loader with cache")
print("="*80)

def _coco_url_from_image_field(image_field: str) -> Optional[str]:
    if not isinstance(image_field, str) or "/" not in image_field:
        return None
    split_dir, filename = image_field.split("/", 1)
    return f"http://images.cocodataset.org/{split_dir}/{filename}"

def load_coco_image(obj: Dict[str,Any]) -> Optional[Image.Image]:
    image_field = obj.get("image")
    url = _coco_url_from_image_field(image_field)
    if url is None:
        return None

    fname = image_field.replace("/", "_")
    fpath = os.path.join(IMG_CACHE_DIR, fname)

    try:
        if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
            return Image.open(fpath).convert("RGB")
    except:
        pass

    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            return None
        img = Image.open(BytesIO(r.content)).convert("RGB")
        try:
            img.save(fpath, format="JPEG", quality=95)
        except:
            pass
        return img
    except:
        return None

# ============================================================
# 4) Enable grads (Vision last3 + Text last3)
# ============================================================
print("\n" + "="*80)
print("4) Enable gradients (Vision last3 + Text last3)")
print("="*80)

def enable_grads_last3_BLIP(m) -> List[str]:
    # freeze all
    for _, p in m.named_parameters():
        p.requires_grad_(False)

    # layer count is usually 12 for BLIP base; we derive from names if possible
    v_ids, t_ids = set(), set()
    for name, _ in m.named_parameters():
        vm = re.search(r"vision_model\.encoder\.layers\.(\d+)\.", name)
        if vm:
            v_ids.add(int(vm.group(1)))
        tm = re.search(r"text_encoder\.encoder\.layer\.(\d+)\.", name)
        if tm:
            t_ids.add(int(tm.group(1)))

    vmax = max(v_ids) if v_ids else 11
    tmax = max(t_ids) if t_ids else 11
    v_start = max(0, vmax - (VISION_LAST_N - 1))
    t_start = max(0, tmax - (TEXT_LAST_M   - 1))

    print(f"  ℹ️ Vision layers: [{v_start}..{vmax}]")
    print(f"  ℹ️ Text   layers: [{t_start}..{tmax}]")

    selected = []
    for name, p in m.named_parameters():
        if p.ndim < 2:
            continue

        enable = False

        # Vision last3: keep self_attn/mlp params
        vm = re.search(r"vision_model\.encoder\.layers\.(\d+)\.", name)
        if vm:
            idx = int(vm.group(1))
            if idx >= v_start and any(kw in name for kw in ["self_attn", "mlp"]):
                enable = True

        # Text last3: keep attention/ffn params
        tm = re.search(r"text_encoder\.encoder\.layer\.(\d+)\.", name)
        if tm:
            idx = int(tm.group(1))
            if idx >= t_start and any(kw in name for kw in [
                "attention", "query", "key", "value",
                "dense", "intermediate", "output"
            ]):
                enable = True

        # Projections/logit_scale (optional but useful)
        if any(k in name for k in ["vision_proj", "text_proj", "logit_scale"]):
            enable = True

        if enable:
            p.requires_grad_(True)
            selected.append(name)

    return selected

selected_params = enable_grads_last3_BLIP(model)
print(f"✅ Trainable params (selected) = {len(selected_params)}")
if len(selected_params) == 0:
    raise RuntimeError("❌ No parameters enabled for grads")

# ============================================================
# 5) Contrastive loss (InfoNCE) + backward
# ============================================================
@torch.enable_grad()
def compute_contrastive_loss(images: torch.Tensor, toks) -> torch.Tensor:
    vision_outputs = model.vision_model(pixel_values=images)
    image_embeds = vision_outputs[1]  # pooler_output
    image_features = model.vision_proj(image_embeds)
    image_features = F.normalize(image_features, dim=-1)

    text_outputs = model.text_encoder(
        input_ids=toks.input_ids,
        attention_mask=toks.attention_mask,
        return_dict=True
    )
    text_embeds = text_outputs.last_hidden_state[:, 0, :]  # CLS
    text_features = model.text_proj(text_embeds)
    text_features = F.normalize(text_features, dim=-1)

    logit_scale = model.logit_scale.exp() if hasattr(model, "logit_scale") else torch.tensor(1.0, device=device)
    logits_per_image = logit_scale * (image_features @ text_features.T)
    logits_per_text  = logits_per_image.T

    B = images.shape[0]
    labels = torch.arange(B, device=device)
    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)
    return (loss_i + loss_t) / 2

@torch.enable_grad()
def contrastive_loss_and_backward(images: torch.Tensor, toks) -> None:
    model.train()
    model.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast():
        loss = compute_contrastive_loss(images, toks)
    loss.backward()
    model.eval()

# ============================================================
# 6) Batch construction (target + negatives)
# ============================================================
def make_batch_for_obj(
    target_obj: Dict[str,Any],
    neg_pool: List[Dict[str,Any]],
    K: int = BATCH_K
) -> Optional[Tuple[torch.Tensor, Any]]:
    tgt_img = load_coco_image(target_obj)
    if tgt_img is None:
        return None

    negs = []
    tries = 0
    while len(negs) < K-1 and tries < K*60:
        cand = random.choice(neg_pool)
        if cand is target_obj:
            tries += 1
            continue
        cim = load_coco_image(cand)
        if cim is None:
            tries += 1
            continue
        negs.append(cand)
        tries += 1

    batch_list = [target_obj] + negs

    imgs, caps = [], []
    for obj in batch_list:
        im = load_coco_image(obj)
        if im is None:
            H = image_processor.size["height"]
            W = image_processor.size["width"]
            im = Image.new("RGB", (W, H), color="black")

        cap = get_caption_from_obj(obj).strip()
        imgs.append(im)
        caps.append(cap)

    images = torch.stack([
        image_processor(im, return_tensors="pt").pixel_values[0]
        for im in imgs
    ]).to(device)

    toks = tokenizer(
        caps,
        padding="max_length",
        truncation=True,
        max_length=77,
        return_tensors="pt"
    ).to(device)

    return images, toks

# ============================================================
# 7) Gradient dict per sample
# ============================================================
def compute_grad_dict_for_obj(
    target_obj: Dict[str,Any],
    neg_pool: List[Dict[str,Any]],
    K: int = BATCH_K
) -> Dict[str, torch.Tensor]:
    made = make_batch_for_obj(target_obj, neg_pool, K=K)
    if made is None:
        return {}

    images, toks = made
    try:
        contrastive_loss_and_backward(images, toks)
    except Exception:
        model.zero_grad(set_to_none=True)
        return {}

    grad_dict = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        g = p.grad
        if g is None:
            continue
        if g.ndim >= 2:
            grad_dict[name] = g.detach().half().cpu()
        p.grad = None

    model.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    return grad_dict

# ============================================================
# 8) Row/Col cosine similarity
# ============================================================
def row_col_cos(a: torch.Tensor, b: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if a.ndim < 2:
        flat = F.cosine_similarity(a.flatten().unsqueeze(0), b.flatten().unsqueeze(0), dim=1)
        return flat, flat
    r = F.cosine_similarity(a, b, dim=1)
    c = F.cosine_similarity(a.T, b.T, dim=1)
    return torch.nan_to_num(r, 0.0), torch.nan_to_num(c, 0.0)

# ============================================================
# 9) Build reference gradient from P_calib
# ============================================================
def build_reference_grad(calib_P: List[Dict[str,Any]], neg_pool: List[Dict[str,Any]]) -> Dict[str, torch.Tensor]:
    ref = {}
    ok = 0
    for obj in tqdm(calib_P, desc="RefGrad(P)", dynamic_ncols=True):
        gd = compute_grad_dict_for_obj(obj, neg_pool, K=BATCH_K)
        if not gd:
            continue
        ok += 1
        if not ref:
            ref = {k: v.clone() for k, v in gd.items()}
        else:
            for k in ref:
                if k in gd and ref[k].shape == gd[k].shape:
                    ref[k] += gd[k]

    if ok == 0:
        raise RuntimeError("❌ Reference gradient build failed (0 valid grads).")

    for k in ref:
        ref[k] /= ok
    print(f"✅ RefGrad built with {ok}/{len(calib_P)} samples, params={len(ref)}")
    return ref

# ============================================================
# 10) Avg sims on calib P / calib N
# ============================================================
def avg_rowcol_sims(samples: List[Dict[str,Any]], ref: Dict[str,torch.Tensor],
                    neg_pool: List[Dict[str,Any]], tag: str) -> Tuple[Dict[str,torch.Tensor], Dict[str,torch.Tensor]]:
    acc_r, acc_c = {}, {}
    cnt = 0
    for obj in tqdm(samples, desc=f"Sims({tag})", dynamic_ncols=True):
        gd = compute_grad_dict_for_obj(obj, neg_pool, K=BATCH_K)
        if not gd:
            continue
        cnt += 1
        for name, g in gd.items():
            if name not in ref or ref[name].shape != g.shape:
                continue
            rs, cs = row_col_cos(g, ref[name])
            if name not in acc_r:
                acc_r[name] = rs.clone()
                acc_c[name] = cs.clone()
            else:
                if acc_r[name].shape == rs.shape: acc_r[name] += rs
                if acc_c[name].shape == cs.shape: acc_c[name] += cs

    if cnt == 0:
        print(f"⚠️ Sims({tag}) got 0 valid grads.")
        return {}, {}

    for k in acc_r:
        acc_r[k] /= cnt
        acc_c[k] /= cnt
    print(f"✅ Sims({tag}) computed with {cnt}/{len(samples)} samples")
    return acc_r, acc_c

# ============================================================
# 11) Sensitive masks (your rule)
#    - if tau keeps <20% -> use top20% (q=0.8)
#    - else -> use tau
# ============================================================
def _mask_tau_or_top20(gap: torch.Tensor, tau: float, q: float, min_keep_ratio: float) -> torch.Tensor:
    """
    gap: 1D tensor (row-sim gaps or col-sim gaps)
    rule:
      tau_mask = gap > tau
      if tau_mask ratio < min_keep_ratio: use top-q mask (gap >= quantile(gap, q))
      else: use tau_mask
    """
    gap = torch.nan_to_num(gap, nan=0.0, posinf=0.0, neginf=0.0)
    if gap.numel() == 0:
        return torch.zeros_like(gap, dtype=torch.bool)

    tau_mask = (gap > float(tau))
    keep_ratio = float(tau_mask.float().mean().item())

    if keep_ratio < float(min_keep_ratio):
        qthr = torch.quantile(gap, float(q))
        q_mask = (gap >= qthr)
        return q_mask
    else:
        return tau_mask

def build_sensitive_masks(P_rc, N_rc, tau: float = SENS_TAU, q: float = Q_TOP, min_keep_ratio: float = MIN_KEEP_RATIO):
    pr, pc = P_rc
    nr, nc = N_rc

    row_masks, col_masks = {}, {}
    total = 0

    for name in pr:
        if name not in nr or name not in pc or name not in nc:
            continue

        rgap = (pr[name] - nr[name]).detach().float().cpu()
        cgap = (pc[name] - nc[name]).detach().float().cpu()

        rm = _mask_tau_or_top20(rgap, tau=tau, q=q, min_keep_ratio=min_keep_ratio)
        cm = _mask_tau_or_top20(cgap, tau=tau, q=q, min_keep_ratio=min_keep_ratio)

        if rm.any() or cm.any():
            row_masks[name] = rm
            col_masks[name] = cm
            total += int(rm.sum().item() + cm.sum().item())

    return row_masks, col_masks, total

# ============================================================
# 12) Probe scoring
# ============================================================
def score_obj(obj: Dict[str,Any], ref: Dict[str,torch.Tensor],
              row_masks: Dict[str,torch.Tensor], col_masks: Dict[str,torch.Tensor],
              neg_pool: List[Dict[str,Any]]) -> float:
    gd = compute_grad_dict_for_obj(obj, neg_pool, K=BATCH_K)
    if not gd:
        return 0.0

    sims = []
    for name, g in gd.items():
        if name not in ref or name not in row_masks or name not in col_masks:
            continue
        if ref[name].shape != g.shape:
            continue

        rs, cs = row_col_cos(g, ref[name])
        rm, cm = row_masks[name], col_masks[name]

        if rm.any():
            sims.extend(rs[rm].cpu().tolist())
        if cm.any():
            sims.extend(cs[cm].cpu().tolist())

    return float(np.mean(sims)) if sims else 0.0

def score_set(objs: List[Dict[str,Any]], ref, row_masks, col_masks, neg_pool, tag: str) -> np.ndarray:
    out = []
    for obj in tqdm(objs, desc=f"Score({tag})", dynamic_ncols=True):
        out.append(score_obj(obj, ref, row_masks, col_masks, neg_pool))
    return np.array(out, dtype=np.float32)

# ============================================================
# 13) Run pipeline
# ============================================================
print("\n" + "="*80)
print("5) Run GradAudit-style pipeline on P/S/N")
print("="*80)

print("Step A) Build reference gradient from P_calib")
ref_grads = build_reference_grad(P_calib, neg_pool_member)

print("\nStep B) Compute avg sims for P_calib vs N_calib")
P_rc = avg_rowcol_sims(P_calib, ref_grads, neg_pool_member, tag="P_calib")
N_rc = avg_rowcol_sims(N_calib, ref_grads, neg_pool_nonmem, tag="N_calib")

print("\nStep C) Build sensitive masks (tau-or-top20 rule)")
row_masks, col_masks, total_sens = build_sensitive_masks(P_rc, N_rc, tau=SENS_TAU, q=Q_TOP, min_keep_ratio=MIN_KEEP_RATIO)
print(f"✅ Sensitive dims: {total_sens}")
if total_sens == 0:
    print("⚠️ total_sens=0 -> scores may collapse. Consider lowering tau or using a higher-q (e.g., 0.9).")

print("\nStep D) Score probes: P_probe / S_probe / N_probe")
P_scores = score_set(P_probe, ref_grads, row_masks, col_masks, neg_pool_member, tag="P")
S_scores = score_set(S_probe, ref_grads, row_masks, col_masks, neg_pool_member, tag="S")
N_scores = score_set(N_probe, ref_grads, row_masks, col_masks, neg_pool_nonmem, tag="N")

# ============================================================
# 14) Evaluations
# ============================================================
print("\n" + "="*80)
print("6) Evaluations (Experiment 1 & 2)")
print("="*80)

def auc_pair(pos_scores: np.ndarray, neg_scores: np.ndarray) -> float:
    y = np.array([1]*len(pos_scores) + [0]*len(neg_scores), dtype=np.int32)
    s = np.concatenate([pos_scores, neg_scores], axis=0)
    if np.all(s == s[0]):
        return 0.5
    return float(roc_auc_score(y, s))

# Experiment 1: P vs S
auc_P_S = auc_pair(P_scores, S_scores)
print(f"🏆 Experiment-1 AUC (P vs S) = {auc_P_S:.4f}")

# sanity
auc_P_N = auc_pair(P_scores, N_scores)
auc_S_N = auc_pair(S_scores, N_scores)
print(f"Sanity AUC (P vs N) = {auc_P_N:.4f}")
print(f"Sanity AUC (S vs N) = {auc_S_N:.4f}")

# Experiment 2: choose threshold to make TPR(P)=80%, report FPR(S)
target_tpr = 0.80
thr = float(np.quantile(P_scores, 1.0 - target_tpr))
TPR_on_P = float(np.mean(P_scores >= thr))
FPR_on_S = float(np.mean(S_scores >= thr))

print("\n✅ Experiment-2 (Fix TPR on P=80%, check FPR on S)")
print(f"Threshold (from P quantile) = {thr:.6f}")
print(f"TPR on P (should be ~0.80)  = {TPR_on_P*100:.2f}%")
print(f"FPR on S (mis-kill rate)    = {FPR_on_S*100:.2f}%")

# Confusion matrix for P vs S
y_ps = np.array([1]*len(P_scores) + [0]*len(S_scores), dtype=np.int32)
s_ps = np.concatenate([P_scores, S_scores], axis=0)
pred_ps = (s_ps >= thr).astype(int)
cm_ps = confusion_matrix(y_ps, pred_ps)
print("\nConfusion Matrix on (P as pos, S as neg) under TPR=80% threshold:")
print(cm_ps)

print("\n✅ Done.")


## Mink P/S/N

In [ ]:
# ==============================================================================
# Text-only Min-K% baseline on your P/S/N setting (COCO Karpathy subset)
# - Data: member_images_1k.pkl, nonmember_images_1k.pkl (your prepared pkl objects)
# - P: member image i + member caption i (paired)
# - S: member image i + member caption j (j != i) (shuffled captions)
# - N: nonmember image + nonmember caption (unseen)
#
# Min-K% baseline (text-dominated):
# - Score caption using a causal LM token NLL
# - Min-K% = mean of smallest p% token NLLs
# - Final score = - MinK_NLL  (higher => more "familiar")
#
# Expected (theory-aligned):
# - AUC(P vs S) ~ 0.5 (both captions are from training-caption pool)
# - Experiment-2: Fix TPR(P)=80%, FPR(S) ~ 20% (random-like)
# ==============================================================================

import os, re, math, gc, pickle, random, warnings
from typing import Dict, Any, Optional, List, Tuple
from io import BytesIO

import numpy as np
from tqdm import tqdm

import torch
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

warnings.filterwarnings("ignore")

# ----------------------------
# 0) Config
# ----------------------------
MEMBER_PKL     = "/content/coco_data/member_images_1k.pkl"
NONMEMBER_PKL  = "/content/coco_data/nonmember_images_1k.pkl"

SEED = 42
NUM_EACH = 1000

# Min-K% hyperparams
MIN_K_FRAC = 0.20          # take lowest 20% token NLLs
MAX_LEN = 64               # cap caption length for speed/stability
BATCH_SIZE = 16            # batch captions for scoring
USE_AMP = True

# You asked "model is BLIP-ITM" - we keep it for pair-input consistency
# but Min-K% scoring is text-only (ability-boundary baseline).
BLIP_MODEL_NAME = "Salesforce/blip-itm-base-coco"

# Text LM for Min-K% scoring (fast & stable baseline)
# You can switch to "gpt2" if you want, but distilgpt2 is faster.
TEXT_LM_NAME = "distilgpt2"

# ----------------------------
# 1) Seed & device
# ----------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "❌ CUDA not available"
device = "cuda"
print(f"✅ Device: {device}")

# ----------------------------
# 2) Load BLIP-ITM (kept for pair-input consistency; not used in score)
# ----------------------------
print("\n" + "="*80)
print("Load BLIP-ITM (pair input consistency; NOT used in Min-K score)")
print("="*80)

from transformers import BlipProcessor, BlipForImageTextRetrieval

blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL_NAME)
blip_model = BlipForImageTextRetrieval.from_pretrained(BLIP_MODEL_NAME).to(device)
blip_model.eval()
print("✅ BLIP-ITM loaded")

# ----------------------------
# 3) Load Text LM for Min-K scoring
# ----------------------------
print("\n" + "="*80)
print("Load Text LM for Min-K% scoring")
print("="*80)

from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(TEXT_LM_NAME)
# GPT2 family has no pad_token by default -> set to eos for batching
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

lm = AutoModelForCausalLM.from_pretrained(TEXT_LM_NAME).to(device)
lm.eval()

print(f"✅ Text LM loaded: {TEXT_LM_NAME}")

# ----------------------------
# 4) Data I/O + construct P/S/N (each 1000)
# ----------------------------
print("\n" + "="*80)
print("Load PKLs + Build P/S/N (each 1000)")
print("="*80)

def load_pickle_list(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

members_all    = load_pickle_list(MEMBER_PKL)
nonmembers_all = load_pickle_list(NONMEMBER_PKL)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)

members_all    = members_all[:NUM_EACH]
nonmembers_all = nonmembers_all[:NUM_EACH]

def get_caption(obj: Dict[str,Any]) -> str:
    if "caption" in obj:
        c = obj["caption"]
        if isinstance(c, str): return c.strip()
        if isinstance(c, list) and c: return str(c[0]).strip()
    if "captions" in obj:
        c = obj["captions"]
        if isinstance(c, str): return c.strip()
        if isinstance(c, list) and c: return str(c[0]).strip()
    return ""

def set_caption(obj: Dict[str,Any], new_cap: str) -> Dict[str,Any]:
    x = dict(obj)
    x["caption"] = str(new_cap)
    return x

# P (paired)
P = members_all

# S (shuffled captions within member pool, ensure j != i)
caps = [get_caption(x) for x in members_all]
perm = list(range(len(caps)))
random.Random(SEED + 999).shuffle(perm)
for i in range(len(perm)):
    if perm[i] == i:
        j = (i + 1) % len(perm)
        perm[i], perm[j] = perm[j], perm[i]
S = [set_caption(members_all[i], caps[perm[i]]) for i in range(len(members_all))]

# N (nonmember)
N = nonmembers_all

print(f"✅ Built: P={len(P)} | S={len(S)} | N={len(N)}")
diff_cnt = sum(get_caption(P[i]) != get_caption(S[i]) for i in range(50))
print(f"Sanity (first 50): P vs S caption different count = {diff_cnt}/50")

# ----------------------------
# 5) Min-K% scoring (token NLL)
# ----------------------------
@torch.inference_mode()
def min_k_percent_score_captions(captions: List[str], min_k_frac: float = MIN_K_FRAC) -> np.ndarray:
    """
    Returns score = -mean(lowest p% token NLLs) per caption.
    Higher => more "familiar".
    """
    scores = []

    for i in tqdm(range(0, len(captions), BATCH_SIZE), desc="MinKScore", dynamic_ncols=True):
        batch_caps = captions[i:i+BATCH_SIZE]

        # tokenize
        enc = tok(
            batch_caps,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        ).to(device)

        input_ids = enc["input_ids"]
        attn = enc["attention_mask"]

        # causal LM next-token loss:
        # logits: [B, T, V]
        # shift for next-token prediction
        with torch.cuda.amp.autocast(enabled=(USE_AMP and torch.cuda.is_available())):
            out = lm(input_ids=input_ids, attention_mask=attn)
            logits = out.logits

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        shift_attn   = attn[:, 1:].contiguous()

        # per-token NLL: CE over vocab for each token position
        # result: [B, T-1]
        nll = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            reduction="none"
        ).view(shift_labels.size(0), shift_labels.size(1))

        # mask padding positions
        nll = nll * shift_attn

        # compute Min-K%: choose lowest p% among valid tokens
        for b in range(nll.size(0)):
            valid = nll[b][shift_attn[b].bool()]
            if valid.numel() == 0:
                scores.append(0.0)
                continue
            k = max(1, int(math.ceil(valid.numel() * float(min_k_frac))))
            # take smallest k NLL values
            vals, _ = torch.topk(valid, k=k, largest=False)
            mk = vals.mean()
            scores.append(float((-mk).item()))

    return np.array(scores, dtype=np.float32)

# captions for each set
P_caps = [get_caption(x) for x in P]
S_caps = [get_caption(x) for x in S]
N_caps = [get_caption(x) for x in N]

# basic caption sanity
assert all(isinstance(c, str) and len(c) > 0 for c in P_caps), "P has empty caption"
assert all(isinstance(c, str) and len(c) > 0 for c in S_caps), "S has empty caption"
assert all(isinstance(c, str) and len(c) > 0 for c in N_caps), "N has empty caption"

print("\n" + "="*80)
print("Scoring Min-K% (text-only) on P/S/N (each 1000)")
print("="*80)
print(f"Hyperparams: MIN_K_FRAC={MIN_K_FRAC}, MAX_LEN={MAX_LEN}, BATCH_SIZE={BATCH_SIZE}")

P_scores = min_k_percent_score_captions(P_caps, min_k_frac=MIN_K_FRAC)
S_scores = min_k_percent_score_captions(S_caps, min_k_frac=MIN_K_FRAC)
N_scores = min_k_percent_score_captions(N_caps, min_k_frac=MIN_K_FRAC)

print(f"✅ Done scoring: P={len(P_scores)}, S={len(S_scores)}, N={len(N_scores)}")

# ----------------------------
# 6) Evaluations
# ----------------------------
print("\n" + "="*80)
print("Evaluations")
print("="*80)

def auc_pair(pos: np.ndarray, neg: np.ndarray) -> float:
    y = np.array([1]*len(pos) + [0]*len(neg), dtype=np.int32)
    s = np.concatenate([pos, neg], axis=0)
    if np.all(s == s[0]):
        return 0.5
    return float(roc_auc_score(y, s))

auc_P_S = auc_pair(P_scores, S_scores)
auc_P_N = auc_pair(P_scores, N_scores)
auc_S_N = auc_pair(S_scores, N_scores)

print(f"🏆 AUC (P vs S) = {auc_P_S:.4f}   (expected ~0.5 for text-only Min-K)")
print(f"Sanity AUC (P vs N) = {auc_P_N:.4f}")
print(f"Sanity AUC (S vs N) = {auc_S_N:.4f}")

# Experiment-2: Fix TPR(P)=80%, compute FPR(S)
target_tpr = 0.80
thr = float(np.quantile(P_scores, 1.0 - target_tpr))
TPR_on_P = float(np.mean(P_scores >= thr))
FPR_on_S = float(np.mean(S_scores >= thr))

print("\n✅ Experiment-2 (Fix TPR on P=80%, check FPR on S)")
print(f"Threshold (from P quantile) = {thr:.6f}")
print(f"TPR on P (should be ~0.80)  = {TPR_on_P*100:.2f}%")
print(f"FPR on S (mis-kill rate)    = {FPR_on_S*100:.2f}%")

# Confusion matrix for P vs S under threshold
y_ps = np.array([1]*len(P_scores) + [0]*len(S_scores), dtype=np.int32)
s_ps = np.concatenate([P_scores, S_scores], axis=0)
pred_ps = (s_ps >= thr).astype(int)
cm_ps = confusion_matrix(y_ps, pred_ps)
print("\nConfusion Matrix (P as pos, S as neg) under TPR=80% threshold:")
print(cm_ps)

print("\n✅ Done.")


## ModRényi P/S/N

In [ ]:
# ==============================================================================
# Correct P/S/N ModRényi*-analog for BLIP-ITM (Salesforce/blip-itm-base-coco)
# - Data: member_images_1k.pkl / nonmember_images_1k.pkl (your prepared pkls)
# - P: paired members (img_i, cap_i)
# - S: shuffled caption members (img_i, cap_perm[i], perm[i]!=i)
# - N: nonmembers (img_u, cap_u)
#
# Score: "Retrieval-Rényi" (ModRényi*-analog)
#   - For each sample, build retrieval distribution over K candidates:
#       Image->Text: softmax( <img, text_candidates> )
#       Text->Image: softmax( <txt, img_candidates> )
#   - Compute Rényi entropy H_alpha of that distribution, aggregate over views,
#     fuse (image-view + text-view) and (alpha=0.5,2.0)
#
# KEY correctness:
#   * S uses pos_txt_idx = perm[i] (NOT i)
#   * P/S negatives are MEMBER-ONLY (to test pairing, not domain shift)
# ==============================================================================

import os, re, math, pickle, random, warnings
from typing import Dict, Any, Optional, List, Tuple
from io import BytesIO

import numpy as np
from PIL import Image
from tqdm import tqdm
import requests

import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_auc_score, confusion_matrix

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Config
# ---------------------------
MEMBER_PKL     = "/content/coco_data/member_images_1k.pkl"
NONMEMBER_PKL  = "/content/coco_data/nonmember_images_1k.pkl"

SEED = 42
NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

K = 32              # 1 pos + (K-1) neg
IMG_VIEWS = 3       # keep small; increase if you want more stability
TXT_VIEWS = 2
ALPHAS = (0.5, 2.0)

# For P/S: use member-only negatives to avoid domain leakage
USE_MEMBER_ONLY_NEG_FOR_PS = True

# Cache dir (works on Colab/local). If Colab, you can set "/content/coco_img_cache"
IMG_CACHE_DIR = "./coco_img_cache"
os.makedirs(IMG_CACHE_DIR, exist_ok=True)

# ---------------------------
# 1) Device & Model
# ---------------------------
assert torch.cuda.is_available(), "❌ CUDA not available"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

from transformers import BlipProcessor, BlipForImageTextRetrieval

MODEL_NAME = "Salesforce/blip-itm-base-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()

image_processor = processor.image_processor
tokenizer = processor.tokenizer

with torch.no_grad():
    LOGIT_SCALE = float(model.logit_scale.exp().item()) if hasattr(model, "logit_scale") else 1.0
print(f"✅ BLIP-ITM ready | logit_scale={LOGIT_SCALE:.3f}")

# ---------------------------
# 2) Load pkls + build P/S/N
# ---------------------------
def load_pickle_list(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

def get_caption(obj: Dict[str,Any]) -> str:
    if "caption" in obj:
        c = obj["caption"]
        if isinstance(c, str): return c.strip()
        if isinstance(c, list) and c: return str(c[0]).strip()
    if "captions" in obj:
        c = obj["captions"]
        if isinstance(c, str): return c.strip()
        if isinstance(c, list) and c: return str(c[0]).strip()
    return ""

def set_caption(obj: Dict[str,Any], new_cap: str) -> Dict[str,Any]:
    x = dict(obj)
    x["caption"] = str(new_cap)
    return x

members_all    = load_pickle_list(MEMBER_PKL)
nonmembers_all = load_pickle_list(NONMEMBER_PKL)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)

members_all    = members_all[:NUM_MEMBERS]
nonmembers_all = nonmembers_all[:NUM_NONMEMBERS]

P = members_all

caps = [get_caption(x) for x in members_all]
perm = list(range(len(caps)))
random.Random(SEED + 999).shuffle(perm)

# force perm[i] != i
for i in range(len(perm)):
    if perm[i] == i:
        j = (i + 1) % len(perm)
        perm[i], perm[j] = perm[j], perm[i]

S = [set_caption(members_all[i], caps[perm[i]]) for i in range(len(members_all))]
N = nonmembers_all

print(f"✅ Built: P={len(P)} | S={len(S)} | N={len(N)}")
diff_cnt = sum(get_caption(P[i]) != get_caption(S[i]) for i in range(50))
print(f"Sanity (first 50): P vs S caption different count = {diff_cnt}/50")

# ---------------------------
# 3) COCO image loader + cache
# ---------------------------
def _coco_url_from_image_field(image_field: str) -> Optional[str]:
    # e.g., "train2014/COCO_train2014_000000123.jpg"
    if not isinstance(image_field, str) or "/" not in image_field:
        return None
    split_dir, filename = image_field.split("/", 1)
    return f"http://images.cocodataset.org/{split_dir}/{filename}"

def load_coco_image(obj: Dict[str,Any]) -> Optional[Image.Image]:
    image_field = obj.get("image")
    url = _coco_url_from_image_field(image_field)
    if url is None:
        return None

    fname = image_field.replace("/", "_")
    fpath = os.path.join(IMG_CACHE_DIR, fname)

    try:
        if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
            return Image.open(fpath).convert("RGB")
    except:
        pass

    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            return None
        img = Image.open(BytesIO(r.content)).convert("RGB")
        try:
            img.save(fpath, format="JPEG", quality=95)
        except:
            pass
        return img
    except:
        return None

def load_pairs(data_list: List[Dict[str,Any]], tag: str) -> Tuple[List[Image.Image], List[str]]:
    imgs, caps_out = [], []
    fail_img, fail_cap = 0, 0
    for obj in tqdm(data_list, desc=f"LoadPairs({tag})", dynamic_ncols=True):
        cap = get_caption(obj)
        if not cap:
            fail_cap += 1
            continue
        im = load_coco_image(obj)
        if im is None:
            fail_img += 1
            continue
        imgs.append(im)
        caps_out.append(cap)
    print(f"📊 {tag}: ok={len(imgs)} | fail_img={fail_img} | fail_cap={fail_cap}")
    return imgs, caps_out

print("\n" + "="*80)
print("Encoding image/text features")
print("="*80)

P_imgs, P_caps = load_pairs(P, "P")
S_imgs, S_caps = load_pairs(S, "S")
N_imgs, N_caps = load_pairs(N, "N")

# align lengths (drop to min)
M = min(len(P_imgs), len(S_imgs), len(P_caps), len(S_caps))
U = min(len(N_imgs), len(N_caps))

P_imgs, P_caps = P_imgs[:M], P_caps[:M]
S_imgs, S_caps = S_imgs[:M], S_caps[:M]
N_imgs, N_caps = N_imgs[:U], N_caps[:U]

# IMPORTANT index maps (after truncation)
P_pos_txt_idx = np.arange(M, dtype=np.int64)            # P: cap_i
S_pos_txt_idx = np.array([perm[i] % M for i in range(M)], dtype=np.int64)  # S: cap_perm[i]
N_pos_txt_idx = np.arange(U, dtype=np.int64) + M        # N: cap in second block

# ---------------------------
# 4) Encode BLIP embeddings
# ---------------------------
@torch.inference_mode()
def encode_image_batch(pils: List[Image.Image], bs: int = 64) -> torch.Tensor:
    out = []
    for i in range(0, len(pils), bs):
        chunk = pils[i:i+bs]
        inputs = image_processor(images=chunk, return_tensors="pt")
        pixel_values = inputs.pixel_values.to(device)
        with torch.cuda.amp.autocast():
            vision_outputs = model.vision_model(pixel_values=pixel_values)
            image_embeds = vision_outputs[1]              # pooler_output
            image_feat = model.vision_proj(image_embeds)  # [B,D]
            z = F.normalize(image_feat, dim=-1)
        out.append(z.float().cpu())
    return torch.cat(out, dim=0)

@torch.inference_mode()
def encode_text_batch(texts: List[str], bs: int = 128, max_len: int = 77) -> torch.Tensor:
    out = []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i+bs]
        inputs = tokenizer(
            text=chunk,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=max_len
        )
        input_ids = inputs.input_ids.to(device)
        attention_mask = inputs.attention_mask.to(device)
        with torch.cuda.amp.autocast():
            text_outputs = model.text_encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )
            text_embeds = text_outputs.last_hidden_state[:, 0, :]
            text_feat = model.text_proj(text_embeds)
            z = F.normalize(text_feat, dim=-1)
        out.append(z.float().cpu())
    return torch.cat(out, dim=0)

print("⏳ EncodeImg P...")
P_img_emb = encode_image_batch(P_imgs, bs=64)
print("⏳ EncodeImg N...")
N_img_emb = encode_image_batch(N_imgs, bs=64)

print("⏳ EncodeTxt P...")
P_txt_emb = encode_text_batch(P_caps, bs=128, max_len=77)
print("⏳ EncodeTxt N...")
N_txt_emb = encode_text_batch(N_caps, bs=128, max_len=77)

# banks: [member block, nonmember block]
all_txt_emb = torch.cat([P_txt_emb, N_txt_emb], dim=0)  # [Nc,D]
all_img_emb = torch.cat([P_img_emb, N_img_emb], dim=0)  # [Ni,D]
Nc = all_txt_emb.size(0)
Ni = all_img_emb.size(0)

print(f"✅ Encoded: P_img={P_img_emb.shape}, N_img={N_img_emb.shape}")
print(f"✅ Text bank: all_txt={all_txt_emb.shape} (Nc={Nc})")
print(f"✅ Image bank: all_img={all_img_emb.shape} (Ni={Ni})")

# ---------------------------
# 5) Views (lightweight)
# ---------------------------
def _normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def make_image_views(pil: Image.Image, want: int = IMG_VIEWS) -> List[Image.Image]:
    views = [pil]
    if want <= 1:
        return views
    try:
        w, h = pil.size
        s = int(min(w, h) * 0.9)
        left = (w - s) // 2
        top = (h - s) // 2
        crop = pil.crop((left, top, left + s, top + s)).resize((w, h))
        views.append(crop)
    except:
        pass
    if want <= 2:
        return views[:want]
    try:
        views.append(pil.transpose(Image.FLIP_LEFT_RIGHT))
    except:
        pass
    return views[:want]

def make_text_views(caption: str, want: int = TXT_VIEWS) -> List[str]:
    base = str(caption or "").strip()
    cand = [base, _normalize_spaces(base)]
    out, seen = [], set()
    for c in cand:
        if c and c not in seen:
            out.append(c); seen.add(c)
    return out[:max(1, want)]

@torch.inference_mode()
def encode_one_image(pil: Image.Image) -> torch.Tensor:
    inputs = image_processor(images=[pil], return_tensors="pt")
    pixel_values = inputs.pixel_values.to(device)
    with torch.cuda.amp.autocast():
        vision_outputs = model.vision_model(pixel_values=pixel_values)
        image_embeds = vision_outputs[1]
        image_feat = model.vision_proj(image_embeds)
        z = F.normalize(image_feat, dim=-1)
    return z[0].float()

@torch.inference_mode()
def encode_one_text(text: str, max_len: int = 77) -> torch.Tensor:
    inputs = tokenizer(
        text=[text],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=max_len
    )
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)
    with torch.cuda.amp.autocast():
        text_outputs = model.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        text_embeds = text_outputs.last_hidden_state[:, 0, :]
        text_feat = model.text_proj(text_embeds)
        z = F.normalize(text_feat, dim=-1)
    return z[0].float()

# ---------------------------
# 6) Retrieval logp
# ---------------------------
txt_pool_members = np.arange(0, M, dtype=np.int64)
img_pool_members = np.arange(0, M, dtype=np.int64)
txt_pool_all     = np.arange(0, Nc, dtype=np.int64)
img_pool_all     = np.arange(0, Ni, dtype=np.int64)

def image_view_logp(img_vec: torch.Tensor, pos_txt_idx: int, Kcand: int, neg_pool_txt: np.ndarray) -> np.ndarray:
    pool = neg_pool_txt[neg_pool_txt != pos_txt_idx] if pos_txt_idx in neg_pool_txt else neg_pool_txt
    neg_idx = np.random.choice(pool, size=Kcand-1, replace=False)
    cand_idx = np.concatenate([[pos_txt_idx], neg_idx])
    cand_txt = all_txt_emb[cand_idx].to(device)
    logits = LOGIT_SCALE * (cand_txt @ img_vec)
    logp = torch.log_softmax(logits, dim=0).unsqueeze(0)  # [1,K]
    return logp.detach().cpu().numpy().astype(np.float32)

def text_view_logp(txt_vec: torch.Tensor, pos_img_idx: int, Kcand: int, neg_pool_img: np.ndarray) -> np.ndarray:
    pool = neg_pool_img[neg_pool_img != pos_img_idx] if pos_img_idx in neg_pool_img else neg_pool_img
    neg_idx = np.random.choice(pool, size=Kcand-1, replace=False)
    cand_idx = np.concatenate([[pos_img_idx], neg_idx])
    cand_img = all_img_emb[cand_idx].to(device)
    logits = LOGIT_SCALE * (cand_img @ txt_vec)
    logp = torch.log_softmax(logits, dim=0).unsqueeze(0)
    return logp.detach().cpu().numpy().astype(np.float32)

# ---------------------------
# 7) Rényi entropy + fusion
# ---------------------------
def renyi_entropy_from_logp(logp_vec: np.ndarray, alpha: float) -> float:
    v = logp_vec.reshape(-1)
    a = alpha * v
    m = np.max(a)
    return float((1.0/(1.0-alpha)) * (m + np.log(np.exp(a - m).sum())))

def renyi_seq_from_logp(logp_seq: np.ndarray, alpha: float) -> float:
    T = logp_seq.shape[0]
    vals = [renyi_entropy_from_logp(logp_seq[t], alpha) for t in range(T)]
    return float(np.mean(vals)) if vals else float("nan")

def robust_aggregate(values: List[float], trim: float = 0.1) -> float:
    arr = np.array([v for v in values if np.isfinite(v)], dtype=float)
    if arr.size == 0: return float("nan")
    arr.sort()
    k = int(math.floor(trim * arr.size))
    if k * 2 < arr.size:
        arr = arr[k: arr.size - k]
    return float(np.mean(arr))

def fused_score(img_logps: List[np.ndarray], txt_logps: List[np.ndarray], alphas=ALPHAS) -> float:
    parts = []
    for logp_views in [img_logps, txt_logps]:
        if len(logp_views) == 0:
            parts.extend([float("nan")] * len(alphas))
            continue
        for a in alphas:
            per_view = []
            for lp in logp_views:
                if lp is None or not np.isfinite(lp).all():
                    continue
                per_view.append(renyi_seq_from_logp(lp, a))
            if not per_view:
                parts.append(float("nan")); continue
            H_bar = robust_aggregate(per_view, trim=0.1)
            parts.append(-H_bar)  # -entropy: higher => more confident
    valid = np.array([x for x in parts if np.isfinite(x)], dtype=float)
    if valid.size == 0:
        return float("nan")
    if valid.size >= 2 and np.std(valid) > 1e-12:
        z = (valid - np.mean(valid)) / np.std(valid)
        return float(np.mean(z))
    return float(np.mean(valid))

# ---------------------------
# 8) Score one sample
# ---------------------------
def score_item(img: Image.Image, cap: str, pos_txt_idx: int, pos_img_idx: int,
               neg_pool_txt: np.ndarray, neg_pool_img: np.ndarray) -> Optional[float]:
    img_logps, txt_logps = [], []
    try:
        for v in make_image_views(img, IMG_VIEWS):
            v_z = encode_one_image(v)
            img_logps.append(image_view_logp(v_z, int(pos_txt_idx), K, neg_pool_txt))
    except:
        pass
    try:
        for t in make_text_views(cap, TXT_VIEWS):
            t_z = encode_one_text(t)
            txt_logps.append(text_view_logp(t_z, int(pos_img_idx), K, neg_pool_img))
    except:
        pass
    if len(img_logps) == 0 and len(txt_logps) == 0:
        return None
    s = fused_score(img_logps, txt_logps, alphas=ALPHAS)
    if not math.isfinite(s):
        return None
    return float(s)

# ---------------------------
# 9) Score P/S/N (correct pools)
# ---------------------------
def score_all_psn() -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    if USE_MEMBER_ONLY_NEG_FOR_PS:
        neg_pool_txt_PS = txt_pool_members
        neg_pool_img_PS = img_pool_members
    else:
        neg_pool_txt_PS = txt_pool_all
        neg_pool_img_PS = img_pool_all

    neg_pool_txt_N = txt_pool_all
    neg_pool_img_N = img_pool_all

    P_scores, S_scores, N_scores = [], [], []

    for i in tqdm(range(M), desc="Score(P)", dynamic_ncols=True):
        s = score_item(P_imgs[i], P_caps[i],
                       pos_txt_idx=int(P_pos_txt_idx[i]),
                       pos_img_idx=int(i),
                       neg_pool_txt=neg_pool_txt_PS,
                       neg_pool_img=neg_pool_img_PS)
        P_scores.append(np.nan if s is None else s)

    for i in tqdm(range(M), desc="Score(S)", dynamic_ncols=True):
        # CRITICAL: pos_txt_idx is perm[i] (caption source)
        s = score_item(S_imgs[i], S_caps[i],
                       pos_txt_idx=int(S_pos_txt_idx[i]),
                       pos_img_idx=int(i),
                       neg_pool_txt=neg_pool_txt_PS,
                       neg_pool_img=neg_pool_img_PS)
        S_scores.append(np.nan if s is None else s)

    for j in tqdm(range(U), desc="Score(N)", dynamic_ncols=True):
        # N has its own indices in banks: text=M+j, image=M+j
        s = score_item(N_imgs[j], N_caps[j],
                       pos_txt_idx=int(N_pos_txt_idx[j]),
                       pos_img_idx=int(M + j),
                       neg_pool_txt=neg_pool_txt_N,
                       neg_pool_img=neg_pool_img_N)
        N_scores.append(np.nan if s is None else s)

    return (np.array(P_scores, dtype=np.float32),
            np.array(S_scores, dtype=np.float32),
            np.array(N_scores, dtype=np.float32))

def compute_auc(pos: np.ndarray, neg: np.ndarray) -> float:
    s = np.concatenate([pos, neg], axis=0)
    y = np.array([1]*len(pos) + [0]*len(neg), dtype=np.int32)
    ok = np.isfinite(s)
    s = s[ok]; y = y[ok]
    if s.size == 0 or np.all(s == s[0]):
        return 0.5
    return float(roc_auc_score(y, s))

def exp2_fpr_at_tpr(Ps: np.ndarray, Ss: np.ndarray, target_tpr: float = 0.80):
    Pn = Ps[np.isfinite(Ps)]
    Sn = Ss[np.isfinite(Ss)]
    thr = float(np.quantile(Pn, 1.0 - target_tpr))
    tpr = float(np.mean(Pn >= thr))
    fpr = float(np.mean(Sn >= thr))
    return thr, tpr, fpr

# ---------------------------
# 10) Run + Evaluate
# ---------------------------
print("\n" + "="*80)
print("Scoring Retrieval-Rényi (ModRényi*-analog) on P/S/N")
print("="*80)
print(f"Hyperparams: K={K}, IMG_VIEWS={IMG_VIEWS}, TXT_VIEWS={TXT_VIEWS}, ALPHAS={ALPHAS}")
print(f"Negatives for P/S: {'MEMBER-ONLY' if USE_MEMBER_ONLY_NEG_FOR_PS else 'ALL'}")

P_scores, S_scores, N_scores = score_all_psn()

print("\n" + "="*80)
print("Evaluations")
print("="*80)

auc_ps = compute_auc(P_scores, S_scores)
auc_pn = compute_auc(P_scores, N_scores)
auc_sn = compute_auc(S_scores, N_scores)

print(f"🏆 AUC (P vs S) = {auc_ps:.4f}")
print(f"Sanity AUC (P vs N) = {auc_pn:.4f}")
print(f"Sanity AUC (S vs N) = {auc_sn:.4f}")

thr, tpr, fpr = exp2_fpr_at_tpr(P_scores, S_scores, target_tpr=0.80)
print("\n✅ Experiment-2 (Fix TPR on P=80%, check FPR on S)")
print(f"Threshold (from P quantile) = {thr:.6f}")
print(f"TPR on P (should be ~0.80)  = {tpr*100:.2f}%")
print(f"FPR on S (mis-kill rate)    = {fpr*100:.2f}%")

Pn = P_scores[np.isfinite(P_scores)]
Sn = S_scores[np.isfinite(S_scores)]
y_ps = np.array([1]*len(Pn) + [0]*len(Sn), dtype=np.int32)
s_ps = np.concatenate([Pn, Sn], axis=0)
pred_ps = (s_ps >= thr).astype(int)
cm_ps = confusion_matrix(y_ps, pred_ps)
print("\nConfusion Matrix on (P as pos, S as neg) under TPR=80% threshold:")
print(cm_ps)

print("\n✅ Done.")


## NA-PDD P/S/N

In [ ]:
# ==============================================================================
# NA-PDD for BLIP-ITM on COCO (Karpathy) - P/S/N Experiment
# Multi-Modal Neuron Activation Pattern Detection
# - Data: member_images_1k.pkl, nonmember_images_1k.pkl
# - Model: Salesforce/blip-itm-base-coco
# - Components: Vision Encoder + Projections + Text Encoder
# - Experiments:
#   (1) AUC on P vs S
#   (2) FPR on S when TPR(P)=80%
# ==============================================================================

import os, io, re, gc, math, pickle, random, warnings
from typing import Dict, Any, Optional, List, Tuple
from collections import Counter
from io import BytesIO

import numpy as np
from PIL import Image
import requests

import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, accuracy_score, precision_recall_fscore_support

warnings.filterwarnings("ignore")

# ============================================================
# 0) Configuration
# ============================================================
MEMBER_PKL     = "/content/coco_data/member_images_1k.pkl"
NONMEMBER_PKL  = "/content/coco_data/nonmember_images_1k.pkl"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

CALIB_P = 200
CALIB_N = 200
PROBE_P = 800
PROBE_S = 800
PROBE_N = 800

# NA-PDD Parameters
ACTIVATION_THRESHOLD = 1.0
TOP_N_LAYERS = 15
RELATIVE_RATIO_THRESHOLD = 1.0

# Image cache
IMG_CACHE_DIR = "/content/coco_img_cache"
os.makedirs(IMG_CACHE_DIR, exist_ok=True)

assert torch.cuda.is_available(), "❌ CUDA not available"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True
print(f"✅ Device: {device}")

# ============================================================
# 1) Load BLIP-ITM model
# ============================================================
print("\n" + "="*80)
print("1) Load BLIP-ITM model")
print("="*80)

try:
    from transformers import BlipProcessor, BlipForImageTextRetrieval
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.28.0"])
    from transformers import BlipProcessor, BlipForImageTextRetrieval

MODEL_NAME = "Salesforce/blip-itm-base-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()

image_processor = processor.image_processor
tokenizer = processor.tokenizer
print("✅ BLIP-ITM loaded")

# ============================================================
# 2) Load data and construct P/S/N
# ============================================================
print("\n" + "="*80)
print("2) Load data + construct P/S/N")
print("="*80)

def load_pickle_list(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

members_all    = load_pickle_list(MEMBER_PKL)
nonmembers_all = load_pickle_list(NONMEMBER_PKL)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)

members_all    = members_all[:NUM_MEMBERS]
nonmembers_all = nonmembers_all[:NUM_NONMEMBERS]

def get_caption_from_obj(obj: Dict[str,Any]) -> str:
    if "caption" in obj:
        c = obj["caption"]
        if isinstance(c, str): return c.strip()
        if isinstance(c, list) and len(c) > 0: return str(c[0]).strip()
        return ""
    if "captions" in obj:
        c = obj["captions"]
        if isinstance(c, str): return c.strip()
        if isinstance(c, list) and len(c) > 0: return str(c[0]).strip()
        return ""
    return ""

def set_caption(obj: Dict[str,Any], new_cap: str) -> Dict[str,Any]:
    x = dict(obj)
    x["caption"] = str(new_cap)
    return x

# P: paired (image + original caption)
P_all = members_all

# S: shuffled captions within member pool (j != i)
caps = [get_caption_from_obj(x) for x in members_all]
perm = list(range(len(caps)))
random.Random(SEED+999).shuffle(perm)
for i in range(len(perm)):
    if perm[i] == i:
        j = (i + 1) % len(perm)
        perm[i], perm[j] = perm[j], perm[i]

S_all = [set_caption(members_all[i], caps[perm[i]]) for i in range(len(members_all))]

# N: nonmembers
N_all = nonmembers_all

print(f"✅ Built sets: P={len(P_all)} | S={len(S_all)} | N={len(N_all)}")
diff_cnt = sum(get_caption_from_obj(P_all[i]) != get_caption_from_obj(S_all[i]) for i in range(50))
print(f"Sanity (first 50): P vs S caption different = {diff_cnt}/50")

# Split
P_calib = P_all[:CALIB_P]
N_calib = N_all[:CALIB_N]

P_probe = P_all[CALIB_P:CALIB_P+PROBE_P]
S_probe = S_all[CALIB_P:CALIB_P+PROBE_S]
N_probe = N_all[CALIB_N:CALIB_N+PROBE_N]

print(f"Calib: P={len(P_calib)}, N={len(N_calib)}")
print(f"Probe: P={len(P_probe)}, S={len(S_probe)}, N={len(N_probe)}")

# ============================================================
# 3) Image loading with cache
# ============================================================
print("\n" + "="*80)
print("3) Image loader")
print("="*80)

def _coco_url_from_image_field(image_field: str) -> Optional[str]:
    if not isinstance(image_field, str) or "/" not in image_field:
        return None
    split_dir, filename = image_field.split("/", 1)
    return f"http://images.cocodataset.org/{split_dir}/{filename}"

def load_coco_image(obj: Dict[str,Any]) -> Optional[Image.Image]:
    image_field = obj.get("image")
    url = _coco_url_from_image_field(image_field)
    if url is None:
        return None

    fname = image_field.replace("/", "_")
    fpath = os.path.join(IMG_CACHE_DIR, fname)

    try:
        if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
            return Image.open(fpath).convert("RGB")
    except:
        pass

    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            return None
        img = Image.open(BytesIO(r.content)).convert("RGB")
        try:
            img.save(fpath, format="JPEG", quality=95)
        except:
            pass
        return img
    except:
        return None

print("✅ Image loader ready")

# ============================================================
# 4) Multi-Component Hook Registration
# ============================================================
print("\n" + "="*80)
print("4) NA-PDD: Multi-Component Hook Registration")
print("="*80)

activations = {}
activated_neurons = {}

def get_activation_hook(name, use_threshold=True):
    """Hook to capture neuron activations"""
    def hook(module, input, output):
        if isinstance(output, tuple):
            output = output[0]

        if output is None or not isinstance(output, torch.Tensor):
            return

        if output.dim() >= 2:
            if use_threshold:
                activation_mask = (output > ACTIVATION_THRESHOLD).detach().cpu()
            else:
                activation_mask = (output.abs() > ACTIVATION_THRESHOLD).detach().cpu()

            activated_neurons[name] = activation_mask

    return hook

hooks = []
vision_hooks = 0
projection_hooks = 0
text_hooks = 0

print("\n🔧 Inspecting BLIP-ITM structure...")

# Inspect model structure
print("\n  Model components:")
for name, module in model.named_children():
    print(f"    - {name}: {type(module).__name__}")

# ==========================================
# Component 1: Vision Encoder
# ==========================================
print("\n📌 Component 1: Vision Encoder")

if hasattr(model, 'vision_model'):
    vision_model = model.vision_model

    if hasattr(vision_model, 'encoder') and hasattr(vision_model.encoder, 'layers'):
        vision_layers = vision_model.encoder.layers
        print(f"  Found {len(vision_layers)} vision encoder layers")

        if len(vision_layers) > 0:
            first_layer = vision_layers[0]
            print(f"  First layer type: {type(first_layer).__name__}")

            for attr in ['mlp', 'ffn', 'fc1', 'fc2']:
                if hasattr(first_layer, attr):
                    print(f"    ✅ Has '{attr}'")

        for i, layer in enumerate(vision_layers):
            if hasattr(layer, 'mlp'):
                mlp = layer.mlp
                for act_name in ['act_fn', 'act', 'activation_fn', 'gelu']:
                    if hasattr(mlp, act_name):
                        try:
                            act_fn = getattr(mlp, act_name)
                            hook = act_fn.register_forward_hook(
                                get_activation_hook(f'vision_layer_{i}_mlp_act', use_threshold=True)
                            )
                            hooks.append(hook)
                            vision_hooks += 1
                            break
                        except:
                            pass

            if vision_hooks == i:
                try:
                    hook = layer.register_forward_hook(
                        get_activation_hook(f'vision_layer_{i}_output', use_threshold=False)
                    )
                    hooks.append(hook)
                    vision_hooks += 1
                except:
                    pass

print(f"  ✅ Registered {vision_hooks} vision hooks")

# ==========================================
# Component 2: Projection Layers
# ==========================================
print("\n📌 Component 2: Projection Layers")

if hasattr(model, 'vision_proj'):
    try:
        hook = model.vision_proj.register_forward_hook(
            get_activation_hook('projection_vision_proj', use_threshold=False)
        )
        hooks.append(hook)
        projection_hooks += 1
        print(f"  ✅ Registered vision_proj hook")
    except:
        pass

if hasattr(model, 'text_proj'):
    try:
        hook = model.text_proj.register_forward_hook(
            get_activation_hook('projection_text_proj', use_threshold=False)
        )
        hooks.append(hook)
        projection_hooks += 1
        print(f"  ✅ Registered text_proj hook")
    except:
        pass

if hasattr(model, 'itm_head'):
    try:
        hook = model.itm_head.register_forward_hook(
            get_activation_hook('projection_itm_head', use_threshold=False)
        )
        hooks.append(hook)
        projection_hooks += 1
        print(f"  ✅ Registered itm_head hook")
    except:
        pass

print(f"  ✅ Registered {projection_hooks} projection hooks")

# ==========================================
# Component 3: Text Encoder
# ==========================================
print("\n📌 Component 3: Text Encoder")

if hasattr(model, 'text_encoder'):
    text_encoder = model.text_encoder

    if hasattr(text_encoder, 'encoder') and hasattr(text_encoder.encoder, 'layer'):
        text_layers = text_encoder.encoder.layer
        print(f"  Found {len(text_layers)} text encoder layers")

        if len(text_layers) > 0:
            first_layer = text_layers[0]
            print(f"  First layer type: {type(first_layer).__name__}")

        for i, layer in enumerate(text_layers):
            if hasattr(layer, 'attention'):
                try:
                    hook = layer.attention.register_forward_hook(
                        get_activation_hook(f'text_layer_{i}_attention', use_threshold=False)
                    )
                    hooks.append(hook)
                    text_hooks += 1
                except:
                    pass

            if hasattr(layer, 'intermediate'):
                intermediate = layer.intermediate
                for act_name in ['intermediate_act_fn', 'act_fn', 'gelu']:
                    if hasattr(intermediate, act_name):
                        try:
                            act_fn = getattr(intermediate, act_name)
                            hook = act_fn.register_forward_hook(
                                get_activation_hook(f'text_layer_{i}_ffn_act', use_threshold=True)
                            )
                            hooks.append(hook)
                            text_hooks += 1
                            break
                        except:
                            pass

print(f"  ✅ Registered {text_hooks} text hooks")

# Summary
print(f"\n{'='*80}")
print(f"📊 Hook Registration Summary:")
print(f"  Vision Encoder:    {vision_hooks} hooks")
print(f"  Projections:       {projection_hooks} hooks")
print(f"  Text Encoder:      {text_hooks} hooks")
print(f"  Total:             {len(hooks)} hooks")
print(f"{'='*80}")

if len(hooks) == 0:
    raise RuntimeError("❌ No hooks registered! Cannot proceed.")

# ============================================================
# 5) Process samples - Simple progress logging
# ============================================================
def process_sample_activation(obj: Dict[str,Any], sample_id: int) -> Dict[str, Any]:
    """Process a single sample and collect activated neuron indices"""
    activations.clear()
    activated_neurons.clear()

    img = load_coco_image(obj)
    if img is None:
        return {'sample_id': sample_id, 'neural_signature': {}}

    caption = get_caption_from_obj(obj)
    if not caption:
        return {'sample_id': sample_id, 'neural_signature': {}}

    try:
        pixel_values = image_processor(img, return_tensors="pt").pixel_values.to(device)
        text_inputs = tokenizer(
            caption,
            padding="max_length",
            truncation=True,
            max_length=77,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(
                pixel_values=pixel_values,
                input_ids=text_inputs.input_ids,
                attention_mask=text_inputs.attention_mask
            )

        sample_neural_signature = {}
        for key, value in activated_neurons.items():
            try:
                if len(value.shape) >= 2:
                    mask = value.squeeze(0).numpy()

                    if len(mask.shape) == 2:
                        active_neurons = set()
                        for pos in range(mask.shape[0]):
                            active_indices = np.where(mask[pos])[0]
                            active_neurons.update(active_indices)

                        if active_neurons:
                            sample_neural_signature[key] = list(active_neurons)
                    elif len(mask.shape) == 1:
                        active_indices = np.where(mask)[0]
                        if len(active_indices) > 0:
                            sample_neural_signature[key] = active_indices.tolist()
            except Exception as e:
                pass

        torch.cuda.empty_cache()
        return {
            'sample_id': sample_id,
            'neural_signature': sample_neural_signature
        }

    except Exception as e:
        torch.cuda.empty_cache()
        return {'sample_id': sample_id, 'neural_signature': {}}

def collect_activations(samples: List[Dict[str,Any]], tag: str) -> List[Dict[str,Any]]:
    """Collect neuron activations - simple progress"""
    results = []
    total = len(samples)

    print(f"  Processing {total} samples for {tag}...")

    for i, obj in enumerate(samples):
        result = process_sample_activation(obj, i)
        results.append(result)

        # Print progress every 50 samples
        if (i + 1) % 50 == 0:
            print(f"    {i+1}/{total} completed ({100*(i+1)/total:.1f}%)")
            torch.cuda.empty_cache()
            gc.collect()

    print(f"  ✅ {tag} completed: {total} samples processed")
    return results

# ============================================================
# 6) Analyze neuron activation patterns
# ============================================================
def analyze_neuron_activation_patterns(member_samples, nonmember_samples):
    """Analyze activation patterns to find discriminative neurons"""
    results = {}

    layer_names = set()
    for sample in member_samples + nonmember_samples:
        layer_names.update(sample['neural_signature'].keys())

    print(f"  Analyzing {len(layer_names)} layers...")

    for layer_name in layer_names:
        member_neuron_counts = Counter()
        nonmember_neuron_counts = Counter()

        for sample in member_samples:
            if layer_name in sample['neural_signature']:
                member_neuron_counts.update(sample['neural_signature'][layer_name])

        for sample in nonmember_samples:
            if layer_name in sample['neural_signature']:
                nonmember_neuron_counts.update(sample['neural_signature'][layer_name])

        member_freq = {n: count / len(member_samples) for n, count in member_neuron_counts.items()}
        nonmember_freq = {n: count / len(nonmember_samples) for n, count in nonmember_neuron_counts.items()}

        member_dominant = {}
        for neuron, freq in member_freq.items():
            if neuron not in nonmember_freq or freq > nonmember_freq[neuron] * 1.5:
                member_dominant[neuron] = freq

        nonmember_dominant = {}
        for neuron, freq in nonmember_freq.items():
            if neuron not in member_freq or freq > member_freq[neuron] * 1.5:
                nonmember_dominant[neuron] = freq

        common_neurons = {}
        for neuron in set(member_freq.keys()) & set(nonmember_freq.keys()):
            if neuron not in member_dominant and neuron not in nonmember_dominant:
                common_neurons[neuron] = (member_freq[neuron], nonmember_freq[neuron])

        results[layer_name] = {
            'member_dominant': member_dominant,
            'nonmember_dominant': nonmember_dominant,
            'common_neurons': common_neurons,
            'member_freq': member_freq,
            'nonmember_freq': nonmember_freq
        }

    return results

def build_reference_patterns(P_samples, N_samples):
    """Build reference activation patterns"""
    print(f"  Building reference patterns: P={len(P_samples)}, N={len(N_samples)}")
    return analyze_neuron_activation_patterns(P_samples, N_samples)

def calculate_layer_discrimination_scores(reference_patterns):
    """Calculate discrimination scores"""
    layer_scores = {}

    for layer_name, data in reference_patterns.items():
        member_dominant_count = len(data['member_dominant'])
        nonmember_dominant_count = len(data['nonmember_dominant'])
        discrimination_score = member_dominant_count - nonmember_dominant_count
        layer_scores[layer_name] = discrimination_score

    return layer_scores

def select_discriminative_layers(layer_scores, top_n=TOP_N_LAYERS):
    """Select most discriminative layers"""
    sorted_scores = sorted(layer_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    selected_layers = [layer for layer, score in sorted_scores[:top_n]]

    print(f"\n  📋 Selected {len(selected_layers)} most discriminative layers:")

    vision_count = sum(1 for l in selected_layers if 'vision' in l)
    proj_count = sum(1 for l in selected_layers if 'projection' in l)
    text_count = sum(1 for l in selected_layers if 'text' in l)

    print(f"    Vision:      {vision_count}")
    print(f"    Projection:  {proj_count}")
    print(f"    Text:        {text_count}")

    return selected_layers

# ============================================================
# 7) Membership prediction
# ============================================================
def predict_membership_by_relative_ratio(sample, reference_patterns, discriminative_layers, threshold=RELATIVE_RATIO_THRESHOLD):
    """Predict membership using relative ratio"""
    if not sample['neural_signature']:
        return 0, 0, 0

    layers_counted = 0
    total_member_ratio = 0
    total_nonmember_ratio = 0

    for layer_name in discriminative_layers:
        if layer_name not in sample['neural_signature'] or layer_name not in reference_patterns:
            continue

        layer_data = reference_patterns[layer_name]
        sample_neurons = set(sample['neural_signature'][layer_name])

        if not sample_neurons:
            continue

        member_dominant_set = set(layer_data['member_dominant'].keys())
        member_overlap = len(sample_neurons.intersection(member_dominant_set))

        nonmember_dominant_set = set(layer_data['nonmember_dominant'].keys())
        nonmember_overlap = len(sample_neurons.intersection(nonmember_dominant_set))

        member_ratio = member_overlap / len(member_dominant_set) if len(member_dominant_set) > 0 else 0
        nonmember_ratio = nonmember_overlap / len(nonmember_dominant_set) if len(nonmember_dominant_set) > 0 else 0

        total_member_ratio += member_ratio
        total_nonmember_ratio += nonmember_ratio
        layers_counted += 1

    if layers_counted == 0:
        return 0, 0, 0

    avg_member_ratio = total_member_ratio / layers_counted
    avg_nonmember_ratio = total_nonmember_ratio / layers_counted

    if avg_nonmember_ratio == 0:
        ratio = float('inf')
    else:
        ratio = avg_member_ratio / avg_nonmember_ratio

    prediction = 1 if ratio >= threshold else 0
    return prediction, avg_member_ratio, avg_nonmember_ratio

def find_best_relative_ratio_threshold(val_samples, reference_patterns, discriminative_layers):
    """Find optimal threshold"""
    print(f"  Finding threshold from {len(val_samples)} validation samples...")
    ratios = []

    for sample in val_samples:
        _, member_ratio, nonmember_ratio = predict_membership_by_relative_ratio(
            sample, reference_patterns, discriminative_layers, threshold=1.0
        )

        ratio = float('inf') if nonmember_ratio == 0 else member_ratio / nonmember_ratio
        ratios.append(ratio)

    filtered_ratios = [r for r in ratios if r != float('inf') and not np.isnan(r)]

    if not filtered_ratios:
        print("  ⚠️ No valid ratios, using default threshold 1.0")
        return 1.0

    best_threshold = float(np.median(filtered_ratios))
    print(f"  ✅ Best threshold = {best_threshold:.4f}")
    return best_threshold

def score_set(samples: List[Dict[str,Any]], reference_patterns, discriminative_layers, tag: str) -> np.ndarray:
    """Score samples - simple progress"""
    scores = []
    total = len(samples)

    print(f"  Scoring {total} samples for {tag}...")

    for i, sample in enumerate(samples):
        _, member_ratio, nonmember_ratio = predict_membership_by_relative_ratio(
            sample, reference_patterns, discriminative_layers
        )

        ratio = float('inf') if nonmember_ratio == 0 else member_ratio / nonmember_ratio

        if ratio == float('inf'):
            ratio = 1000.0
        if np.isnan(ratio):
            ratio = 0.0

        scores.append(ratio)

        if (i + 1) % 100 == 0:
            print(f"    {i+1}/{total} completed ({100*(i+1)/total:.1f}%)")

    print(f"  ✅ {tag} scoring completed")
    return np.array(scores, dtype=np.float32)

# ============================================================
# 8) Run NA-PDD Pipeline
# ============================================================
print("\n" + "="*80)
print("5) Run NA-PDD Pipeline")
print("="*80)

print("\nStep A) Collect neuron activations for calibration sets")
P_calib_acts = collect_activations(P_calib, tag="P_calib")
N_calib_acts = collect_activations(N_calib, tag="N_calib")

print("\nStep B) Build reference activation patterns")
reference_patterns = build_reference_patterns(P_calib_acts, N_calib_acts)

print("\nStep C) Calculate layer discrimination scores")
layer_scores = calculate_layer_discrimination_scores(reference_patterns)

print("\nStep D) Select discriminative layers")
discriminative_layers = select_discriminative_layers(layer_scores, top_n=TOP_N_LAYERS)

print("\nStep E) Collect activations for probe sets")
P_probe_acts = collect_activations(P_probe, tag="P_probe")
S_probe_acts = collect_activations(S_probe, tag="S_probe")
N_probe_acts = collect_activations(N_probe, tag="N_probe")

print("\nStep F) Find optimal threshold")
val_size = min(100, len(P_probe_acts) // 4)
val_samples = P_probe_acts[:val_size]
best_threshold = find_best_relative_ratio_threshold(val_samples, reference_patterns, discriminative_layers)

print("\nStep G) Score all probe sets")
P_scores = score_set(P_probe_acts, reference_patterns, discriminative_layers, tag="P")
S_scores = score_set(S_probe_acts, reference_patterns, discriminative_layers, tag="S")
N_scores = score_set(N_probe_acts, reference_patterns, discriminative_layers, tag="N")

# ============================================================
# 9) Evaluations
# ============================================================
print("\n" + "="*80)
print("6) Evaluations (Experiment 1 & 2)")
print("="*80)

def auc_pair(pos_scores: np.ndarray, neg_scores: np.ndarray) -> float:
    y = np.array([1]*len(pos_scores) + [0]*len(neg_scores), dtype=np.int32)
    s = np.concatenate([pos_scores, neg_scores], axis=0)
    if np.all(s == s[0]):
        return 0.5
    return float(roc_auc_score(y, s))

# Experiment 1
auc_P_S = auc_pair(P_scores, S_scores)
auc_P_N = auc_pair(P_scores, N_scores)
auc_S_N = auc_pair(S_scores, N_scores)

print(f"\n🏆 Experiment-1: AUC Metrics")
print(f"AUC (P vs S) = {auc_P_S:.4f}  [Main result: paired vs shuffled]")
print(f"AUC (P vs N) = {auc_P_N:.4f}  [Sanity check: paired vs nonmember]")
print(f"AUC (S vs N) = {auc_S_N:.4f}  [Sanity check: shuffled vs nonmember]")

# Experiment 2
target_tpr = 0.80
thr = float(np.quantile(P_scores, 1.0 - target_tpr))
TPR_on_P = float(np.mean(P_scores >= thr))
FPR_on_S = float(np.mean(S_scores >= thr))

print(f"\n✅ Experiment-2: Fixed TPR Analysis")
print(f"Threshold (from P quantile) = {thr:.6f}")
print(f"TPR on P (target ~80%)      = {TPR_on_P*100:.2f}%")
print(f"FPR on S (mis-kill rate)    = {FPR_on_S*100:.2f}%")

# Confusion matrix
y_ps = np.array([1]*len(P_scores) + [0]*len(S_scores), dtype=np.int32)
s_ps = np.concatenate([P_scores, S_scores], axis=0)
pred_ps = (s_ps >= thr).astype(int)
cm_ps = confusion_matrix(y_ps, pred_ps)
print("\nConfusion Matrix (P=positive, S=negative):")
print(cm_ps)

# Additional metrics
print(f"\n📈 Additional Metrics (P vs N):")
y_pn = np.array([1]*len(P_scores) + [0]*len(N_scores), dtype=np.int32)
s_pn = np.concatenate([P_scores, N_scores], axis=0)
pred_pn = (s_pn >= best_threshold).astype(int)
acc = accuracy_score(y_pn, pred_pn)
prec, rec, f1, _ = precision_recall_fscore_support(y_pn, pred_pn, average='binary')
print(f"Accuracy  = {acc:.4f}")
print(f"Precision = {prec:.4f}")
print(f"Recall    = {rec:.4f}")
print(f"F1-Score  = {f1:.4f}")

# Cleanup
for hook in hooks:
    hook.remove()

print("\n✅ NA-PDD Pipeline Completed!")


## Zlib P/S/N

In [ ]:
# ==============================================================================
# Zlib Compression MIA for BLIP-ITM - Simplified (No Calibration Needed)
# ==============================================================================

import os, io, re, gc, math, pickle, random, warnings, zlib
from typing import Dict, Any, Optional, List, Tuple
from io import BytesIO

import numpy as np
from PIL import Image
import requests

import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, precision_recall_fscore_support

warnings.filterwarnings("ignore")

# ============================================================
# 0) Configuration
# ============================================================
MEMBER_PKL     = "/content/coco_data/member_images_1k.pkl"
NONMEMBER_PKL  = "/content/coco_data/nonmember_images_1k.pkl"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

# No calibration needed! Use all for testing
PROBE_P = 1000
PROBE_S = 1000
PROBE_N = 1000

IMG_CACHE_DIR = "/content/coco_img_cache"
os.makedirs(IMG_CACHE_DIR, exist_ok=True)

assert torch.cuda.is_available(), "❌ CUDA not available"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True
print(f"✅ Device: {device}")

# ============================================================
# 1) Load BLIP-ITM model
# ============================================================
print("\n" + "="*80)
print("1) Load BLIP-ITM model")
print("="*80)

try:
    from transformers import BlipProcessor, BlipForImageTextRetrieval
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.28.0"])
    from transformers import BlipProcessor, BlipForImageTextRetrieval

MODEL_NAME = "Salesforce/blip-itm-base-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()

image_processor = processor.image_processor
tokenizer = processor.tokenizer
print("✅ BLIP-ITM loaded")

# ============================================================
# 2) Load data and construct P/S/N
# ============================================================
print("\n" + "="*80)
print("2) Load data + construct P/S/N")
print("="*80)

def load_pickle_list(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

members_all    = load_pickle_list(MEMBER_PKL)
nonmembers_all = load_pickle_list(NONMEMBER_PKL)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)

members_all    = members_all[:NUM_MEMBERS]
nonmembers_all = nonmembers_all[:NUM_NONMEMBERS]

def get_caption_from_obj(obj: Dict[str,Any]) -> str:
    if "caption" in obj:
        c = obj["caption"]
        if isinstance(c, str): return c.strip()
        if isinstance(c, list) and len(c) > 0: return str(c[0]).strip()
        return ""
    if "captions" in obj:
        c = obj["captions"]
        if isinstance(c, str): return c.strip()
        if isinstance(c, list) and len(c) > 0: return str(c[0]).strip()
        return ""
    return ""

def set_caption(obj: Dict[str,Any], new_cap: str) -> Dict[str,Any]:
    x = dict(obj)
    x["caption"] = str(new_cap)
    return x

# P: paired
P_all = members_all

# S: shuffled captions
caps = [get_caption_from_obj(x) for x in members_all]
perm = list(range(len(caps)))
random.Random(SEED+999).shuffle(perm)
for i in range(len(perm)):
    if perm[i] == i:
        j = (i + 1) % len(perm)
        perm[i], perm[j] = perm[j], perm[i]

S_all = [set_caption(members_all[i], caps[perm[i]]) for i in range(len(members_all))]

# N: nonmembers
N_all = nonmembers_all

print(f"✅ Built sets: P={len(P_all)} | S={len(S_all)} | N={len(N_all)}")
diff_cnt = sum(get_caption_from_obj(P_all[i]) != get_caption_from_obj(S_all[i]) for i in range(50))
print(f"Sanity (first 50): P vs S caption different = {diff_cnt}/50")

# Use all samples (no calibration split needed)
P_probe = P_all[:PROBE_P]
S_probe = S_all[:PROBE_S]
N_probe = N_all[:PROBE_N]

print(f"Test sets: P={len(P_probe)}, S={len(S_probe)}, N={len(N_probe)}")

# ============================================================
# 3) Image loading
# ============================================================
def _coco_url_from_image_field(image_field: str) -> Optional[str]:
    if not isinstance(image_field, str) or "/" not in image_field:
        return None
    split_dir, filename = image_field.split("/", 1)
    return f"http://images.cocodataset.org/{split_dir}/{filename}"

def load_coco_image(obj: Dict[str,Any]) -> Optional[Image.Image]:
    image_field = obj.get("image")
    url = _coco_url_from_image_field(image_field)
    if url is None:
        return None

    fname = image_field.replace("/", "_")
    fpath = os.path.join(IMG_CACHE_DIR, fname)

    try:
        if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
            return Image.open(fpath).convert("RGB")
    except:
        pass

    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            return None
        img = Image.open(BytesIO(r.content)).convert("RGB")
        try:
            img.save(fpath, format="JPEG", quality=95)
        except:
            pass
        return img
    except:
        return None

# ============================================================
# 4) Zlib Method
# ============================================================
print("\n" + "="*80)
print("3) Zlib Compression Method (No Calibration Required)")
print("="*80)

def compute_loss(obj: Dict[str,Any]) -> Optional[float]:
    img = load_coco_image(obj)
    if img is None:
        return None

    caption = get_caption_from_obj(obj)
    if not caption:
        return None

    try:
        pixel_values = image_processor(img, return_tensors="pt").pixel_values.to(device)
        text_inputs = tokenizer(
            caption,
            padding="max_length",
            truncation=True,
            max_length=77,
            return_tensors="pt"
        )

        input_ids = text_inputs.input_ids.to(device)
        attention_mask = text_inputs.attention_mask.to(device)

        with torch.no_grad():
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )

            if hasattr(outputs, 'logits'):
                logits = outputs.logits
                target = torch.tensor([1], dtype=torch.long, device=device)
                loss = F.cross_entropy(logits, target)
                return float(loss.item())
            else:
                vision_outputs = model.vision_model(pixel_values=pixel_values)
                text_outputs = model.text_encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    return_dict=True
                )

                image_embeds = vision_outputs[1]
                image_features = model.vision_proj(image_embeds)
                image_features = F.normalize(image_features, dim=-1)

                text_embeds = text_outputs.last_hidden_state[:, 0, :]
                text_features = model.text_proj(text_embeds)
                text_features = F.normalize(text_features, dim=-1)

                similarity = (image_features * text_features).sum(dim=-1)
                loss = -torch.log(torch.sigmoid(similarity) + 1e-10)
                return float(loss.item())

    except Exception as e:
        return None
    finally:
        torch.cuda.empty_cache()

def compute_zlib_entropy(text: str) -> float:
    if not text:
        return 1.0

    text_bytes = text.encode('utf-8')
    compressed = zlib.compress(text_bytes, level=9)

    if len(text_bytes) == 0:
        return 1.0

    ratio = len(compressed) / len(text_bytes)
    return ratio

def compute_zlib_score(obj: Dict[str,Any]) -> Optional[float]:
    loss = compute_loss(obj)
    if loss is None:
        return None

    caption = get_caption_from_obj(obj)
    zlib_entropy = compute_zlib_entropy(caption)

    if zlib_entropy < 1e-10:
        zlib_entropy = 1e-10

    # Lower score = more likely member
    # (low loss + low entropy = low score)
    score = loss / zlib_entropy

    return float(score)

def score_samples(samples: List[Dict[str,Any]], tag: str) -> np.ndarray:
    scores = []
    total = len(samples)

    print(f"  Scoring {total} samples for {tag}...")

    for i, obj in enumerate(samples):
        score = compute_zlib_score(obj)
        if score is None:
            score = 0.0
        scores.append(score)

        if (i + 1) % 100 == 0:
            print(f"    {i+1}/{total} completed ({100*(i+1)/total:.1f}%)")
            torch.cuda.empty_cache()
            gc.collect()

    print(f"  ✅ {tag} completed")
    return np.array(scores, dtype=np.float32)

# ============================================================
# 5) Run Pipeline (No Calibration!)
# ============================================================
print("\n" + "="*80)
print("4) Score All Test Sets")
print("="*80)

P_scores = score_samples(P_probe, tag="P")
S_scores = score_samples(S_probe, tag="S")
N_scores = score_samples(N_probe, tag="N")

# ============================================================
# 6) Evaluations
# ============================================================
print("\n" + "="*80)
print("5) Evaluations")
print("="*80)

def auc_pair(pos_scores: np.ndarray, neg_scores: np.ndarray) -> float:
    y = np.array([1]*len(pos_scores) + [0]*len(neg_scores), dtype=np.int32)
    s = np.concatenate([pos_scores, neg_scores], axis=0)
    if np.all(s == s[0]):
        return 0.5
    # Note: Lower score = member for Zlib, so we negate
    return float(roc_auc_score(y, -s))

auc_P_S = auc_pair(P_scores, S_scores)
auc_P_N = auc_pair(P_scores, N_scores)
auc_S_N = auc_pair(S_scores, N_scores)

print(f"\n🏆 Experiment-1: AUC Metrics")
print(f"AUC (P vs S) = {auc_P_S:.4f}  [Main: paired vs shuffled]")
print(f"AUC (P vs N) = {auc_P_N:.4f}  [Sanity: paired vs nonmember]")
print(f"AUC (S vs N) = {auc_S_N:.4f}  [Sanity: shuffled vs nonmember]")

# For Zlib: lower score = member, so use lower quantile
target_tpr = 0.80
thr = float(np.quantile(P_scores, target_tpr))  # Note: using target_tpr directly (not 1-tpr)
TPR_on_P = float(np.mean(P_scores <= thr))
FPR_on_S = float(np.mean(S_scores <= thr))

print(f"\n✅ Experiment-2: Fixed TPR Analysis")
print(f"Threshold (80% quantile of P) = {thr:.6f}")
print(f"TPR on P (target ~80%)        = {TPR_on_P*100:.2f}%")
print(f"FPR on S (mis-kill rate)      = {FPR_on_S*100:.2f}%")

y_ps = np.array([1]*len(P_scores) + [0]*len(S_scores), dtype=np.int32)
s_ps = np.concatenate([P_scores, S_scores], axis=0)
pred_ps = (s_ps <= thr).astype(int)
cm_ps = confusion_matrix(y_ps, pred_ps)
print("\nConfusion Matrix (P=pos, S=neg):")
print(cm_ps)

print(f"\n📈 Additional Metrics (P vs N):")
threshold_pn = float(np.median(P_scores))
y_pn = np.array([1]*len(P_scores) + [0]*len(N_scores), dtype=np.int32)
s_pn = np.concatenate([P_scores, N_scores], axis=0)
pred_pn = (s_pn <= threshold_pn).astype(int)
acc = accuracy_score(y_pn, pred_pn)
prec, rec, f1, _ = precision_recall_fscore_support(y_pn, pred_pn, average='binary')
print(f"Accuracy  = {acc:.4f}")
print(f"Precision = {prec:.4f}")
print(f"Recall    = {rec:.4f}")
print(f"F1-Score  = {f1:.4f}")

print(f"\n📊 Score Statistics (lower = more likely member):")
print(f"P: mean={np.mean(P_scores):.4f}, std={np.std(P_scores):.4f}, median={np.median(P_scores):.4f}")
print(f"S: mean={np.mean(S_scores):.4f}, std={np.std(S_scores):.4f}, median={np.median(S_scores):.4f}")
print(f"N: mean={np.mean(N_scores):.4f}, std={np.std(N_scores):.4f}, median={np.median(N_scores):.4f}")

print("\n✅ Zlib MIA Completed!")

## dc-pdd

In [ ]:
# ==============================================================================
# 生成 BLIP 原生 C4 词频表 (fre_dis_c4_blip.pkl)
# ==============================================================================

import os
import pickle
import numpy as np
from collections import Counter
from transformers import BlipProcessor
from datasets import load_dataset
from tqdm import tqdm
from google.colab import drive

# 1. 挂载 Google Drive
print("正在挂载 Google Drive...")
drive.mount('/content/drive')

# 2. 初始化路径与 BLIP Tokenizer
# 确保该路径是你后续 DC-PDD 代码中引用的路径
FREQ_PATH_BLIP = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_blip.pkl"
MODEL_NAME = "Salesforce/blip-itm-base-coco"

print(f"正在加载 {MODEL_NAME} 的分词器...")
processor = BlipProcessor.from_pretrained(MODEL_NAME)
tokenizer = processor.tokenizer
vocab_size = tokenizer.vocab_size

# 3. 加载 C4 英文子集 (使用 Parquet 格式流式加载)
print("正在从 allenai/c4 加载英文语料子集...")
try:
    # 仅加载第一个分片以确保统计速度，通常 5 万个样本的词频已非常稳定
    dataset = load_dataset(
        "allenai/c4",
        data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
        split="train",
        streaming=True
    )
except Exception as e:
    print(f"加载数据集失败: {e}")
    print("尝试备选加载方式...")
    dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)

# 4. 开始统计
counter = Counter()
MAX_SAMPLES = 50000 # 统计 50,000 个句子，足以覆盖 30522 规模的词表

print(f"开始统计前 {MAX_SAMPLES} 条样本的词频...")
for i, ex in enumerate(tqdm(dataset, total=MAX_SAMPLES, desc="C4 词频统计")):
    if i >= MAX_SAMPLES:
        break
    text = ex['text']
    # 仅统计，不涉及模型推理，忽略长度限制警告
    ids = tokenizer.encode(text, add_special_tokens=False)
    counter.update(ids)

# 5. 计算平滑词频 (Laplace Smoothing)
print("\n正在计算平滑词频...")
total_tokens = sum(counter.values())
# 基础频率设为 1，避免查表时出现 0 频率导致 log(inf)
freq_smo = np.ones(vocab_size, dtype=np.float64)

for tid, count in counter.items():
    if tid < vocab_size:
        freq_smo[tid] += count

# 最终归一化频率
freq_smo = freq_smo / (total_tokens + vocab_size)

# 6. 自动创建目录并保存
os.makedirs(os.path.dirname(FREQ_PATH_BLIP), exist_ok=True)

try:
    with open(FREQ_PATH_BLIP, "wb") as f:
        pickle.dump(freq_smo, f)
    print("\n" + "="*50)
    print(f"✅ 成功！BLIP 原生 C4 词频表已保存至：")
    print(f"👉 {FREQ_PATH_BLIP}")
    print("="*50)
except Exception as e:
    print(f"❌ 保存失败: {e}")

# 7. 快速验证：打印几个常见词的频率
test_words = ["the", "a", "medical", "fashion", "is"]
print("\n验证常见词频率：")
for word in test_words:
    tid = tokenizer.convert_tokens_to_ids(word)
    if tid < vocab_size:
        print(f"Token: '{word}' (ID: {tid}) -> Freq: {freq_smo[tid]:.8f}")

In [ ]:
from google.colab import drive
import os

# 重新挂载或刷新
drive.mount('/content/drive', force_remount=True)

# 检查文件是否真的在那里
FREQ_PATH = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_blip.pkl"
if os.path.exists(FREQ_PATH):
    print("✅ 确认文件已存在，可以继续！")
else:
    print("❌ 文件仍未同步，请稍等 10 秒后再次运行此单元格。")

In [ ]:
import os, io, gc, pickle, random, warnings, requests
from typing import Dict, Any, List, Optional
from io import BytesIO

import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support
from transformers import BlipProcessor, BlipForImageTextRetrieval

warnings.filterwarnings("ignore")

# ============================================================
# 0) Configuration & Paths
# ============================================================
# 数据路径 (请确保这些文件在你的 Drive 中)
MEMBER_PKL     = "/content/coco_data/member_images_1k.pkl"
NONMEMBER_PKL  = "/content/coco_data/nonmember_images_1k.pkl"
# 刚刚生成的 BLIP 专用 C4 频率表
FREQ_PATH      = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_blip.pkl"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

NUM_SAMPLES    = 1000  # 评估样本数
ALPHA          = 1.0   # 截断阈值 (调大以防止信号磨平)
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

IMG_CACHE_DIR = "/content/coco_img_cache"
os.makedirs(IMG_CACHE_DIR, exist_ok=True)

# ============================================================
# 1) Load BLIP-ITM Model & Frequency Map
# ============================================================
print("🚀 Loading BLIP-ITM and Native C4 Frequency Map...")
MODEL_NAME = "Salesforce/blip-itm-base-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# 加载你刚刚生成的频率表
if not os.path.exists(FREQ_PATH):
    raise FileNotFoundError(f"❌ 找不到频率表: {FREQ_PATH}, 请先运行统计脚本。")

with open(FREQ_PATH, "rb") as f:
    freq_smo = pickle.load(f)
print(f"✅ Frequency table loaded. Size: {len(freq_smo)}")

# ============================================================
# 2) Helper Functions
# ============================================================
def load_pickle_list(path: str):
    with open(path, "rb") as f: return pickle.load(f)

def get_caption(obj: Dict) -> str:
    c = obj.get("caption") or obj.get("captions")
    return str(c[0]).strip() if isinstance(c, list) else str(c).strip()

def load_coco_image(obj: Dict) -> Optional[Image.Image]:
    image_field = obj.get("image")
    if not image_field: return None
    fname = image_field.replace("/", "_")
    fpath = os.path.join(IMG_CACHE_DIR, fname)

    # 优先从缓存读取
    if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
        return Image.open(fpath).convert("RGB")

    # 否则从官方下载
    url = f"http://images.cocodataset.org/{image_field}"
    try:
        r = requests.get(url, timeout=10)
        img = Image.open(BytesIO(r.content)).convert("RGB")
        img.save(fpath, format="JPEG")
        return img
    except: return None

# ============================================================
# 3) DC-PDD Scoring Logic
# ============================================================
@torch.inference_mode()
def compute_dc_pdd_score(obj: Dict) -> Optional[float]:
    img = load_coco_image(obj)
    caption = get_caption(obj)
    if img is None or not caption: return None

    # 1. 提取 ITM Logits (不使用 Softmax 保持高动态范围)
    inputs = processor(images=img, text=caption, return_tensors="pt").to(DEVICE)
    outputs = model(**inputs)

    # 信号强度: "Matched" 类别的原始激活值
    p_signal = outputs.itm_score[0, 1].item()

    # 2. 难度校准 (使用 BLIP 原生 Tokenizer 对齐)
    # ids 是根据 BLIP 词表生成的，能直接从 freq_smo 中索引
    ids = processor.tokenizer.encode(caption, add_special_tokens=False)
    if not ids: return None

    # 计算 log(1/freq) 的均值
    # Token 越罕见，diff 越大
    diff_values = [np.log(1.0 / freq_smo[tid]) for tid in ids if tid < len(freq_smo)]
    avg_difficulty = np.mean(diff_values)

    # 3. 最终得分
    # 核心公式: Score = p_signal * log(1/freq)
    # 取负值：Member 倾向于 P 信号高且包含低频难词，因此得分 -Score 越小
    raw_score = p_signal * avg_difficulty
    return -np.minimum(raw_score, ALPHA)

# ============================================================
# 4) Run Scoring Loop
# ============================================================
print("\n🔍 Sampling Member and Non-member Data...")
m_raw = load_pickle_list(MEMBER_PKL)
n_raw = load_pickle_list(NONMEMBER_PKL)

random.Random(SEED).shuffle(m_raw)
random.Random(SEED).shuffle(n_raw)

def run_eval(samples, name):
    scores = []
    for i, obj in enumerate(tqdm(samples[:NUM_SAMPLES], desc=f"Scoring {name}")):
        s = compute_dc_pdd_score(obj)
        if s is not None:
            scores.append(s)
        if (i+1) % 200 == 0:
            gc.collect(); torch.cuda.empty_cache()
    return np.array(scores)

m_scores = run_eval(m_raw, "Members")
n_scores = run_eval(n_raw, "Non-members")

# ============================================================
# 5) Metrics & Results
# ============================================================
def calculate_metrics(m_s, n_s):
    y_true = np.concatenate([np.ones(len(m_s)), np.zeros(len(n_s))])
    all_scores = np.concatenate([m_s, n_s])

    # 自动校准方向: 如果 Member 中值更小，说明符合记忆性假设
    m_med, n_med = np.median(m_s), np.median(n_s)
    final_scores = -all_scores if m_med < n_med else all_scores

    auc = roc_auc_score(y_true, final_scores)
    fpr, tpr, _ = roc_curve(y_true, final_scores)

    # 计算 TPR @ 5% FPR (MIA 的关键指标)
    tpr_5fpr = tpr[np.abs(fpr - 0.05).argmin()]

    # 计算基于中位数的 Accuracy
    thresh = np.median(all_scores)
    preds = (all_scores <= thresh).astype(int) if m_med < n_med else (all_scores >= thresh).astype(int)
    acc = (preds == y_true).mean()

    print(f"\n" + "="*60)
    print(f"🏆 Final DC-PDD MIA Results (Member vs Non-member)")
    print(f"   Frequency Source: Native C4 (BLIP Tokenizer)")
    print(f"-"*60)
    print(f"   AUC:             {auc:.4f}")
    print(f"   TPR @ 5% FPR:    {tpr_5fpr*100:.2f}%")
    print(f"   Accuracy:        {acc*100:.2f}%")
    print(f"   Member Median:   {m_med:.6f}")
    print(f"   Non-member Med:  {n_med:.6f}")
    print(f"="*60)

calculate_metrics(m_scores, n_scores)

## M$^4$I

### data split

In [ ]:
import os, json, pickle, random

SEED = 42
SPLIT_PATH = "/content/splits_blip.json"

# 优先 Drive（因为 /content 会丢）
DRIVE_DIR = "/content/drive/MyDrive/coco_data_mia"
LOCAL_DIR = "/content/coco_data"

def resolve(path1, path2):
    if os.path.exists(path1): return path1
    if os.path.exists(path2): return path2
    raise FileNotFoundError(f"Not found:\n- {path1}\n- {path2}\n\n"
                            f"你需要先运行 COCO 数据生成脚本，或把 pkl 拷贝到 {DRIVE_DIR}")

MEMBER_PKL    = resolve(os.path.join(DRIVE_DIR, "member_images_1k.pkl"),
                        os.path.join(LOCAL_DIR, "member_images_1k.pkl"))
NONMEMBER_PKL = resolve(os.path.join(DRIVE_DIR, "nonmember_images_1k.pkl"),
                        os.path.join(LOCAL_DIR, "nonmember_images_1k.pkl"))

def load_pickle_list(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

members_all    = load_pickle_list(MEMBER_PKL)
nonmembers_all = load_pickle_list(NONMEMBER_PKL)

assert len(members_all) >= 1000 and len(nonmembers_all) >= 1000

rnd = random.Random(SEED)
rnd.shuffle(members_all)
rnd.shuffle(nonmembers_all)

members    = members_all[:1000]
nonmembers = nonmembers_all[:1000]

# --- split sizes ---
ft200           = nonmembers[:200]
train_nonmem200 = nonmembers[200:400]

dev_mem100      = members[:100]
dev_nonmem100   = nonmembers[400:500]

test_mem500     = members[100:600]
test_nonmem500  = nonmembers[500:1000]

print("Split summary:")
print("  FT (from nonmember)         :", len(ft200))
print("  Train nonmember (from nonm) :", len(train_nonmem200))
print("  Dev member                  :", len(dev_mem100))
print("  Dev nonmember               :", len(dev_nonmem100))
print("  Test member                 :", len(test_mem500))
print("  Test nonmember              :", len(test_nonmem500))

splits = dict(
    ft200=ft200,
    train_nonmem200=train_nonmem200,
    dev_mem100=dev_mem100,
    dev_nonmem100=dev_nonmem100,
    test_mem500=test_mem500,
    test_nonmem500=test_nonmem500
)
with open(SPLIT_PATH, "w", encoding="utf-8") as f:
    json.dump(splits, f, ensure_ascii=False)

print("✅ Saved splits to:", SPLIT_PATH)
print("✅ Using member pkl:", MEMBER_PKL)
print("✅ Using nonmember pkl:", NONMEMBER_PKL)


### finetune

In [ ]:
import os, json, random, gc
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import requests
from io import BytesIO
from tqdm.auto import tqdm

from transformers import BlipProcessor, BlipForImageTextRetrieval

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print("DEVICE:", DEVICE, "DTYPE:", DTYPE)

SPLIT_PATH = "/content/splits_blip.json"
with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

ft200 = splits["ft200"]

MODEL_NAME = "Salesforce/blip-itm-base-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(DEVICE)
model.train()

# --------- COCO image loader (same as your script) ----------
def load_coco_image(img_data):
    image_path = img_data.get("image", None)
    if not image_path:
        return None
    try:
        parts = image_path.split("/")
        if len(parts) == 2:
            split_dir, filename = parts
            url = f"http://images.cocodataset.org/{split_dir}/{filename}"
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                return Image.open(BytesIO(r.content)).convert("RGB")
    except:
        return None
    return None

def get_caption(img):
    if "caption" in img:
        c = img["caption"]
        if isinstance(c, str): return c
        if isinstance(c, list) and c: return c[0]
    if "captions" in img:
        c = img["captions"]
        if isinstance(c, str): return c
        if isinstance(c, list) and c: return c[0]
    return ""

class FT200(Dataset):
    def __init__(self, items):
        self.samples = []
        for it in items:
            cap = get_caption(it).strip()
            if not cap:
                continue
            self.samples.append(it)
        print(f"✅ FT usable: {len(self.samples)}/{len(items)}")

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

def collate_ft(batch):
    # build positives
    imgs, caps = [], []
    for it in batch:
        im = load_coco_image(it)
        if im is None:
            im = Image.new("RGB", (384,384), color="black")
        imgs.append(im)
        caps.append(get_caption(it).strip())

    # processor -> pixel_values + tokens
    enc = processor(images=imgs, text=caps, return_tensors="pt", padding=True, truncation=True).to(DEVICE)

    # build negatives via shuffled text in-batch
    bs = len(batch)
    perm = torch.randperm(bs)
    neg_caps = [caps[j] for j in perm.tolist()]
    enc_neg = processor(images=imgs, text=neg_caps, return_tensors="pt", padding=True, truncation=True).to(DEVICE)

    # return both
    return enc, enc_neg

ds = FT200(ft200)
dl = DataLoader(ds, batch_size=8, shuffle=True, collate_fn=collate_ft)

# ------- training loop -------
lr = 1e-5
epochs = 1
grad_accum = 1
opt = torch.optim.AdamW(model.parameters(), lr=lr)

step = 0
torch.manual_seed(SEED)
random.seed(SEED)

for ep in range(epochs):
    for (pos, neg) in tqdm(dl, desc=f"FT epoch {ep+1}/{epochs}"):
        # forward: get embeddings then ITM logits
        # We use model's itm_head via forward() in retrieval model:
        # It returns itm_score for each pair when use_itm_head=True
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE=="cuda")):
            out_pos = model(**pos, use_itm_head=True, return_dict=True)
            out_neg = model(**neg, use_itm_head=True, return_dict=True)

            # out.itm_score: [bs,2] (neg,pos) or (0/1) depending impl
            # We'll treat label=1 for pos, label=0 for neg on index 1 as "match"
            # Safer: use cross entropy on 2-class logits with labels {1 for pos, 0 for neg}
            logits_pos = out_pos.itm_score  # [bs, 2]
            logits_neg = out_neg.itm_score  # [bs, 2]

            y_pos = torch.ones(logits_pos.size(0), dtype=torch.long, device=DEVICE)
            y_neg = torch.zeros(logits_neg.size(0), dtype=torch.long, device=DEVICE)

            loss = (F.cross_entropy(logits_pos, y_pos) + F.cross_entropy(logits_neg, y_neg)) / 2

        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)

        step += 1
        if step % 20 == 0:
            print(f"step={step} loss={loss.item():.4f}")

        if DEVICE == "cuda" and step % 50 == 0:
            torch.cuda.empty_cache(); gc.collect()

FT_MODEL_DIR = "/content/blip_fullFT_200"
os.makedirs(FT_MODEL_DIR, exist_ok=True)
model.save_pretrained(FT_MODEL_DIR)
processor.save_pretrained(FT_MODEL_DIR)
print("✅ Saved FT model to:", FT_MODEL_DIR)


### get acts

In [ ]:
import os, json, gc, shutil
import torch
from tqdm.auto import tqdm
from PIL import Image
import requests
from io import BytesIO

from transformers import BlipProcessor, BlipForImageTextRetrieval

SPLIT_PATH   = "/content/splits_blip.json"
FT_MODEL_DIR = "/content/blip_fullFT_200"
MODEL_NAME   = "Salesforce/blip-itm-base-coco"

ACTS_ROOT = "/content/acts_blip"
TRAIN_DIR = os.path.join(ACTS_ROOT, "train")  # FT model
DEV_DIR   = os.path.join(ACTS_ROOT, "dev")    # baseline
TEST_DIR  = os.path.join(ACTS_ROOT, "test")   # baseline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

# ---- load splits ----
with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

ft200           = splits["ft200"]
train_nonmem200 = splits["train_nonmem200"]
dev_mem100      = splits["dev_mem100"]
dev_nonmem100   = splits["dev_nonmem100"]
test_mem500     = splits["test_mem500"]
test_nonmem500  = splits["test_nonmem500"]

print("✅ Loaded splits:", len(ft200), len(train_nonmem200), len(dev_mem100), len(dev_nonmem100), len(test_mem500), len(test_nonmem500))

# ---- coco loader ----
def load_coco_image(obj):
    image_path = obj.get("image")
    if not image_path:
        return None
    try:
        split_dir, filename = image_path.split("/")
        url = f"http://images.cocodataset.org/{split_dir}/{filename}"
        r = requests.get(url, timeout=20)
        if r.status_code == 200:
            return Image.open(BytesIO(r.content)).convert("RGB")
    except:
        return None
    return None

def get_caption(obj):
    cap = obj.get("caption", "")
    if isinstance(cap, str): return cap.strip()
    if isinstance(cap, list) and cap: return str(cap[0]).strip()
    caps = obj.get("captions", "")
    if isinstance(caps, str): return caps.strip()
    if isinstance(caps, list) and caps: return str(caps[0]).strip()
    return ""

# ---- feature extractor: per-layer text hidden_states[CLS] conditioned on image ----
@torch.no_grad()
def get_layerwise_cls_vectors(model, processor, image, caption):
    image_processor = processor.image_processor
    tokenizer = processor.tokenizer

    pixel_values = image_processor(image, return_tensors="pt").pixel_values.to(model.device)
    toks = tokenizer([caption], padding="max_length", truncation=True, max_length=77, return_tensors="pt").to(model.device)

    # vision embeds
    vision_out = model.vision_model(pixel_values=pixel_values, return_dict=True)
    image_embeds = vision_out.last_hidden_state  # [1, n_patches, dim]
    image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long, device=model.device)

    # text encoder with cross-attn to image
    text_out = model.text_encoder(
        input_ids=toks.input_ids,
        attention_mask=toks.attention_mask,
        encoder_hidden_states=image_embeds,
        encoder_attention_mask=image_atts,
        return_dict=True,
        output_hidden_states=True
    )

    hs = text_out.hidden_states  # tuple: (emb, layer1..layer12)
    vecs = [h[0, 0, :].detach().float().cpu() for h in hs]  # CLS token
    return vecs  # list of [dim]

def ensure_dir(p): os.makedirs(p, exist_ok=True)

def generate_acts(processor, model, dataset, save_dir, idx_offset=0):
    ensure_dir(save_dir)

    # detect layer count once
    dummy = Image.new("RGB", (384, 384), color=(128,128,128))
    dummy_vecs = get_layerwise_cls_vectors(model, processor, dummy, "a photo of something")
    L = len(dummy_vecs)
    print(f"✅ text hidden_states length = {L} (includes embedding). Save to {save_dir}")

    for i in tqdm(range(len(dataset)), desc=f"Acts -> {save_dir}"):
        obj = dataset[i]
        cap = get_caption(obj)
        if not cap:
            continue
        im = load_coco_image(obj)
        if im is None:
            continue

        vecs = get_layerwise_cls_vectors(model, processor, im, cap)
        out_idx = i + idx_offset

        for lid, v in enumerate(vecs):
            torch.save(v.unsqueeze(0), os.path.join(save_dir, f"layer_{lid}_{out_idx}.pt"))

        if torch.cuda.is_available() and (out_idx % 100 == 0):
            torch.cuda.empty_cache()
            gc.collect()

# ---- load models ----
print("\n=== Load FT model (for TRAIN acts) ===")
ft_proc  = BlipProcessor.from_pretrained(FT_MODEL_DIR, use_fast=False)
ft_model = BlipForImageTextRetrieval.from_pretrained(FT_MODEL_DIR).to(device)
ft_model.eval()
print("✅ FT model loaded:", FT_MODEL_DIR)

print("\n=== Load baseline model (for DEV/TEST acts) ===")
base_proc  = BlipProcessor.from_pretrained(MODEL_NAME, use_fast=False)
base_model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
base_model.eval()
print("✅ Baseline loaded:", MODEL_NAME)

# ---- clean dirs ----
shutil.rmtree(ACTS_ROOT, ignore_errors=True)
ensure_dir(TRAIN_DIR); ensure_dir(DEV_DIR); ensure_dir(TEST_DIR)

# 1) TRAIN acts (FT model) on [ft200 (label1) + train_nonmem200 (label0)]
print("\n[1/3] TRAIN acts (FT model)")
generate_acts(ft_proc, ft_model, ft200, TRAIN_DIR, idx_offset=0)
generate_acts(ft_proc, ft_model, train_nonmem200, TRAIN_DIR, idx_offset=len(ft200))

# 2) DEV acts (baseline) on [dev_mem100 + dev_nonmem100]
print("\n[2/3] DEV acts (baseline)")
generate_acts(base_proc, base_model, dev_mem100, DEV_DIR, idx_offset=0)
generate_acts(base_proc, base_model, dev_nonmem100, DEV_DIR, idx_offset=len(dev_mem100))

# 3) TEST acts (baseline) on [test_mem500 + test_nonmem500]
print("\n[3/3] TEST acts (baseline)")
generate_acts(base_proc, base_model, test_mem500, TEST_DIR, idx_offset=0)
generate_acts(base_proc, base_model, test_nonmem500, TEST_DIR, idx_offset=len(test_mem500))

print("\n✅ Done. Acts saved under:", ACTS_ROOT)


###evaluate

In [ ]:
import os
import json
import numpy as np
import torch
from glob import glob
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve, auc

SPLIT_PATH = "/content/splits_blip.json"
ACTS_ROOT  = "/content/acts_blip"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

ft200           = splits["ft200"]
train_nonmem200 = splits["train_nonmem200"]
dev_mem100      = splits["dev_mem100"]
dev_nonmem100   = splits["dev_nonmem100"]
test_mem500     = splits["test_mem500"]
test_nonmem500  = splits["test_nonmem500"]

# -----------------------------
# 1) LRProbe (完全一致风格)
# -----------------------------
class LRProbe(torch.nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(d_in, 1, bias=False),
            torch.nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)
    def score(self, x):
        return self(x)

    @staticmethod
    def from_data(acts, labels, lr=0.001, weight_decay=0.1, epochs=1000, device="cpu"):
        acts, labels = acts.to(device), labels.to(device)
        probe = LRProbe(acts.shape[-1]).to(device)
        opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=weight_decay)
        for _ in range(epochs):
            opt.zero_grad()
            pred = probe(acts)
            loss = torch.nn.BCELoss()(pred, labels)
            loss.backward()
            opt.step()
        return probe

# -----------------------------
# 2) load acts for a layer
# -----------------------------
def load_layer_matrix(split_name: str, layer_id: int, expected_n: int):
    directory = os.path.join(ACTS_ROOT, split_name)
    pattern = os.path.join(directory, f"layer_{layer_id}_*.pt")
    files = sorted(glob(pattern), key=lambda x: int(x.split("_")[-1].split(".")[0]))
    if len(files) == 0:
        raise ValueError(f"No act files: {pattern}")

    mats = []
    for fp in files[:expected_n]:
        v = torch.load(fp, map_location="cpu")  # [1, dim]
        mats.append(v)
    X = torch.cat(mats, dim=0).float()  # [N, dim]
    if X.shape[0] != expected_n:
        raise ValueError(f"Act count mismatch for {split_name} layer {layer_id}: got {X.shape[0]} expect {expected_n}")
    return X

def compute_auc_tpr5(scores, labels):
    fpr, tpr, _ = roc_curve(labels.astype(bool), scores)
    auc_val = auc(fpr, tpr)
    idx = np.where(fpr <= 0.05)[0]
    tpr5 = tpr[idx[-1]] if len(idx) else 0.0
    return float(auc_val), float(tpr5)

# -----------------------------
# 3) build labels
# -----------------------------
# TRAIN: ft200(1) + train_nonmem200(0)
train_labels = np.array([1]*len(ft200) + [0]*len(train_nonmem200), dtype=np.float32)
# DEV: dev_mem100(1) + dev_nonmem100(0)
dev_labels   = np.array([1]*len(dev_mem100) + [0]*len(dev_nonmem100), dtype=np.float32)
# TEST: test_mem500(1) + test_nonmem500(0)
test_labels  = np.array([1]*len(test_mem500) + [0]*len(test_nonmem500), dtype=np.float32)

N_train = len(train_labels)  # 400
N_dev   = len(dev_labels)    # 200
N_test  = len(test_labels)   # 1000

print("N_train:", N_train, "N_dev:", N_dev, "N_test:", N_test)

# -----------------------------
# 4) detect num layers from files
# -----------------------------
train_files = glob(os.path.join(ACTS_ROOT, "train", "layer_*_0.pt"))
layer_ids = sorted(list(set(int(os.path.basename(p).split("_")[1]) for p in train_files)))
if len(layer_ids) == 0:
    raise RuntimeError("No train act files found. Check Module 3 outputs.")
num_layers = max(layer_ids) + 1
print("✅ Detected layers:", num_layers)

# -----------------------------
# 5) run per-layer probe, dev select, test report
# -----------------------------
all_rows = []
best_dev_auc = -1.0
best_layer = None
best_test_auc = None
best_test_tpr5 = None

for layer in range(num_layers):
    Xtr = load_layer_matrix("train", layer, N_train)
    Xdv = load_layer_matrix("dev", layer, N_dev)
    Xte = load_layer_matrix("test", layer, N_test)

    ytr = torch.tensor(train_labels, dtype=torch.float32)
    ydv = dev_labels.astype(np.float32)
    yte = test_labels.astype(np.float32)

    # train probe
    probe = LRProbe.from_data(
        Xtr, ytr,
        lr=0.001, weight_decay=0.1,
        epochs=1000,
        device=str(DEVICE)
    )

    with torch.no_grad():
        sdev = probe.score(Xdv.to(DEVICE)).detach().cpu().numpy()
        ste  = probe.score(Xte.to(DEVICE)).detach().cpu().numpy()

    dev_auc, dev_tpr5 = compute_auc_tpr5(sdev, ydv)
    test_auc, test_tpr5 = compute_auc_tpr5(ste, yte)

    all_rows.append((layer, dev_auc, dev_tpr5, test_auc, test_tpr5))
    print(f"Layer {layer:02d}: dev AUC={dev_auc:.4f} TPR@5%FPR={dev_tpr5:.4f} | test AUC={test_auc:.4f} TPR@5%FPR={test_tpr5:.4f}")

    if dev_auc > best_dev_auc:
        best_dev_auc = dev_auc
        best_layer = layer
        best_test_auc = test_auc
        best_test_tpr5 = test_tpr5

print("\n" + "="*70)
print("✅ Best layer by DEV AUC")
print(f"Best layer = {best_layer}")
print(f"DEV  AUC   = {best_dev_auc:.4f}")
print(f"TEST AUC   = {best_test_auc:.4f}")
print(f"TEST TPR@5%FPR = {best_test_tpr5:.4f}")
print("="*70)


# LLaVA-Med

In [ ]:
# ===================================================================
# 环境安装（合并版，一键初始化）
# - 清理旧库 & cuDF
# - 安装 PyTorch cu121 指定版本
# - 安装其余依赖（固定 pyarrow==19.0.0 / datasets==2.20.0）
# - 设置 HF 数据集缓存目录
# - 自动重启内核
# ===================================================================

print("🚀 开始进行环境优化设置...")

# ---- 0) 可选：把 HF 数据集缓存放到更可控的位置（避免默认缓存爆盘）----
import os
os.environ.setdefault("HF_DATASETS_CACHE", "/content/hf_cache")
os.makedirs(os.environ["HF_DATASETS_CACHE"], exist_ok=True)
print(f"📦 HF_DATASETS_CACHE = {os.environ['HF_DATASETS_CACHE']}")

# ---- 1) 卸载 cuDF（避免与 pyarrow 的二进制/版本冲突）----
print("\n[1/4] 清理可能的 cuDF 组件（如未安装会自动跳过）...")
try:
    import subprocess, sys
    def sh(cmd):
        print(">", " ".join(cmd))
        subprocess.check_call(cmd)
    for pkg in ["cudf-cu12", "pylibcudf-cu12"]:
        sh([sys.executable, "-m", "pip", "uninstall", "-y", pkg])
except Exception as e:
    print("（跳过）", e)

# ---- 2) 卸载常见旧版本库，保证干净环境 ----
print("\n[2/4] 清理旧版本库...")
to_uninstall = [
    "torch", "torchvision", "torchaudio",
    "transformers", "datasets", "accelerate", "bitsandbytes", "peft",
    "pyarrow"
]
for pkg in to_uninstall:
    try:
        sh([sys.executable, "-m", "pip", "uninstall", "-y", pkg])
    except Exception:
        pass
print("✅ 旧库清理完成")

# ---- 3) 安装 PyTorch cu121 指定版本 ----
print("\n[3/4] 安装 PyTorch/cu121 指定版本...")
sh([sys.executable, "-m", "pip", "install", "-q",
    "--index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.3.0", "torchvision==0.18.0", "torchaudio==2.3.0"
])

# ---- 4) 安装其余依赖（含固定的 pyarrow 与 datasets 版本）----
print("\n[4/4] 安装核心依赖（固定 pyarrow==19.0.0, datasets==2.20.0）...")
sh([sys.executable, "-m", "pip", "install", "-q",
    # Transformers 新接口（AutoModelForImageTextToText）
    "transformers>=4.50.0",
    "accelerate",
    "sentencepiece",
    "safetensors",
    "pillow",
    "requests",
    # ✅ 固定 datasets + pyarrow，避免 ABI 冲突
    "datasets==2.20.0",
    "pyarrow==19.0.0",
    # 其他常用依赖
    "huggingface_hub",
    "scikit-learn==1.5.0",
    "tqdm==4.66.4",
    "numpy==1.26.4",
    "pandas==2.2.2",
    "matplotlib==3.9.0",
    # 训练相关可选
    "bitsandbytes==0.43.1",
    "peft==0.11.1"
])

print("\n✅ 环境设置完成！将重启内核以应用所有更改。")
print("   重启后，直接运行你的“主程序/第二步”单元即可。")

import IPython
IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
# 彻底解决版本冲突：同时升级 torch 和 torchvision
%pip install --upgrade torch torchvision torchaudio

## data

In [ ]:
import os, shutil

MOUNT_POINT = "/content/drive"

# 如果目录存在且非空，先清掉（仅清 /content/drive 这个本地目录里的残留，不会删你真实 Google Drive）
if os.path.isdir(MOUNT_POINT) and os.listdir(MOUNT_POINT):
    print("⚠️ /content/drive 非空，正在清理残留文件 ...")
    shutil.rmtree(MOUNT_POINT)

os.makedirs(MOUNT_POINT, exist_ok=True)

from google.colab import drive
drive.mount(MOUNT_POINT)


In [ ]:
# ============================================================
# 放射学专一 members(1000) + ROCO non-members(1000) 构建脚本
# - members 来自 LLaVA-Med instruct_fig_captions 过滤 + URLs JOIN
# - non-members 来自你本地的 ROCO 子集 2000
# - 预览仅下载/展示各自第1条图片，避免大量下载
# ============================================================

import os, json, re, io, tarfile, random, shutil
from pathlib import Path
from typing import Dict, Any, Iterable, List

import pandas as pd
from PIL import Image
from IPython.display import display
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import requests

# --------------------
# 配置区（按需修改）
# --------------------
SEED = 42

# 你的 Drive 路径（根据你上条消息内容）
# 1) LLaVA-Med 原始数据文件（从仓库 data/ 目录下载后放到 Drive）
LLAVA_MED_DIR = "/content/drive/MyDrive/Medical/data"
INSTRUCT_CAPTIONS_PATH = f"{LLAVA_MED_DIR}/llava_med_instruct_fig_captions.json"
INSTRUCT_URLS_PATH     = f"{LLAVA_MED_DIR}/llava_med_image_urls.jsonl"

# 2) ROCO 子集（你已准备 2000 条）
ROCO_BASE = "/content/drive/MyDrive/LLM_MIA/roco_subset_2000"
ROCO_CAPTIONS_TXT = f"{ROCO_BASE}/captions.txt"
ROCO_IMAGES_DIR   = f"{ROCO_BASE}/images"

# 3) 输出目录
OUT_DIR = "/content/drive/MyDrive/Medical/llava_med_pairs"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# 4) 预览：仅下载/解压 member 的第 1 张图片用于展示
PREVIEW_DOWNLOAD_DIR = "/content/member_preview_tmp"
Path(PREVIEW_DOWNLOAD_DIR).mkdir(parents=True, exist_ok=True)

# 5) 放射学 domain 白名单 + caption 黑名单（剔除流程/图表）
DOMAIN_WHITELIST = {"chest_xray", "ct_scan", "mri"}
CAPTION_EXCLUDE_RE = re.compile(
    r"\b("
    r"flow\s*diagram|algorithm|schema|pipeline|chart|graph|plot|"
    r"roc\b|kaplan|table|heat\s*map|heatmap|box[\s-]*plot|bar[\s-]*chart"
    r")\b",
    flags=re.IGNORECASE,
)

# =========================
# 0) 基础检查
# =========================
for p in [INSTRUCT_CAPTIONS_PATH, INSTRUCT_URLS_PATH, ROCO_CAPTIONS_TXT, ROCO_IMAGES_DIR]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"找不到：{p}")

print("✅ 输入文件检查通过")

random.seed(SEED)

# =========================
# 1) 读入 LLaVA-Med instruct captions
# =========================
def load_instruct_captions(path: str) -> List[Dict[str, Any]]:
    """
    支持两种结构：
    A) 顶层是 dict: { 'chest_xray': [ ..items.. ], 'ct_scan': [..], ... }
       每个 item 里可能还有 domain 布尔字段，也可能没有
    B) 顶层是 list: [ { 'fig_caption':..., 'pair_id':..., 'domain': {..} }, ... ]
    """
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    items: List[Dict[str, Any]] = []
    if isinstance(obj, dict):
        # 逐 domain 拉平
        for k, arr in obj.items():
            if not isinstance(arr, list):
                continue
            for x in arr:
                if isinstance(x, dict):
                    # 补一下 domain 键，保留原域名信息
                    dom = x.get("domain", {})
                    if not isinstance(dom, dict):
                        dom = {}
                    dom = {**dom, k: True}
                    y = {**x, "domain": dom}
                    items.append(y)
    elif isinstance(obj, list):
        items = [x for x in obj if isinstance(x, dict)]
    else:
        raise ValueError("instruct_fig_captions.json 结构既不是 dict 也不是 list。")
    return items

print("⏳ 读取 instruct captions …")
instruct_items = load_instruct_captions(INSTRUCT_CAPTIONS_PATH)
print(f"   items: {len(instruct_items):,}")

# -------------------------
# 2) 放射学 domain + caption 过滤
# -------------------------
def is_radiology_domain(sample: Dict[str, Any]) -> bool:
    dom = sample.get("domain", {})
    if isinstance(dom, dict):
        return any(bool(dom.get(d, False)) for d in DOMAIN_WHITELIST)
    return False

def pass_caption_filter(sample: Dict[str, Any]) -> bool:
    cap = str(sample.get("fig_caption", "")).strip()
    if not cap:
        return False
    if CAPTION_EXCLUDE_RE.search(cap):
        return False
    return True

filtered_instruct = [x for x in instruct_items if is_radiology_domain(x) and pass_caption_filter(x)]
print(f"   过滤后放射学专一样本: {len(filtered_instruct):,}")

# -------------------------
# 3) 载入 URLs JSONL（pair_id -> {pmc_tar_url, image_file_path}）
# -------------------------
def load_urls_jsonl(path: str) -> Dict[str, Dict[str, str]]:
    m: Dict[str, Dict[str, str]] = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            pid = obj.get("pair_id")
            if not pid:
                continue
            m[pid] = {
                "pmc_tar_url": obj.get("pmc_tar_url", ""),
                "image_file_path": obj.get("image_file_path", ""),
            }
    return m

print("⏳ 读取 URLs …")
pid2url = load_urls_jsonl(INSTRUCT_URLS_PATH)
print(f"   urls lines: {len(pid2url):,}")

# -------------------------
# 4) JOIN by pair_id
# -------------------------
def join_pairs(items: List[Dict[str, Any]], pid2url: Dict[str, Dict[str, str]]) -> List[Dict[str, Any]]:
    out = []
    for x in items:
        pid = x.get("pair_id") or x.get("pairId") or x.get("pmid_fig_id")
        if not pid:
            continue
        u = pid2url.get(pid)
        if not u:
            # 有些数据的 pair_id 里带下划线/大小写差异，可做兜底尝试：
            pid_low = str(pid).lower()
            u = pid2url.get(pid_low)
        if not u:
            continue
        cap = str(x.get("fig_caption", "")).strip()
        out.append({
            "pair_id": pid,
            "caption": cap,
            "pmc_tar_url": u["pmc_tar_url"],
            "image_file_path": u["image_file_path"],
            # 可选保留一些域信息
            "domain": x.get("domain", {}),
            "pmid": x.get("pmid", None),
            "fig_id": x.get("fig_id", None),
            "fig_label": x.get("fig_label", None),
        })
    return out

print("⏳ JOIN …")
joined = join_pairs(filtered_instruct, pid2url)
print(f"   JOIN 命中: {len(joined):,}")

# -------------------------
# 5) 采样 1000 members（按 PMCID 去重优先，提升多样性）
# -------------------------
def extract_pmcid(image_file_path: str) -> str:
    # 典型路径：PMC4724953/xxx.jpg
    parts = Path(image_file_path).parts
    for p in parts:
        if p.startswith("PMC") and p[3:].isdigit():
            return p
    return "UNKNOWN"

# 先做基于文章的“唯一 PMCID”采样，保证多样性
random.shuffle(joined)
seen_pmc = set()
diverse = []
for r in joined:
    pmcid = extract_pmcid(r["image_file_path"])
    if pmcid not in seen_pmc:
        diverse.append(r)
        seen_pmc.add(pmcid)

# 如果多样性集 < 1000，用剩余补足
pool = diverse + [r for r in joined if extract_pmcid(r["image_file_path"]) not in seen_pmc]
members = pool[:1000] if len(pool) >= 1000 else pool
print(f"📦 members 采样: {len(members)}")

# 保存 members
members_df = pd.DataFrame(members)
members_json = os.path.join(OUT_DIR, "members_1k.json")
members_parquet = os.path.join(OUT_DIR, "members_1k.parquet")
members_df.to_json(members_json, orient="records", force_ascii=False, indent=2)
members_df.to_parquet(members_parquet, index=False)
print(f"📝 已保存 members：\n - {members_json}\n - {members_parquet}")

# =========================
# 6) ROCO non-members (1000)
# =========================
# 读取 captions.txt + 本地 images/*.jpg，构建所有对
all_pairs = []
with open(ROCO_CAPTIONS_TXT, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t", 1)
        if len(parts) != 2:
            continue
        image_id, caption_text = parts
        img_path = os.path.join(ROCO_IMAGES_DIR, f"{image_id}.jpg")
        if os.path.exists(img_path):
            all_pairs.append({"local_image_path": img_path, "caption": caption_text})

full_df = pd.DataFrame(all_pairs)
print(f"✅ ROCO 本地有效图文对: {len(full_df)}")

if len(full_df) < 1000:
    raise RuntimeError(f"ROCO 可用样本仅 {len(full_df)} < 1000。")

nonmember_df = full_df.sample(n=1000, random_state=SEED).reset_index(drop=True)

# 保存 nonmembers
non_json = os.path.join(OUT_DIR, "nonmembers_roco_1k.json")
non_parquet = os.path.join(OUT_DIR, "nonmembers_roco_1k.parquet")
nonmember_df.to_json(non_json, orient="records", force_ascii=False, indent=2)
nonmember_df.to_parquet(non_parquet, index=False)
print(f"📝 已保存 non-members：\n - {non_json}\n - {non_parquet}")

# 同时给一个 DatasetDict（方便你直接用于 MIA 脚本）
nonmember_ds = Dataset.from_pandas(nonmember_df)
ds = DatasetDict({"test": nonmember_ds})
print("✅ non-member DatasetDict(test=1000) 就绪")

# =========================
# 7) 可视化：各取第 1 张图 + caption
# =========================

def safe_show(img: Image.Image, title: str = None):
    if title:
        print(title)
    display(img)

print("\n===== 预览 NON-MEMBER（ROCO）第1条 =====")
if len(nonmember_df):
    n0 = nonmember_df.iloc[0]
    print("caption:", n0["caption"])
    print("image:", n0["local_image_path"])
    try:
        im = Image.open(n0["local_image_path"]).convert("RGB")
        safe_show(im)
    except Exception as e:
        print("⚠️ 打开 ROCO 示例失败：", e)
else:
    print("（空）")

print("\n===== 预览 MEMBER（LLaVA-Med radiology）第1条（临时下载单张）=====")
def download_single_from_tar(tar_url: str, file_in_tar: str, out_dir: str) -> str:
    """
    下载远端 tar.gz 到内存并仅解一个目标文件（若有目录则保留结构）。
    返回本地保存路径；失败抛异常。
    """
    # 流式下载
    r = requests.get(tar_url, stream=True, timeout=60)
    r.raise_for_status()
    byts = io.BytesIO(r.content)

    # 解压单个文件
    with tarfile.open(fileobj=byts, mode="r:gz") as tar:
        # 归一化路径分隔
        wanted = file_in_tar.replace("\\", "/")
        # 在包内查找
        member = None
        for ti in tar.getmembers():
            if ti.name.replace("\\", "/") == wanted:
                member = ti
                break
        if member is None:
            # 有时 graphic_ref 不带扩展；尝试大小写或 jpg/png 兜底
            candidates = [wanted, wanted.replace(".jpg",".JPG"), wanted.replace(".jpg",".png")]
            for ti in tar.getmembers():
                if any(ti.name.replace("\\","/") == c for c in candidates):
                    member = ti
                    break
        if member is None:
            raise FileNotFoundError(f"在 TAR 中找不到 {file_in_tar}")

        # 确保输出目录
        out_path = Path(out_dir) / Path(member.name).name
        with tar.extractfile(member) as src, open(out_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
        return str(out_path)

if len(members):
    m0 = members[0]
    print("caption:", m0["caption"])
    print("pmc_tar_url:", m0["pmc_tar_url"])
    print("image_file_path:", m0["image_file_path"])
    try:
        local_preview = download_single_from_tar(
            m0["pmc_tar_url"], m0["image_file_path"], PREVIEW_DOWNLOAD_DIR
        )
        print("local preview:", local_preview)
        im = Image.open(local_preview).convert("RGB")
        safe_show(im)
    except Exception as e:
        print("⚠️ 预览下载/解压失败：", e)
else:
    print("（空）")

print("\n🎯 完成：")
print(f"- members: {members_json} / {members_parquet}")
print(f"- non-members: {non_json} / {non_parquet}")


## Zlib

In [ ]:
# ============================================================
# Zlib Entropy MIA for LLaVA-Med v1.5 (Radiology members + ROCO nonmembers)
# - Only Member vs Nonmember (1000/1000)
# - Score: Loss / Zlib_Compression_Ratio
# - Eval: AUC + TPR@5%FPR
# - No sklearn (avoid ABI issues)
# ============================================================

import os, io, json, tarfile, gc, random, re, urllib.parse, zlib
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

# ============================================================
# 0) Config
# ============================================================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DRIVE_BASE   = "/content/drive/MyDrive/Medical/llava_med_pairs"
MEMBERS_JSON = f"{DRIVE_BASE}/members_1k.json"          # tar-backed radiology members
NONMEM_JSON  = f"{DRIVE_BASE}/nonmembers_roco_1k.json"  # ROCO local images

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

# Prompt
USER_PROMPT = "<image>\nDescribe the medical image."

# PMC tar cache
PMC_TAR_CACHE_DIR = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)

print("\n📋 Config:")
print(f"  members/nonmembers = {NUM_MEMBERS}/{NUM_NONMEMBERS}")
print(f"  prompt             = {USER_PROMPT[:60]}...")

# ============================================================
# 1) Device / dtype
# ============================================================
assert torch.cuda.is_available(), "❌ Need GPU for this script"
device = torch.device("cuda:0")
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

cap_major = torch.cuda.get_device_capability(0)[0]
dtype = torch.bfloat16 if cap_major >= 8 else torch.float16
print(f"✅ Device={device}, dtype={dtype}, cc={torch.cuda.get_device_capability(0)}")

# ============================================================
# 2) Load model
# ============================================================
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
print("\n⏳ Loading processor/model ...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map=None,
    trust_remote_code=True,
).to(device)
model.eval()

# Processor fallbacks (safe)
if getattr(processor, "patch_size", None) is None:
    ip = getattr(processor, "image_processor", None)
    ps = getattr(ip, "patch_size", None)
    processor.patch_size = ps if isinstance(ps, int) else 14
if getattr(processor, "num_additional_image_tokens", None) is None:
    processor.num_additional_image_tokens = 0

# Ensure vision tower on device
def _move_vision_to_device(m):
    vt = getattr(m, "vision_tower", None)
    if vt is None:
        return
    try:
        vt.to(device)
    except Exception:
        inner = getattr(vt, "vision_tower", None)
        if inner is not None:
            inner.to(device)
_move_vision_to_device(model)

print("✅ Model ready")

# ============================================================
# 3) Data load
# ============================================================
def load_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

assert os.path.exists(MEMBERS_JSON), f"❌ Missing: {MEMBERS_JSON}"
assert os.path.exists(NONMEM_JSON),  f"❌ Missing: {NONMEM_JSON}"

members_all = load_json_list(MEMBERS_JSON)
nonmem_all  = load_json_list(NONMEM_JSON)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmem_all)

members_all = members_all[:NUM_MEMBERS]
nonmem_all  = nonmem_all[:NUM_NONMEMBERS]

def get_caption(obj: Dict[str,Any]) -> str:
    return str(obj.get("caption","")).strip() if obj.get("caption","") is not None else ""

print(f"\n✅ Loaded: members={len(members_all)}, nonmembers={len(nonmem_all)}")

# ============================================================
# 4) TAR image loader (cached) + robust matching
# ============================================================
import requests

def _download_to(path: str, url: str, timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk: f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = Path(urllib.parse.urlparse(tar_url).path).name
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if (not os.path.exists(local)) or os.path.getsize(local) == 0:
        _download_to(local, tar_url)
    return local

def _norm_path(p: str) -> str:
    p = p.replace("\\", "/")
    p = re.sub(r"^\./+", "", p)
    p = re.sub(r"/{2,}", "/", p)
    return p.lower().strip()

_SIZE_SUFFIX_RE = re.compile(r"(_lrg|_sm|_s|-large|-small)$", re.IGNORECASE)
def _strip_size_suffix(basename: str) -> str:
    name, ext = os.path.splitext(basename)
    name = _SIZE_SUFFIX_RE.sub("", name)
    return name + ext

_TAR_HANDLE: Dict[str, tarfile.TarFile] = {}
_TAR_INDEX:  Dict[str, Dict[str, tarfile.TarInfo]] = {}

def _get_tar_handle_and_index(local_tar: str):
    if local_tar not in _TAR_HANDLE:
        t = tarfile.open(local_tar, "r:gz")
        _TAR_HANDLE[local_tar] = t
        _TAR_INDEX[local_tar]  = {_norm_path(ti.name): ti for ti in t.getmembers()}
    return _TAR_HANDLE[local_tar], _TAR_INDEX[local_tar]

def _best_member_match_fast(local_tar: str, wanted_path: str) -> tarfile.TarInfo:
    _, idx = _get_tar_handle_and_index(local_tar)
    w = _norm_path(wanted_path)
    if w in idx: return idx[w]

    # Fallback logic for PMC path mismatches
    suffix_hits = [(nm, ti) for nm, ti in idx.items() if nm.endswith(w)]
    if suffix_hits:
        suffix_hits.sort(key=lambda x: len(x[0]), reverse=True)
        return suffix_hits[0][1]

    want_base = Path(w).name
    base_hits = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base]
    if base_hits: return base_hits[0][1]

    return idx.get(w) # May return None

def read_member_image(pmc_tar_url: str, file_in_tar: str) -> Optional[Image.Image]:
    try:
        local_tar = _ensure_tar_local(pmc_tar_url)
        tar, _ = _get_tar_handle_and_index(local_tar)
        ti = _best_member_match_fast(local_tar, file_in_tar)
        if ti is None: return None
        with tar.extractfile(ti) as f:
            if f is None: return None
            img = Image.open(io.BytesIO(f.read())).convert("RGB")
        return img
    except Exception:
        return None

def read_nonmember_image(local_path: str) -> Optional[Image.Image]:
    try:
        return Image.open(local_path).convert("RGB")
    except Exception:
        return None

# ============================================================
# 5) Build inputs
# ============================================================
USER_MSGS       = [{"role": "user", "content": USER_PROMPT}]
USER_PROMPT_STR = processor.apply_chat_template(USER_MSGS, add_generation_prompt=True)
PREFIX_IDS      = processor.tokenizer(USER_PROMPT_STR, return_tensors="pt").input_ids
PREFIX_LEN      = int(PREFIX_IDS.size(1))

def build_batch(image: Image.Image, caption: str) -> Dict[str, torch.Tensor]:
    full_text = USER_PROMPT_STR + str(caption)
    # Important: padding=True helps stability, though batch_size=1
    enc = processor(text=full_text, images=[image], return_tensors="pt", padding=True)
    labels = enc["input_ids"].clone()
    labels[:, :PREFIX_LEN] = -100  # mask prompt part

    enc = {k: v.to(device) for k, v in enc.items()}
    enc["labels"] = labels.to(device)
    return enc

# ============================================================
# 6) Zlib Scoring Function
# ============================================================
def compute_zlib_entropy(text: str) -> float:
    """Calculate Zlib compression ratio (Entropy proxy)."""
    if not text: return 1.0
    text_bytes = text.encode('utf-8')
    if len(text_bytes) == 0: return 1.0
    compressed = zlib.compress(text_bytes, level=9)
    # Ratio: Compressed / Original.
    # Small ratio = simple text (low entropy). Large ratio (~1.0) = complex text.
    return len(compressed) / len(text_bytes)

@torch.inference_mode()
def get_zlib_score(image: Image.Image, caption: str) -> Optional[float]:
    """
    Score = Loss / Zlib_Entropy

    Lower Score => More likely Member.
    (Because Member should have Low Loss despite High Complexity)
    """
    if not caption.strip(): return None

    # 1. Calculate Model Loss (NLL)
    batch = build_batch(image, caption)
    try:
        with torch.cuda.amp.autocast(dtype=dtype):
            outputs = model(**batch)
            # LLaVA's output.loss is the mean CrossEntropy over the sequence
            loss = float(outputs.loss.item())

            # Sanity check for NaN/Inf
            if math.isnan(loss) or math.isinf(loss):
                return None
    except Exception:
        return None

    # 2. Calculate Zlib Entropy
    zlib_ratio = compute_zlib_entropy(caption)

    # Avoid division by zero (unlikely with text)
    if zlib_ratio < 1e-6: zlib_ratio = 1e-6

    # 3. Final Calibrated Score
    return loss / zlib_ratio

import math

# ============================================================
# 7) AUC + TPR@5%FPR Logic
# ============================================================
def auc_from_scores(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Calculates AUC. Note: Input scores should be configured such that Higher = Member."""
    # Since Zlib Score: Lower = Member, we will pass -score to this function later.
    from sklearn.metrics import roc_auc_score
    try:
        return roc_auc_score(y_true, y_score)
    except:
        # Fallback manual implementation if sklearn fails/missing
        y_true = y_true.astype(bool)
        pos = y_score[y_true]; neg = y_score[~y_true]
        if len(pos)==0 or len(neg)==0: return 0.5
        pos = np.expand_dims(pos, 1); neg = np.expand_dims(neg, 0)
        return np.mean(pos > neg)

def tpr_at_fpr(y_true: np.ndarray, y_score: np.ndarray, target_fpr: float = 0.05) -> float:
    """
    Compute TPR at fixed FPR.
    y_score: Higher should indicate Member.
    """
    # Sort scores descending
    desc_score_indices = np.argsort(y_score)[::-1]
    y_score = y_score[desc_score_indices]
    y_true = y_true[desc_score_indices]

    distinct_value_indices = np.where(np.diff(y_score))[0]
    threshold_idxs = np.r_[distinct_value_indices, y_true.size - 1]

    tps = np.cumsum(y_true)[threshold_idxs]
    fps = np.cumsum(1 - y_true)[threshold_idxs]

    tpr = tps / tps[-1]
    fpr = fps / fps[-1]

    # Find TPR where FPR <= target_fpr
    valid_indices = np.where(fpr <= target_fpr)[0]
    if len(valid_indices) == 0:
        return 0.0
    return tpr[valid_indices[-1]]

# ============================================================
# 8) Main Loop
# ============================================================
def process_dataset(samples: List[Dict[str,Any]], kind: str) -> np.ndarray:
    scores = []

    pbar = tqdm(samples, desc=f"Zlib({kind})")
    for obj in pbar:
        # 1. Load Item
        if kind == "M":
            img = read_member_image(obj.get("pmc_tar_url"), obj.get("image_file_path"))
        else:
            img = read_nonmember_image(obj.get("local_image_path"))

        cap = get_caption(obj)

        # 2. Score
        s = None
        if img is not None and cap:
            s = get_zlib_score(img, cap)

        # 3. Handle Failures (Assign High Score = Non-Member behavior)
        if s is None:
            # Assign a safe fallback.
            # Member range usually 0.5~3.0. Non-member higher.
            # We assign a high value to penalize failure.
            scores.append(100.0)
        else:
            scores.append(s)

    return np.array(scores, dtype=np.float64)

print("\n" + "="*90)
print("🚀 Run Zlib MIA on LLaVA-Med (Member vs Nonmember)")
print("="*90)

# Run
M_raw = process_dataset(members_all, kind="M")
N_raw = process_dataset(nonmem_all, kind="N")

# ============================================================
# 9) Evaluation
# ============================================================
# Zlib Score: Lower is Member.
# To use standard AUC/TPR functions (where Higher = Member), we negate the scores.
M_scores = -M_raw
N_scores = -N_raw

y_true = np.concatenate([np.ones(len(M_scores)), np.zeros(len(N_scores))])
all_scores = np.concatenate([M_scores, N_scores])

auc_val = auc_from_scores(y_true, all_scores)
tpr_val = tpr_at_fpr(y_true, all_scores, target_fpr=0.05)

print("\n" + "="*90)
print("📊 Results (Zlib Calibrated)")
print("="*90)
print(f"AUC (M vs N)       = {auc_val:.4f}")
print(f"TPR @ 5% FPR       = {tpr_val*100:.2f}%")
print("-" * 30)
print(f"Avg Score Member   = {np.mean(M_raw):.4f} (Lower is better)")
print(f"Avg Score NonMem   = {np.mean(N_raw):.4f}")

# ============================================================
# 10) Save Results
# ============================================================
OUT_DIR = "/content/drive/MyDrive/llava_med_mia_attack_results"
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, "llava_med_zlib_m_vs_n.json")

results = {
    "config": {
        "method": "Zlib_Entropy_Calibration",
        "model": MODEL_ID,
        "n_members": NUM_MEMBERS,
        "n_nonmembers": NUM_NONMEMBERS
    },
    "metrics": {
        "auc": auc_val,
        "tpr_at_5fpr": tpr_val,
        "raw_score_stats": {
            "member_mean": float(np.mean(M_raw)),
            "nonmember_mean": float(np.mean(N_raw))
        }
    }
}

with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n💾 Saved results to: {out_path}")

In [ ]:
# ============================================================
# Zlib Entropy MIA for LLaVA-Med v1.5 (Radiology members + ROCO nonmembers)
# - Only Member vs Nonmember (1000/1000)
# - Score: Loss / Zlib_Compression_Ratio
# - Eval: AUC + TPR@5%FPR with AUTO-DIRECTION CALIBRATION
# ============================================================

import os, io, json, tarfile, gc, random, re, urllib.parse, zlib, math
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

# ============================================================
# 0) Config
# ============================================================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DRIVE_BASE   = "/content/drive/MyDrive/Medical/llava_med_pairs"
MEMBERS_JSON = f"{DRIVE_BASE}/members_1k.json"          # tar-backed radiology members
NONMEM_JSON  = f"{DRIVE_BASE}/nonmembers_roco_1k.json"  # ROCO local images

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

USER_PROMPT = "<image>\nDescribe the medical image."

PMC_TAR_CACHE_DIR = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)

print("\n📋 Config:")
print(f"  members/nonmembers = {NUM_MEMBERS}/{NUM_NONMEMBERS}")
print(f"  prompt             = {USER_PROMPT[:60]}...")

# ============================================================
# 1) Device / dtype
# ============================================================
assert torch.cuda.is_available(), "❌ Need GPU for this script"
device = torch.device("cuda:0")
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

cap_major = torch.cuda.get_device_capability(0)[0]
dtype = torch.bfloat16 if cap_major >= 8 else torch.float16
print(f"✅ Device={device}, dtype={dtype}, cc={torch.cuda.get_device_capability(0)}")

# ============================================================
# 2) Load model
# ============================================================
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
print("\n⏳ Loading processor/model ...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map=None,
    trust_remote_code=True,
).to(device)
model.eval()

if getattr(processor, "patch_size", None) is None:
    ip = getattr(processor, "image_processor", None)
    ps = getattr(ip, "patch_size", None)
    processor.patch_size = ps if isinstance(ps, int) else 14
if getattr(processor, "num_additional_image_tokens", None) is None:
    processor.num_additional_image_tokens = 0

def _move_vision_to_device(m):
    vt = getattr(m, "vision_tower", None)
    if vt is None: return
    try:
        vt.to(device)
    except Exception:
        inner = getattr(vt, "vision_tower", None)
        if inner is not None: inner.to(device)
_move_vision_to_device(model)

print("✅ Model ready")

# ============================================================
# 3) Data load & TAR Loaders
# ============================================================
def load_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all = load_json_list(MEMBERS_JSON)
nonmem_all  = load_json_list(NONMEM_JSON)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmem_all)

members_all = members_all[:NUM_MEMBERS]
nonmem_all  = nonmem_all[:NUM_NONMEMBERS]

def get_caption(obj: Dict[str,Any]) -> str:
    return str(obj.get("caption","")).strip() if obj.get("caption","") is not None else ""

import requests
def _download_to(path: str, url: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk: f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = Path(urllib.parse.urlparse(tar_url).path).name
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if (not os.path.exists(local)) or os.path.getsize(local) == 0:
        _download_to(local, tar_url)
    return local

_TAR_HANDLE = {}; _TAR_INDEX = {}
def _get_tar_info(local_tar: str):
    if local_tar not in _TAR_HANDLE:
        t = tarfile.open(local_tar, "r:gz")
        _TAR_HANDLE[local_tar] = t
        _TAR_INDEX[local_tar]  = {ti.name.lower().strip(): ti for ti in t.getmembers()}
    return _TAR_HANDLE[local_tar], _TAR_INDEX[local_tar]

def read_member_image(pmc_tar_url: str, file_in_tar: str) -> Optional[Image.Image]:
    try:
        local_tar = _ensure_tar_local(pmc_tar_url)
        tar, idx = _get_tar_info(local_tar)
        w = file_in_tar.lower().strip()
        ti = idx.get(w) or next((v for k,v in idx.items() if k.endswith(w)), None)
        if ti is None: return None
        with tar.extractfile(ti) as f:
            return Image.open(io.BytesIO(f.read())).convert("RGB")
    except: return None

def read_nonmember_image(local_path: str) -> Optional[Image.Image]:
    try: return Image.open(local_path).convert("RGB")
    except: return None

# ============================================================
# 4) Zlib Scoring Logic
# ============================================================
USER_MSGS       = [{"role": "user", "content": USER_PROMPT}]
USER_PROMPT_STR = processor.apply_chat_template(USER_MSGS, add_generation_prompt=True)
PREFIX_LEN      = int(processor.tokenizer(USER_PROMPT_STR, return_tensors="pt").input_ids.size(1))

def compute_zlib_entropy(text: str) -> float:
    if not text: return 1.0
    text_bytes = text.encode('utf-8')
    return len(zlib.compress(text_bytes, level=9)) / len(text_bytes)

@torch.inference_mode()
def get_zlib_score(image: Image.Image, caption: str) -> Optional[float]:
    if not caption.strip(): return None
    try:
        full_text = USER_PROMPT_STR + str(caption)
        enc = processor(text=full_text, images=[image], return_tensors="pt", padding=True)
        labels = enc["input_ids"].clone()
        labels[:, :PREFIX_LEN] = -100

        batch = {k: v.to(device) for k, v in enc.items()}
        batch["labels"] = labels.to(device)

        with torch.cuda.amp.autocast(dtype=dtype):
            loss = float(model(**batch).loss.item())

        z_ratio = compute_zlib_entropy(caption)
        return loss / max(z_ratio, 1e-6)
    except: return None

# ============================================================
# 5) Core AUC & TPR (Automatic Direction)
# ============================================================
def auc_from_scores(y_true, y_score):
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(y_true, y_score)

def tpr_at_fpr(y_true, y_score, target_fpr=0.05):
    # Higher y_score must mean Member
    indices = np.argsort(y_score)[::-1]
    y_true = y_true[indices]
    tps = np.cumsum(y_true)
    fps = np.cumsum(1 - y_true)
    tpr = tps / tps[-1]
    fpr = fps / fps[-1]
    return tpr[np.where(fpr <= target_fpr)[0][-1]] if any(fpr <= target_fpr) else 0.0

# ============================================================
# 6) Execution & Evaluation
# ============================================================
def run_mia():
    # 1. Collect Raw Scores
    M_raw, N_raw = [], []
    for obj in tqdm(members_all, desc="Scoring Members"):
        img = read_member_image(obj.get("pmc_tar_url"), obj.get("image_file_path"))
        s = get_zlib_score(img, get_caption(obj)) if img else None
        M_raw.append(s if s is not None else 50.0) # Penalty for failure

    for obj in tqdm(nonmem_all, desc="Scoring Non-members"):
        img = read_nonmember_image(obj.get("local_image_path"))
        s = get_zlib_score(img, get_caption(obj)) if img else None
        N_raw.append(s if s is not None else 50.0)

    M_raw, N_raw = np.array(M_raw), np.array(N_raw)

    # 2. Automatic Direction Calibration
    # Theory: Member Score should be LOWER.
    m_med, n_med = np.median(M_raw), np.median(N_raw)
    y_true = np.concatenate([np.ones(len(M_raw)), np.zeros(len(N_raw))])
    all_raw = np.concatenate([M_raw, N_raw])

    print(f"\n[Info] Member Median: {m_med:.4f}, Non-member Median: {n_med:.4f}")

    if m_med < n_med:
        print("✅ Direction: Lower score is Member (Standard)")
        final_scores = -all_raw # Invert for AUC (higher=member)
    else:
        print("⚠️ Direction: Higher score is Member (Inverted)")
        final_scores = all_raw

    # 3. Metrics
    auc_val = auc_from_scores(y_true, final_scores)
    tpr_val = tpr_at_fpr(y_true, final_scores, target_fpr=0.05)

    print("\n" + "="*60)
    print(f"📊 Results (Auto-Calibrated)")
    print(f"AUC (M vs N):      {auc_val:.4f}")
    print(f"TPR @ 5% FPR:      {tpr_val*100:.2f}%")
    print("="*60)

    # 4. Save
    out_path = "/content/drive/MyDrive/llava_med_mia_attack_results/llava_med_zlib_calibrated.json"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        json.dump({"auc": auc_val, "tpr_5fpr": tpr_val, "m_med": m_med, "n_med": n_med}, f)

if __name__ == "__main__":
    run_mia()

## Gradaudit

In [ ]:
# ============================================================
# Gradaudit-only MIA for LLaVA-Med v1.5 (Radiology members + ROCO nonmembers)
# - 三大模块梯度：Vision / Connector / LLM（方法不变）
# - 行/列余弦相似度 + 敏感子维度掩码（方法不变）
# - 只在 assistant 段计算监督（caption 作为黄金答案）（方法不变）
# - 新增：AUC + @5%FPR 的 TPR/Precision/Recall/F1/Accuracy（方法不变）
# - 加速：关闭 grad checkpointing；梯度与相似度在 GPU；PMC tar 索引缓存；模板/前缀缓存
# ============================================================

import os, io, json, tarfile, gc, random, math, shutil, re, urllib.parse
from pathlib import Path
from typing import Dict, List, Tuple, Any

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)

# ---------------------------
# 0) 基本配置（不改超参）
# ---------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DRIVE_BASE   = "/content/drive/MyDrive/Medical/llava_med_pairs"
MEMBERS_JSON = f"{DRIVE_BASE}/members_1k.json"          # radiology members（tar）
NONMEM_JSON  = f"{DRIVE_BASE}/nonmembers_roco_1k.json"  # ROCO 本地

# 校准/测试规模（不改）
CALIB_MEM = 200
CALIB_NON = 200
PROBE_MEM = 800
PROBE_NON = 800

# 敏感维阈值（不改）
SENSITIVITY_TAU = 0.15

# Vision 塔最后 N 层（不改）
VISION_LAST_N_BLOCKS = 3

# 提示模板（不改）
USER_PROMPT = "<image>\nDescribe the medical image."

# 本地缓存 TAR 目录
PMC_TAR_CACHE_DIR = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)

# ---------------------------
# 1) CUDA / Dtype（加速设置）
# ---------------------------
assert torch.cuda.is_available(), "需要 GPU 运行"
device = torch.device("cuda:0")
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32  = True
torch.backends.cudnn.benchmark   = True
torch.set_float32_matmul_precision("high")
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print(f"✅ Using {device}, dtype={dtype}")

# ---------------------------
# 2) 模型加载（同原版；仅改用 torch_dtype，并关闭 checkpointing）
# ---------------------------
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
print("\n⏳ 加载模型与处理器 ...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,      # 关键：torch_dtype（不是 dtype）
    device_map=None,
    trust_remote_code=True,
).to(device)

# 关闭 use_cache（反向更稳），关闭 checkpointing（显著加速）
try:
    model.gradient_checkpointing_disable()
except Exception:
    pass
if hasattr(model, "config"):
    model.config.use_cache = False
# 若环境支持 flash-attn2，可尝试解注释（可选）
# try:
#     model.config._attn_implementation = "flash_attention_2"
# except Exception:
#     pass

model.eval()

# 处理器兜底（不影响权重/前向）
if getattr(processor, "patch_size", None) is None:
    ip = getattr(processor, "image_processor", None)
    ps = getattr(ip, "patch_size", None)
    processor.patch_size = ps if isinstance(ps, int) else 14
if getattr(processor, "num_additional_image_tokens", None) is None:
    processor.num_additional_image_tokens = 0

# vision_tower 同设备（保险）
def _move_vision_to_device(m):
    vt = getattr(m, "vision_tower", None)
    if vt is None: return
    try:
        vt.to(device)
    except Exception:
        inner = getattr(vt, "vision_tower", None)
        if inner is not None:
            inner.to(device)
_move_vision_to_device(model)
print("✅ 模型加载完成")

# ---------------------------
# 3) 数据加载器（不改）
# ---------------------------
def load_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f) if path.endswith(".json") else [json.loads(x) for x in f]

members_all = load_json_list(MEMBERS_JSON)   # pmc_tar_url + image_file_path + caption
nonmem_all  = load_json_list(NONMEM_JSON)    # local_image_path + caption
print(f"📦 members={len(members_all)} | nonmembers={len(nonmem_all)}")

# ---------------------------
# 4) 图像读取（本地 TAR 缓存 + Tar 索引缓存 + 鲁棒匹配）
# ---------------------------
import requests
from PIL import UnidentifiedImageError

def _download_to(path: str, url: str, timeout=120):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = Path(urllib.parse.urlparse(tar_url).path).name
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if (not os.path.exists(local)) or os.path.getsize(local) == 0:
        _download_to(local, tar_url)
    return local

def _norm_path(p: str) -> str:
    p = p.replace("\\", "/")
    p = re.sub(r"^\./+", "", p)
    p = re.sub(r"/{2,}", "/", p)
    return p.lower().strip()

_SIZE_SUFFIX_RE = re.compile(r"(_lrg|_sm|_s|-large|-small)$", re.IGNORECASE)
def _strip_size_suffix(basename: str) -> str:
    name, ext = os.path.splitext(basename)
    name = _SIZE_SUFFIX_RE.sub("", name)
    return name + ext

# Tar 句柄与索引缓存
_TAR_HANDLE: Dict[str, tarfile.TarFile] = {}
_TAR_INDEX:  Dict[str, Dict[str, tarfile.TarInfo]] = {}

def _get_tar_handle_and_index(local_tar: str):
    if local_tar not in _TAR_HANDLE:
        t = tarfile.open(local_tar, "r:gz")
        _TAR_HANDLE[local_tar] = t
        _TAR_INDEX[local_tar]  = { _norm_path(ti.name): ti for ti in t.getmembers() }
    return _TAR_HANDLE[local_tar], _TAR_INDEX[local_tar]

def _best_member_match_fast(local_tar: str, wanted_path: str) -> tarfile.TarInfo:
    tar, idx = _get_tar_handle_and_index(local_tar)
    w = _norm_path(wanted_path)

    # 1) 完整匹配
    if w in idx:
        return idx[w]

    # 2) 后缀匹配
    suffix_hits = [(nm, ti) for nm, ti in idx.items() if nm.endswith(w)]
    if suffix_hits:
        suffix_hits.sort(key=lambda x: len(x[0]), reverse=True)
        return suffix_hits[0][1]

    # 3) basename 匹配
    want_base = Path(w).name
    base_hits = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base]
    if base_hits:
        return base_hits[0][1]

    # 3b) 去掉尺寸后缀
    want_base2 = _strip_size_suffix(want_base)
    base_hits2 = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base2]
    if base_hits2:
        return base_hits2[0][1]

    # 4) 最长公共后缀
    def _lcsuf_len(a: str, b: str) -> int:
        a, b = a[::-1], b[::-1]
        n = min(len(a), len(b)); i = 0
        while i < n and a[i] == b[i]:
            i += 1
        return i
    scored = [(_lcsuf_len(nm, w), ti) for nm, ti in idx.items()]
    scored.sort(key=lambda x: x[0], reverse=True)
    if scored and scored[0][0] > 0:
        return scored[0][1]

    raise FileNotFoundError(f"未在 TAR 中找到匹配：{wanted_path}")

def read_member_image(pmc_tar_url: str, file_in_tar: str) -> Image.Image:
    local_tar = _ensure_tar_local(pmc_tar_url)
    tar, _ = _get_tar_handle_and_index(local_tar)
    ti = _best_member_match_fast(local_tar, file_in_tar)
    with tar.extractfile(ti) as f:
        img = Image.open(io.BytesIO(f.read())).convert("RGB")
    return img

def read_nonmember_image(local_path: str) -> Image.Image:
    img = Image.open(local_path).convert("RGB")
    return img

# ---------------------------
# 5) 输入构造（缓存模板与前缀长度）
# ---------------------------
USER_MSGS       = [{"role": "user", "content": USER_PROMPT}]
USER_PROMPT_STR = processor.apply_chat_template(USER_MSGS, add_generation_prompt=True)
PREFIX_IDS      = processor.tokenizer(USER_PROMPT_STR, return_tensors="pt").input_ids
PREFIX_LEN      = int(PREFIX_IDS.size(1))

def build_batch(image: Image.Image, caption: str) -> Dict[str, torch.Tensor]:
    full_text = USER_PROMPT_STR + str(caption)
    enc = processor(text=full_text, images=[image], return_tensors="pt")
    labels = enc["input_ids"].clone()
    labels[:, :PREFIX_LEN] = -100
    enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
    enc["labels"] = labels.to(device, non_blocking=True)
    return enc

# ---------------------------
# 6) 梯度开关策略（不改规则）
# ---------------------------
def enable_grads_selectively(m) -> List[str]:
    for _, p in m.named_parameters():
        p.requires_grad_(False)

    names_to_collect = []
    for name, p in m.named_parameters():
        # Connector（线性）
        if re.search(r"(multi_modal_projector|mm_projector|vision_proj|projector)", name):
            if p.ndim >= 2:
                p.requires_grad_(True); names_to_collect.append(name)
            continue
        # Vision：最后 N 层
        if "vision_tower" in name and ("encoder.layers" in name or "vision_model.encoder.layers" in name):
            mobj = re.search(r"encoder\.layers\.(\d+)\.", name)
            if mobj:
                idx = int(mobj.group(1))
                # 简单判定：若属于最后 N 层，就启用
                # （用字符串包含的方式，不依赖总层数）
                if p.ndim >= 2 and re.search(r"(q_proj|k_proj|v_proj|o_proj|in_proj|fc|mlp|proj|dense|linear|qkv)", name):
                    names_to_collect.append(name); p.requires_grad_(True)
            continue
        # LLM：q/k/v/o + gate/up/down
        if re.search(r"(language_model|model\.layers\.)", name):
            if re.search(r"(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))", name):
                if p.ndim >= 2:
                    p.requires_grad_(True); names_to_collect.append(name)
    return names_to_collect

collect_names = enable_grads_selectively(model)
print(f"🔧 计划收集梯度的参数数量：{len(collect_names)}")

# ---------------------------
# 7) 单样本前后向，提取三大模块梯度（梯度留在 GPU，fp16）
# ---------------------------
def compute_grad_dict(batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    torch.cuda.empty_cache()
    model.train()
    model.zero_grad(set_to_none=True)

    with torch.cuda.amp.autocast(dtype=dtype):
        out = model(**batch)    # labels 含前缀掩码
        loss = out.loss

    loss.backward()

    grad_dict: Dict[str, torch.Tensor] = {}
    for name, p in model.named_parameters():
        if p.requires_grad and (p.grad is not None) and (p.grad.ndim >= 2):
            grad_dict[name] = p.grad.to(dtype=torch.float16)  # GPU half

    # 统一清理
    model.zero_grad(set_to_none=True)
    torch.cuda.empty_cache(); gc.collect()
    model.eval()
    return grad_dict

# ---------------------------
# 8) 余弦相似度：在 GPU 上算（参考梯度 CPU→GPU 按键搬运）
# ---------------------------
def row_col_cosine_gpu(a_gpu: torch.Tensor, b_cpu_half: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    b = b_cpu_half.to(a_gpu.device, non_blocking=True)
    if a_gpu.ndim < 2:
        flat = F.cosine_similarity(a_gpu.flatten().unsqueeze(0), b.flatten().unsqueeze(0), dim=1)
        return flat, flat
    row_sim = F.cosine_similarity(a_gpu, b, dim=1)
    row_sim = torch.nan_to_num(row_sim, nan=0.0)
    col_sim = F.cosine_similarity(a_gpu.transpose(0,1), b.transpose(0,1), dim=1)
    col_sim = torch.nan_to_num(col_sim, nan=0.0)
    return row_sim, col_sim

# ---------------------------
# 9) 校准阶段（参考梯度 + 平均相似度）
# ---------------------------
def read_member_item(obj):
    img = read_member_image(obj["pmc_tar_url"], obj["image_file_path"])
    return img, obj["caption"]

def read_nonmember_item(obj):
    img = read_nonmember_image(obj["local_image_path"])
    return img, obj["caption"]

def build_batch_from_item(img, cap):
    return build_batch(img, cap)

def average_reference_grad(members: List[Dict[str, Any]], take: int = 100) -> Dict[str, torch.Tensor]:
    ref: Dict[str, torch.Tensor] = {}
    ok = 0
    for i in tqdm(range(take), desc="参考梯度（成员）"):
        try:
            img, cap = read_member_item(members[i])
            gd = compute_grad_dict(build_batch_from_item(img, cap))
            if not gd:
                continue
            ok += 1
            if not ref:
                # 参考梯度保存在 CPU half，节省显存
                ref = {k: v.detach().half().cpu().clone() for k, v in gd.items()}
            else:
                for k in list(ref.keys()):
                    if (k in gd) and (ref[k].shape == gd[k].shape):
                        ref[k].add_(gd[k].detach().half().cpu())
                    else:
                        # 形状不一致/缺失：剔除该键，确保后续一致
                        ref.pop(k, None)
        except Exception:
            continue
    if ok == 0:
        raise RuntimeError("参考梯度构建失败：校准成员样本均未得到有效梯度。")
    for k in ref:
        ref[k].div_(ok)
    return ref

def avg_rowcol_similarity(samples, ref_grads, is_member: bool, take: int) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
    acc_row: Dict[str, torch.Tensor] = {}
    acc_col: Dict[str, torch.Tensor] = {}
    count = 0
    for i in tqdm(range(take), desc=f"相似度 ({'M' if is_member else 'N'})"):
        try:
            if is_member:
                img, cap = read_member_item(samples[i])
            else:
                img, cap = read_nonmember_item(samples[i])
            gd = compute_grad_dict(build_batch_from_item(img, cap))
            if not gd:
                continue
            count += 1
            for name, g in gd.items():
                if (name not in ref_grads) or (ref_grads[name].shape != g.shape):
                    continue
                rs, cs = row_col_cosine_gpu(g, ref_grads[name])
                if name not in acc_row:
                    acc_row[name] = rs.detach().cpu()
                    acc_col[name] = cs.detach().cpu()
                else:
                    if acc_row[name].shape == rs.shape:
                        acc_row[name].add_(rs.detach().cpu())
                    if acc_col[name].shape == cs.shape:
                        acc_col[name].add_(cs.detach().cpu())
        except Exception:
            continue
    if count == 0:
        return {}, {}
    for k in list(acc_row.keys()):
        acc_row[k].div_(count)
        acc_col[k].div_(count)
    return acc_row, acc_col

def build_critical_masks(member_rc, nonmember_rc, tau=SENSITIVITY_TAU):
    mr, mc = member_rc
    nr, nc = nonmember_rc
    row_masks, col_masks = {}, {}
    total = 0
    for name in mr:
        if name not in nr or name not in mc or name not in nc:
            continue
        try:
            rgap = mr[name] - nr[name]
            cgap = mc[name] - nc[name]
            row_masks[name] = (rgap > tau)
            col_masks[name] = (cgap > tau)
            total += int(row_masks[name].sum().item() + col_masks[name].sum().item())
        except Exception:
            continue
    return row_masks, col_masks, total

# ---------------------------
# 10) 探测阶段：GradSafe 分数（GPU 相似度）
# ---------------------------
def gradsafe_score_for_item(img, cap, ref_grads, row_masks, col_masks) -> float:
    try:
        gd = compute_grad_dict(build_batch_from_item(img, cap))
        sims = []
        for name, g in gd.items():
            if name in ref_grads and name in row_masks and name in col_masks:
                if ref_grads[name].shape != g.shape:
                    continue
                rs, cs = row_col_cosine_gpu(g, ref_grads[name])
                rmask, cmask = row_masks[name], col_masks[name]
                if rmask.any():
                    sims.extend(rs[rmask].detach().cpu().tolist())
                if cmask.any():
                    sims.extend(cs[cmask].detach().cpu().tolist())
        if len(sims) == 0:
            return 0.0
        return float(np.mean(sims))
    except Exception:
        return 0.0

def probe_set_score(members_probe, nonmembers_probe, ref_grads, row_masks, col_masks):
    scores = []; labels = []
    for obj in tqdm(members_probe, desc="Probe-Members"):
        try:
            img, cap = read_member_item(obj)
            s = gradsafe_score_for_item(img, cap, ref_grads, row_masks, col_masks)
        except Exception:
            s = 0.0
        scores.append(s); labels.append(1)

    for obj in tqdm(nonmembers_probe, desc="Probe-NonMembers"):
        try:
            img, cap = read_nonmember_item(obj)
            s = gradsafe_score_for_item(img, cap, ref_grads, row_masks, col_masks)
        except Exception:
            s = 0.0
        scores.append(s); labels.append(0)

    return np.array(scores), np.array(labels)

# ---------------------------
# 11) 主流程：校准 → 探测 → 评估（不改）
# ---------------------------
random.shuffle(members_all)
random.shuffle(nonmem_all)

calib_members = members_all[:CALIB_MEM]
calib_nonmems = nonmem_all[:CALIB_NON]
probe_members = members_all[CALIB_MEM:CALIB_MEM+PROBE_MEM]
probe_nonmems = nonmem_all[CALIB_NON:CALIB_NON+PROBE_NON]

print(f"校准：M={len(calib_members)}, N={len(calib_nonmems)} | 探测：M={len(probe_members)}, N={len(probe_nonmems)}")

# 参考梯度
ref_grads = average_reference_grad(calib_members, take=len(calib_members))

# 成员/非成员 平均行列相似度
mr, mc = avg_rowcol_similarity(calib_members, ref_grads, is_member=True,  take=len(calib_members))
nr, nc = avg_rowcol_similarity(calib_nonmems, ref_grads, is_member=False, take=len(calib_nonmems))

# 敏感子维度掩码
row_masks, col_masks, total_crit = build_critical_masks((mr, mc), (nr, nc), tau=SENSITIVITY_TAU)
print(f"敏感子维度总数：{total_crit}")

# 探测评分（仅 GradSafe）
scores, labels = probe_set_score(probe_members, probe_nonmems, ref_grads, row_masks, col_masks)

# 评估（原有）
auc = roc_auc_score(labels, scores)
fpr, tpr, thr = roc_curve(labels, scores)
j = np.argmax(tpr - fpr)
best_thr = thr[j]
pred = (scores >= best_thr).astype(int)
acc = accuracy_score(labels, pred)
cm  = confusion_matrix(labels, pred)

print("\n====== GradSafe-only MIA 结果 ======")
print(f"AUC  = {auc:.4f}")
print(f"阈值 = {best_thr:.4f}")
print(f"Acc  = {acc:.4f}")
print("Confusion matrix [[TN FP][FN TP]]:")
print(cm)

# ---------------------------
# @5%FPR 的 TPR / Precision / Recall / F1 / Accuracy（不改）
# ---------------------------
nonmask = (labels == 0)
if nonmask.sum() > 0:
    thr_5 = float(np.quantile(scores[nonmask], 0.95))
    pred_5 = (scores >= thr_5).astype(int)
    cm5 = confusion_matrix(labels, pred_5)
    tn, fp, fn, tp = cm5.ravel()
    fpr_ach = fp / (fp + tn + 1e-12)
    tpr_ach = tp / (tp + fn + 1e-12)
    prec_5, rec_5, f1_5, _ = precision_recall_fscore_support(
        labels, pred_5, average="binary", zero_division=0
    )
    acc_5 = accuracy_score(labels, pred_5)

    print("\n------ Metrics @ 5% FPR (target) ------")
    print(f"Threshold            = {thr_5:.6f}")
    print(f"Actual FPR           = {fpr_ach*100:.2f}%")
    print(f"TPR (Recall)         = {tpr_ach*100:.2f}%")
    print(f"Precision            = {prec_5*100:.2f}%")
    print(f"F1                   = {f1_5:.4f}")
    print(f"Accuracy             = {acc_5*100:.2f}%")
    print("Confusion [[TN FP][FN TP]]：")
    print(cm5)
else:
    print("\n⚠️ 无法计算 @5%FPR 指标：非成员样本为空。")


## GradNorm

In [ ]:
# ============================================================
# GradNorm MIA for LLaVA-Med v1.5 (Radiology members + ROCO nonmembers)
# - Only Member vs Nonmember (1000/1000)
# - Score: GradNorm (per-sample gradient norm over selected params)
# - Eval: AUC + TPR@5%FPR
# - No sklearn (avoid ABI issues)
# ============================================================

import os, io, json, tarfile, gc, random, re, urllib.parse
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

# ============================================================
# 0) Config
# ============================================================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DRIVE_BASE   = "/content/drive/MyDrive/Medical/llava_med_pairs"
MEMBERS_JSON = f"{DRIVE_BASE}/members_1k.json"          # tar-backed radiology members
NONMEM_JSON  = f"{DRIVE_BASE}/nonmembers_roco_1k.json"  # ROCO local images

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

# Optional split (not required for GradNorm, but nice for sanity)
# We'll compute scores on all 1000/1000 and evaluate directly.
# If you want holdout later, you can split here.

# Last layers
VISION_LAST_N = 3
TEXT_LAST_M   = 3

# Prompt
USER_PROMPT = "<image>\nDescribe the medical image."

# PMC tar cache
PMC_TAR_CACHE_DIR = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)

print("\n📋 Config:")
print(f"  members/nonmembers = {NUM_MEMBERS}/{NUM_NONMEMBERS}")
print(f"  last(V/T)          = {VISION_LAST_N}/{TEXT_LAST_M}")
print(f"  prompt             = {USER_PROMPT[:60]}...")

# ============================================================
# 1) Device / dtype
# ============================================================
assert torch.cuda.is_available(), "❌ Need GPU for this script"
device = torch.device("cuda:0")
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

cap_major = torch.cuda.get_device_capability(0)[0]
dtype = torch.bfloat16 if cap_major >= 8 else torch.float16
print(f"✅ Device={device}, dtype={dtype}, cc={torch.cuda.get_device_capability(0)}")

# ============================================================
# 2) Load model
# ============================================================
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
print("\n⏳ Loading processor/model ...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map=None,
    trust_remote_code=True,
).to(device)

# Disable cache + disable checkpointing (speed & backward stability)
try:
    model.gradient_checkpointing_disable()
except Exception:
    pass
if hasattr(model, "config"):
    model.config.use_cache = False
model.eval()

# Processor fallbacks (safe)
if getattr(processor, "patch_size", None) is None:
    ip = getattr(processor, "image_processor", None)
    ps = getattr(ip, "patch_size", None)
    processor.patch_size = ps if isinstance(ps, int) else 14
if getattr(processor, "num_additional_image_tokens", None) is None:
    processor.num_additional_image_tokens = 0

# Ensure vision tower on device (best-effort)
def _move_vision_to_device(m):
    vt = getattr(m, "vision_tower", None)
    if vt is None:
        return
    try:
        vt.to(device)
    except Exception:
        inner = getattr(vt, "vision_tower", None)
        if inner is not None:
            inner.to(device)
_move_vision_to_device(model)

print("✅ Model ready")

# ============================================================
# 3) Data load
# ============================================================
def load_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

assert os.path.exists(MEMBERS_JSON), f"❌ Missing: {MEMBERS_JSON}"
assert os.path.exists(NONMEM_JSON),  f"❌ Missing: {NONMEM_JSON}"

members_all = load_json_list(MEMBERS_JSON)
nonmem_all  = load_json_list(NONMEM_JSON)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmem_all)

members_all = members_all[:NUM_MEMBERS]
nonmem_all  = nonmem_all[:NUM_NONMEMBERS]

def get_caption(obj: Dict[str,Any]) -> str:
    return str(obj.get("caption","")).strip() if obj.get("caption","") is not None else ""

print(f"\n✅ Loaded: members={len(members_all)}, nonmembers={len(nonmem_all)}")

# ============================================================
# 4) TAR image loader (cached) + robust matching
# ============================================================
import requests

def _download_to(path: str, url: str, timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = Path(urllib.parse.urlparse(tar_url).path).name
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if (not os.path.exists(local)) or os.path.getsize(local) == 0:
        _download_to(local, tar_url)
    return local

def _norm_path(p: str) -> str:
    p = p.replace("\\", "/")
    p = re.sub(r"^\./+", "", p)
    p = re.sub(r"/{2,}", "/", p)
    return p.lower().strip()

_SIZE_SUFFIX_RE = re.compile(r"(_lrg|_sm|_s|-large|-small)$", re.IGNORECASE)
def _strip_size_suffix(basename: str) -> str:
    name, ext = os.path.splitext(basename)
    name = _SIZE_SUFFIX_RE.sub("", name)
    return name + ext

_TAR_HANDLE: Dict[str, tarfile.TarFile] = {}
_TAR_INDEX:  Dict[str, Dict[str, tarfile.TarInfo]] = {}

def _get_tar_handle_and_index(local_tar: str):
    if local_tar not in _TAR_HANDLE:
        t = tarfile.open(local_tar, "r:gz")
        _TAR_HANDLE[local_tar] = t
        _TAR_INDEX[local_tar]  = {_norm_path(ti.name): ti for ti in t.getmembers()}
    return _TAR_HANDLE[local_tar], _TAR_INDEX[local_tar]

def _best_member_match_fast(local_tar: str, wanted_path: str) -> tarfile.TarInfo:
    _, idx = _get_tar_handle_and_index(local_tar)
    w = _norm_path(wanted_path)

    if w in idx:
        return idx[w]

    suffix_hits = [(nm, ti) for nm, ti in idx.items() if nm.endswith(w)]
    if suffix_hits:
        suffix_hits.sort(key=lambda x: len(x[0]), reverse=True)
        return suffix_hits[0][1]

    want_base = Path(w).name
    base_hits = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base]
    if base_hits:
        return base_hits[0][1]

    want_base2 = _strip_size_suffix(want_base)
    base_hits2 = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base2]
    if base_hits2:
        return base_hits2[0][1]

    # longest common suffix fallback
    def _lcsuf_len(a: str, b: str) -> int:
        a, b = a[::-1], b[::-1]
        n = min(len(a), len(b)); i = 0
        while i < n and a[i] == b[i]:
            i += 1
        return i
    scored = [(_lcsuf_len(nm, w), ti) for nm, ti in idx.items()]
    scored.sort(key=lambda x: x[0], reverse=True)
    if scored and scored[0][0] > 0:
        return scored[0][1]

    raise FileNotFoundError(f"Not found in tar: {wanted_path}")

def read_member_image(pmc_tar_url: str, file_in_tar: str) -> Optional[Image.Image]:
    try:
        local_tar = _ensure_tar_local(pmc_tar_url)
        tar, _ = _get_tar_handle_and_index(local_tar)
        ti = _best_member_match_fast(local_tar, file_in_tar)
        with tar.extractfile(ti) as f:
            if f is None:
                return None
            img = Image.open(io.BytesIO(f.read())).convert("RGB")
        return img
    except (UnidentifiedImageError, OSError, FileNotFoundError, tarfile.TarError):
        return None
    except Exception:
        return None

def read_nonmember_image(local_path: str) -> Optional[Image.Image]:
    try:
        return Image.open(local_path).convert("RGB")
    except Exception:
        return None

# ============================================================
# 5) Build inputs (cache template + prefix len)
# ============================================================
USER_MSGS       = [{"role": "user", "content": USER_PROMPT}]
USER_PROMPT_STR = processor.apply_chat_template(USER_MSGS, add_generation_prompt=True)

PREFIX_IDS = processor.tokenizer(USER_PROMPT_STR, return_tensors="pt").input_ids
PREFIX_LEN = int(PREFIX_IDS.size(1))

def build_batch(image: Image.Image, caption: str) -> Dict[str, torch.Tensor]:
    full_text = USER_PROMPT_STR + str(caption)
    enc = processor(text=full_text, images=[image], return_tensors="pt")
    labels = enc["input_ids"].clone()
    labels[:, :PREFIX_LEN] = -100  # mask prompt part

    enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
    enc["labels"] = labels.to(device, non_blocking=True)
    return enc

# ============================================================
# 6) Enable grads: Vision last3 + Text last3 + Connector
# ============================================================
def _discover_last_layers(m) -> Tuple[Tuple[int,int], Tuple[int,int]]:
    v_ids, t_ids = set(), set()

    v_patterns = [
        r"vision_model\.encoder\.layers\.(\d+)\.",
        r"encoder\.layers\.(\d+)\.",
        r"visual\.blocks\.(\d+)\.",
    ]
    t_patterns = [
        r"language_model\.model\.layers\.(\d+)\.",
        r"model\.layers\.(\d+)\.",
        r"transformer\.h\.(\d+)\.",
    ]

    for name, _ in m.named_parameters():
        if ("vision" in name.lower()) or ("vision_tower" in name.lower()) or ("visual" in name.lower()):
            for pat in v_patterns:
                mm = re.search(pat, name)
                if mm:
                    v_ids.add(int(mm.group(1)))
                    break

        if ("language_model" in name) or ("model.layers" in name) or ("transformer.h" in name):
            for pat in t_patterns:
                mm = re.search(pat, name)
                if mm:
                    t_ids.add(int(mm.group(1)))
                    break

    v_max = max(v_ids) if v_ids else -1
    t_max = max(t_ids) if t_ids else -1
    v_start = max(0, v_max - (VISION_LAST_N - 1)) if v_max >= 0 else 0
    t_start = max(0, t_max - (TEXT_LAST_M   - 1)) if t_max >= 0 else 0
    return (v_start, v_max), (t_start, t_max)

def enable_grads_vlast3_tlast3_connector(m) -> List[str]:
    for _, p in m.named_parameters():
        p.requires_grad_(False)

    (v_start, v_max), (t_start, t_max) = _discover_last_layers(m)
    print(f"  ℹ️ Vision layers inferred: [{v_start}..{v_max}] (last {VISION_LAST_N})")
    print(f"  ℹ️ Text   layers inferred: [{t_start}..{t_max}] (last {TEXT_LAST_M})")

    selected = []
    connector_re = re.compile(r"(multi_modal_projector|mm_projector|vision_proj|projector|vision_to_text|visual_projection|vision_adapter)", re.IGNORECASE)

    v_layer_res = [
        re.compile(r"vision_model\.encoder\.layers\.(\d+)\."),
        re.compile(r"encoder\.layers\.(\d+)\."),
        re.compile(r"visual\.blocks\.(\d+)\."),
    ]
    t_layer_res = [
        re.compile(r"language_model\.model\.layers\.(\d+)\."),
        re.compile(r"model\.layers\.(\d+)\."),
        re.compile(r"transformer\.h\.(\d+)\."),
    ]

    for name, p in m.named_parameters():
        if p.ndim < 2:
            continue

        enable = False

        # Connector
        if connector_re.search(name):
            enable = True

        # Vision last-N
        if not enable and (("vision" in name.lower()) or ("vision_tower" in name.lower()) or ("visual" in name.lower())):
            vid = None
            for rr in v_layer_res:
                mm = rr.search(name)
                if mm:
                    vid = int(mm.group(1)); break
            if vid is not None and v_max >= 0 and vid >= v_start:
                if any(k in name for k in ["attn", "self_attn", "mlp", "proj", "q_proj", "k_proj", "v_proj", "o_proj", "fc", "dense", "linear"]):
                    enable = True

        # Text last-M
        if not enable:
            tid = None
            for rr in t_layer_res:
                mm = rr.search(name)
                if mm:
                    tid = int(mm.group(1)); break
            if tid is not None and t_max >= 0 and tid >= t_start:
                if any(k in name for k in [
                    "self_attn", "attn", "q_proj", "k_proj", "v_proj", "o_proj",
                    "mlp", "gate_proj", "up_proj", "down_proj",
                    "wq", "wk", "wv", "wo"
                ]):
                    enable = True

        if enable:
            p.requires_grad_(True)
            selected.append(name)

    return selected

collect_names = enable_grads_vlast3_tlast3_connector(model)
trainable_numel = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Enabled params: {len(collect_names)} | trainable numel={trainable_numel}")
if len(collect_names) == 0:
    raise RuntimeError("❌ No params enabled for gradients")

# ============================================================
# 7) GradNorm score (per-sample)
# ============================================================
@torch.no_grad()
def _noop():
    return

def gradnorm_score_one(batch: Dict[str, torch.Tensor]) -> Optional[float]:
    """
    Compute GradNorm score for one sample:
      score = sqrt(sum ||grad||_2^2 over selected params)
    """
    model.train()
    model.zero_grad(set_to_none=True)

    try:
        with torch.cuda.amp.autocast(dtype=dtype):
            out = model(**batch)
            loss = out.loss

        # backward requires grad enabled
        loss.backward()

        # accumulate squared L2
        sq_sum = 0.0
        for p in model.parameters():
            if p.requires_grad and (p.grad is not None):
                g = p.grad
                if not torch.isfinite(g).all():
                    continue
                sq_sum += float(g.float().pow(2).sum().item())

        model.zero_grad(set_to_none=True)
        model.eval()

        if sq_sum <= 0.0:
            return 0.0
        return float(np.sqrt(sq_sum))

    except RuntimeError:
        model.zero_grad(set_to_none=True)
        model.eval()
        torch.cuda.empty_cache()
        return None
    except Exception:
        model.zero_grad(set_to_none=True)
        model.eval()
        return None

# ============================================================
# 8) AUC + TPR@5%FPR (no sklearn)
# ============================================================
def auc_from_scores(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """
    AUC via rank statistic (handles ties).
    """
    y_true = y_true.astype(np.int32)
    y_score = y_score.astype(np.float64)
    n_pos = int(y_true.sum())
    n_neg = int((1 - y_true).sum())
    if n_pos == 0 or n_neg == 0:
        return 0.5

    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(len(y_score), dtype=np.float64) + 1.0

    # tie correction
    sorted_scores = y_score[order]
    i = 0
    while i < len(sorted_scores):
        j = i
        while j + 1 < len(sorted_scores) and sorted_scores[j + 1] == sorted_scores[i]:
            j += 1
        if j > i:
            avg_rank = 0.5 * (ranks[order[i]] + ranks[order[j]])
            ranks[order[i:j+1]] = avg_rank
        i = j + 1

    sum_ranks_pos = float(ranks[y_true == 1].sum())
    auc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc)

def tpr_at_fpr(y_true: np.ndarray, y_score: np.ndarray, target_fpr: float = 0.05) -> float:
    """
    Compute TPR at target FPR by sweeping thresholds on scores.
    Higher score => more likely member.
    """
    y_true = y_true.astype(np.int32)
    y_score = y_score.astype(np.float64)

    pos = (y_true == 1)
    neg = (y_true == 0)
    n_pos = int(pos.sum())
    n_neg = int(neg.sum())
    if n_pos == 0 or n_neg == 0:
        return 0.0

    # thresholds: unique scores descending
    thr_list = np.unique(y_score)[::-1]
    best_tpr = 0.0

    # For speed: sort scores
    for thr in thr_list:
        pred = (y_score >= thr)
        fp = int((pred & neg).sum())
        tp = int((pred & pos).sum())
        fpr = fp / max(1, n_neg)
        tpr = tp / max(1, n_pos)
        if fpr <= target_fpr:
            if tpr > best_tpr:
                best_tpr = tpr

    return float(best_tpr)

# ============================================================
# 9) Read items
# ============================================================
def read_member_item(obj: Dict[str,Any]) -> Optional[Tuple[Image.Image, str]]:
    img = read_member_image(obj["pmc_tar_url"], obj["image_file_path"])
    if img is None:
        return None
    cap = get_caption(obj)
    return img, cap

def read_nonmember_item(obj: Dict[str,Any]) -> Optional[Tuple[Image.Image, str]]:
    img = read_nonmember_image(obj["local_image_path"])
    if img is None:
        return None
    cap = get_caption(obj)
    return img, cap

# ============================================================
# 10) Score all samples
# ============================================================
def score_dataset(objs: List[Dict[str,Any]], kind: str) -> np.ndarray:
    scores = []
    miss = 0
    fail = 0

    for obj in tqdm(objs, desc=f"GradNorm({kind})"):
        got = read_member_item(obj) if kind == "M" else read_nonmember_item(obj)
        if got is None:
            miss += 1
            scores.append(0.0)
            continue
        img, cap = got

        batch = build_batch(img, cap)
        s = gradnorm_score_one(batch)
        if s is None:
            fail += 1
            scores.append(0.0)
        else:
            scores.append(float(s))

        # keep memory stable
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"  GradNorm({kind}): n={len(objs)} miss_img={miss} backward_fail={fail}")
    return np.array(scores, dtype=np.float64)

print("\n" + "="*90)
print("🚀 Run GradNorm MIA on LLaVA-Med v1.5 (Member vs Nonmember)")
print("="*90)

M_scores = score_dataset(members_all, kind="M")
N_scores = score_dataset(nonmem_all, kind="N")

# ============================================================
# 11) Evaluate
# ============================================================
y = np.array([1]*len(M_scores) + [0]*len(N_scores), dtype=np.int32)
s = np.concatenate([M_scores, N_scores], axis=0)

auc = auc_from_scores(y, s)
tpr5 = tpr_at_fpr(y, s, target_fpr=0.05)

print("\n" + "="*90)
print("📊 Results (GradNorm)")
print("="*90)
print(f"AUC(M vs N)        = {auc:.4f}")
print(f"TPR@5%FPR (M vs N)  = {tpr5*100:.2f}%")

# Optional: save
OUT_DIR = "/content/drive/MyDrive/llava_med_mia_attack_results"
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, "llava_med_gradnorm_m_vs_n.json")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump({
        "config": {
            "seed": SEED,
            "model": MODEL_ID,
            "members_json": MEMBERS_JSON,
            "nonmembers_json": NONMEM_JSON,
            "num_members": int(NUM_MEMBERS),
            "num_nonmembers": int(NUM_NONMEMBERS),
            "vision_last_n": int(VISION_LAST_N),
            "text_last_m": int(TEXT_LAST_M),
            "dtype": str(dtype),
            "prompt": USER_PROMPT,
        },
        "results": {
            "auc": float(auc),
            "tpr_at_5fpr": float(tpr5),
            "mean_score_member": float(np.mean(M_scores)),
            "mean_score_nonmember": float(np.mean(N_scores)),
        }
    }, f, ensure_ascii=False, indent=2)

print(f"\n💾 Saved: {out_path}")

print("\n✅ Done.")


## NA-PDD

In [ ]:
# ============================================================
# NA-PDD baseline (position-level) for LLaVA-Med v1.5  ✅FIXED+FAST
# - EXACT baseline behavior kept:
#   activated_neurons[layer][pos] -> list of neuron_idx (implicit via coords)
#   pattern build: iterate positions & Counter.update(pos_neurons)
#   scoring: union across positions (baseline predict), ratio = avg_mem/avg_non
#
# - Speedups WITHOUT changing algorithm:
#   (1) batch forward (no gen)
#   (2) hook only last V/T layers (3~5 configurable)
#   (3) tar handle + index cache (no per-image tar open/scan)
#   (4) inference_mode + autocast
#   (5) sparse extraction via nonzero()
#   (6) ✅ CRITICAL FIX: vectorized scoring (no Python loops over coords)
#
# - RAM-safe:
#   (a) Calib: store coords (needed for exact position-level Counter.update)
#   (b) Test: stream batches, DO NOT store activations; score on-the-fly
# ============================================================

import os, io, json, tarfile, gc, random, re, urllib.parse, time
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
from collections import Counter

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

import torch
import torch.nn as nn

# ============================================================
# 0) Config
# ============================================================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DRIVE_BASE   = "/content/drive/MyDrive/Medical/llava_med_pairs"
MEMBERS_JSON = f"{DRIVE_BASE}/members_1k.json"
NONMEM_JSON  = f"{DRIVE_BASE}/nonmembers_roco_1k.json"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

CALIB_M = 200
CALIB_N = 200
TEST_M  = 800
TEST_N  = 800

# NA-PDD hyperparams (paper baseline style)
ACTIVATION_THRESHOLD = 0.0
ALPHA_DOM = 1.5
TOP_LAYERS = 10
RATIO_THRESHOLD_FOR_ACC = 1.0

# speed knobs (baseline logic unchanged)
BATCH_SIZE = 16

# IMPORTANT for LLaVA processor:
# DO NOT use truncation='max_length' here. We'll soft-truncate captions only.
MAX_TEXT_TOKENS_SOFT = 2048  # safe upper bound; we truncate caption to keep within this budget

HOOK_VISION_LAST = 3   # you can set 3~5
HOOK_TEXT_LAST   = 3   # you can set 3~5

USER_PROMPT = "<image>\nDescribe the medical image."

PMC_TAR_CACHE_DIR = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)

# silence
PRINT_TAR_INDEX_LOGS = False  # must be False per your request

print("============================================================")
print(f"✅ Config: TOTAL(M/N)={NUM_MEMBERS}/{NUM_NONMEMBERS}, CALIB(M/N)={CALIB_M}/{CALIB_N}, TEST(M/N)={TEST_M}/{TEST_N}")
print(f"📋 NA-PDD(orig): thr={ACTIVATION_THRESHOLD}, alpha={ALPHA_DOM}, top_layers={TOP_LAYERS}, batch={BATCH_SIZE}")
print(f"📋 Hook last(V/T)={HOOK_VISION_LAST}/{HOOK_TEXT_LAST}")
print("============================================================")

# ============================================================
# 1) Device / dtype
# ============================================================
assert torch.cuda.is_available(), "❌ Need GPU"
device = torch.device("cuda:0")
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

cap_major = torch.cuda.get_device_capability(0)[0]
dtype = torch.bfloat16 if cap_major >= 8 else torch.float16
print(f"✅ Device={device}, dtype={dtype}, cc={torch.cuda.get_device_capability(0)}")

# ============================================================
# 2) Load model
# ============================================================
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
print(f"⏳ Loading: {MODEL_ID}")

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map=None,
    trust_remote_code=True,
).to(device)

try:
    model.gradient_checkpointing_disable()
except Exception:
    pass
if hasattr(model, "config"):
    model.config.use_cache = False
model.eval()

# processor fallbacks (safe)
if getattr(processor, "patch_size", None) is None:
    ip = getattr(processor, "image_processor", None)
    ps = getattr(ip, "patch_size", None)
    processor.patch_size = ps if isinstance(ps, int) else 14
if getattr(processor, "num_additional_image_tokens", None) is None:
    processor.num_additional_image_tokens = 0

def _move_vision_to_device(m):
    vt = getattr(m, "vision_tower", None)
    if vt is None:
        return
    try:
        vt.to(device)
    except Exception:
        inner = getattr(vt, "vision_tower", None)
        if inner is not None:
            inner.to(device)
_move_vision_to_device(model)

print("✅ Model ready")

# soft max length
MODEL_MAX_LEN = None
try:
    MODEL_MAX_LEN = int(getattr(getattr(model, "config", None), "max_position_embeddings", 0)) or None
except Exception:
    MODEL_MAX_LEN = None
print(f"✅ MAX_TEXT_TOKENS_SOFT = {MAX_TEXT_TOKENS_SOFT} (MODEL_MAX_LEN={MODEL_MAX_LEN})")

# ============================================================
# 3) Data load + split
# ============================================================
def load_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

assert os.path.exists(MEMBERS_JSON), f"❌ Missing: {MEMBERS_JSON}"
assert os.path.exists(NONMEM_JSON),  f"❌ Missing: {NONMEM_JSON}"

members_all = load_json_list(MEMBERS_JSON)
nonmem_all  = load_json_list(NONMEM_JSON)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmem_all)

members_all = members_all[:NUM_MEMBERS]
nonmem_all  = nonmem_all[:NUM_NONMEMBERS]

M_calib = members_all[:CALIB_M]
M_test  = members_all[CALIB_M:CALIB_M+TEST_M]
N_calib = nonmem_all[:CALIB_N]
N_test  = nonmem_all[CALIB_N:CALIB_N+TEST_N]

print(f"\n✅ Data split:")
print(f"   Calib: M={len(M_calib)} + N={len(N_calib)}")
print(f"   Test : M={len(M_test)} + N={len(N_test)}")

def get_caption(obj: Dict[str,Any]) -> str:
    cap = obj.get("caption", "")
    if isinstance(cap, list) and len(cap) > 0:
        cap = cap[0]
    return str(cap).strip() if cap is not None else ""

# ============================================================
# 4) TAR image loader (handle+index cache, silent)
# ============================================================
import requests

def _download_to(path: str, url: str, timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = Path(urllib.parse.urlparse(tar_url).path).name
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if (not os.path.exists(local)) or os.path.getsize(local) == 0:
        _download_to(local, tar_url)
    return local

def _norm_path(p: str) -> str:
    p = p.replace("\\", "/")
    p = re.sub(r"^\./+", "", p)
    p = re.sub(r"/{2,}", "/", p)
    return p.lower().strip()

_SIZE_SUFFIX_RE = re.compile(r"(_lrg|_sm|_s|-large|-small)$", re.IGNORECASE)
def _strip_size_suffix(basename: str) -> str:
    name, ext = os.path.splitext(basename)
    name = _SIZE_SUFFIX_RE.sub("", name)
    return name + ext

_TAR_HANDLE: Dict[str, tarfile.TarFile] = {}
_TAR_INDEX:  Dict[str, Dict[str, tarfile.TarInfo]] = {}
_TAR_BUILT:  Dict[str, bool] = {}

def _get_tar_handle_and_index(local_tar: str):
    if local_tar not in _TAR_HANDLE:
        t0 = time.time()
        t = tarfile.open(local_tar, "r:gz")
        _TAR_HANDLE[local_tar] = t
        idx = {_norm_path(ti.name): ti for ti in t.getmembers()}
        _TAR_INDEX[local_tar] = idx
        _TAR_BUILT[local_tar] = True
        if PRINT_TAR_INDEX_LOGS:
            print(f"  🧱 Built tar index: {Path(local_tar).name}  members={len(idx)}  time={time.time()-t0:.2f}s")
    return _TAR_HANDLE[local_tar], _TAR_INDEX[local_tar]

def _best_member_match_fast(local_tar: str, wanted_path: str) -> tarfile.TarInfo:
    _, idx = _get_tar_handle_and_index(local_tar)
    w = _norm_path(wanted_path)

    if w in idx:
        return idx[w]

    suffix_hits = [(nm, ti) for nm, ti in idx.items() if nm.endswith(w)]
    if suffix_hits:
        suffix_hits.sort(key=lambda x: len(x[0]), reverse=True)
        return suffix_hits[0][1]

    want_base = Path(w).name
    base_hits = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base]
    if base_hits:
        return base_hits[0][1]

    want_base2 = _strip_size_suffix(want_base)
    base_hits2 = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base2]
    if base_hits2:
        return base_hits2[0][1]

    # longest common suffix fallback
    def _lcsuf_len(a: str, b: str) -> int:
        a, b = a[::-1], b[::-1]
        n = min(len(a), len(b)); i = 0
        while i < n and a[i] == b[i]:
            i += 1
        return i
    scored = [(_lcsuf_len(nm, w), ti) for nm, ti in idx.items()]
    scored.sort(key=lambda x: x[0], reverse=True)
    if scored and scored[0][0] > 0:
        return scored[0][1]

    raise FileNotFoundError(f"Not found in tar: {wanted_path}")

def read_member_image(pmc_tar_url: str, file_in_tar: str) -> Optional[Image.Image]:
    try:
        local_tar = _ensure_tar_local(pmc_tar_url)
        tar, _ = _get_tar_handle_and_index(local_tar)
        ti = _best_member_match_fast(local_tar, file_in_tar)
        with tar.extractfile(ti) as f:
            if f is None:
                return None
            img = Image.open(io.BytesIO(f.read())).convert("RGB")
        return img
    except (UnidentifiedImageError, OSError, FileNotFoundError, tarfile.TarError):
        return None
    except Exception:
        return None

def read_nonmember_image(local_path: str) -> Optional[Image.Image]:
    try:
        return Image.open(local_path).convert("RGB")
    except Exception:
        return None

def read_member_item(obj: Dict[str,Any]) -> Optional[Tuple[Image.Image, str]]:
    img = read_member_image(obj["pmc_tar_url"], obj["image_file_path"])
    if img is None:
        return None
    return img, get_caption(obj)

def read_nonmember_item(obj: Dict[str,Any]) -> Optional[Tuple[Image.Image, str]]:
    img = read_nonmember_image(obj["local_image_path"])
    if img is None:
        return None
    return img, get_caption(obj)

# ============================================================
# 5) Build inputs (NO truncation='max_length'; soft truncate caption only)
# ============================================================
USER_MSGS       = [{"role": "user", "content": USER_PROMPT}]
USER_PROMPT_STR = processor.apply_chat_template(USER_MSGS, add_generation_prompt=True)

# prefix ids length for prompt-only part (not strictly needed here, but kept for consistency)
PREFIX_IDS = processor.tokenizer(USER_PROMPT_STR, return_tensors="pt").input_ids
PREFIX_LEN = int(PREFIX_IDS.size(1))

def _soft_truncate_caption(cap: str, budget_tokens: int) -> str:
    """
    Avoid LLaVA 'image token mismatch' error caused by truncation='max_length'.
    We keep prompt intact; truncate caption token-length if needed.
    """
    if not cap:
        return ""
    # tokenize caption only
    ids = processor.tokenizer(cap, add_special_tokens=False).input_ids
    if len(ids) <= budget_tokens:
        return cap
    ids = ids[:budget_tokens]
    return processor.tokenizer.decode(ids, skip_special_tokens=True)

def build_batch(images: List[Image.Image], captions: List[str]) -> Dict[str, torch.Tensor]:
    # soft truncate caption so total tokens stay under MAX_TEXT_TOKENS_SOFT
    # (approx: prompt tokens + caption tokens)
    prompt_len = PREFIX_LEN
    cap_budget = max(32, int(MAX_TEXT_TOKENS_SOFT - prompt_len - 8))
    caps2 = [_soft_truncate_caption(str(c), cap_budget) for c in captions]
    texts = [USER_PROMPT_STR + c for c in caps2]

    # KEY: no truncation='max_length'
    enc = processor(text=texts, images=images, return_tensors="pt", padding=True)
    enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
    return enc

# ============================================================
# 6) Discover architecture (fixed paths)
# ============================================================
def _get_by_path(root: Any, path: str) -> Optional[Any]:
    cur = root
    for seg in path.split("."):
        if not hasattr(cur, seg):
            return None
        cur = getattr(cur, seg)
    return cur

vision_blocks = _get_by_path(model, "model.vision_tower.vision_model.encoder.layers")
text_layers   = _get_by_path(model, "model.language_model.layers")  # ✅ this is the correct fallback path for this model

print("\n============================================================")
print("🔍 Model architecture check")
print("vision_blocks:", type(vision_blocks).__name__ if vision_blocks is not None else None,
      f"n={len(vision_blocks)} path=model.vision_tower.vision_model.encoder.layers" if vision_blocks is not None else "")
print("text_layers  :", type(text_layers).__name__ if text_layers is not None else None,
      f"n={len(text_layers)} path=model.language_model.layers" if text_layers is not None else "")
print("============================================================")

if vision_blocks is None or text_layers is None:
    print("\n[Hints] named_children:")
    for n, c in model.named_children():
        print(" -", n, "=>", type(c).__name__)
    raise RuntimeError("❌ Cannot locate vision/text layers. Paste the hints and I’ll adapt precisely.")

v_total = len(vision_blocks)
t_total = len(text_layers)
v_start = max(0, v_total - HOOK_VISION_LAST)
t_start = max(0, t_total - HOOK_TEXT_LAST)
print(f"✅ Hook plan: vision[{v_start}..{v_total-1}] text[{t_start}..{t_total-1}]")

# ============================================================
# 7) Hooks: capture MLP outputs (baseline threshold on activations)
# ============================================================
_hook_buffers: Dict[str, torch.Tensor] = {}
_hook_handles: List[Any] = []
global_H_by_layer: Dict[str, int] = {}   # for fast scoring masks

def _make_hook(tag: str):
    def hook(module, inp, out):
        out0 = out[0] if isinstance(out, (tuple, list)) else out
        if torch.is_tensor(out0):
            _hook_buffers[tag] = out0.detach()
    return hook

def register_hooks(vision_blocks, text_layers, v_start, t_start):
    handles, tags = [], []

    # vision blocks: usually have .mlp
    for i in range(v_start, len(vision_blocks)):
        blk = vision_blocks[i]
        if hasattr(blk, "mlp"):
            tag = f"vision::L{i}::mlp"
            handles.append(blk.mlp.register_forward_hook(_make_hook(tag)))
            tags.append(tag)

    # text layers: Mistral decoder layers have .mlp
    for i in range(t_start, len(text_layers)):
        lyr = text_layers[i]
        if hasattr(lyr, "mlp"):
            tag = f"text::L{i}::mlp"
            handles.append(lyr.mlp.register_forward_hook(_make_hook(tag)))
            tags.append(tag)

    return handles, tags

_hook_handles, hook_tags = register_hooks(vision_blocks, text_layers, v_start, t_start)
print(f"✅ Registered hooks: {len(_hook_handles)}  examples={hook_tags[:min(12,len(hook_tags))]}")
if len(_hook_handles) == 0:
    raise RuntimeError("❌ No hooks registered; layer has no `.mlp`")

# ============================================================
# 8) Sparse coords extraction (position-level, baseline-compatible)
#     Return coords per layer: ndarray[K,3] with columns (b,s,h)
# ============================================================
@torch.inference_mode()
def forward_and_get_sparse_coords(images: List[Image.Image], captions: List[str]) -> Tuple[Dict[str,np.ndarray], int, Dict[str,Tuple[int,int,int]]]:
    """
    Returns:
      coords_by_layer[tag] = np.ndarray [K,3] of (b,s,h)
      B = batch size
      layer_shapes[tag] = (B,S,H) or (B,H,1) etc (for debugging)
    """
    _hook_buffers.clear()
    batch = build_batch(images, captions)

    t0 = time.time()
    with torch.amp.autocast("cuda", dtype=dtype):
        _ = model(**batch)
    fw_time = time.time() - t0

    coords_by_layer: Dict[str, np.ndarray] = {}
    layer_shapes: Dict[str, Tuple[int,int,int]] = {}

    for tag, buf in _hook_buffers.items():
        if buf.ndim == 3:
            # [B,S,H]
            B, S, H = buf.shape
            global_H_by_layer[tag] = int(H)
            layer_shapes[tag] = (int(B), int(S), int(H))

            mask = (buf > float(ACTIVATION_THRESHOLD))
            coords = torch.nonzero(mask, as_tuple=False)
            coords_by_layer[tag] = coords.detach().cpu().numpy().astype(np.int32, copy=False)

        elif buf.ndim == 2:
            # treat as [B,1,H] baseline-compatible (pos=0)
            B, H = buf.shape
            global_H_by_layer[tag] = int(H)
            layer_shapes[tag] = (int(B), 1, int(H))

            mask = (buf > float(ACTIVATION_THRESHOLD))
            coords = torch.nonzero(mask, as_tuple=False)  # (b,h)
            if coords.numel() == 0:
                coords_by_layer[tag] = np.zeros((0,3), dtype=np.int32)
            else:
                b = coords[:, 0:1]
                h = coords[:, 1:2]
                s = torch.zeros_like(b)
                coords3 = torch.cat([b, s, h], dim=1)
                coords_by_layer[tag] = coords3.detach().cpu().numpy().astype(np.int32, copy=False)
        else:
            # flatten to [B,1,Hflat]
            B = buf.shape[0]
            flat = buf.reshape(B, -1)
            H = flat.shape[1]
            global_H_by_layer[tag] = int(H)
            layer_shapes[tag] = (int(B), 1, int(H))

            mask = (flat > float(ACTIVATION_THRESHOLD))
            coords = torch.nonzero(mask, as_tuple=False)  # (b,hflat)
            if coords.numel() == 0:
                coords_by_layer[tag] = np.zeros((0,3), dtype=np.int32)
            else:
                b = coords[:, 0:1]
                h = coords[:, 1:2]
                s = torch.zeros_like(b)
                coords3 = torch.cat([b, s, h], dim=1)
                coords_by_layer[tag] = coords3.detach().cpu().numpy().astype(np.int32, copy=False)

    # cleanup
    del batch
    return coords_by_layer, len(images), layer_shapes, fw_time

# ============================================================
# 9) Calib collection (store coords ONLY for baseline pattern build)
# ============================================================
def load_batch_items(objs: List[Dict[str,Any]], kind: str) -> Tuple[List[Image.Image], List[str], int]:
    images, caps = [], []
    used = 0
    for obj in objs:
        got = read_member_item(obj) if kind == "M" else read_nonmember_item(obj)
        if got is None:
            continue
        img, cap = got
        if (img is None) or (not cap):
            continue
        images.append(img)
        caps.append(cap)
        used += 1
    return images, caps, used

def collect_calib_coords(samples: List[Dict[str,Any]], kind: str, batch_size: int):
    """
    Returns list of dict per sample:
      {'label': 1/0, 'coords': {layer: np[K,3] subset for that sample} }
    NOTE: To keep baseline EXACT for pattern build, we must preserve positions.
    We'll store per-sample coords grouped by layer.
    """
    out = []
    pbar = tqdm(range(0, len(samples), batch_size), desc=f"Collect({kind})", leave=True)
    for bi in pbar:
        chunk = samples[bi:bi+batch_size]
        images, caps, used = load_batch_items(chunk, kind)
        if used == 0:
            continue

        coords_by_layer, B, _, fw_time = forward_and_get_sparse_coords(images, caps)

        # split coords into per-sample
        # coords are (b,s,h), group by b
        per_sample = [{"label": 1 if kind=="M" else 0, "coords": {}} for _ in range(B)]
        for layer, coords in coords_by_layer.items():
            if coords.shape[0] == 0:
                for b in range(B):
                    per_sample[b]["coords"][layer] = np.zeros((0,2), dtype=np.int32)  # (s,h)
                continue
            # coords: (b,s,h)
            b_ids = coords[:, 0]
            for b in range(B):
                sel = (b_ids == b)
                if np.any(sel):
                    sh = coords[sel][:, 1:3]  # (s,h)
                    per_sample[b]["coords"][layer] = sh
                else:
                    per_sample[b]["coords"][layer] = np.zeros((0,2), dtype=np.int32)

        out.extend(per_sample)

        # RAM-safe cleanup
        del images, caps, coords_by_layer, per_sample
        if (bi // batch_size) % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

        pbar.set_postfix_str(f"used={used}, fw={fw_time:.2f}s")

    print(f"✅ Collect({kind}): got={len(out)}/{len(samples)}")
    return out

# ============================================================
# 10) Baseline pattern analysis (position-wise Counter.update)
#     We DO NOT build activated_neurons dict; we update counters directly from coords.
# ============================================================
def build_reference_patterns_from_coords(calib_mem, calib_non):
    """
    Inputs:
      calib_mem: list of {'coords': {layer: (K,2) of (s,h)} }
      calib_non: same
    Output: reference_patterns[layer] with member/nonmember dominant sets
    """
    # counters per layer
    m_counts: Dict[str, Counter] = {}
    n_counts: Dict[str, Counter] = {}

    def _update_counts(container: Dict[str, Counter], sample_list: List[Dict[str,Any]]):
        for s in sample_list:
            coords_map = s["coords"]
            for layer, sh in coords_map.items():
                if layer not in container:
                    container[layer] = Counter()
                if sh.shape[0] == 0:
                    continue
                # EXACT baseline: iterate positions and Counter.update(pos_neurons)
                # We have (s,h). We group by s to mimic pos_neurons lists.
                # Grouping with numpy:
                s_ids = sh[:, 0]
                h_ids = sh[:, 1]
                order = np.argsort(s_ids, kind="mergesort")
                s_sorted = s_ids[order]
                h_sorted = h_ids[order]
                # scan runs of equal s
                i = 0
                while i < len(s_sorted):
                    j = i
                    while j + 1 < len(s_sorted) and s_sorted[j+1] == s_sorted[i]:
                        j += 1
                    container[layer].update(h_sorted[i:j+1].tolist())
                    i = j + 1

    _update_counts(m_counts, calib_mem)
    _update_counts(n_counts, calib_non)

    # build reference patterns
    reference_patterns: Dict[str, Any] = {}
    all_layers = sorted(set(list(m_counts.keys()) + list(n_counts.keys())))
    mN = max(1, len(calib_mem))
    nN = max(1, len(calib_non))

    for layer in all_layers:
        mc = m_counts.get(layer, Counter())
        nc = n_counts.get(layer, Counter())

        m_freq = {int(k): v / mN for k, v in mc.items()}
        n_freq = {int(k): v / nN for k, v in nc.items()}

        m_dom = {}
        n_dom = {}

        for neuron, f in m_freq.items():
            if neuron not in n_freq or f > n_freq[neuron] * ALPHA_DOM:
                m_dom[neuron] = f
        for neuron, f in n_freq.items():
            if neuron not in m_freq or f > m_freq[neuron] * ALPHA_DOM:
                n_dom[neuron] = f

        reference_patterns[layer] = {
            "member_dominant": m_dom,
            "nonmember_dominant": n_dom,
            "member_freq": m_freq,
            "nonmember_freq": n_freq,
        }
    return reference_patterns

def calculate_layer_scores(reference_patterns: Dict[str,Any]) -> Dict[str,float]:
    return {layer: float(len(d["member_dominant"]) - len(d["nonmember_dominant"]))
            for layer, d in reference_patterns.items()}

def select_top_layers(layer_scores: Dict[str,float], top_n: int) -> List[str]:
    sorted_layers = sorted(layer_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    return [k for k,_ in sorted_layers[:min(top_n, len(sorted_layers))]]

# ============================================================
# 11) FAST scoring (baseline-equivalent union across positions)
#     Vectorized: unique(b,h) then bincount overlaps using boolean dom masks
# ============================================================
def build_dom_masks(reference_patterns: Dict[str,Any], layers: List[str], H_by_layer: Dict[str,int]):
    m_dom_mask = {}
    n_dom_mask = {}
    for layer in layers:
        H = int(H_by_layer[layer])
        ref = reference_patterns[layer]
        m_dom = list(ref["member_dominant"].keys())
        n_dom = list(ref["nonmember_dominant"].keys())

        mm = np.zeros(H, dtype=np.bool_)
        nm = np.zeros(H, dtype=np.bool_)
        if len(m_dom) > 0:
            mm[np.array(m_dom, dtype=np.int64)] = True
        if len(n_dom) > 0:
            nm[np.array(n_dom, dtype=np.int64)] = True

        m_dom_mask[layer] = mm
        n_dom_mask[layer] = nm
    return m_dom_mask, n_dom_mask

def score_batch_from_coords_fast(coords_by_layer: Dict[str,np.ndarray],
                                 B: int,
                                 discriminative_layers: List[str],
                                 reference_patterns: Dict[str,Any],
                                 m_dom_mask: Dict[str,np.ndarray],
                                 n_dom_mask: Dict[str,np.ndarray]) -> List[float]:
    total_m = np.zeros(B, dtype=np.float64)
    total_n = np.zeros(B, dtype=np.float64)
    layers_counted = np.zeros(B, dtype=np.int32)

    for layer in discriminative_layers:
        coords = coords_by_layer.get(layer, None)
        if coords is None or coords.shape[0] == 0:
            continue

        # union across positions => unique(b,h)
        bh = coords[:, [0, 2]].astype(np.int32, copy=False)
        bh_u = np.unique(bh, axis=0)
        b = bh_u[:, 0].astype(np.int32, copy=False)
        h = bh_u[:, 1].astype(np.int32, copy=False)

        has_any = (np.bincount(b, minlength=B) > 0)

        mm = m_dom_mask[layer]
        nm = n_dom_mask[layer]
        m_hits = mm[h].astype(np.int32, copy=False)
        n_hits = nm[h].astype(np.int32, copy=False)

        m_overlap = np.bincount(b, weights=m_hits, minlength=B).astype(np.float64, copy=False)
        n_overlap = np.bincount(b, weights=n_hits, minlength=B).astype(np.float64, copy=False)

        m_dom_sz = len(reference_patterns[layer]["member_dominant"])
        n_dom_sz = len(reference_patterns[layer]["nonmember_dominant"])

        if m_dom_sz > 0:
            total_m += (m_overlap / float(m_dom_sz)) * has_any
        if n_dom_sz > 0:
            total_n += (n_overlap / float(n_dom_sz)) * has_any

        layers_counted += has_any.astype(np.int32)

    ratios = np.zeros(B, dtype=np.float64)
    valid = layers_counted > 0
    avg_m = np.zeros(B, dtype=np.float64)
    avg_n = np.zeros(B, dtype=np.float64)
    avg_m[valid] = total_m[valid] / layers_counted[valid]
    avg_n[valid] = total_n[valid] / layers_counted[valid]
    ratios[~valid] = 0.0
    ratios[valid] = np.where(avg_n[valid] == 0.0, 1e9, (avg_m[valid] / avg_n[valid]))
    ratios[~np.isfinite(ratios)] = 1e9
    return ratios.tolist()

# ============================================================
# 12) Metrics (no sklearn): AUC + TPR@5%FPR + ACC@ratio>=1
# ============================================================
def auc_from_scores(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = y_true.astype(np.int32)
    y_score = y_score.astype(np.float64)
    n_pos = int(y_true.sum())
    n_neg = int((1 - y_true).sum())
    if n_pos == 0 or n_neg == 0:
        return 0.5

    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(len(y_score), dtype=np.float64) + 1.0

    sorted_scores = y_score[order]
    i = 0
    while i < len(sorted_scores):
        j = i
        while j + 1 < len(sorted_scores) and sorted_scores[j + 1] == sorted_scores[i]:
            j += 1
        if j > i:
            avg_rank = 0.5 * (ranks[order[i]] + ranks[order[j]])
            ranks[order[i:j+1]] = avg_rank
        i = j + 1

    sum_ranks_pos = float(ranks[y_true == 1].sum())
    auc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc)

def tpr_at_fpr(y_true: np.ndarray, y_score: np.ndarray, target_fpr: float = 0.05) -> float:
    y_true = y_true.astype(np.int32)
    y_score = y_score.astype(np.float64)
    pos = (y_true == 1)
    neg = (y_true == 0)
    n_pos = int(pos.sum())
    n_neg = int(neg.sum())
    if n_pos == 0 or n_neg == 0:
        return 0.0

    thr_list = np.unique(y_score)[::-1]
    best_tpr = 0.0
    for thr in thr_list:
        pred = (y_score >= thr)
        fp = int((pred & neg).sum())
        tp = int((pred & pos).sum())
        fpr = fp / max(1, n_neg)
        tpr = tp / max(1, n_pos)
        if fpr <= target_fpr and tpr > best_tpr:
            best_tpr = tpr
    return float(best_tpr)

# ============================================================
# 13) Test streaming (no storing activations)
# ============================================================
def test_stream_scores(samples: List[Dict[str,Any]], kind: str, batch_size: int,
                       discriminative_layers: List[str],
                       reference_patterns: Dict[str,Any],
                       m_dom_mask: Dict[str,np.ndarray],
                       n_dom_mask: Dict[str,np.ndarray]) -> Tuple[List[float], List[int]]:
    scores = []
    labels = []
    pbar = tqdm(range(0, len(samples), batch_size), desc=f"Test({kind})", leave=True)
    for bi in pbar:
        t_load0 = time.time()
        chunk = samples[bi:bi+batch_size]
        images, caps, used = load_batch_items(chunk, kind)
        load_time = time.time() - t_load0

        if used == 0:
            continue

        coords_by_layer, B, _, fw_time = forward_and_get_sparse_coords(images, caps)

        batch_scores = score_batch_from_coords_fast(
            coords_by_layer, B,
            discriminative_layers,
            reference_patterns,
            m_dom_mask, n_dom_mask
        )

        scores.extend(batch_scores)
        labels.extend([1 if kind == "M" else 0] * B)

        # keep only essential progress info (batch bar + load/fw)
        pbar.set_postfix_str(f"used={used}, load={load_time:.2f}s, fw={fw_time:.2f}s")

        del images, caps, coords_by_layer
        if (bi // batch_size) % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    return scores, labels

# ============================================================
# 14) Run pipeline
# ============================================================
print("\n" + "="*90)
print("🚀 Collect activations (Calib)  [store coords only]")
print("="*90)

calib_M = collect_calib_coords(M_calib, kind="M", batch_size=BATCH_SIZE)
calib_N = collect_calib_coords(N_calib, kind="N", batch_size=BATCH_SIZE)

print("\n" + "="*90)
print("🚀 Build reference patterns + select layers")
print("="*90)

reference_patterns = build_reference_patterns_from_coords(calib_M, calib_N)
layer_scores = calculate_layer_scores(reference_patterns)
discriminative_layers = select_top_layers(layer_scores, top_n=TOP_LAYERS)

print(f"✅ Selected layers ({len(discriminative_layers)}):")
for l in discriminative_layers:
    print("  -", l, "score=", layer_scores.get(l, 0.0))

if len(discriminative_layers) == 0:
    raise RuntimeError("❌ No discriminative layers selected (check activations / threshold / alpha).")

# Build dom masks for fast scoring (requires H per layer; gathered during calib forward)
m_dom_mask, n_dom_mask = build_dom_masks(reference_patterns, discriminative_layers, global_H_by_layer)

print("\n" + "="*90)
print("🚀 Test streaming (no activation storage)")
print("="*90)

scores_M, labels_M = test_stream_scores(
    M_test, kind="M", batch_size=BATCH_SIZE,
    discriminative_layers=discriminative_layers,
    reference_patterns=reference_patterns,
    m_dom_mask=m_dom_mask,
    n_dom_mask=n_dom_mask
)

scores_N, labels_N = test_stream_scores(
    N_test, kind="N", batch_size=BATCH_SIZE,
    discriminative_layers=discriminative_layers,
    reference_patterns=reference_patterns,
    m_dom_mask=m_dom_mask,
    n_dom_mask=n_dom_mask
)

scores = np.array(scores_M + scores_N, dtype=np.float64)
labels = np.array(labels_M + labels_N, dtype=np.int32)

auc = auc_from_scores(labels, scores)
tpr5 = tpr_at_fpr(labels, scores, target_fpr=0.05)
pred = (scores >= RATIO_THRESHOLD_FOR_ACC).astype(np.int32)
acc = float((pred == labels).mean()) if labels.size > 0 else 0.0

print("\n" + "="*90)
print("📊 NA-PDD Results (baseline position-level, FAST scoring)")
print("="*90)
print(f"AUC               = {auc:.4f}")
print(f"TPR@5%FPR         = {tpr5*100:.2f}%")
print(f"Accuracy(ratio>=1)= {acc:.4f}")
print(f"n_test_M          = {int((labels==1).sum())}")
print(f"n_test_N          = {int((labels==0).sum())}")

# ============================================================
# 15) Save results
# ============================================================
OUT_DIR = "/content/drive/MyDrive/llava_med_mia_attack_results"
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, "llava_med_na_pdd_baseline_poslevel_m_vs_n_fastscore.json")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump({
        "config": {
            "seed": SEED,
            "model": MODEL_ID,
            "members_json": MEMBERS_JSON,
            "nonmembers_json": NONMEM_JSON,
            "num_members": int(NUM_MEMBERS),
            "num_nonmembers": int(NUM_NONMEMBERS),
            "calib_m": int(CALIB_M),
            "calib_n": int(CALIB_N),
            "test_m": int(TEST_M),
            "test_n": int(TEST_N),
            "activation_threshold": float(ACTIVATION_THRESHOLD),
            "alpha_dom": float(ALPHA_DOM),
            "top_layers": int(TOP_LAYERS),
            "batch_size": int(BATCH_SIZE),
            "hook_vision_last": int(HOOK_VISION_LAST),
            "hook_text_last": int(HOOK_TEXT_LAST),
            "max_text_tokens_soft": int(MAX_TEXT_TOKENS_SOFT),
            "ratio_threshold_for_acc": float(RATIO_THRESHOLD_FOR_ACC),
        },
        "selected_layers": discriminative_layers,
        "layer_scores": {k: float(v) for k, v in layer_scores.items()},
        "results": {
            "auc": float(auc),
            "tpr_at_5fpr": float(tpr5),
            "acc_ratio_ge_1": float(acc),
        }
    }, f, ensure_ascii=False, indent=2)

print(f"\n💾 Saved: {out_path}")

# ============================================================
# 16) Cleanup hooks + tar handles
# ============================================================
for h in _hook_handles:
    try:
        h.remove()
    except Exception:
        pass

for t in list(_TAR_HANDLE.values()):
    try:
        t.close()
    except Exception:
        pass

print("\n✅ Done.")


## Loss MIA

In [ ]:
# ===============================================
# Loss-based MIA for LLaVA-Med (image+caption)
#  - Input  : image + radiology prompt
#  - Target : caption（仅计算 assistant 段的 NLL）
#  - Score  : -loss 作为“成员分数”
# ===============================================

import os, io, json, math, tarfile, random, shutil, warnings
from pathlib import Path
from typing import Dict, Any, Optional

import numpy as np
import torch
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)

warnings.filterwarnings("ignore")

# ---------------------------
# 0) 路径与超参（沿用你上一步的输出）
# ---------------------------
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

# 可先小样本冒烟（如 100/100），再切到 1000/1000
NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

# 下载/缓存目录（members 的远端 *.tar.gz 会按需下载并本地缓存）
PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

# ---------------------------
# 1) CUDA 与模型
# ---------------------------
assert torch.cuda.is_available(), "未检测到 CUDA，请在运行时中启用 GPU（T4/A100/H100）"
device = "cuda:0"
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32  = True
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print(f"✅ Using {device}, dtype={dtype}")

from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
print("\n⏳ 加载模型与处理器 ...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=dtype,             # ≥16bit（bf16 优先，否则 fp16）
    device_map=None,         # 单卡更方便做 loss/MIA 或后续反传
    trust_remote_code=True,
).to(device)
if hasattr(model, "config"):
    model.config.use_cache = False
model.eval()

# 个别镜像处理器可能缺字段，补上（不影响权重或前向）
if getattr(processor, "patch_size", None) is None:
    ip = getattr(processor, "image_processor", None)
    ps = getattr(ip, "patch_size", None)
    processor.patch_size = ps if isinstance(ps, int) else 14   # 常见 ViT-L/14
if getattr(processor, "num_additional_image_tokens", None) is None:
    processor.num_additional_image_tokens = 0

print("✅ 模型加载完成")

# ---------------------------
# 2) 读取上一步包装好的数据
# ---------------------------
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all    = load_json_list(MEMBERS_JSON)      # [{'pair_id','caption','pmc_tar_url','image_file_path',...}, ...]
nonmembers_all = load_json_list(NONMEMBERS_JSON)   # [{'local_image_path','caption'}, ...]
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)

members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 样本就绪：members={len(members)} | non-members={len(nonmembers)}")

# ---------------------------
# 3) Prompt 设计（贴近 LLaVA-Med 训练形态）
# ---------------------------
# 与 Stage-1/2「图像→caption」一致：给出影像 + 指令“请描述该医学影像”
USER_PROMPT = "<image>\nDescribe the medical image."

def truncate_text_by_tokens(text: str, max_tokens: int = 256) -> str:
    if not text:
        return ""
    toks = processor.tokenizer(
        text, truncation=True, max_length=max_tokens, add_special_tokens=False
    )
    return processor.tokenizer.decode(toks["input_ids"], skip_special_tokens=True)

def build_batch(image: Image.Image, caption: str):
    """
    只在 assistant 段计算 loss：
      user: <image>\nDescribe the medical image.
      assistant: <caption>
    """
    caption_short = truncate_text_by_tokens(caption, max_tokens=256)
    user_msgs   = [{"role": "user", "content": USER_PROMPT}]
    user_prompt = processor.apply_chat_template(user_msgs, add_generation_prompt=True)
    full_text   = user_prompt + caption_short

    # 只监督 assistant 段：前缀位置的 label 置 -100
    prefix_ids = processor.tokenizer(user_prompt, return_tensors="pt").input_ids
    enc = processor(text=full_text, images=[image], return_tensors="pt")
    labels = enc["input_ids"].clone()
    labels[:, :prefix_ids.size(1)] = -100

    enc = {k: v.to(device) for k, v in enc.items()}
    enc["labels"] = labels.to(device)
    return enc

# ---------------------------
# 4) 图像读取工具（members: tar 内按需取一图；nonmembers: 本地路径）
# ---------------------------
import requests

def _download_to(path: str, url: str, timeout=60):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = tar_url.strip().split("/")[-1]
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if not os.path.exists(local) or os.path.getsize(local) == 0:
        _download_to(local, tar_url, timeout=90)
    return local

def _extract_one_from_tar(local_tar: str, file_in_tar: str) -> str:
    target_norm = file_in_tar.replace("\\", "/")
    with tarfile.open(local_tar, "r:gz") as tar:
        member = None
        for ti in tar.getmembers():
            if ti.name.replace("\\", "/") == target_norm:
                member = ti
                break
        if member is None:
            # 简单兜底：扩展名/大小写差异
            cands = [target_norm,
                     target_norm.replace(".jpg",".JPG"),
                     target_norm.replace(".jpg",".png"),
                     target_norm.replace(".JPG",".jpg")]
            for ti in tar.getmembers():
                if ti.name.replace("\\","/") in cands:
                    member = ti
                    break
        if member is None:
            raise FileNotFoundError(f"{file_in_tar} 不在 TAR 包中：{local_tar}")

        Path(PMC_EXTRACT_TMP_DIR).mkdir(parents=True, exist_ok=True)
        out_path = os.path.join(PMC_EXTRACT_TMP_DIR, Path(member.name).name)
        with tar.extractfile(member) as src, open(out_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    return out_path

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    tar_url = sample.get("pmc_tar_url")
    path_in = sample.get("image_file_path")
    if not tar_url or not path_in:
        return None
    local_tar = _ensure_tar_local(tar_url)
    local_img = _extract_one_from_tar(local_tar, path_in)
    return Image.open(local_img).convert("RGB")

def load_nonmember_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if not p or not os.path.exists(p):
        return None
    return Image.open(p).convert("RGB")

# ---------------------------
# 5) 单样本 loss & 批量收集
# ---------------------------
FAIL = {"image":0, "encode":0, "model":0, "other":0}

@torch.inference_mode()
def per_sample_loss(image: Image.Image, caption: str) -> Optional[float]:
    try:
        batch = build_batch(image, caption)
    except Exception:
        FAIL["encode"] += 1
        return None

    try:
        if dtype == torch.bfloat16:
            ctx = torch.cuda.amp.autocast(dtype=torch.bfloat16)
        elif dtype == torch.float16:
            ctx = torch.cuda.amp.autocast(dtype=torch.float16)
        else:
            ctx = torch.cpu.amp.autocast(enabled=False)

        with ctx:
            out = model(**batch)
            loss = out.loss
        return float(loss.item())
    except Exception:
        FAIL["model"] += 1
        return None

def collect_losses(samples, source: str) -> list:
    losses = []
    loader = load_member_image if source == "member" else load_nonmember_image
    for i in tqdm(range(len(samples)), desc=f"{source}"):
        s = samples[i]
        cap = str(s.get("caption","")).strip()
        if not cap:
            FAIL["other"] += 1
            continue
        try:
            img = loader(s)
            if img is None:
                FAIL["image"] += 1
                continue
        except (UnidentifiedImageError, FileNotFoundError, tarfile.ReadError):
            FAIL["image"] += 1
            continue
        except Exception:
            FAIL["image"] += 1
            continue

        l = per_sample_loss(img, cap)
        if l is not None and math.isfinite(l):
            losses.append(l)
    return losses

# ---------------------------
# 6) 计算 loss & 做 MIA 指标
# ---------------------------
print("\n⏳ 计算 members loss …（取决于远端 TAR 下载/解压速度）")
member_losses = collect_losses(members, source="member")

print("⏳ 计算 non-members loss …")
nonmember_losses = collect_losses(nonmembers, source="nonmember")

print(f"\n📊 有效样本：members={len(member_losses)} / {len(members)} | non-members={len(nonmember_losses)} / {len(nonmembers)}")
print(f"❗ 失败统计：{FAIL}")

if len(member_losses)==0 or len(nonmember_losses)==0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

# 分数：-loss（loss 越小越像“见过”，分数越大）
y_true = np.array([1]*len(member_losses) + [0]*len(nonmember_losses))
scores = np.array([-x for x in member_losses] + [-x for x in nonmember_losses])

# ---- 基础：AUC + Youden J（与你原逻辑一致）----
auc = roc_auc_score(y_true, scores)
fpr, tpr, thr = roc_curve(y_true, scores)
best_idx = np.argmax(tpr - fpr)      # Youden J
best_thr = thr[best_idx]
y_pred = (scores >= best_thr).astype(int)
acc = accuracy_score(y_true, y_pred)
cm  = confusion_matrix(y_true, y_pred)

def stats(arr, name):
    arr = np.array(arr)
    print(f"{name}: n={len(arr)}, mean={arr.mean():.4f}, median={np.median(arr):.4f}, "
          f"std={arr.std():.4f}, min={arr.min():.4f}, max={arr.max():.4f}")

print("\n====== Loss-MIA 结果 ======")
print(f"AUC = {auc:.4f}")
print(f"Best threshold (Youden J) = {best_thr:.4f}")
print(f"Accuracy = {acc:.4f}")
print("Confusion matrix [[TN FP][FN TP]]：")
print(cm)
print()
stats(member_losses,   "members loss")
stats(nonmember_losses,"non-members loss")

# ---- 新增：@5% FPR 的 TPR / Precision / Recall / F1 / Acc ----
# 阈值 = 非成员分数的 95% 分位（分数越大越像成员）
non_scores = np.array([-x for x in nonmember_losses])
thr_5 = float(np.quantile(non_scores, 0.95))
y_pred_5 = (scores >= thr_5).astype(int)

cm_5 = confusion_matrix(y_true, y_pred_5)
tn, fp, fn, tp = cm_5.ravel()
fpr_ach = fp / (fp + tn + 1e-12)
tpr_ach = tp / (tp + fn + 1e-12)
prec_5, rec_5, f1_5, _ = precision_recall_fscore_support(
    y_true, y_pred_5, average="binary", zero_division=0
)
acc_5 = accuracy_score(y_true, y_pred_5)

print("\n------ Metrics @ 5% FPR (target) ------")
print(f"Threshold            = {thr_5:.6f}")
print(f"Actual FPR           = {fpr_ach*100:.2f}%")
print(f"TPR (Recall)         = {tpr_ach*100:.2f}%")
print(f"Precision            = {prec_5*100:.2f}%")
print(f"F1                   = {f1_5:.4f}")
print(f"Accuracy             = {acc_5*100:.2f}%")
print("Confusion [[TN FP][FN TP]]：")
print(cm_5)


##Entropy

In [ ]:
# ================= Negative Entropy MIA (black-box, processor-aligned, PATCHED) =================
# - Fixes: processor.patch_size / num_additional_image_tokens = None
# - Uses TWO processor(text+images) calls (prefix & full) so <image> expands correctly
# - Masks labels before prefix_len; score = negative mean entropy on assistant tokens
# - Added metrics: AUC + metrics at 5% FPR (TPR/Precision/Recall/F1/Accuracy)

import os, json, tarfile, random, shutil, warnings
from pathlib import Path
from typing import Dict, Any, Optional, List
import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix, precision_recall_fscore_support
import requests

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Paths & seed
# ---------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

# ---------------------------
# 1) Model
# ---------------------------
from transformers import AutoProcessor, LlavaForConditionalGeneration

assert torch.cuda.is_available(), "需要 GPU"
device = "cuda:0"; torch.cuda.set_device(0)
# 若还不稳，可把 dtype 改为 torch.float32 先验证
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print(f"✅ Device: {device}, dtype={dtype}")

MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
print(f"\n⏳ 加载模型: {MODEL_ID}")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map=None,
    trust_remote_code=True,
).to(device)
if hasattr(model, "config"):
    model.config.use_cache = False
model.eval()

# ---- Patch the processor to avoid None patch_size / additional tokens ----
def _patch_llava_processor(proc, mdl):
    # image token 占位（多数模板用 "<image>"）
    if getattr(proc, "image_token", None) is None:
        proc.image_token = "<image>"

    # 1) patch_size
    ps = getattr(proc, "patch_size", None)
    # 尝试从 image_processor 取
    if ps is None:
        ip = getattr(proc, "image_processor", None)
        if ip is not None:
            ps_ip = getattr(ip, "patch_size", None)
            if isinstance(ps_ip, dict):
                ps_ip = ps_ip.get("height") or ps_ip.get("width")
            if isinstance(ps_ip, int):
                ps = ps_ip
    # 再尝试从模型视觉配置取
    if ps is None:
        vt = getattr(mdl, "vision_tower", None)
        if vt is not None and getattr(vt, "config", None) is not None:
            ps_cfg = getattr(vt.config, "patch_size", None)
            if isinstance(ps_cfg, int):
                ps = ps_cfg
    if ps is None:
        vm = getattr(mdl, "vision_model", None)
        if vm is not None and getattr(vm, "config", None) is not None:
            ps_cfg = getattr(vm.config, "patch_size", None)
            if isinstance(ps_cfg, int):
                ps = ps_cfg
    # 最后兜底（LLaVA 常见 14）
    if ps is None:
        ps = 14
    proc.patch_size = int(ps)

    # 2) num_additional_image_tokens
    if getattr(proc, "num_additional_image_tokens", None) is None:
        proc.num_additional_image_tokens = 0

_patch_llava_processor(processor, model)
print(f"✅ 模型&处理器就绪 | patch_size={processor.patch_size} | add_img_tokens={getattr(processor,'num_additional_image_tokens',0)}")

# ---------------------------
# 2) Chat packing aligned with processor
# ---------------------------
IMG_TOKEN   = getattr(processor, "image_token", "<image>")
USER_PROMPT = "Describe the medical image."

def _messages_user_text():
    return [{"role": "user", "content": f"{IMG_TOKEN}\n{USER_PROMPT}"}]

def _messages_full_text(caption_text: str):
    return [
        {"role": "user", "content": f"{IMG_TOKEN}\n{USER_PROMPT}"},
        {"role": "assistant", "content": caption_text},
    ]

def _truncate_caption(caption: str, keep_tokens: int = 256) -> str:
    tok = processor.tokenizer(
        caption, truncation=True, max_length=keep_tokens, add_special_tokens=False
    )
    return processor.tokenizer.decode(tok["input_ids"], skip_special_tokens=True)

def build_batch_llava_chatloss_v4(image: Image.Image, caption: str) -> Optional[dict]:
    """
    1) chat_template → user_text / full_text（纯字符串）
    2) 两次 processor(text=..., images=[image])，由 processor 负责 <image> 展开为正确的视觉 token 数
    3) prefix_len = len(prefix_ids)
    4) labels = full_ids; labels[:,:prefix_len] = -100
    """
    caption = (caption or "").strip()
    if not caption:
        return None

    for attempt in range(2):  # 回退一次（缩短 caption）
        cap_use = caption if attempt == 0 else _truncate_caption(caption, 256)

        user_text = processor.apply_chat_template(
            _messages_user_text(),
            tokenize=False,
            add_generation_prompt=True,
        )
        full_text = processor.apply_chat_template(
            _messages_full_text(cap_use),
            tokenize=False,
            add_generation_prompt=False,
        )

        # 两次都传 text+images，processor 会按 patch_size 展开 <image>
        enc_prefix = processor(text=[user_text], images=[image], return_tensors="pt")
        enc_full   = processor(text=[full_text], images=[image], return_tensors="pt")

        input_ids      = enc_full["input_ids"]
        attention_mask = enc_full.get("attention_mask", torch.ones_like(input_ids))
        # 视觉张量 key（通常是 pixel_values）
        pixel_key = "pixel_values" if "pixel_values" in enc_full else (
            "images" if "images" in enc_full else None
        )
        if pixel_key is None:
            # 兜底：找第一个非文本 key
            pixel_key = next(k for k in enc_full.keys() if k not in ("input_ids","attention_mask"))

        labels = input_ids.clone()
        prefix_len = int(enc_prefix["input_ids"].size(1))
        if labels.size(1) <= prefix_len:
            # 对不齐则重试一次（更短 caption），再不行放弃该样本
            continue

        labels[:, :prefix_len] = -100

        # 安全搬运到 device + dtype
        pv = enc_full[pixel_key].to(device)
        if pv.dtype.is_floating_point:
            pv = pv.to(dtype)

        batch = {
            "input_ids": input_ids.to(device),
            "attention_mask": attention_mask.to(device),
            "labels": labels.to(device),
            pixel_key: pv,
        }
        # 兼容个别 forward 需要 "images" 键
        if pixel_key == "pixel_values":
            batch["images"] = pv
        return batch

    return None

# ---------------------------
# 3) Data & image I/O
# ---------------------------
def load_json_list(p: str) -> List[Dict[str, Any]]:
    with open(p, "r", encoding="utf-8") as f: return json.load(f)

def _download_to(path: str, url: str, timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for ch in r.iter_content(chunk_size=1<<20):
                if ch: f.write(ch)

def _ensure_tar_local(url: str) -> str:
    local = os.path.join(PMC_TAR_CACHE_DIR, url.strip().split("/")[-1])
    if not os.path.exists(local) or os.path.getsize(local) == 0:
        _download_to(local, url)
    return local

def _extract_one(local_tar: str, file_in_tar: str) -> str:
    target = file_in_tar.replace("\\", "/")
    with tarfile.open(local_tar, "r:gz") as tar:
        mem = None
        for ti in tar.getmembers():
            if ti.name.replace("\\", "/") == target:
                mem = ti; break
        if mem is None:
            for alt in [target.replace(".jpg",".JPG"),
                        target.replace(".jpg",".png"),
                        target.replace(".JPG",".jpg")]:
                for ti in tar.getmembers():
                    if ti.name.replace("\\","/")==alt:
                        mem = ti; break
                if mem: break
        if mem is None:
            raise FileNotFoundError(f"{file_in_tar} not in {local_tar}")
        outp = os.path.join(PMC_EXTRACT_TMP_DIR, Path(mem.name).name)
        with tar.extractfile(mem) as src, open(outp, "wb") as dst: shutil.copyfileobj(src, dst)
    return outp

def load_member_image(s: Dict[str, Any]) -> Optional[Image.Image]:
    url = s.get("pmc_tar_url"); p = s.get("image_file_path")
    if not url or not p: return None
    try:
        imgp = _extract_one(_ensure_tar_local(url), p)
        return Image.open(imgp).convert("RGB")
    except Exception:
        return None

def load_nonmember_image(s: Dict[str, Any]) -> Optional[Image.Image]:
    p = s.get("local_image_path")
    if not p or not os.path.exists(p): return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

# ---------------------------
# 4) Score: negative entropy on assistant tokens
# ---------------------------
@torch.inference_mode()
def neg_entropy_score(image: Image.Image, caption: str) -> Optional[float]:
    batch = build_batch_llava_chatloss_v4(image, caption)
    if batch is None: return None

    out = model(**batch)
    logits = out.logits                 # [1, T, V]
    labels = batch["labels"]            # [1, T]

    # 只统计 assistant 段
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    mask = shift_labels.ne(-100)
    if mask.sum() == 0:
        return None

    valid_logits = shift_logits[mask].float()
    p = torch.softmax(valid_logits, dim=-1)
    ent = -(p * (p.add(1e-9).log())).sum(dim=-1)   # per-token entropy
    return float((-ent.mean()).item())             # 负熵越大越像“成员”

# ---------------------------
# 5) Main
# ---------------------------
print("✅ 模型加载完成")
members    = load_json_list(MEMBERS_JSON)
nonmembers = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members)
random.Random(SEED).shuffle(nonmembers)

# —— 冒烟：前 5 条应能看到 mask_tokens>0 和 loss
print("\n[SMOKE TEST] 5 samples ...")
dbg = {"build_none":0, "fwd_err":0}
for s in members[:5]:
    cap = str(s.get("caption","")).strip()
    if not cap:
        print("empty caption"); continue
    img = load_member_image(s)
    if img is None:
        print("no image"); continue
    b = build_batch_llava_chatloss_v4(img, cap)
    if b is None:
        print("batch=None"); dbg["build_none"] += 1; continue
    print("mask_tokens =", int(b["labels"].ne(-100).sum().item()))
    try:
        with torch.inference_mode():
            out = model(**b)
            print("loss =", float(out.loss.item()))
    except Exception as e:
        print("forward error:", repr(e)); dbg["fwd_err"] += 1
print("SMOKE_FAILS:", dbg)

mem_scores, non_scores = [], []

print("\n⏳ members scoring ...")
for s in tqdm(members, desc="members"):
    cap = str(s.get("caption","")).strip()
    if not cap: continue
    try:
        img = load_member_image(s)
        if img is None: continue
        sc = neg_entropy_score(img, cap)
        if sc is not None: mem_scores.append(sc)
    except Exception as e:
        print(f"Member sample failed. Error: {e}")

print("⏳ nonmembers scoring ...")
for s in tqdm(nonmembers, desc="nonmembers"):
    cap = str(s.get("caption","")).strip()
    if not cap: continue
    try:
        img = load_nonmember_image(s)
        if img is None: continue
        sc = neg_entropy_score(img, cap)
        if sc is not None: non_scores.append(sc)
    except Exception as e:
        print(f"Non-member sample failed. Error: {e}")

print(f"\n有效样本数：mem={len(mem_scores)} non={len(non_scores)}")
if len(mem_scores)==0 or len(non_scores)==0:
    print("⚠️ 无法计算 AUC（分数为空）。若 SMOKE TEST 仍失败：")
    print("  1) 把 dtype 改为 torch.float32 再跑；")
    print("  2) 如果 forward 明确要求 'images' 键，已自动镜像；如仍报错，可只保留 'images'、去掉 'pixel_values'。")
else:
    y_true  = np.array([1]*len(mem_scores) + [0]*len(non_scores))
    y_scores= np.array(mem_scores + non_scores)
    auc = roc_auc_score(y_true, y_scores)
    fpr, tpr, thr = roc_curve(y_true, y_scores); j = np.argmax(tpr - fpr)
    best_thr = thr[j]
    acc = accuracy_score(y_true, (y_scores >= best_thr).astype(int))
    cm  = confusion_matrix(y_true, (y_scores >= best_thr).astype(int))

    print("\n====== NegEntropy-MIA 结果 ======")
    print(f"AUC = {auc:.4f}")
    print(f"Best threshold (Youden J) = {best_thr:.4f}")
    print(f"Accuracy at best threshold = {acc:.4f}")
    print("Confusion matrix [[TN FP][FN TP]]:\n", cm)

    # ---- 新增：@5% FPR 的 TPR / Precision / Recall / F1 / Accuracy ----
    # 方向：分数越大越像“成员”。阈值取非成员分数的 95% 分位。
    non_scores_np = np.array(non_scores, dtype=float)
    thr_5 = float(np.quantile(non_scores_np, 0.95))
    y_pred_5 = (y_scores >= thr_5).astype(int)

    cm_5 = confusion_matrix(y_true, y_pred_5)
    tn, fp, fn, tp = cm_5.ravel()
    fpr_ach = fp / (fp + tn + 1e-12)
    tpr_ach = tp / (tp + fn + 1e-12)
    prec_5, rec_5, f1_5, _ = precision_recall_fscore_support(
        y_true, y_pred_5, average="binary", zero_division=0
    )
    acc_5 = accuracy_score(y_true, y_pred_5)

    print("\n------ Metrics @ 5% FPR (target) ------")
    print(f"Threshold            = {thr_5:.6f}")
    print(f"Actual FPR           = {fpr_ach*100:.2f}%")
    print(f"TPR (Recall)         = {tpr_ach*100:.2f}%")
    print(f"Precision            = {prec_5*100:.2f}%")
    print(f"F1                   = {f1_5:.4f}")
    print(f"Accuracy             = {acc_5*100:.2f}%")
    print("Confusion [[TN FP][FN TP]]：")
    print(cm_5)


## Min-K

In [ ]:
# ================= Min-K MIA (black-box, per-token NLL) =================
# - Score: mean of smallest-K% token NLL, then negated (bigger => more member-like)
# - Added metrics: AUC + metrics at 5% FPR (TPR/Precision/Recall/F1/Accuracy)

import os, json, tarfile, random, shutil, warnings
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import torch, torch.nn.functional as F
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)
import requests

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Paths & seed
# ---------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

# ---------------------------
# 1) Model
# ---------------------------
from transformers import AutoProcessor, LlavaForConditionalGeneration

assert torch.cuda.is_available(), "需要 GPU"
device = "cuda:0"; torch.cuda.set_device(0)
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print(f"✅ Device: {device}, dtype={dtype}")

MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
print(f"\n⏳ 加载模型: {MODEL_ID}")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=dtype, device_map=None, trust_remote_code=True
).to(device)
if hasattr(model, "config"): model.config.use_cache = False
model.eval()

def _patch_llava_processor(proc, mdl):
    if getattr(proc, "image_token", None) is None:
        proc.image_token = "<image>"
    ps = getattr(proc, "patch_size", None)
    if ps is None:
        ip = getattr(proc, "image_processor", None)
        if ip is not None:
            ps_ip = getattr(ip, "patch_size", None)
            if isinstance(ps_ip, dict): ps_ip = ps_ip.get("height") or ps_ip.get("width")
            if isinstance(ps_ip, int): ps = ps_ip
    if ps is None:
        vt = getattr(mdl, "vision_tower", None)
        if vt is not None and getattr(vt, "config", None) is not None:
            ps_cfg = getattr(vt, "config").patch_size if hasattr(vt, "config") else None
            if isinstance(ps_cfg, int): ps = ps_cfg
    if ps is None:
        vm = getattr(mdl, "vision_model", None)
        if vm is not None and getattr(vm, "config", None) is not None:
            ps_cfg = getattr(vm, "config").patch_size if hasattr(vm, "config") else None
            if isinstance(ps_cfg, int): ps = ps_cfg
    if ps is None: ps = 14
    proc.patch_size = int(ps)
    if getattr(proc, "num_additional_image_tokens", None) is None:
        proc.num_additional_image_tokens = 0

_patch_llava_processor(processor, model)
print(f"✅ 处理器修补完成 | patch_size={processor.patch_size}, add_img_tokens={processor.num_additional_image_tokens}")

IMG_TOKEN   = getattr(processor, "image_token", "<image>")
USER_PROMPT = "Describe the medical image."

def _messages_user_text():  # 纯字符串
    return [{"role": "user", "content": f"{IMG_TOKEN}\n{USER_PROMPT}"}]

def _messages_full_text(caption: str):
    return [
        {"role": "user", "content": f"{IMG_TOKEN}\n{USER_PROMPT}"},
        {"role": "assistant", "content": caption},
    ]

def _truncate_caption(caption: str, keep_tokens: int = 256) -> str:
    tok = processor.tokenizer(caption, truncation=True, max_length=keep_tokens, add_special_tokens=False)
    return processor.tokenizer.decode(tok["input_ids"], skip_special_tokens=True)

def build_batch_llava(image: Image.Image, caption: str) -> Optional[dict]:
    caption = (caption or "").strip()
    if not caption: return None
    for attempt in range(2):
        cap_use = caption if attempt == 0 else _truncate_caption(caption, 256)
        user_text = processor.apply_chat_template(_messages_user_text(), tokenize=False, add_generation_prompt=True)
        full_text = processor.apply_chat_template(_messages_full_text(cap_use), tokenize=False, add_generation_prompt=False)

        enc_prefix = processor(text=[user_text], images=[image], return_tensors="pt")
        enc_full   = processor(text=[full_text], images=[image], return_tensors="pt")

        input_ids      = enc_full["input_ids"]
        attention_mask = enc_full.get("attention_mask", torch.ones_like(input_ids))
        pixel_key = "pixel_values" if "pixel_values" in enc_full else ("images" if "images" in enc_full else None)
        if pixel_key is None:
            pixel_key = next(k for k in enc_full.keys() if k not in ("input_ids","attention_mask"))

        labels = input_ids.clone()
        prefix_len = int(enc_prefix["input_ids"].size(1))
        if labels.size(1) <= prefix_len:  # 还不对齐，缩短一次
            continue
        labels[:, :prefix_len] = -100

        pv = enc_full[pixel_key].to(device)
        if pv.dtype.is_floating_point: pv = pv.to(dtype)

        batch = {
            "input_ids": input_ids.to(device),
            "attention_mask": attention_mask.to(device),
            "labels": labels.to(device),
            pixel_key: pv,
        }
        if pixel_key == "pixel_values":  # 兼容某些 forward
            batch["images"] = pv
        return batch
    return None

# ---------------------------
# 2) Data & image I/O
# ---------------------------
def load_json_list(p: str) -> List[Dict[str, Any]]:
    with open(p, "r", encoding="utf-8") as f: return json.load(f)

def _download_to(path: str, url: str, timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for ch in r.iter_content(chunk_size=1<<20):
                if ch: f.write(ch)

def _ensure_tar_local(url: str) -> str:
    local = os.path.join(PMC_TAR_CACHE_DIR, url.strip().split("/")[-1])
    if not os.path.exists(local) or os.path.getsize(local) == 0:
        _download_to(local, url)
    return local

def _extract_one(local_tar: str, file_in_tar: str) -> str:
    target = file_in_tar.replace("\\", "/")
    with tarfile.open(local_tar, "r:gz") as tar:
        mem = None
        for ti in tar.getmembers():
            if ti.name.replace("\\","/")==target: mem = ti; break
        if mem is None:
            for alt in [target.replace(".jpg",".JPG"), target.replace(".jpg",".png"), target.replace(".JPG",".jpg")]:
                for ti in tar.getmembers():
                    if ti.name.replace("\\","/")==alt: mem=ti; break
                if mem: break
        if mem is None: raise FileNotFoundError(f"{file_in_tar} not in {local_tar}")
        outp = os.path.join(PMC_EXTRACT_TMP_DIR, Path(mem.name).name)
        with tar.extractfile(mem) as src, open(outp,"wb") as dst: shutil.copyfileobj(src,dst)
    return outp

def load_member_image(s: Dict[str, Any]) -> Optional[Image.Image]:
    url = s.get("pmc_tar_url"); p = s.get("image_file_path")
    if not url or not p: return None
    try:
        imgp = _extract_one(_ensure_tar_local(url), p)
        return Image.open(imgp).convert("RGB")
    except Exception:
        return None

def load_nonmember_image(s: Dict[str, Any]) -> Optional[Image.Image]:
    p = s.get("local_image_path")
    if not p or not os.path.exists(p): return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

# ---------------------------
# 3) Per-sample Min-K score
# ---------------------------
def min_k_score_from_logits(logits: torch.Tensor, labels: torch.Tensor, k_frac: float = 0.05) -> Optional[float]:
    # logits: [1,T,V] ; labels: [1,T] with -100 on prefix
    L = labels[..., 1:]
    Z = logits[..., :-1, :]
    m = L.ne(-100)
    if m.sum() == 0: return None
    with torch.no_grad():
        lp  = F.log_softmax(Z[m], dim=-1)      # [N,V]
        tgt = L[m].unsqueeze(1)                # [N,1]
        nll = -(lp.gather(1, tgt)).squeeze(1)  # [N]
        k = max(1, int(len(nll) * k_frac))
        # 取最小的 k 个 NLL → 等价于取最大的 k 个 (-NLL)
        vals, _ = torch.topk(-nll, k)
        return float(vals.mean().item())       # 值越大 ⇒ 越像成员

@torch.inference_mode()
def min_k_score(image: Image.Image, caption: str, k_frac=0.05) -> Optional[float]:
    b = build_batch_llava(image, caption)
    if b is None: return None
    out = model(**b)
    return min_k_score_from_logits(out.logits, b["labels"], k_frac=k_frac)

# ---------------------------
# 4) Main
# ---------------------------
members    = load_json_list(MEMBERS_JSON)
nonmembers = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members); random.Random(SEED).shuffle(nonmembers)

mem_scores, non_scores = [], []

print("\n⏳ members scoring (Min-K 5%) ...")
for s in tqdm(members, desc="members"):
    cap = str(s.get("caption","")).strip()
    if not cap: continue
    img = load_member_image(s)
    if img is None: continue
    sc = min_k_score(img, cap, k_frac=0.05)
    if sc is not None: mem_scores.append(sc)

print("⏳ nonmembers scoring (Min-K 5%) ...")
for s in tqdm(nonmembers, desc="nonmembers"):
    cap = str(s.get("caption","")).strip()
    if not cap: continue
    img = load_nonmember_image(s)
    if img is None: continue
    sc = min_k_score(img, cap, k_frac=0.05)
    if sc is not None: non_scores.append(sc)

print(f"\n有效样本数：mem={len(mem_scores)} non={len(non_scores)}")
if len(mem_scores)==0 or len(non_scores)==0:
    print("⚠️ 分数为空，检查图像与caption、或把 dtype=torch.float32 再跑。")
else:
    y = np.array([1]*len(mem_scores) + [0]*len(non_scores))
    scores = np.array(mem_scores + non_scores)

    # --- Overall AUC / J-threshold summary (保持原逻辑) ---
    auc = roc_auc_score(y, scores)
    fpr, tpr, thr = roc_curve(y, scores); j = np.argmax(tpr - fpr)
    acc = accuracy_score(y, (scores >= thr[j]).astype(int))
    cm  = confusion_matrix(y, (scores >= thr[j]).astype(int))
    print("\n====== Min-K MIA 结果 ======")
    print(f"AUC = {auc:.4f}")
    print(f"Best threshold (Youden J) = {thr[j]:.4f}")
    print(f"Accuracy at best threshold = {acc:.4f}")
    print("Confusion matrix [[TN FP][FN TP]]:\n", cm)

    # --- 新增：@5% FPR 的 TPR / Precision / Recall / F1 / Accuracy ---
    # 方向：分数越大越像“成员”。阈值取非成员分数的 95% 分位。
    non_scores_np = np.array(non_scores, dtype=float)
    thr_5 = float(np.quantile(non_scores_np, 0.95))
    y_pred_5 = (scores >= thr_5).astype(int)

    cm_5 = confusion_matrix(y, y_pred_5)
    tn, fp, fn, tp = cm_5.ravel()
    fpr_ach = fp / (fp + tn + 1e-12)
    tpr_ach = tp / (tp + fn + 1e-12)
    prec_5, rec_5, f1_5, _ = precision_recall_fscore_support(
        y, y_pred_5, average="binary", zero_division=0
    )
    acc_5 = accuracy_score(y, y_pred_5)

    print("\n------ Metrics @ 5% FPR (target) ------")
    print(f"Threshold            = {thr_5:.6f}")
    print(f"Actual FPR           = {fpr_ach*100:.2f}%")
    print(f"TPR (Recall)         = {tpr_ach*100:.2f}%")
    print(f"Precision            = {prec_5*100:.2f}%")
    print(f"F1                   = {f1_5:.4f}")
    print(f"Accuracy             = {acc_5*100:.2f}%")
    print("Confusion [[TN FP][FN TP]]：")
    print(cm_5)


## Min-K++

In [ ]:
import os, json, tarfile, random, shutil, warnings
from pathlib import Path
from typing import Dict, Any, Optional, List
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch, torch.nn.functional as F
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)
import requests

warnings.filterwarnings("ignore")

# ================= 配置与模型加载 =================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"
PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

from transformers import AutoProcessor, LlavaForConditionalGeneration
device = "cuda:0"; torch.cuda.set_device(0)
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=dtype, device_map=None, trust_remote_code=True
).to(device)
model.eval()

# (此处省略你原有的 _patch_llava_processor, build_batch_llava 等辅助函数，保持不变)

# ================= 核心修正：Min-K++ (无参考模型自归一化版) =================

def min_kpp_fixed_score(logits: torch.Tensor, labels: torch.Tensor, k_list=(0.01, 0.05, 0.1, 0.2)) -> Optional[float]:
    """
    Min-K++ 思想实现：
    1. 计算每个 token 的 Log-Likelihood (LL)。
    2. 计算该序列 LL 的均值 mu 和标准差 std。
    3. 对 LL 进行标准化：z = (LL - mu) / std。
    4. 取标准化后得分最高（即最不像噪声、最确信）的 k% token 的均值。
    """
    L = labels[..., 1:]; Z = logits[..., :-1, :]
    mask = L.ne(-100)
    if mask.sum() < 5: return None # 序列太短则无效

    with torch.no_grad():
        # 得到所有 label token 的 log_probs
        log_probs_all = F.log_softmax(Z[mask], dim=-1)
        target_log_probs = log_probs_all.gather(1, L[mask].unsqueeze(1)).squeeze(1) # [N]

        # 计算序列自身的统计量
        mu = target_log_probs.mean()
        std = target_log_probs.std() + 1e-8

        # 归一化得分：成员数据在某些 token 上的 LL 会显著高于其平均水平
        norm_scores = (target_log_probs - mu) / std

        results = []
        for f in k_list:
            k = max(1, int(len(norm_scores) * f))
            # 取 norm_scores 最大的 k 个（代表这些 token 的表现远超该样本的平均水平）
            vals, _ = torch.topk(norm_scores, k)
            results.append(vals.mean())

        # 取不同 K 下的最大值作为最终倾向性分数
        return float(torch.stack(results).max().item())



@torch.inference_mode()
def get_score(image: Image.Image, caption: str) -> Optional[float]:
    batch = build_batch_llava(image, caption)
    if batch is None: return None
    outputs = model(**batch)
    return min_kpp_fixed_score(outputs.logits, batch["labels"])

# ================= 运行与评价 =================

# (加载 members 和 nonmembers 逻辑保持不变...)

# 评价逻辑修正
def evaluate_mia(mem_scores, non_scores):
    if len(mem_scores) == 0 or len(non_scores) == 0:
        print("❌ 无效分数")
        return

    y_true = np.array([1] * len(mem_scores) + [0] * len(non_scores))
    all_scores = np.array(mem_scores + non_scores)

    # 1. 整体 AUC
    auc = roc_auc_score(y_true, all_scores)

    # 2. 寻找最佳阈值 (Youden's J)
    fpr, tpr, thresholds = roc_curve(y_true, all_scores)
    idx = np.argmax(tpr - fpr)
    best_thr = thresholds[idx]

    # 3. 目标 5% FPR 下的指标
    # 注意：此时分数越大，越可能是成员
    non_scores_np = np.array(non_scores)
    thr_5 = np.quantile(non_scores_np, 0.95)
    y_pred_5 = (all_scores >= thr_5).astype(int)

    prec_5, rec_5, f1_5, _ = precision_recall_fscore_support(y_true, y_pred_5, average="binary")

    print(f"\n" + "="*30)
    print(f"📊 Min-K++ (Normalized) 结果")
    print(f"AUC: {auc:.4f}")
    print(f"最佳阈值 (J): {best_thr:.4f}")
    print(f"\n--- @ 5% FPR 目标 ---")
    print(f"检测阈值: {thr_5:.4f}")
    print(f"召回率 (TPR): {rec_5*100:.2f}%")
    print(f"精确率 (Precision): {prec_5*100:.2f}%")
    print(f"F1 Score: {f1_5:.4f}")
    print(f"准确率: {accuracy_score(y_true, y_pred_5)*100:.2f}%")
    print(f"混淆矩阵:\n{confusion_matrix(y_true, y_pred_5)}")

## ModRényi

In [ ]:
# ===============================================
# ModRényi* (fused) MIA for LLaVA-Med (image+caption)
#  - Inputs : image + radiology prompt, target caption
#  - Score  : fused ModRényi* (img views + text views)
#  - Metric : TPR / Precision / Recall / F1 @ 5% FPR (+ AUC)
# ===============================================

import os, io, json, math, tarfile, random, shutil, warnings, re
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix, precision_recall_fscore_support

warnings.filterwarnings("ignore")

# ---------------------------
# 0) 路径与超参（与你的 loss 脚本一致）
# ---------------------------
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

# 视图数（你可按需调小以加快跑速）
IMG_VIEWS = 5   # 原图 + 4 种增广
TXT_VIEWS = 3   # 原文 + 2 个轻度扰动

# 下载/缓存目录（members 的远端 *.tar.gz 会按需下载并本地缓存）
PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

# ---------------------------
# 1) CUDA 与模型
# ---------------------------
assert torch.cuda.is_available(), "未检测到 CUDA，请在运行时中启用 GPU（T4/A100/H100）"
device = "cuda:0"
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32  = True
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print(f"✅ Using {device}, dtype={dtype}")

from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
print("\n⏳ 加载模型与处理器 ...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=dtype,             # ≥16bit（bf16 优先，否则 fp16）
    device_map=None,         # 单卡更方便做 loss/MIA 或后续反传
    trust_remote_code=True,
).to(device)
if hasattr(model, "config"):
    model.config.use_cache = False
model.eval()

# 个别镜像处理器可能缺字段，补上（不影响权重或前向）
if getattr(processor, "patch_size", None) is None:
    ip = getattr(processor, "image_processor", None)
    ps = getattr(ip, "patch_size", None)
    processor.patch_size = ps if isinstance(ps, int) else 14
if getattr(processor, "num_additional_image_tokens", None) is None:
    processor.num_additional_image_tokens = 0

print("✅ 模型加载完成")

# ---------------------------
# 2) 读取数据
# ---------------------------
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
members_all    = load_json_list(MEMBERS_JSON)      # [{'pair_id','caption','pmc_tar_url','image_file_path',...}, ...]
nonmembers_all = load_json_list(NONMEMBERS_JSON)   # [{'local_image_path','caption'}, ...]
random.shuffle(members_all)
random.shuffle(nonmembers_all)

members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 样本就绪：members={len(members)} | non-members={len(nonmembers)}")

# ---------------------------
# 3) Prompt 与打包（与训练形态一致）
# ---------------------------
USER_PROMPT = "<image>\nDescribe the medical image."

def truncate_text_by_tokens(text: str, max_tokens: int = 256) -> str:
    if not text:
        return ""
    toks = processor.tokenizer(
        text, truncation=True, max_length=max_tokens, add_special_tokens=False
    )
    return processor.tokenizer.decode(toks["input_ids"], skip_special_tokens=True)

def build_full_text_and_prefix_len(image: Image.Image, caption: str) -> Tuple[Dict[str, torch.Tensor], int]:
    """
    构造 full_text，并返回前缀长度（只 user 段），用于在 assistant 段取 logits。
    """
    cap = truncate_text_by_tokens(caption, max_tokens=256)
    user_msgs   = [{"role": "user", "content": USER_PROMPT}]
    user_prompt = processor.apply_chat_template(user_msgs, add_generation_prompt=True)
    full_text   = user_prompt + cap

    # 计算 prefix 长度
    prefix_ids = processor.tokenizer(user_prompt, return_tensors="pt").input_ids

    # 编码
    enc = processor(text=full_text, images=[image], return_tensors="pt")
    prefix_len = int(prefix_ids.size(1))
    return enc, prefix_len

# ---------------------------
# 4) 远端 TAR 解包 & 本地图像读取
# ---------------------------
import requests

def _download_to(path: str, url: str, timeout=60):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = tar_url.strip().split("/")[-1]
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if not os.path.exists(local) or os.path.getsize(local) == 0:
        _download_to(local, tar_url, timeout=90)
    return local

def _extract_one_from_tar(local_tar: str, file_in_tar: str) -> str:
    target_norm = file_in_tar.replace("\\", "/")
    with tarfile.open(local_tar, "r:gz") as tar:
        member = None
        for ti in tar.getmembers():
            if ti.name.replace("\\", "/") == target_norm:
                member = ti
                break
        if member is None:
            cands = [target_norm,
                     target_norm.replace(".jpg",".JPG"),
                     target_norm.replace(".jpg",".png"),
                     target_norm.replace(".JPG",".jpg")]
            for ti in tar.getmembers():
                if ti.name.replace("\\","/") in cands:
                    member = ti
                    break
        if member is None:
            raise FileNotFoundError(f"{file_in_tar} 不在 TAR 包中：{local_tar}")

        Path(PMC_EXTRACT_TMP_DIR).mkdir(parents=True, exist_ok=True)
        out_path = os.path.join(PMC_EXTRACT_TMP_DIR, Path(member.name).name)
        with tar.extractfile(member) as src, open(out_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    return out_path

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    tar_url = sample.get("pmc_tar_url")
    path_in = sample.get("image_file_path")
    if not tar_url or not path_in:
        return None
    local_tar = _ensure_tar_local(tar_url)
    local_img = _extract_one_from_tar(local_tar, path_in)
    return Image.open(local_img).convert("RGB")

def load_nonmember_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if not p or not os.path.exists(p):
        return None
    return Image.open(p).convert("RGB")

# ---------------------------
# 5) 图像/文本视图构造
# ---------------------------
from torchvision.transforms import (
    RandomResizedCrop, RandomRotation, RandomAffine, ColorJitter, InterpolationMode
)

def make_image_views(pil: Image.Image, want: int = IMG_VIEWS) -> List[Image.Image]:
    """
    原图 + 4 种轻中度增广；超出 want 会裁掉。
    """
    views = [pil]
    # 1) RandomResizedCrop
    views.append(RandomResizedCrop(size=(256, 256), scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC)(pil))
    # 2) Rotation
    views.append(RandomRotation(degrees=30, interpolation=InterpolationMode.BICUBIC, expand=False)(pil))
    # 3) Affine
    views.append(RandomAffine(degrees=20, translate=(0.08, 0.08), scale=(0.9, 1.1), interpolation=InterpolationMode.BICUBIC)(pil))
    # 4) ColorJitter
    views.append(ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25, hue=0.05)(pil))
    return views[:max(1, want)]

def normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def make_text_views(caption: str, want: int = TXT_VIEWS) -> List[str]:
    """
    轻度语义保持：原文 / 全小写 / 空白规整（可再加去尾标点）
    """
    base = str(caption or "").strip()
    cand = [base, base.lower(), normalize_spaces(base)]
    # 去重保序
    out, seen = [], set()
    for c in cand:
        if c not in seen:
            out.append(c); seen.add(c)
    return out[:max(1, want)]

# ---------------------------
# 6) 从 logits 提取 assistant 段的 logP (T,V)
# ---------------------------
@torch.no_grad()
def assistant_logp_matrix(image: Image.Image, caption: str) -> Optional[np.ndarray]:
    """
    返回：只含 assistant token 的 logP 矩阵 [T, V]（T>=1）；
    若样本异常（无 token 等）返回 None
    """
    enc, prefix_len = build_full_text_and_prefix_len(image, caption)
    # 组装 labels 以标出前缀
    labels = enc["input_ids"].clone()
    labels[:, :prefix_len] = -100
    enc = {k: v.to(device) for k, v in enc.items()}
    # 为拿 logits，可不传 labels；但传 labels 也会返回 logits，这里保持一致
    with torch.cuda.amp.autocast(dtype=(torch.bfloat16 if dtype==torch.bfloat16 else torch.float16)):
        out = model(**enc, labels=labels.to(device))
        logits = out.logits  # [1, L, V]

    # next-token 对齐
    shift_logits = logits[..., :-1, :]     # [1, L-1, V]
    shift_labels = labels[...,  1: ]       # [1, L-1]
    mask = shift_labels.ne(-100)[0]        # [L-1]，只要 assistant 段
    if mask.sum().item() == 0:
        return None

    logp = F.log_softmax(shift_logits[0].float(), dim=-1)  # [L-1, V] → float32 稳定
    sel = logp[mask]                                       # [T, V]
    if sel.numel() == 0:
        return None
    return sel.detach().cpu().numpy()

# ---------------------------
# 7) Rényi 熵 & 稳健聚合 & 融合
# ---------------------------
def renyi_entropy_from_logp(logp_tok: np.ndarray, alpha: float) -> float:
    # H_α = 1/(1-α) * logsumexp(α log p)
    a = alpha * logp_tok
    m = np.max(a)
    return float((1.0/(1.0-alpha)) * (m + np.log(np.exp(a - m).sum())))

def renyi_seq_from_logp(logp_seq: np.ndarray, alpha: float) -> float:
    vals = [renyi_entropy_from_logp(logp_seq[t], alpha) for t in range(logp_seq.shape[0])]
    return float(np.mean(vals)) if vals else float("nan")

def robust_aggregate(values: List[float], trim: float = 0.1) -> float:
    arr = np.array([v for v in values if np.isfinite(v)], dtype=float)
    if arr.size == 0:
        return float("nan")
    arr.sort()
    k = int(math.floor(trim * arr.size))
    if k * 2 < arr.size:
        arr = arr[k: arr.size - k]
    return float(np.mean(arr))

def fused_modrenyi_score(image_views_logp: List[Optional[np.ndarray]],
                         text_views_logp:  List[Optional[np.ndarray]],
                         alphas=(0.5, 2.0),
                         length_norm=True) -> float:
    """
    返回融合分数 S_fused；若完全失败返回 NaN
    """
    parts = []  # 收集四个分量（img@0.5、img@2、txt@0.5、txt@2）
    for views in [image_views_logp, text_views_logp]:
        # 过滤 None
        vv = [x for x in views if x is not None]
        if not vv:
            parts.extend([float("nan")] * len(alphas))
            continue
        T = vv[0].shape[0]
        for a in alphas:
            per_view = []
            for lp in vv:
                if lp is None or not np.isfinite(lp).all():
                    continue
                H = renyi_seq_from_logp(lp, a)
                per_view.append(H)
            if not per_view:
                parts.append(float("nan")); continue
            H_bar = robust_aggregate(per_view, trim=0.1)
            if length_norm and T > 0:
                H_bar = H_bar / math.sqrt(T)
            score = -H_bar  # 负熵：越大越像“成员”
            parts.append(score)

    # z-score 融合（对可用分量）
    valid = np.array([x for x in parts if np.isfinite(x)], dtype=float)
    if valid.size == 0:
        return float("nan")
    if valid.size >= 2 and np.std(valid) > 1e-12:
        z = (valid - np.mean(valid)) / np.std(valid)
        return float(np.mean(z))
    else:
        return float(np.mean(valid))

# ---------------------------
# 8) 视图推理适配器
# ---------------------------
def image_views_logp_list(image: Image.Image, caption: str) -> List[Optional[np.ndarray]]:
    views = make_image_views(image, IMG_VIEWS)
    out = []
    for v in views:
        try:
            out.append(assistant_logp_matrix(v, caption))
        except Exception:
            out.append(None)
    return out

def text_views_logp_list(image: Image.Image, caption: str) -> List[Optional[np.ndarray]]:
    texts = make_text_views(caption, TXT_VIEWS)
    out = []
    for t in texts:
        try:
            out.append(assistant_logp_matrix(image, t))
        except Exception:
            out.append(None)
    return out

# ---------------------------
# 9) 5% FPR 阈值与指标
# ---------------------------
def metrics_at_target_fpr(y_true: np.ndarray, scores: np.ndarray, target_fpr: float = 0.05):
    fpr, tpr, thr = roc_curve(y_true, scores)
    # 找到 fpr ≈ 目标的插值阈值
    idx = np.searchsorted(fpr, target_fpr, side="right")
    if idx == 0:
        thr_star = thr[0]; tpr_star = tpr[0]
    elif idx >= len(thr):
        thr_star = thr[-1]; tpr_star = tpr[-1]
    else:
        x0, x1 = fpr[idx-1], fpr[idx]
        y0, y1 = tpr[idx-1], tpr[idx]
        t0, t1 = thr[idx-1], thr[idx]
        w = (target_fpr - x0) / (x1 - x0 + 1e-12)
        tpr_star = y0 + w*(y1 - y0)
        thr_star = t0 + w*(t1 - t0)

    y_pred = (scores >= thr_star).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {
        "thr@5%FPR": float(thr_star),
        "TPR@5%FPR": float(tpr_star),
        "Precision": float(prec),
        "Recall": float(rec),
        "F1": float(f1),
        "ROC-AUC": float(roc_auc_score(y_true, scores)),
    }

# ---------------------------
# 10) 主循环：打分 & 评估
# ---------------------------
def fused_score_for_sample(sample: Dict[str,Any], source: str) -> Optional[float]:
    cap = str(sample.get("caption","")).strip()
    if not cap:
        return None
    try:
        if source == "member":
            img = load_member_image(sample)
        else:
            img = load_nonmember_image(sample)
        if img is None:
            return None
    except Exception:
        return None

    try:
        img_views = image_views_logp_list(img, cap)
        txt_views = text_views_logp_list(img, cap)
        return fused_modrenyi_score(img_views, txt_views, alphas=(0.5, 2.0), length_norm=True)
    except Exception:
        return None

print("\n⏳ 计算 ModRényi*（fused）分数 …")
mem_scores, non_scores = [], []
for s in tqdm(members, desc="members"):
    sc = fused_score_for_sample(s, "member")
    if sc is not None and math.isfinite(sc):
        mem_scores.append(sc)
for s in tqdm(nonmembers, desc="nonmembers"):
    sc = fused_score_for_sample(s, "nonmember")
    if sc is not None and math.isfinite(sc):
        non_scores.append(sc)

print(f"\n📊 有效样本：members={len(mem_scores)} / {len(members)} | non-members={len(non_scores)} / {len(nonmembers)}")
if len(mem_scores)==0 or len(non_scores)==0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

y_true = np.array([1]*len(mem_scores) + [0]*len(non_scores))
scores = np.array(mem_scores + non_scores)

# 5% FPR 处的指标
m = metrics_at_target_fpr(y_true, scores, target_fpr=0.05)

print("\n====== ModRényi* (fused) MIA 结果 ======")
print(f"AUC           = {m['ROC-AUC']:.4f}")
print(f"TPR @5% FPR   = {m['TPR@5%FPR']:.4f}")
print(f"Precision     = {m['Precision']:.4f}")
print(f"Recall        = {m['Recall']:.4f}")
print(f"F1            = {m['F1']:.4f}")
print(f"thr @5% FPR   = {m['thr@5%FPR']:.4f}")


## Similarity

In [ ]:
# ===============================================
# Similarity-MIA (Image + Text) with Fusion — robust
# ===============================================
import os, io, json, math, tarfile, random, shutil, warnings, traceback
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import torch
from PIL import Image, ImageFilter, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix, precision_recall_fscore_support

warnings.filterwarnings("ignore")

# ---------------- 基本参数 ----------------
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"
NUM_MEMBERS, NUM_NONMEMBERS = 1000, 1000
SEED = 42

OUT_DIR  = "/content/sim_mia_outputs"
OUT_CSV  = os.path.join(OUT_DIR, "mia_similarity_fused_1000x1000.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# 图像扰动
GAUSS_RADIUS = 1.0
JPEG_QUALITY = 60

# 文本生成（降载）
MAX_NEW_TOKENS = 24
USER_PROMPT    = "<image>\nDescribe the medical image."
TEXT_FAIL_TOL  = 20      # 连续失败达到阈值后禁用文本通道

# 融合权重
W_IMG, W_TEXT = 0.60, 0.40

# ---------------- 设备 & 模型 ----------------
assert torch.cuda.is_available(), "需要 GPU"
device = "cuda:0"
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32  = True
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print(f"✅ Using {device}, dtype={dtype}")

from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
vlm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=dtype, device_map=None, trust_remote_code=True
).to(device)
if hasattr(vlm, "config"):
    vlm.config.use_cache = False  # 降低 KV cache 压力
vlm.eval()
print("✅ LLaVA-Med 就绪")

# 预加载 OpenCLIP（避免运行中临时加载）
try:
    import open_clip
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch>=2.24.0", "timm"])
    import open_clip
OPENCLIP_MODEL, _, OPENCLIP_TRANS = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai", device=device)
OPENCLIP_MODEL.eval()
print("✅ OpenCLIP (ViT-L/14) 就绪")

# 句向量模型
try:
    from sentence_transformers import SentenceTransformer
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer
txt_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
txt_embedder.eval()

# ---------------- 数据 ----------------
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
members_all    = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)
members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 样本就绪：members={len(members)} | non-members={len(nonmembers)}")

# ---------------- I/O（与前一致） ----------------
import requests
PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

def _download_to(path: str, url: str, timeout=90):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1<<20):
                if chunk: f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = tar_url.strip().split("/")[-1]
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if not os.path.exists(local) or os.path.getsize(local) == 0:
        _download_to(local, tar_url)
    return local

def _extract_one_from_tar(local_tar: str, file_in_tar: str) -> str:
    target_norm = file_in_tar.replace("\\", "/")
    with tarfile.open(local_tar, "r:gz") as tar:
        member = None
        for ti in tar.getmembers():
            if ti.name.replace("\\", "/") == target_norm:
                member = ti; break
        if member is None:
            cands = [target_norm,
                     target_norm.replace(".jpg",".JPG"),
                     target_norm.replace(".jpg",".png"),
                     target_norm.replace(".JPG",".jpg")]
            for ti in tar.getmembers():
                if ti.name.replace("\\","/") in cands:
                    member = ti; break
        if member is None: raise FileNotFoundError(f"{file_in_tar} 不在 TAR 包中：{local_tar}")
        out_path = os.path.join(PMC_EXTRACT_TMP_DIR, Path(member.name).name)
        with tar.extractfile(member) as src, open(out_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    return out_path

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    tar_url = sample.get("pmc_tar_url"); path_in = sample.get("image_file_path")
    if not tar_url or not path_in: return None
    local_tar = _ensure_tar_local(tar_url)
    local_img = _extract_one_from_tar(local_tar, path_in)
    return Image.open(local_img).convert("RGB")

def load_nonmember_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if not p or not os.path.exists(p): return None
    return Image.open(p).convert("RGB")

# ---------------- 扰动与生成 ----------------
def corrupt_image(img: Image.Image) -> Image.Image:
    x = img.filter(ImageFilter.GaussianBlur(radius=GAUSS_RADIUS))
    buf = io.BytesIO(); x.save(buf, format="JPEG", quality=JPEG_QUALITY, optimize=True)
    buf.seek(0); x = Image.open(buf).convert("RGB")
    return x

@torch.inference_mode()
def generate_text(image: Image.Image) -> str:
    msgs = [{"role": "user", "content": USER_PROMPT}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    enc = processor(text=[prompt], images=[image], return_tensors="pt").to(device)
    out_ids = vlm.generate(
        **enc, max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False, temperature=0.0, num_beams=1,
        use_cache=False,  # 关闭缓存，稳一点
    )
    text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    torch.cuda.empty_cache()
    return text.strip()

# ---------------- 两通道得分（解耦 try/except） ----------------
first_err = {"img": None, "txt": None}
@torch.inference_mode()
def image_similarity_score(image: Image.Image) -> Optional[float]:
    try:
        img2 = corrupt_image(image)
        t1 = OPENCLIP_TRANS(image).unsqueeze(0).to(device)
        t2 = OPENCLIP_TRANS(img2).unsqueeze(0).to(device)
        z1 = OPENCLIP_MODEL.encode_image(t1).float(); z1 = z1 / (z1.norm(dim=-1, keepdim=True)+1e-12)
        z2 = OPENCLIP_MODEL.encode_image(t2).float(); z2 = z2 / (z2.norm(dim=-1, keepdim=True)+1e-12)
        sim = torch.cosine_similarity(z1[0], z2[0], dim=-1).clamp(-1, 1).item()
        return float(sim)
    except Exception as e:
        if first_err["img"] is None:
            first_err["img"] = "".join(traceback.format_exception_only(type(e), e)).strip()
        return None

@torch.inference_mode()
def text_similarity_score(image: Image.Image) -> Optional[float]:
    try:
        img2 = corrupt_image(image)
        t1 = generate_text(image)
        t2 = generate_text(img2)
        e1 = txt_embedder.encode([t1], convert_to_tensor=True, normalize_embeddings=True, device=device)[0]
        e2 = txt_embedder.encode([t2], convert_to_tensor=True, normalize_embeddings=True, device=device)[0]
        sim = torch.cosine_similarity(e1, e2, dim=-1).clamp(-1, 1).item()
        return float(sim)
    except Exception as e:
        if first_err["txt"] is None:
            first_err["txt"] = "".join(traceback.format_exception_only(type(e), e)).strip()
        return None

# ---------------- 批量收集 ----------------
FAIL = {"image":0, "img_only_fail":0, "txt_only_fail":0}
def collect_scores(samples, source: str, enable_text: bool = True):
    sims_img, sims_txt = [], []
    loader = load_member_image if source == "member" else load_nonmember_image
    text_fail_streak = 0
    for i in tqdm(range(len(samples)), desc=f"{source}"):
        s = samples[i]
        # 读图
        try:
            img = loader(s)
            if img is None:
                FAIL["image"] += 1
                sims_img.append(None); sims_txt.append(None); continue
        except Exception:
            FAIL["image"] += 1
            sims_img.append(None); sims_txt.append(None); continue

        # 图像通道
        si = image_similarity_score(img)
        if si is None: FAIL["img_only_fail"] += 1
        sims_img.append(si)

        # 文本通道（可禁用）
        st = None
        if enable_text:
            st = text_similarity_score(img)
            if st is None:
                text_fail_streak += 1
            else:
                text_fail_streak = 0
            # 连续失败过多 → 自动禁用文本通道，后续只跑图像
            if text_fail_streak >= TEXT_FAIL_TOL:
                enable_text = False
                print(f"⚠️ 文本通道连续失败 {TEXT_FAIL_TOL} 次，后续将禁用文本通道，只保留图像通道。")
        else:
            st = None

        if st is None: FAIL["txt_only_fail"] += 1
        sims_txt.append(st)

        if (i+1) % 50 == 0:
            torch.cuda.empty_cache()

    return sims_img, sims_txt, enable_text

print("\n⏳ 计算相似度分数（members） …")
mem_img_raw, mem_txt_raw, txt_enabled = collect_scores(members, "member", enable_text=True)
print("⏳ 计算相似度分数（nonmembers） …")
non_img_raw, non_txt_raw, _          = collect_scores(nonmembers, "nonmember", enable_text=txt_enabled)

# 过滤 None
def _nan_filter(xs): return np.array([x for x in xs if (x is not None and np.isfinite(x))], dtype=np.float32)
mem_img = _nan_filter(mem_img_raw);  non_img = _nan_filter(non_img_raw)
mem_txt = _nan_filter(mem_txt_raw);  non_txt = _nan_filter(non_txt_raw)

print(f"\n📊 有效样本（按通道 after filter）：")
print(f"image: mem={len(mem_img)} non={len(non_img)}")
print(f"text : mem={len(mem_txt)} non={len(non_txt)}")
if first_err["img"]: print(f"🧪 首个图像通道异常：{first_err['img']}")
if first_err["txt"]: print(f"🧪 首个文本通道异常：{first_err['txt']}")
print(f"❗ 失败统计：{FAIL}")

# ---------------- 融合与指标（容忍单通道） ----------------
def zscore(arr):
    mu, sd = arr.mean(), arr.std() + 1e-8
    return (arr - mu) / (sd + 1e-8), mu, sd

have_img = (len(non_img) > 10) and (len(mem_img) > 10)
have_txt = (len(non_txt) > 10) and (len(mem_txt) > 10)
if not have_img and not have_txt:
    raise RuntimeError("两条通道都没有有效分数，请先根据上面的首个异常信息排查（通常是文本生成 OOM / 接口不兼容）。")

# 仅保留可用通道
parts = []
if have_img: parts.append(("img", mem_img, non_img, W_IMG))
if have_txt: parts.append(("txt", mem_txt, non_txt, W_TEXT))
if len(parts) == 1:
    # 单通道时将权重设为 1
    parts[0] = (parts[0][0], parts[0][1], parts[0][2], 1.0)

# z-score on non-members per-part，然后融合
fused_mem, fused_non = None, None
details = {}
for name, mem_arr, non_arr, w in parts:
    non_z, mu, sd = zscore(non_arr)
    mem_z = (mem_arr - mu) / (sd + 1e-8)
    details[name] = (mem_arr, non_arr, mem_z, non_z, w)
    if fused_mem is None:
        fused_mem = w * mem_z
        fused_non = w * non_z
    else:
        fused_mem = fused_mem + w * mem_z
        fused_non = fused_non + w * non_z

# AUC
y_true = np.array([1]*len(fused_mem) + [0]*len(fused_non))
scores = np.concatenate([fused_mem, fused_non], axis=0)
auc = roc_auc_score(y_true, scores)

# FPR=5% 阈值
thr_5 = float(np.quantile(fused_non, 0.95))
y_pred = (scores >= thr_5).astype(int)

cm  = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn + 1e-12)
tpr = tp / (tp + fn + 1e-12)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
acc = accuracy_score(y_true, y_pred)

print("\n====== Similarity-MIA（Image+Text 融合，稳健版） ======")
print(f"AUC                = {auc:.4f}")
print(f"Threshold @FPR=5%  = {thr_5:.4f}  (实际 FPR={fpr*100:.2f}%)")
print(f"TPR (Recall)       = {tpr*100:.2f}%")
print(f"Precision          = {prec*100:.2f}%")
print(f"F1                 = {f1:.4f}")
print(f"Accuracy           = {acc*100:.2f}%")
print("Confusion [[TN FP][FN TP]]：")
print(cm)

# ---------------- 保存明细 ----------------
import csv
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    hdr = ["split"]
    if have_img: hdr += ["s_img", "z_img(non-z)"]
    if have_txt: hdr += ["s_txt", "z_txt(non-z)"]
    hdr += ["s_fused"]
    w.writerow(hdr)

    def write_side(tag, mem: bool):
        if mem:
            zs = fused_mem; base = {k:v for k,v in details.items()}
        else:
            zs = fused_non
        n = len(zs)
        for i in range(n):
            row = [ "member" if mem else "nonmember" ]
            if have_img:
                mi, ni, miz, niz, wi = details["img"]
                arr = mi if mem else ni
                z   = miz if mem else niz
                row += [ float(arr[i%len(arr)]), float(z[i%len(z)]) ]
            if have_txt:
                mt, nt, mtz, ntz, wt = details["txt"]
                arr = mt if mem else nt
                z   = mtz if mem else ntz
                row += [ float(arr[i%len(arr)]), float(z[i%len(z)]) ]
            row += [ float(zs[i]) ]
            w.writerow(row)

    write_side("member", True)
    write_side("nonmember", False)

print(f"\n💾 明细已保存：{OUT_CSV}")


## dc-pdd

In [ ]:
import os, pickle, torch
import numpy as np
from collections import Counter
from transformers import AutoProcessor
from datasets import load_dataset
from tqdm import tqdm
from google.colab import drive

# 1. 初始化
drive.mount('/content/drive')
MODEL_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
FREQ_PATH = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_llava_med.pkl"
os.makedirs(os.path.dirname(FREQ_PATH), exist_ok=True)

# 加载 LLaVA-Med Processor 以获取其 Tokenizer
print(f"正在加载 {MODEL_ID} 的分词器...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = processor.tokenizer
vocab_size = tokenizer.vocab_size

# 2. 加载 C4 英文子集 (Parquet 格式流式加载)
print("正在从 allenai/c4 加载英文语料...")
dataset = load_dataset(
    "allenai/c4",
    data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
    split="train",
    streaming=True
)

# 3. 统计词频
counter = Counter()
MAX_SAMPLES = 50000

for i, ex in enumerate(tqdm(dataset, total=MAX_SAMPLES, desc="C4 词频统计")):
    if i >= MAX_SAMPLES: break
    ids = tokenizer.encode(ex['text'], add_special_tokens=False)
    counter.update(ids)

# 4. 计算平滑词频 (Laplace Smoothing)
print("\n正在计算平滑词频...")
total_tokens = sum(counter.values())
freq_smo = np.ones(vocab_size, dtype=np.float64) # 初始值为1

for tid, count in counter.items():
    if tid < vocab_size:
        freq_smo[tid] += count

freq_smo = freq_smo / (total_tokens + vocab_size)

# 5. 保存
with open(FREQ_PATH, "wb") as f:
    pickle.dump(freq_smo, f)

print(f"\n✅ LLaVA-Med 原生频率表已保存至: {FREQ_PATH}")

In [ ]:
from google.colab import drive
import os

# 重新挂载或刷新
drive.mount('/content/drive', force_remount=True)

# 检查文件是否真的在那里
FREQ_PATH = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_llava_med.pkl"
if os.path.exists(FREQ_PATH):
    print("✅ 确认文件已存在，可以继续！")
else:
    print("❌ 文件仍未同步，请稍等 10 秒后再次运行此单元格。")

In [ ]:
import os, io, json, math, pickle, random, re, urllib.parse, zlib, gc
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, LlavaForConditionalGeneration
from sklearn.metrics import roc_auc_score, roc_curve

# ============================================================
# 1) 配置与路径
# ============================================================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DRIVE_BASE   = "/content/drive/MyDrive/Medical/llava_med_pairs"
MEMBERS_JSON = f"{DRIVE_BASE}/members_1k.json"
NONMEM_JSON  = f"{DRIVE_BASE}/nonmembers_roco_1k.json"
FREQ_PATH    = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_llava_med.pkl"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
ALPHA          = 0.1
MODEL_ID       = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"
PMC_TAR_CACHE  = "/content/pmc_tar_cache"

os.makedirs(PMC_TAR_CACHE, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 2) 加载模型与补丁处理 (移植 Zlib 成功的加载逻辑)
# ============================================================
print("🚀 正在加载 LLaVA-Med 模型与 C4 词频表...")

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

# 确保频率表存在
if not os.path.exists(FREQ_PATH):
    raise FileNotFoundError(f"❌ 找不到频率表: {FREQ_PATH}")
with open(FREQ_PATH, "rb") as f:
    freq_smo = pickle.load(f)

# 补丁：对齐 Tokenizer
if getattr(processor, "patch_size", None) is None:
    processor.patch_size = 14
if getattr(processor, "num_additional_image_tokens", None) is None:
    processor.num_additional_image_tokens = 0

# ============================================================
# 3) 图像读取辅助 (移植 Zlib 成功的鲁棒匹配逻辑)
# ============================================================
import tarfile, requests

_TAR_HANDLE = {}
_TAR_INDEX = {}

def _norm_path(p: str) -> str:
    p = p.replace("\\", "/")
    p = re.sub(r"^\./+", "", p)
    return p.lower().strip()

def read_member_image(tar_url, file_path):
    try:
        fname = Path(urllib.parse.urlparse(tar_url).path).name
        local = os.path.join(PMC_TAR_CACHE, fname)
        if not os.path.exists(local):
            r = requests.get(tar_url, stream=True, timeout=60)
            r.raise_for_status()
            with open(local, "wb") as f:
                for chunk in r.iter_content(chunk_size=1<<20): f.write(chunk)

        if local not in _TAR_HANDLE:
            t = tarfile.open(local, "r:gz")
            _TAR_HANDLE[local] = t
            _TAR_INDEX[local] = {_norm_path(ti.name): ti for ti in t.getmembers()}

        tar, idx = _TAR_HANDLE[local], _TAR_INDEX[local]
        w = _norm_path(file_path)
        # 鲁棒匹配：直接匹配 or 后缀匹配
        ti = idx.get(w) or next((ti for nm, ti in idx.items() if nm.endswith(w)), None)

        if ti:
            return Image.open(io.BytesIO(tar.extractfile(ti).read())).convert("RGB")
    except: return None
    return None

# ============================================================
# 4) DC-PDD 核心计算 (修复 Prompt Template 和 Token 对齐)
# ============================================================
USER_PROMPT = "Describe the medical image."
USER_MSGS   = [{"role": "user", "content": f"<image>\n{USER_PROMPT}"}]
# 使用 Zlib 中成功的 apply_chat_template
PROMPT_STR  = processor.apply_chat_template(USER_MSGS, add_generation_prompt=True)

@torch.no_grad()
def compute_dc_pdd_score(image: Image.Image, caption: str) -> Optional[float]:
    full_text = PROMPT_STR + str(caption)

    try:
        # 1. 编码
        enc = processor(text=full_text, images=[image], return_tensors="pt").to(device)
        input_ids = enc["input_ids"]

        # 2. 确定 Prompt 长度以 mask 掉它
        prompt_enc = processor.tokenizer(PROMPT_STR, return_tensors="pt")
        cap_start_idx = prompt_enc.input_ids.shape[1] - 1 # 减去最后一个可能的 EOS 或匹配位

        # 3. 推理
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**enc)
        logits = outputs.logits.float() # [1, seq_len, vocab_size]

        # 4. 提取 Caption 部分的 Logits 和目标 Token
        # 预测位置：logits[:, i, :] 预测的是 input_ids[:, i+1]
        target_ids = input_ids[0, cap_start_idx + 1:]
        predict_logits = logits[0, cap_start_idx : -1, :]

        if target_ids.numel() == 0: return None

        # 5. 计算 DC-PDD：P(token) * log(1/freq)
        probs = torch.softmax(predict_logits, dim=-1)
        token_probs = torch.gather(probs, 1, target_ids.unsqueeze(-1)).squeeze(-1).cpu().numpy()

        target_token_ids = target_ids.cpu().numpy()
        freqs = np.array([freq_smo[tid] if tid < len(freq_smo) else 1e-10 for tid in target_token_ids])

        # 核心信号：概率与词频的加权
        difficulty_signals = token_probs * np.log(1.0 / (freqs + 1e-12))

        # 截断处理并取均值
        return float(np.mean(np.minimum(difficulty_signals, ALPHA)))
    except Exception:
        return None

# ============================================================
# 5) 运行评估
# ============================================================
def run_attack(samples, kind="M"):
    scores = []
    for obj in tqdm(samples, desc=f"DC-PDD({kind})"):
        if kind == "M":
            img = read_member_image(obj.get("pmc_tar_url"), obj.get("image_file_path"))
        else:
            p = obj.get("local_image_path")
            img = Image.open(p).convert("RGB") if p and os.path.exists(p) else None

        cap = str(obj.get("caption", "")).strip()
        if img and cap:
            s = compute_dc_pdd_score(img, cap)
            if s is not None:
                scores.append(s)
            else:
                scores.append(0.0) # 失败降级处理
        else:
            scores.append(0.0)

        # 显存管理
        if len(scores) % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()

    return np.array(scores)

print("\n📂 载入数据集...")
members_all = json.load(open(MEMBERS_JSON))[:NUM_MEMBERS]
nonmem_all  = json.load(open(NONMEM_JSON))[:NUM_NONMEMBERS]

M_raw = run_attack(members_all, "M")
N_raw = run_attack(nonmem_all, "N")

# ============================================================
# 6) 指标计算
# ============================================================
# DC-PDD 逻辑：Member 预测更准 -> Score 更高。直接计算 AUC。
y_true = np.concatenate([np.ones(len(M_raw)), np.zeros(len(N_raw))])
all_scores = np.concatenate([M_raw, N_raw])

if len(all_scores) > 0:
    auc_val = roc_auc_score(y_true, all_scores)
    fpr, tpr, _ = roc_curve(y_true, all_scores)
    tpr_5 = tpr[np.abs(fpr - 0.05).argmin()]

    print("\n" + "="*50)
    print(f"📊 DC-PDD MIA 结果 (LLaVA-Med v1.5):")
    print(f"AUC: {auc_val:.4f}")
    print(f"TPR @ 5% FPR: {tpr_5*100:.2f}%")
    print(f"Mean Score (Member): {np.mean(M_raw):.6f}")
    print(f"Mean Score (Non-Mem): {np.mean(N_raw):.6f}")
    print("="*50)

## M$^4$I

### data split

In [ ]:
# ============================================================
# Step 1) Build LRProbe splits + TAR image loader (LLaVA-Med v1.5)
# Data:
#   - members:    /content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json   (tar-backed)
#   - nonmembers: /content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json (local jpg)
#
# Splits (default, same as你之前LRProbe约定):
#   FT train:   ft200  (members 200)   -> used to full-FT model
#   Dev:        mem100 + nonmem100
#   Test:       mem500 + nonmem500
#
# Saves:
#   /content/splits_llava_med_lrprobe.json
#
# Provides:
#   read_member_image(pmc_tar_url, image_file_path)  # tar cached
# ============================================================

import os, io, re, json, tarfile, random, urllib.parse
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import requests
from PIL import Image, UnidentifiedImageError

# ---------------------------
# 0) Paths / Seed
# ---------------------------
SEED = 42
random.seed(SEED)

PAIR_DIR      = "/content/drive/MyDrive/Medical/llava_med_pairs"
MEMBERS_JSON  = f"{PAIR_DIR}/members_1k.json"
NONMEM_JSON   = f"{PAIR_DIR}/nonmembers_roco_1k.json"

OUT_SPLIT_JSON = "/content/splits_llava_med_lrprobe.json"

assert os.path.exists(MEMBERS_JSON), f"❌ Missing {MEMBERS_JSON}"
assert os.path.exists(NONMEM_JSON),  f"❌ Missing {NONMEM_JSON}"
print("✅ Found:", MEMBERS_JSON)
print("✅ Found:", NONMEM_JSON)

# ---------------------------
# 1) Load json
# ---------------------------
def load_json_list(path: str) -> List[Dict[str,Any]]:
    with open(path, "r", encoding="utf-8") as f:
        x = json.load(f)
    assert isinstance(x, list), "JSON must be a list[dict]"
    return x

members_all = load_json_list(MEMBERS_JSON)
nonmem_all  = load_json_list(NONMEM_JSON)

# basic sanity
print(f"📦 loaded: members={len(members_all)} nonmembers={len(nonmem_all)}")
for k in ["pmc_tar_url","image_file_path","caption"]:
    assert k in members_all[0], f"members item missing key: {k}"
for k in ["local_image_path","caption"]:
    assert k in nonmem_all[0], f"nonmembers item missing key: {k}"

# shuffle (deterministic)
rnd = random.Random(SEED)
rnd.shuffle(members_all)
rnd.shuffle(nonmem_all)

# ---------------------------
# 2) Split policy (can tweak here)
# ---------------------------
FT_MEM   = 200
DEV_MEM  = 100
DEV_NON  = 100
TEST_MEM = 500
TEST_NON = 500

need_mem  = FT_MEM + DEV_MEM + TEST_MEM
need_non  = DEV_NON + TEST_NON

assert len(members_all) >= need_mem, f"❌ members not enough: need {need_mem}, got {len(members_all)}"
assert len(nonmem_all)  >= need_non, f"❌ nonmembers not enough: need {need_non}, got {len(nonmem_all)}"

ft200      = members_all[:FT_MEM]
dev_mem100 = members_all[FT_MEM:FT_MEM+DEV_MEM]
test_mem500= members_all[FT_MEM+DEV_MEM:FT_MEM+DEV_MEM+TEST_MEM]

dev_non100  = nonmem_all[:DEV_NON]
test_non500 = nonmem_all[DEV_NON:DEV_NON+TEST_NON]

# (optional) keep an extra 200 nonmem for debugging / alt-train
train_nonmem200 = nonmem_all[DEV_NON+TEST_NON:DEV_NON+TEST_NON+200] if len(nonmem_all) >= (DEV_NON+TEST_NON+200) else []

print("✅ Split sizes:",
      f"ft200={len(ft200)}",
      f"dev=({len(dev_mem100)},{len(dev_non100)})",
      f"test=({len(test_mem500)},{len(test_non500)})",
      f"train_nonmem200={len(train_nonmem200)}")

# ---------------------------
# 3) Save splits
# ---------------------------
splits = {
    "meta": {
        "seed": SEED,
        "members_json": MEMBERS_JSON,
        "nonmembers_json": NONMEM_JSON,
        "policy": {
            "ft200": FT_MEM,
            "dev_mem100": DEV_MEM,
            "dev_nonmem100": DEV_NON,
            "test_mem500": TEST_MEM,
            "test_nonmem500": TEST_NON
        }
    },
    "ft200": ft200,
    "train_nonmem200": train_nonmem200,
    "dev_mem100": dev_mem100,
    "dev_nonmem100": dev_non100,
    "test_mem500": test_mem500,
    "test_nonmem500": test_non500
}

with open(OUT_SPLIT_JSON, "w", encoding="utf-8") as f:
    json.dump(splits, f, ensure_ascii=False, indent=2)

print("📝 Saved splits ->", OUT_SPLIT_JSON)

# ---------------------------
# 4) TAR image loader (for members)  [REQUIRED for Step2/Step3]
# ---------------------------
PMC_TAR_CACHE_DIR = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)

def _download_to(path: str, url: str, timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = Path(urllib.parse.urlparse(tar_url).path).name
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if (not os.path.exists(local)) or os.path.getsize(local) == 0:
        print(f"📦 Download tar -> {local}")
        _download_to(local, tar_url)
    return local

def _norm_path(p: str) -> str:
    p = p.replace("\\", "/")
    p = re.sub(r"^\./+", "", p)
    p = re.sub(r"/{2,}", "/", p)
    return p.lower().strip()

_SIZE_SUFFIX_RE = re.compile(r"(_lrg|_sm|_s|-large|-small)$", re.IGNORECASE)
def _strip_size_suffix(basename: str) -> str:
    name, ext = os.path.splitext(basename)
    name = _SIZE_SUFFIX_RE.sub("", name)
    return name + ext

_TAR_HANDLE: Dict[str, tarfile.TarFile] = {}
_TAR_INDEX:  Dict[str, Dict[str, tarfile.TarInfo]] = {}

def _get_tar_handle_and_index(local_tar: str) -> Tuple[tarfile.TarFile, Dict[str, tarfile.TarInfo]]:
    if local_tar not in _TAR_HANDLE:
        t = tarfile.open(local_tar, "r:gz")
        _TAR_HANDLE[local_tar] = t
        _TAR_INDEX[local_tar]  = {_norm_path(ti.name): ti for ti in t.getmembers()}
    return _TAR_HANDLE[local_tar], _TAR_INDEX[local_tar]

def _best_member_match_fast(local_tar: str, wanted_path: str) -> tarfile.TarInfo:
    _, idx = _get_tar_handle_and_index(local_tar)
    w = _norm_path(wanted_path)

    if w in idx:
        return idx[w]

    suffix_hits = [(nm, ti) for nm, ti in idx.items() if nm.endswith(w)]
    if suffix_hits:
        suffix_hits.sort(key=lambda x: len(x[0]), reverse=True)
        return suffix_hits[0][1]

    want_base = Path(w).name
    base_hits = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base]
    if base_hits:
        return base_hits[0][1]

    want_base2 = _strip_size_suffix(want_base)
    base_hits2 = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base2]
    if base_hits2:
        return base_hits2[0][1]

    def _lcsuf_len(a: str, b: str) -> int:
        a, b = a[::-1], b[::-1]
        n = min(len(a), len(b)); i = 0
        while i < n and a[i] == b[i]:
            i += 1
        return i

    scored = [(_lcsuf_len(nm, w), ti) for nm, ti in idx.items()]
    scored.sort(key=lambda x: x[0], reverse=True)
    if scored and scored[0][0] > 0:
        return scored[0][1]

    raise FileNotFoundError(f"Not found in tar: {wanted_path}")

def read_member_image(pmc_tar_url: str, file_in_tar: str) -> Optional[Image.Image]:
    try:
        local_tar = _ensure_tar_local(pmc_tar_url)
        tar, _ = _get_tar_handle_and_index(local_tar)
        ti = _best_member_match_fast(local_tar, file_in_tar)
        with tar.extractfile(ti) as f:
            if f is None:
                return None
            img = Image.open(io.BytesIO(f.read())).convert("RGB")
        return img
    except (UnidentifiedImageError, OSError, FileNotFoundError, tarfile.TarError):
        return None
    except Exception:
        return None

print("✅ Step1 done: splits saved + read_member_image defined")


### finetune

In [ ]:
# ============================================================
# Step 2) Fine-tune (ft200) on LLaVA-Med v1.5  [SAFE + QUIET]
# - Uses radiology members (from tar) only: ft200=200
# - LoRA finetune (bnb-free, triton-free)
# - Quiet tar download (no spam)
# - Fix processor.patch_size / num_additional_image_tokens
# - Disable truncation to avoid image-token mismatch
# - OOM guard: gradient checkpointing + small batch + accum + skip-on-oom
# ============================================================

import os, io, json, tarfile, gc, random, re, urllib.parse, shutil
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoProcessor, AutoModelForImageTextToText

# ---------------------------
# 0) Config
# ---------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Input data (created by your dataset script)
PAIRS_DIR     = "/content/drive/MyDrive/Medical/llava_med_pairs"
MEMBERS_JSON  = f"{PAIRS_DIR}/members_1k.json"   # radiology member list (tar-backed)

# Output model dir
OUT_DIR = "/content/llava_med_loraBaseline_ft200_safe"
os.makedirs(OUT_DIR, exist_ok=True)

# Model
BASE_ID = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"

# Prompt
USER_PROMPT = "<image>\nDescribe the medical image."

# Train settings (safe defaults)
FT_N            = 200     # ft200
EPOCHS          = 1
LR              = 2e-4
WEIGHT_DECAY    = 0.0
MAX_STEPS       = None    # None => run full epoch
LOG_EVERY       = 1        # tqdm already shows loss
SAVE_FINAL      = True

# Memory settings
BATCH_SIZE      = 1        # keep small
GRAD_ACCUM      = 4        # effective batch = 4
MAX_GRAD_NORM   = 1.0
USE_GC          = True     # gradient checkpointing
OOM_SKIP        = True     # skip batch on OOM

# LoRA settings (bnb-free)
USE_LORA        = True
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05

# TAR cache (quiet)
PMC_TAR_CACHE_DIR = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)

# ---------------------------
# 1) Device / dtype
# ---------------------------
assert torch.cuda.is_available(), "❌ Need GPU"
device = torch.device("cuda:0")
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

cap_major = torch.cuda.get_device_capability(0)[0]
DTYPE = torch.bfloat16 if cap_major >= 8 else torch.float16
print(f"✅ device: {device} DTYPE: {DTYPE}")

# ---------------------------
# 2) Load data (members) -> ft200
# ---------------------------
assert os.path.exists(MEMBERS_JSON), f"❌ Missing: {MEMBERS_JSON}"
with open(MEMBERS_JSON, "r", encoding="utf-8") as f:
    members_all = json.load(f)

random.Random(SEED).shuffle(members_all)
ft200 = members_all[:FT_N]
print("✅ ft200 size:", len(ft200))

# ---------------------------
# 3) TAR image loader (QUIET) + robust matching
# ---------------------------
import requests

def _download_to(path: str, url: str, timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = Path(urllib.parse.urlparse(tar_url).path).name
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if (not os.path.exists(local)) or os.path.getsize(local) == 0:
        _download_to(local, tar_url)
    return local

def _norm_path(p: str) -> str:
    p = p.replace("\\", "/")
    p = re.sub(r"^\./+", "", p)
    p = re.sub(r"/{2,}", "/", p)
    return p.lower().strip()

_SIZE_SUFFIX_RE = re.compile(r"(_lrg|_sm|_s|-large|-small)$", re.IGNORECASE)
def _strip_size_suffix(basename: str) -> str:
    name, ext = os.path.splitext(basename)
    name = _SIZE_SUFFIX_RE.sub("", name)
    return name + ext

_TAR_HANDLE: Dict[str, tarfile.TarFile] = {}
_TAR_INDEX:  Dict[str, Dict[str, tarfile.TarInfo]] = {}

def _get_tar_handle_and_index(local_tar: str):
    if local_tar not in _TAR_HANDLE:
        t = tarfile.open(local_tar, "r:gz")
        _TAR_HANDLE[local_tar] = t
        _TAR_INDEX[local_tar]  = {_norm_path(ti.name): ti for ti in t.getmembers()}
    return _TAR_HANDLE[local_tar], _TAR_INDEX[local_tar]

def _best_member_match_fast(local_tar: str, wanted_path: str) -> tarfile.TarInfo:
    _, idx = _get_tar_handle_and_index(local_tar)
    w = _norm_path(wanted_path)

    if w in idx:
        return idx[w]

    suffix_hits = [(nm, ti) for nm, ti in idx.items() if nm.endswith(w)]
    if suffix_hits:
        suffix_hits.sort(key=lambda x: len(x[0]), reverse=True)
        return suffix_hits[0][1]

    want_base = Path(w).name
    base_hits = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base]
    if base_hits:
        return base_hits[0][1]

    want_base2 = _strip_size_suffix(want_base)
    base_hits2 = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base2]
    if base_hits2:
        return base_hits2[0][1]

    # longest common suffix fallback
    def _lcsuf_len(a: str, b: str) -> int:
        a, b = a[::-1], b[::-1]
        n = min(len(a), len(b)); i = 0
        while i < n and a[i] == b[i]:
            i += 1
        return i
    scored = [(_lcsuf_len(nm, w), ti) for nm, ti in idx.items()]
    scored.sort(key=lambda x: x[0], reverse=True)
    if scored and scored[0][0] > 0:
        return scored[0][1]

    raise FileNotFoundError(f"Not found in tar: {wanted_path}")

def read_member_image(pmc_tar_url: str, file_in_tar: str) -> Optional[Image.Image]:
    try:
        local_tar = _ensure_tar_local(pmc_tar_url)
        tar, _ = _get_tar_handle_and_index(local_tar)
        ti = _best_member_match_fast(local_tar, file_in_tar)
        with tar.extractfile(ti) as f:
            if f is None:
                return None
            img = Image.open(io.BytesIO(f.read())).convert("RGB")
        return img
    except (UnidentifiedImageError, OSError, FileNotFoundError, tarfile.TarError):
        return None
    except Exception:
        return None

# ---------------------------
# 4) Load processor/model + fix processor fields
# ---------------------------
def fix_processor_fields(processor):
    if getattr(processor, "patch_size", None) is None:
        ip = getattr(processor, "image_processor", None)
        ps = getattr(ip, "patch_size", None)
        if isinstance(ps, int):
            processor.patch_size = ps
        elif isinstance(ps, (tuple, list)) and len(ps) > 0 and isinstance(ps[0], int):
            processor.patch_size = ps[0]
        else:
            processor.patch_size = 14
    if getattr(processor, "num_additional_image_tokens", None) is None:
        processor.num_additional_image_tokens = 0
    return processor

print("⏳ Loading processor/model ...")
processor = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True)
processor = fix_processor_fields(processor)

model = AutoModelForImageTextToText.from_pretrained(
    BASE_ID,
    torch_dtype=DTYPE,
    device_map=None,
    trust_remote_code=True,
).to(device)

# gradient checkpointing
if USE_GC:
    try:
        model.gradient_checkpointing_enable()
        print("✅ gradient checkpointing: ON")
    except Exception:
        pass

# must disable cache if GC on
if hasattr(model, "config"):
    model.config.use_cache = False
model.train()
print("✅ model ready")

# ---------------------------
# 5) Build prompt template + PREFIX_LEN
# ---------------------------
USER_MSGS = [{"role": "user", "content": USER_PROMPT}]
USER_PROMPT_STR = processor.apply_chat_template(USER_MSGS, add_generation_prompt=True)
PREFIX_IDS = processor.tokenizer(USER_PROMPT_STR, return_tensors="pt").input_ids
PREFIX_LEN = int(PREFIX_IDS.size(1))
print("✅ PREFIX_LEN:", PREFIX_LEN)

# ---------------------------
# 6) LoRA enable (bnb-free)
# ---------------------------
if USE_LORA:
    from peft import LoraConfig, get_peft_model, TaskType

    for p in model.parameters():
        p.requires_grad_(False)

    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
    lcfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
    )
    model = get_peft_model(model, lcfg)
    model.print_trainable_parameters()
    print("✅ LoRA enabled (bnb-free)")

# ---------------------------
# 7) Dataset / Collate  (NO truncation max_length!)
# ---------------------------
def get_caption(obj: Dict[str,Any]) -> str:
    return str(obj.get("caption","")).strip()

class Ft200Dataset(Dataset):
    def __init__(self, items: List[Dict[str,Any]]):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        obj = self.items[idx]
        cap = get_caption(obj)
        img = read_member_image(obj["pmc_tar_url"], obj["image_file_path"])
        if img is None:
            img = Image.new("RGB", (448, 448), color=(0,0,0))
        if not cap:
            cap = "No caption."
        return img, cap

def collate_fn(batch):
    # batch: list[(PIL, caption)]
    images, caps = zip(*batch)
    full_texts = [USER_PROMPT_STR + str(c) for c in caps]

    # IMPORTANT: avoid truncation='max_length' to prevent image-token mismatch
    enc = processor(
        text=list(full_texts),
        images=list(images),
        return_tensors="pt",
        padding=True,
        truncation=False,   # <-- critical
    )

    labels = enc["input_ids"].clone()
    labels[:, :PREFIX_LEN] = -100
    enc["labels"] = labels
    return enc

ds = Ft200Dataset(ft200)
dl = DataLoader(
    ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,   # <-- avoid "cannot pin cuda tensor"
    collate_fn=collate_fn,
)

# ---------------------------
# 8) Optimizer (pure torch AdamW)
# ---------------------------
trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)

# GradScaler only for fp16
use_scaler = (DTYPE == torch.float16)
scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

# ---------------------------
# 9) Train loop (quiet + oom-safe)
# ---------------------------
steps_per_epoch = len(dl)
total_steps = steps_per_epoch * EPOCHS
if MAX_STEPS is not None:
    total_steps = min(total_steps, int(MAX_STEPS))
print(f"steps_per_epoch: {steps_per_epoch} total_steps: {total_steps}")

global_step = 0
pbar = tqdm(total=total_steps, desc="FT(ft200)")

model.train()
for epoch in range(EPOCHS):
    for it, batch in enumerate(dl):
        if global_step >= total_steps:
            break

        # move to GPU here (so DataLoader stays CPU-only)
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        try:
            with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=True):
                out = model(**batch)
                loss = out.loss / GRAD_ACCUM

            if use_scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (global_step + 1) % GRAD_ACCUM == 0:
                if use_scaler:
                    scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainable, MAX_GRAD_NORM)

                if use_scaler:
                    scaler.step(opt)
                    scaler.update()
                else:
                    opt.step()

                opt.zero_grad(set_to_none=True)

            # progress bar (only one line)
            pbar.set_postfix(loss=float(loss.detach().cpu()) * GRAD_ACCUM)
            pbar.update(1)
            global_step += 1

        except torch.cuda.OutOfMemoryError:
            # OOM guard
            if OOM_SKIP:
                opt.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                gc.collect()
                # skip this step (still advance progress bar to keep UI smooth)
                pbar.set_postfix(loss="OOM-skip")
                pbar.update(1)
                global_step += 1
                continue
            else:
                raise

    if global_step >= total_steps:
        break

pbar.close()

# ---------------------------
# 10) Save
# ---------------------------
if SAVE_FINAL:
    model.save_pretrained(OUT_DIR)
    processor.save_pretrained(OUT_DIR)
    print("✅ Saved to:", OUT_DIR)


### get acts

In [ ]:
# ============================================================
# Step 3) Extract Activations (LLaVA-Med v1.5)  [FINAL + SAFE]
# - TRAIN acts: FT model on ft200 (members tar-backed)
# - DEV/TEST acts: BASE model on dev/test
# - Feature: last-token hidden state for each layer (includes embedding)
# - Saves: layer_{layer_id}_{index}.pt  (shape [1, D], float32 on CPU)
# ============================================================

import os, io, json, tarfile, gc, random, re, urllib.parse
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# ---------------------------
# 0) Config
# ---------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Step1 output dir
PAIRS_DIR = "/content/drive/MyDrive/Medical/llava_med_pairs"
MEMBERS_JSON = f"{PAIRS_DIR}/members_1k.json"
NONMEM_JSON  = f"{PAIRS_DIR}/nonmembers_roco_1k.json"

# Step2 output model dir (your fine-tuned model)
FT_MODEL_DIR = "/content/llava_med_loraBaseline_ft200_safe"  # <-- Step2保存目录
BASE_ID      = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"

# splits file (created here, for Step4)
SPLIT_PATH = "/content/splits_llava_med_lrprobe.json"

# activations output
ACTS_ROOT = "/content/acts_llava_med"
TRAIN_DIR = os.path.join(ACTS_ROOT, "train")  # FT model on ft200
DEV_DIR   = os.path.join(ACTS_ROOT, "dev")    # BASE model on dev
TEST_DIR  = os.path.join(ACTS_ROOT, "test")   # BASE model on test

# prompt (must match Step2/Step4)
USER_PROMPT = "<image>\nDescribe the medical image."

# TAR cache (quiet)
PMC_TAR_CACHE_DIR = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)

# sizes (LRProbe default)
FT200_N       = 200
DEV_MEM_N     = 100
DEV_NONMEM_N  = 100
TEST_MEM_N    = 500
TEST_NONMEM_N = 500

# ---------------------------
# 1) Device / dtype
# ---------------------------
assert torch.cuda.is_available(), "❌ Need GPU"
device = torch.device("cuda:0")
torch.cuda.set_device(0)
cap_major = torch.cuda.get_device_capability(0)[0]
DTYPE = torch.bfloat16 if cap_major >= 8 else torch.float16
print(f"✅ device: {device} DTYPE: {DTYPE}")

# ---------------------------
# 2) Load data lists
# ---------------------------
assert os.path.exists(MEMBERS_JSON), f"❌ Missing: {MEMBERS_JSON}"
assert os.path.exists(NONMEM_JSON),  f"❌ Missing: {NONMEM_JSON}"

with open(MEMBERS_JSON, "r", encoding="utf-8") as f:
    members_all = json.load(f)
with open(NONMEM_JSON, "r", encoding="utf-8") as f:
    nonmem_all = json.load(f)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmem_all)

members_all = members_all[:1000]
nonmem_all  = nonmem_all[:1000]

# ---------------------------
# 3) Build splits for LRProbe (and save SPLIT_PATH)
# ---------------------------
# Train(ft200): member 200
ft200 = members_all[:FT200_N]

# Dev: 100 member + 100 nonmember
dev_mem100    = members_all[FT200_N:FT200_N + DEV_MEM_N]
dev_nonmem100 = nonmem_all[:DEV_NONMEM_N]

# Test: 500 member + 500 nonmember
test_mem500    = members_all[FT200_N + DEV_MEM_N:FT200_N + DEV_MEM_N + TEST_MEM_N]
test_nonmem500 = nonmem_all[DEV_NONMEM_N:DEV_NONMEM_N + TEST_NONMEM_N]

splits = {
    "ft200": ft200,
    "dev_mem100": dev_mem100,
    "dev_nonmem100": dev_nonmem100,
    "test_mem500": test_mem500,
    "test_nonmem500": test_nonmem500
}
with open(SPLIT_PATH, "w", encoding="utf-8") as f:
    json.dump(splits, f, ensure_ascii=False, indent=2)

print("✅ Split sizes: train(ft200)=", len(ft200),
      "dev=(", len(dev_mem100), len(dev_nonmem100), ")",
      "test=(", len(test_mem500), len(test_nonmem500), ")")
print("✅ SPLIT_PATH:", SPLIT_PATH)

# ---------------------------
# 4) TAR loader (quiet) for members
# ---------------------------
import requests

def _download_to(path: str, url: str, timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = Path(urllib.parse.urlparse(tar_url).path).name
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if (not os.path.exists(local)) or os.path.getsize(local) == 0:
        _download_to(local, tar_url)
    return local

def _norm_path(p: str) -> str:
    p = p.replace("\\", "/")
    p = re.sub(r"^\./+", "", p)
    p = re.sub(r"/{2,}", "/", p)
    return p.lower().strip()

_SIZE_SUFFIX_RE = re.compile(r"(_lrg|_sm|_s|-large|-small)$", re.IGNORECASE)
def _strip_size_suffix(basename: str) -> str:
    name, ext = os.path.splitext(basename)
    name = _SIZE_SUFFIX_RE.sub("", name)
    return name + ext

_TAR_HANDLE: Dict[str, tarfile.TarFile] = {}
_TAR_INDEX:  Dict[str, Dict[str, tarfile.TarInfo]] = {}

def _get_tar_handle_and_index(local_tar: str):
    if local_tar not in _TAR_HANDLE:
        t = tarfile.open(local_tar, "r:gz")
        _TAR_HANDLE[local_tar] = t
        _TAR_INDEX[local_tar]  = {_norm_path(ti.name): ti for ti in t.getmembers()}
    return _TAR_HANDLE[local_tar], _TAR_INDEX[local_tar]

def _best_member_match_fast(local_tar: str, wanted_path: str) -> tarfile.TarInfo:
    _, idx = _get_tar_handle_and_index(local_tar)
    w = _norm_path(wanted_path)

    if w in idx:
        return idx[w]

    suffix_hits = [(nm, ti) for nm, ti in idx.items() if nm.endswith(w)]
    if suffix_hits:
        suffix_hits.sort(key=lambda x: len(x[0]), reverse=True)
        return suffix_hits[0][1]

    want_base = Path(w).name
    base_hits = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base]
    if base_hits:
        return base_hits[0][1]

    want_base2 = _strip_size_suffix(want_base)
    base_hits2 = [(nm, ti) for nm, ti in idx.items() if Path(nm).name == want_base2]
    if base_hits2:
        return base_hits2[0][1]

    # longest common suffix fallback
    def _lcsuf_len(a: str, b: str) -> int:
        a, b = a[::-1], b[::-1]
        n = min(len(a), len(b)); i = 0
        while i < n and a[i] == b[i]:
            i += 1
        return i
    scored = [(_lcsuf_len(nm, w), ti) for nm, ti in idx.items()]
    scored.sort(key=lambda x: x[0], reverse=True)
    if scored and scored[0][0] > 0:
        return scored[0][1]

    raise FileNotFoundError(f"Not found in tar: {wanted_path}")

def read_member_image(pmc_tar_url: str, file_in_tar: str) -> Optional[Image.Image]:
    try:
        local_tar = _ensure_tar_local(pmc_tar_url)
        tar, _ = _get_tar_handle_and_index(local_tar)
        ti = _best_member_match_fast(local_tar, file_in_tar)
        with tar.extractfile(ti) as f:
            if f is None:
                return None
            img = Image.open(io.BytesIO(f.read())).convert("RGB")
        return img
    except (UnidentifiedImageError, OSError, FileNotFoundError, tarfile.TarError):
        return None
    except Exception:
        return None

def read_nonmember_image(local_path: str) -> Optional[Image.Image]:
    try:
        if (not local_path) or (not os.path.exists(local_path)):
            return None
        return Image.open(local_path).convert("RGB")
    except Exception:
        return None

# ---------------------------
# 5) Processor fix (patch_size)  [critical]
# ---------------------------
def fix_processor_fields(processor):
    if getattr(processor, "patch_size", None) is None:
        ip = getattr(processor, "image_processor", None)
        ps = getattr(ip, "patch_size", None)
        if isinstance(ps, int):
            processor.patch_size = ps
        elif isinstance(ps, (tuple, list)) and len(ps) > 0 and isinstance(ps[0], int):
            processor.patch_size = ps[0]
        else:
            processor.patch_size = 14
    if getattr(processor, "num_additional_image_tokens", None) is None:
        processor.num_additional_image_tokens = 0
    return processor

# ---------------------------
# 6) Build template prompt string
# ---------------------------
def build_user_prompt_str(processor):
    msgs = [{"role":"user","content": USER_PROMPT}]
    return processor.apply_chat_template(msgs, add_generation_prompt=True)

# ---------------------------
# 7) Hidden states extractor (last token)
# ---------------------------
@torch.no_grad()
def get_last_token_hiddens(model, processor, image: Image.Image, caption: str) -> List[torch.Tensor]:
    user_prompt_str = build_user_prompt_str(processor)
    full_text = user_prompt_str + str(caption)

    # IMPORTANT: no truncation to avoid image-token mismatch
    enc = processor(text=full_text, images=[image], return_tensors="pt",
                    padding=True, truncation=False)
    enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

    out = model(**enc, output_hidden_states=True, return_dict=True)
    hstates = out.hidden_states  # tuple: (emb, layer1..layerN)

    vecs = []
    for h in hstates:
        vecs.append(h[0, -1, :].detach().to(torch.float32).cpu())  # [D] on CPU float32
    return vecs

def detect_num_layers(model, processor) -> int:
    dummy = Image.new("RGB", (448, 448), color=(128,128,128))
    v = get_last_token_hiddens(model, processor, dummy, caption="dummy")
    return len(v)

def ensure_dir(p): os.makedirs(p, exist_ok=True)

def generate_split_acts(
    processor,
    model,
    dataset: List[Dict[str,Any]],
    save_dir: str,
    idx_offset: int,
    kind: str,  # "member" or "nonmember"
):
    ensure_dir(save_dir)
    num_layers = detect_num_layers(model, processor)
    print(f"✅ hidden_states length = {num_layers} (includes embedding).")

    for i in tqdm(range(len(dataset)), desc=f"Acts -> {os.path.basename(save_dir)}"):
        obj = dataset[i]
        cap = str(obj.get("caption","")).strip()
        if not cap:
            cap = "No caption."

        # load image
        if kind == "member":
            img = read_member_image(obj["pmc_tar_url"], obj["image_file_path"])
        else:
            img = read_nonmember_image(obj["local_image_path"])

        if img is None:
            # fallback black image to keep indices aligned (important for Step4)
            img = Image.new("RGB", (448, 448), color=(0,0,0))

        vecs = get_last_token_hiddens(model, processor, img, caption=cap)

        out_idx = i + idx_offset
        for layer_id, v in enumerate(vecs):
            torch.save(v.unsqueeze(0), os.path.join(save_dir, f"layer_{layer_id}_{out_idx}.pt"))

        if (out_idx % 50) == 0:
            torch.cuda.empty_cache()
            gc.collect()

# ---------------------------
# 8) Load models (FT + BASE)
# ---------------------------
def load_model(model_id_or_dir: str):
    proc = AutoProcessor.from_pretrained(model_id_or_dir, trust_remote_code=True)
    proc = fix_processor_fields(proc)
    m = AutoModelForImageTextToText.from_pretrained(
        model_id_or_dir,
        torch_dtype=DTYPE,
        device_map=None,
        trust_remote_code=True
    ).to(device)
    try:
        m.gradient_checkpointing_disable()
    except Exception:
        pass
    if hasattr(m, "config"):
        m.config.use_cache = False
    m.eval()
    return proc, m

# clean dirs
shutil.rmtree(ACTS_ROOT, ignore_errors=True)
ensure_dir(TRAIN_DIR); ensure_dir(DEV_DIR); ensure_dir(TEST_DIR)

print("\n=== Load FT model (TRAIN acts) ===")
ft_proc, ft_model = load_model(FT_MODEL_DIR)
print("✅ FT model loaded:", FT_MODEL_DIR)

print("\n=== Load BASE model (DEV/TEST acts) ===")
base_proc, base_model = load_model(BASE_ID)
print("✅ Base model loaded:", BASE_ID)

# ---------------------------
# 9) RUN
# ---------------------------
# [1/3] TRAIN acts: FT model on ft200 (members)
print("\n[1/3] TRAIN acts: FT model on ft200 (members)")
generate_split_acts(ft_proc, ft_model, ft200, TRAIN_DIR, idx_offset=0, kind="member")

# [2/3] DEV acts: BASE model on dev
print("\n[2/3] DEV acts: BASE model")
print("  - dev member (idx 0..99)")
generate_split_acts(base_proc, base_model, dev_mem100, DEV_DIR, idx_offset=0, kind="member")
print("  - dev nonmember (offset=100 => idx 100..199)")
generate_split_acts(base_proc, base_model, dev_nonmem100, DEV_DIR, idx_offset=len(dev_mem100), kind="nonmember")

# [3/3] TEST acts: BASE model on test
print("\n[3/3] TEST acts: BASE model")
print("  - test member (idx 0..499)")
generate_split_acts(base_proc, base_model, test_mem500, TEST_DIR, idx_offset=0, kind="member")
print("  - test nonmember (offset=500 => idx 500..999)")
generate_split_acts(base_proc, base_model, test_nonmem500, TEST_DIR, idx_offset=len(test_mem500), kind="nonmember")

print("\n✅ Done. Acts saved under:", ACTS_ROOT)
print("train:", TRAIN_DIR)
print("dev  :", DEV_DIR)
print("test :", TEST_DIR)

# quick sanity check for a few files (should exist)
import glob
print("\nSanity check example files:")
print("DEV layer0 idx0:", os.path.exists(os.path.join(DEV_DIR, "layer_0_0.pt")))
print("DEV layer0 idx150:", os.path.exists(os.path.join(DEV_DIR, "layer_0_150.pt")))
print("TEST layer0 idx999:", os.path.exists(os.path.join(TEST_DIR, "layer_0_999.pt")))
print("Train layer0 idx0:", os.path.exists(os.path.join(TRAIN_DIR, "layer_0_0.pt")))


### evaluate

In [ ]:
# ============================================================
# Step 4) LRProbe MIA (LLaVA-Med v1.5)  [FINAL + NO-SKLEARN]
# - Train probe on TRAIN acts (ft200, label=1)
#   + also uses DEV nonmember acts as negatives to fit scaler/probe stably
# - Evaluate on DEV and TEST: (members vs nonmembers)
# - Metrics:
#     (1) AUC (no sklearn)
#     (2) Fix TPR=80% threshold from DEV positives -> report FPR on DEV negatives
#     (3) Same threshold -> report TEST TPR/FPR
# - Reads Step3 outputs:
#     ACTS_ROOT/train, dev, test with files layer_{L}_{idx}.pt
# - Uses per-layer Logistic Regression implemented in torch
# ============================================================

import os, json, math, gc
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import torch
from tqdm.auto import tqdm

# ---------------------------
# 0) Paths
# ---------------------------
SPLIT_PATH = "/content/splits_llava_med_lrprobe.json"
ACTS_ROOT  = "/content/acts_llava_med"
TRAIN_DIR  = os.path.join(ACTS_ROOT, "train")
DEV_DIR    = os.path.join(ACTS_ROOT, "dev")
TEST_DIR   = os.path.join(ACTS_ROOT, "test")

assert os.path.exists(SPLIT_PATH), f"❌ Missing: {SPLIT_PATH}"
assert os.path.isdir(TRAIN_DIR), f"❌ Missing dir: {TRAIN_DIR}"
assert os.path.isdir(DEV_DIR),   f"❌ Missing dir: {DEV_DIR}"
assert os.path.isdir(TEST_DIR),  f"❌ Missing dir: {TEST_DIR}"

with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

# sizes (must match Step3 offsets)
N_TRAIN = len(splits["ft200"])                 # 200
N_DEV_M = len(splits["dev_mem100"])            # 100
N_DEV_N = len(splits["dev_nonmem100"])         # 100
N_TST_M = len(splits["test_mem500"])           # 500
N_TST_N = len(splits["test_nonmem500"])        # 500

train_idx = list(range(N_TRAIN))               # train idx: 0..199
dev_mem_idx = list(range(N_DEV_M))             # dev mem: 0..99
dev_non_idx = list(range(N_DEV_M, N_DEV_M+N_DEV_N))  # dev non: 100..199
test_mem_idx = list(range(N_TST_M))            # test mem: 0..499
test_non_idx = list(range(N_TST_M, N_TST_M+N_TST_N))  # test non: 500..999

print("✅ Split sizes:",
      "train(ft200)=", N_TRAIN,
      "dev=", (N_DEV_M, N_DEV_N),
      "test=", (N_TST_M, N_TST_N))
print("✅ Index sanity: train", (train_idx[0], train_idx[-1]),
      "dev", (dev_mem_idx[0], dev_non_idx[-1]),
      "test", (test_mem_idx[0], test_non_idx[-1]))

# ---------------------------
# 1) Helpers: load layer vectors
# ---------------------------
def discover_layers(acts_dir: str) -> List[int]:
    # find all layer ids by checking files "layer_{L}_0.pt"
    layer_ids = []
    for fn in os.listdir(acts_dir):
        if fn.startswith("layer_") and fn.endswith(".pt"):
            # layer_{L}_{idx}.pt
            parts = fn.replace(".pt","").split("_")
            if len(parts) == 3:
                try:
                    L = int(parts[1])
                    layer_ids.append(L)
                except:
                    pass
    layer_ids = sorted(list(set(layer_ids)))
    return layer_ids

def load_vec(fp: str) -> torch.Tensor:
    t = torch.load(fp, map_location="cpu")
    # saved as [1, D] float32
    if t.ndim == 2 and t.size(0) == 1:
        t = t[0]
    return t.to(torch.float32)  # [D]

def load_layer_matrix(acts_dir: str, layer_id: int, idx_list: List[int]) -> torch.Tensor:
    # returns [N, D] on CPU float32
    rows = []
    for idx in idx_list:
        fp = os.path.join(acts_dir, f"layer_{layer_id}_{idx}.pt")
        if not os.path.exists(fp):
            raise FileNotFoundError(f"Missing act: {fp}")
        rows.append(load_vec(fp).unsqueeze(0))
    return torch.cat(rows, dim=0)

# ---------------------------
# 2) Metrics (no sklearn)
# ---------------------------
def auc_from_scores(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = y_true.astype(np.int32)
    y_score = y_score.astype(np.float64)
    n_pos = int(y_true.sum())
    n_neg = int((1 - y_true).sum())
    if n_pos == 0 or n_neg == 0:
        return 0.5

    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(len(y_score), dtype=np.float64) + 1.0

    sorted_scores = y_score[order]
    i = 0
    while i < len(sorted_scores):
        j = i
        while j + 1 < len(sorted_scores) and sorted_scores[j + 1] == sorted_scores[i]:
            j += 1
        if j > i:
            avg_rank = 0.5 * (ranks[order[i]] + ranks[order[j]])
            ranks[order[i:j+1]] = avg_rank
        i = j + 1

    sum_ranks_pos = float(ranks[y_true == 1].sum())
    auc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc)

def tpr_fpr_at_threshold(pos_scores: np.ndarray, neg_scores: np.ndarray, thr: float) -> Tuple[float,float]:
    tpr = float(np.mean(pos_scores >= thr)) if len(pos_scores) else 0.0
    fpr = float(np.mean(neg_scores >= thr)) if len(neg_scores) else 0.0
    return tpr, fpr

# ---------------------------
# 3) Torch LR (standardize + L2)
# ---------------------------
class TorchStandardScaler:
    def __init__(self, eps: float = 1e-6):
        self.eps = eps
        self.mean = None
        self.std  = None

    def fit(self, X: torch.Tensor):
        self.mean = X.mean(dim=0, keepdim=True)
        self.std  = X.std(dim=0, keepdim=True).clamp_min(self.eps)
        return self

    def transform(self, X: torch.Tensor) -> torch.Tensor:
        return (X - self.mean) / self.std

class TorchLogReg:
    def __init__(self, dim: int, l2: float = 1e-3):
        self.w = torch.zeros(dim, dtype=torch.float32)
        self.b = torch.tensor(0.0, dtype=torch.float32)
        self.l2 = float(l2)

    def predict_logits(self, X: torch.Tensor) -> torch.Tensor:
        # X: [N,D]
        return X.matmul(self.w) + self.b  # [N]

    def fit(self, X: torch.Tensor, y: torch.Tensor, lr=0.1, steps=300):
        # CPU training for stability
        X = X.to(torch.float32)
        y = y.to(torch.float32)

        w = self.w.clone().requires_grad_(True)
        b = self.b.clone().requires_grad_(True)

        opt = torch.optim.Adam([w, b], lr=lr)

        for _ in range(int(steps)):
            opt.zero_grad(set_to_none=True)
            logits = X.matmul(w) + b
            loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y)
            # L2
            loss = loss + 0.5 * self.l2 * (w * w).sum()
            loss.backward()
            opt.step()

        self.w = w.detach()
        self.b = b.detach()
        return self

    def predict_proba(self, X: torch.Tensor) -> torch.Tensor:
        logits = self.predict_logits(X)
        return torch.sigmoid(logits)

# ---------------------------
# 4) Run LRProbe per layer + pick best DEV AUC layer
# ---------------------------
layers = discover_layers(DEV_DIR)
assert len(layers) > 0, "❌ No layers found in DEV_DIR"
print(f"✅ Discovered layers: {len(layers)} layers. First/last =", layers[0], layers[-1])

results = []
best = {"layer": None, "dev_auc": -1.0}

for L in tqdm(layers, desc="LRProbe over layers"):
    # Train features:
    #   positives: TRAIN(ft200) from TRAIN_DIR
    #   negatives: DEV nonmembers from DEV_DIR (idx 100..199)
    Xp = load_layer_matrix(TRAIN_DIR, L, train_idx)     # [200, D]
    Xn = load_layer_matrix(DEV_DIR,   L, dev_non_idx)   # [100, D]
    Xtr = torch.cat([Xp, Xn], dim=0)                    # [300, D]
    ytr = torch.cat([torch.ones(len(Xp)), torch.zeros(len(Xn))], dim=0)

    # Fit scaler on train
    scaler = TorchStandardScaler().fit(Xtr)
    Xtr_s = scaler.transform(Xtr)

    # Fit LR
    lr_model = TorchLogReg(dim=Xtr_s.size(1), l2=1e-3).fit(Xtr_s, ytr, lr=0.05, steps=350)

    # DEV eval: mem vs nonmem
    Xdv_pos = scaler.transform(load_layer_matrix(DEV_DIR, L, dev_mem_idx))
    Xdv_neg = scaler.transform(load_layer_matrix(DEV_DIR, L, dev_non_idx))
    s_pos = lr_model.predict_proba(Xdv_pos).numpy()
    s_neg = lr_model.predict_proba(Xdv_neg).numpy()

    y = np.array([1]*len(s_pos) + [0]*len(s_neg), dtype=np.int32)
    s = np.concatenate([s_pos, s_neg], axis=0)
    dev_auc = auc_from_scores(y, s)

    results.append((L, dev_auc, float(s_pos.mean()), float(s_neg.mean())))

    if dev_auc > best["dev_auc"]:
        best = {"layer": L, "dev_auc": dev_auc}

print("\n" + "="*90)
print("📌 Layer sweep summary (ALL layers, sorted by DEV AUC)")
results_sorted = sorted(results, key=lambda x: x[1], reverse=True)
for (L, aucv, mpos, mneg) in results_sorted:
    print(f"Layer {L:02d} | DEV AUC={aucv:.4f} | mean(p)={mpos:.3f} mean(n)={mneg:.3f}")

best_layer = best["layer"]
print(f"\n🏆 Best layer by DEV AUC: layer={best_layer}  DEV_AUC={best['dev_auc']:.4f}")

# ---------------------------
# 5) Retrain best-layer probe (same recipe) + evaluate DEV/TEST
# ---------------------------
L = best_layer

Xp = load_layer_matrix(TRAIN_DIR, L, train_idx)
Xn = load_layer_matrix(DEV_DIR,   L, dev_non_idx)
Xtr = torch.cat([Xp, Xn], dim=0)
ytr = torch.cat([torch.ones(len(Xp)), torch.zeros(len(Xn))], dim=0)

scaler = TorchStandardScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr)

probe = TorchLogReg(dim=Xtr_s.size(1), l2=1e-3).fit(Xtr_s, ytr, lr=0.05, steps=500)

# DEV scores
Xdv_pos = scaler.transform(load_layer_matrix(DEV_DIR,  L, dev_mem_idx))
Xdv_neg = scaler.transform(load_layer_matrix(DEV_DIR,  L, dev_non_idx))
dev_pos = probe.predict_proba(Xdv_pos).numpy()
dev_neg = probe.predict_proba(Xdv_neg).numpy()

dev_auc = auc_from_scores(
    np.array([1]*len(dev_pos) + [0]*len(dev_neg), dtype=np.int32),
    np.concatenate([dev_pos, dev_neg], axis=0)
)

# threshold for TPR=80% on DEV positives
target_tpr = 0.80
thr = float(np.quantile(dev_pos, 1.0 - target_tpr))  # 0.2 quantile
dev_tpr, dev_fpr = tpr_fpr_at_threshold(dev_pos, dev_neg, thr)

# TEST scores
Xts_pos = scaler.transform(load_layer_matrix(TEST_DIR, L, test_mem_idx))
Xts_neg = scaler.transform(load_layer_matrix(TEST_DIR, L, test_non_idx))
test_pos = probe.predict_proba(Xts_pos).numpy()
test_neg = probe.predict_proba(Xts_neg).numpy()

test_auc = auc_from_scores(
    np.array([1]*len(test_pos) + [0]*len(test_neg), dtype=np.int32),
    np.concatenate([test_pos, test_neg], axis=0)
)
test_tpr, test_fpr = tpr_fpr_at_threshold(test_pos, test_neg, thr)

print("\n" + "="*90)
print("📊 LRProbe Results (Best layer)")
print("="*90)
print(f"Best Layer: {L}")
print(f"DEV  AUC: {dev_auc:.4f}")
print(f"TEST AUC: {test_auc:.4f}")

print("\n✅ Thresholding (choose thr so DEV TPR≈80%)")
print(f"thr = {thr:.6f}")
print(f"DEV : TPR={dev_tpr*100:.2f}% | FPR={dev_fpr*100:.2f}%")
print(f"TEST: TPR={test_tpr*100:.2f}% | FPR={test_fpr*100:.2f}%")

# ---------------------------
# 6) Optional: save summary
# ---------------------------
out_summary = {
    "best_layer": int(L),
    "dev_auc": float(dev_auc),
    "test_auc": float(test_auc),
    "thr_dev_tpr80": float(thr),
    "dev_tpr": float(dev_tpr),
    "dev_fpr": float(dev_fpr),
    "test_tpr": float(test_tpr),
    "test_fpr": float(test_fpr),
    "layer_sweep": [
        {"layer": int(r[0]), "dev_auc": float(r[1]), "dev_pos_mean": float(r[2]), "dev_neg_mean": float(r[3])}
        for r in results_sorted
    ]
}
SAVE_JSON = "/content/lrprobe_llava_med_results.json"
with open(SAVE_JSON, "w", encoding="utf-8") as f:
    json.dump(out_summary, f, ensure_ascii=False, indent=2)
print("\n📝 Saved:", SAVE_JSON)


# BiomedCLIP

## data

In [ ]:

import os
from google.colab import drive

print("🚀 挂载 Google Drive...")
drive.mount('/content/drive')

In [ ]:
import io, os, tarfile, json, random, requests
from PIL import Image
from IPython.display import display

# 文件位置（你前面已生成）
MEMBERS_JSON    = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

with open(MEMBERS_JSON, "r", encoding="utf-8") as f:
    members = json.load(f)
with open(NONMEMBERS_JSON, "r", encoding="utf-8") as f:
    nonmembers = json.load(f)

print(f"members={len(members)}, nonmembers={len(nonmembers)}")

def fetch_img_from_pmc_tar(tar_url: str, inside: str):
    # 下载 tar.gz 到内存并只取一个目标文件
    r = requests.get(tar_url, stream=True, timeout=60)
    r.raise_for_status()
    byts = io.BytesIO(r.content)
    wanted = inside.replace("\\","/")
    with tarfile.open(fileobj=byts, mode="r:gz") as tar:
        member = None
        for ti in tar.getmembers():
            if ti.name.replace("\\","/") == wanted:
                member = ti; break
        if member is None:
            cands = [wanted,
                     wanted.replace(".jpg",".JPG"),
                     wanted.replace(".jpg",".png"),
                     wanted.replace(".JPG",".jpg")]
            for ti in tar.getmembers():
                if ti.name.replace("\\","/") in cands:
                    member = ti; break
        if member is None:
            raise FileNotFoundError(f"图 {inside} 不在 {tar_url} 中")
        with tar.extractfile(member) as src:
            return Image.open(src).convert("RGB")

print("\n=== Members（抽 8 张）===")
for s in random.sample(members, k=8):
    try:
        img = fetch_img_from_pmc_tar(s["pmc_tar_url"], s["image_file_path"])
        print("caption:", s["caption"][:160].replace("\n"," "))
        display(img)
    except Exception as e:
        print("跳过（提取失败）:", e)

print("\n=== Non-members（ROCO 抽 8 张）===")
for s in random.sample(nonmembers, k=8):
    try:
        img = Image.open(s["local_image_path"]).convert("RGB")
        print("caption:", s["caption"][:160].replace("\n"," "))
        display(img)
    except Exception as e:
        print("跳过（本地图失败）:", e)



## zlib

In [ ]:
# ============================================================
# Zlib-Calibrated Contrastive MIA for BiomedCLIP
# - Model : BiomedCLIP (Contrastive Vision-Language)
# - Score : Cosine_Similarity * Zlib_Compression_Ratio
# - Eval  : AUC + TPR@5%FPR + Calibration Stats
# ============================================================

import os, io, json, tarfile, random, shutil, zlib, warnings, urllib.parse
from pathlib import Path
from typing import Dict, Any, Optional, List

import numpy as np
import torch
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve

# ---------------------------
# 0) 配置与路径
# ---------------------------
DRIVE_BASE   = "/content/drive/MyDrive/Medical/llava_med_pairs"
MEMBERS_JSON = f"{DRIVE_BASE}/members_1k.json"
NONMEM_JSON  = f"{DRIVE_BASE}/nonmembers_roco_1k.json"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

PMC_TAR_CACHE_DIR = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)

# ---------------------------
# 1) 模型加载 (BiomedCLIP)
# ---------------------------
try:
    import open_clip
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "open_clip_torch"])
    import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
print(f"⏳ 加载模型：{MODEL_NAME}")
model, _, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()

# ---------------------------
# 2) 核心算法：Zlib 复杂度校准
# ---------------------------
def get_zlib_ratio(text: str) -> float:
    """计算文本压缩率：压缩后字节数 / 原始字节数"""
    if not text or len(text.strip()) == 0:
        return 1.0
    original_bytes = text.encode('utf-8')
    compressed_bytes = zlib.compress(original_bytes, level=9)
    # 比例越高 = 越复杂/信息量越大
    return len(compressed_bytes) / len(original_bytes)

# ---------------------------
# 3) 图像加载逻辑 (针对 PMC Tar)
# ---------------------------
import requests
_TAR_HANDLE = {}; _TAR_INDEX = {}

def _get_tar_info(local_tar: str):
    if local_tar not in _TAR_HANDLE:
        t = tarfile.open(local_tar, "r:gz")
        _TAR_HANDLE[local_tar] = t
        _TAR_INDEX[local_tar]  = {ti.name.lower().strip(): ti for ti in t.getmembers()}
    return _TAR_HANDLE[local_tar], _TAR_INDEX[local_tar]

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    try:
        url = sample.get("pmc_tar_url")
        fname = Path(urllib.parse.urlparse(url).path).name
        local_tar = os.path.join(PMC_TAR_CACHE_DIR, fname)
        if not os.path.exists(local_tar):
            with requests.get(url, stream=True) as r:
                with open(local_tar, "wb") as f: shutil.copyfileobj(r.raw, f)

        tar, idx = _get_tar_info(local_tar)
        path_in = sample.get("image_file_path").lower().strip()
        ti = idx.get(path_in) or next((v for k,v in idx.items() if k.endswith(path_in)), None)
        if ti is None: return None
        with tar.extractfile(ti) as f:
            return Image.open(io.BytesIO(f.read())).convert("RGB")
    except: return None

def load_nonmember_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if p and os.path.exists(p):
        try: return Image.open(p).convert("RGB")
        except: return None
    return None

# ---------------------------
# 4) 推理函数
# ---------------------------
@torch.inference_mode()
def compute_scores(images, captions):
    scores_raw = []   # 纯余弦相似度
    scores_zlib = []  # 校准后的分数

    for img_pil, cap in tqdm(zip(images, captions), total=len(images), desc="Inference"):
        # 图像编码
        img_in = preprocess_val(img_pil).unsqueeze(0).to(device)
        img_emb = model.encode_image(img_in)
        img_emb /= img_emb.norm(dim=-1, keepdim=True)

        # 文本编码
        txt_tok = tokenizer([cap]).to(device)
        txt_emb = model.encode_text(txt_tok)
        txt_emb /= txt_emb.norm(dim=-1, keepdim=True)

        # 计算相似度
        cos_sim = (img_emb * txt_emb).sum().item()

        # 计算 Zlib 比例
        z_ratio = get_zlib_ratio(cap)

        scores_raw.append(cos_sim)
        # 核心逻辑：相似度 * 复杂度。
        # 含义：如果模型能对“极其复杂”的文本依然给出“很高相似度”，则成员嫌疑极大。
        scores_zlib.append(cos_sim * z_ratio)

    return np.array(scores_raw), np.array(scores_zlib)

# ---------------------------
# 5) 主程序
# ---------------------------
def main():
    with open(MEMBERS_JSON, "r") as f: m_data = json.load(f)[:NUM_MEMBERS]
    with open(NONMEM_JSON, "r") as f: n_data = json.load(f)[:NUM_NONMEMBERS]

    # 收集数据
    def collect(data, loader, label):
        imgs, caps = [], []
        for s in tqdm(data, desc=f"Loading {label}"):
            im = loader(s)
            cp = s.get("caption", "").strip()
            if im and cp:
                imgs.append(im); caps.append(cp)
        return imgs, caps

    m_imgs, m_caps = collect(m_data, load_member_image, "Members")
    n_imgs, n_caps = collect(n_data, load_nonmember_image, "Non-members")

    print(f"\n✅ 成功加载：Member={len(m_imgs)}, Non-member={len(n_imgs)}")

    # 运行得分计算
    m_raw, m_zlib = compute_scores(m_imgs, m_caps)
    n_raw, n_zlib = compute_scores(n_imgs, n_caps)

    # 评估函数
    def evaluate(m_s, n_s, title):
        y_true = np.array([1]*len(m_s) + [0]*len(n_s))
        y_score = np.concatenate([m_s, n_s])

        auc = roc_auc_score(y_true, y_score)
        fpr, tpr, _ = roc_curve(y_true, y_score)
        tpr_at_5fpr = tpr[np.where(fpr <= 0.05)[0][-1]]

        print(f"\n--- {title} ---")
        print(f"AUC:         {auc:.4f}")
        print(f"TPR @ 5%FPR: {tpr_at_5fpr*100:.2f}%")
        print(f"Mean (M):    {np.mean(m_s):.4f}")
        print(f"Mean (N):    {np.mean(n_s):.4f}")

    # 对比原始相似度 vs Zlib 校准后的分数
    evaluate(m_raw, n_raw, "Standard Cosine Similarity (Baseline)")
    evaluate(m_zlib, n_zlib, "Zlib-Calibrated Similarity (MIA Attack)")

    # 自动保存
    res = {"auc_zlib": roc_auc_score([1]*len(m_zlib)+[0]*len(n_zlib), np.concatenate([m_zlib, n_zlib]))}
    save_path = "/content/biomedclip_zlib_mia_results.json"
    with open(save_path, "w") as f: json.dump(res, f)
    print(f"\n💾 结果已保存至: {save_path}")

if __name__ == "__main__":
    main()

## Gradaudit

In [ ]:
# =========================================================================
# Gradaudit-only MIA for BiomedCLIP (open_clip) - 改进版 V2.1 (参数调优 + 最终语法修正)
#
# [FIXED] Corrected ALL remaining SyntaxErrors from code compression.
# [TRY] 1. 更严格分位数: SENS_QUANTILE = 0.95 (Top 5%)
# [TRY] 2. 更少层数: N=6, M=6 -> N=6, M=6 (聚焦末端层 + proj)
# [KEPT] 3. 保留 V2 的改进: 差分梯度, 扩增层梯度(含proj), 中位数聚合。
# =========================================================================

import os, io, json, tarfile, random, shutil, warnings, re, gc
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

# ---------------------------
# 0) 路径与超参
# ---------------------------
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

SEED = 42
NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
CALIB_MEM = 200
CALIB_NON = 200
PROBE_MEM = 800
PROBE_NON = 800
BATCH_K = 8
SENS_QUANTILE = 0.95
VISION_LAST_N = 6
TEXT_LAST_M   = 6

PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True); os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

# ---------------------------
# 1) 设备 & 性能设置
# ---------------------------
assert torch.cuda.is_available(), "未检测到 CUDA"
device = "cuda"; tbc.allow_tf32 = True; cudnn.allow_tf32 = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

# ---------------------------
# 2) 加载 BiomedCLIP
# ---------------------------
try: import open_clip
except Exception: import sys, subprocess; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch>=2.24.0", "ftfy", "regex", "tqdm"]); import open_clip # Added ftfy, regex, tqdm
MODEL_NAME = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
print(f"⏳ 加载模型：{MODEL_NAME}")
model, _, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval(); print("✅ BiomedCLIP 加载完成")

# ---------------------------
# 3) 读取数据
# ---------------------------
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all = load_json_list(MEMBERS_JSON); nonmembers_all = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members_all); random.Random(SEED).shuffle(nonmembers_all)
members = members_all[:NUM_MEMBERS]; nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 样本就绪：members={len(members)} | non-members={len(nonmembers)}")

# ---------------------------
# 4) 图像读取 (语法修正完毕)
# ---------------------------
import requests
def _download_to(path: str, url: str, timeout=120):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = tar_url.strip().split("/")[-1]
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if not os.path.exists(local) or os.path.getsize(local) == 0:
        _download_to(local, tar_url, timeout=180)
    return local

def _normalize_path(s: str):
    s = s.replace("\\", "/").lstrip("./")
    return s

# [FIXED] Corrected SyntaxError
def _best_ext_variants(path: str):
    exts = [".jpg", ".jpeg", ".png", ".tif", ".tiff"]
    base = os.path.splitext(path)[0]
    cand = []
    for e in exts:
        cand.append(base + e)
        cand.append(base.replace("_lrg", "").replace("_large", "") + e)
        cand.append((base + "_lrg") + e)
        cand.append((base + "_large") + e)
    uniq, seen = [], set()
    for x in cand:
        x2 = x.lower()
        if x2 not in seen:
            seen.add(x2)
            uniq.append(x)
    return uniq

def _extract_one_from_tar(local_tar: str, file_in_tar: str) -> Optional[str]:
    target = _normalize_path(file_in_tar); cand_paths = _best_ext_variants(target)
    try:
        with tarfile.open(local_tar, "r:gz") as tar:
            members = tar.getmembers(); names_map = {_normalize_path(ti.name): ti for ti in members}; names_low_map = {_normalize_path(ti.name).lower(): ti for ti in members}
            for cand_path in [target] + cand_paths:
                cand_norm = _normalize_path(cand_path)
                if cand_norm in names_map:
                    ti = names_map[cand_norm]; out_path = os.path.join(PMC_EXTRACT_TMP_DIR, Path(ti.name).name)
                    with tar.extractfile(ti) as src, open(out_path, "wb") as dst: shutil.copyfileobj(src, dst)
                    return out_path
                cand_low = cand_norm.lower()
                if cand_low in names_low_map:
                    ti = names_low_map[cand_low]; out_path = os.path.join(PMC_EXTRACT_TMP_DIR, Path(ti.name).name)
                    with tar.extractfile(ti) as src, open(out_path, "wb") as dst: shutil.copyfileobj(src, dst)
                    return out_path
    except (tarfile.ReadError, EOFError, IsADirectoryError): return None
    except Exception: return None
    return None

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    tar_url = sample.get("pmc_tar_url"); path_in = sample.get("image_file_path")
    if not tar_url or not path_in: return None
    try:
        local_tar = _ensure_tar_local(tar_url); local_img = _extract_one_from_tar(local_tar, path_in)
        if local_img is None: return None
        img = Image.open(local_img).convert("RGB"); os.remove(local_img)
        return img
    except (FileNotFoundError, UnidentifiedImageError, OSError, ValueError, requests.exceptions.RequestException, Exception): return None

def load_nonmember_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if not p or not os.path.exists(p): return None
    try: return Image.open(p).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError, Exception): return None


# ---------------------------
# 5) 选择性开梯度 (语法修正完毕)
# ---------------------------
def enable_grads_selectively_CLIP(m) -> List[str]:
    selected = []
    for _, p in m.named_parameters(): p.requires_grad_(False)
    max_v_layer, max_t_layer = 0, 0
    for name, _ in m.named_parameters():
        mobj = re.search(r"(blocks|resblocks)\.(\d+)\.", name)
        if "visual" in name and mobj: max_v_layer = max(max_v_layer, int(mobj.group(2)))
        tobj = re.search(r"bert\.encoder\.layer\.(\d+)\.", name)
        if tobj: max_t_layer = max(max_t_layer, int(tobj.group(1)))

    v_start = max(0, max_v_layer - VISION_LAST_N + 1)
    t_start = max(0, max_t_layer - TEXT_LAST_M + 1)

    for name, p in m.named_parameters():
        if name == "visual.proj" and p.ndim >= 2: p.requires_grad_(True); selected.append(name); continue
        if name == "text_projection.weight" and p.ndim >= 2: p.requires_grad_(True); selected.append(name); continue
        if name == "text.proj" and p.ndim >= 2: p.requires_grad_(True); selected.append(name); continue
        if "visual" in name and re.search(r"(blocks|resblocks)\.(\d+)\.", name):
            idx = int(re.search(r"(blocks|resblocks)\.(\d+)\.", name).group(2))
            if idx >= v_start and p.ndim >= 2 and re.search(r"(attn|mlp|fc|proj|linear|dense|qkv)", name):
                p.requires_grad_(True); selected.append(name); continue
        if "bert.encoder.layer." in name:
            idx = int(re.search(r"bert\.encoder\.layer\.(\d+)\.", name).group(1))
            if idx >= t_start and p.ndim >= 2 and re.search(r"(attention|query|key|value|dense|intermediate|output)", name):
                p.requires_grad_(True); selected.append(name); continue
    return selected

collect_names = enable_grads_selectively_CLIP(model)
print(f"🔧 计划收集梯度的参数数量 (N={VISION_LAST_N}, M={TEXT_LAST_M} + Proj): {len(collect_names)}")

# ---------------------------
# 6) 构建 InfoNCE 批 (语法修正完毕)
# ---------------------------
def make_batch_for_item(target: Dict[str, Any], neg_pool: List[Dict[str, Any]], kind: str, K: int = BATCH_K) -> Optional[Tuple[torch.Tensor, torch.Tensor]]:
    tgt_img = load_member_image(target) if kind == "member" else load_nonmember_image(target)
    if tgt_img is None: return None
    negs, tries = [], 0
    while len(negs) < K - 1 and tries < K * 40:
        cand = random.choice(neg_pool)
        if cand is target: tries += 1; continue
        cand_kind = "member" if "pmc_tar_url" in cand else "nonmember"; cim = load_member_image(cand) if cand_kind == "member" else load_nonmember_image(cand)
        if cim is None: tries += 1; continue
        negs.append(cand); tries += 1
    if len(negs) < K - 1: return None
    batch_list = [target] + negs; imgs, caps = [], []
    for obj in batch_list:
        obj_kind = "member" if "pmc_tar_url" in obj else "nonmember"; im = load_member_image(obj) if obj_kind == "member" else load_nonmember_image(obj)
        if im is None: im = Image.new("RGB", (224,224), color="black")
        imgs.append(preprocess_val(im)); caps.append(str(obj.get("caption","")).strip())
    images = torch.stack(imgs, dim=0).to(device)
    try:
        toks = tokenizer(caps); toks = torch.as_tensor(toks).to(device)
    except Exception: return None
    return images, toks

# ---------------------------
# 7) 对比损失 & 梯度提取 (语法修正完毕)
# ---------------------------
def contrastive_loss_and_backward(images: torch.Tensor, toks: torch.Tensor) -> bool:
    model.train(); model.zero_grad(set_to_none=True)
    try:
        with torch.cuda.amp.autocast():
            img_feats  = model.encode_image(images); txt_feats  = model.encode_text(toks)
            if not torch.isfinite(img_feats).all() or not torch.isfinite(txt_feats).all(): return False
            img_feats  = img_feats / (img_feats.norm(dim=-1, keepdim=True) + 1e-12); txt_feats  = txt_feats / (txt_feats.norm(dim=-1, keepdim=True) + 1e-12)
            if not torch.isfinite(img_feats).all() or not torch.isfinite(txt_feats).all(): return False
            logit_scale = model.logit_scale.exp(); logits_i = logit_scale * img_feats @ txt_feats.t(); labels   = torch.arange(images.size(0), device=images.device)
            loss_i2t = F.cross_entropy(logits_i, labels); loss_t2i = F.cross_entropy(logits_i.t(), labels); loss = (loss_i2t + loss_t2i) / 2
        if not torch.isfinite(loss): return False
        loss.backward()
        return True
    except (RuntimeError, Exception): model.zero_grad(set_to_none=True); torch.cuda.empty_cache(); return False

def compute_grad_dict_for_item(target: Dict[str,Any], neg_pool: List[Dict[str,Any]], kind: str, K: int = BATCH_K) -> Dict[str, torch.Tensor]:
    torch.cuda.empty_cache(); made = make_batch_for_item(target, neg_pool, kind=kind, K=K)
    if made is None: return {}
    images, toks = made
    success = contrastive_loss_and_backward(images, toks)
    if not success: return {}
    grad_dict = {}
    for name, p in model.named_parameters():
        if p.requires_grad and p.grad is not None:
            if torch.isfinite(p.grad).all():
                grad_dict[name] = p.grad.detach().half().cpu()
        if p.grad is not None: p.grad = None
    model.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache(); model.eval()
    return grad_dict if grad_dict else {}


# ---------------------------
# 8) 行/列余弦 (语法修正完毕)
# ---------------------------
def row_col_cos(a: torch.Tensor, b: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    try:
        a_float = a.float(); b_float = b.float()
        if a_float.ndim < 2:
            flat = F.cosine_similarity(a_float.flatten().unsqueeze(0), b_float.flatten().unsqueeze(0), dim=1)
            return torch.nan_to_num(flat, 0.0), torch.nan_to_num(flat, 0.0)
        r = F.cosine_similarity(a_float,   b_float,   dim=1); c = F.cosine_similarity(a_float.T, b_float.T, dim=1)
        return torch.nan_to_num(r, 0.0), torch.nan_to_num(c, 0.0)
    except Exception:
        if a.ndim >= 2: return torch.zeros(a.shape[0]), torch.zeros(a.shape[1])
        else: return torch.zeros(1), torch.zeros(1)

# ---------------------------
# 9) 校准 (语法修正完毕)
# ---------------------------
def _build_avg_grad(calib_samples: List[Dict[str,Any]], neg_pool: List[Dict[str,Any]], kind: str, take: int) -> Dict[str, torch.Tensor]:
    ref = {}; ok = 0; valid_grads_list = []
    for i in tqdm(range(take), desc=f"计算平均梯度 ({kind})"):
        gd = compute_grad_dict_for_item(calib_samples[i], neg_pool, kind=kind, K=BATCH_K)
        if gd: valid_grads_list.append(gd); ok += 1
    if ok == 0: raise RuntimeError(f"{kind} 平均梯度构建失败")
    common_keys = set(valid_grads_list[0].keys())
    for gd in valid_grads_list[1:]: common_keys.intersection_update(gd.keys())
    ref = {}
    for k in common_keys:
        first_shape = valid_grads_list[0][k].shape
        if all(gd[k].shape == first_shape for gd in valid_grads_list):
            sum_tensor = torch.zeros_like(valid_grads_list[0][k], dtype=torch.float32)
            for gd in valid_grads_list: sum_tensor += gd[k].float()
            ref[k] = sum_tensor / ok
    if not ref: raise RuntimeError(f"{kind} 平均梯度聚合失败")
    return ref

def avg_rowcol_sims(samples: List[Dict[str,Any]], ref: Dict[str,torch.Tensor], neg_pool: List[Dict[str,Any]], kind: str, take: int) -> Tuple[Dict[str,torch.Tensor], Dict[str,torch.Tensor]]:
    acc_r, acc_c = {}, {}; cnt = 0
    for i in tqdm(range(take), desc=f"计算相似度 ({kind})"):
        gd = compute_grad_dict_for_item(samples[i], neg_pool, kind=kind, K=BATCH_K)
        if not gd: continue
        cnt += 1
        for name, g in gd.items():
            if name not in ref or ref[name].shape != g.shape: continue
            rs, cs = row_col_cos(g, ref[name])
            if name not in acc_r:
                acc_r[name] = rs.clone().float(); acc_c[name] = cs.clone().float()
            else:
                if acc_r[name].shape == rs.shape: acc_r[name] += rs.float()
                if acc_c[name].shape == cs.shape: acc_c[name] += cs.float()
    if cnt == 0: return {}, {}
    for k in list(acc_r.keys()): acc_r[k] /= cnt; acc_c[k] /= cnt
    return acc_r, acc_c

def build_sensitive_masks(mem_rc, non_rc, q: float):
    mr, mc = mem_rc; nr, nc = non_rc; row_masks, col_masks, total = {}, {}, 0
    if not mr or not nr: return {}, {}, 0
    for name in mr:
        if name not in nr or name not in mc or name not in nc: continue
        if mr[name].shape != nr[name].shape or mc[name].shape != nc[name].shape: continue
        rgap = torch.nan_to_num(mr[name] - nr[name], nan=0.0).cpu(); cgap = torch.nan_to_num(mc[name] - nc[name], nan=0.0).cpu()
        rt = torch.quantile(rgap, q) if rgap.numel() > 0 else torch.tensor(float("inf")); ct = torch.quantile(cgap, q) if cgap.numel() > 0 else torch.tensor(float("inf"))
        rm = (rgap > rt) & (rgap > 0); cm = (cgap > ct) & (cgap > 0)
        if rm.any() or cm.any():
            row_masks[name] = rm; col_masks[name] = cm; total += int(rm.sum().item() + cm.sum().item())
    return row_masks, col_masks, total

# ---------------------------
# 10) 探测：单样本 GradSafe 分数 (语法修正完毕)
# ---------------------------
def gradsafe_score_item(obj: Dict[str,Any], ref: Dict[str,torch.Tensor], row_masks, col_masks, neg_pool: List[Dict[str,Any]], kind: str) -> float:
    if not row_masks and not col_masks: return 0.0
    gd = compute_grad_dict_for_item(obj, neg_pool, kind=kind, K=BATCH_K)
    if not gd: return 0.0
    sims = []
    for name, g in gd.items():
        if name in ref and name in row_masks and name in col_masks and ref[name].shape == g.shape:
            rs, cs = row_col_cos(g, ref[name])
            rm, cm = row_masks.get(name), col_masks.get(name)
            if rm is not None and rm.any(): sims.extend(rs[rm].cpu().tolist())
            if cm is not None and cm.any(): sims.extend(cs[cm].cpu().tolist())
    return float(np.median(sims)) if sims else 0.0

def score_probe_sets(probe_members, probe_nonmembers, ref, row_masks, col_masks, neg_pool_mem, neg_pool_non):
    scores, labels = [], []
    for obj in tqdm(probe_members, desc="Probe-Members"):
        s = gradsafe_score_item(obj, ref, row_masks, col_masks, neg_pool_mem, kind="member")
        scores.append(s); labels.append(1)
    for obj in tqdm(probe_nonmembers, desc="Probe-NonMembers"):
        s = gradsafe_score_item(obj, ref, row_masks, col_masks, neg_pool_non, kind="nonmember")
        scores.append(s); labels.append(0)
    return np.array(scores), np.array(labels)

# ---------------------------
# 11) 主流程 (保持 V2.1 设置)
# ---------------------------
random.shuffle(members_all); random.shuffle(nonmembers_all)
calib_members = members_all[:CALIB_MEM]; calib_nonmems = nonmembers_all[:CALIB_NON]
probe_members = members_all[CALIB_MEM:CALIB_MEM+PROBE_MEM]; probe_nonmems = nonmembers_all[CALIB_NON:CALIB_NON+PROBE_NON]
print(f"\n划分 | 校准: M={len(calib_members)} / N={len(calib_nonmems)}  探测: M={len(probe_members)} / N={len(probe_nonmems)}")
print(f"Batch(K)={BATCH_K}, q={SENS_QUANTILE}, Vision_N={VISION_LAST_N}, Text_M={TEXT_LAST_M}")
neg_pool_for_member   = members_all[CALIB_MEM+PROBE_MEM:] or members_all
neg_pool_for_nonmem   = nonmembers_all[CALIB_NON+PROBE_NON:] or nonmembers_all

try:
    print("\n[V2.1] 正在计算差分参考梯度...")
    avg_grad_mem = _build_avg_grad(calib_members, neg_pool_for_member, kind="member", take=len(calib_members))
    avg_grad_non = _build_avg_grad(calib_nonmems, neg_pool_for_nonmem, kind="nonmember", take=len(calib_nonmems))
    ref_grads = {}
    for k in avg_grad_mem:
        if k in avg_grad_non and avg_grad_mem[k].shape == avg_grad_non[k].shape:
            ref_grads[k] = (avg_grad_mem[k].float() - avg_grad_non[k].float()).detach()
    print(f"✅ 差分参考梯度已构建 (层数: {len(ref_grads)})")

    mr, mc = avg_rowcol_sims(calib_members, ref_grads, neg_pool_for_member,   kind="member",    take=len(calib_members))
    nr, nc = avg_rowcol_sims(calib_nonmems, ref_grads, neg_pool_for_nonmem,   kind="nonmember", take=len(calib_nonmems))

    row_masks, col_masks, total_crit = build_sensitive_masks((mr, mc), (nr, nc), q=SENS_QUANTILE)
    print(f"敏感子维度总数 (q={SENS_QUANTILE}): {total_crit}")

    if total_crit > 0:
        scores, labels = score_probe_sets(probe_members, probe_nonmems, ref_grads, row_masks, col_masks, neg_pool_for_member, neg_pool_for_nonmem)
        if len(np.unique(scores)) <= 1:
            print("\n⚠️ 探测评分失败或所有分数相同")
            auc, best_thr, acc, cm = np.nan, np.nan, np.nan, np.array([[0,0],[0,0]])
        else:
            auc = roc_auc_score(labels, scores); fpr, tpr, thr = roc_curve(labels, scores); best_idx = np.argmax(tpr - fpr); best_thr = thr[best_idx]; pred = (scores >= best_thr).astype(int); acc = accuracy_score(labels, pred); cm  = confusion_matrix(labels, pred)

        print("\n====== GradSafe-MIA 结果 ([V2.1] q=0.95, N=6, M=6) ======")
        print(f"AUC  = {auc:.4f}"); print(f"阈值 = {best_thr:.4f}"); print(f"Acc  = {acc:.4f}"); print("Confusion matrix [[TN FP][FN TP]]："); print(cm)

        non_scores_np = scores[labels == 0].astype(np.float64)
        if len(non_scores_np) > 0 and len(np.unique(non_scores_np)) > 1:
            thr_5 = float(np.quantile(non_scores_np, 0.95)); pred_5 = (scores >= thr_5).astype(int); cm_5 = confusion_matrix(labels, pred_5); tn, fp, fn, tp = cm_5.ravel(); fpr_actual = fp / (fp + tn + 1e-12); tpr_5 = tp / (tp + fn + 1e-12); prec_5 = precision_score(labels, pred_5, zero_division=0); rec_5  = recall_score(labels, pred_5, zero_division=0); f1_5   = f1_score(labels, pred_5, zero_division=0)
            print("\n------ @5% FPR Metrics ------"); print(f"Threshold @FPR=5% = {thr_5:.4f}  (实际 FPR={fpr_actual*100:.2f}%)"); print(f"TPR (Recall)      = {tpr_5*100:.2f}%"); print(f"Precision          = {prec_5*100:.2f}%"); print(f"F1                 = {f1_5:.4f}"); print("Confusion @5%FPR [[TN FP][FN TP]]："); print(cm_5)
        else: print("\n------ @5% FPR Metrics ------"); print("⚠️ 无法计算 @5% FPR 指标。")
    else: print("\n⚠️ 敏感子维度数量为 0，无法进行探测评分。")

except RuntimeError as e: print(f"\n❌ 主流程执行失败: {e}")
except Exception as e: print(f"\n❌ 主流程发生意外错误: {e}")

## GradNorm

In [ ]:
# ============================================================
# GradNorm MIA for BiomedCLIP (open_clip) — Member vs Non-member (M/N)
# (AUC only, NO CALIB split)
#
# ✅ 目标：
# - 不使用 P/S/N
# - 不使用 CALIB
# - 只输出 AUC(M vs N)
#
# ✅ GradNorm 定义（参考 mimir.attacks.gradnorm 思路）：
# - 对每个样本构造一个 CLIP InfoNCE batch（target + K-1 negatives）
# - 反向传播得到 ∇θ loss
# - 计算选定参数集合的梯度 p-norm，并对参数取均值
# - score = - grad_norm  (梯度越小 -> score 越大 -> 越像 member)
#
# ✅ 数据格式假设（与你现有代码一致）：
# - member obj: {"pmc_tar_url":..., "image_file_path":..., "caption":...}
# - nonmember obj: {"local_image_path":..., "caption":...}
#
# ============================================================

import os, io, json, tarfile, random, shutil, warnings, re, gc
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

# ============================================================
# 0) Paths & Hyperparams
# ============================================================
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

SEED = 42

TOTAL_M = 1000
TOTAL_N = 1000

# CLIP contrastive batch size (target + K-1 negatives)
BATCH_K = 8

# GradNorm hyperparams
PNORM = 2                 # 1 / 2 / np.inf
REDUCE = "mean"           # "mean" or "sum" across params
GRAD_CLIP = None          # e.g. 1.0 or None

# Selective grads: Vision last-N + Text last-M + Projection
VISION_LAST_N = 3
TEXT_LAST_M = 3
USE_SELECTIVE_GRADS = True

# PMC cache dirs
PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

print("\n📋 Hyperparams:")
print(f"  TOTAL M/N     = {TOTAL_M}/{TOTAL_N}")
print(f"  BATCH_K       = {BATCH_K}")
print(f"  PNORM         = {PNORM}")
print(f"  REDUCE        = {REDUCE}")
print(f"  Vision lastN  = {VISION_LAST_N}")
print(f"  Text lastM    = {TEXT_LAST_M}")
print(f"  selective     = {USE_SELECTIVE_GRADS}")

# ============================================================
# 1) Device & perf
# ============================================================
assert torch.cuda.is_available(), "❌ CUDA not available"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"✅ Device: {device}")

# ============================================================
# 2) Load BiomedCLIP
# ============================================================
try:
    import open_clip
except Exception:
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "open_clip_torch>=2.24.0", "ftfy", "regex", "tqdm"
    ])
    import open_clip

MODEL_NAME = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
print(f"⏳ Loading model: {MODEL_NAME}")

model, _, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()
print("✅ BiomedCLIP loaded.")

# ============================================================
# 3) Load data
# ============================================================
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)

rnd = random.Random(SEED)
rnd.shuffle(members_all)
rnd.shuffle(nonmembers_all)

members_all = members_all[:TOTAL_M]
nonmembers_all = nonmembers_all[:TOTAL_N]

print(f"\n📦 Data loaded:")
print(f"   Members:     {len(members_all)}")
print(f"   Non-members: {len(nonmembers_all)}")

if min(len(members_all), len(nonmembers_all)) == 0:
    raise RuntimeError("❌ members or nonmembers is empty")

# Union pool for negatives (same construction for both M and N to avoid bias)
union_pool = list(members_all) + list(nonmembers_all)
rnd = random.Random(SEED + 123)
rnd.shuffle(union_pool)

# ============================================================
# 4) Image utilities (Member: PMC tar; Non-member: local path)
# ============================================================
import requests

def _download_to(path: str, url: str, timeout=120):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = tar_url.strip().split("/")[-1]
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if (not os.path.exists(local)) or (os.path.getsize(local) == 0):
        _download_to(local, tar_url, timeout=180)
    return local

def _normalize_path(s: str):
    return s.replace("\\", "/").lstrip("./")

def _best_ext_variants(path: str):
    exts = [".jpg", ".jpeg", ".png", ".tif", ".tiff"]
    base = os.path.splitext(path)[0]
    cand = []
    for e in exts:
        cand.append(base + e)
        cand.append(base.replace("_lrg", "").replace("_large", "") + e)
        cand.append((base + "_lrg") + e)
        cand.append((base + "_large") + e)
    uniq, seen = [], set()
    for x in cand:
        xl = x.lower()
        if xl not in seen:
            seen.add(xl)
            uniq.append(x)
    return uniq

def _extract_one_from_tar(local_tar: str, file_in_tar: str) -> Optional[str]:
    target = _normalize_path(file_in_tar)
    cand_paths = _best_ext_variants(target)
    try:
        with tarfile.open(local_tar, "r:gz") as tar:
            members = tar.getmembers()
            names_map = {_normalize_path(ti.name): ti for ti in members}
            names_low_map = {_normalize_path(ti.name).lower(): ti for ti in members}

            for cand_path in [target] + cand_paths:
                cand_norm = _normalize_path(cand_path)

                if cand_norm in names_map:
                    ti = names_map[cand_norm]
                    out_path = os.path.join(PMC_EXTRACT_TMP_DIR, Path(ti.name).name)
                    with tar.extractfile(ti) as src, open(out_path, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    return out_path

                cand_low = cand_norm.lower()
                if cand_low in names_low_map:
                    ti = names_low_map[cand_low]
                    out_path = os.path.join(PMC_EXTRACT_TMP_DIR, Path(ti.name).name)
                    with tar.extractfile(ti) as src, open(out_path, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    return out_path
    except Exception:
        return None
    return None

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    tar_url = sample.get("pmc_tar_url")
    path_in = sample.get("image_file_path")
    if not tar_url or not path_in:
        return None
    try:
        local_tar = _ensure_tar_local(tar_url)
        local_img = _extract_one_from_tar(local_tar, path_in)
        if local_img is None:
            return None
        img = Image.open(local_img).convert("RGB")
        try:
            os.remove(local_img)
        except Exception:
            pass
        return img
    except (FileNotFoundError, UnidentifiedImageError, OSError, ValueError,
            requests.exceptions.RequestException, Exception):
        return None

def load_nonmember_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if (not p) or (not os.path.exists(p)):
        return None
    try:
        return Image.open(p).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError, Exception):
        return None

def is_member_obj(obj: Dict[str, Any]) -> bool:
    return ("pmc_tar_url" in obj) and ("image_file_path" in obj)

def load_image_by_obj(obj: Dict[str, Any]) -> Optional[Image.Image]:
    return load_member_image(obj) if is_member_obj(obj) else load_nonmember_image(obj)

# ============================================================
# 5) Selective gradients
# ============================================================
def enable_grads_selectively_CLIP(m) -> List[str]:
    selected = []
    for _, p in m.named_parameters():
        p.requires_grad_(False)

    max_v_layer, max_t_layer = 0, 0
    for name, _ in m.named_parameters():
        mobj = re.search(r"(blocks|resblocks)\.(\d+)\.", name)
        if "visual" in name and mobj:
            max_v_layer = max(max_v_layer, int(mobj.group(2)))
        tobj = re.search(r"bert\.encoder\.layer\.(\d+)\.", name)
        if tobj:
            max_t_layer = max(max_t_layer, int(tobj.group(1)))

    v_start = max(0, max_v_layer - VISION_LAST_N + 1)
    t_start = max(0, max_t_layer - TEXT_LAST_M + 1)

    for name, p in m.named_parameters():
        # Projection layers
        if name == "visual.proj" and p.ndim >= 2:
            p.requires_grad_(True); selected.append(name); continue
        if name == "text_projection.weight" and p.ndim >= 2:
            p.requires_grad_(True); selected.append(name); continue
        if name == "text.proj" and p.ndim >= 2:
            p.requires_grad_(True); selected.append(name); continue

        # Vision last-N
        if "visual" in name and re.search(r"(blocks|resblocks)\.(\d+)\.", name):
            idx = int(re.search(r"(blocks|resblocks)\.(\d+)\.", name).group(2))
            if idx >= v_start and p.ndim >= 2:
                if re.search(r"(attn|mlp|fc|proj|linear|dense|qkv)", name):
                    p.requires_grad_(True); selected.append(name); continue

        # Text last-M (PubMedBERT)
        if "bert.encoder.layer." in name:
            idx = int(re.search(r"bert\.encoder\.layer\.(\d+)\.", name).group(1))
            if idx >= t_start and p.ndim >= 2:
                if re.search(r"(attention|query|key|value|dense|intermediate|output)", name):
                    p.requires_grad_(True); selected.append(name); continue

    return selected

if USE_SELECTIVE_GRADS:
    collect_names = enable_grads_selectively_CLIP(model)
    print(f"\n🔧 Selective grads enabled: params={len(collect_names)}")
else:
    for _, p in model.named_parameters():
        p.requires_grad_(True)
    print("\n🔧 All grads enabled.")

# ============================================================
# 6) Build InfoNCE batch (target + negatives)
# ============================================================
def make_batch_for_item(
    target: Dict[str, Any],
    neg_pool: List[Dict[str, Any]],
    K: int = BATCH_K,
    max_tries: int = 500
) -> Optional[Tuple[torch.Tensor, torch.Tensor]]:
    tgt_img = load_image_by_obj(target)
    if tgt_img is None:
        return None

    negs = []
    tries = 0
    while len(negs) < K - 1 and tries < max_tries:
        cand = random.choice(neg_pool)
        if cand is target:
            tries += 1
            continue
        cim = load_image_by_obj(cand)
        if cim is None:
            tries += 1
            continue
        negs.append(cand)
        tries += 1

    if len(negs) < K - 1:
        return None

    batch_list = [target] + negs
    imgs, caps = [], []

    for obj in batch_list:
        im = load_image_by_obj(obj)
        if im is None:
            im = Image.new("RGB", (224, 224), color="black")
        imgs.append(preprocess_val(im))
        caps.append(str(obj.get("caption", "")).strip())

    images = torch.stack(imgs, dim=0).to(device)

    try:
        toks = tokenizer(caps)
        toks = torch.as_tensor(toks).to(device)
    except Exception:
        return None

    return images, toks

# ============================================================
# 7) InfoNCE loss + backward
# ============================================================
def infonce_backward(images: torch.Tensor, toks: torch.Tensor) -> bool:
    model.train()
    model.zero_grad(set_to_none=True)

    try:
        with torch.cuda.amp.autocast():
            img_feats = model.encode_image(images)
            txt_feats = model.encode_text(toks)

            if (not torch.isfinite(img_feats).all()) or (not torch.isfinite(txt_feats).all()):
                return False

            img_feats = img_feats / (img_feats.norm(dim=-1, keepdim=True) + 1e-12)
            txt_feats = txt_feats / (txt_feats.norm(dim=-1, keepdim=True) + 1e-12)

            logit_scale = model.logit_scale.exp()
            logits_i = logit_scale * img_feats @ txt_feats.t()
            labels = torch.arange(images.size(0), device=images.device)

            loss_i2t = F.cross_entropy(logits_i, labels)
            loss_t2i = F.cross_entropy(logits_i.t(), labels)
            loss = (loss_i2t + loss_t2i) / 2

        if not torch.isfinite(loss):
            return False

        loss.backward()

        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad and p.grad is not None],
                max_norm=float(GRAD_CLIP)
            )

        return True

    except Exception:
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
        return False

# ============================================================
# 8) GradNorm score (core)
# ============================================================
def gradnorm_score_item(
    obj: Dict[str, Any],
    neg_pool: List[Dict[str, Any]],
    K: int = BATCH_K,
    p: float = PNORM,
    reduce: str = REDUCE
) -> float:
    """
    score = - mean_pnorm(gradients)   (larger => more likely member)
    """
    made = make_batch_for_item(obj, neg_pool=neg_pool, K=K)
    if made is None:
        return 0.0

    images, toks = made
    ok = infonce_backward(images, toks)
    if not ok:
        return 0.0

    norms = []
    with torch.no_grad():
        for _, p0 in model.named_parameters():
            if (not p0.requires_grad) or (p0.grad is None):
                continue
            g = p0.grad.detach()
            if not torch.isfinite(g).all():
                continue
            try:
                if p == np.inf:
                    norms.append(g.norm(float("inf")).float())
                else:
                    norms.append(g.norm(float(p)).float())
            except Exception:
                continue

    # cleanup
    model.zero_grad(set_to_none=True)
    model.eval()
    gc.collect()
    torch.cuda.empty_cache()

    if not norms:
        return 0.0

    st = torch.stack(norms)
    grad_norm = st.sum() if reduce == "sum" else st.mean()
    return -float(grad_norm.item())

def score_set(objs: List[Dict[str, Any]], neg_pool: List[Dict[str, Any]], desc: str) -> np.ndarray:
    out = []
    for obj in tqdm(objs, desc=desc):
        out.append(gradnorm_score_item(obj, neg_pool=neg_pool, K=BATCH_K, p=PNORM, reduce=REDUCE))
    return np.array(out, dtype=np.float32)

def auc_pair(pos: np.ndarray, neg: np.ndarray) -> float:
    y = np.array([1]*len(pos) + [0]*len(neg), dtype=np.int32)
    s = np.concatenate([pos, neg], axis=0)
    if len(s) == 0 or np.all(s == s[0]):
        return 0.5
    return float(roc_auc_score(y, s))

# ============================================================
# 9) Run: score all & compute AUC
# ============================================================
print("\n" + "="*90)
print("BiomedCLIP GradNorm MIA — Member vs Non-member (AUC only)")
print("="*90)

try:
    print("\nScoring Members...")
    M_scores = score_set(members_all, union_pool, desc="M (member)")

    print("\nScoring Non-members...")
    N_scores = score_set(nonmembers_all, union_pool, desc="N (nonmember)")

    auc_M_N = auc_pair(M_scores, N_scores)

    print("\n" + "="*90)
    print("📊 Results (GradNorm, AUC only)")
    print("="*90)
    print(f"  AUC(M vs N) = {auc_M_N:.4f}")
    print(f"  mean score M = {float(np.mean(M_scores)):.6f} | std = {float(np.std(M_scores)):.6f}")
    print(f"  mean score N = {float(np.mean(N_scores)):.6f} | std = {float(np.std(N_scores)):.6f}")

    # Save
    OUT_DIR = "/content/drive/MyDrive/biomedclip_mia_attack_results"
    os.makedirs(OUT_DIR, exist_ok=True)
    out_json = os.path.join(OUT_DIR, "biomedclip_gradnorm_mn_auc_only.json")

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump({
            "config": {
                "seed": SEED,
                "TOTAL_M": TOTAL_M,
                "TOTAL_N": TOTAL_N,
                "BATCH_K": BATCH_K,
                "PNORM": float(PNORM) if PNORM != np.inf else "inf",
                "REDUCE": REDUCE,
                "GRAD_CLIP": GRAD_CLIP,
                "VISION_LAST_N": VISION_LAST_N if USE_SELECTIVE_GRADS else "ALL",
                "TEXT_LAST_M": TEXT_LAST_M if USE_SELECTIVE_GRADS else "ALL",
                "MODEL_NAME": MODEL_NAME,
            },
            "results": {
                "auc_M_N": float(auc_M_N),
                "n_M": int(len(M_scores)),
                "n_N": int(len(N_scores)),
                "mean_score_M": float(np.mean(M_scores)),
                "std_score_M": float(np.std(M_scores)),
                "mean_score_N": float(np.mean(N_scores)),
                "std_score_N": float(np.std(N_scores)),
            }
        }, f, ensure_ascii=False, indent=2)

    print(f"\n💾 Saved: {out_json}")

except Exception as e:
    print(f"\n❌ Unexpected error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*90)
print("✅ Done.")
print("="*90)


## NA_PDD

In [ ]:
# ==============================================================================
# ✅ 原始 NA-PDD 逻辑（LLM 版）→ 迁移到 BiomedCLIP（Vision+Text = VLLM/多模态）
#
# 核心：阈值激活 -> 激活神经元集合 -> 统计差异 -> 选层 -> relative ratio score
# ✅ 速度：单样本仅 1 次 forward（可 batch），无 K=8、无 passes=3
# ✅ Hook：Vision last-3 的 MLP GELU + Text last-3 的 BertIntermediate（等价 FFN 激活）
# ==============================================================================

import os, io, json, tarfile, random, warnings, re, gc, time
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
from collections import Counter

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score

warnings.filterwarnings("ignore")

# =========================
# 0) Config
# =========================
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

SEED = 42
TOTAL_MEMBERS = 1000
TOTAL_NONMEMBERS = 1000

CALIB_MEM = 200
CALIB_NON = 200
TEST_MEM  = 800
TEST_NON  = 800

# Hook 只选最后三层（你要求）
VISION_LAST = 3
TEXT_LAST   = 3

# 激活阈值（原始 NA-PDD 用这个）
ACTIVATION_THRESHOLD = 0.0

# 选 top 判别层数
TOP_LAYERS = 10

# relative ratio 候选阈值（可用于 accuracy）
RATIO_THRESHOLD_FOR_ACC = 1.0

# 推理 batch（越大越快，但受显存限制）
BATCH_SIZE = 16

# tar 缓存
PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)

# bytes LRU 缓存（避免重复解 tar）
BYTES_LRU_MAX = 4096

# 可选：先把所有 tar 预下载一次（第一次跑推荐开）
PREFETCH_TARS = True

# =========================
# 1) Device
# =========================
assert torch.cuda.is_available(), "❌ 未检测到 CUDA"
device = "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

tbc.allow_tf32 = True
cudnn.allow_tf32 = True

print(f"✅ Device: {device}")
print(f"📋 Config: TOTAL(M/N)={TOTAL_MEMBERS}/{TOTAL_NONMEMBERS}, CALIB(M/N)={CALIB_MEM}/{CALIB_NON}, TEST(M/N)={TEST_MEM}/{TEST_NON}")
print(f"📋 NA-PDD(orig): thr={ACTIVATION_THRESHOLD}, top_layers={TOP_LAYERS}, batch={BATCH_SIZE}")
print(f"📋 Hook last(V/T)={VISION_LAST}/{TEXT_LAST}, bytes_LRU={BYTES_LRU_MAX}, prefetch_tars={PREFETCH_TARS}")

# =========================
# 2) Load BiomedCLIP (open_clip)
# =========================
try:
    import open_clip
except Exception:
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "open_clip_torch>=2.24.0", "ftfy", "regex", "tqdm"
    ])
    import open_clip

MODEL_NAME = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
print(f"⏳ Loading model: {MODEL_NAME}")
model, _, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()
print("✅ BiomedCLIP loaded")

# =========================
# 3) Load data
# =========================
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)

rnd = random.Random(SEED)
rnd.shuffle(members_all)
rnd.shuffle(nonmembers_all)

members_all = members_all[:TOTAL_MEMBERS]
nonmembers_all = nonmembers_all[:TOTAL_NONMEMBERS]

calib_member = members_all[:CALIB_MEM]
calib_non    = nonmembers_all[:CALIB_NON]
test_member  = members_all[CALIB_MEM:CALIB_MEM+TEST_MEM]
test_non     = nonmembers_all[CALIB_NON:CALIB_NON+TEST_NON]

print(f"\n✅ Data split:")
print(f"   Calib: {len(calib_member)} + {len(calib_non)}")
print(f"   Test : {len(test_member)} + {len(test_non)}")

def get_caption(obj: Dict[str,Any]) -> str:
    c = obj.get("caption", "")
    if isinstance(c, list) and len(c) > 0:
        c = c[0]
    return str(c).strip()

# =========================
# 4) Tar tools + bytes LRU
# =========================
import requests
from functools import lru_cache

def _download_to(path: str, url: str, timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = tar_url.strip().split("/")[-1]
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if not os.path.exists(local) or os.path.getsize(local) == 0:
        _download_to(local, tar_url, timeout=180)
    return local

def _normalize_path(s: str):
    return s.replace("\\", "/").lstrip("./")

def _best_ext_variants(path: str):
    exts = [".jpg", ".jpeg", ".png", ".tif", ".tiff"]
    base = os.path.splitext(path)[0]
    cand = []
    for e in exts:
        cand.append(base + e)
        cand.append(base.replace("_lrg", "").replace("_large", "") + e)
        cand.append((base + "_lrg") + e)
        cand.append((base + "_large") + e)
    uniq, seen = [], set()
    for x in cand:
        lx = x.lower()
        if lx not in seen:
            seen.add(lx)
            uniq.append(x)
    return uniq

@lru_cache(maxsize=BYTES_LRU_MAX)
def _load_member_image_bytes(tar_url: str, path_in: str) -> Optional[bytes]:
    if not tar_url or not path_in:
        return None
    try:
        local_tar = _ensure_tar_local(tar_url)
        target = _normalize_path(path_in)
        cand_paths = _best_ext_variants(target)

        with tarfile.open(local_tar, "r:gz") as tar:
            members = tar.getmembers()
            names_map = {_normalize_path(ti.name): ti for ti in members}
            names_low = {_normalize_path(ti.name).lower(): ti for ti in members}

            for cand in [target] + cand_paths:
                cn = _normalize_path(cand)
                ti = names_map.get(cn, None)
                if ti is None:
                    ti = names_low.get(cn.lower(), None)
                if ti is None:
                    continue
                f = tar.extractfile(ti)
                if f is None:
                    continue
                return f.read()
    except Exception:
        return None
    return None

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    tar_url = sample.get("pmc_tar_url")
    path_in = sample.get("image_file_path")
    if not tar_url or not path_in:
        return None
    b = _load_member_image_bytes(str(tar_url), str(path_in))
    if b is None:
        return None
    try:
        return Image.open(io.BytesIO(b)).convert("RGB")
    except Exception:
        return None

def load_nonmember_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if not p or not os.path.exists(p):
        return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

def load_image(obj: Dict[str,Any], kind: str) -> Optional[Image.Image]:
    return load_member_image(obj) if kind == "member" else load_nonmember_image(obj)

# ---- 可选：预下载所有 tar（第一次跑建议开，能避免中途频繁网络阻塞） ----
if PREFETCH_TARS:
    tar_urls = sorted({str(o.get("pmc_tar_url")) for o in members_all if o.get("pmc_tar_url")})
    print(f"\n📦 Prefetch tars: {len(tar_urls)} unique")
    for u in tqdm(tar_urls, desc="Download tars"):
        try:
            _ensure_tar_local(u)
        except Exception:
            pass
    print("✅ Prefetch done")

# =========================
# 5) Hook：收集“激活神经元索引”（原始 NA-PDD 方式）
# =========================
# 存每个 batch 的激活索引： layer_name -> List[List[int]] (len=B)
batch_activated: Dict[str, List[List[int]]] = {}

def _hook_collect_indices(name: str):
    def hook(module, inp, out):
        # out: [B, Seq, H] or [B, H] or tuple
        if isinstance(out, (tuple, list)):
            out = out[0]
        if not torch.is_tensor(out):
            return

        t = out.detach()

        # 只做阈值激活，不搬大 tensor 到 CPU
        if t.ndim == 3:
            # [B,Seq,H] -> [B,H] 是否被任一 token 激活
            m = (t > ACTIVATION_THRESHOLD).any(dim=1)
        elif t.ndim == 2:
            m = (t > ACTIVATION_THRESHOLD)
        else:
            # 其它：展平到 [B, H]
            if t.ndim >= 1:
                t2 = t.reshape(t.shape[0], -1)
                m = (t2 > ACTIVATION_THRESHOLD)
            else:
                return

        # 每个样本取激活神经元索引（只搬索引，很小）
        idx_lists = []
        for b in range(m.shape[0]):
            idx = torch.nonzero(m[b], as_tuple=False).squeeze(1)
            if idx.numel() == 0:
                idx_lists.append([])
            else:
                idx_lists.append(idx.to("cpu", dtype=torch.int32).tolist())

        batch_activated[name] = idx_lists
    return hook

def register_hooks_last_layers_biomedclip(m: nn.Module, v_last: int, t_last: int):
    hooks = []
    names = []

    # -------- Vision: visual.trunk.blocks.i.mlp.act (GELU) --------
    num_v = len(m.visual.trunk.blocks)
    v_start = max(0, num_v - v_last)

    for i in range(v_start, num_v):
        blk = m.visual.trunk.blocks[i]
        if hasattr(blk, "mlp") and hasattr(blk.mlp, "act"):
            act_mod = blk.mlp.act
            # act_mod 通常就是 nn.GELU
            h = act_mod.register_forward_hook(_hook_collect_indices(f"vision_L{i}_ffn_act"))
            hooks.append(h); names.append(f"vision_L{i}_ffn_act")

    # -------- Text: text.transformer.encoder.layer.i.intermediate (BertIntermediate) --------
    # open_clip 的 text 大多是 HF Transformer/BERT 结构
    # 从你打印名可以看出：text.transformer.encoder.layer.9.intermediate
    text_encoder = getattr(m, "text", None)
    if text_encoder is not None and hasattr(text_encoder, "transformer"):
        enc = text_encoder.transformer
        if hasattr(enc, "encoder") and hasattr(enc.encoder, "layer"):
            layers = enc.encoder.layer
            num_t = len(layers)
            t_start = max(0, num_t - t_last)
            for i in range(t_start, num_t):
                lyr = layers[i]
                if hasattr(lyr, "intermediate"):
                    h = lyr.intermediate.register_forward_hook(_hook_collect_indices(f"text_L{i}_ffn_intermediate"))
                    hooks.append(h); names.append(f"text_L{i}_ffn_intermediate")

    print("\n" + "="*80)
    print("✅ Hooks registered (orig NA-PDD style)")
    print("="*80)
    print(f"  total={len(hooks)}")
    print("  names:", names)
    return hooks, names

hooks, hook_names = register_hooks_last_layers_biomedclip(model, VISION_LAST, TEXT_LAST)

# =========================
# 6) 批量跑模型并拿到每个样本的 activated_neurons（按层存 set）
# =========================
def run_batch_collect(samples: List[Dict[str,Any]], kind: str) -> List[Dict[str,Any]]:
    """对一批 samples：跑 encode_image + encode_text 触发 hooks，返回每个样本层->激活集合"""
    batch_activated.clear()

    imgs = []
    caps = []
    valid_idx = []

    for idx, obj in enumerate(samples):
        im = load_image(obj, kind)
        cap = get_caption(obj)
        if im is None or not cap:
            continue
        imgs.append(preprocess_val(im))
        caps.append(cap)
        valid_idx.append(idx)

    if len(imgs) == 0:
        return []

    images = torch.stack(imgs, dim=0).to(device, non_blocking=True)
    toks = tokenizer(caps)
    toks = torch.as_tensor(toks).to(device, non_blocking=True)

    with torch.no_grad(), torch.cuda.amp.autocast():
        _ = model.encode_image(images)   # 触发 vision hooks
        _ = model.encode_text(toks)      # 触发 text hooks

    # 组织输出（每个样本一个 dict）
    out = []
    B = images.shape[0]

    for b in range(B):
        act_map = {}
        for lname, idx_lists in batch_activated.items():
            if b < len(idx_lists) and len(idx_lists[b]) > 0:
                act_map[lname] = set(idx_lists[b])
        out.append({"activated_neurons": act_map})

    return out

def collect_activations_dataset(samples: List[Dict[str,Any]], kind: str, label: int, batch_size: int) -> List[Dict[str,Any]]:
    results = []
    for i in tqdm(range(0, len(samples), batch_size), desc=f"Collect {kind}"):
        batch = samples[i:i+batch_size]
        batch_res = run_batch_collect(batch, kind)
        for j, r in enumerate(batch_res):
            r["label"] = label
            results.append(r)
        if (i // batch_size) % 50 == 0:
            gc.collect()
            torch.cuda.empty_cache()
    return results

# =========================
# 7) 原始 NA-PDD：统计频率差异，建 reference_patterns
# =========================
def analyze_patterns(member_samples: List[Dict[str,Any]], nonmember_samples: List[Dict[str,Any]]):
    results = {}

    layer_names = set()
    for s in member_samples + nonmember_samples:
        layer_names.update(s["activated_neurons"].keys())

    for layer in layer_names:
        mem_cnt = Counter()
        non_cnt = Counter()

        for s in member_samples:
            if layer in s["activated_neurons"]:
                mem_cnt.update(list(s["activated_neurons"][layer]))

        for s in nonmember_samples:
            if layer in s["activated_neurons"]:
                non_cnt.update(list(s["activated_neurons"][layer]))

        mem_freq = {n: c / max(1, len(member_samples)) for n, c in mem_cnt.items()}
        non_freq = {n: c / max(1, len(nonmember_samples)) for n, c in non_cnt.items()}

        member_dominant = {}
        for n, f in mem_freq.items():
            if (n not in non_freq) or (f > non_freq[n] * 1.5):
                member_dominant[n] = f

        nonmember_dominant = {}
        for n, f in non_freq.items():
            if (n not in mem_freq) or (f > mem_freq[n] * 1.5):
                nonmember_dominant[n] = f

        results[layer] = {
            "member_dominant": member_dominant,
            "nonmember_dominant": nonmember_dominant,
            "member_freq": mem_freq,
            "nonmember_freq": non_freq
        }
    return results

def calculate_layer_scores(reference_patterns: Dict[str,Any]):
    layer_scores = {}
    for layer, d in reference_patterns.items():
        layer_scores[layer] = len(d["member_dominant"]) - len(d["nonmember_dominant"])
    return layer_scores

def select_top_layers(layer_scores: Dict[str,float], top_n: int):
    sorted_layers = sorted(layer_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    return [k for k,_ in sorted_layers[:top_n]]

def ratio_score(sample: Dict[str,Any], reference_patterns: Dict[str,Any], layers: List[str]):
    layers_counted = 0
    total_mem = 0.0
    total_non = 0.0

    for layer in layers:
        if layer not in sample["activated_neurons"]:
            continue
        if layer not in reference_patterns:
            continue

        s_neurons = sample["activated_neurons"][layer]
        if not s_neurons:
            continue

        mem_dom = set(reference_patterns[layer]["member_dominant"].keys())
        non_dom = set(reference_patterns[layer]["nonmember_dominant"].keys())

        mem_overlap = len(s_neurons & mem_dom)
        non_overlap = len(s_neurons & non_dom)

        mem_ratio = mem_overlap / len(mem_dom) if len(mem_dom) > 0 else 0.0
        non_ratio = non_overlap / len(non_dom) if len(non_dom) > 0 else 0.0

        total_mem += mem_ratio
        total_non += non_ratio
        layers_counted += 1

    if layers_counted == 0:
        return 0.0

    avg_mem = total_mem / layers_counted
    avg_non = total_non / layers_counted
    if avg_non == 0:
        return float("inf")
    return avg_mem / avg_non

# =========================
# 8) 主流程：Calib 建 reference；Test 打分评估
# =========================
print("\n" + "="*90)
print("🚀 NA-PDD (orig) on BiomedCLIP — Calib collect")
print("="*90)

calib_mem_acts = collect_activations_dataset(calib_member, "member", label=1, batch_size=BATCH_SIZE)
calib_non_acts = collect_activations_dataset(calib_non,    "nonmember", label=0, batch_size=BATCH_SIZE)

print("\n" + "="*90)
print("🔧 Build reference patterns")
print("="*90)

reference_patterns = analyze_patterns(calib_mem_acts, calib_non_acts)
layer_scores = calculate_layer_scores(reference_patterns)
discriminative_layers = select_top_layers(layer_scores, TOP_LAYERS)

print("✅ Selected layers:")
for x in discriminative_layers:
    print("  -", x, "score=", layer_scores.get(x, 0))

print("\n" + "="*90)
print("🧪 NA-PDD (orig) on BiomedCLIP — Test collect")
print("="*90)

test_mem_acts = collect_activations_dataset(test_member, "member", label=1, batch_size=BATCH_SIZE)
test_non_acts = collect_activations_dataset(test_non,    "nonmember", label=0, batch_size=BATCH_SIZE)

test_all = test_mem_acts + test_non_acts
y = np.array([s["label"] for s in test_all], dtype=np.int32)

scores = []
for s in tqdm(test_all, desc="Score"):
    r = ratio_score(s, reference_patterns, discriminative_layers)
    # 过滤 inf 方便ROC；也可以保留并替换成一个很大值
    if r == float("inf") or np.isnan(r):
        r = 1e6
    scores.append(float(r))

scores = np.array(scores, dtype=np.float64)

# AUC + TPR@5%FPR
if len(np.unique(y)) < 2:
    auc = 0.5
    tpr5 = 0.0
else:
    fpr, tpr, thr = roc_curve(y, scores)
    auc = roc_auc_score(y, scores)
    idx = np.where(fpr >= 0.05)[0]
    tpr5 = float(tpr[idx[0]]) if len(idx) > 0 else 0.0

# Accuracy（用固定阈值 ratio>=1.0，与你原始代码一致）
pred = (scores >= RATIO_THRESHOLD_FOR_ACC).astype(np.int32)
acc = accuracy_score(y, pred)

print("\n" + "="*90)
print("📊 Results (orig NA-PDD -> BiomedCLIP)")
print("="*90)
print(f"  AUC               = {auc:.4f}")
print(f"  TPR@5%FPR         = {tpr5*100:.2f}%")
print(f"  Accuracy(ratio>=1)= {acc:.4f}")
print(f"  score mean (mem)  = {scores[y==1].mean():.4f} ± {scores[y==1].std():.4f}  n={int((y==1).sum())}")
print(f"  score mean (non)  = {scores[y==0].mean():.4f} ± {scores[y==0].std():.4f}  n={int((y==0).sum())}")

# cleanup hooks
for h in hooks:
    try:
        h.remove()
    except:
        pass

print("\n✅ Done.")


## Loss MIA

In [ ]:
# ===============================================
# Contrastive-similarity MIA for BiomedCLIP/CLIP
#  - Members : PMC tar + caption（members_1k.json）
#  - Non-mem : ROCO 本地（nonmembers_roco_1k.json）
#  - Model   : BiomedCLIP via open_clip (HF Hub)
#  - Score   : cos(image_emb, text_emb)
# ===============================================

import os, io, json, tarfile, random, shutil, math, warnings
from pathlib import Path
from typing import Dict, Any, Optional, List

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix

import torch
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

warnings.filterwarnings("ignore")

# ---------------------------
# 0) 路径与超参
# ---------------------------
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

# 下载/缓存目录（members 的远端 *.tar.gz 会按需下载并本地缓存）
PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

# ---------------------------
# 1) 设备 & 性能设置
# ---------------------------
assert torch.cuda.is_available(), "未检测到 CUDA，请在运行时中启用 GPU（T4/A100/H100）"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True
print(f"✅ Device: {device}")

# ---------------------------
# 2) 加载 BiomedCLIP (open_clip)
# ---------------------------
# 依赖：open_clip_torch
try:
    import open_clip
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch>=2.24.0", "ftfy", "regex", "tqdm"])
    import open_clip

# 直接通过 HF Hub 前缀拉权重 + 处理器 + 分词器
MODEL_NAME = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
print(f"⏳ 加载模型：{MODEL_NAME}")
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()
print("✅ BiomedCLIP 加载完成")

# ---------------------------
# 3) 读取数据（上一阶段你已生成）
# ---------------------------
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all    = load_json_list(MEMBERS_JSON)      # [{'pair_id','caption','pmc_tar_url','image_file_path',...}, ...]
nonmembers_all = load_json_list(NONMEMBERS_JSON)   # [{'local_image_path','caption'}, ...]
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)

members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 样本就绪：members={len(members)} | non-members={len(nonmembers)}")

# ---------------------------
# 4) PMC tar 读取 & 本地图读取
# ---------------------------
import requests

def _download_to(path: str, url: str, timeout=120):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk: f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = tar_url.strip().split("/")[-1]
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if not os.path.exists(local) or os.path.getsize(local) == 0:
        _download_to(local, tar_url, timeout=180)
    return local

def _extract_one_from_tar(local_tar: str, file_in_tar: str) -> str:
    target_norm = file_in_tar.replace("\\", "/")
    with tarfile.open(local_tar, "r:gz") as tar:
        member = None
        for ti in tar.getmembers():
            if ti.name.replace("\\", "/") == target_norm:
                member = ti; break
        if member is None:
            # 扩展名/大小写兜底
            cands = [target_norm,
                     target_norm.replace(".jpg",".JPG"),
                     target_norm.replace(".jpg",".png"),
                     target_norm.replace(".JPG",".jpg")]
            for ti in tar.getmembers():
                if ti.name.replace("\\","/") in cands:
                    member = ti; break
        if member is None:
            raise FileNotFoundError(f"{file_in_tar} 不在 TAR 包中：{local_tar}")
        Path(PMC_EXTRACT_TMP_DIR).mkdir(parents=True, exist_ok=True)
        out_path = os.path.join(PMC_EXTRACT_TMP_DIR, Path(member.name).name)
        with tar.extractfile(member) as src, open(out_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    return out_path

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    tar_url = sample.get("pmc_tar_url"); path_in = sample.get("image_file_path")
    if not tar_url or not path_in: return None
    local_tar = _ensure_tar_local(tar_url)
    local_img = _extract_one_from_tar(local_tar, path_in)
    return Image.open(local_img).convert("RGB")

def load_nonmember_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if not p or not os.path.exists(p): return None
    return Image.open(p).convert("RGB")

# ---------------------------
# 5) 编码工具（批处理）
# ---------------------------
@torch.inference_mode()
def encode_image_batch(pils: List[Image.Image], batch_size: int = 32) -> torch.Tensor:
    """将 PIL 列表转为图像嵌入（L2 归一化后返回 CPU float32）"""
    out = []
    for i in range(0, len(pils), batch_size):
        chunk = pils[i:i+batch_size]
        # 注意：一定要对每张 PIL 应用 preprocess_val，再 stack
        imgs = torch.stack([preprocess_val(im) for im in chunk]).to(device)
        with torch.cuda.amp.autocast():
            feats = model.encode_image(imgs)
            feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
        out.append(feats.float().cpu())
    return torch.cat(out, dim=0) if out else torch.zeros((0, model.text_projection.shape[1]))

@torch.inference_mode()
def encode_text_batch(texts: List[str], batch_size: int = 64, max_len: int = 256) -> torch.Tensor:
    """将 caption 列表转为文本嵌入（L2 归一化后返回 CPU float32）"""
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        toks = tokenizer(chunk)              # tokenizer 接受 list[str]
        toks = torch.as_tensor(toks)         # 可能是 np.ndarray，转 torch
        toks = toks[:, :max_len].to(device)  # 截断到最大长度
        with torch.cuda.amp.autocast():
            feats = model.encode_text(toks)
            feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
        out.append(feats.float().cpu())
    return torch.cat(out, dim=0) if out else torch.zeros((0, model.text_projection.shape[1]))

# ---------------------------
# 6) 收集样本（读取图像+caption）
# ---------------------------
def collect_pairs(samples: List[Dict[str,Any]], loader, kind="member"):
    imgs, caps = [], []
    fail = {"image":0, "other":0}
    for s in tqdm(samples, desc=f"load-{kind}"):
        cap = str(s.get("caption","")).strip()
        if not cap:
            fail["other"] += 1; continue
        try:
            im = loader(s)
            if im is None: fail["image"] += 1; continue
        except (UnidentifiedImageError, FileNotFoundError, tarfile.ReadError):
            fail["image"] += 1; continue
        except Exception:
            fail["image"] += 1; continue
        imgs.append(im); caps.append(cap)
    return imgs, caps, fail

print("\n⏳ 读取 members（TAR 下载/解包可能略慢）")
mem_imgs, mem_caps, mem_fail = collect_pairs(members, load_member_image, kind="member")
print("⏳ 读取 non-members（本地）")
non_imgs, non_caps, non_fail = collect_pairs(nonmembers, load_nonmember_image, kind="nonmember")

print(f"\n📊 可用样本：members={len(mem_imgs)} | nonmembers={len(non_imgs)}")
print(f"❗ 失败统计：member={mem_fail} | nonmember={non_fail}")
if len(mem_imgs)==0 or len(non_imgs)==0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

# ---------------------------
# 7) 编码 & 相似度
# ---------------------------
print("\n⏳ 编码图像 …")
mem_img_emb  = encode_image_batch(mem_imgs, batch_size=32)
non_img_emb  = encode_image_batch(non_imgs, batch_size=32)

print("⏳ 编码文本 …")
mem_txt_emb  = encode_text_batch(mem_caps, batch_size=64, max_len=256)
non_txt_emb  = encode_text_batch(non_caps, batch_size=64, max_len=256)

def cosine_batch(a: torch.Tensor, b: torch.Tensor) -> np.ndarray:
    # a,b: [N, D] 已归一化向量，逐行点积 -> cos 相似度
    sim = (a * b).sum(dim=-1)
    return sim.numpy()

mem_scores  = cosine_batch(mem_img_emb,  mem_txt_emb)
non_scores  = cosine_batch(non_img_emb,  non_txt_emb)

# ---------------------------
# 8) 评估（AUC / 阈值 / 混淆矩阵 / 统计）
# ---------------------------
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix

y_true = np.array([1]*len(mem_scores) + [0]*len(non_scores))
scores = np.concatenate([mem_scores, non_scores], axis=0)

auc = roc_auc_score(y_true, scores)
fpr, tpr, thr = roc_curve(y_true, scores)
best_idx = np.argmax(tpr - fpr)      # Youden J
best_thr = thr[best_idx]
y_pred = (scores >= best_thr).astype(int)
acc = accuracy_score(y_true, y_pred)
cm  = confusion_matrix(y_true, y_pred)

def stats(arr, name):
    arr = np.array(arr)
    print(f"{name}: n={len(arr)}, mean={arr.mean():.4f}, median={np.median(arr):.4f}, "
          f"std={arr.std():.4f}, min={arr.min():.4f}, max={arr.max():.4f}")

print("\n====== Contrastive-MIA 结果 (BiomedCLIP) ======")
print(f"AUC = {auc:.4f}")
print(f"Best threshold (Youden J) = {best_thr:.4f}")
print(f"Accuracy = {acc:.4f}")
print("Confusion matrix [[TN FP][FN TP]]：")
print(cm)
print()
stats(mem_scores,   "members cos-sim")
stats(non_scores,   "non-members cos-sim")


## Entropy

In [ ]:
# ===============================================
# Entropy-based MIA for BiomedCLIP (open_clip)
#  - Data   : members_1k.json / nonmembers_roco_1k.json
#  - Model  : microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224
#  - Score  : negative entropy over K-way softmax (avg over R trials)
# ===============================================

import os, json, tarfile, random, shutil, warnings
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
# 新增指标
from sklearn.metrics import precision_score, recall_score, f1_score

import torch
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Paths & hyperparams
# ---------------------------
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

# candidate pool per trial: 1 pos + (K-1) neg captions
K = 32
# how many trials per sample (variance reduction)
R = 8

# TAR cache
PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

# ---------------------------
# 1) Device
# ---------------------------
assert torch.cuda.is_available(), "需要 CUDA GPU"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

# ---------------------------
# 2) Load BiomedCLIP via open_clip
# ---------------------------
try:
    import open_clip
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch>=2.24.0", "ftfy", "regex", "tqdm"])
    import open_clip

MODEL_NAME = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
print(f"⏳ Loading model: {MODEL_NAME}")
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()
# logit scale
with torch.no_grad():
    LOGIT_SCALE = model.logit_scale.exp().item() if hasattr(model, "logit_scale") else 1.0
print(f"✅ BiomedCLIP ready | logit_scale={LOGIT_SCALE:.3f}")

# ---------------------------
# 3) Data I/O
# ---------------------------
import requests

def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all    = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)
members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 Samples: members={len(members)} | non-members={len(nonmembers)}")

def _download_to(path: str, url: str, timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path,"wb") as f:
            for ch in r.iter_content(chunk_size=1<<20):
                if ch: f.write(ch)

def _ensure_tar_local(url: str) -> str:
    local=os.path.join(PMC_TAR_CACHE_DIR, url.strip().split("/")[-1])
    if not os.path.exists(local) or os.path.getsize(local)==0: _download_to(local,url)
    return local

def _extract_one(local_tar: str, file_in_tar: str) -> str:
    target=file_in_tar.replace("\\","/")
    with tarfile.open(local_tar,"r:gz") as tar:
        mem=None
        for ti in tar.getmembers():
            if ti.name.replace("\\","/")==target: mem=ti; break
        if mem is None:
            for alt in [target.replace(".jpg",".JPG"),
                        target.replace(".jpg",".png"),
                        target.replace(".JPG",".jpg")]:
                for ti in tar.getmembers():
                    if ti.name.replace("\\","/")==alt: mem=ti; break
                if mem: break
        if mem is None: raise FileNotFoundError(f"{file_in_tar} not in {local_tar}")
        outp=os.path.join(PMC_EXTRACT_TMP_DIR, Path(mem.name).name)
        with tar.extractfile(mem) as src, open(outp,"wb") as dst: shutil.copyfileobj(src,dst)
    return outp

from PIL import Image, UnidentifiedImageError

def load_member_image(s: Dict[str,Any]) -> Optional[Image.Image]:
    url=s.get("pmc_tar_url"); p=s.get("image_file_path")
    if not url or not p: return None
    try:
        imgp=_extract_one(_ensure_tar_local(url), p)
        return Image.open(imgp).convert("RGB")
    except Exception:
        return None

def load_nonmember_image(s: Dict[str,Any]) -> Optional[Image.Image]:
    p=s.get("local_image_path")
    if not p or not os.path.exists(p): return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

def collect_pairs(samples, loader, tag="member"):
    imgs, caps = [], []
    fail = {"image":0, "caption":0}
    for s in tqdm(samples, desc=f"load-{tag}"):
        cap=str(s.get("caption","")).strip()
        if not cap: fail["caption"]+=1; continue
        im=loader(s)
        if im is None: fail["image"]+=1; continue
        imgs.append(im); caps.append(cap)
    print(f"📊 {tag}: ok={len(imgs)} | fail={fail}")
    return imgs, caps

print("\n⏳ Loading images …")
mem_imgs, mem_caps = collect_pairs(members, load_member_image, "member")
non_imgs, non_caps = collect_pairs(nonmembers, load_nonmember_image, "nonmember")
if len(mem_imgs)==0 or len(non_imgs)==0: raise RuntimeError("有效样本为 0")

# ---------------------------
# 4) Encode
# ---------------------------
@torch.inference_mode()
def encode_image_batch(pils: List[Image.Image], bs=64) -> torch.Tensor:
    out=[]
    for i in range(0,len(pils),bs):
        batch=torch.stack([preprocess_val(im) for im in pils[i:i+bs]]).to(device)
        with torch.cuda.amp.autocast():
            z=model.encode_image(batch)
            z=z/(z.norm(dim=-1,keepdim=True)+1e-12)
        out.append(z.float().cpu())
    return torch.cat(out,0)

@torch.inference_mode()
def encode_text_batch(texts: List[str], bs=128, max_len=256) -> torch.Tensor:
    out=[]
    for i in range(0,len(texts),bs):
        toks=tokenizer(texts[i:i+bs])
        toks=torch.as_tensor(toks)[:, :max_len].to(device)
        with torch.cuda.amp.autocast():
            z=model.encode_text(toks)
            z=z/(z.norm(dim=-1,keepdim=True)+1e-12)
        out.append(z.float().cpu())
    return torch.cat(out,0)

print("\n⏳ Encoding …")
mem_img = encode_image_batch(mem_imgs, bs=64)
non_img = encode_image_batch(non_imgs, bs=64)
mem_txt = encode_text_batch(mem_caps, bs=128, max_len=256)
non_txt = encode_text_batch(non_caps, bs=128, max_len=256)

# caption pool for negatives
all_txt = torch.cat([mem_txt, non_txt], dim=0)            # [Nc, D]
Nc = all_txt.size(0)
all_txt_gpu = all_txt.to(device)  # keep on GPU for fast gather

# ---------------------------
# 5) Entropy score
# ---------------------------
def entropy_over_candidates(img_vec: torch.Tensor, pos_idx: int, K:int=32, R:int=8) -> float:
    """
    img_vec: [D] on CPU; pos_idx: index in [0, Nc)
    Return mean negative entropy over R trials.
    """
    D = img_vec.numel()
    img_d = img_vec.to(device)  # [D]
    scores=[]
    for _ in range(R):
        # sample (K-1) negatives excluding pos_idx
        neg_pool = np.delete(np.arange(Nc), pos_idx)
        neg_idx  = np.random.choice(neg_pool, size=K-1, replace=False)
        cand_idx = np.concatenate([[pos_idx], neg_idx])
        cand = all_txt_gpu[cand_idx]                 # [K, D]
        # logits = logit_scale * cos
        logits = LOGIT_SCALE * (cand @ img_d)        # [K]
        p = torch.softmax(logits, dim=0)
        ent = -(p * (p.clamp_min(1e-9).log())).sum()
        scores.append(float((-ent).item()))          # negative entropy
    return float(np.mean(scores))

# ---------------------------
# 6) Score all & evaluate
# ---------------------------
mem_scores, non_scores = [], []

print("\n⏳ Scoring (Entropy)…")
# mapping pos index in caption pool
for i in tqdm(range(mem_img.size(0)), desc="members"):
    s = entropy_over_candidates(mem_img[i], pos_idx=i, K=K, R=R)
    mem_scores.append(s)
for j in tqdm(range(non_img.size(0)), desc="nonmembers"):
    pos = mem_txt.size(0) + j
    s = entropy_over_candidates(non_img[j], pos_idx=pos, K=K, R=R)
    non_scores.append(s)

print(f"\n有效样本数：mem={len(mem_scores)} non={len(non_scores)}")
y_true  = np.array([1]*len(mem_scores) + [0]*len(non_scores))
scores  = np.array(mem_scores + non_scores)
auc = roc_auc_score(y_true, scores)
fpr, tpr, thr = roc_curve(y_true, scores); j=np.argmax(tpr-fpr)
acc = accuracy_score(y_true, (scores>=thr[j]).astype(int))
cm  = confusion_matrix(y_true, (scores>=thr[j]).astype(int))

print("\n====== Entropy-MIA 结果 ======")
print(f"AUC = {auc:.4f}")
print(f"Best threshold (Youden J) = {thr[j]:.4f}")
print(f"Accuracy = {acc:.4f}")
print("Confusion matrix [[TN FP][FN TP]]:\n", cm)

# -------- 新增：@5% FPR 的 TPR / Precision / Recall / F1 --------
# 阈值取“非成员分数”的 95% 分位；然后在全体上计算指标
thr_5 = float(np.quantile(non_scores, 0.95))
y_pred_5 = (scores >= thr_5).astype(int)

cm_5 = confusion_matrix(y_true, y_pred_5)
tn, fp, fn, tp = cm_5.ravel()
fpr_actual = fp / (fp + tn + 1e-12)
tpr_5 = tp / (tp + fn + 1e-12)
prec_5 = precision_score(y_true, y_pred_5, zero_division=0)
rec_5  = recall_score(y_true, y_pred_5, zero_division=0)   # 与 tpr_5 等价
f1_5   = f1_score(y_true, y_pred_5, zero_division=0)

print("\n------ @5% FPR Metrics ------")
print(f"Threshold @FPR=5% = {thr_5:.4f}  (实际 FPR={fpr_actual*100:.2f}%)")
print(f"TPR (Recall)      = {tpr_5*100:.2f}%")
print(f"Precision          = {prec_5*100:.2f}%")
print(f"F1                 = {f1_5:.4f}")
print("Confusion @5%FPR [[TN FP][FN TP]]：")
print(cm_5)


## Min-K

In [ ]:
# ===============================================
# Min-K% MIA for BiomedCLIP (open_clip)
#  - Score: run R trials -> NLL_pos per trial -> take mean of smallest ceil(R*p) NLLs
#           score = -mean_min_p (bigger => member-like)
# ===============================================

import os, json, tarfile, random, shutil, warnings
from pathlib import Path
from typing import Dict, Any, Optional, List
import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
# 新增：@5%FPR 处 TPR/Precision/Recall/F1
from sklearn.metrics import precision_score, recall_score, f1_score

import torch
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Paths & hyperparams
# ---------------------------
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

K_candidates = 32
R_trials     = 8
P_frac       = 0.25   # 使用最小的前 25% 的 NLL

PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

# ---------------------------
# 1) Device & model
# ---------------------------
assert torch.cuda.is_available(), "需要 CUDA GPU"
device="cuda"
tbc.allow_tf32=True; cudnn.allow_tf32=True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

try:
    import open_clip
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable,"-m","pip","install","-q","open_clip_torch>=2.24.0","ftfy","regex","tqdm"])
    import open_clip

MODEL_NAME='hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
print(f"⏳ Loading model: {MODEL_NAME}")
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()
with torch.no_grad():
    LOGIT_SCALE = model.logit_scale.exp().item() if hasattr(model,"logit_scale") else 1.0
print(f"✅ BiomedCLIP ready | logit_scale={LOGIT_SCALE:.3f}")

# ---------------------------
# 2) Data & I/O
# ---------------------------
import requests
def load_json_list(p):
    with open(p,"r",encoding="utf-8") as f: return json.load(f)

members_all    = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)
members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 Samples: members={len(members)} | non-members={len(nonmembers)}")

def _download_to(path,url,timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url,stream=True,timeout=timeout) as r:
        r.raise_for_status()
        with open(path,"wb") as f:
            for ch in r.iter_content(chunk_size=1<<20):
                if ch: f.write(ch)
def _ensure_tar_local(url):
    local=os.path.join(PMC_TAR_CACHE_DIR, url.strip().split("/")[-1])
    if not os.path.exists(local) or os.path.getsize(local)==0: _download_to(local,url)
    return local
def _extract_one(local_tar,file_in_tar):
    target=file_in_tar.replace("\\","/")
    with tarfile.open(local_tar,"r:gz") as tar:
        mem=None
        for ti in tar.getmembers():
            if ti.name.replace("\\","/")==target: mem=ti; break
        if mem is None:
            for alt in [target.replace(".jpg",".JPG"),target.replace(".jpg",".png"),target.replace(".JPG",".jpg")]:
                for ti in tar.getmembers():
                    if ti.name.replace("\\","/")==alt: mem=ti; break
                if mem: break
        if mem is None: raise FileNotFoundError(f"{file_in_tar} not in {local_tar}")
        outp=os.path.join(PMC_EXTRACT_TMP_DIR, Path(mem.name).name)
        with tar.extractfile(mem) as src, open(outp,"wb") as dst: shutil.copyfileobj(src,dst)
    return outp

from PIL import Image
def load_member_image(s):
    url=s.get("pmc_tar_url"); p=s.get("image_file_path")
    if not url or not p: return None
    try: return Image.open(_extract_one(_ensure_tar_local(url),p)).convert("RGB")
    except Exception: return None
def load_nonmember_image(s):
    p=s.get("local_image_path")
    if not p or not os.path.exists(p): return None
    try: return Image.open(p).convert("RGB")
    except Exception: return None

def collect_pairs(samples, loader, tag):
    imgs, caps = [], []
    for s in tqdm(samples, desc=f"load-{tag}"):
        cap=str(s.get("caption","")).strip()
        if not cap: continue
        im=loader(s)
        if im is None: continue
        imgs.append(im); caps.append(cap)
    print(f"📊 {tag}: ok={len(imgs)}")
    return imgs, caps

mem_imgs, mem_caps = collect_pairs(members, load_member_image, "member")
non_imgs, non_caps = collect_pairs(nonmembers, load_nonmember_image, "nonmember")
if len(mem_imgs)==0 or len(non_imgs)==0: raise RuntimeError("有效样本为 0")

# ---------------------------
# 3) Encode
# ---------------------------
@torch.inference_mode()
def encode_image_batch(pils, bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        batch=torch.stack([preprocess_val(im) for im in pils[i:i+bs]]).to(device)
        with torch.cuda.amp.autocast():
            z=model.encode_image(batch); z=z/(z.norm(dim=-1,keepdim=True)+1e-12)
        out.append(z.float().cpu())
    return torch.cat(out,0)

@torch.inference_mode()
def encode_text_batch(texts, bs=128, max_len=256):
    out=[]
    for i in range(0,len(texts),bs):
        toks=tokenizer(texts[i:i+bs])
        toks=torch.as_tensor(toks)[:, :max_len].to(device)
        with torch.cuda.amp.autocast():
            z=model.encode_text(toks); z=z/(z.norm(dim=-1,keepdim=True)+1e-12)
        out.append(z.float().cpu())
    return torch.cat(out,0)

print("\n⏳ Encoding …")
mem_img = encode_image_batch(mem_imgs)
non_img = encode_image_batch(non_imgs)
mem_txt = encode_text_batch(mem_caps)
non_txt = encode_text_batch(non_caps)
all_txt = torch.cat([mem_txt, non_txt], 0)
Nc = all_txt.size(0)
all_txt_gpu = all_txt.to(device)

# ---------------------------
# 4) Min-K% score
# ---------------------------
def minK_frac_score(img_vec: torch.Tensor, pos_idx: int, Kcand:int, R:int, p:float) -> float:
    img_d = img_vec.to(device)
    nlls=[]
    for _ in range(R):
        neg_pool=np.delete(np.arange(Nc), pos_idx)
        neg_idx =np.random.choice(neg_pool, size=Kcand-1, replace=False)
        cand_idx=np.concatenate([[pos_idx],neg_idx])
        cand=all_txt_gpu[cand_idx]
        logits=LOGIT_SCALE*(cand @ img_d)
        logp = torch.log_softmax(logits, dim=0)
        nll  = -logp[0]
        nlls.append(float(nll.item()))
    nlls=np.array(nlls)
    k=max(1, int(np.ceil(len(nlls)*p)))
    best=np.partition(nlls, k-1)[:k].mean()
    return float(-best)

# ---------------------------
# 5) Score & eval
# ---------------------------
mem_scores, non_scores = [], []
print("\n⏳ Scoring (Min-K%) …")
for i in tqdm(range(mem_img.size(0)), desc="members"):
    mem_scores.append(minK_frac_score(mem_img[i], i, K_candidates, R_trials, P_frac))
for j in tqdm(range(non_img.size(0)), desc="nonmembers"):
    pos=mem_txt.size(0)+j
    non_scores.append(minK_frac_score(non_img[j], pos, K_candidates, R_trials, P_frac))

print(f"\n有效样本数：mem={len(mem_scores)} non={len(non_scores)}")
y=np.array([1]*len(mem_scores)+[0]*len(non_scores))
scores=np.array(mem_scores+non_scores)
auc=roc_auc_score(y,scores)
fpr,tpr,thr=roc_curve(y,scores); j=np.argmax(tpr-fpr)
acc=accuracy_score(y,(scores>=thr[j]).astype(int))
cm=confusion_matrix(y,(scores>=thr[j]).astype(int))
print("\n====== Min-K% MIA 结果 ======")
print(f"AUC = {auc:.4f}")
print(f"Best threshold (Youden J) = {thr[j]:.4f}")
print(f"Accuracy = {acc:.4f}")
print("Confusion matrix [[TN FP][FN TP]]:\n", cm)

# -------- 新增：@5% FPR 的 TPR / Precision / Recall / F1 --------
# 采用“非成员分数 95% 分位数”作为阈值（实际 FPR 可能略偏离 5%，便于快速复现实验）
thr_5 = float(np.quantile(non_scores, 0.95))
y_pred_5 = (scores >= thr_5).astype(int)

cm_5 = confusion_matrix(y, y_pred_5)
tn, fp, fn, tp = cm_5.ravel()
fpr_actual = fp / (fp + tn + 1e-12)
tpr_5 = tp / (tp + fn + 1e-12)
prec_5 = precision_score(y, y_pred_5, zero_division=0)
rec_5  = recall_score(y, y_pred_5, zero_division=0)   # = TPR
f1_5   = f1_score(y, y_pred_5, zero_division=0)

print("\n------ @5% FPR Metrics ------")
print(f"Threshold @FPR=5% = {thr_5:.4f}  (实际 FPR={fpr_actual*100:.2f}%)")
print(f"TPR (Recall)      = {tpr_5*100:.2f}%")
print(f"Precision          = {prec_5*100:.2f}%")
print(f"F1                 = {f1_5:.4f}")
print("Confusion @5%FPR [[TN FP][FN TP]]：")
print(cm_5)


## Min-K++

In [ ]:
# ===============================================
# Min-K++ MIA for BiomedCLIP (open_clip)
#  - Score: for each image, build hard-negative pool by nearest texts (excl. pos)
#           run R trials; in each trial:
#             - sample K-1 negatives (HARD_RATIO from hard-pool, rest from global)
#             - evaluate NLL under multiple softmax temperatures TAUS
#           collect R * len(TAUS) NLLs -> take mean of smallest ceil(p * count)
#           score = -mean_min_p (bigger => member-like)
# ===============================================

import os, json, tarfile, random, shutil, warnings
from pathlib import Path
from typing import Dict, Any, Optional, List
import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
# 新增：@5%FPR 处的 Precision / Recall / F1
from sklearn.metrics import precision_score, recall_score, f1_score

import torch
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Paths & hyperparams
# ---------------------------
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

# candidates per trial: 1 pos + (K-1) neg captions
K = 32
# how many trials per sample (variance reduction)
R = 8
# robust aggregation: smallest p-fraction
P_FRAC = 0.25

# Min-K++ extras
HARD_TOP    = 256    # size of hard negative pool (top-N nearest texts to the image, excl. positive)
HARD_RATIO  = 0.75   # fraction of negatives drawn from hard pool each trial
TAUS        = [1.0, 0.8, 1.2]  # temperature jitters; softmax(logits / tau)

# TAR cache
PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

# ---------------------------
# 1) Device & model
# ---------------------------
assert torch.cuda.is_available(), "需要 CUDA GPU"
device="cuda"
tbc.allow_tf32=True; cudnn.allow_tf32=True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

try:
    import open_clip
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable,"-m","pip","install","-q","open_clip_torch>=2.24.0","ftfy","regex","tqdm"])
    import open_clip

MODEL_NAME='hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
print(f"⏳ Loading model: {MODEL_NAME}")
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()
with torch.no_grad():
    LOGIT_SCALE = model.logit_scale.exp().item() if hasattr(model,"logit_scale") else 1.0
print(f"✅ BiomedCLIP ready | logit_scale={LOGIT_SCALE:.3f}")

# ---------------------------
# 2) Data & I/O
# ---------------------------
import requests
def load_json_list(p):
    with open(p,"r",encoding="utf-8") as f: return json.load(f)

members_all    = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)
members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 Samples: members={len(members)} | non-members={len(nonmembers)}")

def _download_to(path,url,timeout=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url,stream=True,timeout=timeout) as r:
        r.raise_for_status()
        with open(path,"wb") as f:
            for ch in r.iter_content(chunk_size=1<<20):
                if ch: f.write(ch)
def _ensure_tar_local(url):
    local=os.path.join(PMC_TAR_CACHE_DIR, url.strip().split("/")[-1])
    if not os.path.exists(local) or os.path.getsize(local)==0: _download_to(local,url)
    return local
def _extract_one(local_tar,file_in_tar):
    target=file_in_tar.replace("\\","/")
    with tarfile.open(local_tar,"r:gz") as tar:
        mem=None
        for ti in tar.getmembers():
            if ti.name.replace("\\","/")==target: mem=ti; break
        if mem is None:
            for alt in [target.replace(".jpg",".JPG"),target.replace(".jpg",".png"),target.replace(".JPG",".jpg")]:
                for ti in tar.getmembers():
                    if ti.name.replace("\\","/")==alt: mem=ti; break
                if mem: break
        if mem is None: raise FileNotFoundError(f"{file_in_tar} not in {local_tar}")
        outp=os.path.join(PMC_EXTRACT_TMP_DIR, Path(mem.name).name)
        with tar.extractfile(mem) as src, open(outp,"wb") as dst: shutil.copyfileobj(src,dst)
    return outp

def load_member_image(s):
    url=s.get("pmc_tar_url"); p=s.get("image_file_path")
    if not url or not p: return None
    try: return Image.open(_extract_one(_ensure_tar_local(url),p)).convert("RGB")
    except Exception: return None
def load_nonmember_image(s):
    p=s.get("local_image_path")
    if not p or not os.path.exists(p): return None
    try: return Image.open(p).convert("RGB")
    except Exception: return None

def collect_pairs(samples, loader, tag):
    imgs, caps = [], []
    for s in tqdm(samples, desc=f"load-{tag}"):
        cap=str(s.get("caption","")).strip()
        if not cap: continue
        im=loader(s)
        if im is None: continue
        imgs.append(im); caps.append(cap)
    print(f"📊 {tag}: ok={len(imgs)}")
    return imgs, caps

mem_imgs, mem_caps = collect_pairs(members, load_member_image, "member")
non_imgs, non_caps = collect_pairs(nonmembers, load_nonmember_image, "nonmember")
if len(mem_imgs)==0 or len(non_imgs)==0: raise RuntimeError("有效样本为 0")

# ---------------------------
# 3) Encode
# ---------------------------
@torch.inference_mode()
def encode_image_batch(pils, bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        batch=torch.stack([preprocess_val(im) for im in pils[i:i+bs]]).to(device)
        with torch.cuda.amp.autocast():
            z=model.encode_image(batch); z=z/(z.norm(dim=-1,keepdim=True)+1e-12)
        out.append(z.float().cpu())
    return torch.cat(out,0)

@torch.inference_mode()
def encode_text_batch(texts, bs=128, max_len=256):
    out=[]
    for i in range(0,len(texts),bs):
        toks=tokenizer(texts[i:i+bs])
        toks=torch.as_tensor(toks)[:, :max_len].to(device)
        with torch.cuda.amp.autocast():
            z=model.encode_text(toks); z=z/(z.norm(dim=-1,keepdim=True)+1e-12)
        out.append(z.float().cpu())
    return torch.cat(out,0)

print("\n⏳ Encoding …")
mem_img = encode_image_batch(mem_imgs)
non_img = encode_image_batch(non_imgs)
mem_txt = encode_text_batch(mem_caps)
non_txt = encode_text_batch(non_caps)

all_txt = torch.cat([mem_txt, non_txt], 0)         # [Nc, D]
Nc = all_txt.size(0)
all_txt_gpu = all_txt.to(device)

# ---------------------------
# 4) Min-K++ scorer
# ---------------------------
def _build_hard_pool_for_image(img_vec_gpu: torch.Tensor, pos_idx: int, hard_top: int) -> np.ndarray:
    """
    返回该图像在文本库中的 top-HARD_TOP 相似文本索引（排除 pos_idx）
    """
    with torch.no_grad():
        sim = all_txt_gpu @ img_vec_gpu  # [Nc]
        # 取 topk(hard_top+1)，然后滤掉 pos_idx
        k = min(hard_top + 1, sim.numel())
        vals, idx = torch.topk(sim, k=k, largest=True)
        idx = idx.detach().cpu().numpy()
        idx = idx[idx != pos_idx]
        if idx.size > hard_top:
            idx = idx[:hard_top]
        return idx

def minKpp_score(img_vec: torch.Tensor, pos_idx: int) -> float:
    """
    Min-K++:
      - hard pool: top-HARD_TOP nearest texts to the image (exclude pos)
      - per trial: sample K-1 negatives; HARD_RATIO from hard pool, rest random
      - temperature jitters TAUS
      - aggregate: collect R * len(TAUS) NLLs -> mean of smallest ceil(p*count)
      - score = -mean_min_p
    """
    img_d = img_vec.to(device)
    hard_pool = _build_hard_pool_for_image(img_d, pos_idx, HARD_TOP)

    # 全局负例索引（除去正例）
    global_pool = np.delete(np.arange(Nc), pos_idx)

    nlls = []
    for _ in range(R):
        # 按比例从 hard_pool / global_pool 采样
        need_neg = K - 1
        n_hard = min(need_neg, int(round(need_neg * HARD_RATIO)))
        n_rand = need_neg - n_hard

        if hard_pool.size > 0 and n_hard > 0:
            hard_idx = np.random.choice(hard_pool, size=n_hard, replace=(hard_pool.size < n_hard))
        else:
            hard_idx = np.array([], dtype=int)

        # global 池里也要避免选到 hard_idx / pos_idx（虽然 pos_idx 已经被排除）
        remain_pool = np.setdiff1d(global_pool, hard_idx, assume_unique=False)
        if remain_pool.size > 0 and n_rand > 0:
            rand_idx = np.random.choice(remain_pool, size=n_rand, replace=(remain_pool.size < n_rand))
        else:
            rand_idx = np.array([], dtype=int)

        neg_idx = np.concatenate([hard_idx, rand_idx])
        # 万一负例数不足，做兜底补齐（极端小池）
        if neg_idx.size < need_neg:
            extra = np.random.choice(global_pool, size=need_neg - neg_idx.size, replace=True)
            neg_idx = np.concatenate([neg_idx, extra])

        cand_idx = np.concatenate([[pos_idx], neg_idx])  # [K]
        cand = all_txt_gpu[cand_idx]                      # [K, D]
        logits_raw = LOGIT_SCALE * (cand @ img_d)         # [K]

        # 温度扰动
        for tau in TAUS:
            logits = logits_raw / float(tau)
            logp = torch.log_softmax(logits, dim=0)
            nll  = -logp[0]
            nlls.append(float(nll.item()))

    nlls = np.asarray(nlls, dtype=np.float64)
    m = max(1, int(np.ceil(P_FRAC * len(nlls))))
    best_mean = np.partition(nlls, m-1)[:m].mean()
    return float(-best_mean)

# ---------------------------
# 5) Score & eval
# ---------------------------
mem_scores, non_scores = [], []
print("\n⏳ Scoring (Min-K++) …")

for i in tqdm(range(mem_img.size(0)), desc="members"):
    mem_scores.append(minKpp_score(mem_img[i], i))

offset = mem_txt.size(0)
for j in tqdm(range(non_img.size(0)), desc="nonmembers"):
    pos = offset + j
    non_scores.append(minKpp_score(non_img[j], pos))

print(f"\n有效样本数：mem={len(mem_scores)} non={len(non_scores)}")
y = np.array([1]*len(mem_scores) + [0]*len(non_scores))
scores = np.array(mem_scores + non_scores)

auc = roc_auc_score(y, scores)
fpr, tpr, thr = roc_curve(y, scores); j = np.argmax(tpr - fpr)
acc = accuracy_score(y, (scores >= thr[j]).astype(int))
cm  = confusion_matrix(y, (scores >= thr[j]).astype(int))

print("\n====== Min-K++ MIA 结果 ======")
print(f"AUC = {auc:.4f}")
print(f"Best threshold (Youden J) = {thr[j]:.4f}")
print(f"Accuracy = {acc:.4f}")
print("Confusion matrix [[TN FP][FN TP]]:\n", cm)

# -------- 新增：@5% FPR 的 TPR / Precision / Recall / F1 --------
# 采用“非成员分数 95% 分位数”作为阈值（与前面脚本风格一致；实际 FPR 可能略有偏差）
thr_5 = float(np.quantile(non_scores, 0.95))
y_pred_5 = (scores >= thr_5).astype(int)

cm_5 = confusion_matrix(y, y_pred_5)
tn, fp, fn, tp = cm_5.ravel()
fpr_actual = fp / (fp + tn + 1e-12)
tpr_5 = tp / (tp + fn + 1e-12)
prec_5 = precision_score(y, y_pred_5, zero_division=0)
rec_5  = recall_score(y, y_pred_5, zero_division=0)   # 与 TPR 相同
f1_5   = f1_score(y, y_pred_5, zero_division=0)

print("\n------ @5% FPR Metrics ------")
print(f"Threshold @FPR=5% = {thr_5:.4f}  (实际 FPR={fpr_actual*100:.2f}%)")
print(f"TPR (Recall)      = {tpr_5*100:.2f}%")
print(f"Precision          = {prec_5*100:.2f}%")
print(f"F1                 = {f1_5:.4f}")
print("Confusion @5%FPR [[TN FP][FN TP]]：")
print(cm_5)


## ModRényi

In [ ]:
# ===============================================
# ModRényi* (fused) MIA for BiomedCLIP (open_clip)
#  - Members : PMC tar + caption（members_1k.json）
#  - Non-mem : ROCO 本地（nonmembers_roco_1k.json）
#  - Model   : microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224
#  - Score   : fused ModRényi*（图像视图 + 文本视图）
#  - Metric  : AUC + TPR / Precision / Recall / F1 @ 5% FPR
# ===============================================

import os, io, json, tarfile, random, shutil, math, warnings, re
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support, confusion_matrix

warnings.filterwarnings("ignore")

# ---------------------------
# 0) 路径与超参（与你的 BiomedCLIP 脚本保持一致）
# ---------------------------
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

# 候选池规模：1 正 + (K-1) 负
K = 32

# 视图数（可按需调小以提速）
IMG_VIEWS = 5   # 原图 + 4 种增广
TXT_VIEWS = 3   # 原文 + 2 个轻度扰动

# TAR 下载/缓存目录
PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

# ---------------------------
# 1) 设备与模型
# ---------------------------
assert torch.cuda.is_available(), "未检测到 CUDA，请在运行时中启用 GPU（T4/A100/H100）"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

# 依赖：open_clip_torch
try:
    import open_clip
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch>=2.24.0", "ftfy", "regex", "tqdm", "torchvision"])
    import open_clip

MODEL_NAME = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
print(f"⏳ 加载模型：{MODEL_NAME}")
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()
with torch.no_grad():
    LOGIT_SCALE = model.logit_scale.exp().item() if hasattr(model, "logit_scale") else 1.0
print(f"✅ BiomedCLIP 加载完成 | logit_scale={LOGIT_SCALE:.3f}")

# ---------------------------
# 2) 数据读取（与对比脚本一致）
# ---------------------------
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all    = load_json_list(MEMBERS_JSON)[:NUM_MEMBERS]
nonmembers_all = load_json_list(NONMEMBERS_JSON)[:NUM_NONMEMBERS]
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)

import requests
def _download_to(path: str, url: str, timeout=120):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk: f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = tar_url.strip().split("/")[-1]
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if not os.path.exists(local) or os.path.getsize(local) == 0:
        _download_to(local, tar_url, timeout=180)
    return local

def _extract_one_from_tar(local_tar: str, file_in_tar: str) -> str:
    target_norm = file_in_tar.replace("\\", "/")
    with tarfile.open(local_tar, "r:gz") as tar:
        member = None
        for ti in tar.getmembers():
            if ti.name.replace("\\", "/") == target_norm:
                member = ti; break
        if member is None:
            cands = [target_norm,
                     target_norm.replace(".jpg",".JPG"),
                     target_norm.replace(".jpg",".png"),
                     target_norm.replace(".JPG",".jpg")]
            for ti in tar.getmembers():
                if ti.name.replace("\\","/") in cands:
                    member = ti; break
        if member is None:
            raise FileNotFoundError(f"{file_in_tar} 不在 TAR 包中：{local_tar}")
        out_path = os.path.join(PMC_EXTRACT_TMP_DIR, Path(member.name).name)
        with tar.extractfile(member) as src, open(out_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    return out_path

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    tar_url = sample.get("pmc_tar_url"); path_in = sample.get("image_file_path")
    if not tar_url or not path_in: return None
    local_tar = _ensure_tar_local(tar_url)
    local_img = _extract_one_from_tar(local_tar, path_in)
    return Image.open(local_img).convert("RGB")

def load_nonmember_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if not p or not os.path.exists(p): return None
    return Image.open(p).convert("RGB")

def collect_pairs(samples: List[Dict[str,Any]], loader, kind="member"):
    imgs, caps = [], []
    fail = {"image":0, "other":0}
    for s in tqdm(samples, desc=f"load-{kind}"):
        cap = str(s.get("caption","")).strip()
        if not cap:
            fail["other"] += 1; continue
        try:
            im = loader(s)
            if im is None: fail["image"] += 1; continue
        except (UnidentifiedImageError, FileNotFoundError, tarfile.ReadError):
            fail["image"] += 1; continue
        except Exception:
            fail["image"] += 1; continue
        imgs.append(im); caps.append(cap)
    print(f"📊 {kind}: ok={len(imgs)} | fail={fail}")
    return imgs, caps

print("\n⏳ 读取 members（TAR 下载/解包可能略慢）")
mem_imgs, mem_caps = collect_pairs(members_all, load_member_image, kind="member")
print("⏳ 读取 non-members（本地）")
non_imgs, non_caps = collect_pairs(nonmembers_all, load_nonmember_image, kind="nonmember")

if len(mem_imgs)==0 or len(non_imgs)==0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

# ---------------------------
# 3) 编码（批处理，与对比脚本一致）
# ---------------------------
@torch.inference_mode()
def encode_image_batch(pils: List[Image.Image], batch_size: int = 32) -> torch.Tensor:
    out = []
    for i in range(0, len(pils), batch_size):
        chunk = pils[i:i+batch_size]
        imgs = torch.stack([preprocess_val(im) for im in chunk]).to(device)
        with torch.cuda.amp.autocast():
            feats = model.encode_image(imgs)
            feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
        out.append(feats.float().cpu())
    return torch.cat(out, dim=0)

@torch.inference_mode()
def encode_text_batch(texts: List[str], batch_size: int = 64, max_len: int = 256) -> torch.Tensor:
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        toks = tokenizer(chunk)
        toks = torch.as_tensor(toks)[:, :max_len].to(device)
        with torch.cuda.amp.autocast():
            feats = model.encode_text(toks)
            feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
        out.append(feats.float().cpu())
    return torch.cat(out, dim=0)

print("\n⏳ 编码图像 …")
mem_img_emb  = encode_image_batch(mem_imgs, batch_size=32)
non_img_emb  = encode_image_batch(non_imgs, batch_size=32)
print("⏳ 编码文本 …")
mem_txt_emb  = encode_text_batch(mem_caps, batch_size=64, max_len=256)
non_txt_emb  = encode_text_batch(non_caps, batch_size=64, max_len=256)

# 池：用于负例采样/快速拼接
all_txt_emb = torch.cat([mem_txt_emb, non_txt_emb], dim=0)  # [Nc, D]
all_img_emb = torch.cat([mem_img_emb, non_img_emb], dim=0)  # [Ni, D]
Nc = all_txt_emb.size(0)
Ni = all_img_emb.size(0)

# ---------------------------
# 4) 视图构造（图像与文本）
# ---------------------------
from torchvision.transforms import RandomResizedCrop, RandomRotation, RandomAffine, ColorJitter, InterpolationMode

def make_image_views(pil: Image.Image, want: int = IMG_VIEWS) -> List[Image.Image]:
    views = [pil]
    views.append(RandomResizedCrop(size=(256, 256), scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC)(pil))
    views.append(RandomRotation(degrees=30, interpolation=InterpolationMode.BICUBIC, expand=False)(pil))
    views.append(RandomAffine(degrees=20, translate=(0.08, 0.08), scale=(0.9, 1.1), interpolation=InterpolationMode.BICUBIC)(pil))
    views.append(ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25, hue=0.05)(pil))
    return views[:max(1, want)]

def _normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def make_text_views(caption: str, want: int = TXT_VIEWS) -> List[str]:
    base = str(caption or "").strip()
    cand = [base, base.lower(), _normalize_spaces(base)]
    # 去重保序
    out, seen = [], set()
    for c in cand:
        if c not in seen:
            out.append(c); seen.add(c)
    return out[:max(1, want)]

# ---------------------------
# 5) 计算单视图的 logP（候选池 softmax）
# ---------------------------
@torch.inference_mode()
def _encode_one_image_view(pil: Image.Image) -> torch.Tensor:
    tens = preprocess_val(pil).unsqueeze(0).to(device)
    with torch.cuda.amp.autocast():
        z = model.encode_image(tens)
        z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
    return z[0].float()  # [D]

@torch.inference_mode()
def _encode_one_text_view(text: str, max_len=256) -> torch.Tensor:
    toks = tokenizer([text])
    toks = torch.as_tensor(toks)[:, :max_len].to(device)
    with torch.cuda.amp.autocast():
        z = model.encode_text(toks)
        z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
    return z[0].float()  # [D]

def image_view_logp(img_vec: torch.Tensor, pos_txt_idx: int, K: int) -> np.ndarray:
    """
    给定图像向量 img_vec（已归一化），在候选 caption 池上算 log-softmax 概率分布（长度 K）。
    返回 shape [1, K] 的 logP（np.float32）。
    """
    img_d = img_vec.to(device)
    # 负例索引
    neg_pool = np.delete(np.arange(Nc), pos_txt_idx)
    neg_idx  = np.random.choice(neg_pool, size=K-1, replace=False)
    cand_idx = np.concatenate([[pos_txt_idx], neg_idx])
    cand_txt = all_txt_emb[cand_idx].to(device)  # [K, D]
    logits   = LOGIT_SCALE * (cand_txt @ img_d)  # [K]
    logp     = torch.log_softmax(logits, dim=0).unsqueeze(0)  # [1, K]
    return logp.detach().cpu().numpy().astype(np.float32)

def text_view_logp(txt_vec: torch.Tensor, pos_img_idx: int, K: int) -> np.ndarray:
    """
    给定文本向量 txt_vec（已归一化），在候选图像池上算 log-softmax 概率分布（长度 K）。
    返回 shape [1, K] 的 logP（np.float32）。
    """
    txt_d = txt_vec.to(device)
    neg_pool = np.delete(np.arange(Ni), pos_img_idx)
    neg_idx  = np.random.choice(neg_pool, size=K-1, replace=False)
    cand_idx = np.concatenate([[pos_img_idx], neg_idx])
    cand_img = all_img_emb[cand_idx].to(device)  # [K, D]
    logits   = LOGIT_SCALE * (cand_img @ txt_d)  # [K]
    logp     = torch.log_softmax(logits, dim=0).unsqueeze(0)  # [1, K]
    return logp.detach().cpu().numpy().astype(np.float32)

# ---------------------------
# 6) Rényi 熵、稳健聚合、融合
# ---------------------------
def renyi_entropy_from_logp(logp_vec: np.ndarray, alpha: float) -> float:
    # logp_vec: [K] 或 [1,K]；支持统一处理
    v = logp_vec.reshape(-1)
    a = alpha * v
    m = np.max(a)
    return float((1.0/(1.0-alpha)) * (m + np.log(np.exp(a - m).sum())))

def renyi_seq_from_logp(logp_seq: np.ndarray, alpha: float) -> float:
    # 这里 logp_seq 是 [T, K]；CLIP 情况下 T=1（单行）
    T = logp_seq.shape[0]
    vals = [renyi_entropy_from_logp(logp_seq[t], alpha) for t in range(T)]
    return float(np.mean(vals)) if vals else float("nan")

def robust_aggregate(values: List[float], trim: float = 0.1) -> float:
    arr = np.array([v for v in values if np.isfinite(v)], dtype=float)
    if arr.size == 0:
        return float("nan")
    arr.sort()
    k = int(math.floor(trim * arr.size))
    if k * 2 < arr.size:
        arr = arr[k: arr.size - k]
    return float(np.mean(arr))

def fused_modrenyi_score(img_logps: List[np.ndarray],
                         txt_logps: List[np.ndarray],
                         alphas=(0.5, 2.0)) -> float:
    """
    - 对每个通道（图像/文本），对每个视图的一行 logP 计算 Rényi 熵（α∈{0.5,2.0}）
    - 视图内做稳健均值（trimmed mean）
    - 对两个通道、两种 α 的 4 个分量取负熵并 z-score 融合
    """
    parts = []
    for logp_views in [img_logps, txt_logps]:
        if len(logp_views) == 0:
            parts.extend([float("nan")] * len(alphas))
            continue
        for a in alphas:
            per_view = []
            for lp in logp_views:
                if lp is None or not np.isfinite(lp).all():
                    continue
                H = renyi_seq_from_logp(lp, a)  # T=1
                per_view.append(H)
            if not per_view:
                parts.append(float("nan")); continue
            H_bar = robust_aggregate(per_view, trim=0.1)
            score = -H_bar
            parts.append(score)

    valid = np.array([x for x in parts if np.isfinite(x)], dtype=float)
    if valid.size == 0:
        return float("nan")
    if valid.size >= 2 and np.std(valid) > 1e-12:
        z = (valid - np.mean(valid)) / np.std(valid)
        return float(np.mean(z))
    else:
        return float(np.mean(valid))

# ---------------------------
# 7) 单样本打分（融合）
# ---------------------------
def fused_score_for_member(i: int) -> Optional[float]:
    """ members 第 i 个样本（img/cap 都在 mem_* 列表），其正例索引为：
        - 文本：pos_txt_idx = i
        - 图像：pos_img_idx = i
    """
    try:
        img = mem_imgs[i]; cap = mem_caps[i]
    except Exception:
        return None

    # 图像视图 → logP 行
    img_views = make_image_views(img, IMG_VIEWS)
    img_logps = []
    try:
        pos_txt_idx = i
        for v in img_views:
            v_z = _encode_one_image_view(v)
            lp  = image_view_logp(v_z, pos_txt_idx, K)
            img_logps.append(lp)
    except Exception:
        pass

    # 文本视图 → logP 行
    txt_views = make_text_views(cap, TXT_VIEWS)
    txt_logps = []
    try:
        pos_img_idx = i
        for t in txt_views:
            t_z = _encode_one_text_view(t)
            lp  = text_view_logp(t_z, pos_img_idx, K)
            txt_logps.append(lp)
    except Exception:
        pass

    if len(img_logps)==0 and len(txt_logps)==0:
        return None
    return fused_modrenyi_score(img_logps, txt_logps, alphas=(0.5, 2.0))

def fused_score_for_nonmember(j: int) -> Optional[float]:
    """ nonmembers 第 j 个样本，其在拼接池中的正例索引：
        - 文本：pos_txt_idx = len(mem_txt_emb) + j
        - 图像：pos_img_idx = len(mem_img_emb) + j
    """
    try:
        img = non_imgs[j]; cap = non_caps[j]
    except Exception:
        return None

    img_logps = []
    try:
        pos_txt_idx = mem_txt_emb.size(0) + j
        for v in make_image_views(img, IMG_VIEWS):
            v_z = _encode_one_image_view(v)
            lp  = image_view_logp(v_z, pos_txt_idx, K)
            img_logps.append(lp)
    except Exception:
        pass

    txt_logps = []
    try:
        pos_img_idx = mem_img_emb.size(0) + j
        for t in make_text_views(cap, TXT_VIEWS):
            t_z = _encode_one_text_view(t)
            lp  = text_view_logp(t_z, pos_img_idx, K)
            txt_logps.append(lp)
    except Exception:
        pass

    if len(img_logps)==0 and len(txt_logps)==0:
        return None
    return fused_modrenyi_score(img_logps, txt_logps, alphas=(0.5, 2.0))

# ---------------------------
# 8) @5%FPR 指标（线性插值阈值）
# ---------------------------
def metrics_at_target_fpr(y_true: np.ndarray, scores: np.ndarray, target_fpr: float = 0.05):
    fpr, tpr, thr = roc_curve(y_true, scores)
    # 找到 fpr ≈ 目标的插值阈值
    idx = np.searchsorted(fpr, target_fpr, side="right")
    if idx == 0:
        thr_star = thr[0]; tpr_star = tpr[0]
    elif idx >= len(thr):
        thr_star = thr[-1]; tpr_star = tpr[-1]
    else:
        x0, x1 = fpr[idx-1], fpr[idx]
        y0, y1 = tpr[idx-1], tpr[idx]
        t0, t1 = thr[idx-1], thr[idx]
        w = (target_fpr - x0) / (x1 - x0 + 1e-12)
        tpr_star = y0 + w*(y1 - y0)
        thr_star = t0 + w*(t1 - t0)

    y_pred = (scores >= thr_star).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr_actual = fp / (fp + tn + 1e-12)

    return {
        "thr@5%FPR": float(thr_star),
        "TPR@5%FPR": float(tpr_star),
        "Precision": float(prec),
        "Recall": float(rec),
        "F1": float(f1),
        "ROC-AUC": float(roc_auc_score(y_true, scores)),
        "FPR(actual)": float(fpr_actual),
        "CM@5%FPR": cm,
    }

# ---------------------------
# 9) 主流程：打分 & 评估
# ---------------------------
print("\n⏳ 计算 ModRényi*（fused）分数 …")
mem_scores, non_scores = [], []

for i in tqdm(range(len(mem_imgs)), desc="members"):
    s = fused_score_for_member(i)
    if s is not None and math.isfinite(s):
        mem_scores.append(s)

for j in tqdm(range(len(non_imgs)), desc="nonmembers"):
    s = fused_score_for_nonmember(j)
    if s is not None and math.isfinite(s):
        non_scores.append(s)

print(f"\n📊 有效样本：members={len(mem_scores)} / {len(mem_imgs)} | non-members={len(non_scores)} / {len(non_imgs)}")
if len(mem_scores)==0 or len(non_scores)==0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

y_true = np.array([1]*len(mem_scores) + [0]*len(non_scores))
scores = np.array(mem_scores + non_scores)

m = metrics_at_target_fpr(y_true, scores, target_fpr=0.05)

print("\n====== ModRényi* (fused) MIA 结果（BiomedCLIP） ======")
print(f"AUC                 = {m['ROC-AUC']:.4f}")
print(f"TPR  @5% FPR        = {m['TPR@5%FPR']:.4f}")
print(f"Precision @5% FPR   = {m['Precision']:.4f}")
print(f"Recall    @5% FPR   = {m['Recall']:.4f}")
print(f"F1        @5% FPR   = {m['F1']:.4f}")
print(f"thr       @5% FPR   = {m['thr@5%FPR']:.4f}")
print(f"FPR(actual)         = {m['FPR(actual)']*100:.2f}%")
print("Confusion @5%FPR [[TN FP][FN TP]]：")
print(m["CM@5%FPR"])


## Similarity

In [ ]:
# ===============================================
# Similarity-MIA (Image + Text) with Fusion — robust (BiomedCLIP)
#  - Data   : members_1k.json (PMC tar) / nonmembers_roco_1k.json (local)
#  - Model  : microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224 via open_clip
#  - Score  : self-similarity under light perturbations (image & text), z-score-fused
#  - Metric : AUC + TPR / Precision / Recall / F1 @ 5% FPR
# ===============================================
import os, io, json, math, tarfile, random, shutil, warnings, re, csv
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import torch
from PIL import Image, ImageFilter, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)

warnings.filterwarnings("ignore")

# ---------------- 基本参数 ----------------
MEMBERS_JSON     = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON  = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"
NUM_MEMBERS, NUM_NONMEMBERS = 1000, 1000
SEED = 42

OUT_DIR  = "/content/sim_mia_outputs"
OUT_CSV  = os.path.join(OUT_DIR, "mia_similarity_fused_1000x1000_biomedclip.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# 图像扰动
GAUSS_RADIUS = 1.0     # 轻度高斯模糊
JPEG_QUALITY = 60      # 有损重压缩

# 文本扰动（与生成式不同，这里做轻语义保持的规范化）
TXT_VIEWS = 2          # 只取两种：lower / 空白规整（二者与原文成对比）
MAX_TEXT_LEN = 256     # 文本截断长度（BiomedCLIP 文本编码器）

# 融合权重（单通道时自动视为 1.0）
W_IMG, W_TEXT = 0.60, 0.40

# ---------------- 设备 & 模型（BiomedCLIP via open_clip） ----------------
assert torch.cuda.is_available(), "需要 GPU"
device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32  = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Using {device}")

try:
    import open_clip
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch>=2.24.0", "ftfy", "regex", "tqdm", "torchvision"])
    import open_clip

MODEL_NAME = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
print(f"⏳ 加载 BiomedCLIP：{MODEL_NAME}")
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()
with torch.no_grad():
    LOGIT_SCALE = model.logit_scale.exp().item() if hasattr(model, "logit_scale") else 1.0
print(f"✅ BiomedCLIP 就绪 | logit_scale={LOGIT_SCALE:.3f}")

# ---------------- 数据加载（与对比脚本一致） ----------------
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all    = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)
members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 样本就绪：members={len(members)} | non-members={len(nonmembers)}")

# 远端 TAR & 本地 I/O（与对比脚本一致）
import requests
PMC_TAR_CACHE_DIR   = "/content/pmc_tar_cache"
PMC_EXTRACT_TMP_DIR = "/content/pmc_extract_tmp"
os.makedirs(PMC_TAR_CACHE_DIR, exist_ok=True)
os.makedirs(PMC_EXTRACT_TMP_DIR, exist_ok=True)

def _download_to(path: str, url: str, timeout=120):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1<<20):
                if chunk: f.write(chunk)

def _ensure_tar_local(tar_url: str) -> str:
    fname = tar_url.strip().split("/")[-1]
    local = os.path.join(PMC_TAR_CACHE_DIR, fname)
    if not os.path.exists(local) or os.path.getsize(local) == 0:
        _download_to(local, tar_url, timeout=180)
    return local

def _extract_one_from_tar(local_tar: str, file_in_tar: str) -> str:
    target_norm = file_in_tar.replace("\\", "/")
    with tarfile.open(local_tar, "r:gz") as tar:
        member = None
        for ti in tar.getmembers():
            if ti.name.replace("\\", "/") == target_norm:
                member = ti; break
        if member is None:
            cands = [target_norm,
                     target_norm.replace(".jpg",".JPG"),
                     target_norm.replace(".jpg",".png"),
                     target_norm.replace(".JPG",".jpg")]
            for ti in tar.getmembers():
                if ti.name.replace("\\","/") in cands:
                    member = ti; break
        if member is None: raise FileNotFoundError(f"{file_in_tar} 不在 TAR 包中：{local_tar}")
        out_path = os.path.join(PMC_EXTRACT_TMP_DIR, Path(member.name).name)
        with tar.extractfile(member) as src, open(out_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    return out_path

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    tar_url = sample.get("pmc_tar_url"); path_in = sample.get("image_file_path")
    if not tar_url or not path_in: return None
    local_tar = _ensure_tar_local(tar_url)
    local_img = _extract_one_from_tar(local_tar, path_in)
    return Image.open(local_img).convert("RGB")

def load_nonmember_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if not p or not os.path.exists(p): return None
    return Image.open(p).convert("RGB")

def collect_pairs(samples: List[Dict[str,Any]], loader, kind="member"):
    imgs, caps = [], []
    fail = {"image":0, "other":0}
    for s in tqdm(samples, desc=f"load-{kind}"):
        cap = str(s.get("caption","")).strip()
        if not cap:
            fail["other"] += 1; continue
        try:
            im = loader(s)
            if im is None: fail["image"] += 1; continue
        except (UnidentifiedImageError, FileNotFoundError, tarfile.ReadError):
            fail["image"] += 1; continue
        except Exception:
            fail["image"] += 1; continue
        imgs.append(im); caps.append(cap)
    print(f"📊 {kind}: ok={len(imgs)} | fail={fail}")
    return imgs, caps

print("\n⏳ 读取 members（TAR 下载/解包可能略慢）")
mem_imgs, mem_caps = collect_pairs(members, load_member_image, kind="member")
print("⏳ 读取 non-members（本地）")
non_imgs, non_caps = collect_pairs(nonmembers, load_nonmember_image, kind="nonmember")

if len(mem_imgs)==0 or len(non_imgs)==0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

# ---------------- 扰动与编码工具 ----------------
def corrupt_image(img: Image.Image) -> Image.Image:
    x = img.filter(ImageFilter.GaussianBlur(radius=GAUSS_RADIUS))
    buf = io.BytesIO(); x.save(buf, format="JPEG", quality=JPEG_QUALITY, optimize=True)
    buf.seek(0); x = Image.open(buf).convert("RGB")
    return x

def _normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def make_text_views(caption: str, want: int = TXT_VIEWS) -> List[str]:
    base = str(caption or "").strip()
    # 轻度语义保持 & 可逆扰动
    cand = [base, base.lower(), _normalize_spaces(base)]
    out, seen = [], set()
    for c in cand:
        if c not in seen:
            out.append(c); seen.add(c)
    return out[:max(2, want)]  # 至少两种视图以形成配对

@torch.inference_mode()
def encode_image_batch(pils: List[Image.Image], batch_size: int = 64) -> torch.Tensor:
    out = []
    for i in range(0, len(pils), batch_size):
        chunk = pils[i:i+batch_size]
        imgs = torch.stack([preprocess_val(im) for im in chunk]).to(device)
        with torch.cuda.amp.autocast():
            z = model.encode_image(imgs)
            z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
        out.append(z.float().cpu())
    return torch.cat(out, dim=0) if out else torch.zeros((0, 512))

@torch.inference_mode()
def encode_text_batch(texts: List[str], batch_size: int = 128, max_len: int = MAX_TEXT_LEN) -> torch.Tensor:
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        toks = tokenizer(chunk)
        toks = torch.as_tensor(toks)[:, :max_len].to(device)
        with torch.cuda.amp.autocast():
            z = model.encode_text(toks)
            z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
        out.append(z.float().cpu())
    return torch.cat(out, dim=0) if out else torch.zeros((0, 512))

# ---------------- 通道得分（批量、自相似） ----------------
def pair_cos_sims(z1: torch.Tensor, z2: torch.Tensor) -> np.ndarray:
    # 每行一对 (i,i) 的余弦，相当于逐元素点积（已归一化）
    sims = (z1 * z2).sum(dim=-1).numpy()
    return sims.astype(np.float32)

print("\n⏳ 构造图像扰动并编码 …")
# 原图 & 扰动图（members + nonmembers 一起批量编码，减少显存抖动）
all_imgs = mem_imgs + non_imgs
all_imgs_corrupt = [corrupt_image(im) for im in tqdm(all_imgs, desc="corrupt-images")]
z_img_orig    = encode_image_batch(all_imgs, batch_size=64)
z_img_corrupt = encode_image_batch(all_imgs_corrupt, batch_size=64)

mem_img_sim = pair_cos_sims(z_img_orig[:len(mem_imgs)], z_img_corrupt[:len(mem_imgs)])
non_img_sim = pair_cos_sims(z_img_orig[len(mem_imgs):], z_img_corrupt[len(mem_imgs):])

print("⏳ 构造文本扰动并编码 …")
# 为每个 caption 取两种视图：v0=原文，v1=lower 或空白规整（与 make_text_views 对齐）
def pick_two_views(caps: List[str]) -> Tuple[List[str], List[str]]:
    v0, v1 = [], []
    for c in caps:
        views = make_text_views(c, want=2)
        if len(views) < 2:
            # 极端兜底：复制原文
            views = [c, c]
        v0.append(views[0]); v1.append(views[1])
    return v0, v1

mem_v0, mem_v1 = pick_two_views(mem_caps)
non_v0, non_v1 = pick_two_views(non_caps)

z_txt_mem_v0 = encode_text_batch(mem_v0, batch_size=256)
z_txt_mem_v1 = encode_text_batch(mem_v1, batch_size=256)
z_txt_non_v0 = encode_text_batch(non_v0, batch_size=256)
z_txt_non_v1 = encode_text_batch(non_v1, batch_size=256)

mem_txt_sim = pair_cos_sims(z_txt_mem_v0, z_txt_mem_v1)
non_txt_sim = pair_cos_sims(z_txt_non_v0, z_txt_non_v1)

# ---------------- 融合与指标（容忍单通道） ----------------
def zscore(arr: np.ndarray):
    mu, sd = float(arr.mean()), float(arr.std() + 1e-8)
    return (arr - mu) / (sd + 1e-8), mu, sd

have_img = (len(mem_img_sim) > 10 and len(non_img_sim) > 10)
have_txt = (len(mem_txt_sim) > 10 and len(non_txt_sim) > 10)
if not have_img and not have_txt:
    raise RuntimeError("两条通道都没有有效分数（请检查数据读入或编码阶段）")

parts = []
if have_img: parts.append(("img", mem_img_sim, non_img_sim, W_IMG))
if have_txt: parts.append(("txt", mem_txt_sim, non_txt_sim, W_TEXT))
if len(parts) == 1:
    # 单通道时将权重设为 1
    parts[0] = (parts[0][0], parts[0][1], parts[0][2], 1.0)

fused_mem, fused_non = None, None
details = {}
for name, mem_arr, non_arr, w in parts:
    non_z, mu, sd = zscore(non_arr)
    mem_z = (mem_arr - mu) / (sd + 1e-8)
    details[name] = (mem_arr, non_arr, mem_z, non_z, w)
    if fused_mem is None:
        fused_mem = w * mem_z
        fused_non = w * non_z
    else:
        fused_mem = fused_mem + w * mem_z
        fused_non = fused_non + w * non_z

# AUC
y_true = np.array([1]*len(fused_mem) + [0]*len(fused_non))
scores = np.concatenate([fused_mem, fused_non], axis=0)
auc = roc_auc_score(y_true, scores)

# FPR=5% 阈值（以 nonmember 的 95 分位为阈值，直观可控）
thr_5 = float(np.quantile(fused_non, 0.95))
y_pred = (scores >= thr_5).astype(int)

cm  = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn + 1e-12)
tpr = tp / (tp + fn + 1e-12)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
acc = accuracy_score(y_true, y_pred)

print("\n====== Similarity-MIA（Image+Text 融合，稳健版，BiomedCLIP） ======")
print(f"AUC                = {auc:.4f}")
print(f"Threshold @FPR=5%  = {thr_5:.4f}  (实际 FPR={fpr*100:.2f}%)")
print(f"TPR (Recall)       = {tpr*100:.2f}%")
print(f"Precision          = {prec*100:.2f}%")
print(f"F1                 = {f1:.4f}")
print(f"Accuracy           = {acc*100:.2f}%")
print("Confusion [[TN FP][FN TP]]：")
print(cm)

# ---------------- 保存明细 ----------------
os.makedirs(OUT_DIR, exist_ok=True)
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    hdr = ["split"]
    if have_img: hdr += ["s_img", "z_img(non-z)"]
    if have_txt: hdr += ["s_txt", "z_txt(non-z)"]
    hdr += ["s_fused"]
    w.writerow(hdr)

    def write_side(mem: bool):
        zs = fused_mem if mem else fused_non
        n  = len(zs)
        for i in range(n):
            row = ["member" if mem else "nonmember"]
            if have_img:
                mi, ni, miz, niz, wi = details["img"]
                row += [ float(mi[i % len(mi)]) if mem else float(ni[i % len(ni)]) ,
                         float(miz[i % len(miz)]) if mem else float(niz[i % len(niz)]) ]
            if have_txt:
                mt, nt, mtz, ntz, wt = details["txt"]
                row += [ float(mt[i % len(mt)]) if mem else float(nt[i % len(nt)]) ,
                         float(mtz[i % len(mtz)]) if mem else float(ntz[i % len(ntz)]) ]
            row += [ float(zs[i]) ]
            w.writerow(row)

    write_side(mem=True)
    write_side(mem=False)

print(f"\n💾 明细已保存：{OUT_CSV}")


## dc-pdd

In [ ]:
import os, pickle, torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from tqdm import tqdm
from google.colab import drive

# 1. 初始化
drive.mount('/content/drive')
try:
    import open_clip
except ImportError:
    os.system('pip install open_clip_torch')
    import open_clip

# 使用 BiomedCLIP 指定的 Tokenizer
MODEL_NAME = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

# 自动获取词表大小 (PubMedBERT 通常是 28895 或 30522)
# 获取方法：由于 open_clip 的 tokenizer 是一个 wrapper，我们直接编码一个测试字符串
test_ids = tokenizer(["test"])
vocab_size = 30522 # PubMedBERT 默认标准
# 路径
FREQ_PATH = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_biomedclip.pkl"
os.makedirs(os.path.dirname(FREQ_PATH), exist_ok=True)

# 2. 加载 C4 数据集
print("正在加载 C4 英文语料 (Streaming)...")
dataset = load_dataset(
    "allenai/c4",
    data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
    split="train",
    streaming=True
)

# 3. 统计
counter = Counter()
MAX_SAMPLES = 50000

for i, ex in enumerate(tqdm(dataset, total=MAX_SAMPLES, desc="BiomedCLIP 词频统计")):
    if i >= MAX_SAMPLES: break
    # open_clip tokenizer 返回的是 tensor [1, context_length]
    tokens = tokenizer([ex['text']])[0].numpy()
    # 过滤掉填充词 (0)
    tokens = tokens[tokens > 0]
    counter.update(tokens)

# 4. 平滑与归一化
print("\n正在计算平滑词频...")
freq_smo = np.ones(vocab_size, dtype=np.float64)
total_tokens = sum(counter.values())

for tid, count in counter.items():
    if tid < vocab_size:
        freq_smo[tid] += count

freq_smo = freq_smo / (total_tokens + vocab_size)

# 5. 保存
with open(FREQ_PATH, "wb") as f:
    pickle.dump(freq_smo, f)

print(f"\n✅ BiomedCLIP (PubMedBERT) 词频表已保存至: {FREQ_PATH}")

In [ ]:
from google.colab import drive
import os

# 重新挂载或刷新
drive.mount('/content/drive', force_remount=True)

# 检查文件是否真的在那里
FREQ_PATH = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_biomedclip.pkl"
if os.path.exists(FREQ_PATH):
    print("✅ 确认文件已存在，可以继续！")
else:
    print("❌ 文件仍未同步，请稍等 10 秒后再次运行此单元格。")

In [ ]:
# ============================================================
# DC-PDD MIA for BiomedCLIP
# - Score: Cosine_Similarity * Mean(Difficulty_Calibration)
# ============================================================

import os, io, json, pickle, random, urllib.parse, tarfile, shutil
from pathlib import Path
from typing import Dict, Any, Optional, List

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve

# ---------------------------
# 0) 配置与路径
# ---------------------------
DRIVE_BASE   = "/content/drive/MyDrive/Medical/llava_med_pairs"
MEMBERS_JSON = f"{DRIVE_BASE}/members_1k.json"
NONMEM_JSON  = f"{DRIVE_BASE}/nonmembers_roco_1k.json"
FREQ_PATH    = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_biomedclip.pkl"

ALPHA = 1.0  # 难度截断阈值
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'

# ---------------------------
# 1) 模型与数据加载
# ---------------------------
import open_clip
print(f"⏳ 加载模型：{MODEL_NAME}")
model, _, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME, device=DEVICE)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()

with open(FREQ_PATH, "rb") as f:
    freq_smo = pickle.load(f)

# --- 图片读取辅助函数 ---
PMC_TAR_CACHE = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE, exist_ok=True)
_TAR_HANDLE = {}; _TAR_INDEX = {}

def load_member_image(sample):
    try:
        url = sample.get("pmc_tar_url")
        fname = Path(urllib.parse.urlparse(url).path).name
        local_tar = os.path.join(PMC_TAR_CACHE, fname)
        if not os.path.exists(local_tar):
            import requests
            with requests.get(url, stream=True) as r:
                with open(local_tar, "wb") as f: shutil.copyfileobj(r.raw, f)
        if local_tar not in _TAR_HANDLE:
            t = tarfile.open(local_tar, "r:gz")
            _TAR_HANDLE[local_tar] = t
            _TAR_INDEX[local_tar] = {ti.name.lower(): ti for ti in t.getmembers()}
        tar, idx = _TAR_HANDLE[local_tar], _TAR_INDEX[local_tar]
        p = sample.get("image_file_path").lower()
        ti = idx.get(p) or next((v for k,v in idx.items() if p in k), None)
        return Image.open(io.BytesIO(tar.extractfile(ti).read())).convert("RGB")
    except: return None

# ---------------------------
# 2) DC-PDD 核心逻辑
# ---------------------------
@torch.inference_mode()
def compute_dc_pdd_biomed(img_pil, caption):
    # A. 计算相似度 (Similarity Signal)
    img_in = preprocess_val(img_pil).unsqueeze(0).to(DEVICE)
    img_emb = model.encode_image(img_in)
    img_emb /= img_emb.norm(dim=-1, keepdim=True)

    txt_tok = tokenizer([caption]).to(DEVICE)
    txt_emb = model.encode_text(txt_tok)
    txt_emb /= txt_emb.norm(dim=-1, keepdim=True)

    cosine_sim = (img_emb * txt_emb).sum().item()

    # B. 计算难度校准 (Difficulty Calibration)
    # 获取 Caption 的 Token IDs
    tokens = tokenizer([caption])[0].numpy()
    tokens = tokens[tokens > 0] # 移除填充

    if len(tokens) == 0: return None

    # 计算 log(1/freq)
    diff_scores = [np.log(1.0 / freq_smo[tid]) if tid < len(freq_smo) else 0 for tid in tokens]
    # 截断校准
    calib_factor = np.mean([np.minimum(d, ALPHA) for d in diff_scores])

    # C. 结合
    return cosine_sim * calib_factor

# ---------------------------
# 3) 执行评估
# ---------------------------
def main():
    with open(MEMBERS_JSON, "r") as f: m_data = json.load(f)[:1000]
    with open(NONMEM_JSON, "r") as f: n_data = json.load(f)[:1000]

    m_scores, n_scores = [], []

    # Member Scoring
    for s in tqdm(m_data, desc="DC-PDD (Members)"):
        img = load_member_image(s)
        cap = s.get("caption", "").strip()
        if img and cap:
            score = compute_dc_pdd_biomed(img, cap)
            if score: m_scores.append(score)

    # Non-member Scoring
    for s in tqdm(n_data, desc="DC-PDD (Non-members)"):
        p = s.get("local_image_path")
        cap = s.get("caption", "").strip()
        if p and os.path.exists(p) and cap:
            try:
                img = Image.open(p).convert("RGB")
                score = compute_dc_pdd_biomed(img, cap)
                if score: n_scores.append(score)
            except: continue

    # 4) 指标展示
    m_s, n_s = np.array(m_scores), np.array(n_scores)
    y_true = np.concatenate([np.ones(len(m_s)), np.zeros(len(n_s))])
    y_score = np.concatenate([m_s, n_s])

    auc = roc_auc_score(y_true, y_score)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    tpr_at_5 = tpr[np.abs(fpr - 0.05).argmin()]

    print("\n" + "="*40)
    print("📊 BiomedCLIP DC-PDD Results")
    print("="*40)
    print(f"AUC:         {auc:.4f}")
    print(f"TPR @ 5%FPR: {tpr_at_5*100:.2f}%")
    print(f"Mean M:      {np.mean(m_s):.4f}")
    print(f"Mean N:      {np.mean(n_s):.4f}")

if __name__ == "__main__":
    main()

## M$^4$I

In [ ]:
# ============================================================
# M⁴I (CLIP version) on BiomedCLIP — Colab Full Pipeline (Strict, No-Leak)
#
# ✅ 目标：shadow→target 迁移的 membership inference
#   1) 从 nonmembers_1k 抽样：
#      - target_ft (member for target') = 100  -> 微调 target'
#      - shadow_ft (member for shadow)  = 100  -> 微调 shadow
#      - shadow_non (nonmember shadow)  = 100  -> 训练攻击器的负样本（shadow 未见）
#      - eval_non (nonmember target')   = 600  -> 评估负样本（target' 未见）
#      - eval_mem (member target')      = target_ft (100)
#
# ✅ 攻击特征（M⁴I 的 CLIP 化）：
#   feat = [ abs(img_emb - txt_emb) ; cos(img_emb, txt_emb) ; logit(img_emb, txt_emb) ]
#   (emb 维度=512，所以总维度=514)
#
# ✅ 攻击器：
#   用 shadow 的输出特征训练 MLP / Logistic(这里用小 MLP)
#   在 target' 的输出特征上评估 AUC / Acc / TPR@5%FPR
#
# ⚠️ 注意：
#   - membership 标签严格相对“某个微调模型的训练集”定义
#   - 所以评估 member 只有 target_ft 这 100 条（除非你扩大微调集）
#   - members_1k（PMC）在这个严格闭环中不作为 member；可选作为额外 nonmember（默认关闭）
# ============================================================

import os, json, random, math, time, re
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.metrics import precision_recall_fscore_support

# -----------------------------
# 0) 环境与依赖
# -----------------------------
try:
    import open_clip
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"])
    import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------------
# 1) 读取数据
# -----------------------------
MEMBERS_JSON    = "/content/drive/MyDrive/Medical/llava_med_pairs/members_1k.json"
NONMEMBERS_JSON = "/content/drive/MyDrive/Medical/llava_med_pairs/nonmembers_roco_1k.json"

with open(MEMBERS_JSON, "r", encoding="utf-8") as f:
    members_1k = json.load(f)
with open(NONMEMBERS_JSON, "r", encoding="utf-8") as f:
    nonmembers_1k = json.load(f)

print(f"members={len(members_1k)}, nonmembers={len(nonmembers_1k)}")

# -----------------------------
# 2) 关键超参数（按你要求：每次微调<=100；两次=200）
# -----------------------------
SEED = 42
FT_TARGET_N = 100
FT_SHADOW_N = 100
ATTACK_SHADOW_NON_N = 100
EVAL_NON_N = 600

FINETUNE_EPOCHS = 2          # 你可调 1~3
FINETUNE_BS = 32
FINETUNE_LR = 1e-5           # CLIP 微调通常小 LR 更稳
FINETUNE_WD = 0.01

ATTACK_EPOCHS = 80
ATTACK_BS = 64
ATTACK_LR = 1e-3

OUT_DIR = "/content/drive/MyDrive/Medical/biomedclip_m4i_strict"
os.makedirs(OUT_DIR, exist_ok=True)

USE_PMC_AS_EXTRA_NONMEM = False   # 可选：把 members_1k 当作额外 nonmember（默认关闭）

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# -----------------------------
# 3) 划分（严格不泄漏）
#    只从 nonmembers_1k 里抽样用于微调与训练/评估
# -----------------------------
idx = list(range(len(nonmembers_1k)))
random.shuffle(idx)

def take(n):
    global idx
    out = idx[:n]
    idx = idx[n:]
    return out

i_target_ft = take(FT_TARGET_N)
i_shadow_ft = take(FT_SHADOW_N)
i_shadow_non = take(ATTACK_SHADOW_NON_N)
i_eval_non   = take(EVAL_NON_N)

target_ft = [nonmembers_1k[i] for i in i_target_ft]
shadow_ft = [nonmembers_1k[i] for i in i_shadow_ft]
shadow_non = [nonmembers_1k[i] for i in i_shadow_non]
eval_non = [nonmembers_1k[i] for i in i_eval_non]

eval_mem = target_ft  # ✅ target' 的 member 只能是它的训练集（这里就是 100 条）

print("\n[Split]")
print(" target_ft (member for target'):", len(target_ft))
print(" shadow_ft (member for shadow): ", len(shadow_ft))
print(" shadow_non (nonmember shadow):", len(shadow_non))
print(" eval_mem (member target'):", len(eval_mem))
print(" eval_non (nonmember target'):", len(eval_non))
print(" remaining nonmembers unused:", len(idx))

# 保存 split 便于复现实验
split_path = os.path.join(OUT_DIR, "split.json")
with open(split_path, "w", encoding="utf-8") as f:
    json.dump({
        "seed": SEED,
        "FT_TARGET_N": FT_TARGET_N,
        "FT_SHADOW_N": FT_SHADOW_N,
        "ATTACK_SHADOW_NON_N": ATTACK_SHADOW_NON_N,
        "EVAL_NON_N": EVAL_NON_N,
        "target_ft_idx": i_target_ft,
        "shadow_ft_idx": i_shadow_ft,
        "shadow_non_idx": i_shadow_non,
        "eval_non_idx": i_eval_non
    }, f, indent=2)
print("Saved split:", split_path)

# -----------------------------
# 4) 加载 BiomedCLIP
# -----------------------------
MODEL_NAME = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
print("\nLoading:", MODEL_NAME)
base_model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
base_model.eval()
print("BiomedCLIP loaded.")

# -----------------------------
# 5) 数据集（ROCO nonmembers：本地路径）
# -----------------------------
def safe_open_image(path: str) -> Optional[Image.Image]:
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

class RocPairs(Dataset):
    def __init__(self, samples: List[Dict[str, Any]], preprocess, tokenizer):
        self.samples = samples
        self.preprocess = preprocess
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        img_path = s.get("local_image_path", "")
        caption = s.get("caption", "")

        img = safe_open_image(img_path)
        if img is None:
            # 返回一个占位避免 dataloader 崩
            img = Image.new("RGB", (224,224), (128,128,128))
            caption = caption if isinstance(caption, str) else ""

        img_t = self.preprocess(img)
        # open_clip tokenizer 返回 numpy/torch 版本均可，这里统一成 torch.LongTensor
        tok = self.tokenizer([caption])[0]
        tok = torch.as_tensor(tok, dtype=torch.long)
        return img_t, tok

def make_loader(samples, bs, shuffle):
    ds = RocPairs(samples, preprocess, tokenizer)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=True, drop_last=False)

# -----------------------------
# 6) 微调函数（CLIP 对比损失）
# -----------------------------
def clone_model_from_base() -> torch.nn.Module:
    m, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, device=device)
    return m

def clip_contrastive_loss(model, images, tokens):
    # encode
    img_feat = model.encode_image(images)
    txt_feat = model.encode_text(tokens)

    img_feat = F.normalize(img_feat, dim=-1)
    txt_feat = F.normalize(txt_feat, dim=-1)

    # scale
    logit_scale = model.logit_scale.exp()
    logits = logit_scale * img_feat @ txt_feat.t()
    labels = torch.arange(images.size(0), device=images.device)

    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.t(), labels)
    return (loss_i2t + loss_t2i) / 2

def finetune_clip(train_samples, save_name: str):
    model = clone_model_from_base()
    model.train()

    loader = make_loader(train_samples, bs=FINETUNE_BS, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=FINETUNE_LR, weight_decay=FINETUNE_WD)

    scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))

    for ep in range(FINETUNE_EPOCHS):
        ep_loss = 0.0
        n = 0
        t0 = time.time()

        for images, toks in loader:
            images = images.to(device, non_blocking=True)
            toks = toks.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(device=="cuda")):
                loss = clip_contrastive_loss(model, images, toks)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            ep_loss += float(loss.item())
            n += 1

        print(f"[{save_name}] epoch {ep+1}/{FINETUNE_EPOCHS} | loss={ep_loss/max(n,1):.4f} | time={time.time()-t0:.1f}s")

    model.eval()
    save_path = os.path.join(OUT_DIR, f"{save_name}.pt")
    torch.save(model.state_dict(), save_path)
    print("Saved:", save_path)
    return model

# -----------------------------
# 7) 特征提取（M⁴I 的 CLIP 化特征）
# -----------------------------
@torch.no_grad()
def extract_m4i_features(model, samples: List[Dict[str,Any]], batch_size=64) -> np.ndarray:
    loader = make_loader(samples, bs=batch_size, shuffle=False)
    feats = []

    for images, toks in loader:
        images = images.to(device, non_blocking=True)
        toks = toks.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            img = model.encode_image(images)
            txt = model.encode_text(toks)

        img = F.normalize(img.float(), dim=-1)
        txt = F.normalize(txt.float(), dim=-1)

        diff = torch.abs(img - txt)                 # [B, 512]
        cos  = (img * txt).sum(dim=-1, keepdim=True) # [B, 1]
        logit_scale = model.logit_scale.exp().float()
        logit = (logit_scale * cos)                  # [B, 1]

        f = torch.cat([diff, cos, logit], dim=-1)    # [B, 514]
        feats.append(f.cpu().numpy())

    return np.concatenate(feats, axis=0)

# -----------------------------
# 8) 攻击器（小 MLP）
# -----------------------------
class AttackMLP(nn.Module):
    def __init__(self, d_in=514):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

def train_attack_mlp(X, y):
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    y_t = torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(1)

    ds = torch.utils.data.TensorDataset(X_t, y_t)
    dl = DataLoader(ds, batch_size=ATTACK_BS, shuffle=True, drop_last=False)

    model = AttackMLP(d_in=X.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=ATTACK_LR)
    bce = nn.BCEWithLogitsLoss()

    for ep in range(ATTACK_EPOCHS):
        model.train()
        total = 0.0
        for bx, by in dl:
            opt.zero_grad(set_to_none=True)
            logits = model(bx)
            loss = bce(logits, by)
            loss.backward()
            opt.step()
            total += float(loss.item())

        if (ep+1) % 20 == 0:
            model.eval()
            with torch.no_grad():
                p = torch.sigmoid(model(X_t)).detach().cpu().numpy().reshape(-1)
            auc = roc_auc_score(y, p)
            pred = (p >= 0.5).astype(int)
            acc = accuracy_score(y, pred)
            print(f"[Attack] ep {ep+1:3d}/{ATTACK_EPOCHS} | loss={total/max(len(dl),1):.4f} | AUC={auc:.4f} | Acc={acc:.4f}")

    return model

def metrics_at_fpr(y_true, scores, target_fpr=0.05):
    fpr, tpr, thr = roc_curve(y_true, scores)
    idx = np.searchsorted(fpr, target_fpr, side="right")
    if idx == 0:
        thr_star = thr[0]; tpr_star = tpr[0]
    elif idx >= len(thr):
        thr_star = thr[-1]; tpr_star = tpr[-1]
    else:
        x0, x1 = fpr[idx-1], fpr[idx]
        y0, y1 = tpr[idx-1], tpr[idx]
        t0, t1 = thr[idx-1], thr[idx]
        w = (target_fpr - x0) / (x1 - x0 + 1e-12)
        thr_star = t0 + w*(t1 - t0)
        tpr_star = y0 + w*(y1 - y0)

    y_pred = (scores >= thr_star).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr_act = fp / (fp + tn + 1e-12)
    tpr_act = tp / (tp + fn + 1e-12)
    acc = accuracy_score(y_true, y_pred)
    return {
        "thr@5%FPR": float(thr_star),
        "TPR@5%FPR(interp)": float(tpr_star),
        "FPR(actual)": float(fpr_act),
        "TPR(actual)": float(tpr_act),
        "Precision": float(prec),
        "Recall": float(rec),
        "F1": float(f1),
        "Accuracy": float(acc),
        "AUC": float(roc_auc_score(y_true, scores)),
        "CM": cm
    }

# ============================================================
# 9) 主流程开始
# ============================================================
print("\n" + "="*70)
print("STEP 1) 微调 target'（member=target_ft）")
print("="*70)
target_model = finetune_clip(target_ft, "target_finetuned_nonmember100")

print("\n" + "="*70)
print("STEP 2) 微调 shadow（member=shadow_ft）")
print("="*70)
shadow_model = finetune_clip(shadow_ft, "shadow_finetuned_nonmember100")

# ============================================================
# 10) 用 shadow 输出训练攻击器（shadow_ft=1, shadow_non=0）
# ============================================================
print("\n" + "="*70)
print("STEP 3) 提取 shadow 特征并训练攻击器")
print("="*70)
X_shadow_mem = extract_m4i_features(shadow_model, shadow_ft, batch_size=64)
X_shadow_non = extract_m4i_features(shadow_model, shadow_non, batch_size=64)

X_train = np.concatenate([X_shadow_mem, X_shadow_non], axis=0)
y_train = np.array([1]*len(X_shadow_mem) + [0]*len(X_shadow_non), dtype=np.int64)

print("Attack train shapes:", X_train.shape, y_train.shape)
attack = train_attack_mlp(X_train, y_train)

attack_path = os.path.join(OUT_DIR, "attack_mlp.pt")
torch.save(attack.state_dict(), attack_path)
print("Saved attack:", attack_path)

# ============================================================
# 11) 在 target' 上评估（eval_mem=target_ft=1, eval_non=600=0）
# ============================================================
print("\n" + "="*70)
print("STEP 4) 在 target' 上评估攻击器（严格 membership）")
print("="*70)

X_eval_mem = extract_m4i_features(target_model, eval_mem, batch_size=64)
X_eval_non = extract_m4i_features(target_model, eval_non, batch_size=64)

X_eval = np.concatenate([X_eval_mem, X_eval_non], axis=0)
y_eval = np.array([1]*len(X_eval_mem) + [0]*len(X_eval_non), dtype=np.int64)

attack.eval()
with torch.no_grad():
    s = torch.sigmoid(attack(torch.tensor(X_eval, dtype=torch.float32, device=device))).cpu().numpy().reshape(-1)

auc = roc_auc_score(y_eval, s)
pred = (s >= 0.5).astype(int)
acc = accuracy_score(y_eval, pred)
cm = confusion_matrix(y_eval, pred)

print(f"AUC = {auc:.4f}")
print(f"Acc = {acc:.4f}")
print("Confusion [[TN FP][FN TP]]:\n", cm)

m5 = metrics_at_fpr(y_eval, s, target_fpr=0.05)
print("\n@5%FPR:")
print(f" AUC                = {m5['AUC']:.4f}")
print(f" thr@5%FPR          = {m5['thr@5%FPR']:.6f}")
print(f" TPR@5%FPR (interp) = {m5['TPR@5%FPR(interp)']:.4f} | "
      f"实际FPR={m5['FPR(actual)']*100:.2f}% 实际TPR={m5['TPR(actual)']*100:.2f}%")
print(f" Precision          = {m5['Precision']:.4f}")
print(f" Recall             = {m5['Recall']:.4f}")
print(f" F1                 = {m5['F1']:.4f}")
print(f" Accuracy           = {m5['Accuracy']:.4f}")
print(" CM:\n", m5["CM"])

# 保存评估结果
res_path = os.path.join(OUT_DIR, "results.json")
with open(res_path, "w", encoding="utf-8") as f:
    json.dump({
        "seed": SEED,
        "finetune": {
            "epochs": FINETUNE_EPOCHS,
            "bs": FINETUNE_BS,
            "lr": FINETUNE_LR,
            "wd": FINETUNE_WD
        },
        "attack": {
            "epochs": ATTACK_EPOCHS,
            "bs": ATTACK_BS,
            "lr": ATTACK_LR,
            "feat_dim": int(X_train.shape[1])
        },
        "sizes": {
            "target_ft": len(target_ft),
            "shadow_ft": len(shadow_ft),
            "shadow_non": len(shadow_non),
            "eval_mem": len(eval_mem),
            "eval_non": len(eval_non),
        },
        "eval": {
            "AUC": float(auc),
            "Acc@0.5": float(acc),
            "CM@0.5": cm.tolist(),
            "@5%FPR": {
                k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in m5.items()
            }
        }
    }, f, indent=2)

print("\nSaved results:", res_path)
print("OUT_DIR:", OUT_DIR)


# Pubmedclip

## data

In [ ]:
!pip install -U datasets transformers accelerate tqdm
!pip install pillow==10.4.0


In [ ]:
# ============================================================
# PubMedCLIP (HF) + ROCO (HF streaming)
# - Member: ROCO train 取 1000
# - Non-member: ROCO test 取 1000
# - 全部保存到 Colab 本地 /content (不写 Drive)
# - 只用 streaming，不下载全量数据
# - 输出尽量少：每个大步骤只打印 1 行
# ============================================================

import os, json, random, hashlib
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor

# -----------------------------
# 0) Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# -----------------------------
# 1) Local paths (Colab local)
# -----------------------------
ROOT = "/content/pubmedclip_roco_mia"
MODEL_DIR = f"{ROOT}/model_pubmedclip"
MEM_IMG_DIR = f"{ROOT}/members_train_1k/images"
NONMEM_IMG_DIR = f"{ROOT}/nonmembers_test_1k/images"
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
Path(MEM_IMG_DIR).mkdir(parents=True, exist_ok=True)
Path(NONMEM_IMG_DIR).mkdir(parents=True, exist_ok=True)

MEMBERS_JSON = f"{ROOT}/members_roco_train_1k.json"
NONMEMBERS_JSON = f"{ROOT}/nonmembers_roco_test_1k.json"

# -----------------------------
# 2) Load PubMedCLIP from HF (no Drive)
# -----------------------------
MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"[OK] PubMedCLIP loaded ({MODEL_ID}) on {device}")

# -----------------------------
# 3) Stream ROCO from HF
# -----------------------------
DATASET_ID = "mdwiratathya/ROCO-radiology"
roco = load_dataset(DATASET_ID, streaming=True)
print(f"[OK] ROCO streaming loaded ({DATASET_ID}) splits={list(roco.keys())}")

# -----------------------------
# 4) Helper: save image locally + build record
# -----------------------------
def md5_bytes(b: bytes) -> str:
    return hashlib.md5(b).hexdigest()

def save_pil_as_jpg(pil_img: Image.Image, out_path: str):
    pil_img.convert("RGB").save(out_path, format="JPEG", quality=95, optimize=True)

def take_k_samples(stream_iter, k, out_dir, split_name, seed=42):
    """
    从 streaming iterable 中取 k 个样本（顺序扫描 + 去重）
    - 去重策略：对 image bytes 做 md5
    - 返回 records: [{image_id, caption, split, local_image_path}]
    """
    records = []
    seen_hash = set()

    pbar = tqdm(total=k, desc=f"{split_name}", leave=True)
    for ex in stream_iter:
        if len(records) >= k:
            break

        img = ex.get("image", None)
        cap = ex.get("caption", "") or ""
        image_id = ex.get("image_id", None)

        if img is None:
            continue

        if not isinstance(img, Image.Image):
            try:
                img = Image.fromarray(img)
            except:
                continue

        try:
            b = img.tobytes()
        except:
            img = img.convert("RGB")
            b = img.tobytes()

        h = md5_bytes(b)
        if h in seen_hash:
            continue
        seen_hash.add(h)

        if image_id is None or str(image_id).strip() == "":
            image_id = f"{split_name}_{h[:12]}"

        fn = f"{str(image_id)}.jpg".replace("/", "_")
        local_path = os.path.join(out_dir, fn)
        save_pil_as_jpg(img, local_path)

        records.append({
            "image_id": str(image_id),
            "caption": cap,
            "split": split_name,
            "local_image_path": local_path
        })
        pbar.update(1)

    pbar.close()
    return records

# -----------------------------
# 5) Collect member/nonmember (streaming)
# -----------------------------
MEM_K = 1000
NONMEM_K = 1000

members = take_k_samples(
    roco["train"], MEM_K, MEM_IMG_DIR, split_name="member(roco_train)", seed=SEED
)
nonmembers = take_k_samples(
    roco["test"], NONMEM_K, NONMEM_IMG_DIR, split_name="nonmember(roco_test)", seed=SEED+1
)
print(f"[OK] Collected members={len(members)} nonmembers={len(nonmembers)}")

# -----------------------------
# 6) Save JSON locally
# -----------------------------
with open(MEMBERS_JSON, "w", encoding="utf-8") as f:
    json.dump(members, f, ensure_ascii=False, indent=2)

with open(NONMEMBERS_JSON, "w", encoding="utf-8") as f:
    json.dump(nonmembers, f, ensure_ascii=False, indent=2)

print(f"[OK] Saved JSON -> {MEMBERS_JSON} | {NONMEMBERS_JSON}")

# -----------------------------
# 7) Sanity check: show 8 images from each
# -----------------------------
from IPython.display import display

def preview(records, k=8, title=""):
    print(f"\n=== {title} (show {min(k, len(records))}/{len(records)}) ===")
    for s in random.sample(records, k=min(k, len(records))):
        try:
            img = Image.open(s["local_image_path"]).convert("RGB")
            print("image_id:", s["image_id"])
            print("caption:", (s["caption"] or "")[:160].replace("\n"," "))
            display(img)
        except Exception as e:
            print("skip:", e)

preview(members, k=8, title="Members")
preview(nonmembers, k=8, title="Non-members")

# -----------------------------
# 8) Optional: quick forward check on 1 sample
# -----------------------------
one = members[0]
img = Image.open(one["local_image_path"]).convert("RGB")
text = one["caption"] if one["caption"] else "medical image"

inputs = processor(images=img, text=[text], return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    out = model(**inputs)

print(f"[OK] Forward check: image_embeds={tuple(out.image_embeds.shape)} text_embeds={tuple(out.text_embeds.shape)}")


## Loss MIA

In [ ]:
# =========================================================================
# Loss Attack MIA for PubMedCLIP (Full Runnable Script)
#
# Method: Log-Likelihood (Loss) Attack
# Logic:  Calculate log(P(True_Caption | Image)) among K candidates.
#         Members are expected to have higher probability (closer to 0 log-score).
#
# Model:  flaviagiammarino/pubmed-clip-vit-base-patch32
#
# Requirements:
#   pip install torch transformers pillow scikit-learn tqdm numpy
# =========================================================================

import os, json, random, warnings
from typing import Dict, Any, Optional, List, Tuple
from collections import OrderedDict

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Hyper-parameters & Paths
# ---------------------------
# 请修改为你的实际 JSON 路径
MEMBERS_JSON     = "/content/pubmedclip_roco_mia/members_roco_train_1k.json"
NONMEMBERS_JSON  = "/content/pubmedclip_roco_mia/nonmembers_roco_test_1k.json"

MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"

SEED = 42
NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

# Attack Config
K = 8          # 1 Positive + (K-1) Negatives
R = 8          # Repeat R times to reduce variance from random negatives
CLIP_MAX_LEN = 77  # Critical for CLIP

# ---------------------------
# 1) Setup Device & Seed
# ---------------------------
assert torch.cuda.is_available(), "CUDA GPU is required for efficiency."
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed(SEED)
print(f"✅ Device: {device} | Seed: {SEED}")

# ---------------------------
# 2) Load PubMedCLIP Model
# ---------------------------
print(f"⏳ Loading model: {MODEL_ID} ...")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

# Get Logit Scale (Temperature)
with torch.no_grad():
    try:
        # Standard CLIP stores logit_scale as a learnable parameter
        LOGIT_SCALE = float(model.logit_scale.exp().detach().cpu())
    except Exception:
        LOGIT_SCALE = 1.0

print(f"✅ Model loaded | Logit Scale: {LOGIT_SCALE:.3f}")

# ---------------------------
# 3) Load Data
# ---------------------------
def load_json_list(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"JSON not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("⏳ Loading datasets...")
members_all = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)

# Shuffle and slice
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)
members_all = members_all[:NUM_MEMBERS]
nonmembers_all = nonmembers_all[:NUM_NONMEMBERS]

print(f"📦 Samples: members={len(members_all)} | nonmembers={len(nonmembers_all)}")

# ---------------------------
# 4) Helper: Prepare Negative Pool
# ---------------------------
# Use ONLY non-member captions for negatives to be fair/clean
def clean_cap(x: str) -> str:
    return str(x).replace("\n", " ").strip()

neg_pool = [clean_cap(x.get("caption", "")) for x in nonmembers_all]
neg_pool = [c for c in neg_pool if c]  # filter empty
if len(neg_pool) < K:
    raise ValueError(f"Not enough non-member captions for negatives! Need at least {K}.")

print(f"🧰 Negative Pool Size: {len(neg_pool)}")

# ---------------------------
# 5) Helper: Image Loading & Encoding
# ---------------------------
# Simple LRU Cache to avoid disk thrashing
_IMG_CACHE = OrderedDict()
_MAX_CACHE = 512

def get_image(path):
    if path in _IMG_CACHE:
        _IMG_CACHE.move_to_end(path)
        return _IMG_CACHE[path]
    try:
        img = Image.open(path).convert("RGB")
        _IMG_CACHE[path] = img
        if len(_IMG_CACHE) > _MAX_CACHE:
            _IMG_CACHE.popitem(last=False)
        return img
    except Exception:
        return None

@torch.inference_mode()
def encode_images_batch(sample_list, bs=64):
    """
    Pre-encode all images to speed up the loop.
    Returns: Tensor [N, D] (normalized) on CPU
             List[str] valid captions
             List[int] valid indices
    """
    valid_embs = []
    valid_caps = []

    # We process in batches
    buffer_imgs = []
    buffer_caps = []

    for item in tqdm(sample_list, desc="Encoding Images"):
        path = item.get("local_image_path")
        cap = clean_cap(item.get("caption", ""))

        if not path or not cap:
            continue

        img = get_image(path)
        if img is None:
            continue

        buffer_imgs.append(img)
        buffer_caps.append(cap)

        if len(buffer_imgs) >= bs:
            # Process batch
            inputs = processor(images=buffer_imgs, return_tensors="pt").to(device)
            feats = model.get_image_features(**inputs)
            feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
            valid_embs.append(feats.cpu())

            valid_caps.extend(buffer_caps)
            buffer_imgs = []
            buffer_caps = []

    # Process remaining
    if buffer_imgs:
        inputs = processor(images=buffer_imgs, return_tensors="pt").to(device)
        feats = model.get_image_features(**inputs)
        feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
        valid_embs.append(feats.cpu())
        valid_caps.extend(buffer_caps)

    if not valid_embs:
        return None, [], []

    final_embs = torch.cat(valid_embs, dim=0)
    return final_embs, valid_caps

# ---------------------------
# 6) Helper: Text Encoding (On-the-fly)
# ---------------------------
@torch.inference_mode()
def encode_texts(text_list):
    """
    Encode a small batch of texts (size K).
    Returns: [K, D] normalized on GPU
    """
    inputs = processor(
        text=text_list,
        padding=True,
        truncation=True,
        max_length=CLIP_MAX_LEN,  # <-- FIX applied
        return_tensors="pt"
    ).to(device)

    feats = model.get_text_features(**inputs)
    feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
    return feats

# ---------------------------
# 7) Core Logic: Score Dataset
# ---------------------------
@torch.inference_mode()
def score_dataset_loss(img_embs_cpu, pos_caps, seed_offset=0):
    """
    img_embs_cpu: [N, D]
    pos_caps: List[str] length N
    Returns: numpy array of scores [N]
    """
    N = len(pos_caps)
    scores = np.zeros(N, dtype=np.float32)
    local_rng = np.random.default_rng(SEED + seed_offset)

    # Move huge neg pool to a simple list for fast sampling
    pool_size = len(neg_pool)

    for i in tqdm(range(N), desc="Calculating Loss Scores"):
        # 1. Get Image Embedding
        img_vec = img_embs_cpu[i].to(device) # [D]
        pos_txt = pos_caps[i]

        sample_logs = []

        # 2. Repeat R trials
        for _ in range(R):
            # Sample K-1 negatives
            negs = []
            while len(negs) < (K - 1):
                # Random index
                idx = local_rng.integers(0, pool_size)
                cand = neg_pool[idx]
                # Simple dedup: don't use the exact same text as positive
                if cand != pos_txt:
                    negs.append(cand)

            # Candidates: [Positive, Neg1, Neg2, ..., NegK-1]
            candidates = [pos_txt] + negs

            # Encode Texts [K, D]
            txt_feats = encode_texts(candidates)

            # Compute Logits: scale * (txt @ img) -> shape [K]
            # txt_feats [K, D] @ img_vec [D] -> [K]
            sims = torch.mv(txt_feats, img_vec)
            logits = LOGIT_SCALE * sims

            # Softmax to get probabilities
            probs = torch.softmax(logits, dim=0)

            # We want the log-probability of the positive class (index 0)
            # log(p_0)
            log_p_pos = torch.log(probs[0].clamp_min(1e-9))
            sample_logs.append(log_p_pos.item())

        # 3. Average over R trials
        scores[i] = np.mean(sample_logs)

    return scores

# ---------------------------
# 8) Execute Attack
# ---------------------------

print("\n[Step] Encoding Member Images...")
mem_embs, mem_caps = encode_images_batch(members_all)
print(f"  > Encoded {len(mem_caps)} members.")

print("\n[Step] Encoding Non-Member Images...")
non_embs, non_caps = encode_images_batch(nonmembers_all)
print(f"  > Encoded {len(non_caps)} non-members.")

if mem_embs is None or non_embs is None:
    raise RuntimeError("Failed to encode images. Check paths.")

print("\n[Step] Scoring Members (Loss Attack)...")
mem_scores = score_dataset_loss(mem_embs, mem_caps, seed_offset=100)

print("\n[Step] Scoring Non-Members (Loss Attack)...")
non_scores = score_dataset_loss(non_embs, non_caps, seed_offset=200)

# ---------------------------
# 9) Evaluation Metrics
# ---------------------------
# Label: 1 for Member, 0 for Non-Member
y_true = np.concatenate([np.ones(len(mem_scores)), np.zeros(len(non_scores))])
y_scores = np.concatenate([mem_scores, non_scores])

# AUC
auc = roc_auc_score(y_true, y_scores)
fpr, tpr, thresholds = roc_curve(y_true, y_scores)

# Best Threshold (Youden's J statistic)
J = tpr - fpr
best_idx = np.argmax(J)
best_thr = thresholds[best_idx]

# Accuracy at Best Threshold
y_pred_best = (y_scores >= best_thr).astype(int)
acc_best = accuracy_score(y_true, y_pred_best)
cm_best = confusion_matrix(y_true, y_pred_best)

# Fixed FPR @ 5%
# We want FPR <= 0.05. Since "Member" is positive class (1),
# FPR is determined by Non-Member scores (0).
# Threshold should be the 95th percentile of Non-Member scores.
thr_5fpr = np.percentile(non_scores, 95)
y_pred_5 = (y_scores >= thr_5fpr).astype(int)

cm_5 = confusion_matrix(y_true, y_pred_5)
tn, fp, fn, tp = cm_5.ravel()
actual_fpr = fp / (fp + tn + 1e-12)
recall_5 = recall_score(y_true, y_pred_5)
prec_5 = precision_score(y_true, y_pred_5)

# ---------------------------
# 10) Print Results
# ---------------------------
def print_stats(name, arr):
    print(f"{name}: Mean={np.mean(arr):.4f}, Std={np.std(arr):.4f}")

print("\n" + "="*40)
print("     MIA RESULTS: Loss Attack (PubMedCLIP)")
print("="*40)
print_stats("Member Scores (LogLikelihood)", mem_scores)
print_stats("NonMember Scores (LogLikelihood)", non_scores)
print("-" * 40)
print(f"AUC Score:               {auc:.4f}")
print(f"Best Accuracy:           {acc_best:.4f} (Thr={best_thr:.4f})")
print(f"Confusion Matrix (Best):\n{cm_best}")
print("-" * 40)
print(f"Metrics @ Fixed 5% FPR:")
print(f"  Threshold:             {thr_5fpr:.4f}")
print(f"  Actual FPR:            {actual_fpr*100:.2f}%")
print(f"  TP Rate (Recall):      {recall_5*100:.2f}%")
print(f"  Precision:             {prec_5*100:.2f}%")
print(f"  Confusion Matrix:\n{cm_5}")
print("="*40)

## Entropy

In [ ]:
# =========================================================================
# Entropy-based MIA for PubMedCLIP (Transformers CLIPModel)
#
# 设计与 BiomedCLIP 版本一致：
#   - 对每个样本，构造 K-way 候选 caption 集合：1 个正例 + (K-1) 个负例
#   - logits = logit_scale * cos(img_emb, text_emb)
#   - p = softmax(logits)
#   - score = -Entropy(p)   （R 次随机负例抽样取平均，降低方差）
#
# 你要求的修复（同前）：
#   ✅ target/neg 每个样本只加载一次（PIL LRU cache）
#   ✅ neg 失败就重采样
#   ✅ 不用黑图补位；凑不齐K直接丢弃该样本
#   ✅ 文本强制 truncation 到 77（避免 CLIP max_position_embeddings 报错）
#
# 运行前提：
#   - members_all / nonmembers_all 已经在内存里（list[dict]），每条至少包含：
#       {"local_image_path": "...", "caption": "..."}
#     若没有，也可用 JSON 路径加载（下面提供）
#   - PubMedCLIP 模型来自 HF：
#       MODEL_ID="flaviagiammarino/pubmed-clip-vit-base-patch32"
#   - K=8, R=8 默认按你要求（可改）
# =========================================================================

import os, json, random, warnings
from typing import Dict, Any, Optional, List, Tuple
from collections import OrderedDict

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from transformers import CLIPModel, CLIPProcessor

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

# ---------------------------
# 0) 参数
# ---------------------------
MEMBERS_JSON     = "/content/pubmedclip_roco_mia/members_roco_train_1k.json"
NONMEMBERS_JSON  = "/content/pubmedclip_roco_mia/nonmembers_roco_test_1k.json"

MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"

SEED = 42
NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

K = 8          # 1 pos + (K-1) neg
R = 8          # 每个样本重复采样次数（降方差）

# 负例池的大小：从池里抽 K-1 个
# 如果你想更“LLM-like”，可以让负例池固定为 nonmember captions
NEG_POOL_SIZE = 1000

CLIP_MAX_LEN = 77

# ---------------------------
# 1) Device
# ---------------------------
assert torch.cuda.is_available(), "需要 CUDA GPU"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

# ---------------------------
# 2) Load PubMedCLIP
# ---------------------------
print(f"⏳ Loading model: {MODEL_ID}")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

with torch.no_grad():
    try:
        LOGIT_SCALE = float(model.logit_scale.exp().detach().cpu())
    except Exception:
        LOGIT_SCALE = 1.0
print(f"✅ PubMedCLIP ready | logit_scale={LOGIT_SCALE:.3f} | max_len={CLIP_MAX_LEN}")

# ---------------------------
# 3) Load data
# ---------------------------
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

if "members_all" not in globals() or "nonmembers_all" not in globals():
    members_all = load_json_list(MEMBERS_JSON)
    nonmembers_all = load_json_list(NONMEMBERS_JSON)

random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)
members_all = members_all[:NUM_MEMBERS]
nonmembers_all = nonmembers_all[:NUM_NONMEMBERS]
print(f"📦 Samples: members={len(members_all)} | nonmembers={len(nonmembers_all)}")

# ---------------------------
# 4) Image LRU cache (避免重复打开/重复decode)
# ---------------------------
_IMG_CACHE = OrderedDict()
_IMG_CACHE_MAX = 512

def _key(obj: Dict[str,Any]) -> str:
    return str(obj.get("local_image_path",""))

def _cache_get(k: str):
    if k in _IMG_CACHE:
        _IMG_CACHE.move_to_end(k)
        return _IMG_CACHE[k]
    return None

def _cache_put(k: str, v: Image.Image):
    _IMG_CACHE[k] = v
    _IMG_CACHE.move_to_end(k)
    if len(_IMG_CACHE) > _IMG_CACHE_MAX:
        _IMG_CACHE.popitem(last=False)

def load_img_cached(obj: Dict[str,Any]) -> Optional[Image.Image]:
    k = _key(obj)
    if not k:
        return None
    cached = _cache_get(k)
    if cached is not None:
        return cached
    p = obj.get("local_image_path")
    if not p or not os.path.exists(p):
        return None
    try:
        im = Image.open(p).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError, Exception):
        return None
    _cache_put(k, im)
    return im

# ---------------------------
# 5) Encode helpers
# ---------------------------
@torch.inference_mode()
def encode_image_one(img_pil: Image.Image) -> torch.Tensor:
    # returns [D] on CPU float32 (normalized)
    inputs = processor(images=img_pil, return_tensors="pt").to(device)
    feat = model.get_image_features(pixel_values=inputs["pixel_values"])
    feat = feat / (feat.norm(dim=-1, keepdim=True) + 1e-12)
    return feat.squeeze(0).float().cpu()

@torch.inference_mode()
def encode_text_many(texts: List[str]) -> torch.Tensor:
    # returns [K, D] on CPU float32 (normalized)
    inputs = processor(
        text=texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CLIP_MAX_LEN
    ).to(device)
    feat = model.get_text_features(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
    feat = feat / (feat.norm(dim=-1, keepdim=True) + 1e-12)
    return feat.float().cpu()

# ---------------------------
# 6) Candidate caption pool (negatives)
#    - 默认用 nonmembers captions（更干净，且符合“负例来自未见过的分布”）
# ---------------------------
# 收集可用 captions（去空）
all_non_caps = [str(x.get("caption","")).strip() for x in nonmembers_all]
all_non_caps = [c for c in all_non_caps if c]
if len(all_non_caps) < (K - 1):
    raise RuntimeError("Nonmember captions 太少，无法采样负例。")

# 固定一个负例池（大小=NEG_POOL_SIZE 或全部）
rng = random.Random(SEED)
rng.shuffle(all_non_caps)
neg_pool_caps = all_non_caps[:min(NEG_POOL_SIZE, len(all_non_caps))]
print(f"🧰 Negative caption pool size = {len(neg_pool_caps)}")

# ---------------------------
# 7) Entropy scoring
# ---------------------------
def neg_entropy_score(img_vec_cpu: torch.Tensor, pos_caption: str, K: int = 8, R: int = 8) -> Optional[float]:
    """
    img_vec_cpu: [D] CPU float32 normalized
    pos_caption: 正例 caption
    返回：R 次试验的 (-entropy) 平均；失败返回 None
    """
    pos_caption = str(pos_caption).strip()
    if not pos_caption:
        return None

    # 为避免“pos caption 恰好也在 neg pool”带来的重复，我们采样时会过滤相同字符串
    scores = []
    for _ in range(R):
        # sample K-1 negatives
        # 允许文本重复概率很低；我们做简单过滤，最多尝试几次凑齐
        negs = []
        tries = 0
        while len(negs) < (K - 1) and tries < 200:
            tries += 1
            c = random.choice(neg_pool_caps)
            if c == pos_caption:
                continue
            negs.append(c)
        if len(negs) < (K - 1):
            return None

        cand_caps = [pos_caption] + negs  # pos at index 0
        # encode texts
        txt_mat = encode_text_many(cand_caps)  # [K,D] CPU
        # logits = scale * cos = scale * (txt @ img)
        logits = LOGIT_SCALE * (txt_mat @ img_vec_cpu)  # [K]
        p = torch.softmax(logits, dim=0)
        ent = -(p * (p.clamp_min(1e-9).log())).sum()
        scores.append(float((-ent).item()))
    return float(np.mean(scores)) if scores else None

def score_dataset(samples: List[Dict[str,Any]], tag: str) -> Tuple[List[float], int]:
    """
    对一个集合打分；返回 scores 列表、以及失败数
    """
    out = []
    fail = 0
    for s in tqdm(samples, desc=f"scoring-{tag}"):
        img = load_img_cached(s)
        cap = str(s.get("caption","")).strip()
        if img is None or not cap:
            fail += 1
            continue

        img_vec = encode_image_one(img)  # [D] CPU
        sc = neg_entropy_score(img_vec, cap, K=K, R=R)
        if sc is None or (not np.isfinite(sc)):
            fail += 1
            continue
        out.append(sc)
    return out, fail

print(f"\n[Config] K={K}, R={R}, neg_pool={len(neg_pool_caps)}, logit_scale={LOGIT_SCALE:.3f}")

print("\n[Step] Scoring members...")
mem_scores, mem_fail = score_dataset(members_all, "members")

print("\n[Step] Scoring nonmembers...")
non_scores, non_fail = score_dataset(nonmembers_all, "nonmembers")

print("\n[Done] valid scores:")
print(f"  members   ok={len(mem_scores)}  fail={mem_fail}")
print(f"  nonmembers ok={len(non_scores)}  fail={non_fail}")

if len(mem_scores) == 0 or len(non_scores) == 0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

# ---------------------------
# 8) Evaluate (AUC / best thr / @5%FPR)
# ---------------------------
y_true = np.array([1]*len(mem_scores) + [0]*len(non_scores))
scores = np.array(mem_scores + non_scores, dtype=np.float64)

auc = roc_auc_score(y_true, scores)
fpr, tpr, thr = roc_curve(y_true, scores)
best_idx = np.argmax(tpr - fpr)      # Youden J
best_thr = thr[best_idx]
y_pred = (scores >= best_thr).astype(int)
acc = accuracy_score(y_true, y_pred)
cm  = confusion_matrix(y_true, y_pred)

def stats(arr, name):
    arr = np.asarray(arr, dtype=np.float64)
    print(f"{name}: n={len(arr)}, mean={arr.mean():.4f}, median={np.median(arr):.4f}, "
          f"std={arr.std():.4f}, min={arr.min():.4f}, max={arr.max():.4f}")

print("\n====== Entropy-MIA (PubMedCLIP) ======")
print(f"AUC = {auc:.4f}")
print(f"Best threshold (Youden J) = {best_thr:.4f}")
print(f"Accuracy = {acc:.4f}")
print("Confusion matrix [[TN FP][FN TP]]:")
print(cm)
print()
stats(mem_scores, "members (-entropy)")
stats(non_scores, "nonmembers (-entropy)")

# @5% FPR (threshold = 95% quantile of nonmember scores)
thr_5 = float(np.quantile(non_scores, 0.95))
y_pred_5 = (scores >= thr_5).astype(int)
cm_5 = confusion_matrix(y_true, y_pred_5)
tn, fp, fn, tp = cm_5.ravel()
fpr_actual = fp / (fp + tn + 1e-12)
tpr_5 = tp / (tp + fn + 1e-12)
prec_5 = precision_score(y_true, y_pred_5, zero_division=0)
rec_5  = recall_score(y_true, y_pred_5, zero_division=0)
f1_5   = f1_score(y_true, y_pred_5, zero_division=0)

print("\n------ @5% FPR Metrics ------")
print(f"Threshold @FPR=5% = {thr_5:.4f}  (实际 FPR={fpr_actual*100:.2f}%)")
print(f"TPR (Recall)      = {tpr_5*100:.2f}%")
print(f"Precision          = {prec_5*100:.2f}%")
print(f"F1                 = {f1_5:.4f}")
print("Confusion @5%FPR [[TN FP][FN TP]]:")
print(cm_5)


## Mink

In [ ]:
# =========================================================================
# Min-K% MIA for PubMedCLIP (Transformers CLIPModel)  ✅按你要求“和 BiomedCLIP 版本一模一样”
#
# Score:
#   - 对每个样本做 R 次 trial
#   - 每次构造 K-way captions: 1 pos + (K-1) neg（neg 从 nonmember caption pool 采样）
#   - logits = logit_scale * cos(txt, img)
#   - nll_pos = -log_softmax(logits)[pos]
#   - Min-K%: 取最小的 ceil(R*p) 个 nll_pos 平均 -> score = -(mean_min)
#     （score 越大越 member-like）
#
# 关键修复（你刚才指出的 confound）：
#   ✅ 图像 loader 只读一次：本地 local_image_path + PIL LRU cache（不重复读/不黑图补位）
#   ✅ 负例如果凑不齐（caption 太少/空）直接跳过该 trial 或样本
#   ✅ 文本强制 truncation 到 CLIP_MAX_LEN=77（避免 max_position_embeddings 报错）
#   ✅ 负例池固定来自 nonmember captions（避免 member captions 混入负例带来的不稳）
#
# 运行前提：
#   - 你已经生成并保存了 ROCO train/test 1k 的 JSON（本地 /content），格式至少包含：
#       {"local_image_path": "...", "caption": "..."}
#   - 你已经能从 HF 拉到 PubMedCLIP：
#       flaviagiammarino/pubmed-clip-vit-base-patch32
# =========================================================================

import os, json, random, warnings
from typing import Dict, Any, Optional, List, Tuple
from collections import OrderedDict

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from transformers import CLIPModel, CLIPProcessor

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Paths & hyperparams
# ---------------------------
MEMBERS_JSON     = "/content/pubmedclip_roco_mia/members_roco_train_1k.json"
NONMEMBERS_JSON  = "/content/pubmedclip_roco_mia/nonmembers_roco_test_1k.json"

MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

K_candidates = 8     # 你要求：K=8
R_trials     = 8     # 你要求：R=8
P_frac       = 0.25  # Min-K% fraction（同你给的 BiomedCLIP 版本）

# 固定负例池大小（从 nonmember captions 中取，默认=1000）
NEG_POOL_SIZE = 1000

# CLIP text max length (HF CLIP max_position_embeddings=77)
CLIP_MAX_LEN = 77

# PIL LRU cache size（避免重复 decode）
IMG_CACHE_MAX = 512

# ---------------------------
# 1) Device & model
# ---------------------------
assert torch.cuda.is_available(), "需要 CUDA GPU"
device="cuda"
tbc.allow_tf32=True; cudnn.allow_tf32=True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

print(f"⏳ Loading PubMedCLIP: {MODEL_ID}")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

with torch.no_grad():
    try:
        LOGIT_SCALE = float(model.logit_scale.exp().detach().cpu())
    except Exception:
        LOGIT_SCALE = 1.0
print(f"✅ PubMedCLIP ready | logit_scale={LOGIT_SCALE:.3f} | max_len={CLIP_MAX_LEN}")

# ---------------------------
# 2) Data
# ---------------------------
def load_json_list(p: str):
    with open(p,"r",encoding="utf-8") as f:
        return json.load(f)

members_all    = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)

members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 Samples: members={len(members)} | non-members={len(nonmembers)}")

# ---------------------------
# 3) Image loader (本地) + LRU cache
# ---------------------------
_IMG_CACHE = OrderedDict()

def _img_key(s: Dict[str,Any]) -> str:
    return str(s.get("local_image_path",""))

def _cache_get(k: str):
    if k in _IMG_CACHE:
        _IMG_CACHE.move_to_end(k)
        return _IMG_CACHE[k]
    return None

def _cache_put(k: str, v: Image.Image):
    _IMG_CACHE[k] = v
    _IMG_CACHE.move_to_end(k)
    if len(_IMG_CACHE) > IMG_CACHE_MAX:
        _IMG_CACHE.popitem(last=False)

def load_image_cached(s: Dict[str,Any]) -> Optional[Image.Image]:
    p = s.get("local_image_path")
    if not p or (not os.path.exists(p)):
        return None
    k = _img_key(s)
    cached = _cache_get(k)
    if cached is not None:
        return cached
    try:
        im = Image.open(p).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError, Exception):
        return None
    _cache_put(k, im)
    return im

# ---------------------------
# 4) Encode helpers
# ---------------------------
@torch.inference_mode()
def encode_image_one(img_pil: Image.Image) -> torch.Tensor:
    """return: [D] CPU float32 normalized"""
    inputs = processor(images=img_pil, return_tensors="pt").to(device)
    z = model.get_image_features(pixel_values=inputs["pixel_values"])
    z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
    return z.squeeze(0).float().cpu()

@torch.inference_mode()
def encode_text_many(texts: List[str]) -> torch.Tensor:
    """return: [K,D] CPU float32 normalized"""
    inputs = processor(
        text=texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CLIP_MAX_LEN
    ).to(device)
    z = model.get_text_features(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
    z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
    return z.float().cpu()

# ---------------------------
# 5) Prepare NEG caption pool (固定来自 nonmember captions)
# ---------------------------
non_caps = [str(x.get("caption","")).strip() for x in nonmembers]
non_caps = [c for c in non_caps if c]
if len(non_caps) < (K_candidates - 1):
    raise RuntimeError("Nonmember captions 太少，无法采样负例。")

rng = random.Random(SEED)
rng.shuffle(non_caps)
neg_pool_caps = non_caps[:min(NEG_POOL_SIZE, len(non_caps))]
print(f"🧰 Negative caption pool size (nonmember-only) = {len(neg_pool_caps)}")

# ---------------------------
# 6) Min-K% score
# ---------------------------
def minK_frac_score(
    img_vec_cpu: torch.Tensor,
    pos_caption: str,
    Kcand: int,
    R: int,
    p: float
) -> Optional[float]:
    """
    img_vec_cpu: [D] CPU normalized
    pos_caption: str
    Return: score = -mean(min ceil(R*p) NLLs) ; bigger => member-like
    """
    pos_caption = str(pos_caption).strip()
    if not pos_caption:
        return None

    nlls = []
    for _ in range(R):
        # sample K-1 negatives from fixed pool (exclude exact same string)
        negs = []
        tries = 0
        while len(negs) < (Kcand - 1) and tries < 300:
            tries += 1
            c = random.choice(neg_pool_caps)
            if c == pos_caption:
                continue
            negs.append(c)
        if len(negs) < (Kcand - 1):
            return None

        cand_caps = [pos_caption] + negs  # pos at index 0
        txt_mat = encode_text_many(cand_caps)  # [K,D] CPU
        logits = LOGIT_SCALE * (txt_mat @ img_vec_cpu)  # [K]
        logp = torch.log_softmax(logits, dim=0)
        nll = float((-logp[0]).item())
        if np.isfinite(nll):
            nlls.append(nll)

    if len(nlls) == 0:
        return None

    nlls = np.array(nlls, dtype=np.float64)
    k = max(1, int(np.ceil(len(nlls) * p)))  # how many to keep (smallest)
    best = np.partition(nlls, k-1)[:k].mean()
    return float(-best)

# ---------------------------
# 7) Score & eval
# ---------------------------
def score_samples(samples: List[Dict[str,Any]], tag: str) -> Tuple[List[float], int]:
    scores = []
    fail = 0
    for s in tqdm(samples, desc=f"scoring-{tag}"):
        cap = str(s.get("caption","")).strip()
        if not cap:
            fail += 1
            continue
        img = load_image_cached(s)
        if img is None:
            fail += 1
            continue
        img_vec = encode_image_one(img)
        sc = minK_frac_score(img_vec, cap, K_candidates, R_trials, P_frac)
        if sc is None or (not np.isfinite(sc)):
            fail += 1
            continue
        scores.append(sc)
    return scores, fail

print(f"\n[Config] K={K_candidates}, R={R_trials}, p={P_frac}, neg_pool={len(neg_pool_caps)}, logit_scale={LOGIT_SCALE:.3f}")

print("\n[Step] Scoring members...")
mem_scores, mem_fail = score_samples(members, "members")

print("\n[Step] Scoring nonmembers...")
non_scores, non_fail = score_samples(nonmembers, "nonmembers")

print(f"\n有效样本数：mem={len(mem_scores)} (fail={mem_fail}) | non={len(non_scores)} (fail={non_fail})")
if len(mem_scores) == 0 or len(non_scores) == 0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

y = np.array([1]*len(mem_scores) + [0]*len(non_scores))
scores = np.array(mem_scores + non_scores, dtype=np.float64)

auc = roc_auc_score(y, scores)
fpr, tpr, thr = roc_curve(y, scores)
j = np.argmax(tpr - fpr)
best_thr = thr[j]
acc = accuracy_score(y, (scores >= best_thr).astype(int))
cm  = confusion_matrix(y, (scores >= best_thr).astype(int))

print("\n====== Min-K% MIA 结果 (PubMedCLIP) ======")
print(f"AUC = {auc:.4f}")
print(f"Best threshold (Youden J) = {best_thr:.4f}")
print(f"Accuracy = {acc:.4f}")
print("Confusion matrix [[TN FP][FN TP]]:\n", cm)

# -------- @5% FPR metrics --------
thr_5 = float(np.quantile(non_scores, 0.95))
y_pred_5 = (scores >= thr_5).astype(int)

cm_5 = confusion_matrix(y, y_pred_5)
tn, fp, fn, tp = cm_5.ravel()
fpr_actual = fp / (fp + tn + 1e-12)
tpr_5 = tp / (tp + fn + 1e-12)
prec_5 = precision_score(y, y_pred_5, zero_division=0)
rec_5  = recall_score(y, y_pred_5, zero_division=0)
f1_5   = f1_score(y, y_pred_5, zero_division=0)

print("\n------ @5% FPR Metrics ------")
print(f"Threshold @FPR=5% = {thr_5:.4f}  (实际 FPR={fpr_actual*100:.2f}%)")
print(f"TPR (Recall)      = {tpr_5*100:.2f}%")
print(f"Precision          = {prec_5*100:.2f}%")
print(f"F1                 = {f1_5:.4f}")
print("Confusion @5%FPR [[TN FP][FN TP]]：")
print(cm_5)


## Min-k++

In [ ]:
# =========================================================================
# Min-K++ MIA for PubMedCLIP (Transformers CLIPModel)  ✅你给的 BiomedCLIP Min-K++ 版本“等价迁移”
#
# Score (Min-K++):
#  - 对每个 image：
#    1) 用该 image embedding 在文本库里做 top-N 检索，构建 hard negative pool（排除正例）
#    2) 做 R 次 trial；每次 trial：
#       - 采样 K-1 个负例：HARD_RATIO 来自 hard pool，其余来自 global pool
#       - 计算 logits_raw = logit_scale * cos(txt, img)
#       - 对每个 tau in TAUS：计算 NLL_pos = -log_softmax(logits_raw / tau)[0]
#    3) 收集 R * len(TAUS) 个 NLL，取最小的 ceil(P_FRAC * count) 个均值
#       score = -mean_min  （越大越 member-like）
#
# 关键修复（消除 confound）：
#   ✅ 图像只从本地 local_image_path 读取 + PIL LRU cache，不重复读取/不黑图补位
#   ✅ 文本强制 truncation 到 CLIP_MAX_LEN=77（HF CLIP position embeddings 限制）
#
# 运行前提（你前面已经跑好）：
#   - members_roco_train_1k.json / nonmembers_roco_test_1k.json 在 /content 下
#     每条至少包含 {"local_image_path": "...", "caption": "..."}
#   - 模型：flaviagiammarino/pubmed-clip-vit-base-patch32 可从 HF 拉取
# =========================================================================

import os, json, random, warnings
from typing import Dict, Any, Optional, List, Tuple
from collections import OrderedDict

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from transformers import CLIPModel, CLIPProcessor

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Paths & hyperparams
# ---------------------------
MEMBERS_JSON     = "/content/pubmedclip_roco_mia/members_roco_train_1k.json"
NONMEMBERS_JSON  = "/content/pubmedclip_roco_mia/nonmembers_roco_test_1k.json"

MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

# ---- Min-K++ params (对齐你给的 BiomedCLIP 版本) ----
K          = 32
R          = 8
P_FRAC     = 0.25

HARD_TOP   = 256
HARD_RATIO = 0.75
TAUS       = [1.0, 0.8, 1.2]

# CLIP text max length (HF CLIP: max_position_embeddings=77)
CLIP_MAX_LEN = 77

# PIL LRU cache size
IMG_CACHE_MAX = 512

# ---------------------------
# 1) Device & model
# ---------------------------
assert torch.cuda.is_available(), "需要 CUDA GPU"
device = "cuda"
tbc.allow_tf32=True; cudnn.allow_tf32=True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

print(f"⏳ Loading PubMedCLIP: {MODEL_ID}")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

with torch.no_grad():
    try:
        LOGIT_SCALE = float(model.logit_scale.exp().detach().cpu())
    except Exception:
        LOGIT_SCALE = 1.0
print(f"✅ PubMedCLIP ready | logit_scale={LOGIT_SCALE:.3f} | max_len={CLIP_MAX_LEN}")

# ---------------------------
# 2) Data
# ---------------------------
def load_json_list(p: str):
    with open(p,"r",encoding="utf-8") as f:
        return json.load(f)

members_all    = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)

members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 Samples: members={len(members)} | non-members={len(nonmembers)}")

# ---------------------------
# 3) Image loader (local) + LRU cache
# ---------------------------
_IMG_CACHE = OrderedDict()

def _img_key(s: Dict[str,Any]) -> str:
    return str(s.get("local_image_path",""))

def _cache_get(k: str):
    if k in _IMG_CACHE:
        _IMG_CACHE.move_to_end(k)
        return _IMG_CACHE[k]
    return None

def _cache_put(k: str, v: Image.Image):
    _IMG_CACHE[k] = v
    _IMG_CACHE.move_to_end(k)
    if len(_IMG_CACHE) > IMG_CACHE_MAX:
        _IMG_CACHE.popitem(last=False)

def load_image_cached(s: Dict[str,Any]) -> Optional[Image.Image]:
    p = s.get("local_image_path")
    if not p or (not os.path.exists(p)):
        return None
    k = _img_key(s)
    cached = _cache_get(k)
    if cached is not None:
        return cached
    try:
        im = Image.open(p).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError, Exception):
        return None
    _cache_put(k, im)
    return im

# ---------------------------
# 4) Encode helpers (PubMedCLIP)
# ---------------------------
@torch.inference_mode()
def encode_image_one(img_pil: Image.Image) -> torch.Tensor:
    """return: [D] CPU float32 normalized"""
    inputs = processor(images=img_pil, return_tensors="pt").to(device)
    z = model.get_image_features(pixel_values=inputs["pixel_values"])
    z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
    return z.squeeze(0).float().cpu()

@torch.inference_mode()
def encode_text_batch(texts: List[str], bs: int = 256) -> torch.Tensor:
    """return: [N,D] CPU float32 normalized"""
    outs = []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i+bs]
        inputs = processor(
            text=chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=CLIP_MAX_LEN,
        ).to(device)
        z = model.get_text_features(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
        outs.append(z.float().cpu())
    return torch.cat(outs, dim=0) if outs else torch.zeros((0, model.projection_dim), dtype=torch.float32)

# ---------------------------
# 5) Encode ALL captions once (避免每次 trial 重复跑 text encoder)
#    注意：这一步和你 BiomedCLIP 脚本一致（先得到 mem_txt/non_txt/all_txt）
# ---------------------------
def valid_caption_list(samples: List[Dict[str,Any]]) -> List[str]:
    caps = []
    for s in samples:
        c = str(s.get("caption","")).strip()
        if c:
            caps.append(c)
        else:
            caps.append("")  # 保持索引对齐
    return caps

mem_caps = valid_caption_list(members)
non_caps = valid_caption_list(nonmembers)

print("\n⏳ Encoding text (members)...")
mem_txt = encode_text_batch(mem_caps, bs=256)   # [Nm, D]
print("⏳ Encoding text (nonmembers)...")
non_txt = encode_text_batch(non_caps, bs=256)   # [Nn, D]

all_txt = torch.cat([mem_txt, non_txt], dim=0)  # [Nc, D]
Nc = all_txt.size(0)
all_txt_gpu = all_txt.to(device)
print(f"✅ Text bank ready: Nc={Nc}, D={all_txt.size(1)}")

# ---------------------------
# 6) Encode image embeddings once (避免重复读取/重复 encode)
# ---------------------------
def encode_images_for_samples(samples: List[Dict[str,Any]], tag: str) -> Tuple[torch.Tensor, List[int]]:
    """
    返回:
      img_emb: [N_valid, D] CPU
      kept_idx: 原 samples 的索引（用于对齐 pos caption idx）
    """
    embs = []
    kept = []
    for i, s in enumerate(tqdm(samples, desc=f"encode-image-{tag}")):
        img = load_image_cached(s)
        cap = str(s.get("caption","")).strip()
        if img is None or (not cap):
            continue
        v = encode_image_one(img)
        embs.append(v)
        kept.append(i)
    if len(embs) == 0:
        return torch.zeros((0, all_txt.size(1)), dtype=torch.float32), []
    return torch.stack(embs, dim=0), kept

print("\n⏳ Encoding images (members)...")
mem_img, mem_kept = encode_images_for_samples(members, "members")
print("⏳ Encoding images (nonmembers)...")
non_img, non_kept = encode_images_for_samples(nonmembers, "nonmembers")

print(f"\n📊 Valid after I/O filter:")
print(f"  members:    {len(mem_kept)} / {len(members)}")
print(f"  nonmembers: {len(non_kept)} / {len(nonmembers)}")
if len(mem_kept) == 0 or len(non_kept) == 0:
    raise RuntimeError("有效样本为 0，无法运行 Min-K++。")

# ---------------------------
# 7) Hard pool builder
# ---------------------------
@torch.inference_mode()
def _build_hard_pool_for_image(img_vec_gpu: torch.Tensor, pos_idx: int, hard_top: int) -> np.ndarray:
    """
    return: top-HARD_TOP nearest text indices to the image, excluding pos_idx
    """
    sim = all_txt_gpu @ img_vec_gpu  # [Nc]
    k = min(hard_top + 1, sim.numel())
    _, idx = torch.topk(sim, k=k, largest=True)
    idx = idx.detach().cpu().numpy()
    idx = idx[idx != pos_idx]
    if idx.size > hard_top:
        idx = idx[:hard_top]
    return idx

# ---------------------------
# 8) Min-K++ score
# ---------------------------
def minKpp_score(img_vec_cpu: torch.Tensor, pos_idx: int) -> float:
    """
    Min-K++:
      - hard pool = top-HARD_TOP nearest texts to image (exclude pos)
      - per trial: sample K-1 negatives with HARD_RATIO from hard pool, rest from global
      - temperature jitters TAUS: logits_raw/tau
      - aggregate: smallest ceil(P_FRAC * (R*len(TAUS))) NLLs average
      - score = -mean_min
    """
    img_d = img_vec_cpu.to(device)
    hard_pool = _build_hard_pool_for_image(img_d, pos_idx, HARD_TOP)  # np array

    global_pool = np.delete(np.arange(Nc), pos_idx)

    nlls = []
    for _ in range(R):
        need_neg = K - 1
        n_hard = min(need_neg, int(round(need_neg * HARD_RATIO)))
        n_rand = need_neg - n_hard

        if hard_pool.size > 0 and n_hard > 0:
            hard_idx = np.random.choice(hard_pool, size=n_hard, replace=(hard_pool.size < n_hard))
        else:
            hard_idx = np.array([], dtype=int)

        remain_pool = np.setdiff1d(global_pool, hard_idx, assume_unique=False)
        if remain_pool.size > 0 and n_rand > 0:
            rand_idx = np.random.choice(remain_pool, size=n_rand, replace=(remain_pool.size < n_rand))
        else:
            rand_idx = np.array([], dtype=int)

        neg_idx = np.concatenate([hard_idx, rand_idx])
        if neg_idx.size < need_neg:
            extra = np.random.choice(global_pool, size=need_neg - neg_idx.size, replace=True)
            neg_idx = np.concatenate([neg_idx, extra])

        cand_idx = np.concatenate([[pos_idx], neg_idx])
        cand = all_txt_gpu[cand_idx]                   # [K,D]
        logits_raw = LOGIT_SCALE * (cand @ img_d)      # [K]

        for tau in TAUS:
            logits = logits_raw / float(tau)
            logp = torch.log_softmax(logits, dim=0)
            nll = float((-logp[0]).item())
            if np.isfinite(nll):
                nlls.append(nll)

    nlls = np.asarray(nlls, dtype=np.float64)
    m = max(1, int(np.ceil(P_FRAC * len(nlls))))
    best_mean = np.partition(nlls, m-1)[:m].mean()
    return float(-best_mean)

# ---------------------------
# 9) Score & eval
# ---------------------------
mem_scores, non_scores = [], []

print("\n⏳ Scoring (Min-K++ on PubMedCLIP)…")
# member 的 pos_idx 在 all_txt 里的位置：就是原 members 的 index（因为 all_txt = mem_txt + non_txt）
for t in tqdm(range(mem_img.size(0)), desc="members"):
    orig_i = mem_kept[t]           # 对应 members 的原索引
    pos_idx = orig_i               # 在 mem_txt 范围内
    mem_scores.append(minKpp_score(mem_img[t], pos_idx))

# nonmember 的 pos_idx 在 all_txt 里的位置：offset + 原 nonmembers 索引
offset = mem_txt.size(0)
for t in tqdm(range(non_img.size(0)), desc="nonmembers"):
    orig_j = non_kept[t]
    pos_idx = offset + orig_j
    non_scores.append(minKpp_score(non_img[t], pos_idx))

print(f"\n有效样本数：mem={len(mem_scores)} non={len(non_scores)}")

y = np.array([1]*len(mem_scores) + [0]*len(non_scores))
scores = np.array(mem_scores + non_scores, dtype=np.float64)

auc = roc_auc_score(y, scores)
fpr, tpr, thr = roc_curve(y, scores)
j = np.argmax(tpr - fpr)
best_thr = thr[j]
acc = accuracy_score(y, (scores >= best_thr).astype(int))
cm  = confusion_matrix(y, (scores >= best_thr).astype(int))

print("\n====== Min-K++ MIA 结果 (PubMedCLIP) ======")
print(f"AUC = {auc:.4f}")
print(f"Best threshold (Youden J) = {best_thr:.4f}")
print(f"Accuracy = {acc:.4f}")
print("Confusion matrix [[TN FP][FN TP]]:\n", cm)

# -------- @5% FPR metrics (同你脚本风格：non_scores 95% 分位作阈值) --------
thr_5 = float(np.quantile(non_scores, 0.95))
y_pred_5 = (scores >= thr_5).astype(int)

cm_5 = confusion_matrix(y, y_pred_5)
tn, fp, fn, tp = cm_5.ravel()
fpr_actual = fp / (fp + tn + 1e-12)
tpr_5 = tp / (tp + fn + 1e-12)
prec_5 = precision_score(y, y_pred_5, zero_division=0)
rec_5  = recall_score(y, y_pred_5, zero_division=0)
f1_5   = f1_score(y, y_pred_5, zero_division=0)

print("\n------ @5% FPR Metrics ------")
print(f"Threshold @FPR=5% = {thr_5:.4f}  (实际 FPR={fpr_actual*100:.2f}%)")
print(f"TPR (Recall)      = {tpr_5*100:.2f}%")
print(f"Precision          = {prec_5*100:.2f}%")
print(f"F1                 = {f1_5:.4f}")
print("Confusion @5%FPR [[TN FP][FN TP]]：")
print(cm_5)


## ModRényi

In [ ]:
# =========================================================================
# ModRényi* (fused) MIA for PubMedCLIP (Transformers CLIPModel)  ✅迁移版
#  - Data   : ROCO streaming 生成的本地 json（members_roco_train_1k.json / nonmembers_roco_test_1k.json）
#  - Model  : flaviagiammarino/pubmed-clip-vit-base-patch32  (HF transformers)
#  - Score  : fused ModRényi*（image-view + text-view 两个视图通道融合）
#  - Metric : AUC + TPR/Precision/Recall/F1 @ 5% FPR
#
# 设计对齐你给的 BiomedCLIP 脚本：
#   - K-way softmax: 1 pos + (K-1) neg
#   - IMG_VIEWS / TXT_VIEWS：多视图扰动
#   - Rényi α ∈ {0.5, 2.0}
#   - “fused” = 4个分量(-H_img^a, -H_img^b, -H_txt^a, -H_txt^b) 做 z-score 再平均
#
# 关键修复（防 confound）：
#   ✅ 图像只从 local_image_path 读，不做 tar 解包；PIL LRU cache，避免重复读/重复失败
#   ✅ HF CLIP 文本 max_position_embeddings=77：所有 tokenization 强制 truncation 到 77
#
# 你只要把 JSON 路径对上即可。
# =========================================================================

import os, json, random, math, re, warnings
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
from collections import OrderedDict

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from transformers import CLIPModel, CLIPProcessor

from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.metrics import precision_recall_fscore_support

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Paths & hyperparams
# ---------------------------
MEMBERS_JSON     = "/content/pubmedclip_roco_mia/members_roco_train_1k.json"
NONMEMBERS_JSON  = "/content/pubmedclip_roco_mia/nonmembers_roco_test_1k.json"

MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

K = 32

IMG_VIEWS = 5  # 原图 + 4 增广
TXT_VIEWS = 3  # 原文 + 2 轻度扰动

CLIP_MAX_LEN = 77         # HF CLIP hard limit
IMG_CACHE_MAX = 512       # PIL cache size

# ---------------------------
# 1) Device & model
# ---------------------------
assert torch.cuda.is_available(), "需要 CUDA GPU"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

print(f"⏳ Loading PubMedCLIP: {MODEL_ID}")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

with torch.no_grad():
    try:
        LOGIT_SCALE = float(model.logit_scale.exp().detach().cpu())
    except Exception:
        LOGIT_SCALE = 1.0
print(f"✅ PubMedCLIP ready | logit_scale={LOGIT_SCALE:.3f} | max_len={CLIP_MAX_LEN}")

# ---------------------------
# 2) Data load
# ---------------------------
def load_json_list(p: str):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

members_all    = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)

members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 Samples: members={len(members)} | non-members={len(nonmembers)}")

# ---------------------------
# 3) Local image loader + LRU cache
# ---------------------------
_IMG_CACHE = OrderedDict()

def _img_key(s: Dict[str,Any]) -> str:
    return str(s.get("local_image_path", ""))

def _cache_get(k: str):
    if k in _IMG_CACHE:
        _IMG_CACHE.move_to_end(k)
        return _IMG_CACHE[k]
    return None

def _cache_put(k: str, v: Image.Image):
    _IMG_CACHE[k] = v
    _IMG_CACHE.move_to_end(k)
    if len(_IMG_CACHE) > IMG_CACHE_MAX:
        _IMG_CACHE.popitem(last=False)

def load_image_cached(s: Dict[str,Any]) -> Optional[Image.Image]:
    p = s.get("local_image_path")
    if not p or (not os.path.exists(p)):
        return None
    k = _img_key(s)
    cached = _cache_get(k)
    if cached is not None:
        return cached
    try:
        im = Image.open(p).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError, Exception):
        return None
    _cache_put(k, im)
    return im

# ---------------------------
# 4) Collect valid pairs (keep indices for pos mapping)
# ---------------------------
def collect_valid(samples: List[Dict[str,Any]], tag: str) -> Tuple[List[Image.Image], List[str], List[int]]:
    imgs, caps, kept_idx = [], [], []
    fail = {"image": 0, "caption": 0}
    for i, s in enumerate(tqdm(samples, desc=f"collect-{tag}")):
        cap = str(s.get("caption", "")).strip()
        if not cap:
            fail["caption"] += 1
            continue
        im = load_image_cached(s)
        if im is None:
            fail["image"] += 1
            continue
        imgs.append(im)
        caps.append(cap)
        kept_idx.append(i)  # index in original samples list
    print(f"📊 {tag}: ok={len(imgs)} | fail={fail}")
    return imgs, caps, kept_idx

mem_imgs, mem_caps, mem_kept = collect_valid(members, "members")
non_imgs, non_caps, non_kept = collect_valid(nonmembers, "nonmembers")
if len(mem_imgs) == 0 or len(non_imgs) == 0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

# ---------------------------
# 5) Encode full pools once (for fast K-way sampling)
# ---------------------------
@torch.inference_mode()
def encode_text_batch(texts: List[str], bs: int = 256) -> torch.Tensor:
    outs = []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i+bs]
        inputs = processor(
            text=chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=CLIP_MAX_LEN,
        ).to(device)
        z = model.get_text_features(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
        outs.append(z.float().cpu())
    return torch.cat(outs, dim=0)

@torch.inference_mode()
def encode_image_batch(pils: List[Image.Image], bs: int = 64) -> torch.Tensor:
    outs = []
    for i in range(0, len(pils), bs):
        chunk = pils[i:i+bs]
        inputs = processor(images=chunk, return_tensors="pt").to(device)
        z = model.get_image_features(pixel_values=inputs["pixel_values"])
        z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
        outs.append(z.float().cpu())
    return torch.cat(outs, dim=0)

print("\n⏳ Encoding pools (image/text) …")
mem_img_emb = encode_image_batch(mem_imgs, bs=64)
non_img_emb = encode_image_batch(non_imgs, bs=64)
mem_txt_emb = encode_text_batch(mem_caps, bs=256)
non_txt_emb = encode_text_batch(non_caps, bs=256)

all_txt_emb = torch.cat([mem_txt_emb, non_txt_emb], dim=0)  # [Nc, D]
all_img_emb = torch.cat([mem_img_emb, non_img_emb], dim=0)  # [Ni, D]
Nc = all_txt_emb.size(0)
Ni = all_img_emb.size(0)

all_txt_gpu = all_txt_emb.to(device)
all_img_gpu = all_img_emb.to(device)

print(f"✅ Pools ready: Nc(txt)={Nc}, Ni(img)={Ni}, D={all_txt_emb.size(1)}")

# ---------------------------
# 6) Views (image/text)
# ---------------------------
from torchvision.transforms import (
    RandomResizedCrop, RandomRotation, RandomAffine, ColorJitter, InterpolationMode
)

def make_image_views(pil: Image.Image, want: int = IMG_VIEWS) -> List[Image.Image]:
    views = [pil]
    # 下面这些对 PIL 输出仍是 PIL
    views.append(RandomResizedCrop(size=(256, 256), scale=(0.8, 1.0),
                                   interpolation=InterpolationMode.BICUBIC)(pil))
    views.append(RandomRotation(degrees=30, interpolation=InterpolationMode.BICUBIC, expand=False)(pil))
    views.append(RandomAffine(degrees=20, translate=(0.08, 0.08), scale=(0.9, 1.1),
                              interpolation=InterpolationMode.BICUBIC)(pil))
    views.append(ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25, hue=0.05)(pil))
    return views[:max(1, want)]

def _normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def make_text_views(caption: str, want: int = TXT_VIEWS) -> List[str]:
    base = str(caption or "").strip()
    cand = [base, base.lower(), _normalize_spaces(base)]
    out, seen = [], set()
    for c in cand:
        if c not in seen:
            out.append(c); seen.add(c)
    return out[:max(1, want)]

# ---------------------------
# 7) Encode one view (fast path)
# ---------------------------
@torch.inference_mode()
def _encode_one_image_view(pil: Image.Image) -> torch.Tensor:
    inp = processor(images=pil, return_tensors="pt").to(device)
    with torch.cuda.amp.autocast():
        z = model.get_image_features(pixel_values=inp["pixel_values"])
        z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
    return z[0].float()  # [D] on GPU

@torch.inference_mode()
def _encode_one_text_view(text: str) -> torch.Tensor:
    inp = processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CLIP_MAX_LEN,
    ).to(device)
    with torch.cuda.amp.autocast():
        z = model.get_text_features(input_ids=inp["input_ids"], attention_mask=inp["attention_mask"])
        z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
    return z[0].float()  # [D] on GPU

# ---------------------------
# 8) K-way logP (image-view / text-view)
# ---------------------------
def image_view_logp(img_vec_gpu: torch.Tensor, pos_txt_idx: int, K: int) -> np.ndarray:
    """
    img_vec_gpu: [D] on GPU
    return: [1, K] logP (np.float32)
    """
    neg_pool = np.delete(np.arange(Nc), pos_txt_idx)
    replace = (neg_pool.size < (K-1))
    neg_idx = np.random.choice(neg_pool, size=K-1, replace=replace)
    cand_idx = np.concatenate([[pos_txt_idx], neg_idx])
    cand_txt = all_txt_gpu[cand_idx]                # [K, D]
    logits   = LOGIT_SCALE * (cand_txt @ img_vec_gpu)  # [K]
    logp     = torch.log_softmax(logits, dim=0).unsqueeze(0)
    return logp.detach().cpu().numpy().astype(np.float32)

def text_view_logp(txt_vec_gpu: torch.Tensor, pos_img_idx: int, K: int) -> np.ndarray:
    """
    txt_vec_gpu: [D] on GPU
    return: [1, K] logP (np.float32)
    """
    neg_pool = np.delete(np.arange(Ni), pos_img_idx)
    replace = (neg_pool.size < (K-1))
    neg_idx = np.random.choice(neg_pool, size=K-1, replace=replace)
    cand_idx = np.concatenate([[pos_img_idx], neg_idx])
    cand_img = all_img_gpu[cand_idx]                # [K, D]
    logits   = LOGIT_SCALE * (cand_img @ txt_vec_gpu)  # [K]
    logp     = torch.log_softmax(logits, dim=0).unsqueeze(0)
    return logp.detach().cpu().numpy().astype(np.float32)

# ---------------------------
# 9) Rényi entropy + fusion
# ---------------------------
def renyi_entropy_from_logp(logp_vec: np.ndarray, alpha: float) -> float:
    v = logp_vec.reshape(-1)
    a = alpha * v
    m = np.max(a)
    # H_alpha = (1/(1-a)) * log sum p^a ；logp -> log sum exp(alpha*logp)
    return float((1.0 / (1.0 - alpha)) * (m + np.log(np.exp(a - m).sum())))

def renyi_seq_from_logp(logp_seq: np.ndarray, alpha: float) -> float:
    # logp_seq: [T,K]；此处 T=1，但保持接口一致
    T = logp_seq.shape[0]
    vals = [renyi_entropy_from_logp(logp_seq[t], alpha) for t in range(T)]
    return float(np.mean(vals)) if vals else float("nan")

def robust_aggregate(values: List[float], trim: float = 0.1) -> float:
    arr = np.array([v for v in values if np.isfinite(v)], dtype=float)
    if arr.size == 0:
        return float("nan")
    arr.sort()
    k = int(math.floor(trim * arr.size))
    if k * 2 < arr.size:
        arr = arr[k: arr.size - k]
    return float(np.mean(arr))

def fused_modrenyi_score(img_logps: List[np.ndarray],
                         txt_logps: List[np.ndarray],
                         alphas=(0.5, 2.0)) -> float:
    """
    4 个分量：
      -img: alpha=0.5,2.0 -> score=-H
      -txt: alpha=0.5,2.0 -> score=-H
    融合：valid 分量做 z-score 再平均；不足则直接均值
    """
    parts = []
    for logp_views in [img_logps, txt_logps]:
        if len(logp_views) == 0:
            parts.extend([float("nan")] * len(alphas))
            continue
        for a in alphas:
            per_view = []
            for lp in logp_views:
                if lp is None or (not np.isfinite(lp).all()):
                    continue
                H = renyi_seq_from_logp(lp, a)
                per_view.append(H)
            if not per_view:
                parts.append(float("nan"))
                continue
            H_bar = robust_aggregate(per_view, trim=0.1)
            parts.append(float(-H_bar))

    valid = np.array([x for x in parts if np.isfinite(x)], dtype=float)
    if valid.size == 0:
        return float("nan")
    if valid.size >= 2 and np.std(valid) > 1e-12:
        z = (valid - np.mean(valid)) / (np.std(valid) + 1e-12)
        return float(np.mean(z))
    return float(np.mean(valid))

# ---------------------------
# 10) Per-sample fused score
# ---------------------------
def fused_score_member(i: int) -> Optional[float]:
    """
    对于 members 第 i 个（在 mem_* 列表里）：
      pos_txt_idx = i
      pos_img_idx = i
    """
    img = mem_imgs[i]
    cap = mem_caps[i]

    img_logps, txt_logps = [], []

    # image views -> logP over candidate texts
    try:
        pos_txt_idx = i
        for v in make_image_views(img, IMG_VIEWS):
            v_z = _encode_one_image_view(v)
            img_logps.append(image_view_logp(v_z, pos_txt_idx, K))
    except Exception:
        pass

    # text views -> logP over candidate images
    try:
        pos_img_idx = i
        for t in make_text_views(cap, TXT_VIEWS):
            t_z = _encode_one_text_view(t)
            txt_logps.append(text_view_logp(t_z, pos_img_idx, K))
    except Exception:
        pass

    if len(img_logps) == 0 and len(txt_logps) == 0:
        return None
    s = fused_modrenyi_score(img_logps, txt_logps, alphas=(0.5, 2.0))
    return s if (s is not None and math.isfinite(s)) else None

def fused_score_nonmember(j: int) -> Optional[float]:
    """
    对于 nonmembers 第 j 个（在 non_* 列表里）：
      pos_txt_idx = len(mem_txt_emb) + j
      pos_img_idx = len(mem_img_emb) + j
    """
    img = non_imgs[j]
    cap = non_caps[j]

    img_logps, txt_logps = [], []

    try:
        pos_txt_idx = mem_txt_emb.size(0) + j
        for v in make_image_views(img, IMG_VIEWS):
            v_z = _encode_one_image_view(v)
            img_logps.append(image_view_logp(v_z, pos_txt_idx, K))
    except Exception:
        pass

    try:
        pos_img_idx = mem_img_emb.size(0) + j
        for t in make_text_views(cap, TXT_VIEWS):
            t_z = _encode_one_text_view(t)
            txt_logps.append(text_view_logp(t_z, pos_img_idx, K))
    except Exception:
        pass

    if len(img_logps) == 0 and len(txt_logps) == 0:
        return None
    s = fused_modrenyi_score(img_logps, txt_logps, alphas=(0.5, 2.0))
    return s if (s is not None and math.isfinite(s)) else None

# ---------------------------
# 11) Metrics @ 5% FPR (interp threshold)
# ---------------------------
def metrics_at_target_fpr(y_true: np.ndarray, scores: np.ndarray, target_fpr: float = 0.05):
    fpr, tpr, thr = roc_curve(y_true, scores)
    idx = np.searchsorted(fpr, target_fpr, side="right")

    if idx == 0:
        thr_star = thr[0]; tpr_star = tpr[0]
    elif idx >= len(thr):
        thr_star = thr[-1]; tpr_star = tpr[-1]
    else:
        x0, x1 = fpr[idx-1], fpr[idx]
        y0, y1 = tpr[idx-1], tpr[idx]
        t0, t1 = thr[idx-1], thr[idx]
        w = (target_fpr - x0) / (x1 - x0 + 1e-12)
        tpr_star = y0 + w * (y1 - y0)
        thr_star = t0 + w * (t1 - t0)

    y_pred = (scores >= thr_star).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr_actual = fp / (fp + tn + 1e-12)

    return {
        "ROC-AUC": float(roc_auc_score(y_true, scores)),
        "thr@5%FPR": float(thr_star),
        "TPR@5%FPR": float(tpr_star),
        "Precision": float(prec),
        "Recall": float(rec),
        "F1": float(f1),
        "FPR(actual)": float(fpr_actual),
        "CM@5%FPR": cm,
    }

# ---------------------------
# 12) Main: score & evaluate
# ---------------------------
print("\n⏳ Scoring ModRényi* (fused) on PubMedCLIP …")
mem_scores, non_scores = [], []

for i in tqdm(range(len(mem_imgs)), desc="members"):
    s = fused_score_member(i)
    if s is not None:
        mem_scores.append(s)

for j in tqdm(range(len(non_imgs)), desc="nonmembers"):
    s = fused_score_nonmember(j)
    if s is not None:
        non_scores.append(s)

print(f"\n📊 Valid scores: members={len(mem_scores)}/{len(mem_imgs)} | nonmembers={len(non_scores)}/{len(non_imgs)}")
if len(mem_scores) == 0 or len(non_scores) == 0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

y_true = np.array([1]*len(mem_scores) + [0]*len(non_scores))
scores = np.array(mem_scores + non_scores, dtype=np.float64)

m = metrics_at_target_fpr(y_true, scores, target_fpr=0.05)

print("\n====== ModRényi* (fused) MIA 结果（PubMedCLIP） ======")
print(f"AUC                 = {m['ROC-AUC']:.4f}")
print(f"TPR  @5% FPR        = {m['TPR@5%FPR']:.4f}")
print(f"Precision @5% FPR   = {m['Precision']:.4f}")
print(f"Recall    @5% FPR   = {m['Recall']:.4f}")
print(f"F1        @5% FPR   = {m['F1']:.4f}")
print(f"thr       @5% FPR   = {m['thr@5%FPR']:.4f}")
print(f"FPR(actual)         = {m['FPR(actual)']*100:.2f}%")
print("Confusion @5%FPR [[TN FP][FN TP]]：")
print(m["CM@5%FPR"])


## Similarity

In [ ]:
# ===============================================
# Similarity-MIA (Image + Text) with Fusion — robust (PubMedCLIP)
#  - Data   : members_pubmedclip_*.json / nonmembers_pubmedclip_*.json (LOCAL images)
#  - Model  : PubMedCLIP via HuggingFace transformers (CLIPModel)
#  - Score  : self-similarity under light perturbations (image & text), z-score-fused
#  - Metric : AUC + TPR / Precision / Recall / F1 @ 5% FPR
#
# ✅ 你要的版本特点（对照你 BiomedCLIP 脚本）：
# - 保持“同一套路”：图像轻扰动相似度 + 文本轻扰动相似度 -> nonmember z-score -> 按权重融合
# - 阈值：nonmember fused 分数 95% 分位（目标 ~5% FPR）
# - 输出：AUC + @5%FPR 的 TPR/Precision/Recall/F1 + Confusion matrix
# - 保存 CSV 明细（member/nonmember 的通道分数、z分数、fused）
#
# ✅ PubMedCLIP 关键差异：
# - 不用 open_clip；改用 transformers 的 CLIPModel + CLIPProcessor
# - 文本 max_length=77（CLIP hard limit），必须 truncation=True
# - 数据读取：默认认为 members/nonmembers 都是本地图片路径 local_image_path
#
# ✅ 数据格式假设（你 PubMedCLIP 的数据）：
# - member obj:    {"local_image_path": "...", "caption": "..."}   (或你可改 KEY)
# - nonmember obj: {"local_image_path": "...", "caption": "..."}
# ===============================================

import os, io, json, math, random, warnings, re, csv
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List
from collections import OrderedDict

import numpy as np
import torch
from PIL import Image, ImageFilter, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)

from transformers import CLIPModel, CLIPProcessor

warnings.filterwarnings("ignore")

# ---------------- 基本参数 ----------------
# ✅ 按你 PubMedCLIP 数据实际路径改这里
DATA_BASE = "/content/pubmedclip_roco_mia"
MEMBERS_JSON     = f"{DATA_BASE}/members_roco_train_1k.json"
NONMEMBERS_JSON  = f"{DATA_BASE}/nonmembers_roco_test_1k.json"

NUM_MEMBERS, NUM_NONMEMBERS = 1000, 1000
SEED = 42

OUT_DIR  = "/content/sim_mia_outputs_pubmedclip"
OUT_CSV  = os.path.join(OUT_DIR, "mia_similarity_fused_1000x1000_pubmedclip.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# 图像扰动
GAUSS_RADIUS = 1.0     # 轻度高斯模糊

# 文本扰动（轻语义保持）
TXT_VIEWS = 2          # 两种视图：原文 vs (lower / 空白规整)
CLIP_MAX_LEN = 77      # ✅ transformers-CLIP hard limit

# 融合权重（单通道时自动视为 1.0）
W_IMG, W_TEXT = 0.60, 0.40

# 图像 cache（避免重复读取）
IMG_CACHE_MAX = 4096

# ---------------- 设备 & 模型（PubMedCLIP via transformers） ----------------
assert torch.cuda.is_available(), "需要 GPU"
device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32  = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Using {device}")

MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"  # ✅ 如你用别的 PubMedCLIP repo，在此替换
print(f"⏳ 加载 PubMedCLIP：{MODEL_ID}")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

with torch.no_grad():
    # HF CLIP: logit_scale is a parameter (stored in log space)
    LOGIT_SCALE = float(model.logit_scale.exp().item()) if hasattr(model, "logit_scale") else 1.0
print(f"✅ PubMedCLIP 就绪 | logit_scale={LOGIT_SCALE:.3f}")

# ---------------- 数据加载 ----------------
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all    = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)
members    = members_all[:NUM_MEMBERS]
nonmembers = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 样本就绪：members={len(members)} | non-members={len(nonmembers)}")

# ---------------- 本地 I/O + cache ----------------
_IMG_CACHE = OrderedDict()

def _cache_get(k: str):
    if k in _IMG_CACHE:
        _IMG_CACHE.move_to_end(k)
        return _IMG_CACHE[k]
    return None

def _cache_put(k: str, v: Image.Image):
    _IMG_CACHE[k] = v
    _IMG_CACHE.move_to_end(k)
    if len(_IMG_CACHE) > IMG_CACHE_MAX:
        _IMG_CACHE.popitem(last=False)

# ✅ 如果你的 key 不是 local_image_path，就改这里
IMG_KEY = "local_image_path"

def load_local_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get(IMG_KEY)
    if not p or (not os.path.exists(p)):
        return None

    cached = _cache_get(p)
    if cached is not None:
        return cached

    try:
        im = Image.open(p).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError, Exception):
        return None

    _cache_put(p, im)
    return im

def collect_pairs(samples: List[Dict[str,Any]], kind="member"):
    imgs, caps = [], []
    fail = {"image":0, "other":0}
    for s in tqdm(samples, desc=f"load-{kind}"):
        cap = str(s.get("caption","")).strip()
        if not cap:
            fail["other"] += 1
            continue
        im = load_local_image(s)
        if im is None:
            fail["image"] += 1
            continue
        imgs.append(im)
        caps.append(cap)
    print(f"📊 {kind}: ok={len(imgs)} | fail={fail}")
    return imgs, caps

print("\n⏳ 读取 members（本地）")
mem_imgs, mem_caps = collect_pairs(members, kind="member")
print("⏳ 读取 non-members（本地）")
non_imgs, non_caps = collect_pairs(nonmembers, kind="nonmember")

if len(mem_imgs)==0 or len(non_imgs)==0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

# ---------------- 扰动与编码工具 ----------------
def corrupt_image(img: Image.Image) -> Image.Image:
    # 与你原脚本一致：轻度高斯模糊（不做 JPEG 重压缩，避免额外 codec 变量）
    return img.filter(ImageFilter.GaussianBlur(radius=GAUSS_RADIUS))

def _normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def make_text_views(caption: str, want: int = TXT_VIEWS) -> List[str]:
    base = str(caption or "").strip()
    cand = [base, base.lower(), _normalize_spaces(base)]
    out, seen = [], set()
    for c in cand:
        if c not in seen:
            out.append(c); seen.add(c)
    # 至少两种视图以形成配对
    if len(out) < 2:
        out = [base, base]
    return out[:max(2, want)]

def pick_two_views(caps: List[str]) -> Tuple[List[str], List[str]]:
    v0, v1 = [], []
    for c in caps:
        views = make_text_views(c, want=2)
        v0.append(views[0])
        v1.append(views[1])
    return v0, v1

@torch.inference_mode()
def encode_images(pils: List[Image.Image], batch_size: int = 64) -> torch.Tensor:
    feats = []
    for i in range(0, len(pils), batch_size):
        chunk = pils[i:i+batch_size]
        inputs = processor(images=chunk, return_tensors="pt").to(device)
        with torch.cuda.amp.autocast():
            z = model.get_image_features(pixel_values=inputs["pixel_values"])
            z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
        feats.append(z.float().cpu())
    return torch.cat(feats, dim=0) if feats else torch.zeros((0, 512), dtype=torch.float32)

@torch.inference_mode()
def encode_texts(texts: List[str], batch_size: int = 128) -> torch.Tensor:
    feats = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        inputs = processor(
            text=chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=CLIP_MAX_LEN
        ).to(device)
        with torch.cuda.amp.autocast():
            z = model.get_text_features(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"]
            )
            z = z / (z.norm(dim=-1, keepdim=True) + 1e-12)
        feats.append(z.float().cpu())
    return torch.cat(feats, dim=0) if feats else torch.zeros((0, 512), dtype=torch.float32)

def pair_cos_sims(z1: torch.Tensor, z2: torch.Tensor) -> np.ndarray:
    # 每行一对 (i,i) 的余弦，相当于逐元素点积（已归一化）
    sims = (z1 * z2).sum(dim=-1).numpy()
    return sims.astype(np.float32)

# ---------------- 计算图像通道分数 ----------------
print("\n⏳ 构造图像扰动并编码 …")
all_imgs = mem_imgs + non_imgs
all_imgs_corrupt = [corrupt_image(im) for im in tqdm(all_imgs, desc="corrupt-images")]

z_img_orig    = encode_images(all_imgs, batch_size=64)
z_img_corrupt = encode_images(all_imgs_corrupt, batch_size=64)

mem_img_sim = pair_cos_sims(z_img_orig[:len(mem_imgs)], z_img_corrupt[:len(mem_imgs)])
non_img_sim = pair_cos_sims(z_img_orig[len(mem_imgs):], z_img_corrupt[len(mem_imgs):])

# ---------------- 计算文本通道分数 ----------------
print("⏳ 构造文本扰动并编码 …")
mem_v0, mem_v1 = pick_two_views(mem_caps)
non_v0, non_v1 = pick_two_views(non_caps)

z_txt_mem_v0 = encode_texts(mem_v0, batch_size=256)
z_txt_mem_v1 = encode_texts(mem_v1, batch_size=256)
z_txt_non_v0 = encode_texts(non_v0, batch_size=256)
z_txt_non_v1 = encode_texts(non_v1, batch_size=256)

mem_txt_sim = pair_cos_sims(z_txt_mem_v0, z_txt_mem_v1)
non_txt_sim = pair_cos_sims(z_txt_non_v0, z_txt_non_v1)

# ---------------- 融合与指标（容忍单通道） ----------------
def zscore_by_nonmember(non_arr: np.ndarray):
    mu, sd = float(non_arr.mean()), float(non_arr.std() + 1e-8)
    return mu, sd

have_img = (len(mem_img_sim) > 10 and len(non_img_sim) > 10)
have_txt = (len(mem_txt_sim) > 10 and len(non_txt_sim) > 10)
if not have_img and not have_txt:
    raise RuntimeError("两条通道都没有有效分数（请检查数据读入或编码阶段）")

parts = []
if have_img: parts.append(("img", mem_img_sim, non_img_sim, W_IMG))
if have_txt: parts.append(("txt", mem_txt_sim, non_txt_sim, W_TEXT))
if len(parts) == 1:
    # 单通道时将权重设为 1
    name, ma, na, _w = parts[0]
    parts = [(name, ma, na, 1.0)]

fused_mem, fused_non = None, None
details = {}

for name, mem_arr, non_arr, w in parts:
    mu, sd = zscore_by_nonmember(non_arr)
    non_z = (non_arr - mu) / (sd + 1e-8)
    mem_z = (mem_arr - mu) / (sd + 1e-8)

    details[name] = {
        "mem_raw": mem_arr, "non_raw": non_arr,
        "mem_z": mem_z, "non_z": non_z,
        "w": w, "mu": mu, "sd": sd
    }

    if fused_mem is None:
        fused_mem = w * mem_z
        fused_non = w * non_z
    else:
        fused_mem = fused_mem + w * mem_z
        fused_non = fused_non + w * non_z

# 评估
y_true = np.array([1]*len(fused_mem) + [0]*len(fused_non))
scores = np.concatenate([fused_mem, fused_non], axis=0)
auc = roc_auc_score(y_true, scores)

# 阈值：nonmember 95 分位（目标 ~5% FPR）
thr_5 = float(np.quantile(fused_non, 0.95))
y_pred = (scores >= thr_5).astype(int)

cm  = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn + 1e-12)
tpr = tp / (tp + fn + 1e-12)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
acc = accuracy_score(y_true, y_pred)

print("\n====== Similarity-MIA（Image+Text 融合，稳健版，PubMedCLIP） ======")
print(f"AUC                = {auc:.4f}")
print(f"Threshold @FPR=5%  = {thr_5:.4f}  (实际 FPR={fpr*100:.2f}%)")
print(f"TPR (Recall)       = {tpr*100:.2f}%")
print(f"Precision          = {prec*100:.2f}%")
print(f"F1                 = {f1:.4f}")
print(f"Accuracy           = {acc*100:.2f}%")
print("Confusion [[TN FP][FN TP]]：")
print(cm)

# ---------------- 保存明细 CSV ----------------
os.makedirs(OUT_DIR, exist_ok=True)
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    hdr = ["split"]
    if have_img: hdr += ["s_img_raw", "s_img_z(non-calib)"]
    if have_txt: hdr += ["s_txt_raw", "s_txt_z(non-calib)"]
    hdr += ["s_fused"]
    w.writerow(hdr)

    def write_rows(mem: bool):
        n = len(fused_mem) if mem else len(fused_non)
        for i in range(n):
            row = ["member" if mem else "nonmember"]
            if have_img:
                d = details["img"]
                row += [
                    float(d["mem_raw"][i]) if mem else float(d["non_raw"][i]),
                    float(d["mem_z"][i])   if mem else float(d["non_z"][i]),
                ]
            if have_txt:
                d = details["txt"]
                row += [
                    float(d["mem_raw"][i]) if mem else float(d["non_raw"][i]),
                    float(d["mem_z"][i])   if mem else float(d["non_z"][i]),
                ]
            row += [float(fused_mem[i]) if mem else float(fused_non[i])]
            w.writerow(row)

    write_rows(mem=True)
    write_rows(mem=False)

print(f"\n💾 明细已保存：{OUT_CSV}")


## zlib

In [ ]:
# ============================================================
# Zlib-Calibrated Contrastive MIA for PubMedCLIP (HF transformers)
# - Model : PubMedCLIP (CLIP-style contrastive VLM)
# - Score : Cosine_Similarity * Zlib_Compression_Ratio
# - Eval  : AUC + TPR@5%FPR + Calibration Stats
#
# ✅ 改动点（相对你 BiomedCLIP 版）：
#  1) open_clip -> transformers: CLIPModel + CLIPProcessor
#  2) 去掉 PMC tar 解包逻辑：PubMedCLIP/ROCO 数据默认 local_image_path
#  3) 强制文本 truncation 到 CLIP_MAX_LEN=77（避免 77 限制报错）
#  4) 加 PIL 图像 LRU cache，减少重复 IO/失败噪声
# ============================================================

import os, json, random, zlib, warnings
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
from collections import OrderedDict

import numpy as np
import torch
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve

from transformers import CLIPModel, CLIPProcessor

warnings.filterwarnings("ignore")

# ---------------------------
# 0) 配置与路径
# ---------------------------
# ✅ 按你 PubMedCLIP/ROCO 的本地 json 修改这里
DATA_BASE     = "/content/pubmedclip_roco_mia"
MEMBERS_JSON  = f"{DATA_BASE}/members_roco_train_1k.json"
NONMEM_JSON   = f"{DATA_BASE}/nonmembers_roco_test_1k.json"

NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000
SEED = 42

# PubMedCLIP HF 模型（你也可替换成你实际在用的 pubmedclip HF repo）
MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"

# HF CLIP text hard limit
CLIP_MAX_LEN = 77

# PIL cache
IMG_CACHE_MAX = 512

# ---------------------------
# 1) 模型加载 (PubMedCLIP)
# ---------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")

print(f"⏳ Loading model: {MODEL_ID}")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

with torch.no_grad():
    try:
        LOGIT_SCALE = float(model.logit_scale.exp().detach().cpu())
    except Exception:
        LOGIT_SCALE = 1.0
print(f"✅ PubMedCLIP ready | logit_scale={LOGIT_SCALE:.3f} | max_len={CLIP_MAX_LEN}")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ---------------------------
# 2) 核心算法：Zlib 复杂度校准
# ---------------------------
def get_zlib_ratio(text: str) -> float:
    """压缩率 = compressed_bytes / original_bytes；越高越“复杂/信息密度高”"""
    if not text or len(text.strip()) == 0:
        return 1.0
    b = text.encode("utf-8")
    c = zlib.compress(b, level=9)
    return len(c) / max(1, len(b))

# ---------------------------
# 3) 图像加载逻辑 (local path + LRU cache)
# ---------------------------
_IMG_CACHE = OrderedDict()

def _cache_get(k: str):
    if k in _IMG_CACHE:
        _IMG_CACHE.move_to_end(k)
        return _IMG_CACHE[k]
    return None

def _cache_put(k: str, v: Image.Image):
    _IMG_CACHE[k] = v
    _IMG_CACHE.move_to_end(k)
    if len(_IMG_CACHE) > IMG_CACHE_MAX:
        _IMG_CACHE.popitem(last=False)

def load_local_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if not p or (not os.path.exists(p)):
        return None

    cached = _cache_get(p)
    if cached is not None:
        return cached

    try:
        im = Image.open(p).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError, Exception):
        return None

    _cache_put(p, im)
    return im

# ---------------------------
# 4) 计算分数（cos & cos*zlib）
# ---------------------------
@torch.inference_mode()
def compute_scores(images: List[Image.Image], captions: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    scores_raw = []   # 纯 cosine
    scores_zlb = []   # cosine * zlib_ratio

    for img_pil, cap in tqdm(list(zip(images, captions)), total=len(images), desc="Inference"):
        cap = str(cap or "").strip()
        if not cap:
            # 与 collect 阶段对齐：理论上不会出现
            scores_raw.append(0.0)
            scores_zlb.append(0.0)
            continue

        # image embedding
        inp_img = processor(images=img_pil, return_tensors="pt").to(device)
        img_emb = model.get_image_features(pixel_values=inp_img["pixel_values"])
        img_emb = img_emb / (img_emb.norm(dim=-1, keepdim=True) + 1e-12)

        # text embedding (✅ truncation to 77)
        inp_txt = processor(
            text=[cap],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=CLIP_MAX_LEN,
        ).to(device)
        txt_emb = model.get_text_features(
            input_ids=inp_txt["input_ids"],
            attention_mask=inp_txt["attention_mask"],
        )
        txt_emb = txt_emb / (txt_emb.norm(dim=-1, keepdim=True) + 1e-12)

        cos_sim = float((img_emb * txt_emb).sum().item())
        z_ratio = float(get_zlib_ratio(cap))

        scores_raw.append(cos_sim)
        scores_zlb.append(cos_sim * z_ratio)

    return np.asarray(scores_raw, dtype=np.float64), np.asarray(scores_zlb, dtype=np.float64)

# ---------------------------
# 5) 主程序
# ---------------------------
def main():
    with open(MEMBERS_JSON, "r", encoding="utf-8") as f:
        m_data = json.load(f)[:NUM_MEMBERS]
    with open(NONMEM_JSON, "r", encoding="utf-8") as f:
        n_data = json.load(f)[:NUM_NONMEMBERS]

    def collect(data: List[Dict[str,Any]], label: str):
        imgs, caps = [], []
        fail = {"image": 0, "caption": 0}
        for s in tqdm(data, desc=f"Loading {label}"):
            cp = str(s.get("caption", "")).strip()
            if not cp:
                fail["caption"] += 1
                continue
            im = load_local_image(s)
            if im is None:
                fail["image"] += 1
                continue
            imgs.append(im)
            caps.append(cp)
        print(f"📊 {label}: ok={len(imgs)} | fail={fail}")
        return imgs, caps

    m_imgs, m_caps = collect(m_data, "Members")
    n_imgs, n_caps = collect(n_data, "Non-members")

    print(f"\n✅ Loaded: Member={len(m_imgs)}, Non-member={len(n_imgs)}")
    if len(m_imgs) == 0 or len(n_imgs) == 0:
        raise RuntimeError("有效样本为 0，无法评估。请检查 local_image_path 是否存在。")

    # compute
    m_raw, m_zlb = compute_scores(m_imgs, m_caps)
    n_raw, n_zlb = compute_scores(n_imgs, n_caps)

    def tpr_at_fpr5(y_true: np.ndarray, y_score: np.ndarray) -> float:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        idxs = np.where(fpr <= 0.05)[0]
        if idxs.size == 0:
            return float(tpr[0])
        return float(tpr[idxs[-1]])

    def evaluate(m_s: np.ndarray, n_s: np.ndarray, title: str):
        y_true  = np.array([1]*len(m_s) + [0]*len(n_s), dtype=np.int32)
        y_score = np.concatenate([m_s, n_s]).astype(np.float64)

        auc = float(roc_auc_score(y_true, y_score))
        tpr5 = tpr_at_fpr5(y_true, y_score)

        print(f"\n--- {title} ---")
        print(f"AUC:         {auc:.4f}")
        print(f"TPR @ 5%FPR: {tpr5*100:.2f}%")
        print(f"Mean (M):    {float(np.mean(m_s)):.6f}")
        print(f"Mean (N):    {float(np.mean(n_s)):.6f}")
        print(f"Std  (M):    {float(np.std(m_s)):.6f}")
        print(f"Std  (N):    {float(np.std(n_s)):.6f}")

    evaluate(m_raw, n_raw, "Standard Cosine Similarity (Baseline)")
    evaluate(m_zlb, n_zlb, "Zlib-Calibrated Similarity (MIA Attack)")

    # save
    y_true  = np.array([1]*len(m_zlb) + [0]*len(n_zlb), dtype=np.int32)
    y_score = np.concatenate([m_zlb, n_zlb]).astype(np.float64)
    auc_zlb = float(roc_auc_score(y_true, y_score))
    fpr, tpr, thr = roc_curve(y_true, y_score)

    res = {
        "model_id": MODEL_ID,
        "members_json": MEMBERS_JSON,
        "nonmembers_json": NONMEM_JSON,
        "num_members_scored": int(len(m_zlb)),
        "num_nonmembers_scored": int(len(n_zlb)),
        "auc_zlib": auc_zlb,
        "tpr_at_5fpr_zlib": float(tpr[np.where(fpr <= 0.05)[0][-1]]) if np.any(fpr <= 0.05) else float(tpr[0]),
        "mean_member_zlib": float(np.mean(m_zlb)),
        "mean_nonmember_zlib": float(np.mean(n_zlb)),
    }

    save_path = "/content/pubmedclip_zlib_mia_results.json"
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(res, f, ensure_ascii=False, indent=2)
    print(f"\n💾 Saved: {save_path}")

if __name__ == "__main__":
    main()


## GradNorm

In [ ]:
# ============================================================
# GradNorm MIA for PubMedCLIP (HF transformers) — Member vs Non-member (M/N)
# (AUC only, NO CALIB split)
#
# ✅ 目标：
# - 不使用 P/S/N
# - 不使用 CALIB
# - 只输出 AUC(M vs N)
#
# ✅ GradNorm 定义（参考 mimir.attacks.gradnorm 思路）：
# - 对每个样本构造一个 CLIP InfoNCE batch（target + K-1 negatives）
# - 反向传播得到 ∇θ loss
# - 计算选定参数集合的梯度 p-norm，并对参数取均值
# - score = - grad_norm  (梯度越小 -> score 越大 -> 越像 member)
#
# ✅ PubMedCLIP 数据格式假设（本地图片）：
# - member obj:    {"local_image_path":..., "caption":...}
# - nonmember obj: {"local_image_path":..., "caption":...}
#
# ✅ 关键修复：
# - 强制 max_length=77 + truncation=True（CLIP text 上限 77，避免报错）
# - PIL 图像 LRU cache，避免重复读图导致 batch 实际不一致
# - neg_pool 使用 union_pool（M+N 混合）以减少采样偏置
# ============================================================

import os, io, json, random, warnings, re, gc
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
from collections import OrderedDict

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_auc_score

from transformers import CLIPModel, CLIPProcessor

warnings.filterwarnings("ignore")

# ============================================================
# 0) Paths & Hyperparams
# ============================================================
# ✅ 按你 PubMedCLIP 数据实际路径改这里
DATA_BASE     = "/content/pubmedclip_roco_mia"
MEMBERS_JSON  = f"{DATA_BASE}/members_roco_train_1k.json"
NONMEMBERS_JSON = f"{DATA_BASE}/nonmembers_roco_test_1k.json"

SEED = 42
TOTAL_M = 1000
TOTAL_N = 1000

# CLIP contrastive batch size (target + K-1 negatives)
BATCH_K = 8

# GradNorm hyperparams
PNORM = 2                 # 1 / 2 / np.inf
REDUCE = "mean"           # "mean" or "sum" across params
GRAD_CLIP = None          # e.g. 1.0 or None

# Selective grads: Vision last-N + Text last-M + Projection
VISION_LAST_N = 3
TEXT_LAST_M = 3
USE_SELECTIVE_GRADS = True

# PubMedCLIP HF model
MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"

# CLIP tokenizer max length (hard limit)
CLIP_MAX_LEN = 77

# PIL cache
IMG_CACHE_MAX = 512

print("\n📋 Hyperparams:")
print(f"  TOTAL M/N     = {TOTAL_M}/{TOTAL_N}")
print(f"  BATCH_K       = {BATCH_K}")
print(f"  PNORM         = {PNORM}")
print(f"  REDUCE        = {REDUCE}")
print(f"  Vision lastN  = {VISION_LAST_N}")
print(f"  Text lastM    = {TEXT_LAST_M}")
print(f"  selective     = {USE_SELECTIVE_GRADS}")
print(f"  MODEL_ID      = {MODEL_ID}")
print(f"  CLIP_MAX_LEN  = {CLIP_MAX_LEN}")

# ============================================================
# 1) Device & perf
# ============================================================
assert torch.cuda.is_available(), "❌ CUDA not available"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"✅ Device: {device}")

# ============================================================
# 2) Load PubMedCLIP (transformers)
# ============================================================
print(f"⏳ Loading model: {MODEL_ID}")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()
print("✅ PubMedCLIP loaded.")

# ============================================================
# 3) Load data
# ============================================================
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)

rnd = random.Random(SEED)
rnd.shuffle(members_all)
rnd.shuffle(nonmembers_all)

members_all = members_all[:TOTAL_M]
nonmembers_all = nonmembers_all[:TOTAL_N]

print(f"\n📦 Data loaded:")
print(f"   Members:     {len(members_all)}")
print(f"   Non-members: {len(nonmembers_all)}")

if min(len(members_all), len(nonmembers_all)) == 0:
    raise RuntimeError("❌ members or nonmembers is empty")

# Union pool for negatives (same construction for both M and N to avoid bias)
union_pool = list(members_all) + list(nonmembers_all)
rnd2 = random.Random(SEED + 123)
rnd2.shuffle(union_pool)

# ============================================================
# 4) Image loader (local path) + LRU cache
# ============================================================
_IMG_CACHE = OrderedDict()

def _cache_get(k: str):
    if k in _IMG_CACHE:
        _IMG_CACHE.move_to_end(k)
        return _IMG_CACHE[k]
    return None

def _cache_put(k: str, v: Image.Image):
    _IMG_CACHE[k] = v
    _IMG_CACHE.move_to_end(k)
    if len(_IMG_CACHE) > IMG_CACHE_MAX:
        _IMG_CACHE.popitem(last=False)

def load_local_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path")
    if (not p) or (not os.path.exists(p)):
        return None

    cached = _cache_get(p)
    if cached is not None:
        return cached

    try:
        im = Image.open(p).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError, Exception):
        return None

    _cache_put(p, im)
    return im

# ============================================================
# 5) Selective gradients (transformers CLIP)
# ============================================================
def enable_grads_selectively_hf_clip(m: CLIPModel) -> List[str]:
    """
    Select: vision encoder last-N layers + text encoder last-M layers + projections.
    Works for CLIPModel in transformers (vision_model.encoder.layers / text_model.encoder.layers).
    """
    selected = []

    # freeze all
    for p in m.parameters():
        p.requires_grad_(False)

    # projections (always include)
    for name, p in m.named_parameters():
        if any(x in name for x in [
            "visual_projection", "text_projection",  # HF CLIP
            "vision_projection", "text_projection"   # some variants
        ]):
            if p.ndim >= 2:
                p.requires_grad_(True)
                selected.append(name)

    # vision last-N
    try:
        v_layers = m.vision_model.encoder.layers
        v_start = max(0, len(v_layers) - VISION_LAST_N)
        for li in range(v_start, len(v_layers)):
            for n, p in v_layers[li].named_parameters():
                p.requires_grad_(True)
                selected.append(f"vision_model.encoder.layers.{li}.{n}")
    except Exception:
        # fallback: regex by name
        for name, p in m.named_parameters():
            if re.search(r"vision_model\.encoder\.layers\.(\d+)\.", name):
                idx = int(re.search(r"vision_model\.encoder\.layers\.(\d+)\.", name).group(1))
                # unknown total, so only take last-N by scanning max later is hard; keep fallback simple
                # (You can ignore: on normal HF CLIP this branch won't trigger)
                pass

    # text last-M
    try:
        t_layers = m.text_model.encoder.layers
        t_start = max(0, len(t_layers) - TEXT_LAST_M)
        for li in range(t_start, len(t_layers)):
            for n, p in t_layers[li].named_parameters():
                p.requires_grad_(True)
                selected.append(f"text_model.encoder.layers.{li}.{n}")
    except Exception:
        pass

    # de-dup
    # (selected can include duplicates if projection matches multiple patterns)
    selected = list(dict.fromkeys(selected))
    return selected

if USE_SELECTIVE_GRADS:
    collect_names = enable_grads_selectively_hf_clip(model)
    print(f"\n🔧 Selective grads enabled: params={len(collect_names)}")
else:
    for p in model.parameters():
        p.requires_grad_(True)
    collect_names = [n for n, _ in model.named_parameters()]
    print("\n🔧 All grads enabled.")

# ============================================================
# 6) Build InfoNCE batch (target + negatives)
# ============================================================
def make_batch_for_item(
    target: Dict[str, Any],
    neg_pool: List[Dict[str, Any]],
    K: int = BATCH_K,
    max_tries: int = 800
) -> Optional[Tuple[List[Image.Image], List[str]]]:
    """
    Return: (images_pil_list length K, captions_list length K)
    We keep PIL until forward; processor will handle transforms/tokenization.
    """
    tgt_img = load_local_image(target)
    tgt_cap = str(target.get("caption", "")).strip()
    if tgt_img is None or not tgt_cap:
        return None

    negs = []
    tries = 0
    while len(negs) < K - 1 and tries < max_tries:
        cand = random.choice(neg_pool)
        if cand is target:
            tries += 1
            continue
        cim = load_local_image(cand)
        ccap = str(cand.get("caption", "")).strip()
        if cim is None or not ccap:
            tries += 1
            continue
        negs.append(cand)
        tries += 1

    if len(negs) < K - 1:
        return None

    batch_list = [target] + negs
    images, caps = [], []
    for obj in batch_list:
        im = load_local_image(obj)
        cp = str(obj.get("caption", "")).strip()
        if im is None:
            im = Image.new("RGB", (224, 224), color="black")
        if not cp:
            cp = " "  # avoid empty-string edge
        images.append(im)
        caps.append(cp)

    return images, caps

# ============================================================
# 7) InfoNCE loss + backward (HF CLIP)
# ============================================================
def infonce_backward(images_pil: List[Image.Image], caps: List[str]) -> bool:
    """
    Build KxK logits and compute symmetric InfoNCE (i2t and t2i).
    """
    model.train()
    model.zero_grad(set_to_none=True)

    try:
        # processor handles vision transforms + tokenization
        inputs = processor(
            text=caps,
            images=images_pil,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=CLIP_MAX_LEN,
        ).to(device)

        with torch.cuda.amp.autocast():
            out = model(**inputs)

            # image/text embeddings
            img = out.image_embeds
            txt = out.text_embeds
            if img is None or txt is None:
                # fallback: compute features
                img = model.get_image_features(pixel_values=inputs["pixel_values"])
                txt = model.get_text_features(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                )

            img = img / (img.norm(dim=-1, keepdim=True) + 1e-12)
            txt = txt / (txt.norm(dim=-1, keepdim=True) + 1e-12)

            logit_scale = model.logit_scale.exp()
            logits = logit_scale * (img @ txt.t())  # [K,K]
            labels = torch.arange(logits.size(0), device=logits.device)

            loss_i2t = F.cross_entropy(logits, labels)
            loss_t2i = F.cross_entropy(logits.t(), labels)
            loss = 0.5 * (loss_i2t + loss_t2i)

        if not torch.isfinite(loss):
            return False

        loss.backward()

        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad and p.grad is not None],
                max_norm=float(GRAD_CLIP)
            )
        return True

    except Exception:
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
        return False

# ============================================================
# 8) GradNorm score (core)
# ============================================================
def gradnorm_score_item(
    obj: Dict[str, Any],
    neg_pool: List[Dict[str, Any]],
    K: int = BATCH_K,
    p: float = PNORM,
    reduce: str = REDUCE
) -> float:
    """
    score = - mean_pnorm(gradients)   (larger => more likely member)
    """
    made = make_batch_for_item(obj, neg_pool=neg_pool, K=K)
    if made is None:
        return 0.0

    images_pil, caps = made
    ok = infonce_backward(images_pil, caps)
    if not ok:
        return 0.0

    norms = []
    with torch.no_grad():
        for _, p0 in model.named_parameters():
            if (not p0.requires_grad) or (p0.grad is None):
                continue
            g = p0.grad.detach()
            if not torch.isfinite(g).all():
                continue
            try:
                if p == np.inf:
                    norms.append(g.norm(float("inf")).float())
                else:
                    norms.append(g.norm(float(p)).float())
            except Exception:
                continue

    # cleanup
    model.zero_grad(set_to_none=True)
    model.eval()
    gc.collect()
    torch.cuda.empty_cache()

    if not norms:
        return 0.0

    st = torch.stack(norms)
    grad_norm = st.sum() if reduce == "sum" else st.mean()
    return -float(grad_norm.item())

def score_set(objs: List[Dict[str, Any]], neg_pool: List[Dict[str, Any]], desc: str) -> np.ndarray:
    out = []
    for obj in tqdm(objs, desc=desc):
        out.append(gradnorm_score_item(obj, neg_pool=neg_pool, K=BATCH_K, p=PNORM, reduce=REDUCE))
    return np.array(out, dtype=np.float32)

def auc_pair(pos: np.ndarray, neg: np.ndarray) -> float:
    y = np.array([1]*len(pos) + [0]*len(neg), dtype=np.int32)
    s = np.concatenate([pos, neg], axis=0)
    if len(s) == 0 or np.all(s == s[0]):
        return 0.5
    return float(roc_auc_score(y, s))

# ============================================================
# 9) Run: score all & compute AUC
# ============================================================
print("\n" + "="*90)
print("PubMedCLIP GradNorm MIA — Member vs Non-member (AUC only)")
print("="*90)

try:
    print("\nScoring Members...")
    M_scores = score_set(members_all, union_pool, desc="M (member)")

    print("\nScoring Non-members...")
    N_scores = score_set(nonmembers_all, union_pool, desc="N (nonmember)")

    auc_M_N = auc_pair(M_scores, N_scores)

    print("\n" + "="*90)
    print("📊 Results (GradNorm, AUC only)")
    print("="*90)
    print(f"  AUC(M vs N) = {auc_M_N:.4f}")
    print(f"  mean score M = {float(np.mean(M_scores)):.6f} | std = {float(np.std(M_scores)):.6f}")
    print(f"  mean score N = {float(np.mean(N_scores)):.6f} | std = {float(np.std(N_scores)):.6f}")

    # Save
    OUT_DIR = "/content/drive/MyDrive/pubmedclip_mia_attack_results"
    os.makedirs(OUT_DIR, exist_ok=True)
    out_json = os.path.join(OUT_DIR, "pubmedclip_gradnorm_mn_auc_only.json")

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump({
            "config": {
                "seed": SEED,
                "TOTAL_M": TOTAL_M,
                "TOTAL_N": TOTAL_N,
                "BATCH_K": BATCH_K,
                "PNORM": float(PNORM) if PNORM != np.inf else "inf",
                "REDUCE": REDUCE,
                "GRAD_CLIP": GRAD_CLIP,
                "VISION_LAST_N": VISION_LAST_N if USE_SELECTIVE_GRADS else "ALL",
                "TEXT_LAST_M": TEXT_LAST_M if USE_SELECTIVE_GRADS else "ALL",
                "MODEL_ID": MODEL_ID,
                "CLIP_MAX_LEN": CLIP_MAX_LEN,
            },
            "results": {
                "auc_M_N": float(auc_M_N),
                "n_M": int(len(M_scores)),
                "n_N": int(len(N_scores)),
                "mean_score_M": float(np.mean(M_scores)),
                "std_score_M": float(np.std(M_scores)),
                "mean_score_N": float(np.mean(N_scores)),
                "std_score_N": float(np.std(N_scores)),
            }
        }, f, ensure_ascii=False, indent=2)

    print(f"\n💾 Saved: {out_json}")

except Exception as e:
    print(f"\n❌ Unexpected error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*90)
print("✅ Done.")
print("="*90)


## NA_PDD

In [ ]:
# ==============================================================================
# ✅ NA-PDD (orig LLM logic) -> PubMedCLIP (Vision+Text)
#
# Core:
#   threshold activation -> activated neuron set
#   calib stats diff -> select top discriminative layers
#   relative ratio score (member-dom / nonmember-dom)
#
# Speed:
#   single forward per batch (no K=8, no multi-pass)
#
# Hooks:
#   Vision last-3: ViT encoder layer MLP activation_fn (GELU/QuickGELU)
#   Text   last-3: CLIP text encoder layer MLP activation_fn (FFN act)
# ==============================================================================

import os, json, random, warnings, gc, re
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
from collections import Counter

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score

warnings.filterwarnings("ignore")

# =========================
# 0) Config (✅ PubMedCLIP data path)
# =========================
DATA_BASE = "/content/pubmedclip_roco_mia"
MEMBERS_JSON     = os.path.join(DATA_BASE, "members_roco_train_1k.json")
NONMEMBERS_JSON  = os.path.join(DATA_BASE, "nonmembers_roco_test_1k.json")

SEED = 42
TOTAL_MEMBERS = 1000
TOTAL_NONMEMBERS = 1000

CALIB_MEM = 200
CALIB_NON = 200
TEST_MEM  = 800
TEST_NON  = 800

# Hook last-3
VISION_LAST = 3
TEXT_LAST   = 3

# NA-PDD threshold
ACTIVATION_THRESHOLD = 0.0

# select top discriminative layers
TOP_LAYERS = 10

# ratio threshold for accuracy
RATIO_THRESHOLD_FOR_ACC = 1.0

# inference batch size
BATCH_SIZE = 8

# ===== IMPORTANT: pubmedclip model =====
MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"
MAX_TEXT_LEN = 77  # CLIP default

print("="*100)
print("NA-PDD (orig) -> PubMedCLIP")
print("="*100)

# =========================
# 1) Device
# =========================
assert torch.cuda.is_available(), "❌ CUDA not available"
device = "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

tbc.allow_tf32 = True
cudnn.allow_tf32 = True

print(f"✅ Device: {device}")
print(f"📋 DATA_BASE={DATA_BASE}")
print(f"📋 TOTAL(M/N)={TOTAL_MEMBERS}/{TOTAL_NONMEMBERS}, CALIB(M/N)={CALIB_MEM}/{CALIB_NON}, TEST(M/N)={TEST_MEM}/{TEST_NON}")
print(f"📋 thr={ACTIVATION_THRESHOLD}, top_layers={TOP_LAYERS}, batch={BATCH_SIZE}")
print(f"📋 hook last(V/T)={VISION_LAST}/{TEXT_LAST}")

# =========================
# 2) Load PubMedCLIP (transformers)
# =========================
try:
    from transformers import CLIPModel, CLIPProcessor
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
    from transformers import CLIPModel, CLIPProcessor

print(f"⏳ Loading model: {MODEL_ID}")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()
print("✅ PubMedCLIP loaded")

# =========================
# 3) Load data
# =========================
def load_json_list(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

members_all = load_json_list(MEMBERS_JSON)
nonmembers_all = load_json_list(NONMEMBERS_JSON)

rnd = random.Random(SEED)
rnd.shuffle(members_all)
rnd.shuffle(nonmembers_all)

members_all = members_all[:TOTAL_MEMBERS]
nonmembers_all = nonmembers_all[:TOTAL_NONMEMBERS]

calib_member = members_all[:CALIB_MEM]
calib_non    = nonmembers_all[:CALIB_NON]
test_member  = members_all[CALIB_MEM:CALIB_MEM+TEST_MEM]
test_non     = nonmembers_all[CALIB_NON:CALIB_NON+TEST_NON]

print(f"\n✅ Data split:")
print(f"   Calib: {len(calib_member)} + {len(calib_non)}")
print(f"   Test : {len(test_member)} + {len(test_non)}")

def get_caption(obj: Dict[str,Any]) -> str:
    c = obj.get("caption", "")
    if isinstance(c, list) and len(c) > 0:
        c = c[0]
    return str(c).strip()

# =========================
# 4) Image loader (PubMedCLIP data usually local paths)
# =========================
def load_local_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = sample.get("local_image_path") or sample.get("image_path")
    if not p or (not os.path.exists(p)):
        return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

def load_image(obj: Dict[str,Any], kind: str) -> Optional[Image.Image]:
    # PubMedCLIP data: both sides are local in your setup
    return load_local_image(obj)

# =========================
# 5) Hooks: collect activated neuron indices per layer (orig NA-PDD style)
# =========================
# batch_activated[name] = List[List[int]]  # length=B, each inner list are neuron indices
batch_activated: Dict[str, List[List[int]]] = {}

def _hook_collect_indices(name: str):
    def hook(module, inp, out):
        """
        out can be:
          - [B, Seq, H]
          - [B, H]
          - tuple/list -> take first tensor
        We compute threshold activation and store indices per sample.
        """
        if isinstance(out, (tuple, list)):
            out = out[0]
        if not torch.is_tensor(out):
            return

        t = out.detach()

        # boolean activation mask per neuron
        if t.ndim == 3:
            # [B, Seq, H] -> [B, H] (any token > thr)
            m = (t > ACTIVATION_THRESHOLD).any(dim=1)
        elif t.ndim == 2:
            # [B, H]
            m = (t > ACTIVATION_THRESHOLD)
        else:
            # fallback: flatten to [B, -1]
            if t.ndim >= 1:
                t2 = t.reshape(t.shape[0], -1)
                m = (t2 > ACTIVATION_THRESHOLD)
            else:
                return

        idx_lists = []
        for b in range(m.shape[0]):
            idx = torch.nonzero(m[b], as_tuple=False).squeeze(1)
            if idx.numel() == 0:
                idx_lists.append([])
            else:
                idx_lists.append(idx.to("cpu", dtype=torch.int32).tolist())

        batch_activated[name] = idx_lists
    return hook

def register_hooks_last_layers_pubmedclip(m: CLIPModel, v_last: int, t_last: int):
    """
    HF CLIP structure:
      m.vision_model.encoder.layers[i].mlp.activation_fn
      m.text_model.encoder.layers[i].mlp.activation_fn
    """
    hooks = []
    names = []

    # ---- Vision last-N ----
    v_layers = m.vision_model.encoder.layers
    num_v = len(v_layers)
    v_start = max(0, num_v - v_last)

    for i in range(v_start, num_v):
        lyr = v_layers[i]
        # mlp.activation_fn is a Module (QuickGELUActivation)
        if hasattr(lyr, "mlp") and hasattr(lyr.mlp, "activation_fn"):
            act_mod = lyr.mlp.activation_fn
            h = act_mod.register_forward_hook(_hook_collect_indices(f"vision_L{i}_ffn_act"))
            hooks.append(h); names.append(f"vision_L{i}_ffn_act")

    # ---- Text last-M ----
    t_layers = m.text_model.encoder.layers
    num_t = len(t_layers)
    t_start = max(0, num_t - t_last)

    for i in range(t_start, num_t):
        lyr = t_layers[i]
        if hasattr(lyr, "mlp") and hasattr(lyr.mlp, "activation_fn"):
            act_mod = lyr.mlp.activation_fn
            h = act_mod.register_forward_hook(_hook_collect_indices(f"text_L{i}_ffn_act"))
            hooks.append(h); names.append(f"text_L{i}_ffn_act")

    print("\n" + "="*80)
    print("✅ Hooks registered (orig NA-PDD style) — PubMedCLIP")
    print("="*80)
    print(f"  total={len(hooks)}")
    print("  names:", names)
    return hooks, names

hooks, hook_names = register_hooks_last_layers_pubmedclip(model, VISION_LAST, TEXT_LAST)

# =========================
# 6) Run forward once per batch, collect activated neurons
# =========================
@torch.inference_mode()
def run_batch_collect(samples: List[Dict[str,Any]], kind: str) -> List[Dict[str,Any]]:
    """
    For a batch:
      - build pixel_values + text tokens
      - single forward: model(pixel_values, input_ids, attention_mask)
      - hooks capture activations for selected layers
      - return per-sample dict: activated_neurons map (layer -> set(indices))
    """
    batch_activated.clear()

    images = []
    caps = []
    for obj in samples:
        im = load_image(obj, kind)
        cap = get_caption(obj)
        if im is None or not cap:
            continue
        images.append(im)
        caps.append(cap)

    if len(images) == 0:
        return []

    # processor handles image normalization to pixel_values
    img_inputs = processor(images=images, return_tensors="pt").to(device)
    txt_inputs = tokenizer(
        caps,
        padding="max_length",
        truncation=True,
        max_length=MAX_TEXT_LEN,
        return_tensors="pt"
    ).to(device)

    # ONE forward pass triggers both vision & text encoder internals -> hooks fire
    _ = model(
        pixel_values=img_inputs["pixel_values"],
        input_ids=txt_inputs["input_ids"],
        attention_mask=txt_inputs["attention_mask"],
        return_dict=True
    )

    B = img_inputs["pixel_values"].shape[0]
    out = []
    for b in range(B):
        act_map = {}
        for lname, idx_lists in batch_activated.items():
            if b < len(idx_lists) and len(idx_lists[b]) > 0:
                act_map[lname] = set(idx_lists[b])
        out.append({"activated_neurons": act_map})
    return out

def collect_activations_dataset(samples: List[Dict[str,Any]], kind: str, label: int, batch_size: int) -> List[Dict[str,Any]]:
    results = []
    for i in tqdm(range(0, len(samples), batch_size), desc=f"Collect {kind}"):
        batch = samples[i:i+batch_size]
        batch_res = run_batch_collect(batch, kind)
        for r in batch_res:
            r["label"] = label
            results.append(r)

        if (i // batch_size) % 50 == 0:
            gc.collect()
            torch.cuda.empty_cache()
    return results

# =========================
# 7) Orig NA-PDD: build reference patterns from calib
# =========================
def analyze_patterns(member_samples: List[Dict[str,Any]], nonmember_samples: List[Dict[str,Any]]):
    results = {}

    layer_names = set()
    for s in member_samples + nonmember_samples:
        layer_names.update(s["activated_neurons"].keys())

    for layer in layer_names:
        mem_cnt = Counter()
        non_cnt = Counter()

        for s in member_samples:
            if layer in s["activated_neurons"]:
                mem_cnt.update(list(s["activated_neurons"][layer]))

        for s in nonmember_samples:
            if layer in s["activated_neurons"]:
                non_cnt.update(list(s["activated_neurons"][layer]))

        mem_freq = {n: c / max(1, len(member_samples)) for n, c in mem_cnt.items()}
        non_freq = {n: c / max(1, len(nonmember_samples)) for n, c in non_cnt.items()}

        # dominant = freq higher than other side * 1.5 (same as your code)
        member_dominant = {}
        for n, f in mem_freq.items():
            if (n not in non_freq) or (f > non_freq[n] * 1.5):
                member_dominant[n] = f

        nonmember_dominant = {}
        for n, f in non_freq.items():
            if (n not in mem_freq) or (f > mem_freq[n] * 1.5):
                nonmember_dominant[n] = f

        results[layer] = {
            "member_dominant": member_dominant,
            "nonmember_dominant": nonmember_dominant,
            "member_freq": mem_freq,
            "nonmember_freq": non_freq
        }
    return results

def calculate_layer_scores(reference_patterns: Dict[str,Any]):
    layer_scores = {}
    for layer, d in reference_patterns.items():
        layer_scores[layer] = len(d["member_dominant"]) - len(d["nonmember_dominant"])
    return layer_scores

def select_top_layers(layer_scores: Dict[str,float], top_n: int):
    sorted_layers = sorted(layer_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    return [k for k,_ in sorted_layers[:top_n]]

def ratio_score(sample: Dict[str,Any], reference_patterns: Dict[str,Any], layers: List[str]):
    layers_counted = 0
    total_mem = 0.0
    total_non = 0.0

    for layer in layers:
        if layer not in sample["activated_neurons"]:
            continue
        if layer not in reference_patterns:
            continue

        s_neurons = sample["activated_neurons"][layer]
        if not s_neurons:
            continue

        mem_dom = set(reference_patterns[layer]["member_dominant"].keys())
        non_dom = set(reference_patterns[layer]["nonmember_dominant"].keys())

        mem_overlap = len(s_neurons & mem_dom)
        non_overlap = len(s_neurons & non_dom)

        mem_ratio = mem_overlap / len(mem_dom) if len(mem_dom) > 0 else 0.0
        non_ratio = non_overlap / len(non_dom) if len(non_dom) > 0 else 0.0

        total_mem += mem_ratio
        total_non += non_ratio
        layers_counted += 1

    if layers_counted == 0:
        return 0.0

    avg_mem = total_mem / layers_counted
    avg_non = total_non / layers_counted
    if avg_non == 0:
        return float("inf")
    return avg_mem / avg_non

# =========================
# 8) Main: Calib -> reference, Test -> score & eval
# =========================
print("\n" + "="*90)
print("🚀 NA-PDD (orig) on PubMedCLIP — Calib collect")
print("="*90)

calib_mem_acts = collect_activations_dataset(calib_member, "member", label=1, batch_size=BATCH_SIZE)
calib_non_acts = collect_activations_dataset(calib_non,    "nonmember", label=0, batch_size=BATCH_SIZE)

print("\n" + "="*90)
print("🔧 Build reference patterns")
print("="*90)

reference_patterns = analyze_patterns(calib_mem_acts, calib_non_acts)
layer_scores = calculate_layer_scores(reference_patterns)
discriminative_layers = select_top_layers(layer_scores, TOP_LAYERS)

print("✅ Selected layers:")
for x in discriminative_layers:
    print("  -", x, "score=", layer_scores.get(x, 0))

print("\n" + "="*90)
print("🧪 NA-PDD (orig) on PubMedCLIP — Test collect")
print("="*90)

test_mem_acts = collect_activations_dataset(test_member, "member", label=1, batch_size=BATCH_SIZE)
test_non_acts = collect_activations_dataset(test_non,    "nonmember", label=0, batch_size=BATCH_SIZE)

test_all = test_mem_acts + test_non_acts
y = np.array([s["label"] for s in test_all], dtype=np.int32)

scores = []
for s in tqdm(test_all, desc="Score"):
    r = ratio_score(s, reference_patterns, discriminative_layers)
    if r == float("inf") or np.isnan(r):
        r = 1e6
    scores.append(float(r))
scores = np.array(scores, dtype=np.float64)

# AUC + TPR@5%FPR
if len(np.unique(y)) < 2 or len(scores) == 0:
    auc = 0.5
    tpr5 = 0.0
else:
    fpr, tpr, thr = roc_curve(y, scores)
    auc = roc_auc_score(y, scores)
    idx = np.where(fpr >= 0.05)[0]
    tpr5 = float(tpr[idx[0]]) if len(idx) > 0 else 0.0

# Accuracy using fixed ratio threshold
pred = (scores >= RATIO_THRESHOLD_FOR_ACC).astype(np.int32)
acc = accuracy_score(y, pred) if len(scores) > 0 else 0.0

print("\n" + "="*90)
print("📊 Results (orig NA-PDD -> PubMedCLIP)")
print("="*90)
print(f"  AUC               = {auc:.4f}")
print(f"  TPR@5%FPR         = {tpr5*100:.2f}%")
print(f"  Accuracy(ratio>=1)= {acc:.4f}")
if (y==1).sum() > 0:
    print(f"  score mean (mem)  = {scores[y==1].mean():.4f} ± {scores[y==1].std():.4f}  n={int((y==1).sum())}")
if (y==0).sum() > 0:
    print(f"  score mean (non)  = {scores[y==0].mean():.4f} ± {scores[y==0].std():.4f}  n={int((y==0).sum())}")

# cleanup hooks
for h in hooks:
    try:
        h.remove()
    except:
        pass

print("\n✅ Done.")


## dc-pdd

In [ ]:
import os, pickle, torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from tqdm import tqdm
from google.colab import drive

# 1. 初始化
drive.mount('/content/drive')

# transformers (PubMedCLIP)
try:
    from transformers import CLIPProcessor
except ImportError:
    os.system("pip -q install transformers datasets")
    from transformers import CLIPProcessor

# ✅ PubMedCLIP model id（按你实际的 PubMedCLIP 替换）
MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(MODEL_ID)
tok = processor.tokenizer

# 自动获取 vocab_size
vocab_size = int(tok.vocab_size)
print(f"✅ PubMedCLIP tokenizer vocab_size = {vocab_size}")

# 路径
FREQ_PATH = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_pubmedclip.pkl"
os.makedirs(os.path.dirname(FREQ_PATH), exist_ok=True)

# 2. 加载 C4 数据集
print("正在加载 C4 英文语料 (Streaming)...")
dataset = load_dataset(
    "allenai/c4",
    data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
    split="train",
    streaming=True
)

# 3. 统计
counter = Counter()
MAX_SAMPLES = 50000

for i, ex in enumerate(tqdm(dataset, total=MAX_SAMPLES, desc="PubMedCLIP 词频统计")):
    if i >= MAX_SAMPLES:
        break

    text = ex.get("text", "")
    if not text:
        continue

    # ✅ CLIP tokenizer：用 input_ids 统计（去掉 pad）
    enc = tok(
        text,
        truncation=True,
        max_length=77,
        padding="max_length",
        return_tensors=None
    )
    tokens = np.array(enc["input_ids"], dtype=np.int64)
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else 0
    tokens = tokens[tokens != pad_id]
    counter.update(tokens.tolist())

# 4. 平滑与归一化
print("\n正在计算平滑词频...")
freq_smo = np.ones(vocab_size, dtype=np.float64)
total_tokens = int(sum(counter.values()))

for tid, count in counter.items():
    if 0 <= tid < vocab_size:
        freq_smo[tid] += int(count)

freq_smo = freq_smo / (total_tokens + vocab_size)

# 5. 保存
with open(FREQ_PATH, "wb") as f:
    pickle.dump(freq_smo, f)

print(f"\n✅ PubMedCLIP 词频表已保存至: {FREQ_PATH}")


In [ ]:
from google.colab import drive
import os

# 重新挂载或刷新
drive.mount('/content/drive', force_remount=True)

# 检查文件是否真的在那里
FREQ_PATH = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_pubmedclip.pkl"
if os.path.exists(FREQ_PATH):
    print("✅ 确认文件已存在，可以继续！")
else:
    print("❌ 文件仍未同步，请稍等 10 秒后再次运行此单元格。")

In [ ]:
# ============================================================
# DC-PDD MIA for PubMedCLIP
# - Score: Cosine_Similarity * Mean(Difficulty_Calibration)
# ============================================================

import os, io, json, pickle, random, urllib.parse, tarfile, shutil
from pathlib import Path
from typing import Dict, Any, Optional, List

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings("ignore")

# ---------------------------
# 0) 配置与路径（✅ 修正）
# ---------------------------
DATA_BASE = "/content/pubmedclip_roco_mia"
MEMBERS_JSON    = os.path.join(DATA_BASE, "members_roco_train_1k.json")
NONMEMBERS_JSON = os.path.join(DATA_BASE, "nonmembers_roco_test_1k.json")

# ✅ 对应代码一保存的词频文件
FREQ_PATH = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_pubmedclip.pkl"

ALPHA  = 1.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ✅ PubMedCLIP 模型（transformers）
MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"

# ---- 可选：如果 member 仍来自 PMC tar（pmc_tar_url + image_file_path），打开这个开关 ----
USE_PMC_TAR_FOR_MEMBER = False

# ---------------------------
# 0.1) 基础检查（避免 silent fail）
# ---------------------------
print("📌 Path check:")
print("  DATA_BASE        =", DATA_BASE)
print("  MEMBERS_JSON     =", MEMBERS_JSON, " | exists =", os.path.exists(MEMBERS_JSON))
print("  NONMEMBERS_JSON  =", NONMEMBERS_JSON, " | exists =", os.path.exists(NONMEMBERS_JSON))
print("  FREQ_PATH        =", FREQ_PATH, " | exists =", os.path.exists(FREQ_PATH))

if not os.path.exists(MEMBERS_JSON):
    raise FileNotFoundError(f"members json not found: {MEMBERS_JSON}")
if not os.path.exists(NONMEMBERS_JSON):
    raise FileNotFoundError(f"nonmembers json not found: {NONMEMBERS_JSON}")
if not os.path.exists(FREQ_PATH):
    raise FileNotFoundError(f"freq pkl not found: {FREQ_PATH}")

# ---------------------------
# 1) 模型与数据加载
# ---------------------------
try:
    from transformers import CLIPModel, CLIPProcessor
except ImportError:
    os.system("pip -q install transformers")
    from transformers import CLIPModel, CLIPProcessor

print(f"\n⏳ 加载模型：{MODEL_ID}")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
model = CLIPModel.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

with torch.no_grad():
    LOGIT_SCALE = float(model.logit_scale.exp().item()) if hasattr(model, "logit_scale") else 1.0
print(f"✅ PubMedCLIP ready | device={DEVICE} | logit_scale={LOGIT_SCALE:.3f}")

with open(FREQ_PATH, "rb") as f:
    freq_smo = pickle.load(f)
freq_smo = np.asarray(freq_smo, dtype=np.float64)

# ---------------------------
# 1.1 图片读取辅助函数
# ---------------------------
def load_local_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    # ✅ 按你数据常见字段：local_image_path / image_path
    p = sample.get("local_image_path") or sample.get("image_path")
    if not p or not os.path.exists(p):
        return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

# ---- 可选：PMC tar（如果你 member 还沿用 BiomedCLIP 的 tar 格式）----
PMC_TAR_CACHE = "/content/pmc_tar_cache"
os.makedirs(PMC_TAR_CACHE, exist_ok=True)
_TAR_HANDLE = {}
_TAR_INDEX = {}

def load_member_image_from_tar(sample: Dict[str, Any]) -> Optional[Image.Image]:
    try:
        url = sample.get("pmc_tar_url")
        if not url:
            return None
        fname = Path(urllib.parse.urlparse(url).path).name
        local_tar = os.path.join(PMC_TAR_CACHE, fname)

        if not os.path.exists(local_tar):
            import requests
            with requests.get(url, stream=True, timeout=180) as r:
                r.raise_for_status()
                with open(local_tar, "wb") as f:
                    shutil.copyfileobj(r.raw, f)

        if local_tar not in _TAR_HANDLE:
            t = tarfile.open(local_tar, "r:gz")
            _TAR_HANDLE[local_tar] = t
            _TAR_INDEX[local_tar] = {ti.name.lower(): ti for ti in t.getmembers()}

        tar, idx = _TAR_HANDLE[local_tar], _TAR_INDEX[local_tar]
        p = (sample.get("image_file_path") or "").lower().strip()
        if not p:
            return None

        ti = idx.get(p) or next((v for k, v in idx.items() if p in k), None)
        if ti is None:
            return None

        raw = tar.extractfile(ti).read()
        return Image.open(io.BytesIO(raw)).convert("RGB")
    except Exception:
        return None

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    return load_member_image_from_tar(sample) if USE_PMC_TAR_FOR_MEMBER else load_local_image(sample)

# ---------------------------
# 2) DC-PDD 核心逻辑（PubMedCLIP）
# ---------------------------
def get_caption_token_ids(caption: str) -> np.ndarray:
    if not caption or not caption.strip():
        return np.array([], dtype=np.int64)

    enc = tokenizer(
        caption,
        truncation=True,
        max_length=77,
        padding="max_length",
        return_tensors=None
    )
    ids = np.array(enc["input_ids"], dtype=np.int64)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    ids = ids[ids != pad_id]
    return ids

@torch.inference_mode()
def compute_dc_pdd_pubmedclip(img_pil: Image.Image, caption: str) -> Optional[float]:
    # A. cosine similarity
    inputs = processor(images=img_pil, return_tensors="pt").to(DEVICE)
    img_emb = model.get_image_features(pixel_values=inputs["pixel_values"])
    img_emb = img_emb / (img_emb.norm(dim=-1, keepdim=True) + 1e-12)

    tinputs = processor(
        text=[caption],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    ).to(DEVICE)
    txt_emb = model.get_text_features(
        input_ids=tinputs["input_ids"],
        attention_mask=tinputs["attention_mask"]
    )
    txt_emb = txt_emb / (txt_emb.norm(dim=-1, keepdim=True) + 1e-12)

    cosine_sim = float((img_emb * txt_emb).sum().item())

    # B. difficulty calibration
    tokens = get_caption_token_ids(caption)
    if tokens.size == 0:
        return None

    V = int(len(freq_smo))
    diffs = []
    for tid in tokens.tolist():
        if 0 <= tid < V:
            diffs.append(float(np.log(1.0 / float(freq_smo[tid]))))
        else:
            diffs.append(0.0)

    calib_factor = float(np.mean([min(d, ALPHA) for d in diffs]))

    return cosine_sim * calib_factor

# ---------------------------
# 3) 执行评估
# ---------------------------
def main():
    with open(MEMBERS_JSON, "r", encoding="utf-8") as f:
        m_data = json.load(f)[:1000]
    with open(NONMEMBERS_JSON, "r", encoding="utf-8") as f:
        n_data = json.load(f)[:1000]

    print(f"\n📦 Loaded json:")
    print(f"  members    = {len(m_data)}")
    print(f"  nonmembers = {len(n_data)}")

    m_scores, n_scores = [], []

    # Member
    for s in tqdm(m_data, desc="DC-PDD (Members)"):
        img = load_member_image(s)
        cap = str(s.get("caption", "")).strip()
        if img is None or not cap:
            continue
        score = compute_dc_pdd_pubmedclip(img, cap)
        if score is not None and np.isfinite(score):
            m_scores.append(score)

    # Non-member
    for s in tqdm(n_data, desc="DC-PDD (Non-members)"):
        img = load_local_image(s)
        cap = str(s.get("caption", "")).strip()
        if img is None or not cap:
            continue
        score = compute_dc_pdd_pubmedclip(img, cap)
        if score is not None and np.isfinite(score):
            n_scores.append(score)

    m_s = np.array(m_scores, dtype=np.float64)
    n_s = np.array(n_scores, dtype=np.float64)

    print(f"\n✅有效得分数：members={len(m_s)} | nonmembers={len(n_s)}")
    if len(m_s) == 0 or len(n_s) == 0:
        print("❌ 有效样本为 0（member 或 nonmember），请检查：")
        print("  - json 内图片字段是否是 local_image_path 或 image_path")
        print("  - 路径是否真实存在（建议打印一个 sample 看看）")
        return

    y_true = np.concatenate([np.ones(len(m_s)), np.zeros(len(n_s))])
    y_score = np.concatenate([m_s, n_s])

    auc = roc_auc_score(y_true, y_score)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    tpr_at_5 = float(tpr[np.abs(fpr - 0.05).argmin()])

    print("\n" + "="*40)
    print("📊 PubMedCLIP DC-PDD Results (ROCO train vs test)")
    print("="*40)
    print(f"AUC:         {auc:.4f}")
    print(f"TPR @ 5%FPR: {tpr_at_5*100:.2f}%")
    print(f"Mean M:      {float(np.mean(m_s)):.4f}  | n={len(m_s)}")
    print(f"Mean N:      {float(np.mean(n_s)):.4f}  | n={len(n_s)}")

if __name__ == "__main__":
    main()


## M$^4$I

In [ ]:
# ============================================================
# M⁴I (CLIP version) on PubMedCLIP — Colab Full Pipeline (Strict, No-Leak)
#
# ✅ 目标：shadow→target 迁移的 membership inference
#   - 复刻你 BiomedCLIP 版本的“严格 membership = 微调训练集”定义
#   - 攻击特征： [ abs(img_emb - txt_emb) ; cos ; logit ]
#   - 攻击器：shadow 训练 -> target' 评估（AUC / Acc / TPR@5%FPR）
#
# 支持两种模式：
#   MODE="nonmember_pool"（默认，与你 BiomedCLIP 严格闭环一致）
#     - 只从 nonmembers_test_1k 里抽：
#         target_ft=100, shadow_ft=100, shadow_non=100, eval_non=600
#       eval_mem = target_ft (100)
#
#   MODE="classic"（传统：members_train=member, nonmembers_test=nonmember）
#     - target_ft = 从 members_train 抽 100 （target' 的 member）
#       eval_mem  = members_train 剩余 900
#       eval_non  = nonmembers_test 全部 1000
#     - shadow_ft/shadow_non 从 members_train/nonmembers_test 再抽（不与 eval 重叠）
# ============================================================

import os, json, random, math, time
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.metrics import precision_recall_fscore_support

from transformers import CLIPModel, CLIPProcessor

# -----------------------------
# 0) Config
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

ROOT = "/content/pubmedclip_roco_mia"
MEMBERS_JSON    = f"{ROOT}/members_roco_train_1k.json"
NONMEMBERS_JSON = f"{ROOT}/nonmembers_roco_test_1k.json"

OUT_DIR = f"{ROOT}/m4i_strict_out"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"

# ===== 模式选择 =====
MODE = "nonmember_pool"   # "nonmember_pool" 或 "classic"

# ===== 你要求的规模（与 BiomedCLIP 版本一致）=====
FT_TARGET_N = 100
FT_SHADOW_N = 100
ATTACK_SHADOW_NON_N = 100
EVAL_NON_N = 600

# ===== 微调超参 =====
FINETUNE_EPOCHS = 2
FINETUNE_BS = 32
FINETUNE_LR = 1e-5
FINETUNE_WD = 0.01
MAX_TEXT_LEN = 77  # CLIP 常用 77

# ===== 攻击器超参 =====
ATTACK_EPOCHS = 80
ATTACK_BS = 64
ATTACK_LR = 1e-3

# -----------------------------
# 1) Load JSON data
# -----------------------------
with open(MEMBERS_JSON, "r", encoding="utf-8") as f:
    members_1k = json.load(f)
with open(NONMEMBERS_JSON, "r", encoding="utf-8") as f:
    nonmembers_1k = json.load(f)

print(f"Loaded: members_train={len(members_1k)} nonmembers_test={len(nonmembers_1k)}")

# -----------------------------
# 2) Build strict split
# -----------------------------
def shuffled_indices(n, seed=42):
    idx = list(range(n))
    rnd = random.Random(seed)
    rnd.shuffle(idx)
    return idx

split = {}

if MODE == "nonmember_pool":
    # 完全复刻你 BiomedCLIP 的严格闭环：只从 nonmembers_test_1k 抽样
    idx = shuffled_indices(len(nonmembers_1k), seed=SEED)

    def take(k):
        nonlocal_idx = idx[:k]
        del idx[:k]
        return nonlocal_idx

    i_target_ft  = take(FT_TARGET_N)
    i_shadow_ft  = take(FT_SHADOW_N)
    i_shadow_non = take(ATTACK_SHADOW_NON_N)
    i_eval_non   = take(EVAL_NON_N)

    target_ft   = [nonmembers_1k[i] for i in i_target_ft]
    shadow_ft   = [nonmembers_1k[i] for i in i_shadow_ft]
    shadow_non  = [nonmembers_1k[i] for i in i_shadow_non]
    eval_non    = [nonmembers_1k[i] for i in i_eval_non]
    eval_mem    = target_ft  # member=target' 的训练集

    split = {
        "mode": MODE,
        "seed": SEED,
        "target_ft_idx": i_target_ft,
        "shadow_ft_idx": i_shadow_ft,
        "shadow_non_idx": i_shadow_non,
        "eval_non_idx": i_eval_non,
        "remaining_nonmembers_unused": len(idx)
    }

elif MODE == "classic":
    # 传统：members_train=member, nonmembers_test=nonmember，同时仍然做 shadow→target
    mem_idx = shuffled_indices(len(members_1k), seed=SEED)
    non_idx = shuffled_indices(len(nonmembers_1k), seed=SEED+1)

    # target' 微调集（member）
    i_target_ft = mem_idx[:FT_TARGET_N]
    mem_idx = mem_idx[FT_TARGET_N:]

    # shadow 微调集（member）与 shadow_non（nonmember）
    i_shadow_ft = mem_idx[:FT_SHADOW_N]
    mem_idx = mem_idx[FT_SHADOW_N:]

    i_shadow_non = non_idx[:ATTACK_SHADOW_NON_N]
    non_idx = non_idx[ATTACK_SHADOW_NON_N:]

    # eval：target' member 用 members_train 剩余（不与 target_ft 重叠）
    eval_mem = [members_1k[i] for i in mem_idx]  # 900
    # eval_non 用 nonmembers_test 全部剩余
    eval_non = [nonmembers_1k[i] for i in non_idx]  # 900 (因为抽走了 100 做 shadow_non)

    target_ft  = [members_1k[i] for i in i_target_ft]
    shadow_ft  = [members_1k[i] for i in i_shadow_ft]
    shadow_non = [nonmembers_1k[i] for i in i_shadow_non]

    split = {
        "mode": MODE,
        "seed": SEED,
        "target_ft_idx(mem_train)": i_target_ft,
        "shadow_ft_idx(mem_train)": i_shadow_ft,
        "shadow_non_idx(non_test)": i_shadow_non,
        "eval_mem_size": len(eval_mem),
        "eval_non_size": len(eval_non),
    }
else:
    raise ValueError("MODE must be 'nonmember_pool' or 'classic'")

print("\n[Split Summary]")
print(" target_ft:", len(target_ft))
print(" shadow_ft:", len(shadow_ft))
print(" shadow_non:", len(shadow_non))
print(" eval_mem:", len(eval_mem))
print(" eval_non:", len(eval_non))

with open(os.path.join(OUT_DIR, "split.json"), "w", encoding="utf-8") as f:
    json.dump(split, f, indent=2, ensure_ascii=False)

# -----------------------------
# 3) Load PubMedCLIP
# -----------------------------
processor = CLIPProcessor.from_pretrained(MODEL_ID)
base_model = CLIPModel.from_pretrained(MODEL_ID).to(device)
base_model.eval()
print(f"\n[OK] Loaded PubMedCLIP: {MODEL_ID}")

# -----------------------------
# 4) Dataset + Collate (processor handles padding)
# -----------------------------
def safe_open_image(path: str) -> Image.Image:
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return Image.new("RGB", (224, 224), (128, 128, 128))

class LocalPairs(Dataset):
    def __init__(self, samples: List[Dict[str,Any]]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        img_path = s.get("local_image_path", "")
        cap = s.get("caption", "") or ""
        img = safe_open_image(img_path)
        return img, cap

def make_loader(samples, bs, shuffle):
    ds = LocalPairs(samples)

    def collate(batch):
        imgs, caps = zip(*batch)
        enc = processor(
            images=list(imgs),
            text=list(caps),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_TEXT_LEN
        )
        # enc: pixel_values, input_ids, attention_mask
        return enc

    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=True, drop_last=False, collate_fn=collate)

# -----------------------------
# 5) CLIP contrastive loss (HF)
# -----------------------------
def clip_contrastive_loss_hf(model: CLIPModel, batch):
    # batch -> device
    batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
    out = model(**batch, return_loss=False)

    # HF 输出：image_embeds/text_embeds + logit_scale
    img = F.normalize(out.image_embeds.float(), dim=-1)
    txt = F.normalize(out.text_embeds.float(), dim=-1)

    logit_scale = model.logit_scale.exp().float()
    logits = logit_scale * img @ txt.t()  # [B,B]
    labels = torch.arange(img.size(0), device=device)

    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.t(), labels)
    return (loss_i2t + loss_t2i) / 2

def clone_model_from_base() -> CLIPModel:
    m = CLIPModel.from_pretrained(MODEL_ID).to(device)
    return m

def finetune_clip(train_samples, save_name: str):
    model = clone_model_from_base()
    model.train()

    loader = make_loader(train_samples, bs=FINETUNE_BS, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=FINETUNE_LR, weight_decay=FINETUNE_WD)
    scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))

    for ep in range(FINETUNE_EPOCHS):
        total = 0.0
        n = 0
        t0 = time.time()
        for batch in loader:
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device=="cuda")):
                loss = clip_contrastive_loss_hf(model, batch)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total += float(loss.item()); n += 1

        print(f"[{save_name}] epoch {ep+1}/{FINETUNE_EPOCHS} | loss={total/max(n,1):.4f} | time={time.time()-t0:.1f}s")

    model.eval()
    save_path = os.path.join(OUT_DIR, f"{save_name}.pt")
    torch.save(model.state_dict(), save_path)
    print("Saved:", save_path)
    return model

# -----------------------------
# 6) Feature extraction (M⁴I CLIP features)
#    feat = [absdiff(512), cos(1), logit(1)] => 514
# -----------------------------
@torch.no_grad()
def extract_m4i_features(model: CLIPModel, samples: List[Dict[str,Any]], batch_size=64) -> np.ndarray:
    loader = make_loader(samples, bs=batch_size, shuffle=False)
    feats = []

    for batch in loader:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            out = model(**batch, return_loss=False)

        img = F.normalize(out.image_embeds.float(), dim=-1)  # [B,512]
        txt = F.normalize(out.text_embeds.float(), dim=-1)   # [B,512]

        diff = torch.abs(img - txt)                          # [B,512]
        cos  = (img * txt).sum(dim=-1, keepdim=True)         # [B,1]
        logit = (model.logit_scale.exp().float() * cos)      # [B,1]

        f = torch.cat([diff, cos, logit], dim=-1)            # [B,514]
        feats.append(f.cpu().numpy())

    return np.concatenate(feats, axis=0)

# -----------------------------
# 7) Attack MLP (same as your BiomedCLIP pipeline)
# -----------------------------
class AttackMLP(nn.Module):
    def __init__(self, d_in=514):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

def train_attack_mlp(X, y):
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    y_t = torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(1)

    ds = torch.utils.data.TensorDataset(X_t, y_t)
    dl = DataLoader(ds, batch_size=ATTACK_BS, shuffle=True, drop_last=False)

    model = AttackMLP(d_in=X.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=ATTACK_LR)
    bce = nn.BCEWithLogitsLoss()

    for ep in range(ATTACK_EPOCHS):
        model.train()
        total = 0.0
        for bx, by in dl:
            opt.zero_grad(set_to_none=True)
            logits = model(bx)
            loss = bce(logits, by)
            loss.backward()
            opt.step()
            total += float(loss.item())

        if (ep+1) % 20 == 0:
            model.eval()
            with torch.no_grad():
                p = torch.sigmoid(model(X_t)).detach().cpu().numpy().reshape(-1)
            auc = roc_auc_score(y, p)
            acc = accuracy_score(y, (p >= 0.5).astype(int))
            print(f"[Attack] ep {ep+1:3d}/{ATTACK_EPOCHS} | loss={total/max(len(dl),1):.4f} | AUC={auc:.4f} | Acc={acc:.4f}")

    return model

def metrics_at_fpr(y_true, scores, target_fpr=0.05):
    fpr, tpr, thr = roc_curve(y_true, scores)
    idx = np.searchsorted(fpr, target_fpr, side="right")
    if idx == 0:
        thr_star = thr[0]; tpr_star = tpr[0]
    elif idx >= len(thr):
        thr_star = thr[-1]; tpr_star = tpr[-1]
    else:
        x0, x1 = fpr[idx-1], fpr[idx]
        y0, y1 = tpr[idx-1], tpr[idx]
        t0, t1 = thr[idx-1], thr[idx]
        w = (target_fpr - x0) / (x1 - x0 + 1e-12)
        thr_star = t0 + w*(t1 - t0)
        tpr_star = y0 + w*(y1 - y0)

    y_pred = (scores >= thr_star).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr_act = fp / (fp + tn + 1e-12)
    tpr_act = tp / (tp + fn + 1e-12)
    acc = accuracy_score(y_true, y_pred)
    return {
        "thr@5%FPR": float(thr_star),
        "TPR@5%FPR(interp)": float(tpr_star),
        "FPR(actual)": float(fpr_act),
        "TPR(actual)": float(tpr_act),
        "Precision": float(prec),
        "Recall": float(rec),
        "F1": float(f1),
        "Accuracy": float(acc),
        "AUC": float(roc_auc_score(y_true, scores)),
        "CM": cm
    }

# ============================================================
# 8) Main pipeline
# ============================================================
print("\n" + "="*70)
print("STEP 1) Finetune target' (member=target_ft)")
print("="*70)
target_model = finetune_clip(target_ft, "target_pubmedclip_finetuned_100")

print("\n" + "="*70)
print("STEP 2) Finetune shadow (member=shadow_ft)")
print("="*70)
shadow_model = finetune_clip(shadow_ft, "shadow_pubmedclip_finetuned_100")

print("\n" + "="*70)
print("STEP 3) Train attack model on shadow features (shadow_ft=1, shadow_non=0)")
print("="*70)
X_shadow_mem = extract_m4i_features(shadow_model, shadow_ft, batch_size=64)
X_shadow_non = extract_m4i_features(shadow_model, shadow_non, batch_size=64)

X_train = np.concatenate([X_shadow_mem, X_shadow_non], axis=0)
y_train = np.array([1]*len(X_shadow_mem) + [0]*len(X_shadow_non), dtype=np.int64)

print("Attack train:", X_train.shape, y_train.shape)
attack = train_attack_mlp(X_train, y_train)

attack_path = os.path.join(OUT_DIR, "attack_mlp.pt")
torch.save(attack.state_dict(), attack_path)
print("Saved attack:", attack_path)

print("\n" + "="*70)
print("STEP 4) Evaluate on target' features (eval_mem vs eval_non)")
print("="*70)
X_eval_mem = extract_m4i_features(target_model, eval_mem, batch_size=64)
X_eval_non = extract_m4i_features(target_model, eval_non, batch_size=64)

X_eval = np.concatenate([X_eval_mem, X_eval_non], axis=0)
y_eval = np.array([1]*len(X_eval_mem) + [0]*len(X_eval_non), dtype=np.int64)

attack.eval()
with torch.no_grad():
    s = torch.sigmoid(attack(torch.tensor(X_eval, dtype=torch.float32, device=device))).cpu().numpy().reshape(-1)

auc = roc_auc_score(y_eval, s)
acc = accuracy_score(y_eval, (s >= 0.5).astype(int))
cm = confusion_matrix(y_eval, (s >= 0.5).astype(int))

print(f"AUC = {auc:.4f}")
print(f"Acc = {acc:.4f}")
print("CM@0.5:\n", cm)

m5 = metrics_at_fpr(y_eval, s, target_fpr=0.05)
print("\n@5%FPR:")
print(f" thr@5%FPR          = {m5['thr@5%FPR']:.6f}")
print(f" TPR@5%FPR (interp) = {m5['TPR@5%FPR(interp)']:.4f} | 实际FPR={m5['FPR(actual)']*100:.2f}% 实际TPR={m5['TPR(actual)']*100:.2f}%")
print(f" Precision          = {m5['Precision']:.4f}")
print(f" Recall             = {m5['Recall']:.4f}")
print(f" F1                 = {m5['F1']:.4f}")
print(f" Accuracy           = {m5['Accuracy']:.4f}")
print(" CM:\n", m5["CM"])

# Save results
res_path = os.path.join(OUT_DIR, "results.json")
with open(res_path, "w", encoding="utf-8") as f:
    json.dump({
        "model_id": MODEL_ID,
        "mode": MODE,
        "seed": SEED,
        "finetune": {
            "epochs": FINETUNE_EPOCHS,
            "bs": FINETUNE_BS,
            "lr": FINETUNE_LR,
            "wd": FINETUNE_WD
        },
        "attack": {
            "epochs": ATTACK_EPOCHS,
            "bs": ATTACK_BS,
            "lr": ATTACK_LR,
            "feat_dim": int(X_train.shape[1])
        },
        "sizes": {
            "target_ft": len(target_ft),
            "shadow_ft": len(shadow_ft),
            "shadow_non": len(shadow_non),
            "eval_mem": len(eval_mem),
            "eval_non": len(eval_non),
        },
        "eval": {
            "AUC": float(auc),
            "Acc@0.5": float(acc),
            "CM@0.5": cm.tolist(),
            "@5%FPR": {
                "thr@5%FPR": m5["thr@5%FPR"],
                "TPR@5%FPR(interp)": m5["TPR@5%FPR(interp)"],
                "FPR(actual)": m5["FPR(actual)"],
                "TPR(actual)": m5["TPR(actual)"],
                "Precision": m5["Precision"],
                "Recall": m5["Recall"],
                "F1": m5["F1"],
                "Accuracy": m5["Accuracy"],
                "AUC": m5["AUC"],
                "CM": m5["CM"].tolist()
            }
        }
    }, f, indent=2, ensure_ascii=False)

print("\nSaved:", res_path)
print("OUT_DIR:", OUT_DIR)


## GradAudit

In [ ]:
# =========================================================================
# GradAudit-only (GradSafe-style) MIA for PubMedCLIP (Transformers CLIPModel)
# 目标：把你 BiomedCLIP(open_clip) 那套“校准->差分参考梯度->敏感行/列mask->probe打分”
#       一模一样搬到 PubMedCLIP 上，
#       且【只修复】你指出的 confound：
#         - make_batch_for_item 里重复读图/重复decode
#         - neg 失败用黑图补位导致 batch 不一致
#       => 改为：每个样本只加载一次，neg失败重采样，凑不齐K直接丢弃(返回None)
#
# 依赖前提：
#   - members_all / nonmembers_all 已经是 list[dict]，每个元素至少包含：
#       {"local_image_path": "...", "caption": "..."}
#   - 或者你想从 JSON 加载也可以（下面提供路径）
#   - model/processor 已可从 HuggingFace 拉取：
#       MODEL_ID="flaviagiammarino/pubmed-clip-vit-base-patch32"
# =========================================================================

import os, json, random, warnings, re, gc
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
from collections import OrderedDict

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.backends.cuda as tbc

from transformers import CLIPModel, CLIPProcessor

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

# ---------------------------
# 0) 路径与超参（保持你原逻辑）
# ---------------------------
# 如果你已经在notebook里有 members_all/nonmembers_all 变量，可以把下面两行置空不读json
MEMBERS_JSON     = "/content/pubmedclip_roco_mia/members_roco_train_1k.json"
NONMEMBERS_JSON  = "/content/pubmedclip_roco_mia/nonmembers_roco_test_1k.json"

SEED = 42
NUM_MEMBERS    = 1000
NUM_NONMEMBERS = 1000

CALIB_MEM = 200
CALIB_NON = 200
PROBE_MEM = 800
PROBE_NON = 800

BATCH_K = 8
SENS_QUANTILE = 0.95
VISION_LAST_N = 6
TEXT_LAST_M   = 6

# PubMedCLIP / CLIP text max len
CLIP_MAX_LEN = 77

# ---------------------------
# 1) 设备 & 性能设置
# ---------------------------
assert torch.cuda.is_available(), "未检测到 CUDA"
device = "cuda"
tbc.allow_tf32 = True
cudnn.allow_tf32 = True
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device}")

# ---------------------------
# 2) 加载 PubMedCLIP (HF transformers)
# ---------------------------
MODEL_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"
print(f"⏳ 加载模型：{MODEL_ID}")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()
print("✅ PubMedCLIP 加载完成")

with torch.no_grad():
    try:
        LOGIT_SCALE = float(model.logit_scale.exp().detach().cpu())
    except Exception:
        LOGIT_SCALE = 1.0
print(f"🔧 logit_scale={LOGIT_SCALE:.3f} | max_len={CLIP_MAX_LEN}")

# ---------------------------
# 3) 读取数据（若你已在上游生成 members_all/nonmembers_all，可跳过）
# ---------------------------
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

if "members_all" not in globals() or "nonmembers_all" not in globals():
    members_all = load_json_list(MEMBERS_JSON)
    nonmembers_all = load_json_list(NONMEMBERS_JSON)

# 截断到指定数量（保持你原逻辑）
random.Random(SEED).shuffle(members_all)
random.Random(SEED).shuffle(nonmembers_all)
members_all = members_all[:NUM_MEMBERS]
nonmembers_all = nonmembers_all[:NUM_NONMEMBERS]
print(f"\n📦 样本就绪：members={len(members_all)} | non-members={len(nonmembers_all)}")

# ---------------------------
# 4) 【关键修复】图片缓存 + 单次加载（避免重复读取/失败补位）
# ---------------------------
_PMC_IMG_CACHE = OrderedDict()
_PMC_IMG_CACHE_MAX = 512  # 只缓存PIL RGB，避免重复decode

def _pmc_key(obj: Dict[str,Any]) -> str:
    return str(obj.get("local_image_path",""))

def _cache_get(k: str):
    if k in _PMC_IMG_CACHE:
        _PMC_IMG_CACHE.move_to_end(k)
        return _PMC_IMG_CACHE[k]
    return None

def _cache_put(k: str, v: Image.Image):
    _PMC_IMG_CACHE[k] = v
    _PMC_IMG_CACHE.move_to_end(k)
    if len(_PMC_IMG_CACHE) > _PMC_IMG_CACHE_MAX:
        _PMC_IMG_CACHE.popitem(last=False)

def load_pmc_image_cached(obj: Dict[str,Any]) -> Optional[Image.Image]:
    """
    返回 PIL RGB；失败返回 None
    - 缓存避免同一图片被重复打开/重复decode
    """
    k = _pmc_key(obj)
    if not k:
        return None

    cached = _cache_get(k)
    if cached is not None:
        return cached

    p = obj.get("local_image_path")
    if not p or not os.path.exists(p):
        return None

    try:
        im = Image.open(p).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError, Exception):
        return None

    _cache_put(k, im)
    return im

def make_batch_for_item(
    target: Dict[str,Any],
    neg_pool: List[Dict[str,Any]],
    kind: str,                 # 保持签名一致，不改变上层逻辑
    K: int = BATCH_K,
    max_tries: int = 400
) -> Optional[Tuple[List[Image.Image], List[str]]]:
    """
    ✅ 修复版 K-way batch 构造（只修这个点，其他不变）：
      - target 只加载一次
      - negatives 失败就重采样
      - 不用黑图补位；凑不齐 K 直接返回 None
    返回：images_pil(list length K), captions(list length K)
    """
    tgt_img = load_pmc_image_cached(target)
    tgt_cap = str(target.get("caption","")).strip()
    if tgt_img is None or not tgt_cap:
        return None

    imgs = [tgt_img]
    caps = [tgt_cap]
    used = set([_pmc_key(target)])

    tries = 0
    while len(imgs) < K and tries < max_tries:
        tries += 1
        cand = random.choice(neg_pool)
        ck = _pmc_key(cand)
        if not ck or ck in used:
            continue
        ccap = str(cand.get("caption","")).strip()
        if not ccap:
            continue
        cim = load_pmc_image_cached(cand)
        if cim is None:
            continue

        used.add(ck)
        imgs.append(cim)
        caps.append(ccap)

    if len(imgs) < K:
        return None

    return imgs, caps

# ---------------------------
# 5) 选择性开梯度（匹配 transformers CLIP 命名）
# ---------------------------
def enable_grads_selectively_pubmedclip(m: CLIPModel) -> List[str]:
    """
    类似你原来的 enable_grads_selectively_CLIP：
    - 只在末端 VISION_LAST_N / TEXT_LAST_M 的 encoder layers 上打开梯度
    - 额外包含 visual_projection / text_projection
    - 仅对 ndims>=2 的权重收集(跟你原规则对齐)
    """
    selected = []
    for _, p in m.named_parameters():
        p.requires_grad_(False)

    # 找最大层号
    max_v_layer, max_t_layer = 0, 0
    for name, _ in m.named_parameters():
        mv = re.search(r"vision_model\.encoder\.layers\.(\d+)\.", name)
        mt = re.search(r"text_model\.encoder\.layers\.(\d+)\.", name)
        if mv:
            max_v_layer = max(max_v_layer, int(mv.group(1)))
        if mt:
            max_t_layer = max(max_t_layer, int(mt.group(1)))

    v_start = max(0, max_v_layer - VISION_LAST_N + 1)
    t_start = max(0, max_t_layer - TEXT_LAST_M + 1)

    for name, p in m.named_parameters():
        # projection（CLIPModel 里通常是 Linear）
        if name in ["visual_projection.weight", "text_projection.weight"] and p.ndim >= 2:
            p.requires_grad_(True)
            selected.append(name)
            continue

        mv = re.search(r"vision_model\.encoder\.layers\.(\d+)\.", name)
        if mv:
            idx = int(mv.group(1))
            if idx >= v_start and p.ndim >= 2 and re.search(r"(attn|mlp|fc|proj|linear|dense|qkv|out_proj)", name):
                p.requires_grad_(True)
                selected.append(name)
                continue

        mt = re.search(r"text_model\.encoder\.layers\.(\d+)\.", name)
        if mt:
            idx = int(mt.group(1))
            if idx >= t_start and p.ndim >= 2 and re.search(r"(attention|query|key|value|dense|intermediate|output|fc)", name):
                p.requires_grad_(True)
                selected.append(name)
                continue

    return selected

collect_names = enable_grads_selectively_pubmedclip(model)
print(f"🔧 计划收集梯度的参数数量 (N={VISION_LAST_N}, M={TEXT_LAST_M} + Proj): {len(collect_names)}")

# ---------------------------
# 6) 对比损失 & 梯度提取（PubMedCLIP版，保持你原结构）
# ---------------------------
def _encode_batch(images_pil: List[Image.Image], caps: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    返回归一化后的 img_feats/txt_feats: [K, D] on device
    必须 truncation 到 77，避免你之前的报错。
    """
    inputs = processor(
        images=images_pil,
        text=caps,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CLIP_MAX_LEN
    ).to(device)

    # image/text features
    img_feats = model.get_image_features(pixel_values=inputs["pixel_values"])
    txt_feats = model.get_text_features(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])

    img_feats = img_feats / (img_feats.norm(dim=-1, keepdim=True) + 1e-12)
    txt_feats = txt_feats / (txt_feats.norm(dim=-1, keepdim=True) + 1e-12)
    return img_feats, txt_feats

def contrastive_loss_and_backward(images_pil: List[Image.Image], caps: List[str]) -> bool:
    """
    计算 CLIP 对称 InfoNCE loss 并 backward
    - 与你 open_clip 版本保持一致：loss = (CE(i2t)+CE(t2i))/2
    """
    model.train()
    model.zero_grad(set_to_none=True)

    try:
        with torch.cuda.amp.autocast():
            img_feats, txt_feats = _encode_batch(images_pil, caps)  # [K,D],[K,D]
            if (not torch.isfinite(img_feats).all()) or (not torch.isfinite(txt_feats).all()):
                return False

            logit_scale = model.logit_scale.exp()
            logits_i = logit_scale * (img_feats @ txt_feats.t())    # [K,K]
            labels = torch.arange(len(caps), device=device)

            loss_i2t = F.cross_entropy(logits_i, labels)
            loss_t2i = F.cross_entropy(logits_i.t(), labels)
            loss = (loss_i2t + loss_t2i) / 2

        if not torch.isfinite(loss):
            return False

        loss.backward()
        return True

    except (RuntimeError, Exception):
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
        return False

def compute_grad_dict_for_item(
    target: Dict[str,Any],
    neg_pool: List[Dict[str,Any]],
    kind: str,
    K: int = BATCH_K
) -> Dict[str, torch.Tensor]:
    """
    返回 grad_dict[name]=grad(cpu)；只收集 requires_grad 且 grad 非空的参数
    """
    made = make_batch_for_item(target, neg_pool, kind=kind, K=K)
    if made is None:
        return {}

    images_pil, caps = made
    ok = contrastive_loss_and_backward(images_pil, caps)
    if not ok:
        return {}

    grad_dict = {}
    for name, p in model.named_parameters():
        if p.requires_grad and (p.grad is not None):
            if torch.isfinite(p.grad).all():
                # 这里保持你原风格：存到 CPU（建议 float32 更稳，但你说其他不变，这里仍可 half）
                grad_dict[name] = p.grad.detach().half().cpu()
        if p.grad is not None:
            p.grad = None

    model.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()
    model.eval()
    return grad_dict if grad_dict else {}

# ---------------------------
# 7) 行/列余弦（保持你原函数）
# ---------------------------
def row_col_cos(a: torch.Tensor, b: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    try:
        a_float = a.float()
        b_float = b.float()
        if a_float.ndim < 2:
            flat = F.cosine_similarity(a_float.flatten().unsqueeze(0), b_float.flatten().unsqueeze(0), dim=1)
            return torch.nan_to_num(flat, 0.0), torch.nan_to_num(flat, 0.0)
        r = F.cosine_similarity(a_float,   b_float,   dim=1)
        c = F.cosine_similarity(a_float.T, b_float.T, dim=1)
        return torch.nan_to_num(r, 0.0), torch.nan_to_num(c, 0.0)
    except Exception:
        if a.ndim >= 2:
            return torch.zeros(a.shape[0]), torch.zeros(a.shape[1])
        else:
            return torch.zeros(1), torch.zeros(1)

# ---------------------------
# 8) 校准（保持你原逻辑）
# ---------------------------
def _build_avg_grad(calib_samples: List[Dict[str,Any]], neg_pool: List[Dict[str,Any]], kind: str, take: int) -> Dict[str, torch.Tensor]:
    ok = 0
    valid_grads_list = []
    for i in tqdm(range(take), desc=f"计算平均梯度 ({kind})"):
        gd = compute_grad_dict_for_item(calib_samples[i], neg_pool, kind=kind, K=BATCH_K)
        if gd:
            valid_grads_list.append(gd)
            ok += 1

    if ok == 0:
        raise RuntimeError(f"{kind} 平均梯度构建失败（有效样本为0）")

    common_keys = set(valid_grads_list[0].keys())
    for gd in valid_grads_list[1:]:
        common_keys.intersection_update(gd.keys())

    ref = {}
    for k in common_keys:
        first_shape = valid_grads_list[0][k].shape
        if all((k in gd and gd[k].shape == first_shape) for gd in valid_grads_list):
            s = torch.zeros_like(valid_grads_list[0][k], dtype=torch.float32)
            for gd in valid_grads_list:
                s += gd[k].float()
            ref[k] = s / ok

    if not ref:
        raise RuntimeError(f"{kind} 平均梯度聚合失败（common_keys为空或shape不一致）")
    return ref

def avg_rowcol_sims(samples: List[Dict[str,Any]], ref: Dict[str,torch.Tensor], neg_pool: List[Dict[str,Any]], kind: str, take: int) -> Tuple[Dict[str,torch.Tensor], Dict[str,torch.Tensor]]:
    acc_r, acc_c = {}, {}
    cnt = 0
    for i in tqdm(range(take), desc=f"计算相似度 ({kind})"):
        gd = compute_grad_dict_for_item(samples[i], neg_pool, kind=kind, K=BATCH_K)
        if not gd:
            continue
        cnt += 1
        for name, g in gd.items():
            if name not in ref or ref[name].shape != g.shape:
                continue
            rs, cs = row_col_cos(g, ref[name])
            if name not in acc_r:
                acc_r[name] = rs.clone().float()
                acc_c[name] = cs.clone().float()
            else:
                if acc_r[name].shape == rs.shape:
                    acc_r[name] += rs.float()
                if acc_c[name].shape == cs.shape:
                    acc_c[name] += cs.float()

    if cnt == 0:
        return {}, {}

    for k in list(acc_r.keys()):
        acc_r[k] /= cnt
        acc_c[k] /= cnt
    return acc_r, acc_c

def build_sensitive_masks(mem_rc, non_rc, q: float):
    mr, mc = mem_rc
    nr, nc = non_rc
    row_masks, col_masks, total = {}, {}, 0
    if (not mr) or (not nr):
        return {}, {}, 0

    for name in mr:
        if name not in nr or name not in mc or name not in nc:
            continue
        if mr[name].shape != nr[name].shape or mc[name].shape != nc[name].shape:
            continue

        rgap = torch.nan_to_num(mr[name] - nr[name], nan=0.0).cpu()
        cgap = torch.nan_to_num(mc[name] - nc[name], nan=0.0).cpu()

        rt = torch.quantile(rgap, q) if rgap.numel() > 0 else torch.tensor(float("inf"))
        ct = torch.quantile(cgap, q) if cgap.numel() > 0 else torch.tensor(float("inf"))

        rm = (rgap > rt) & (rgap > 0)
        cm = (cgap > ct) & (cgap > 0)

        if rm.any() or cm.any():
            row_masks[name] = rm
            col_masks[name] = cm
            total += int(rm.sum().item() + cm.sum().item())

    return row_masks, col_masks, total

# ---------------------------
# 9) 探测：单样本打分（保持你原逻辑）
# ---------------------------
def gradsafe_score_item(obj: Dict[str,Any], ref: Dict[str,torch.Tensor], row_masks, col_masks, neg_pool: List[Dict[str,Any]], kind: str) -> float:
    if (not row_masks) and (not col_masks):
        return 0.0

    gd = compute_grad_dict_for_item(obj, neg_pool, kind=kind, K=BATCH_K)
    if not gd:
        return 0.0

    sims = []
    for name, g in gd.items():
        if name in ref and name in row_masks and name in col_masks and ref[name].shape == g.shape:
            rs, cs = row_col_cos(g, ref[name])
            rm = row_masks.get(name)
            cm = col_masks.get(name)
            if rm is not None and rm.any():
                sims.extend(rs[rm].cpu().tolist())
            if cm is not None and cm.any():
                sims.extend(cs[cm].cpu().tolist())

    return float(np.median(sims)) if sims else 0.0

def score_probe_sets(probe_members, probe_nonmembers, ref, row_masks, col_masks, neg_pool_mem, neg_pool_non):
    scores, labels = [], []
    for obj in tqdm(probe_members, desc="Probe-Members"):
        s = gradsafe_score_item(obj, ref, row_masks, col_masks, neg_pool_mem, kind="member")
        scores.append(s)
        labels.append(1)
    for obj in tqdm(probe_nonmembers, desc="Probe-NonMembers"):
        s = gradsafe_score_item(obj, ref, row_masks, col_masks, neg_pool_non, kind="nonmember")
        scores.append(s)
        labels.append(0)
    return np.array(scores), np.array(labels)

# ---------------------------
# 10) 主流程（保持你原设置/结构；只修 batch 构造）
# ---------------------------
random.shuffle(members_all)
random.shuffle(nonmembers_all)

calib_members = members_all[:CALIB_MEM]
calib_nonmems = nonmembers_all[:CALIB_NON]

probe_members = members_all[CALIB_MEM:CALIB_MEM + PROBE_MEM]
probe_nonmems = nonmembers_all[CALIB_NON:CALIB_NON + PROBE_NON]

print(f"\n划分 | 校准: M={len(calib_members)} / N={len(calib_nonmems)}  探测: M={len(probe_members)} / N={len(probe_nonmems)}")
print(f"Batch(K)={BATCH_K}, q={SENS_QUANTILE}, Vision_N={VISION_LAST_N}, Text_M={TEXT_LAST_M}")

# 负例池（保持你原写法）
neg_pool_for_member = members_all[CALIB_MEM + PROBE_MEM:] or members_all
neg_pool_for_nonmem = nonmembers_all[CALIB_NON + PROBE_NON:] or nonmembers_all

try:
    print("\n[V2.1-PubMedCLIP] 正在计算差分参考梯度...")
    avg_grad_mem = _build_avg_grad(calib_members, neg_pool_for_member, kind="member", take=len(calib_members))
    avg_grad_non = _build_avg_grad(calib_nonmems, neg_pool_for_nonmem, kind="nonmember", take=len(calib_nonmems))

    ref_grads = {}
    for k in avg_grad_mem:
        if k in avg_grad_non and avg_grad_mem[k].shape == avg_grad_non[k].shape:
            ref_grads[k] = (avg_grad_mem[k].float() - avg_grad_non[k].float()).detach()

    print(f"✅ 差分参考梯度已构建 (层数: {len(ref_grads)})")

    mr, mc = avg_rowcol_sims(calib_members, ref_grads, neg_pool_for_member, kind="member", take=len(calib_members))
    nr, nc = avg_rowcol_sims(calib_nonmems, ref_grads, neg_pool_for_nonmem, kind="nonmember", take=len(calib_nonmems))

    row_masks, col_masks, total_crit = build_sensitive_masks((mr, mc), (nr, nc), q=SENS_QUANTILE)
    print(f"敏感子维度总数 (q={SENS_QUANTILE}): {total_crit}")

    if total_crit > 0:
        scores, labels = score_probe_sets(
            probe_members, probe_nonmems,
            ref_grads, row_masks, col_masks,
            neg_pool_for_member, neg_pool_for_nonmem
        )

        if len(np.unique(scores)) <= 1:
            print("\n⚠️ 探测评分失败或所有分数相同")
            auc, best_thr, acc, cm = np.nan, np.nan, np.nan, np.array([[0, 0], [0, 0]])
        else:
            auc = roc_auc_score(labels, scores)
            fpr, tpr, thr = roc_curve(labels, scores)
            best_idx = np.argmax(tpr - fpr)
            best_thr = thr[best_idx]
            pred = (scores >= best_thr).astype(int)
            acc = accuracy_score(labels, pred)
            cm = confusion_matrix(labels, pred)

        print("\n====== GradSafe-MIA 结果 (PubMedCLIP) ======")
        print(f"AUC  = {auc:.4f}")
        print(f"阈值 = {best_thr:.4f}")
        print(f"Acc  = {acc:.4f}")
        print("Confusion matrix [[TN FP][FN TP]]：")
        print(cm)

        non_scores_np = scores[labels == 0].astype(np.float64)
        if len(non_scores_np) > 0 and len(np.unique(non_scores_np)) > 1:
            thr_5 = float(np.quantile(non_scores_np, 0.95))
            pred_5 = (scores >= thr_5).astype(int)
            cm_5 = confusion_matrix(labels, pred_5)
            tn, fp, fn, tp = cm_5.ravel()
            fpr_actual = fp / (fp + tn + 1e-12)
            tpr_5 = tp / (tp + fn + 1e-12)
            prec_5 = precision_score(labels, pred_5, zero_division=0)
            rec_5  = recall_score(labels, pred_5, zero_division=0)
            f1_5   = f1_score(labels, pred_5, zero_division=0)

            print("\n------ @5% FPR Metrics ------")
            print(f"Threshold @FPR=5% = {thr_5:.4f}  (实际 FPR={fpr_actual*100:.2f}%)")
            print(f"TPR (Recall)      = {tpr_5*100:.2f}%")
            print(f"Precision          = {prec_5*100:.2f}%")
            print(f"F1                 = {f1_5:.4f}")
            print("Confusion @5%FPR [[TN FP][FN TP]]：")
            print(cm_5)
        else:
            print("\n------ @5% FPR Metrics ------")
            print("⚠️ 无法计算 @5% FPR 指标。")
    else:
        print("\n⚠️ 敏感子维度数量为 0，无法进行探测评分。")

except RuntimeError as e:
    print(f"\n❌ 主流程执行失败: {e}")
except Exception as e:
    print(f"\n❌ 主流程发生意外错误: {e}")


# Qwen-FashionGen

In [ ]:
# ============================================================
# 必须先挂载 Drive (否则数据会丢)
# @title 0. Data setup
# ============================================================
from google.colab import drive
import os
print("📂 正在挂载 Google Drive...")
drive.mount('/content/drive')

# ============================================================
# A. Install deps (保持最轻量)
# ============================================================
import subprocess, sys
def pip_install(pkgs):
    for p in pkgs:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

pip_install(["tqdm", "huggingface_hub", "h5py", "numpy", "pillow"])

# ============================================================
# B. Download FashionGen (.h5) from HuggingFace -> Drive
# ============================================================
import json, random, time, shutil
import numpy as np
import h5py
from PIL import Image
from tqdm import tqdm
from huggingface_hub import hf_hub_download

# -----------------------------
# 0) 路径设置
# -----------------------------
drive_root = "/content/drive/MyDrive/FashionGen_Preprocessed_A100"
drive_dataset_root = os.path.join(drive_root, "FashionGen_h5")
os.makedirs(drive_dataset_root, exist_ok=True)

train_h5_path = os.path.join(drive_dataset_root, "fashiongen_256_256_train.h5")
val_h5_path   = os.path.join(drive_dataset_root, "fashiongen_256_256_validation.h5")

# 目标输出（给你后续 MIA 用）
dst_root = "/content/drive/MyDrive/FashionGen_MIA"
member_root = os.path.join(dst_root, "member_5k")
nonmember_root = os.path.join(dst_root, "nonmember_5k")
member_img_dir = os.path.join(member_root, "images")
nonmember_img_dir = os.path.join(nonmember_root, "images")
os.makedirs(member_img_dir, exist_ok=True)
os.makedirs(nonmember_img_dir, exist_ok=True)

member_json_path = os.path.join(member_root, "mllm_data.json")
nonmember_json_path = os.path.join(nonmember_root, "mllm_data.json")

# -----------------------------
# 1) 目标数量 & 随机种子
# -----------------------------
SEED = 20251012
random.seed(SEED)
np.random.seed(SEED)

TARGET_MEMBER = 5000
TARGET_NONMEMBER = 5000

PROMPT_USER_EN = (
    "<image> You are a fashion vision-language model. "
    "Please describe the product in the image with detailed, factual attributes "
    "(e.g., category, color, material, style, pattern, notable details)."
)

# -----------------------------
# 2) 下载 h5（若已存在则跳过）
# -----------------------------
REPO_ID = "hieupth/fashiongen"

def ensure_file(local_path, filename):
    if os.path.exists(local_path) and os.path.getsize(local_path) > 1024 * 1024:
        print(f"✅ 已存在: {local_path}")
        return local_path
    print(f"⬇️ 下载: {filename} -> {local_path}")
    tmp = hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset",
        cache_dir="/content/hf_cache",
    )
    shutil.copy2(tmp, local_path)
    print(f"✅ 已落盘到 Drive: {local_path}")
    return local_path

train_h5_path = ensure_file(train_h5_path, "fashiongen_256_256_train.h5")
val_h5_path   = ensure_file(val_h5_path,   "fashiongen_256_256_validation.h5")

print("🎉 FashionGen h5 准备完成")
print("  -", train_h5_path)
print("  -", val_h5_path)

# ============================================================
# C. 从 h5 里抽取 10k 条有效样本，构建 member/nonmember
# ============================================================

# ✅ 适配你现在的真实格式：np.ndarray(shape=(1,), dtype='|S800') -> bytes
def decode_text(x):
    # numpy scalar
    if isinstance(x, np.generic):
        x = x.item()

    # 关键：np.ndarray (shape=(1,), dtype='|S800')
    if isinstance(x, np.ndarray):
        if x.size == 0:
            return ""
        if x.size == 1:
            return decode_text(x.reshape(-1)[0])
        return " ".join([decode_text(xx) for xx in x.reshape(-1)]).strip()

    if isinstance(x, (bytes, np.bytes_)):
        try:
            return x.decode("utf-8", errors="ignore").strip()
        except:
            return str(x).strip()

    if isinstance(x, str):
        return x.strip()

    return str(x).strip()

def open_h5_and_get_handles(h5_path):
    f = h5py.File(h5_path, "r")

    img_key = "input_image"
    txt_key = "input_concat_description"

    if img_key not in f or txt_key not in f:
        keys = list(f.keys())
        f.close()
        raise RuntimeError(f"❌ 找不到 {img_key}/{txt_key}。当前顶层 keys: {keys[:80]}")

    imgs = f[img_key]
    txts = f[txt_key]
    n = min(len(imgs), len(txts))
    print(f"✅ Using keys: images='{img_key}', text='{txt_key}', N={n}")

    # 样例确认
    t0 = decode_text(txts[0])
    print(f"🔎 Sample text length: {len(t0.split())} words | {len(t0)} chars")
    print("🔎 Sample text preview:", t0[:220].replace("\n", " "), "...")
    return f, imgs, txts, n

def assign_bucket(member_records, nonmember_records):
    if len(member_records) >= TARGET_MEMBER:
        return "nonmember"
    if len(nonmember_records) >= TARGET_NONMEMBER:
        return "member"
    return "member" if random.random() < 0.5 else "nonmember"

def save_image_array(arr, out_path):
    a = np.array(arr)
    # HWC or CHW
    if a.ndim == 3 and a.shape[0] == 3 and a.shape[-1] != 3:
        a = np.transpose(a, (1, 2, 0))
    a = a.astype(np.uint8)
    Image.fromarray(a).save(out_path, format="JPEG", quality=95)
    return True

member_records, nonmember_records = [], []

print("\n🚀 开始从 h5 抽取 10k 样本并写入 Drive ...")
for p in [member_json_path, nonmember_json_path]:
    if os.path.exists(p):
        print(f"⚠️ 检测到已有 {p}，将覆盖重写")

# ✅ 不刷屏：每 5 秒刷新一次；每 50 step 才强制刷新
pbar = tqdm(
    total=TARGET_MEMBER + TARGET_NONMEMBER,
    desc="FashionGen -> member/nonmember",
    mininterval=5.0,
    miniters=50
)

global_idx = 0
bad = 0

sources = [train_h5_path, val_h5_path]

# ✅ 关键优化：不创建巨大的 indices=list(range(N)) 再 shuffle
# 用“随机分块扫描”：每次随机抽一批 index，直到凑够 10k
BATCH_SCAN = 20000  # 每次随机扫描多少条（可调 10k~50k，越大越随机但越慢一点）

for h5_path in sources:
    if len(member_records) >= TARGET_MEMBER and len(nonmember_records) >= TARGET_NONMEMBER:
        break

    f, imgs, txts, N = open_h5_and_get_handles(h5_path)

    scanned = 0
    while scanned < N:
        if len(member_records) >= TARGET_MEMBER and len(nonmember_records) >= TARGET_NONMEMBER:
            break

        # 从 [0, N) 里随机抽一批 index（无放回），避免全量 permutation
        batch = min(BATCH_SCAN, N - scanned)
        idxs = np.random.choice(N, size=batch, replace=False)

        for i in idxs:
            if len(member_records) >= TARGET_MEMBER and len(nonmember_records) >= TARGET_NONMEMBER:
                break

            try:
                text = decode_text(txts[int(i)])
                if (not text) or (len(text) < 10):
                    bad += 1
                    continue

                arr = imgs[int(i)]
                if arr is None:
                    bad += 1
                    continue

                bucket = assign_bucket(member_records, nonmember_records)
                if bucket == "member":
                    img_dir = member_img_dir
                    out_list = member_records
                else:
                    img_dir = nonmember_img_dir
                    out_list = nonmember_records

                img_filename = f"{global_idx:08d}.jpg"
                abs_img_path = os.path.join(img_dir, img_filename)
                save_image_array(arr, abs_img_path)

                out_list.append({
                    "messages": [
                        {"role": "user", "content": PROMPT_USER_EN},
                        {"role": "assistant", "content": text},
                    ],
                    "images": [os.path.join("images", img_filename)]
                })

                global_idx += 1
                pbar.update(1)

                if global_idx % 400 == 0:
                    time.sleep(0.02)

            except Exception:
                bad += 1
                continue

        scanned += batch

    f.close()

pbar.close()
print(f"✅ Done. member={len(member_records)}, nonmember={len(nonmember_records)}, bad/skip={bad}")

# 写 JSON
with open(member_json_path, "w", encoding="utf-8") as f:
    json.dump(member_records, f, ensure_ascii=False, indent=2)
with open(nonmember_json_path, "w", encoding="utf-8") as f:
    json.dump(nonmember_records, f, ensure_ascii=False, indent=2)

# eval_1000
member_eval = random.sample(member_records, min(1000, len(member_records)))
nonmember_eval = random.sample(nonmember_records, min(1000, len(nonmember_records)))

with open(os.path.join(dst_root, "member_eval_1000.json"), "w", encoding="utf-8") as f:
    json.dump(member_eval, f, ensure_ascii=False, indent=2)
with open(os.path.join(dst_root, "nonmember_eval_1000.json"), "w", encoding="utf-8") as f:
    json.dump(nonmember_eval, f, ensure_ascii=False, indent=2)

print("🎉 FashionGen member/nonmember 数据准备完成")
print(f"  - member:    {member_root}")
print(f"  - nonmember: {nonmember_root}")
print(f"  - eval:      {dst_root}/member_eval_1000.json, {dst_root}/nonmember_eval_1000.json")


In [ ]:
# ============================================================
# Part 0: 环境准备 + 模型下载（2B + 7B）【稳定可重复运行版】
# 说明：这一段每次新会话可直接运行，不会因 Drive 状态报错
# ============================================================

# @title 1. Environment Setup & Model Download (Safe Version)

import os
from google.colab import drive

print("=" * 80)
print("🚀 Part 0: 环境准备与模型下载（Safe Version）")
print("=" * 80)

# ============================================================
# 0.1 挂载 Google Drive（防炸版本）
# ============================================================
print("\n📂 挂载 Google Drive...")

try:
    # 如果之前残留了目录或文件，强制重新挂载
    drive.mount("/content/drive", force_remount=True)
except Exception as e:
    print("⚠️ force_remount 失败，尝试清理后重新挂载:", e)
    !rm -rf /content/drive
    drive.mount("/content/drive")

print("✅ Drive 已挂载")

# ============================================================
# 0.2 登录 Hugging Face（一次即可，token 会缓存）
# ============================================================
print("\n🔐 登录 Hugging Face（若已登录可直接回车）")
!huggingface-cli login

# ============================================================
# 0.3 安装系统依赖
# ============================================================
print("\n📦 安装系统依赖...")
!apt-get -qq update
!apt-get -y install git-lfs > /dev/null 2>&1
!git lfs install > /dev/null 2>&1
print("✅ 系统依赖完成")

# ============================================================
# 0.4 克隆 LLaMA-Factory（每次保证干净）
# ============================================================
print("\n📥 克隆 LLaMA-Factory...")
%cd /content
if os.path.exists("/content/LLaMA-Factory"):
    !rm -rf /content/LLaMA-Factory
!git clone https://github.com/hiyouga/LLaMA-Factory.git > /dev/null 2>&1
print("✅ LLaMA-Factory 克隆完成")

# ============================================================
# 0.5 安装 Python 依赖（锁版本，避免不兼容）
# ============================================================
print("\n📦 安装 Python 依赖...")
!pip install -q -U pip
!pip install -q \
    "transformers==4.52.4" \
    "tokenizers==0.21.1" \
    "accelerate==1.7.0" \
    "peft==0.15.2" \
    "datasets>=2.19.0" \
    "tqdm" \
    "Pillow" \
    "scikit-learn" \
    "matplotlib" \
    "seaborn" \
    "pandas"
!pip install -q "llamafactory==0.9.3"
print("✅ Python 依赖完成")

# ============================================================
# 0.6 下载 Qwen2-VL-2B
# ============================================================
print("\n📥 下载 Qwen2-VL-2B...")
if os.path.exists("/content/Qwen2-VL-2B-Instruct/config.json"):
    print("✅ Qwen2-VL-2B 已存在，跳过下载")
else:
    %cd /content
    !git clone https://www.modelscope.cn/Qwen/Qwen2-VL-2B-Instruct.git
    if not os.path.exists("/content/Qwen2-VL-2B-Instruct/config.json"):
        print("⚠️ ModelScope 失败，改用 HuggingFace")
        !git clone https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct
    print("✅ Qwen2-VL-2B 下载完成")


# ============================================================
# 0.8 检查 GPU
# ============================================================
print("\n🖥️ GPU 信息:")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

# ============================================================
# 0.9 缓存目录
# ============================================================
os.makedirs("/content/hf_cache", exist_ok=True)

print("\n" + "=" * 80)
print("✅ 环境准备完成（Safe Version）")
print("=" * 80)


In [ ]:
# 强制重装兼容版本的 Numpy
!pip install "numpy<2.0" --force-reinstall

In [ ]:
# 1. 先彻底删除可能存在的冲突文件夹
!rm -rf /content/drive

# 2. 重新挂载
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 1. 挂载 Google Drive (每次重连 Colab 都必须做这一步)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. (可选) 验证文件是否存在，确保挂载成功
import os
target_file = "/content/drive/MyDrive/FashionGen_MIA/member_5k/mllm_data.json"

if os.path.exists(target_file):
    print(f"✅ Drive 挂载成功，文件存在: {target_file}")
else:
    print(f"❌ 错误: 找不到文件。请检查 Drive 路径或账号是否正确。")
    print(f"   尝试访问的路径: {target_file}")

In [ ]:
# @title 2. Lora & GradAudit

import os, json, random, subprocess, sys, re, gc, shutil, warnings
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional
from tqdm import tqdm

import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve

from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

warnings.filterwarnings("ignore")

# ============================================================
# 🔧 单一实验配置 (FashionGen 1k)
# ============================================================

config = {
    "id": "fashiongen_1k_2b",   # 实验ID（仅名字变更，流程不变）
    "train_size": 1000,         # 训练样本数
    "model": "2B",              # 模型大小
    "ref_size": 200             # MIA 参考集大小
}

print("=" * 80)
print(f"🔬 实验启动: {config['id']}")
print(f"  训练数据: {config['train_size']}")
print(f"  模型大小: {config['model']}")
print(f"  参考数据集: {config['ref_size']}")
print("=" * 80)

# ============================================================
# 📂 路径配置 (FashionGen_MIA)
# ============================================================

LLAMA_REPO = "/content/LLaMA-Factory"
BASE_MODEL_2B = "/content/Qwen2-VL-2B-Instruct"
CACHE_DIR = "/content/hf_cache"

# Drive 根目录 (FashionGen 项目)
DRIVE_ROOT = "/content/drive/MyDrive/FashionGen_MIA"
MEMBER_5K_ROOT = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_5K_ROOT = f"{DRIVE_ROOT}/nonmember_5k"

# 数据集 JSON 路径
MEMBER_5K_JSON = f"{MEMBER_5K_ROOT}/mllm_data.json"
NONMEMBER_5K_JSON = f"{NONMEMBER_5K_ROOT}/mllm_data.json"

# Colab 本地工作区 (提高 I/O 速度)
COLAB_WORK_DIR = "/content/experiment_workspace"
os.makedirs(COLAB_WORK_DIR, exist_ok=True)

# GradAudit 参数
SENSITIVITY_TAU = 0.10

# 梯度策略
LAYER_CONFIGS = [
    {"name": "only_lora", "mode": "lora_only"},          # 仅 LoRA
    {"name": "lora_plus_last3", "mode": "lora_last3"},   # LoRA + Vision Last 3 + Text Last 3
]

# ============================================================
# ✅ 强制：训练 exposure 严格对齐 1k（防止 max_samples 触发重复采样）
# ============================================================
TRAIN_MAX_SAMPLES = int(config["train_size"])  # ✅ 改为 1000
TRAIN_EPOCHS = 5.0
SAVE_TOTAL_LIMIT = 30  # ✅ 对齐你“原始正确脚本”的保存上限（便于回溯）

# ============================================================
# ✅ 训练复现种子（重要：要同时喂给 Python / numpy / torch）
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# ✅ FashionGen Prompt（默认 prompt）
# - 数据 JSON 里已有 prompt，但为了保证一致性，这里也提供一个默认 prompt
# - 进入 processor 前会自动 strip 掉 prompt 里的 "<image>"，避免双 image token
# ============================================================
PROMPT_USER_EN = "As a fashion vision-language model, please describe the product in the image with detailed, factual attributes."

# ============================================================
# ✅ 环境“止血贴”（对齐你原始正确训练脚本的稳定配置）
# ============================================================
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
os.environ["MPLBACKEND"] = "Agg"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # 调试用，正常训练可注释提高速度

# ============================================================
# Section 1: 数据准备
# ============================================================

print("\n" + "=" * 80)
print("📊 Section 1: 数据准备")
print("=" * 80)

def _assert_dataset_format(sample: Dict[str, Any], name: str):
    assert "images" in sample and isinstance(sample["images"], list) and len(sample["images"]) > 0, f"{name}: 缺少 images"
    assert "messages" in sample and isinstance(sample["messages"], list) and len(sample["messages"]) >= 2, f"{name}: 缺少 messages"
    assert "role" in sample["messages"][0] and "content" in sample["messages"][0], f"{name}: messages[0] 非 sharegpt"
    assert "role" in sample["messages"][1] and "content" in sample["messages"][1], f"{name}: messages[1] 非 sharegpt"

def prepare_training_data(train_size: int, config_id: str):
    """准备训练数据（Colab 本地）"""
    print(f"  📖 读取 Member 数据: {MEMBER_5K_JSON}")
    with open(MEMBER_5K_JSON, "r", encoding="utf-8") as f:
        all_members = json.load(f)

    print(f"  📖 读取 Non-Member 数据: {NONMEMBER_5K_JSON}")
    with open(NONMEMBER_5K_JSON, "r", encoding="utf-8") as f:
        all_nonmembers = json.load(f)

    # ✅ 固定随机种子进行打乱（保持你原逻辑）
    config_seed = SEED + 123
    random.Random(config_seed).shuffle(all_members)
    random.Random(config_seed).shuffle(all_nonmembers)

    # ✅ 切分数据：训练集 = 前 train_size
    train_data = all_members[:train_size]

    # ✅ MIA：从训练集中采样 1000 个 Member（若 train_size<1000 则取全部），Non-member 取前 1000
    mia_members = random.Random(config_seed).sample(train_data, min(1000, len(train_data)))
    mia_nonmembers = all_nonmembers[:1000]

    # ✅ 基本格式检查（防止 silently wrong）
    _assert_dataset_format(train_data[0], "train_data[0]")
    _assert_dataset_format(mia_members[0], "mia_members[0]")
    _assert_dataset_format(mia_nonmembers[0], "mia_nonmembers[0]")

    # 保存到 Colab 本地
    work_dir = f"{COLAB_WORK_DIR}/{config_id}"
    os.makedirs(f"{work_dir}/train_data", exist_ok=True)
    os.makedirs(f"{work_dir}/mia_data", exist_ok=True)

    train_json = f"{work_dir}/train_data/mllm_data.json"
    with open(train_json, "w", encoding="utf-8") as f:
        json.dump(train_data, f, ensure_ascii=False)

    mia_member_json = f"{work_dir}/mia_data/member_eval.json"
    with open(mia_member_json, "w", encoding="utf-8") as f:
        json.dump(mia_members, f, ensure_ascii=False)

    mia_nonmember_json = f"{work_dir}/mia_data/nonmember_eval.json"
    with open(mia_nonmember_json, "w", encoding="utf-8") as f:
        json.dump(mia_nonmembers, f, ensure_ascii=False)

    print(f"✅ 数据准备完成: Train={len(train_data)}, MIA_M={len(mia_members)}, MIA_N={len(mia_nonmembers)}")
    return {
        "train_json": train_json,
        "mia_member_json": mia_member_json,
        "mia_nonmember_json": mia_nonmember_json,
    }

data_paths = prepare_training_data(config["train_size"], config["id"])

# ============================================================
# Section 2: LoRA 训练（FIXED: 严格 1k exposure + 稳定续训机制）
# ============================================================

print("\n" + "=" * 80)
print("🚀 Section 2: LoRA 训练 (FIXED)")
print("=" * 80)

def register_dataset(config_id: str, train_json: str):
    """注册数据集到 LLaMA-Factory"""
    assert os.path.isdir(LLAMA_REPO), f"❌ LLaMA-Factory 不存在: {LLAMA_REPO}"
    os.chdir(LLAMA_REPO)
    dataset_info_path = "data/dataset_info.json"

    with open(dataset_info_path, "r", encoding="utf-8") as f:
        info = json.load(f)

    dataset_key = f"fashiongen_{config_id}"
    info[dataset_key] = {
        "file_name": train_json,
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": {"role_tag": "role", "content_tag": "content", "user_tag": "user", "assistant_tag": "assistant"}
    }

    with open(dataset_info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)

    print(f"✅ dataset_info.json 已注册: {dataset_key}")
    return dataset_key

def _latest_ckpt(root: str) -> Optional[str]:
    """找最近 checkpoint（用于自动续训）"""
    p = Path(root)
    if not p.exists():
        return None
    cands = []
    for d in p.glob("checkpoint-*"):
        m = re.search(r"checkpoint-(\d+)$", d.name)
        if m:
            cands.append((int(m.group(1)), str(d)))
    if not cands:
        return None
    cands.sort(key=lambda x: x[0], reverse=True)
    return cands[0][1]

def train_lora_model(config: Dict[str, Any], train_json: str):
    """训练 LoRA 模型（只保存 adapter 到 Drive；FIXED：严格 1k exposure）"""
    config_id = config["id"]
    base_model = BASE_MODEL_2B

    # 训练输出到 Colab 本地 (速度快)
    output_dir_local = f"{COLAB_WORK_DIR}/{config_id}/lora_model"
    log_path = f"{COLAB_WORK_DIR}/{config_id}/train.log"

    # Drive 最终存储位置（仅 adapter）
    drive_adapter_dir = f"{DRIVE_ROOT}/adapters/{config_id}"

    # 检查 Drive 是否已有 adapter，有则跳过训练
    if os.path.exists(f"{drive_adapter_dir}/adapter_config.json") and (
        os.path.exists(f"{drive_adapter_dir}/adapter_model.safetensors") or os.path.exists(f"{drive_adapter_dir}/adapter_model.bin")
    ):
        print(f"✅ 发现 Drive 中已有 adapter，跳过训练: {drive_adapter_dir}")
        return {"base_model": base_model, "lora_dir": drive_adapter_dir, "log_path": None}

    dataset_key = register_dataset(config_id, train_json)

    # ✅ 续训逻辑（对齐你“原始正确脚本”的行为）
    resume_dir = _latest_ckpt(output_dir_local)
    if resume_dir:
        print(f"🔁 将从最近的 checkpoint 续训：{resume_dir}")
    else:
        print("🔰 没有已存在的 checkpoint，本次从头训练。")
        if os.path.isdir(output_dir_local):
            shutil.rmtree(output_dir_local)

    # ✅ 核心修正 1：max_samples = train_size，避免重复采样导致 loss-based MIA 异常变强
    # ✅ 核心修正 2：save_total_limit 提高，便于回溯
    # ✅ 核心修正 3：保留与原脚本一致的一系列稳定选项
    script = f"""#!/usr/bin/env bash
set -euo pipefail
cd "{LLAMA_REPO}"

python -m llamafactory.cli train \\
  --stage sft \\
  --do_train \\
  --model_name_or_path "{base_model}" \\
  --cache_dir "{CACHE_DIR}" \\
  --trust_remote_code \\
  --finetuning_type lora \\
  --template qwen2_vl \\
  --media_dir "{MEMBER_5K_ROOT}" \\
  --dataset_dir data \\
  --dataset {dataset_key} \\
  --cutoff_len 2048 \\
  --learning_rate 1e-4 \\
  --num_train_epochs {TRAIN_EPOCHS} \\
  --max_samples {TRAIN_MAX_SAMPLES} \\
  --per_device_train_batch_size 4 \\
  --gradient_accumulation_steps 4 \\
  --lr_scheduler_type cosine \\
  --max_grad_norm 1.0 \\
  --logging_steps 5 \\
  --save_strategy steps \\
  --save_steps 100 \\
  --save_total_limit {SAVE_TOTAL_LIMIT} \\
  --warmup_steps 0 \\
  --optim adamw_torch \\
  --report_to none \\
  --output_dir "{output_dir_local}" \\
  --overwrite_output_dir \\
  --bf16 \\
  --plot_loss False \\
  --flash_attn sdpa \\
  --dataloader_num_workers 0 \\
  --no_dataloader_pin_memory \\
  --torch_empty_cache_steps 20 \\
  --lora_rank 64 \\
  --lora_alpha 128 \\
  --lora_dropout 0.0 \\
  --lora_target all \\
  --gradient_checkpointing \\
  {"--resume_from_checkpoint " + resume_dir if resume_dir else ""}
"""

    sh_path = f"/content/run_lora_{config_id}.sh"
    Path(sh_path).write_text(script)
    os.chmod(sh_path, 0o755)

    print("🚀 调用 LLaMA-Factory 开始训练...")
    with open(log_path, "wb") as logf:
        proc = subprocess.Popen(
            ["bash", "-lc", sh_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1
        )
        for line in iter(proc.stdout.readline, b""):
            logf.write(line)
            try:
                print(line.decode("utf-8"), end="")
            except:
                pass
        proc.wait()
        code = proc.returncode

    print(f"\n🧾 Log saved to: {log_path}")
    if code != 0:
        print(f"\n❌ 训练失败，查看日志: {log_path}")
        raise SystemExit(code)

    # 找最佳 adapter（按 checkpoint step 最大）
    def find_best_adapter(root: str) -> Optional[str]:
        hits = []
        for p in Path(root).rglob("adapter_config.json"):
            d = str(p.parent)
            m = re.search(r"checkpoint[-_]?(\d+)", d)
            step = int(m.group(1)) if m else -1
            hits.append((step, d))
        if not hits:
            if (Path(root) / "adapter_config.json").exists():
                return root
            return None
        return sorted(hits, key=lambda x: x[0], reverse=True)[0][1]

    best_adapter_local = find_best_adapter(output_dir_local)
    if not best_adapter_local:
        raise RuntimeError("❌ 训练完成但未找到 adapter_config.json（请查看 train.log 尾部）")

    print(f"\n✅ 训练完成，最佳权重目录: {best_adapter_local}")

    # 📤 保存到 Drive（仅 adapter 文件）
    print(f"📤 保存 adapter 到 Drive: {drive_adapter_dir}")
    os.makedirs(drive_adapter_dir, exist_ok=True)

    for filename in ["adapter_config.json", "adapter_model.safetensors", "adapter_model.bin"]:
        src = os.path.join(best_adapter_local, filename)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(drive_adapter_dir, filename))
            print(f"  - Copied {filename}")

    # ✅ 不强制删整个 output_dir_local（保留 checkpoint 便于排查/复现）
    # 如果你一定要清理，把下面两行取消注释即可：
    # print("🧹 清理 Colab 本地训练文件...")
    # shutil.rmtree(output_dir_local, ignore_errors=True)

    torch.cuda.empty_cache()
    gc.collect()

    return {"base_model": base_model, "lora_dir": drive_adapter_dir, "log_path": log_path}

model_info = train_lora_model(config, data_paths["train_json"])

# ============================================================
# Section 3: 模型加载工具
# ============================================================

def load_model_with_lora(base_model_path: str, lora_dir: str):
    print(f"  🔧 加载基座: {base_model_path}")
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        base_model_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    print(f"  🔧 挂载 LoRA: {lora_dir}")
    model = PeftModel.from_pretrained(model, lora_dir, is_trainable=False)
    model.eval()
    return model

# ============================================================
# Section 4: GradAudit MIA (Attack)
# ============================================================

print("\n" + "=" * 80)
print("🔬 Section 4: GradAudit MIA 攻击")
print("=" * 80)

def mark_trainable_params(model, mode: str):
    for p in model.parameters():
        p.requires_grad_(False)

    selected = []

    if mode == "lora_only":
        for name, param in model.named_parameters():
            if "lora" in name.lower() and param.ndim >= 2:
                param.requires_grad_(True)
                selected.append(name)

    elif mode == "lora_last3":
        text_indices, vision_indices = set(), set()
        for name, _ in model.named_parameters():
            if "model.layers." in name:
                m = re.search(r"model\.layers\.(\d+)\.", name)
                if m:
                    text_indices.add(int(m.group(1)))
            if "visual.blocks." in name:
                m = re.search(r"visual\.blocks\.(\d+)\.", name)
                if m:
                    vision_indices.add(int(m.group(1)))

        max_text = max(text_indices) if text_indices else 0
        max_vision = max(vision_indices) if vision_indices else 0
        text_start = max(0, max_text - 2)
        vision_start = max(0, max_vision - 2)

        print(f"  ℹ️ Layers: Text [{text_start}-{max_text}], Vision [{vision_start}-{max_vision}]")

        for name, param in model.named_parameters():
            if param.ndim < 2:
                continue
            enable = False
            if "lora" in name.lower():
                enable = True
            else:
                t_match = re.search(r"model\.layers\.(\d+)\.", name)
                if t_match and int(t_match.group(1)) >= text_start:
                    enable = True
                v_match = re.search(r"visual\.blocks\.(\d+)\.", name)
                if v_match and int(v_match.group(1)) >= vision_start:
                    enable = True

            if enable:
                param.requires_grad_(True)
                selected.append(name)

    return selected

# ✅ 关键：保证 prompt 不含 <image>，避免双 image token
_IMAGE_TOKEN_RE = re.compile(r"<\s*image\s*>", flags=re.IGNORECASE)

def _strip_image_token_from_prompt(p: str) -> str:
    if not isinstance(p, str):
        return ""
    p = _IMAGE_TOKEN_RE.sub("", p)
    p = p.replace("\n", " ").strip()
    p = re.sub(r"\s+", " ", p).strip()
    return p

def _get_user_prompt_from_record(item: Dict[str, Any]) -> str:
    try:
        p = item["messages"][0]["content"]
        if isinstance(p, str) and p.strip():
            p = _strip_image_token_from_prompt(p)
            return p if p else PROMPT_USER_EN
    except:
        pass
    return PROMPT_USER_EN

def _get_assistant_text_from_record(item: Dict[str, Any]) -> str:
    t = item["messages"][-1]["content"]
    if isinstance(t, list):
        parts = []
        for seg in t:
            if isinstance(seg, dict) and seg.get("text"):
                parts.append(seg["text"])
            elif isinstance(seg, str):
                parts.append(seg)
        t = " ".join(parts)
    return str(t)

def backward_and_collect(image, user_prompt: str, target_text: str, model, processor):
    """计算单个样本梯度（保持你原始的“全序列 labels=inputs”设计不变）"""
    try:
        messages = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_prompt}]},
            {"role": "assistant", "content": [{"type": "text", "text": target_text}]}
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

        labels = inputs["input_ids"].clone()

        model.train()
        model.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss

        loss.backward()

        grad_dict = {}
        for n, p in model.named_parameters():
            if p.requires_grad and p.grad is not None:
                grad_dict[n] = p.grad.detach().float().cpu()

        model.zero_grad()
        model.eval()
        return grad_dict

    except:
        model.zero_grad()
        model.eval()
        return {}

def gradaudit_attack(layer_config: Dict[str, str]):
    print(f"\n⚡ 运行策略: {layer_config['name']}")

    base_model = model_info["base_model"]
    lora_dir = model_info["lora_dir"]

    processor = AutoProcessor.from_pretrained(base_model, trust_remote_code=True)
    model = load_model_with_lora(base_model, lora_dir)

    selected_params = mark_trainable_params(model, layer_config["mode"])
    print(f"  参数数量: {len(selected_params)}")
    if not selected_params:
        return None

    with open(data_paths["mia_member_json"], "r", encoding="utf-8") as f:
        members = json.load(f)
    with open(data_paths["mia_nonmember_json"], "r", encoding="utf-8") as f:
        nonmembers = json.load(f)

    ref_size = config["ref_size"]
    ref_members = members[:ref_size]
    ref_nonmembers = nonmembers[:ref_size]
    probe_members = members[ref_size:]
    probe_nonmembers = nonmembers[ref_size:ref_size + len(probe_members)]

    print("  Step 1: 计算参考梯度...")
    ref_grads = {}
    ref_count = 0

    for item in tqdm(ref_members, desc="Ref Grads"):
        try:
            img_path = f"{MEMBER_5K_ROOT}/{item['images'][0]}"
            image = Image.open(img_path).convert("RGB")

            user_prompt = _get_user_prompt_from_record(item)
            target = _get_assistant_text_from_record(item)

            gd = backward_and_collect(image, user_prompt, target, model, processor)
            if gd:
                ref_count += 1
                for k, v in gd.items():
                    if k not in ref_grads:
                        ref_grads[k] = v
                    else:
                        ref_grads[k] += v
        except:
            continue

    for k in ref_grads:
        ref_grads[k] /= max(1, ref_count)

    def get_sims(samples, root):
        r_acc, c_acc = {}, {}
        cnt = 0
        for item in tqdm(samples, desc="Sims", leave=False):
            try:
                img_path = f"{root}/{item['images'][0]}"
                image = Image.open(img_path).convert("RGB")

                user_prompt = _get_user_prompt_from_record(item)
                target = _get_assistant_text_from_record(item)

                gd = backward_and_collect(image, user_prompt, target, model, processor)
                if not gd:
                    continue

                cnt += 1
                for n, g in gd.items():
                    if n in ref_grads:
                        if g.ndim < 2:
                            g = g.unsqueeze(0)
                            rg = ref_grads[n].unsqueeze(0)
                        else:
                            rg = ref_grads[n]

                        row_sim = F.cosine_similarity(g, rg, dim=1)
                        col_sim = F.cosine_similarity(g.T, rg.T, dim=1)

                        r_acc[n] = r_acc.get(n, 0) + row_sim
                        c_acc[n] = c_acc.get(n, 0) + col_sim
            except:
                continue

        for k in r_acc:
            r_acc[k] /= max(1, cnt)
        for k in c_acc:
            c_acc[k] /= max(1, cnt)
        return r_acc, c_acc

    print("  Step 2: 筛选敏感神经元...")
    mr, mc = get_sims(ref_members, MEMBER_5K_ROOT)
    nr, nc = get_sims(ref_nonmembers, NONMEMBER_5K_ROOT)

    row_mask, col_mask = {}, {}
    total_sens = 0
    for n in mr:
        if n in nr:
            r_gap = torch.nan_to_num(mr[n] - nr[n], 0)
            c_gap = torch.nan_to_num(mc[n] - nc[n], 0)
            rm = r_gap > SENSITIVITY_TAU
            cm = c_gap > SENSITIVITY_TAU
            if rm.any() or cm.any():
                row_mask[n] = rm
                col_mask[n] = cm
                total_sens += (rm.sum() + cm.sum()).item()

    print(f"  敏感维度数: {int(total_sens)}")

    print("  Step 3: 对 Probe 数据打分...")

    def get_score(item, root):
        try:
            img_path = f"{root}/{item['images'][0]}"
            image = Image.open(img_path).convert("RGB")

            user_prompt = _get_user_prompt_from_record(item)
            target = _get_assistant_text_from_record(item)

            gd = backward_and_collect(image, user_prompt, target, model, processor)
            if not gd:
                return 0.0

            scores = []
            for n, g in gd.items():
                if n in row_mask:
                    if g.ndim < 2:
                        g = g.unsqueeze(0)
                        rg = ref_grads[n].unsqueeze(0)
                    else:
                        rg = ref_grads[n]

                    rs = F.cosine_similarity(g, rg, dim=1)
                    cs = F.cosine_similarity(g.T, rg.T, dim=1)

                    scores.extend(rs[row_mask[n]].tolist())
                    scores.extend(cs[col_mask[n]].tolist())

            return float(np.mean(scores)) if scores else 0.0
        except:
            return 0.0

    final_scores, final_labels = [], []
    for item in tqdm(probe_members, desc="Probe Member"):
        final_scores.append(get_score(item, MEMBER_5K_ROOT))
        final_labels.append(1)
    for item in tqdm(probe_nonmembers, desc="Probe NonMember"):
        final_scores.append(get_score(item, NONMEMBER_5K_ROOT))
        final_labels.append(0)

    auc_val = roc_auc_score(final_labels, final_scores)
    fpr, tpr, _ = roc_curve(final_labels, final_scores)
    idx = np.abs(fpr - 0.05).argmin()
    tpr_5 = tpr[idx]

    print(f"  🏆 AUC: {auc_val:.4f}")
    print(f"  🏆 TPR@5%FPR: {tpr_5:.4f}")

    del model, processor
    torch.cuda.empty_cache()
    gc.collect()

    return {"auc": auc_val, "tpr_5": tpr_5, "sens": total_sens}

# ============================================================
# 执行所有梯度策略
# ============================================================

final_results = []
for layer_conf in LAYER_CONFIGS:
    res = gradaudit_attack(layer_conf)
    if res:
        final_results.append({"config": config["id"], "strategy": layer_conf["name"], **res})

# ============================================================
# 保存结果
# ============================================================

save_path = f"{DRIVE_ROOT}/results/{config['id']}_results.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(final_results, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 80)
print(f"💾 实验结束，结果已保存: {save_path}")
print("=" * 80)


In [ ]:
# ==============================================================================
# Min-K% Sensitivity Analysis - FashionGen (Strict Training Split)
# ✅ 严格复刻训练脚本随机种子与切分：确保取到真正参与微调的那 1000 条 member
# ✅ 复刻“原始 Min-K 逻辑”：只在 assistant 段计算 NLL（labels 前缀置 -100）
# ✅ Min-K% = 取“最小 NLL”的 k% token 平均（不是最大 NLL）
# ✅ 适配 FashionGen_MIA sharegpt 格式：messages + images
# ✅ 防止双 image token：自动 strip prompt 中的 "<image>"
# @title 3. Min-K% Analysis
# ==============================================================================

import subprocess, sys, os, json, warnings, random, re, math
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

# ----------------------------
# 0) 安装最少依赖（可选）
# ----------------------------
def install_package(pkg):
    try:
        __import__(pkg)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_package("peft")
warnings.filterwarnings("ignore")

# ============================================================
# 1) 路径配置（✅ FashionGen）
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive/FashionGen_MIA"
BASE_MODEL = "/content/Qwen2-VL-2B-Instruct"
LORA_PATH  = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"

MEMBER_DIR = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_DIR = f"{DRIVE_ROOT}/nonmember_5k"
MEMBERS_FULL_JSON = f"{MEMBER_DIR}/mllm_data.json"
NONMEMBERS_FULL_JSON = f"{NONMEMBER_DIR}/mllm_data.json"

OUTPUT_DIR = f"{DRIVE_ROOT}/results/mink_grid_search_fashiongen_1k_2b_strict_minK"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 🔍 配置（与训练保持一致）
K_GRID = [0.1]
TRAIN_SIZE = 1000
SEED = 42
CONFIG_SEED = SEED + 123  # 训练脚本：config_seed = SEED + 123

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# 2) 数据复现（严格复刻训练 split）
# ============================================================
print("🔄 Reconstructing exact training split (FashionGen)...")

def get_exact_training_data():
    with open(MEMBERS_FULL_JSON, "r", encoding="utf-8") as f:
        all_members = json.load(f)

    with open(NONMEMBERS_FULL_JSON, "r", encoding="utf-8") as f:
        all_nonmembers = json.load(f)

    random.Random(CONFIG_SEED).shuffle(all_members)
    random.Random(CONFIG_SEED).shuffle(all_nonmembers)

    real_members = all_members[:TRAIN_SIZE]
    real_nonmembers = all_nonmembers[:TRAIN_SIZE]

    print(f"  ✅ Extracted {len(real_members)} Members (Training Set)")
    print(f"  ✅ Extracted {len(real_nonmembers)} Non-Members (Held-out Set)")
    return real_members, real_nonmembers

m_data, n_data = get_exact_training_data()

# ============================================================
# 3) 模型加载（Qwen2-VL-2B + LoRA）
# ============================================================
print(f"\n🚀 Loading FashionGen model + LoRA: {LORA_PATH}")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16

processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True
)

try:
    model = PeftModel.from_pretrained(base_model, LORA_PATH, is_trainable=False)
    model.eval()
    print("✅ Model loaded successfully")
except Exception as e:
    print(f"❌ Failed to load LoRA: {e}")
    raise

# 统一拿一个 device（避免你手动 to(cuda:0) 引发不一致）
def _model_device(m):
    try:
        return next(m.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_DEVICE = _model_device(model)

# ============================================================
# 4) 辅助函数（适配 sharegpt + strip <image>）
# ============================================================

_IMAGE_TOKEN_RE = re.compile(r"<\s*image\s*>", flags=re.IGNORECASE)

def strip_image_token(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = _IMAGE_TOKEN_RE.sub("", s)
    s = s.replace("\n", " ").strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def robust_extract_text(c):
    if isinstance(c, str):
        return strip_image_token(c)
    if isinstance(c, list):
        parts = []
        for seg in c:
            if isinstance(seg, dict) and seg.get("type") == "text" and seg.get("text"):
                parts.append(seg["text"])
            elif isinstance(seg, str):
                parts.append(seg)
        return strip_image_token(" ".join(parts))
    return strip_image_token(str(c))

def load_img(root, rel_path):
    try:
        p = os.path.join(root, rel_path)
        if os.path.exists(p):
            return Image.open(p).convert("RGB")
        return None
    except Exception:
        return None

def build_msgs_from_record(item):
    """
    严格复刻原始 mink 的逻辑：messages -> 标准 chat content(list) 结构
    - user: [{"type":"image"},{"type":"text","text":prompt}]
    - assistant: [{"type":"text","text":answer}]
    """
    # prompt
    u = item["messages"][0]["content"]
    prompt = robust_extract_text(u)
    # answer
    a = item["messages"][1]["content"]
    answer = robust_extract_text(a)

    return [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]},
        {"role": "assistant", "content": [{"type": "text", "text": answer}]}
    ]

def pack_for_minK(image, msgs):
    """
    ✅ 复刻原始 _pack：
    - enc_p: user-only (add_generation_prompt=True) -> prefix_len
    - enc_f: full msgs (add_generation_prompt=False)
    - labels: 前缀置 -100，只在 assistant 段算 NLL
    """
    user_only = [msgs[0]]
    user_text = processor.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

    enc_p = processor(text=[user_text], images=[image], return_tensors="pt")
    enc_f = processor(text=[full_text], images=[image], return_tensors="pt")

    # move to model device (device_map="auto" safe-ish)
    enc_p = {k: v.to(MODEL_DEVICE) for k, v in enc_p.items()}
    enc_f = {k: v.to(MODEL_DEVICE) for k, v in enc_f.items()}

    input_ids = enc_f["input_ids"]
    labels = input_ids.clone()

    prefix_len = int(enc_p["input_ids"].size(1))
    if input_ids.size(1) <= prefix_len:
        return None

    labels[:, :prefix_len] = -100
    enc_f["labels"] = labels
    return enc_f, prefix_len

# ============================================================
# 5) Min-K% 计算（✅ 最小 NLL 的 k% token 平均）
# ============================================================

@torch.no_grad()
def mink_percent_scores(rec, root, k_list):
    """
    返回 dict[k] = score_k
    其中 score_k 定义为：-(mean of smallest k% token NLL on assistant)
    => 分数越大越像 member（与原始代码一致）
    """
    img_path = os.path.join(root, rec["images"][0])
    image = Image.open(img_path).convert("RGB")

    msgs = build_msgs_from_record(rec)
    packed = pack_for_minK(image, msgs)
    if packed is None:
        return None
    inputs, _ = packed

    # forward
    with torch.autocast(device_type="cuda", dtype=dtype, enabled=torch.cuda.is_available()):
        out = model(**inputs)

    logits = out.logits           # [B, T, V]
    labels = inputs["labels"]     # [B, T]

    # per-token NLL on assistant (same as original)
    sl = logits[..., :-1, :]      # predict next token
    tl = labels[..., 1:]          # next-token labels
    mask = tl.ne(-100)[0]
    if mask.sum().item() == 0:
        return None

    logp = F.log_softmax(sl[0].float(), dim=-1)
    gather = tl[0].clamp(min=0)
    tok_lp = logp[torch.arange(logp.size(0), device=logp.device), gather]  # [T-1]
    tok_nll = (-tok_lp)[mask]                                              # [N]

    if tok_nll.numel() == 0:
        return None

    tok_nll = tok_nll.detach()
    N = int(tok_nll.numel())

    results = {}
    for k in k_list:
        k_len = max(1, int(N * float(k)))
        # ✅ smallest NLL
        smallest, _ = torch.topk(tok_nll, k_len, largest=False)
        # score = -mean(smallest_nll)  => bigger = more member-like
        results[float(k)] = float((-smallest.mean()).item())

    return results

# ============================================================
# 6) 主处理循环
# ============================================================

final_data = []

def process(data, root, label):
    kept = 0
    for item in tqdm(data, desc=f"Label {label}"):
        try:
            if not item.get("images"):
                continue
            # quick image existence check
            img = load_img(root, item["images"][0])
            if img is None:
                continue

            scores = mink_percent_scores(item, root, K_GRID)
            if scores is None:
                continue

            final_data.append({"label": int(label), "scores": scores})
            kept += 1
        except Exception:
            continue
    print(f"  ✅ kept={kept} / {len(data)}")

print("\n🚀 Processing Training Members (Label 1)...")
process(m_data, MEMBER_DIR, 1)

print("\n🚀 Processing Non-Members (Label 0)...")
process(n_data, NONMEMBER_DIR, 0)

print(f"\n✅ Finished. Collected samples: {len(final_data)}")

# ============================================================
# 7) 结果分析 + 保存
# ============================================================

print("\n📊 Min-K% Grid Search Results (FashionGen Strict 1k, original logic):")
plt.figure(figsize=(10, 8))

y_true = np.array([d["label"] for d in final_data], dtype=np.int32)

for k in K_GRID:
    scores = np.array([d["scores"][float(k)] for d in final_data], dtype=np.float32)
    scores = np.nan_to_num(scores, nan=-100.0, posinf=100.0, neginf=-100.0)

    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc = auc(fpr, tpr)

    tpr5_idx = np.where(fpr <= 0.05)[0]
    tpr5 = float(tpr[tpr5_idx[-1]]) if len(tpr5_idx) > 0 else 0.0

    plt.plot(fpr, tpr, lw=2, label=f'k={int(k*100)}% (AUC={roc_auc:.4f})')
    print(f"  ✅ k={k:<4} | AUC: {roc_auc:.4f} | TPR@5%FPR: {tpr5*100:.2f}%")

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Min-K% MIA (FashionGen Strict 1k) - Original Min-K Logic")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

save_path = os.path.join(OUTPUT_DIR, "mink_percent_roc_strict_original_logic.png")
plt.savefig(save_path, dpi=300)
print(f"\n📈 Chart saved to: {save_path}")

json_path = os.path.join(OUTPUT_DIR, "mink_percent_scores_strict_original_logic.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=2)

print(f"🧾 Scores saved to: {json_path}")


In [ ]:
# ==============================================================================
# ModRényi* (Original-Logic Replica) - FashionGen (Strict Training Split)
# - Strictly reconstruct the exact 1k training member split (SEED=42, CONFIG_SEED=SEED+123)
# - Load Qwen2-VL-2B + LoRA adapter
# - Compute assistant-only logP matrix (token x vocab) via labels=-100 prefix masking
# - Multi-view:
#    * Image views: original + light augmentations (crop/rotate/affine/jitter)
#    * Text views : original answer + (lower, normalize spaces)
# - Per-view: Rényi entropy (alpha in {0.5, 2.0}) averaged over tokens
# - Robust aggregation (trim mean), optional length normalization (/sqrt(T))
# - Fuse parts with z-score mean (same spirit as original)
# @title 4. ModRényi Analysis
# ==============================================================================

import os, json, random, re, math, warnings
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt

from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

warnings.filterwarnings("ignore")
assert torch.cuda.is_available(), "Need GPU for this script."

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ============================================================
# 1) Paths (FashionGen)
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive/FashionGen_MIA"
BASE_MODEL = "/content/Qwen2-VL-2B-Instruct"
LORA_PATH  = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"

MEM_IMGROOT = f"{DRIVE_ROOT}/member_5k"
NON_IMGROOT = f"{DRIVE_ROOT}/nonmember_5k"
MEM_JSON    = f"{MEM_IMGROOT}/mllm_data.json"
NON_JSON    = f"{NON_IMGROOT}/mllm_data.json"

OUT_DIR = f"{DRIVE_ROOT}/results/modrenyi_star_fashiongen_strict_1k"
os.makedirs(OUT_DIR, exist_ok=True)

# Strict split config (must match training)
TRAIN_SIZE  = 1000
SEED        = 42
CONFIG_SEED = SEED + 123

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# 2) Reconstruct exact training split
# ============================================================

print("🔄 Reconstructing exact FashionGen training split...")

with open(MEM_JSON, "r", encoding="utf-8") as f:
    all_members = json.load(f)
with open(NON_JSON, "r", encoding="utf-8") as f:
    all_nonmembers = json.load(f)

random.Random(CONFIG_SEED).shuffle(all_members)
random.Random(CONFIG_SEED).shuffle(all_nonmembers)

members_1k    = all_members[:TRAIN_SIZE]
nonmembers_1k = all_nonmembers[:TRAIN_SIZE]

print(f"✅ members={len(members_1k)}  nonmembers={len(nonmembers_1k)}")

# ============================================================
# 3) Load Qwen2-VL-2B + LoRA
# ============================================================

dtype = torch.bfloat16
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True
)

print(f"🔧 Loading LoRA: {LORA_PATH}")
model = PeftModel.from_pretrained(base_model, LORA_PATH, is_trainable=False).eval()

def _model_device(m):
    try:
        return next(m.parameters()).device
    except StopIteration:
        return torch.device("cuda")

device = _model_device(model)
print(f"✅ device={device} dtype={next(iter(model.parameters())).dtype}")

# ============================================================
# 4) ModRényi* view configs (same spirit as original)
# ============================================================

IMG_VIEWS = 5
TXT_VIEWS = 3
RENYI_ALPHAS = (0.5, 2.0)
LENGTH_NORM  = True

# ---- text views ----
def normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def make_text_views(answer_text: str, want: int = TXT_VIEWS):
    base = str(answer_text or "").strip()
    cand = [base, base.lower(), normalize_spaces(base)]
    out, seen = [], set()
    for c in cand:
        if c not in seen:
            out.append(c); seen.add(c)
    return out[:max(1, want)]

# ---- image views ----
from torchvision.transforms import (
    RandomResizedCrop, RandomRotation, RandomAffine, ColorJitter, InterpolationMode
)

def make_image_views(pil: Image.Image, want: int = IMG_VIEWS):
    views = [pil]
    views.append(RandomResizedCrop(size=(256, 256), scale=(0.8, 1.0),
                                   interpolation=InterpolationMode.BICUBIC)(pil))
    views.append(RandomRotation(degrees=30, interpolation=InterpolationMode.BICUBIC, expand=False)(pil))
    views.append(RandomAffine(degrees=20, translate=(0.08, 0.08), scale=(0.9, 1.1),
                              interpolation=InterpolationMode.BICUBIC)(pil))
    views.append(ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25, hue=0.05)(pil))
    return views[:max(1, want)]

# ============================================================
# 5) ShareGPT helpers (FashionGen) + strip "<image>"
# ============================================================

_IMAGE_TOKEN_RE = re.compile(r"<\s*image\s*>", flags=re.IGNORECASE)

def strip_image_token(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = _IMAGE_TOKEN_RE.sub("", s)
    s = s.replace("\n", " ").strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def robust_extract_text(c):
    if isinstance(c, str):
        return strip_image_token(c)
    if isinstance(c, list):
        parts = []
        for seg in c:
            if isinstance(seg, dict) and seg.get("type") == "text" and seg.get("text"):
                parts.append(seg["text"])
            elif isinstance(seg, str):
                parts.append(seg)
        return strip_image_token(" ".join(parts))
    return strip_image_token(str(c))

def _extract_answer_text(rec):
    msgs = rec.get("messages", None)
    if not msgs or not isinstance(msgs, list) or len(msgs) < 2:
        return str(rec.get("caption", "")).strip()
    return robust_extract_text(msgs[1].get("content", ""))

def _build_msgs_with_answer(rec, answer_text: str):
    """
    Build:
      user:   [{"type":"image"},{"type":"text","text":prompt}]
      assistant: [{"type":"text","text":answer_text}]
    """
    msgs = rec.get("messages", None)
    if msgs and isinstance(msgs, list) and len(msgs) >= 2:
        u = msgs[0].get("content", "")
        prompt = robust_extract_text(u)
    else:
        prompt = "Please describe the product in the image with detailed, factual attributes."
    return [
        {"role": "user", "content": [{"type":"image"}, {"type":"text", "text": prompt}]},
        {"role": "assistant", "content": [{"type":"text", "text": str(answer_text)}]},
    ]

def _safe_open_image(path: str):
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

# ============================================================
# 6) Pack (prefix -100 masking) + assistant logP matrix [T, V]
# ============================================================

def _pack_for_qwen(image: Image.Image, msgs):
    user_only = [msgs[0]]
    user_text = processor.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

    enc_prefix = processor(text=[user_text], images=[image], return_tensors="pt")
    enc_full   = processor(text=[full_text], images=[image], return_tensors="pt")

    enc_prefix = {k: v.to(device) for k, v in enc_prefix.items()}
    enc_full   = {k: v.to(device) for k, v in enc_full.items()}

    input_ids  = enc_full["input_ids"]
    labels     = input_ids.clone()
    prefix_len = int(enc_prefix["input_ids"].size(1))
    if input_ids.size(1) <= prefix_len:
        return None

    labels[:, :prefix_len] = -100
    enc_full["labels"] = labels
    return enc_full

@torch.no_grad()
def assistant_logp_matrix_qwen(image: Image.Image, rec, answer_text: str):
    msgs = _build_msgs_with_answer(rec, answer_text)
    enc = _pack_for_qwen(image, msgs)
    if enc is None:
        return None

    with torch.autocast(device_type="cuda", dtype=dtype, enabled=True):
        out = model(**enc)

    logits = out.logits
    labels = enc["labels"]

    shift_logits = logits[..., :-1, :]
    shift_labels = labels[...,  1: ]
    mask = shift_labels.ne(-100)[0]
    if mask.sum().item() == 0:
        return None

    logp = F.log_softmax(shift_logits[0].float(), dim=-1)  # [L-1, V] float32
    sel = logp[mask]                                       # [T, V]
    if sel.numel() == 0:
        return None

    return sel.detach().cpu().numpy()

# ============================================================
# 7) Rényi entropy + robust aggregation + fusion (same as original)
# ============================================================

def renyi_entropy_from_logp(logp_tok: np.ndarray, alpha: float) -> float:
    # H_alpha(p) = 1/(1-alpha) * log sum_i p_i^alpha
    # logp_tok are log(p_i)
    a = alpha * logp_tok
    m = np.max(a)
    return float((1.0/(1.0-alpha)) * (m + np.log(np.exp(a - m).sum())))

def renyi_seq_from_logp(logp_seq: np.ndarray, alpha: float) -> float:
    vals = [renyi_entropy_from_logp(logp_seq[t], alpha) for t in range(logp_seq.shape[0])]
    return float(np.mean(vals)) if vals else float("nan")

def robust_aggregate(values, trim: float = 0.1) -> float:
    arr = np.array([v for v in values if np.isfinite(v)], dtype=float)
    if arr.size == 0:
        return float("nan")
    arr.sort()
    k = int(math.floor(trim * arr.size))
    if k * 2 < arr.size:
        arr = arr[k: arr.size - k]
    return float(np.mean(arr))

def fused_modrenyi_score(image_views_logp, text_views_logp,
                         alphas=RENYI_ALPHAS, length_norm=LENGTH_NORM):
    parts = []
    for views in [image_views_logp, text_views_logp]:
        vv = [x for x in views if x is not None]
        if not vv:
            parts.extend([float("nan")] * len(alphas))
            continue

        T = vv[0].shape[0]
        for a in alphas:
            per_view = []
            for lp in vv:
                if lp is None or not np.isfinite(lp).all():
                    continue
                H = renyi_seq_from_logp(lp, a)
                per_view.append(H)

            if not per_view:
                parts.append(float("nan"))
                continue

            H_bar = robust_aggregate(per_view, trim=0.1)
            if length_norm and T > 0:
                H_bar = H_bar / math.sqrt(T)

            parts.append(-H_bar)  # negative entropy => bigger = more member-like

    valid = np.array([x for x in parts if np.isfinite(x)], dtype=float)
    if valid.size == 0:
        return float("nan")

    if valid.size >= 2 and np.std(valid) > 1e-12:
        z = (valid - np.mean(valid)) / np.std(valid)
        return float(np.mean(z))
    else:
        return float(np.mean(valid))

# ============================================================
# 8) View adapters (same spirit as original)
# ============================================================

def image_views_logp_list_qwen(image: Image.Image, rec, answer_text: str):
    views = make_image_views(image, IMG_VIEWS)
    out = []
    for v in views:
        try:
            out.append(assistant_logp_matrix_qwen(v, rec, answer_text))
        except Exception:
            out.append(None)
    return out

def text_views_logp_list_qwen(image: Image.Image, rec, base_answer: str):
    texts = make_text_views(base_answer, TXT_VIEWS)
    out = []
    for t in texts:
        try:
            out.append(assistant_logp_matrix_qwen(image, rec, t))
        except Exception:
            out.append(None)
    return out

def fused_score_for_record(rec, source: str):
    rel = rec["images"][0] if rec.get("images") else None
    if not rel:
        return None

    root = MEM_IMGROOT if source == "member" else NON_IMGROOT
    img_path = os.path.join(root, rel)
    image = _safe_open_image(img_path)
    if image is None:
        return None

    base_answer = _extract_answer_text(rec)
    if not str(base_answer).strip():
        return None

    img_views = image_views_logp_list_qwen(image, rec, base_answer)
    txt_views = text_views_logp_list_qwen(image, rec, base_answer)

    s = fused_modrenyi_score(img_views, txt_views, alphas=RENYI_ALPHAS, length_norm=LENGTH_NORM)
    if not math.isfinite(s):
        return None
    return s

# ============================================================
# 9) Run + Evaluate
# ============================================================

print("\n⏳ Computing ModRényi* fused scores (FashionGen strict 1k)...")
mem_scores, non_scores = [], []

for r in tqdm(members_1k, desc="members"):
    s = fused_score_for_record(r, "member")
    if s is not None:
        mem_scores.append(s)

for r in tqdm(nonmembers_1k, desc="nonmembers"):
    s = fused_score_for_record(r, "nonmember")
    if s is not None:
        non_scores.append(s)

print(f"\n📊 Valid samples: members={len(mem_scores)}/{len(members_1k)} | nonmembers={len(non_scores)}/{len(nonmembers_1k)}")
if len(mem_scores) == 0 or len(non_scores) == 0:
    raise RuntimeError("No valid samples collected.")

y_true = np.array([1]*len(mem_scores) + [0]*len(non_scores))
scores = np.array(mem_scores + non_scores)

def metrics_at_target_fpr(y_true, scores, target_fpr=0.05):
    fpr, tpr, thr = roc_curve(y_true, scores)
    idx = np.searchsorted(fpr, target_fpr, side="right")
    if idx == 0:
        thr_star = thr[0]; tpr_star = tpr[0]
    elif idx >= len(thr):
        thr_star = thr[-1]; tpr_star = tpr[-1]
    else:
        x0, x1 = fpr[idx-1], fpr[idx]
        y0, y1 = tpr[idx-1], tpr[idx]
        t0, t1 = thr[idx-1], thr[idx]
        w = (target_fpr - x0) / (x1 - x0 + 1e-12)
        tpr_star = y0 + w*(y1 - y0)
        thr_star = t0 + w*(t1 - t0)

    aucv = roc_auc_score(y_true, scores)
    return float(aucv), float(tpr_star), float(thr_star), fpr, tpr

aucv, tpr5, thr5, fpr, tpr = metrics_at_target_fpr(y_true, scores, target_fpr=0.05)
print("\n====== ModRényi* (Replica, FashionGen Strict 1k) ======")
print(f"AUC         = {aucv:.4f}")
print(f"TPR@5%FPR   = {tpr5:.4f}")
print(f"thr@5%FPR   = {thr5:.6f}")

# plot ROC
plt.figure(figsize=(8, 7))
plt.plot(fpr, tpr, lw=2, label=f"ModRényi* (AUC={aucv:.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ModRényi* ROC (FashionGen Strict 1k, Replica)")
plt.grid(alpha=0.3)
plt.legend(loc="lower right")

fig_path = os.path.join(OUT_DIR, "modrenyi_star_replica_roc.png")
plt.savefig(fig_path, dpi=300)
print(f"📈 Saved: {fig_path}")

json_path = os.path.join(OUT_DIR, "modrenyi_star_replica_scores.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "members_n": len(mem_scores),
            "nonmembers_n": len(non_scores),
            "scores_member": mem_scores,
            "scores_nonmember": non_scores,
            "alphas": list(RENYI_ALPHAS),
            "img_views": IMG_VIEWS,
            "txt_views": TXT_VIEWS,
            "length_norm": LENGTH_NORM
        },
        f,
        ensure_ascii=False,
        indent=2
    )
print(f"💾 Saved: {json_path}")


In [ ]:
# ==============================================================================
# Min-K++ (adaptive, filtered, length-normalized) - FashionGen (Strict Training Split)
# ✅ 严格复刻训练脚本的随机种子与切分：真实参与微调的那 1000 条 member
# ✅ 适配 FashionGen_MIA sharegpt 格式：messages + images
# ✅ 防止双 image token：自动 strip prompt 中的 "<image>"
# ✅ Qwen2-VL-2B + LoRA（PeftModel）
# ✅ 逻辑严格对齐原始 Min-K++：
#    - assistant 段 mask（labels[:prefix_len] = -100）
#    - per-token NLL
#    - 过滤无信息 token（空白/纯标点）
#    - 自适应 k：k = ceil(alpha * L) + clamp [K_MIN, K_MAX]
#    - 长度归一：/ sqrt(L)
# @title 5. Min-K++ Analysis
# ==============================================================================

import os, json, random, math, csv, warnings, re
from typing import List, Dict, Any, Optional, Tuple
from PIL import Image
from tqdm import tqdm
import numpy as np

import torch
import torch.nn.functional as F
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)

from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

warnings.filterwarnings("ignore")
assert torch.cuda.is_available(), "需要 GPU"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ============================================================
# 1) 路径配置（FashionGen）
# ============================================================

DRIVE_ROOT   = "/content/drive/MyDrive/FashionGen_MIA"
BASE_MODEL   = "/content/Qwen2-VL-2B-Instruct"
LORA_PATH    = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"

MEM_IMGROOT  = f"{DRIVE_ROOT}/member_5k"
NON_IMGROOT  = f"{DRIVE_ROOT}/nonmember_5k"
MEM_JSON     = f"{MEM_IMGROOT}/mllm_data.json"
NON_JSON     = f"{NON_IMGROOT}/mllm_data.json"

OUT_DIR      = f"{DRIVE_ROOT}/results/minkpp_fashiongen_1k_2b"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV      = os.path.join(OUT_DIR, "mia_minkPP_strict_1000x1000.csv")

# ============================================================
# 2) Strict split（必须与训练脚本一致）
# ============================================================

SEED = 42
TRAIN_SIZE = 1000
CONFIG_SEED = SEED + 123  # 训练脚本：config_seed = SEED + 123

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def reconstruct_strict_split() -> Tuple[List[Dict[str,Any]], List[Dict[str,Any]]]:
    with open(MEM_JSON, "r", encoding="utf-8") as f:
        all_members = json.load(f)
    random.Random(CONFIG_SEED).shuffle(all_members)
    members = all_members[:TRAIN_SIZE]

    with open(NON_JSON, "r", encoding="utf-8") as f:
        all_nonmembers = json.load(f)
    # 训练脚本里 nonmembers 也 shuffle 再取前 1000（你现在就是这么做的）
    random.Random(CONFIG_SEED).shuffle(all_nonmembers)
    nonmembers = all_nonmembers[:TRAIN_SIZE]

    return members, nonmembers

members_1k, nonmembers_1k = reconstruct_strict_split()
print(f"✅ Strict split reconstructed: member={len(members_1k)} nonmember={len(nonmembers_1k)}")

# ============================================================
# 3) 模型加载（Qwen2-VL-2B + LoRA）
# ============================================================

dtype = torch.bfloat16
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=dtype,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, LORA_PATH, is_trainable=False).eval()

def _model_device(m):
    try:
        return next(m.parameters()).device
    except StopIteration:
        return torch.device("cuda")

device = _model_device(model)
print(f"✅ Model loaded. device={device}, dtype={next(model.parameters()).dtype}")

# ============================================================
# 4) 超参（与原 Min-K++ 一致）
# ============================================================

ALPHA   = 0.20   # 底部比例（20%）
K_MIN   = 8      # 最小 k
K_MAX   = 64     # 最大 k

# 过滤“无信息 token”：空白 / 纯标点
RE_PUNC = re.compile(r"^\W+$", re.UNICODE)

# 防止双 image token
_IMAGE_TOKEN_RE = re.compile(r"<\s*image\s*>", flags=re.IGNORECASE)

def strip_image_token(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = _IMAGE_TOKEN_RE.sub("", s)
    s = s.replace("\n", " ").strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def robust_extract_text(c):
    # FashionGen: messages[x]["content"] 多数是 str（含<image>）
    if isinstance(c, str):
        return strip_image_token(c)
    if isinstance(c, list):
        parts = []
        for seg in c:
            if isinstance(seg, dict) and seg.get("type") == "text" and seg.get("text"):
                parts.append(seg["text"])
            elif isinstance(seg, str):
                parts.append(seg)
        return strip_image_token(" ".join(parts))
    return strip_image_token(str(c))

def _safe_open_image(path: str) -> Optional[Image.Image]:
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

def _is_informative(tok_str: str) -> bool:
    s = (tok_str or "").strip()
    if not s:
        return False
    if RE_PUNC.match(s):
        return False
    return True

# ============================================================
# 5) 构造 messages（对齐你 GradAudit 的做法：显式 image + text）
# ============================================================

def build_msgs_fashiongen(rec: Dict[str,Any]) -> Optional[List[Dict[str,Any]]]:
    """
    返回 ShareGPT messages：
      user: [image, text(prompt)]
      assistant: [text(answer)]
    """
    try:
        msgs = rec.get("messages", None)
        if not msgs or not isinstance(msgs, list) or len(msgs) < 2:
            return None

        q = robust_extract_text(msgs[0]["content"])
        a = robust_extract_text(msgs[1]["content"])

        if not q or not a:
            return None

        return [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
            {"role": "assistant", "content": [{"type": "text", "text": a}]},
        ]
    except Exception:
        return None

# ============================================================
# 6) pack：只在 assistant 段监督（与原始 Min-K++ 一致）
# ============================================================

def pack_qwen(image: Image.Image, msgs: List[Dict[str,Any]]) -> Optional[Dict[str,torch.Tensor]]:
    """
    用两次 processor 打包以获得 prefix_len（user-only + full）
    然后 labels[:prefix_len] = -100，仅监督 assistant 段
    """
    user_only = [msgs[0]]
    user_text = processor.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(msgs,      tokenize=False, add_generation_prompt=False)

    enc_prefix = processor(text=[user_text], images=[image], return_tensors="pt")
    enc_full   = processor(text=[full_text], images=[image], return_tensors="pt")

    # move to model device
    enc_prefix = {k: v.to(device) for k, v in enc_prefix.items()}
    enc_full   = {k: v.to(device) for k, v in enc_full.items()}

    input_ids  = enc_full["input_ids"]
    labels     = input_ids.clone()

    prefix_len = int(enc_prefix["input_ids"].size(1))
    if input_ids.size(1) <= prefix_len:
        return None

    labels[:, :prefix_len] = -100
    enc_full["labels"] = labels
    return enc_full

# ============================================================
# 7) Min-K++ 评分（严格对齐原始逻辑）
# ============================================================

@torch.no_grad()
def mink_pp_score(rec: Dict[str,Any], root: str) -> Optional[float]:
    """
    返回 raw score（NLL均值/长度归一）：
      - 越小越像 member
    """
    # 1) image
    rel = rec.get("images", [None])[0]
    if not rel:
        return None
    img_path = os.path.join(root, rel)
    image = _safe_open_image(img_path)
    if image is None:
        return None

    # 2) messages
    msgs = build_msgs_fashiongen(rec)
    if msgs is None:
        return None

    # 3) pack w/ assistant labels
    inputs = pack_qwen(image, msgs)
    if inputs is None:
        return None

    # 4) forward
    with torch.autocast(device_type="cuda", dtype=dtype, enabled=True):
        out = model(**inputs)

    logits = out.logits          # [1, L, V]
    labels = inputs["labels"]    # [1, L]

    # next-token align
    sl = logits[..., :-1, :]     # [1, L-1, V]
    tl = labels[..., 1:]         # [1, L-1]

    mask = tl.ne(-100)[0]        # assistant token mask on tl
    if mask.sum().item() == 0:
        return None

    # token log-prob
    logp = F.log_softmax(sl[0].float(), dim=-1)  # [L-1, V]
    tgt  = tl[0].clamp(min=0)                    # [L-1]

    # assistant token indices in tl-space
    idxs = torch.nonzero(mask, as_tuple=False).squeeze(1)  # positions in [0..L-2]

    # 过滤无信息 token
    keep = []
    for i in idxs.tolist():
        # i 对应 tl 的位置，也对应 input_ids 的位置 i+1（next-token label）
        tok_id = int(inputs["input_ids"][0, i+1].item())
        piece = processor.tokenizer.decode([tok_id], skip_special_tokens=True)
        if _is_informative(piece):
            keep.append(i)

    if not keep:
        return None

    keep = torch.tensor(keep, device=logp.device, dtype=torch.long)

    # per-token NLL on filtered positions
    filtered_nll = -logp[keep, tgt[keep]]   # [L_filtered]
    L = int(filtered_nll.numel())
    if L <= 0:
        return None

    # 自适应 k
    k = int(math.ceil(ALPHA * L))
    k = max(K_MIN, min(K_MAX, k))
    k = min(k, L)

    # 取最小 NLL 的 k 个（=> -NLL 最大）
    vals, _ = torch.topk(-filtered_nll, k)  # [k]
    score = (-vals).mean().item() / math.sqrt(L)  # 长度归一
    return float(score)

def run_side(recs: List[Dict[str,Any]], root: str, desc: str) -> List[float]:
    out = []
    for r in tqdm(recs, desc=desc):
        try:
            s = mink_pp_score(r, root)
            if s is not None and math.isfinite(s):
                out.append(s)
        except Exception:
            pass
    return out

mem_scores = run_side(members_1k,    MEM_IMGROOT, "members")
non_scores = run_side(nonmembers_1k, NON_IMGROOT, "nonmembers")
print(f"有效样本：M={len(mem_scores)} N={len(non_scores)}")

if len(mem_scores) == 0 or len(non_scores) == 0:
    raise RuntimeError("有效样本为 0，无法计算指标。")

# ============================================================
# 8) 评估（关键修复：方向统一）
# ============================================================

y = np.array([1]*len(mem_scores) + [0]*len(non_scores))
raw = np.array(mem_scores + non_scores, dtype=np.float64)

# ✅ Min-K++ raw 越小越像 member，所以正向分数应取负号：越大越像 member
sc = -raw

auc_val = roc_auc_score(y, sc)
fpr, tpr, thr = roc_curve(y, sc)
j = int(np.argmax(tpr - fpr))

acc = accuracy_score(y, (sc >= thr[j]).astype(int))
cm  = confusion_matrix(y, (sc >= thr[j]).astype(int))

print("\n====== Min-K++ MIA 结果（基础, 已修正方向） ======")
print(f"AUC={auc_val:.4f}  thr={thr[j]:.6f}  acc={acc:.4f}")
print("Confusion [[TN FP][FN TP]]:\n", cm)

# ---------------- @5%FPR 的完整指标（对齐你原始代码逻辑） ----------------
def metrics_at_target_fpr(y_true: np.ndarray, scores: np.ndarray, target_fpr: float = 0.05):
    fpr, tpr, thr = roc_curve(y_true, scores)
    idx = np.searchsorted(fpr, target_fpr, side="right")
    if idx == 0:
        thr_star = thr[0]; tpr_star = tpr[0]
    elif idx >= len(thr):
        thr_star = thr[-1]; tpr_star = tpr[-1]
    else:
        x0, x1 = fpr[idx-1], fpr[idx]
        y0, y1 = tpr[idx-1], tpr[idx]
        t0, t1 = thr[idx-1], thr[idx]
        w = (target_fpr - x0) / (x1 - x0 + 1e-12)
        thr_star = t0 + w*(t1 - t0)
        tpr_star = y0 + w*(y1 - y0)

    y_pred = (scores >= thr_star).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    cm2 = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm2.ravel()
    fpr_actual = fp / (fp + tn + 1e-12)
    tpr_actual = tp / (tp + fn + 1e-12)
    acc2 = accuracy_score(y_true, y_pred)
    return {
        "thr@5%FPR": float(thr_star),
        "TPR@5%FPR(interp)": float(tpr_star),
        "FPR(actual)": float(fpr_actual),
        "TPR(actual)": float(tpr_actual),
        "Precision": float(prec),
        "Recall": float(rec),
        "F1": float(f1),
        "Accuracy": float(acc2),
        "AUC": float(roc_auc_score(y_true, scores)),
        "CM": cm2,
    }

m5 = metrics_at_target_fpr(y, sc, target_fpr=0.05)

print("\n====== Min-K++ MIA 结果（@5% FPR, 已修正方向） ======")
print(f"AUC                 = {m5['AUC']:.4f}")
print(f"thr @5%FPR          = {m5['thr@5%FPR']:.6f}")
print(f"TPR@5%FPR (interp)  = {m5['TPR@5%FPR(interp)']:.4f} | "
      f"实际 FPR={m5['FPR(actual)']*100:.2f}%  实际 TPR={m5['TPR(actual)']*100:.2f}%")
print(f"Precision           = {m5['Precision']:.4f}")
print(f"Recall              = {m5['Recall']:.4f}")
print(f"F1                  = {m5['F1']:.4f}")
print(f"Accuracy            = {m5['Accuracy']*100:.2f}%")
print("Confusion [[TN FP][FN TP]]：")
print(m5["CM"])

# ============================================================
# 9) 保存 CSV（同时保存 raw 与 corrected 分数，方便你对齐论文）
# ============================================================

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["split", "raw_score(minkPP, lower=more_member)", "score_for_ROC(higher=more_member)"])
    for s in mem_scores:
        w.writerow(["member", float(s), float(-s)])
    for s in non_scores:
        w.writerow(["nonmember", float(s), float(-s)])

print(f"\n💾 Saved: {OUT_CSV}")

# 额外：打印分布（快速 sanity check）
print("\n[Sanity] raw score stats (lower=more_member):")
print(" member mean/std:", float(np.mean(mem_scores)), float(np.std(mem_scores)))
print(" nonmem  mean/std:", float(np.mean(non_scores)), float(np.std(non_scores)))


In [ ]:
# @title 6. Entropy (FashionGen / Qwen2-VL-2B + LoRA / Strict Split 1000x1000)
# ========================= MIA: Entropy (negative mean entropy) =========================

import os, json, random, math, csv, warnings, re
from typing import List, Dict, Any, Optional, Tuple
from PIL import Image
from tqdm import tqdm
import numpy as np

import torch
import torch.nn.functional as F
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)

from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

warnings.filterwarnings("ignore")
assert torch.cuda.is_available(), "需要 GPU"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ============================================================
# 1) 路径配置（✅ FashionGen）
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive/FashionGen_MIA"

BASE_MODEL = "/content/Qwen2-VL-2B-Instruct"                 # base
LORA_PATH  = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"       # adapter

MEM_IMGROOT = f"{DRIVE_ROOT}/member_5k"
NON_IMGROOT = f"{DRIVE_ROOT}/nonmember_5k"
MEM_JSON    = f"{MEM_IMGROOT}/mllm_data.json"
NON_JSON    = f"{NON_IMGROOT}/mllm_data.json"

OUT_DIR = f"{DRIVE_ROOT}/results/entropy_fashiongen_1k_2b"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, "mia_entropy_strict_1000x1000.csv")

# ============================================================
# 2) Strict split（✅ 复刻训练脚本）
# ============================================================

SEED = 42
TRAIN_SIZE = 1000
CONFIG_SEED = SEED + 123  # 训练脚本：config_seed = SEED + 123

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def reconstruct_strict_split() -> Tuple[List[Dict[str,Any]], List[Dict[str,Any]]]:
    with open(MEM_JSON, "r", encoding="utf-8") as f:
        all_members = json.load(f)
    random.Random(CONFIG_SEED).shuffle(all_members)
    members = all_members[:TRAIN_SIZE]

    with open(NON_JSON, "r", encoding="utf-8") as f:
        all_nonmembers = json.load(f)
    # 与你训练脚本一致：也 shuffle，再取前 1000
    random.Random(CONFIG_SEED).shuffle(all_nonmembers)
    nonmembers = all_nonmembers[:TRAIN_SIZE]
    return members, nonmembers

members_1k, nonmembers_1k = reconstruct_strict_split()
print(f"✅ Strict split reconstructed: member={len(members_1k)} nonmember={len(nonmembers_1k)}")

# ============================================================
# 3) 模型加载（✅ Qwen2-VL-2B + LoRA）
# ============================================================

dtype = torch.bfloat16
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=dtype,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, LORA_PATH, is_trainable=False).eval()

def _model_device(m):
    try:
        return next(m.parameters()).device
    except StopIteration:
        return torch.device("cuda")

device = _model_device(model)
print(f"✅ Model loaded. device={device}, dtype={next(model.parameters()).dtype}")

# ============================================================
# 4) FashionGen 文本处理（防双 <image>）
# ============================================================

_IMAGE_TOKEN_RE = re.compile(r"<\s*image\s*>", flags=re.IGNORECASE)

def strip_image_token(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = _IMAGE_TOKEN_RE.sub("", s)
    s = s.replace("\n", " ").strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def robust_extract_text(c):
    """
    FashionGen_MIA:
      - messages[0]['content'] 多数为 str（含 <image>）
      - messages[1]['content'] 多数为 str（caption）
    """
    if isinstance(c, str):
        return strip_image_token(c)
    if isinstance(c, list):
        parts = []
        for seg in c:
            if isinstance(seg, dict) and seg.get("type") == "text" and seg.get("text"):
                parts.append(seg["text"])
            elif isinstance(seg, str):
                parts.append(seg)
        return strip_image_token(" ".join(parts))
    return strip_image_token(str(c))

def _safe_open_image(path: str) -> Optional[Image.Image]:
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

# ============================================================
# 5) 构造 messages（对齐你原始 Entropy 逻辑：显式 image + text）
# ============================================================

def _build_msgs_fashiongen(rec: Dict[str, Any]) -> Optional[List[Dict[str, Any]]]:
    try:
        if not rec.get("images"):
            return None
        msgs = rec.get("messages", None)
        if not msgs or not isinstance(msgs, list) or len(msgs) < 2:
            return None

        q = robust_extract_text(msgs[0]["content"])
        a = robust_extract_text(msgs[1]["content"])
        if not q or not a:
            return None

        return [
            {"role": "user", "content": [{"type":"image"}, {"type":"text","text": q}]},
            {"role": "assistant", "content": [{"type":"text","text": a}]},
        ]
    except Exception:
        return None

# ============================================================
# 6) pack：两次打包计算 prefix_len，仅统计 assistant 段
# ============================================================

def _pack(image: Image.Image, msgs: List[Dict[str,Any]]) -> Optional[Tuple[Dict[str,torch.Tensor], int]]:
    user_only = [msgs[0]]
    user_text = processor.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(msgs,      tokenize=False, add_generation_prompt=False)

    enc_prefix = processor(text=[user_text], images=[image], return_tensors="pt")
    enc_full   = processor(text=[full_text], images=[image], return_tensors="pt")

    # to model device
    enc_prefix = {k: v.to(device) for k, v in enc_prefix.items()}
    enc_full   = {k: v.to(device) for k, v in enc_full.items()}

    input_ids  = enc_full["input_ids"]
    labels     = input_ids.clone()

    prefix_len = int(enc_prefix["input_ids"].size(1))
    if input_ids.size(1) <= prefix_len:
        return None

    labels[:, :prefix_len] = -100
    enc_full["labels"] = labels
    return enc_full, prefix_len

# ============================================================
# 7) Entropy 分数：assistant token 上的负平均熵（越大越像 member）
# ============================================================

@torch.no_grad()
def neg_entropy_one(rec: Dict[str,Any], img_root: str) -> Optional[float]:
    rel = rec.get("images", [None])[0]
    if not rel:
        return None

    img_path = os.path.join(img_root, rel)
    image = _safe_open_image(img_path)
    if image is None:
        return None

    msgs = _build_msgs_fashiongen(rec)
    if msgs is None:
        return None

    packed = _pack(image, msgs)
    if packed is None:
        return None

    inputs, _ = packed

    with torch.autocast(device_type="cuda", dtype=dtype, enabled=True):
        out = model(**inputs)

    logits = out.logits                # [1, T, V]
    labels = inputs["labels"]          # [1, T]

    # next-token align，仅 assistant 段
    shift_logits = logits[..., :-1, :] # [1, T-1, V]
    shift_labels = labels[...,  1: ]   # [1, T-1]
    mask = shift_labels.ne(-100)[0]    # [T-1]
    if mask.sum().item() == 0:
        return None

    valid_logits = shift_logits[0][mask].float()  # [N, V]
    p = torch.softmax(valid_logits, dim=-1)
    ent = -(p * (p.add(1e-9).log())).sum(dim=-1)  # per-token entropy [N]
    return float((-ent.mean()).item())            # 负熵：越大越像成员

def run_side(recs: List[Dict[str,Any]], root: str, desc: str) -> List[float]:
    scores = []
    for r in tqdm(recs, desc=desc):
        try:
            s = neg_entropy_one(r, root)
            if s is not None and math.isfinite(s):
                scores.append(s)
        except Exception:
            pass
    return scores

print("🧮 计算 entropy 分数 …")
mem_scores = run_side(members_1k,    MEM_IMGROOT, "members")
non_scores = run_side(nonmembers_1k, NON_IMGROOT, "nonmembers")
print(f"✅ 有效样本：member={len(mem_scores)}  nonmember={len(non_scores)}")

if len(mem_scores) == 0 or len(non_scores) == 0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

# ============================================================
# 8) 指标评估（与原版一致）
# ============================================================

y  = np.array([1]*len(mem_scores) + [0]*len(non_scores))
sc = np.array(mem_scores + non_scores, dtype=np.float64)

auc_val = roc_auc_score(y, sc)
fpr, tpr, thr = roc_curve(y, sc)
j = int(np.argmax(tpr - fpr))
acc = accuracy_score(y, (sc >= thr[j]).astype(int))
cm  = confusion_matrix(y, (sc >= thr[j]).astype(int))

print("\n====== Entropy-MIA 结果（基础） ======")
print(f"AUC = {auc_val:.4f}")
print(f"Best thr = {thr[j]:.6f}")
print(f"Acc = {acc:.4f}")
print("Confusion [[TN FP][FN TP]]:\n", cm)

def metrics_at_target_fpr(y_true: np.ndarray, scores: np.ndarray, target_fpr: float = 0.05):
    fpr, tpr, thr = roc_curve(y_true, scores)
    idx = np.searchsorted(fpr, target_fpr, side="right")
    if idx == 0:
        thr_star = thr[0]; tpr_star = tpr[0]
    elif idx >= len(thr):
        thr_star = thr[-1]; tpr_star = tpr[-1]
    else:
        x0, x1 = fpr[idx-1], fpr[idx]
        y0, y1 = tpr[idx-1], tpr[idx]
        t0, t1 = thr[idx-1], thr[idx]
        w = (target_fpr - x0) / (x1 - x0 + 1e-12)
        thr_star = t0 + w*(t1 - t0)
        tpr_star = y0 + w*(y1 - y0)

    y_pred = (scores >= thr_star).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    cm2 = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm2.ravel()
    fpr_actual = fp / (fp + tn + 1e-12)
    tpr_actual = tp / (tp + fn + 1e-12)
    acc2 = accuracy_score(y_true, y_pred)
    return {
        "thr@5%FPR": float(thr_star),
        "TPR@5%FPR(interp)": float(tpr_star),
        "FPR(actual)": float(fpr_actual),
        "TPR(actual)": float(tpr_actual),
        "Precision": float(prec),
        "Recall": float(rec),
        "F1": float(f1),
        "Accuracy": float(acc2),
        "AUC": float(roc_auc_score(y_true, scores)),
        "CM": cm2,
    }

m5 = metrics_at_target_fpr(y, sc, target_fpr=0.05)

print("\n====== Entropy-MIA 结果（@5% FPR） ======")
print(f"AUC                 = {m5['AUC']:.4f}")
print(f"thr @5%FPR          = {m5['thr@5%FPR']:.6f}")
print(f"TPR@5%FPR (interp)  = {m5['TPR@5%FPR(interp)']:.4f} | "
      f"实际 FPR={m5['FPR(actual)']*100:.2f}%  实际 TPR={m5['TPR(actual)']*100:.2f}%")
print(f"Precision           = {m5['Precision']:.4f}")
print(f"Recall              = {m5['Recall']:.4f}")
print(f"F1                  = {m5['F1']:.4f}")
print(f"Accuracy            = {m5['Accuracy']*100:.2f}%")
print("Confusion [[TN FP][FN TP]]：")
print(m5["CM"])

# ============================================================
# 9) 保存 CSV（与原版一致）
# ============================================================

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["split", "score(neg-entropy, higher=more_member)"])
    for s in mem_scores:
        w.writerow(["member", float(s)])
    for s in non_scores:
        w.writerow(["nonmember", float(s)])

print(f"\n💾 已保存明细：{OUT_CSV}")

# 额外：简单 sanity check
print("\n[Sanity] score stats (higher=more_member):")
print(" member mean/std:", float(np.mean(mem_scores)), float(np.std(mem_scores)))
print(" nonmem  mean/std:", float(np.mean(non_scores)), float(np.std(non_scores)))


In [ ]:
# ==============================================================================
# Similarity-MIA (Image + Text) with Fusion — FashionGen (Strict Training Split)
# - Model: Qwen2-VL-2B-Instruct + LoRA adapter (FashionGen)
# - Strict split reconstructed (same seed as training)
# - Auto-install deps + verify imports (open_clip, sentence_transformers, peft)
# - No invalid generation flags => no warning spam
# @title 7. Similarity-MIA
# ==============================================================================

import os, io, json, random, re, warnings, traceback, sys, subprocess, importlib
from typing import Dict, Any, Optional, List

import numpy as np
import torch
from PIL import Image, ImageFilter
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, precision_recall_fscore_support

warnings.filterwarnings("ignore")

# ============================================================
# 0) Robust install (pip) + import verification
# ============================================================
def pip_install(packages: List[str]):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages
    subprocess.check_call(cmd)

def ensure_import(module_name: str, pip_names: List[str]):
    try:
        return importlib.import_module(module_name)
    except Exception:
        print(f"🔧 Installing missing dependency for: {module_name}  via pip {pip_names} ...")
        pip_install(pip_names)
        # retry import
        return importlib.import_module(module_name)

# peft
ensure_import("peft", ["peft"])
# sentence_transformers
ensure_import("sentence_transformers", ["sentence-transformers"])
# open_clip (module is open_clip; pip package is open_clip_torch)
ensure_import("open_clip", ["open_clip_torch", "timm"])

# ============================================================
# 1) Paths / Config (FashionGen)
# ============================================================
DRIVE_ROOT = "/content/drive/MyDrive/FashionGen_MIA"

# ✅ 如果你没有本地 /content/Qwen2-VL-2B-Instruct，就改成 HF repo id:
# BASE_MODEL = "Qwen/Qwen2-VL-2B-Instruct"
BASE_MODEL = "/content/Qwen2-VL-2B-Instruct"

LORA_PATH  = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"

MEMBER_DIR = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_DIR = f"{DRIVE_ROOT}/nonmember_5k"
MEMBERS_FULL_JSON = f"{MEMBER_DIR}/mllm_data.json"
NONMEMBERS_FULL_JSON = f"{NONMEMBER_DIR}/mllm_data.json"

OUT_DIR = f"{DRIVE_ROOT}/results/similarity_fused_fashiongen_1k_2b"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, "mia_similarity_fused_strict_1000x1000.csv")

TRAIN_SIZE = 1000
SEED = 42
CONFIG_SEED = SEED + 123

# image corruption
GAUSS_RADIUS = 1.0
JPEG_QUALITY = 60

# text generation
MAX_NEW_TOKENS = 24
TEXT_FAIL_TOL = 20

# fusion weights
W_IMG, W_TEXT = 0.60, 0.40

# ============================================================
# 2) Device / dtype
# ============================================================
assert torch.cuda.is_available(), "Need GPU"
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
dtype = torch.bfloat16
device0 = "cuda:0"
print(f"✅ Using {device0}, dtype={dtype}")

# ============================================================
# 3) Resolve BASE_MODEL: local path vs HF repo_id
# ============================================================
def resolve_model_id_or_path(x: str) -> str:
    if os.path.isdir(x) and os.path.isfile(os.path.join(x, "config.json")):
        return x
    if x.startswith("/") and (not os.path.isdir(x)):
        raise FileNotFoundError(
            f"BASE_MODEL looks like a local path but not found: {x}\n"
            f"Either:\n"
            f"  (1) make sure the folder exists and contains config.json\n"
            f"  (2) change BASE_MODEL to HF repo id like 'Qwen/Qwen2-VL-2B-Instruct'"
        )
    return x

BASE_MODEL_RESOLVED = resolve_model_id_or_path(BASE_MODEL)
print(f"✅ BASE_MODEL resolved as: {BASE_MODEL_RESOLVED}")

# ============================================================
# 4) Load Qwen2-VL + LoRA
# ============================================================
from transformers import AutoProcessor, GenerationConfig
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

processor = AutoProcessor.from_pretrained(BASE_MODEL_RESOLVED, trust_remote_code=True)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_RESOLVED,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, LORA_PATH, is_trainable=False).eval()

def _model_device(m):
    try:
        return next(m.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_DEVICE = _model_device(model)
print(f"✅ Qwen2-VL(+LoRA) ready. device={MODEL_DEVICE}, dtype={next(iter(model.parameters())).dtype}")

GENCFG = GenerationConfig(
    do_sample=False,
    num_beams=1,
    max_new_tokens=MAX_NEW_TOKENS,
    use_cache=False,
)

# ============================================================
# 5) OpenCLIP + SentenceTransformer
# ============================================================
import open_clip
OPENCLIP_MODEL, _, OPENCLIP_TRANS = open_clip.create_model_and_transforms(
    "ViT-L-14", pretrained="openai", device=device0
)
OPENCLIP_MODEL.eval()
print("✅ OpenCLIP ready")

from sentence_transformers import SentenceTransformer
txt_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device0)
txt_embedder.eval()
print("✅ SentenceTransformer ready")

# ============================================================
# 6) Strict split reconstruction
# ============================================================
def load_json_list(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def get_exact_training_data():
    all_members = load_json_list(MEMBERS_FULL_JSON)
    rng = random.Random(CONFIG_SEED)
    rng.shuffle(all_members)
    real_members = all_members[:TRAIN_SIZE]

    all_nonmembers = load_json_list(NONMEMBERS_FULL_JSON)
    rng2 = random.Random(CONFIG_SEED)
    rng2.shuffle(all_nonmembers)
    real_nonmembers = all_nonmembers[:TRAIN_SIZE]
    return real_members, real_nonmembers

members, nonmembers = get_exact_training_data()
print(f"\n📦 Strict samples ready: members={len(members)} | nonmembers={len(nonmembers)}")

# ============================================================
# 7) Helpers
# ============================================================
_IMAGE_TOKEN_RE = re.compile(r"<\s*image\s*>", flags=re.IGNORECASE)

def strip_image_token(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = _IMAGE_TOKEN_RE.sub("", s)
    s = s.replace("\n", " ").strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def robust_extract_text(c):
    if isinstance(c, str):
        return strip_image_token(c)
    if isinstance(c, list):
        parts = []
        for seg in c:
            if isinstance(seg, dict) and seg.get("type") == "text" and seg.get("text"):
                parts.append(seg["text"])
            elif isinstance(seg, str):
                parts.append(seg)
        return strip_image_token(" ".join(parts))
    return strip_image_token(str(c))

def load_img(root: str, rel_path: str) -> Optional[Image.Image]:
    try:
        p = os.path.join(root, rel_path)
        if os.path.exists(p):
            return Image.open(p).convert("RGB")
        return None
    except Exception:
        return None

def corrupt_image(img: Image.Image) -> Image.Image:
    x = img.filter(ImageFilter.GaussianBlur(radius=GAUSS_RADIUS))
    buf = io.BytesIO()
    x.save(buf, format="JPEG", quality=JPEG_QUALITY, optimize=True)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

# ============================================================
# 8) Generate text (Qwen2-VL)
# ============================================================
USER_PROMPT = "Describe the clothing item in the image in one short sentence."

@torch.inference_mode()
def generate_text_qwen(image: Image.Image, user_text: str) -> str:
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
    prompt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    enc = processor(text=[prompt], images=[image], return_tensors="pt")
    enc = {k: v.to(MODEL_DEVICE) for k, v in enc.items()}

    out_ids = model.generate(**enc, generation_config=GENCFG)
    text = processor.tokenizer.batch_decode(out_ids, skip_special_tokens=True)[0].strip()
    return text

# ============================================================
# 9) Similarity scores
# ============================================================
first_err = {"img": None, "txt": None}
FAIL = {"image": 0, "img_only_fail": 0, "txt_only_fail": 0}

@torch.inference_mode()
def image_similarity_score(img: Image.Image) -> Optional[float]:
    try:
        img2 = corrupt_image(img)
        t1 = OPENCLIP_TRANS(img).unsqueeze(0).to(device0)
        t2 = OPENCLIP_TRANS(img2).unsqueeze(0).to(device0)
        z1 = OPENCLIP_MODEL.encode_image(t1).float()
        z2 = OPENCLIP_MODEL.encode_image(t2).float()
        z1 = z1 / (z1.norm(dim=-1, keepdim=True) + 1e-12)
        z2 = z2 / (z2.norm(dim=-1, keepdim=True) + 1e-12)
        sim = torch.cosine_similarity(z1[0], z2[0], dim=-1).clamp(-1, 1).item()
        return float(sim)
    except Exception as e:
        if first_err["img"] is None:
            first_err["img"] = "".join(traceback.format_exception_only(type(e), e)).strip()
        return None

@torch.inference_mode()
def text_similarity_score(img: Image.Image) -> Optional[float]:
    try:
        img2 = corrupt_image(img)
        t1 = generate_text_qwen(img, USER_PROMPT)
        t2 = generate_text_qwen(img2, USER_PROMPT)
        e1 = txt_embedder.encode([t1], convert_to_tensor=True, normalize_embeddings=True, device=device0)[0]
        e2 = txt_embedder.encode([t2], convert_to_tensor=True, normalize_embeddings=True, device=device0)[0]
        sim = torch.cosine_similarity(e1, e2, dim=-1).clamp(-1, 1).item()
        return float(sim)
    except Exception as e:
        if first_err["txt"] is None:
            first_err["txt"] = "".join(traceback.format_exception_only(type(e), e)).strip()
        return None

def _nan_filter(xs):
    return np.array([x for x in xs if (x is not None and np.isfinite(x))], dtype=np.float32)

def collect_scores(samples: List[Dict[str, Any]], source: str, enable_text: bool = True):
    sims_img, sims_txt = [], []
    root = MEMBER_DIR if source == "member" else NONMEMBER_DIR
    text_fail_streak = 0

    for i in tqdm(range(len(samples)), desc=source):
        s = samples[i]
        try:
            if not s.get("images"):
                FAIL["image"] += 1
                sims_img.append(None); sims_txt.append(None)
                continue
            img = load_img(root, s["images"][0])
            if img is None:
                FAIL["image"] += 1
                sims_img.append(None); sims_txt.append(None)
                continue
        except Exception:
            FAIL["image"] += 1
            sims_img.append(None); sims_txt.append(None)
            continue

        si = image_similarity_score(img)
        if si is None: FAIL["img_only_fail"] += 1
        sims_img.append(si)

        st = None
        if enable_text:
            st = text_similarity_score(img)
            if st is None:
                text_fail_streak += 1
            else:
                text_fail_streak = 0
            if text_fail_streak >= TEXT_FAIL_TOL:
                enable_text = False
                print(f"⚠️ Text channel failed {TEXT_FAIL_TOL} times consecutively -> disabled.")
        if st is None: FAIL["txt_only_fail"] += 1
        sims_txt.append(st)

        if (i + 1) % 50 == 0:
            torch.cuda.empty_cache()

    return sims_img, sims_txt, enable_text

print("\n⏳ Collecting similarity scores (members)...")
mem_img_raw, mem_txt_raw, txt_enabled = collect_scores(members, "member", enable_text=True)
print("⏳ Collecting similarity scores (nonmembers)...")
non_img_raw, non_txt_raw, _ = collect_scores(nonmembers, "nonmember", enable_text=txt_enabled)

mem_img = _nan_filter(mem_img_raw); non_img = _nan_filter(non_img_raw)
mem_txt = _nan_filter(mem_txt_raw); non_txt = _nan_filter(non_txt_raw)

print(f"\n📊 Valid samples (per-channel):")
print(f"image: mem={len(mem_img)} non={len(non_img)}")
print(f"text : mem={len(mem_txt)} non={len(non_txt)}")
if first_err["img"]: print(f"🧪 First image-channel error: {first_err['img']}")
if first_err["txt"]: print(f"🧪 First text-channel error : {first_err['txt']}")
print(f"❗ Fail stats: {FAIL}")

# ============================================================
# 10) Fusion + Metrics
# ============================================================
def zscore(arr: np.ndarray):
    mu = float(arr.mean())
    sd = float(arr.std() + 1e-8)
    return (arr - mu) / (sd + 1e-8), mu, sd

have_img = (len(mem_img) > 10) and (len(non_img) > 10)
have_txt = (len(mem_txt) > 10) and (len(non_txt) > 10)
if not have_img and not have_txt:
    raise RuntimeError("No valid scores in both channels. Check first_err above.")

parts = []
if have_img: parts.append(("img", mem_img, non_img, W_IMG))
if have_txt: parts.append(("txt", mem_txt, non_txt, W_TEXT))
if len(parts) == 1:
    parts[0] = (parts[0][0], parts[0][1], parts[0][2], 1.0)

details = {}
fused_mem, fused_non = None, None

for name, mem_arr, non_arr, w in parts:
    non_z, mu, sd = zscore(non_arr)
    mem_z = (mem_arr - mu) / (sd + 1e-8)
    details[name] = {"mem": mem_arr, "non": non_arr, "mem_z": mem_z, "non_z": non_z, "w": w}
    if fused_mem is None:
        fused_mem = w * mem_z
        fused_non = w * non_z
    else:
        fused_mem = fused_mem + w * mem_z
        fused_non = fused_non + w * non_z

y_true = np.array([1]*len(fused_mem) + [0]*len(fused_non), dtype=np.int32)
scores = np.concatenate([fused_mem, fused_non], axis=0)

auc_val = roc_auc_score(y_true, scores)

thr_5 = float(np.quantile(fused_non, 0.95))
y_pred = (scores >= thr_5).astype(int)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr_actual = fp / (fp + tn + 1e-12)
tpr_actual = tp / (tp + fn + 1e-12)

prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
acc = accuracy_score(y_true, y_pred)

print("\n====== Similarity-MIA (FashionGen, fused) ======")
print(f"AUC                = {auc_val:.4f}")
print(f"Threshold @FPR=5%  = {thr_5:.4f} (actual FPR={fpr_actual*100:.2f}%)")
print(f"TPR (Recall)       = {tpr_actual*100:.2f}%")
print(f"Precision          = {prec*100:.2f}%")
print(f"F1                 = {f1:.4f}")
print(f"Accuracy           = {acc*100:.2f}%")
print("Confusion [[TN FP][FN TP]]:")
print(cm)

# ============================================================
# 11) Save CSV
# ============================================================
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    header = ["split"]
    if have_img: header += ["s_img", "z_img"]
    if have_txt: header += ["s_txt", "z_txt"]
    header += ["s_fused"]
    w.writerow(header)

    def write_side(is_member: bool):
        tag = "member" if is_member else "nonmember"
        zs  = fused_mem if is_member else fused_non
        n   = len(zs)
        for i in range(n):
            row = [tag]
            if have_img:
                arr = details["img"]["mem"] if is_member else details["img"]["non"]
                z   = details["img"]["mem_z"] if is_member else details["img"]["non_z"]
                row += [float(arr[i % len(arr)]), float(z[i % len(z)])]
            if have_txt:
                arr = details["txt"]["mem"] if is_member else details["txt"]["non"]
                z   = details["txt"]["mem_z"] if is_member else details["txt"]["non_z"]
                row += [float(arr[i % len(arr)]), float(z[i % len(z)])]
            row += [float(zs[i])]
            w.writerow(row)

    write_side(True)
    write_side(False)

print(f"\n💾 Saved: {OUT_CSV}")


In [ ]:
# ==============================================================================
# GradNorm MIA Baseline (Layer-scope, GradAudit-style layer parsing)
# ONLY change: gradient parameter scope (vt_last3 or vt_all), exclude LoRA params
# No LOSS_TOKEN_KEEP_PROB, No RMS. Standard L2 GradNorm.
# @title 8. GradNorm MIA
# ==============================================================================

import os, json, math, random, re, warnings, csv
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix, precision_recall_fscore_support

from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

warnings.filterwarnings("ignore")

# ============================================================
# 0) Paths & Config
# ============================================================
DRIVE_ROOT   = "/content/drive/MyDrive/FashionGen_MIA"
BASE_MODEL   = "/content/Qwen2-VL-2B-Instruct"
LORA_PATH    = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"

MEMBER_DIR        = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_DIR     = f"{DRIVE_ROOT}/nonmember_5k"
MEMBERS_FULL_JSON = f"{MEMBER_DIR}/mllm_data.json"
NONMEMBERS_FULL_JSON = f"{NONMEMBER_DIR}/mllm_data.json"

OUTPUT_DIR = f"{DRIVE_ROOT}/results/gradnorm_scope_fashiongen_1k_2b"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# strict split (match training)
TRAIN_SIZE   = 1000
SEED         = 42
CONFIG_SEED  = SEED + 123

# ---- ONLY knob you need ----
GRAD_PARAM_SCOPE = "vt_all"   # "vt_last3" or "vt_all"
LAST_K = 3                      # used when vt_last3

# standard options (unchanged)
LENGTH_NORM = True
MAX_TOKENS_CAP = None
USE_AMP = True
DTYPE = torch.bfloat16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "Need GPU"

# ============================================================
# 1) Strict split reconstruction
# ============================================================
print("🔄 Reconstructing exact training split (FashionGen)...")

def _load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def get_exact_training_data() -> Tuple[List[Dict[str,Any]], List[Dict[str,Any]]]:
    all_members = _load_json(MEMBERS_FULL_JSON)
    rng = random.Random(CONFIG_SEED)
    rng.shuffle(all_members)
    real_members = all_members[:TRAIN_SIZE]

    all_non = _load_json(NONMEMBERS_FULL_JSON)
    rng = random.Random(CONFIG_SEED)
    rng.shuffle(all_non)
    real_non = all_non[:TRAIN_SIZE]

    print(f"✅ Strict split reconstructed: member={len(real_members)} nonmember={len(real_non)}")
    return real_members, real_non

m_data, n_data = get_exact_training_data()

# ============================================================
# 2) Load model (Qwen2-VL-2B + LoRA)
# ============================================================
def _ensure_local_model_dir(path: str):
    if not os.path.isdir(path):
        raise RuntimeError(f"BASE_MODEL directory not found: {path}")

_ensure_local_model_dir(BASE_MODEL)

print(f"\n🚀 Loading Qwen2-VL base from: {BASE_MODEL}")
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True
)

print(f"🔧 Loading LoRA from: {LORA_PATH}")
# is_trainable=True so selected backbone params can receive grads after we enable them
model = PeftModel.from_pretrained(base_model, LORA_PATH, is_trainable=True)
model.eval()

MODEL_DEVICE = next(model.parameters()).device
print(f"✅ Model loaded. device={MODEL_DEVICE}, dtype={next(model.parameters()).dtype}")

# ============================================================
# 3) Data helpers
# ============================================================
_IMAGE_TOKEN_RE = re.compile(r"<\s*image\s*>", flags=re.IGNORECASE)

def strip_image_token(s: str) -> str:
    if not isinstance(s, str): return ""
    s = _IMAGE_TOKEN_RE.sub("", s)
    s = s.replace("\n", " ").strip()
    return re.sub(r"\s+", " ", s).strip()

def robust_extract_text(c) -> str:
    if isinstance(c, str): return strip_image_token(c)
    if isinstance(c, list):
        parts = []
        for seg in c:
            if isinstance(seg, dict) and seg.get("type") == "text" and seg.get("text"):
                parts.append(seg["text"])
            elif isinstance(seg, str):
                parts.append(seg)
        return strip_image_token(" ".join(parts))
    return strip_image_token(str(c))

def load_img(root: str, rel_path: str) -> Optional[Image.Image]:
    try:
        p = os.path.join(root, rel_path)
        if os.path.exists(p):
            return Image.open(p).convert("RGB")
        return None
    except Exception:
        return None

# ============================================================
# 4) Packing (labels only on assistant segment)
# ============================================================
def pack_qwen(image: Image.Image, question: str, answer: str):
    msgs_full = [
        {"role":"user", "content":[{"type":"image"}, {"type":"text", "text":question}]},
        {"role":"assistant", "content":[{"type":"text", "text":answer}]}
    ]
    msgs_u = [
        {"role":"user", "content":[{"type":"image"}, {"type":"text", "text":question}]}
    ]

    text_full = processor.apply_chat_template(msgs_full, tokenize=False, add_generation_prompt=False)
    text_u    = processor.apply_chat_template(msgs_u,    tokenize=False, add_generation_prompt=True)

    enc_full = processor(text=[text_full], images=[image], return_tensors="pt")
    enc_u    = processor(text=[text_u],    images=[image], return_tensors="pt")

    enc_full = {k: v.to(MODEL_DEVICE) for k, v in enc_full.items()}
    enc_u    = {k: v.to(MODEL_DEVICE) for k, v in enc_u.items()}

    input_ids = enc_full["input_ids"]
    labels = input_ids.clone()

    prefix_len = int(enc_u["input_ids"].shape[1])
    if input_ids.shape[1] <= prefix_len:
        return None, 0

    labels[:, :prefix_len] = -100

    if MAX_TOKENS_CAP is not None:
        cap = int(MAX_TOKENS_CAP)
        keep_len = min(input_ids.shape[1], prefix_len + cap)
        for k in list(enc_full.keys()):
            if isinstance(enc_full[k], torch.Tensor) and enc_full[k].dim() >= 2 and enc_full[k].shape[1] == input_ids.shape[1]:
                enc_full[k] = enc_full[k][:, :keep_len]
        input_ids = input_ids[:, :keep_len]
        labels    = labels[:, :keep_len]

    enc_full["input_ids"] = input_ids
    enc_full["labels"] = labels

    shift_labels = labels[:, 1:]
    assistant_tokens = int((shift_labels != -100).sum().item())
    return enc_full, assistant_tokens

# ============================================================
# 5) ONLY CHANGE: choose gradient param set (GradAudit-style parsing)
# ============================================================
_RE_TEXT = re.compile(r"model\.layers\.(\d+)\.")
_RE_VIS  = re.compile(r"visual\.blocks\.(\d+)\.")

def _is_lora_name(name: str) -> bool:
    return ("lora" in name.lower())

def select_scoring_params(model, scope: str = "vt_last3", last_k: int = 3, exclude_lora: bool = True):
    """
    Scope options:
      - vt_last3: Vision last3 blocks + Text last3 layers
      - vt_all  : Vision all blocks + Text all layers
    Uses parameter-name parsing like your GradAudit script.
    """
    # freeze all
    for p in model.parameters():
        p.requires_grad_(False)

    # collect indices
    text_indices = set()
    vis_indices  = set()
    for name, _ in model.named_parameters():
        m = _RE_TEXT.search(name)
        if m:
            text_indices.add(int(m.group(1)))
        m = _RE_VIS.search(name)
        if m:
            vis_indices.add(int(m.group(1)))

    if not text_indices:
        raise RuntimeError("❌ Cannot find any 'model.layers.{i}.' in parameter names. Print a few named_parameters to inspect.")
    if not vis_indices:
        raise RuntimeError("❌ Cannot find any 'visual.blocks.{j}.' in parameter names. Print a few named_parameters to inspect.")

    max_text  = max(text_indices)
    max_vis   = max(vis_indices)

    if scope == "vt_all":
        text_start  = min(text_indices)
        vis_start   = min(vis_indices)
    elif scope == "vt_last3":
        text_start  = max(0, max_text - (last_k - 1))
        vis_start   = max(0, max_vis  - (last_k - 1))
    else:
        raise ValueError(f"Unknown scope: {scope}")

    print(f"🧪 Layer scope={scope}: Text [{text_start}-{max_text}] | Vision [{vis_start}-{max_vis}] | exclude_lora={exclude_lora}")

    scoring_params = []
    enabled_count = 0
    for name, p in model.named_parameters():
        if p.ndim < 2:
            continue
        if exclude_lora and _is_lora_name(name):
            continue

        enable = False
        tm = _RE_TEXT.search(name)
        if tm and int(tm.group(1)) >= text_start:
            enable = True
        vm = _RE_VIS.search(name)
        if vm and int(vm.group(1)) >= vis_start:
            enable = True

        if enable:
            p.requires_grad_(True)
            scoring_params.append(p)
            enabled_count += 1

    if enabled_count == 0:
        raise RuntimeError("❌ No scoring params enabled. Something wrong with name patterns / scope.")

    # de-dup by id
    uniq = []
    seen = set()
    for p in scoring_params:
        if id(p) not in seen:
            uniq.append(p)
            seen.add(id(p))
    return uniq

SCORING_PARAMS = select_scoring_params(
    model,
    scope=GRAD_PARAM_SCOPE,
    last_k=LAST_K,
    exclude_lora=True   # ✅ default: exclude LoRA (weaken baseline, avoid oracle)
)
print(f"✅ Scoring params enabled: {len(SCORING_PARAMS)}")

# ============================================================
# 6) GradNorm score (standard L2)
# ============================================================
@torch.enable_grad()
def gradnorm_score_one(image: Image.Image, question: str, answer: str) -> Optional[float]:
    packed, t_assist = pack_qwen(image, question, answer)
    if packed is None or t_assist <= 0:
        return None

    model.zero_grad(set_to_none=True)

    if USE_AMP and torch.cuda.is_available():
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            out = model(**packed)
            loss = out.loss
    else:
        out = model(**packed)
        loss = out.loss

    if loss is None or not torch.isfinite(loss).item():
        return None

    loss.backward()

    sq_sum = 0.0
    any_grad = False
    for p in SCORING_PARAMS:
        g = p.grad
        if g is None:
            continue
        g = g.detach()
        if not torch.isfinite(g).all():
            continue
        gf = g.float()
        sq_sum += float(torch.sum(gf * gf).item())
        any_grad = True

    if not any_grad:
        return None

    gn = math.sqrt(max(sq_sum, 0.0))
    if LENGTH_NORM:
        gn = gn / math.sqrt(float(t_assist) + 1e-12)

    # Higher score => more likely member
    return float(-gn)

# ============================================================
# 7) Run loop
# ============================================================
def process_side(data: List[Dict[str,Any]], root: str, label: int):
    scores = []
    pbar = tqdm(data, desc=f"Label {label}")
    for item in pbar:
        try:
            if not item.get("images"):
                continue
            img = load_img(root, item["images"][0])
            if img is None:
                continue

            q = robust_extract_text(item["messages"][0]["content"])
            a = robust_extract_text(item["messages"][1]["content"])
            if not a.strip():
                continue

            s = gradnorm_score_one(img, q, a)
            if s is None or (not math.isfinite(s)):
                continue
            scores.append(float(s))

            if len(scores) % 50 == 0:
                pbar.set_postfix({"kept": len(scores), "avg": float(np.mean(scores))})
        except Exception:
            continue
    return scores

print("\n⏳ Computing GradNorm scores (members)...")
mem_scores = process_side(m_data, MEMBER_DIR, 1)
print("⏳ Computing GradNorm scores (nonmembers)...")
non_scores = process_side(n_data, NONMEMBER_DIR, 0)

print(f"\n✅ Valid samples: M={len(mem_scores)} N={len(non_scores)}")
if len(mem_scores) == 0 or len(non_scores) == 0:
    raise RuntimeError("No valid scores collected.")

# ============================================================
# 8) Metrics
# ============================================================
y_true = np.array([1]*len(mem_scores) + [0]*len(non_scores), dtype=np.int32)
scores = np.array(mem_scores + non_scores, dtype=np.float64)

def metrics_at_target_fpr(y_true: np.ndarray, scores: np.ndarray, target_fpr: float = 0.05):
    fpr, tpr, thr = roc_curve(y_true, scores)
    idx = np.searchsorted(fpr, target_fpr, side="right")
    if idx == 0:
        thr_star = thr[0]; tpr_star = tpr[0]
    elif idx >= len(thr):
        thr_star = thr[-1]; tpr_star = tpr[-1]
    else:
        x0, x1 = fpr[idx-1], fpr[idx]
        y0, y1 = tpr[idx-1], tpr[idx]
        t0, t1 = thr[idx-1], thr[idx]
        w = (target_fpr - x0) / (x1 - x0 + 1e-12)
        thr_star = t0 + w*(t1 - t0)
        tpr_star = y0 + w*(y1 - y0)

    y_pred = (scores >= thr_star).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr_actual = fp / (fp + tn + 1e-12)
    tpr_actual = tp / (tp + fn + 1e-12)
    acc = accuracy_score(y_true, y_pred)
    return {
        "AUC": float(roc_auc_score(y_true, scores)),
        "thr@5%FPR": float(thr_star),
        "TPR@5%FPR(interp)": float(tpr_star),
        "FPR(actual)": float(fpr_actual),
        "TPR(actual)": float(tpr_actual),
        "Precision": float(prec),
        "Recall": float(rec),
        "F1": float(f1),
        "Accuracy": float(acc),
        "CM": cm
    }

m5 = metrics_at_target_fpr(y_true, scores, target_fpr=0.05)

print("\n====== GradNorm MIA (Scope-controlled Baseline) ======")
print(f"Config: GRAD_PARAM_SCOPE={GRAD_PARAM_SCOPE}, LAST_K={LAST_K}, exclude_lora=True")
print(f"AUC                 = {m5['AUC']:.4f}")
print(f"thr @5%FPR          = {m5['thr@5%FPR']:.6f}")
print(f"TPR@5%FPR (interp)  = {m5['TPR@5%FPR(interp)']:.4f}")
print(f"Accuracy            = {m5['Accuracy']*100:.2f}%")
print("Confusion [[TN FP][FN TP]]：")
print(m5["CM"])

# ============================================================
# 9) Save outputs
# ============================================================
out_json = os.path.join(OUTPUT_DIR, f"gradnorm_scope_{GRAD_PARAM_SCOPE}_scores_strict.json")
payload = {
    "meta": {
        "method": "GradNorm (scope-controlled; LoRA excluded)",
        "grad_param_scope": GRAD_PARAM_SCOPE,
        "last_k": LAST_K,
        "length_norm": bool(LENGTH_NORM),
        "max_tokens_cap": MAX_TOKENS_CAP,
        "train_size": TRAIN_SIZE,
        "seed": SEED
    },
    "member_scores": mem_scores,
    "nonmember_scores": non_scores,
    "metrics": {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in m5.items()}
}
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

out_csv = os.path.join(OUTPUT_DIR, f"mia_gradnorm_scope_{GRAD_PARAM_SCOPE}_1000x1000.csv")
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["split", "score(-gradnorm)"])
    for s in mem_scores: w.writerow(["member", float(s)])
    for s in non_scores: w.writerow(["nonmember", float(s)])

print(f"\n💾 Saved JSON: {out_json}")
print(f"💾 Saved CSV : {out_csv}")


In [ ]:
# ==============================================================================
# NA-PDD (STRICT baseline) for FashionGen
# - Model: Qwen2-VL-2B-Instruct (local) + LoRA
# - Data : FashionGen strict training split reconstruction (same as your GradNorm code)
# - Baseline-consistent:
#     (1) threshold activation, (2) position-level map, (3) Counter.update by pos,
#     (4) reference_patterns -> layer_scores -> top_layers -> ratio score
# - Speed/RAM:
#     * hook only last3 vision + last3 text
#     * STREAMING calib counts (no saving all activations)
#     * STREAMING test scoring (no saving all activations)
# @title 8. NA-PDD MIA
# ==============================================================================

import os, json, math, random, re, warnings, csv, gc, time
from typing import Dict, Any, List, Optional, Tuple
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix

from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

warnings.filterwarnings("ignore")

# ============================================================
# 0) Paths & Config (same as your GradNorm script)
# ============================================================
DRIVE_ROOT   = "/content/drive/MyDrive/FashionGen_MIA"
BASE_MODEL   = "/content/Qwen2-VL-2B-Instruct"
LORA_PATH    = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"

MEMBER_DIR        = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_DIR     = f"{DRIVE_ROOT}/nonmember_5k"
MEMBERS_FULL_JSON = f"{MEMBER_DIR}/mllm_data.json"
NONMEMBERS_FULL_JSON = f"{NONMEMBER_DIR}/mllm_data.json"

OUTPUT_DIR = f"{DRIVE_ROOT}/results/na_pdd_fashiongen_1k_2b"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# strict split
TRAIN_SIZE   = 1000
SEED         = 42
CONFIG_SEED  = SEED + 123

# NA-PDD baseline hyperparams
ACTIVATION_THRESHOLD = 0.0
ALPHA = 1.5                   # member-dominant if f_mem > f_non * alpha
TOP_LAYERS = 6                # last3+last3 =>最多6层，top_layers<=6更合理
BATCH_SIZE = 8                # 你可用 8/16，看显存
USE_AMP = True
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float16

# Hook last-N layers (match your "只要最后3层到5层"需求，这里默认3)
VISION_LAST = 3
TEXT_LAST   = 3

# Optional cap to prevent extreme long sequences (does NOT change baseline logic, only prevents OOM)
# If you insist "absolutely no change", set MAX_SEQ_LEN=None.
MAX_SEQ_LEN = None

# Score threshold (baseline often uses >=1.0 as member)
RATIO_THRESHOLD_FOR_ACC = 1.0

# ============================================================
# 1) Utilities: strict split reconstruction
# ============================================================
print("🔄 Reconstructing exact training split (FashionGen)...")

def _load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def get_exact_training_data() -> Tuple[List[Dict[str,Any]], List[Dict[str,Any]]]:
    all_members = _load_json(MEMBERS_FULL_JSON)
    rng = random.Random(CONFIG_SEED)
    rng.shuffle(all_members)
    real_members = all_members[:TRAIN_SIZE]

    all_non = _load_json(NONMEMBERS_FULL_JSON)
    rng = random.Random(CONFIG_SEED)
    rng.shuffle(all_non)
    real_non = all_non[:TRAIN_SIZE]

    print(f"✅ Strict split reconstructed: member={len(real_members)} nonmember={len(real_non)}")
    return real_members, real_non

m_data, n_data = get_exact_training_data()

# ============================================================
# 2) Load model (base + LoRA)
#    注意：NA-PDD不需要grad，但我们沿用你LoRA的加载方式（eval）
# ============================================================
def _ensure_local_model_dir(path: str):
    if not os.path.isdir(path):
        raise RuntimeError(f"BASE_MODEL directory not found: {path}")

_ensure_local_model_dir(BASE_MODEL)

print(f"\n🚀 Loading Qwen2-VL base from: {BASE_MODEL}")
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True
)

print(f"🔧 Loading LoRA from: {LORA_PATH}")
# 这里 is_trainable 不影响 NA-PDD（no_grad forward），但保持与你当前环境一致
model = PeftModel.from_pretrained(base_model, LORA_PATH, is_trainable=False)
model.eval()

def _model_device(m) -> torch.device:
    try:
        return next(m.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_DEVICE = _model_device(model)
print(f"✅ Model loaded. device={MODEL_DEVICE}, dtype={next(model.parameters()).dtype}")

# ============================================================
# 3) Data helpers
# ============================================================
_IMAGE_TOKEN_RE = re.compile(r"<\s*image\s*>", flags=re.IGNORECASE)

def strip_image_token(s: str) -> str:
    if not isinstance(s, str): return ""
    s = _IMAGE_TOKEN_RE.sub("", s)
    s = s.replace("\n", " ").strip()
    return re.sub(r"\s+", " ", s).strip()

def robust_extract_text(c) -> str:
    if isinstance(c, str): return strip_image_token(c)
    if isinstance(c, list):
        parts = []
        for seg in c:
            if isinstance(seg, dict) and seg.get("type") == "text" and seg.get("text"):
                parts.append(seg["text"])
            elif isinstance(seg, str):
                parts.append(seg)
        return strip_image_token(" ".join(parts))
    return strip_image_token(str(c))

def load_img(root: str, rel_path: str) -> Optional[Image.Image]:
    try:
        p = os.path.join(root, rel_path)
        if os.path.exists(p):
            return Image.open(p).convert("RGB")
        return None
    except Exception:
        return None

# ============================================================
# 4) Pack Qwen inputs (same logic as your GradNorm, but no-grad forward)
# ============================================================
def pack_qwen_for_forward(image: Image.Image, question: str, answer: str) -> Optional[Dict[str, torch.Tensor]]:
    msgs_full = [
        {"role":"user", "content":[{"type":"image"}, {"type":"text", "text":question}]},
        {"role":"assistant", "content":[{"type":"text", "text":answer}]}
    ]

    text_full = processor.apply_chat_template(
        msgs_full, tokenize=False, add_generation_prompt=False
    )

    # IMPORTANT: avoid truncation that can break multimodal token alignment
    enc = processor(
        text=[text_full],
        images=[image],
        return_tensors="pt",
        padding=False,
        truncation=False
    )

    enc = {k: v.to(MODEL_DEVICE, non_blocking=True) if isinstance(v, torch.Tensor) else v for k, v in enc.items()}

    if MAX_SEQ_LEN is not None and "input_ids" in enc:
        L = enc["input_ids"].shape[1]
        if L > int(MAX_SEQ_LEN):
            keep = int(MAX_SEQ_LEN)
            for k in list(enc.keys()):
                if isinstance(enc[k], torch.Tensor) and enc[k].dim() >= 2 and enc[k].shape[1] == L:
                    enc[k] = enc[k][:, :keep]
    return enc

# ============================================================
# 5) Locate vision/text layers robustly (auto)
# ============================================================
def _find_modulelist_by_keywords(root: nn.Module, must_have: List[str], min_len: int = 2) -> Optional[Tuple[str, nn.ModuleList]]:
    # returns (path, modulelist)
    for name, mod in root.named_modules():
        if isinstance(mod, nn.ModuleList) and len(mod) >= min_len:
            lname = name.lower()
            ok = True
            for kw in must_have:
                if kw.lower() not in lname:
                    ok = False
                    break
            if ok:
                return name, mod
    return None

def locate_vision_blocks(m: nn.Module) -> Tuple[str, nn.ModuleList]:
    # Try common Qwen2-VL patterns first
    candidates = [
        (["vision", "encoder", "layers"], 4),
        (["vision_model", "encoder", "layers"], 4),
        (["vision", "blocks"], 4),
        (["visual", "blocks"], 4),
    ]
    for must, mn in candidates:
        found = _find_modulelist_by_keywords(m, must_have=must, min_len=mn)
        if found:
            return found[0], found[1]
    # fallback: any long ModuleList containing "vision"
    found = _find_modulelist_by_keywords(m, must_have=["vision"], min_len=4)
    if found:
        return found[0], found[1]
    raise RuntimeError("❌ Cannot locate vision blocks ModuleList. Please print model.named_modules() around vision.")

def locate_text_layers(m: nn.Module) -> Tuple[str, nn.ModuleList]:
    candidates = [
        (["language_model", "layers"], 4),
        (["model", "layers"], 4),
        (["transformer", "layers"], 4),
    ]
    for must, mn in candidates:
        found = _find_modulelist_by_keywords(m, must_have=must, min_len=mn)
        if found:
            return found[0], found[1]
    # fallback: any long ModuleList containing "layers" and "model"
    found = _find_modulelist_by_keywords(m, must_have=["layers"], min_len=4)
    if found:
        return found[0], found[1]
    raise RuntimeError("❌ Cannot locate text layers ModuleList. Please print model.named_modules() around language model.")

vision_path, vision_blocks = locate_vision_blocks(model)
text_path, text_layers = locate_text_layers(model)

print("\n" + "="*80)
print("🔍 Model architecture check (auto)")
print("="*80)
print(f"vision_blocks: n={len(vision_blocks)}  path={vision_path}")
print(f"text_layers : n={len(text_layers)}   path={text_path}")

# ============================================================
# 6) Register NA-PDD hooks (STRICT position-level, no token-OR)
# ============================================================
batch_activated_pos: Dict[str, List[Dict[int, List[int]]]] = {}

def _hook_collect_pos_indices(layer_name: str):
    """
    Strict baseline:
      - if out: [B,S,H], store per position activated neuron indices
      - if out: [B,H], store pos=0
    We ONLY move indices to CPU (int32), not whole tensors.
    """
    def hook(module, inp, out):
        if isinstance(out, (tuple, list)):
            out = out[0]
        if not torch.is_tensor(out):
            return
        t = out.detach()

        if t.ndim == 3:
            # [B,S,H]
            m = (t > ACTIVATION_THRESHOLD)
            B, S, H = m.shape
            per_sample: List[Dict[int, List[int]]] = []
            for b in range(B):
                pos_map: Dict[int, List[int]] = {}
                mb = m[b]  # [S,H]
                for pos in range(S):
                    row = mb[pos]
                    if row.any():
                        idx = torch.nonzero(row, as_tuple=False).squeeze(1)
                        if idx.numel() > 0:
                            pos_map[pos] = idx.to("cpu", dtype=torch.int32).tolist()
                per_sample.append(pos_map)
            batch_activated_pos[layer_name] = per_sample
            return

        if t.ndim == 2:
            # [B,H] => pos=0
            m = (t > ACTIVATION_THRESHOLD)
            B, H = m.shape
            per_sample: List[Dict[int, List[int]]] = []
            for b in range(B):
                idx = torch.nonzero(m[b], as_tuple=False).squeeze(1)
                if idx.numel() > 0:
                    per_sample.append({0: idx.to("cpu", dtype=torch.int32).tolist()})
                else:
                    per_sample.append({})
            batch_activated_pos[layer_name] = per_sample
            return

        # other shapes -> ignore (keeps baseline clean)
        return

    return hook

def _get_mlp_module(block: nn.Module) -> Optional[nn.Module]:
    # Try common names
    for attr in ["mlp", "feed_forward", "ffn", "mlp_fc", "ff"]:
        if hasattr(block, attr):
            return getattr(block, attr)
    return None

def register_hooks_last_layers(m: nn.Module, v_last: int, t_last: int) -> Tuple[List[Any], List[str]]:
    hooks = []
    names = []

    # vision last layers
    v_start = max(0, len(vision_blocks) - v_last)
    for i in range(v_start, len(vision_blocks)):
        blk = vision_blocks[i]
        mlp = _get_mlp_module(blk)
        if mlp is None:
            # fallback: hook the whole block output (still valid activations)
            h = blk.register_forward_hook(_hook_collect_pos_indices(f"vision::L{i}::block"))
            hooks.append(h); names.append(f"vision::L{i}::block")
        else:
            h = mlp.register_forward_hook(_hook_collect_pos_indices(f"vision::L{i}::mlp"))
            hooks.append(h); names.append(f"vision::L{i}::mlp")

    # text last layers
    t_start = max(0, len(text_layers) - t_last)
    for i in range(t_start, len(text_layers)):
        lyr = text_layers[i]
        mlp = _get_mlp_module(lyr)
        if mlp is None:
            h = lyr.register_forward_hook(_hook_collect_pos_indices(f"text::L{i}::block"))
            hooks.append(h); names.append(f"text::L{i}::block")
        else:
            h = mlp.register_forward_hook(_hook_collect_pos_indices(f"text::L{i}::mlp"))
            hooks.append(h); names.append(f"text::L{i}::mlp")

    print("\n" + "="*80)
    print(f"✅ Hook plan: vision[{v_start}..{len(vision_blocks)-1}] text[{t_start}..{len(text_layers)-1}]")
    print(f"✅ Registered hooks: {len(hooks)}")
    print(f"   examples: {names[:6]}")
    return hooks, names

hooks, hook_names = register_hooks_last_layers(model, VISION_LAST, TEXT_LAST)

# ============================================================
# 7) Run forward for a batch and return per-sample posmaps (for streaming)
# ============================================================
def run_forward_collect_posmaps(batch_items: List[Tuple[Image.Image, str, str]]) -> List[Dict[str, Dict[int, List[int]]]]:
    """
    Input: list of (image, question, answer)
    Output: list length=B, each:
        { layer_name: {pos: [neuron_ids]} }
    """
    batch_activated_pos.clear()

    # Qwen2-VL processor doesn't support real batching with variable images reliably if you pre-stack tensors;
    # We'll just do per-sample forward in a micro-batch loop (keeps baseline stable).
    # Speed is dominated by model forward anyway; last3 hooks minimizes overhead.
    outs = []

    for (img, q, a) in batch_items:
        enc = pack_qwen_for_forward(img, q, a)
        if enc is None:
            outs.append({})
            continue

        # clear hook storage for this single sample forward
        batch_activated_pos.clear()

        with torch.no_grad():
            if USE_AMP and torch.cuda.is_available():
                with torch.autocast(device_type="cuda", dtype=DTYPE):
                    _ = model(**enc)
            else:
                _ = model(**enc)

        # For single sample, hook stored per_sample_list length should be 1
        per_sample_map: Dict[str, Dict[int, List[int]]] = {}
        for lname, per_list in batch_activated_pos.items():
            if isinstance(per_list, list) and len(per_list) >= 1:
                pm = per_list[0]
                if pm:
                    per_sample_map[lname] = pm
        outs.append(per_sample_map)

    return outs

# ============================================================
# 8) Streaming calib: build counters per layer (position-level Counter.update)
# ============================================================
def iter_valid_examples(data: List[Dict[str,Any]], root_dir: str):
    """
    yields (image, question, answer)
    - question: from messages[0]
    - answer  : from messages[1]
    """
    for item in data:
        try:
            imgs = item.get("images", None)
            if not imgs:
                continue
            img = load_img(root_dir, imgs[0])
            if img is None:
                continue
            msgs = item.get("messages", None)
            if not isinstance(msgs, list) or len(msgs) < 2:
                continue
            q = robust_extract_text(msgs[0].get("content", ""))
            a = robust_extract_text(msgs[1].get("content", ""))
            if not a.strip():
                continue
            yield (img, q, a)
        except Exception:
            continue

def build_layer_counters_stream(data: List[Dict[str,Any]], root_dir: str, tag: str) -> Dict[str, Counter]:
    """
    Returns:
      layer -> Counter(neuron_id -> count)  where count aggregates across POSITIONS (strict baseline)
    """
    layer_cnts: Dict[str, Counter] = {}
    ok = 0

    pbar = tqdm(iter_valid_examples(data, root_dir), desc=f"CalibCollect({tag})", total=len(data), dynamic_ncols=True)
    micro_batch: List[Tuple[Image.Image, str, str]] = []

    for ex in pbar:
        micro_batch.append(ex)
        if len(micro_batch) < BATCH_SIZE:
            continue

        posmaps_list = run_forward_collect_posmaps(micro_batch)
        for posmaps in posmaps_list:
            if not posmaps:
                continue
            ok += 1
            for layer, pos_dict in posmaps.items():
                if layer not in layer_cnts:
                    layer_cnts[layer] = Counter()
                # STRICT: update by EACH position's neuron list
                for neu_list in pos_dict.values():
                    layer_cnts[layer].update(neu_list)

        micro_batch = []
        if ok % 50 == 0:
            pbar.set_postfix({"ok": ok})
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # flush remaining
    if micro_batch:
        posmaps_list = run_forward_collect_posmaps(micro_batch)
        for posmaps in posmaps_list:
            if not posmaps:
                continue
            ok += 1
            for layer, pos_dict in posmaps.items():
                if layer not in layer_cnts:
                    layer_cnts[layer] = Counter()
                for neu_list in pos_dict.values():
                    layer_cnts[layer].update(neu_list)

    print(f"✅ CalibCollect({tag}) done: ok={ok}/{len(data)} layers={len(layer_cnts)}")
    return layer_cnts

# ============================================================
# 9) Reference patterns + layer scores + top layers (baseline)
# ============================================================
def build_reference_patterns(
    mem_cnts: Dict[str, Counter],
    non_cnts: Dict[str, Counter],
    n_mem_samples: int,
    n_non_samples: int,
    alpha: float = ALPHA
) -> Dict[str, Any]:
    """
    For each layer:
      member_freq[n] = count / n_mem_samples
      non_freq[n]    = count / n_non_samples
      member_dominant: f_mem > f_non * alpha or neuron absent in non
      nonmember_dominant: symmetric
    """
    layers = set(mem_cnts.keys()) | set(non_cnts.keys())
    ref = {}

    for layer in layers:
        mc = mem_cnts.get(layer, Counter())
        nc = non_cnts.get(layer, Counter())

        mem_freq = {n: c / max(1, n_mem_samples) for n, c in mc.items()}
        non_freq = {n: c / max(1, n_non_samples) for n, c in nc.items()}

        member_dom = {}
        for n, f in mem_freq.items():
            if (n not in non_freq) or (f > non_freq[n] * alpha):
                member_dom[n] = f

        non_dom = {}
        for n, f in non_freq.items():
            if (n not in mem_freq) or (f > mem_freq[n] * alpha):
                non_dom[n] = f

        ref[layer] = {
            "member_dominant": member_dom,
            "nonmember_dominant": non_dom,
            "member_freq": mem_freq,
            "non_freq": non_freq
        }

    return ref

def calculate_layer_scores(reference_patterns: Dict[str,Any]) -> Dict[str, float]:
    return {layer: (len(d["member_dominant"]) - len(d["nonmember_dominant"])) for layer, d in reference_patterns.items()}

def select_top_layers(layer_scores: Dict[str,float], top_n: int) -> List[str]:
    # baseline: abs(score) desc
    sorted_layers = sorted(layer_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    return [k for k,_ in sorted_layers[:top_n]]

# ============================================================
# 10) Ratio score per sample (baseline)
# ============================================================
def ratio_score_from_posmaps(posmaps: Dict[str, Dict[int, List[int]]],
                             reference_patterns: Dict[str,Any],
                             layers: List[str]) -> float:
    layers_counted = 0
    total_mem = 0.0
    total_non = 0.0

    for layer in layers:
        if layer not in posmaps:
            continue
        if layer not in reference_patterns:
            continue

        pos_dict = posmaps[layer]
        if not pos_dict:
            continue

        # baseline scoring uses union over positions for overlap
        sample_neurons = set()
        for neu in pos_dict.values():
            sample_neurons.update(neu)
        if not sample_neurons:
            continue

        mem_dom = set(reference_patterns[layer]["member_dominant"].keys())
        non_dom = set(reference_patterns[layer]["nonmember_dominant"].keys())

        mem_overlap = len(sample_neurons & mem_dom)
        non_overlap = len(sample_neurons & non_dom)

        mem_ratio = mem_overlap / len(mem_dom) if len(mem_dom) > 0 else 0.0
        non_ratio = non_overlap / len(non_dom) if len(non_dom) > 0 else 0.0

        total_mem += mem_ratio
        total_non += non_ratio
        layers_counted += 1

    if layers_counted == 0:
        return 0.0

    avg_mem = total_mem / layers_counted
    avg_non = total_non / layers_counted
    if avg_non == 0:
        return float("inf")
    return avg_mem / avg_non

# ============================================================
# 11) Test streaming scoring
# ============================================================
def score_dataset_stream(data: List[Dict[str,Any]], root_dir: str,
                         reference_patterns: Dict[str,Any], layers: List[str],
                         label: int, tag: str) -> Tuple[List[float], List[int], int]:
    scores, labels = [], []
    ok = 0

    pbar = tqdm(iter_valid_examples(data, root_dir), desc=f"TestScore({tag})", total=len(data), dynamic_ncols=True)
    micro_batch: List[Tuple[Image.Image, str, str]] = []

    for ex in pbar:
        micro_batch.append(ex)
        if len(micro_batch) < BATCH_SIZE:
            continue

        posmaps_list = run_forward_collect_posmaps(micro_batch)
        for posmaps in posmaps_list:
            if not posmaps:
                continue
            ok += 1
            r = ratio_score_from_posmaps(posmaps, reference_patterns, layers)
            if r == float("inf") or (isinstance(r, float) and (math.isnan(r) or not math.isfinite(r))):
                r = 1e6
            scores.append(float(r))
            labels.append(int(label))

        micro_batch = []
        if ok % 50 == 0:
            pbar.set_postfix({"ok": ok})
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # flush
    if micro_batch:
        posmaps_list = run_forward_collect_posmaps(micro_batch)
        for posmaps in posmaps_list:
            if not posmaps:
                continue
            ok += 1
            r = ratio_score_from_posmaps(posmaps, reference_patterns, layers)
            if r == float("inf") or (isinstance(r, float) and (math.isnan(r) or not math.isfinite(r))):
                r = 1e6
            scores.append(float(r))
            labels.append(int(label))

    print(f"✅ TestScore({tag}) done: ok={ok}/{len(data)}")
    return scores, labels, ok

def tpr_at_fpr_5(y: np.ndarray, s: np.ndarray, target_fpr: float = 0.05) -> float:
    fpr, tpr, _ = roc_curve(y, s)
    idx = np.where(fpr >= target_fpr)[0]
    if len(idx) == 0:
        return 0.0
    return float(tpr[idx[0]])

# ============================================================
# 12) Main
# ============================================================
print("\n" + "="*90)
print("🚀 NA-PDD (STRICT baseline) — FashionGen (Qwen2-VL-2B + LoRA) 1k/1k")
print("="*90)
print(f"thr={ACTIVATION_THRESHOLD} alpha={ALPHA} top_layers={TOP_LAYERS} batch={BATCH_SIZE}")
print(f"hook_last(V/T)={VISION_LAST}/{TEXT_LAST}")

# Calib sets (use the same strict 1k split; we can do 200/200 calib like your earlier pipelines)
CALIB_MEM = 200
CALIB_NON = 200
TEST_MEM  = 800
TEST_NON  = 800

mem_calib = m_data[:CALIB_MEM]
non_calib = n_data[:CALIB_NON]
mem_test  = m_data[CALIB_MEM:CALIB_MEM+TEST_MEM]
non_test  = n_data[CALIB_NON:CALIB_NON+TEST_NON]

print(f"\n✅ Split: calib M={len(mem_calib)} N={len(non_calib)} | test M={len(mem_test)} N={len(non_test)}")

# ---- Calib streaming counters ----
mem_cnts = build_layer_counters_stream(mem_calib, MEMBER_DIR, tag="M")
non_cnts = build_layer_counters_stream(non_calib, NONMEMBER_DIR, tag="N")

# Build reference patterns
reference_patterns = build_reference_patterns(
    mem_cnts, non_cnts,
    n_mem_samples=len(mem_calib),
    n_non_samples=len(non_calib),
    alpha=ALPHA
)
layer_scores = calculate_layer_scores(reference_patterns)

# Pick top layers (cap by available layers)
top_n = min(TOP_LAYERS, len(layer_scores))
selected_layers = select_top_layers(layer_scores, top_n)

print("\n✅ Selected layers:")
for k in selected_layers:
    print(f"  - {k} score={layer_scores.get(k, 0)}")

# ---- Test streaming scores ----
m_scores, m_labels, m_ok = score_dataset_stream(mem_test, MEMBER_DIR, reference_patterns, selected_layers, label=1, tag="M")
n_scores, n_labels, n_ok = score_dataset_stream(non_test, NONMEMBER_DIR, reference_patterns, selected_layers, label=0, tag="N")

scores = np.array(m_scores + n_scores, dtype=np.float64)
labels = np.array(m_labels + n_labels, dtype=np.int32)

print(f"\n✅ Valid scored: M={int((labels==1).sum())}, N={int((labels==0).sum())}")

# Metrics
if len(np.unique(labels)) < 2 or len(scores) < 10:
    auc = 0.5
    tpr5 = 0.0
else:
    auc = float(roc_auc_score(labels, scores))
    tpr5 = tpr_at_fpr_5(labels, scores, target_fpr=0.05)

pred = (scores >= RATIO_THRESHOLD_FOR_ACC).astype(np.int32)
acc = float(accuracy_score(labels, pred)) if len(scores) > 0 else 0.0
cm = confusion_matrix(labels, pred) if len(scores) > 0 else np.array([[0,0],[0,0]])

print("\n" + "="*90)
print("📊 NA-PDD Results (STRICT baseline)")
print("="*90)
print(f"AUC       = {auc:.4f}")
print(f"TPR@5%FPR = {tpr5*100:.2f}%")
print(f"ACC(ratio>=1.0) = {acc:.4f}")
print("CM [[TN FP][FN TP]]:")
print(cm)

# ---- Save outputs ----
out_json = os.path.join(OUTPUT_DIR, "na_pdd_scores_strict_1k.json")
payload = {
    "meta": {
        "method": "NA-PDD strict baseline (position-level counts)",
        "activation_threshold": ACTIVATION_THRESHOLD,
        "alpha": ALPHA,
        "top_layers": int(top_n),
        "batch_size": BATCH_SIZE,
        "vision_last": VISION_LAST,
        "text_last": TEXT_LAST,
        "train_size": TRAIN_SIZE,
        "config_seed": CONFIG_SEED,
        "calib_mem": CALIB_MEM,
        "calib_non": CALIB_NON,
        "test_mem": TEST_MEM,
        "test_non": TEST_NON,
        "base_model": BASE_MODEL,
        "lora_path": LORA_PATH
    },
    "selected_layers": selected_layers,
    "layer_scores": layer_scores,
    "metrics": {
        "AUC": auc,
        "TPR@5%FPR": tpr5,
        "ACC_ratio>=1": acc,
        "CM": cm.tolist()
    },
    "member_scores": m_scores,
    "nonmember_scores": n_scores
}
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

out_csv = os.path.join(OUTPUT_DIR, "na_pdd_mia_800x800.csv")
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["split", "ratio_score"])
    for s in m_scores:
        w.writerow(["member", float(s)])
    for s in n_scores:
        w.writerow(["nonmember", float(s)])

print(f"\n💾 Saved JSON: {out_json}")
print(f"💾 Saved CSV : {out_csv}")

# ---- Cleanup hooks ----
for h in hooks:
    try:
        h.remove()
    except Exception:
        pass

print("\n✅ Done.")


## zlib

In [ ]:
# ==============================================================================
# Zlib-Calibrated MIA for Qwen2-VL (FashionGen)
# - Score: Loss / Zlib_Compression_Ratio
# - Features: Auto-direction calibration, Correct Masking for Qwen2-VL
# ==============================================================================

import os, json, math, random, re, warnings, csv, zlib
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve

from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel

warnings.filterwarnings("ignore")

# ============================================================
# 0) Paths & Config
# ============================================================
DRIVE_ROOT   = "/content/drive/MyDrive/FashionGen_MIA"
BASE_MODEL   = "/content/Qwen2-VL-2B-Instruct"
LORA_PATH    = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"

MEMBER_DIR        = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_DIR     = f"{DRIVE_ROOT}/nonmember_5k"
MEMBERS_FULL_JSON = f"{MEMBER_DIR}/mllm_data.json"
NONMEMBERS_FULL_JSON = f"{NONMEMBER_DIR}/mllm_data.json"

OUTPUT_DIR = f"{DRIVE_ROOT}/results/zlib_mia_fashiongen_2b"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_SIZE   = 1000
SEED         = 42
DTYPE        = torch.bfloat16 # Qwen2-VL 推荐使用 bf16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# 1) Reconstruct Training Split
# ============================================================
def _load_json(path: str):
    with open(path, "r", encoding="utf-8") as f: return json.load(f)

print("🔄 Reconstructing splits...")
all_members = _load_json(MEMBERS_FULL_JSON)
random.Random(SEED+123).shuffle(all_members)
m_data = all_members[:TRAIN_SIZE]

all_non = _load_json(NONMEMBERS_FULL_JSON)
random.Random(SEED+123).shuffle(all_non)
n_data = all_non[:TRAIN_SIZE]

# ============================================================
# 2) Load Model
# ============================================================
print(f"🚀 Loading Qwen2-VL + LoRA...")
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()
MODEL_DEVICE = next(model.parameters()).device

# ============================================================
# 3) Core Zlib Logic
# ============================================================
def get_zlib_entropy(text: str) -> float:
    if not text: return 1.0
    text_bytes = text.encode('utf-8')
    # 压缩率 = 压缩后长度 / 原始长度。复杂文本压缩率更高（接近1）
    return len(zlib.compress(text_bytes, level=9)) / len(text_bytes)

def load_img(root: str, rel_path: str) -> Optional[Image.Image]:
    try:
        p = os.path.join(root, rel_path)
        return Image.open(p).convert("RGB") if os.path.exists(p) else None
    except: return None

@torch.inference_mode()
def compute_zlib_score(image: Image.Image, question: str, answer: str) -> Optional[float]:
    # 1. Prepare Qwen2-VL Inputs (Same as training)
    msgs_full = [
        {"role":"user", "content":[{"type":"image"}, {"type":"text", "text":question}]},
        {"role":"assistant", "content":[{"type":"text", "text":answer}]}
    ]
    msgs_u = [{"role":"user", "content":[{"type":"image"}, {"type":"text", "text":question}]}]

    text_full = processor.apply_chat_template(msgs_full, tokenize=False, add_generation_prompt=False)
    text_u    = processor.apply_chat_template(msgs_u,    tokenize=False, add_generation_prompt=True)

    enc_full = processor(text=[text_full], images=[image], return_tensors="pt").to(MODEL_DEVICE)
    enc_u    = processor(text=[text_u],    images=[image], return_tensors="pt").to(MODEL_DEVICE)

    labels = enc_full["input_ids"].clone()
    prefix_len = enc_u["input_ids"].shape[1]
    labels[:, :prefix_len] = -100 # Mask prompt tokens

    # 2. Get Loss
    with torch.cuda.amp.autocast(dtype=DTYPE):
        outputs = model(**enc_full, labels=labels)
        loss = outputs.loss.item()

    if not math.isfinite(loss): return None

    # 3. Zlib Calibration
    z_ratio = get_zlib_entropy(answer)
    return loss / max(z_ratio, 1e-6)

# ============================================================
# 4) Run Scoring Loop
# ============================================================
def run_scoring(data, root, label_name):
    scores = []
    for item in tqdm(data, desc=f"Scoring {label_name}"):
        img = load_img(root, item["images"][0])
        if img is None: continue

        # Extract text (Assuming standard message format)
        q = item["messages"][0]["content"]
        if isinstance(q, list): q = "".join([x['text'] for x in q if x['type']=='text'])
        a = item["messages"][1]["content"]
        if isinstance(a, list): a = "".join([x['text'] for x in q if x['type']=='text'])

        s = compute_zlib_score(img, q, a)
        if s is not None: scores.append(s)
    return np.array(scores)

print("\n⏳ Processing...")
m_scores_raw = run_scoring(m_data, MEMBER_DIR, "Members")
n_scores_raw = run_scoring(n_data, NONMEMBER_DIR, "Non-members")

# ============================================================
# 5) Auto-Calibrated Evaluation
# ============================================================
def evaluate_mia(m_s, n_s):
    m_med, n_med = np.median(m_s), np.median(n_s)
    y_true = np.concatenate([np.ones(len(m_s)), np.zeros(len(n_s))])
    all_scores = np.concatenate([m_s, n_s])

    # Standard: Member Loss/Zlib should be LOWER than Non-member
    if m_med < n_med:
        print(f"✅ Direction Standard (M < N). Medians: M={m_med:.4f}, N={n_med:.4f}")
        final_scores = -all_scores # Invert for AUC (higher=member)
    else:
        print(f"⚠️ Direction Inverted (M > N). Medians: M={m_med:.4f}, N={n_med:.4f}")
        final_scores = all_scores

    auc = roc_auc_score(y_true, final_scores)
    fpr, tpr, _ = roc_curve(y_true, final_scores)
    tpr_at_5fpr = tpr[np.where(fpr <= 0.05)[0][-1]]

    return auc, tpr_at_5fpr, m_med, n_med

auc_val, tpr_val, m_m, n_m = evaluate_mia(m_scores_raw, n_scores_raw)

print("\n" + "="*50)
print(f"📊 Zlib MIA Results")
print(f"AUC:          {auc_val:.4f}")
print(f"TPR @ 5% FPR: {tpr_val*100:.2f}%")
print(f"Member Med:   {m_m:.4f}")
print(f"NonMem Med:   {n_m:.4f}")
print("="*50)

# ============================================================
# 6) Save
# ============================================================
res_file = os.path.join(OUTPUT_DIR, "zlib_mia_results.json")
with open(res_file, "w") as f:
    json.dump({
        "auc": auc_val, "tpr_5fpr": tpr_val,
        "m_median": m_m, "n_median": n_m,
        "m_scores": m_scores_raw.tolist(),
        "n_scores": n_scores_raw.tolist()
    }, f, indent=2)
print(f"💾 Results saved to {res_file}")

In [ ]:
# ==============================================================================
# Zlib-Calibrated MIA for Qwen2-VL (FashionGen)
# - Score: Loss / Zlib_Compression_Ratio
# - Features: Auto-direction calibration, Correct Masking for Qwen2-VL
# - Fixed: Text extraction bug, Added raw vs calibrated comparison
# ==============================================================================

import os, json, math, random, re, warnings, csv, zlib
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve

from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel

warnings.filterwarnings("ignore")

# ============================================================
# 0) Paths & Config
# ============================================================
DRIVE_ROOT   = "/content/drive/MyDrive/FashionGen_MIA"
BASE_MODEL   = "/content/Qwen2-VL-2B-Instruct"
LORA_PATH    = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"

MEMBER_DIR        = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_DIR     = f"{DRIVE_ROOT}/nonmember_5k"
MEMBERS_FULL_JSON = f"{MEMBER_DIR}/mllm_data.json"
NONMEMBERS_FULL_JSON = f"{NONMEMBER_DIR}/mllm_data.json"

OUTPUT_DIR = f"{DRIVE_ROOT}/results/zlib_mia_fashiongen_2b"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_SIZE   = 1000
SEED         = 42
DTYPE        = torch.bfloat16  # Qwen2-VL 推荐使用 bf16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# 1) Reconstruct Training Split
# ============================================================
def _load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("🔄 Reconstructing splits...")
all_members = _load_json(MEMBERS_FULL_JSON)
random.Random(SEED+123).shuffle(all_members)
m_data = all_members[:TRAIN_SIZE]

all_non = _load_json(NONMEMBERS_FULL_JSON)
random.Random(SEED+123).shuffle(all_non)
n_data = all_non[:TRAIN_SIZE]

print(f"✅ Member samples: {len(m_data)}")
print(f"✅ Non-member samples: {len(n_data)}")

# ============================================================
# 2) Load Model
# ============================================================
print(f"\n🚀 Loading Qwen2-VL + LoRA...")
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()
MODEL_DEVICE = next(model.parameters()).device
print(f"✅ Model loaded on {MODEL_DEVICE}")

# ============================================================
# 3) Core Zlib Logic
# ============================================================
def get_zlib_entropy(text: str) -> float:
    """
    计算文本的zlib压缩率
    返回值: compressed_size / original_size
    - 接近0: 高度可压缩（简单/重复文本）
    - 接近1: 难以压缩（复杂/随机文本）
    """
    if not text:
        return 1.0
    text_bytes = text.encode('utf-8')
    compressed_size = len(zlib.compress(text_bytes, level=9))
    original_size = len(text_bytes)
    return compressed_size / original_size

def load_img(root: str, rel_path: str) -> Optional[Image.Image]:
    try:
        p = os.path.join(root, rel_path)
        return Image.open(p).convert("RGB") if os.path.exists(p) else None
    except:
        return None

@torch.inference_mode()
def compute_zlib_score(image: Image.Image, question: str, answer: str) -> Optional[Tuple[float, float, float]]:
    """
    计算一个样本的原始loss、zlib ratio和calibrated score
    Returns: (raw_loss, zlib_ratio, calibrated_score) or None
    """
    # 1. Prepare Qwen2-VL Inputs (Same as training)
    msgs_full = [
        {"role":"user", "content":[{"type":"image"}, {"type":"text", "text":question}]},
        {"role":"assistant", "content":[{"type":"text", "text":answer}]}
    ]
    msgs_u = [
        {"role":"user", "content":[{"type":"image"}, {"type":"text", "text":question}]}
    ]

    text_full = processor.apply_chat_template(msgs_full, tokenize=False, add_generation_prompt=False)
    text_u    = processor.apply_chat_template(msgs_u,    tokenize=False, add_generation_prompt=True)

    enc_full = processor(text=[text_full], images=[image], return_tensors="pt").to(MODEL_DEVICE)
    enc_u    = processor(text=[text_u],    images=[image], return_tensors="pt").to(MODEL_DEVICE)

    labels = enc_full["input_ids"].clone()
    prefix_len = enc_u["input_ids"].shape[1]
    labels[:, :prefix_len] = -100  # Mask prompt tokens

    # 2. Get Loss
    with torch.cuda.amp.autocast(dtype=DTYPE):
        outputs = model(**enc_full, labels=labels)
        loss = outputs.loss.item()

    if not math.isfinite(loss):
        return None

    # 3. Zlib Calibration
    z_ratio = get_zlib_entropy(answer)
    calibrated_score = loss / max(z_ratio, 1e-6)

    return loss, z_ratio, calibrated_score

# ============================================================
# 4) Run Scoring Loop
# ============================================================
def run_scoring(data, root, label_name):
    """
    对数据集进行评分
    Returns: (raw_losses, zlib_ratios, calibrated_scores)
    """
    raw_losses = []
    zlib_ratios = []
    calib_scores = []

    for item in tqdm(data, desc=f"Scoring {label_name}"):
        img = load_img(root, item["images"][0])
        if img is None:
            continue

        # ✅ 修复：正确提取question和answer
        q = item["messages"][0]["content"]
        if isinstance(q, list):
            q = "".join([x['text'] for x in q if x.get('type')=='text'])
        elif not isinstance(q, str):
            q = str(q)

        a = item["messages"][1]["content"]
        if isinstance(a, list):
            # ✅ 关键修复：遍历a而不是q
            a = "".join([x['text'] for x in a if x.get('type')=='text'])
        elif not isinstance(a, str):
            a = str(a)

        result = compute_zlib_score(img, q, a)
        if result is not None:
            raw_loss, z_ratio, calib_score = result
            raw_losses.append(raw_loss)
            zlib_ratios.append(z_ratio)
            calib_scores.append(calib_score)

    return np.array(raw_losses), np.array(zlib_ratios), np.array(calib_scores)

print("\n⏳ Processing members...")
m_raw, m_zlib, m_calib = run_scoring(m_data, MEMBER_DIR, "Members")

print("⏳ Processing non-members...")
n_raw, n_zlib, n_calib = run_scoring(n_data, NONMEMBER_DIR, "Non-members")

print(f"\n✅ Successfully scored {len(m_raw)} members and {len(n_raw)} non-members")

# ============================================================
# 5) Auto-Calibrated Evaluation
# ============================================================
def evaluate_mia(m_s, n_s, score_name="Score"):
    """
    评估MIA性能，自动判断方向
    """
    m_med, n_med = np.median(m_s), np.median(n_s)
    m_mean, n_mean = np.mean(m_s), np.mean(n_s)

    y_true = np.concatenate([np.ones(len(m_s)), np.zeros(len(n_s))])
    all_scores = np.concatenate([m_s, n_s])

    # Standard: Member score应该更低（lower is more confident member）
    # 如果M < N，说明member分数更低，需要反转让AUC计算正确
    if m_med < n_med:
        direction = "Standard (M < N)"
        final_scores = -all_scores  # 反转：让member有更高的"member confidence"
    else:
        direction = "Inverted (M > N)"
        final_scores = all_scores

    auc = roc_auc_score(y_true, final_scores)
    fpr, tpr, thresholds = roc_curve(y_true, final_scores)

    # TPR at 5% FPR
    idx_5fpr = np.where(fpr <= 0.05)[0]
    tpr_at_5fpr = tpr[idx_5fpr[-1]] if len(idx_5fpr) > 0 else 0.0

    print(f"\n{'='*60}")
    print(f"📊 {score_name} Results")
    print(f"{'='*60}")
    print(f"Direction:      {direction}")
    print(f"AUC:            {auc:.4f}")
    print(f"TPR @ 5% FPR:   {tpr_at_5fpr*100:.2f}%")
    print(f"Member   - Median: {m_med:.4f}, Mean: {m_mean:.4f}")
    print(f"NonMem   - Median: {n_med:.4f}, Mean: {n_mean:.4f}")
    print(f"{'='*60}")

    return {
        "auc": float(auc),
        "tpr_5fpr": float(tpr_at_5fpr),
        "m_median": float(m_med),
        "n_median": float(n_med),
        "m_mean": float(m_mean),
        "n_mean": float(n_mean),
        "direction": direction
    }

# ============================================================
# 6) Evaluate & Compare
# ============================================================
print("\n" + "🔬 EVALUATION RESULTS ".center(60, "="))

# 评估原始loss
results_raw = evaluate_mia(m_raw, n_raw, "Raw Loss (Baseline)")

# 评估calibrated score
results_calib = evaluate_mia(m_calib, n_calib, "Zlib-Calibrated Score")

# 分析zlib ratios本身的分布
print(f"\n{'='*60}")
print(f"📈 Zlib Compression Ratio Statistics")
print(f"{'='*60}")
print(f"Member   - Median: {np.median(m_zlib):.4f}, Mean: {np.mean(m_zlib):.4f}")
print(f"NonMem   - Median: {np.median(n_zlib):.4f}, Mean: {np.mean(n_zlib):.4f}")
print(f"Note: Lower ratio = more compressible (simpler text)")
print(f"{'='*60}")

# 对比分析
print(f"\n{'='*60}")
print(f"🎯 COMPARISON: Raw vs Calibrated")
print(f"{'='*60}")
auc_diff = results_calib['auc'] - results_raw['auc']
tpr_diff = results_calib['tpr_5fpr'] - results_raw['tpr_5fpr']

print(f"AUC Improvement:          {auc_diff:+.4f} ({auc_diff*100:+.2f}%)")
print(f"TPR@5%FPR Improvement:    {tpr_diff:+.4f} ({tpr_diff*100:+.2f}%)")

if auc_diff > 0.01:
    print(f"✅ Zlib calibration IMPROVES MIA performance")
elif auc_diff < -0.01:
    print(f"⚠️  Zlib calibration DEGRADES MIA performance")
else:
    print(f"➖ Zlib calibration has MINIMAL effect")
print(f"{'='*60}")

# ============================================================
# 7) Save Results
# ============================================================
res_file = os.path.join(OUTPUT_DIR, "zlib_mia_results.json")
with open(res_file, "w") as f:
    json.dump({
        "raw_loss": results_raw,
        "zlib_calibrated": results_calib,
        "zlib_stats": {
            "m_median": float(np.median(m_zlib)),
            "m_mean": float(np.mean(m_zlib)),
            "n_median": float(np.median(n_zlib)),
            "n_mean": float(np.mean(n_zlib))
        },
        "improvement": {
            "auc_diff": float(auc_diff),
            "tpr_diff": float(tpr_diff)
        },
        "scores": {
            "m_raw": m_raw.tolist(),
            "n_raw": n_raw.tolist(),
            "m_zlib": m_zlib.tolist(),
            "n_zlib": n_zlib.tolist(),
            "m_calib": m_calib.tolist(),
            "n_calib": n_calib.tolist()
        }
    }, f, indent=2)

print(f"\n💾 Results saved to {res_file}")

# ============================================================
# 8) Additional Analysis: Correlation
# ============================================================
from scipy.stats import spearmanr

print(f"\n{'='*60}")
print(f"🔗 Correlation Analysis")
print(f"{'='*60}")

# Member样本中loss与zlib ratio的相关性
m_corr, m_pval = spearmanr(m_raw, m_zlib)
print(f"Member:    Loss vs Zlib Ratio")
print(f"           Spearman ρ = {m_corr:.4f} (p={m_pval:.4e})")

# Non-member样本中loss与zlib ratio的相关性
n_corr, n_pval = spearmanr(n_raw, n_zlib)
print(f"NonMember: Loss vs Zlib Ratio")
print(f"           Spearman ρ = {n_corr:.4f} (p={n_pval:.4e})")

if abs(m_corr) < 0.1 and abs(n_corr) < 0.1:
    print(f"\n💡 Insight: Loss与Zlib ratio相关性很弱，calibration可能不适用")
elif m_corr > 0.3 or n_corr > 0.3:
    print(f"\n💡 Insight: 存在正相关，说明复杂文本确实导致更高loss")
    print(f"   Calibration理论上应该有效")
print(f"{'='*60}")

print("\n✨ All done!")

In [ ]:
# ==============================================================================
# Text-Only Zlib MIA for Qwen2-VL (FashionGen)
# - Baseline: Text-only loss (without image encoding)
# - Calibrated: Text-only loss / Zlib_Compression_Ratio
# - Purpose: Test if text memorization exists when isolating from vision
# ==============================================================================

import os, json, math, random, re, warnings, csv, zlib
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve

from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel

warnings.filterwarnings("ignore")

# ============================================================
# 0) Paths & Config
# ============================================================
DRIVE_ROOT   = "/content/drive/MyDrive/FashionGen_MIA"
BASE_MODEL   = "/content/Qwen2-VL-2B-Instruct"
LORA_PATH    = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"

MEMBER_DIR        = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_DIR     = f"{DRIVE_ROOT}/nonmember_5k"
MEMBERS_FULL_JSON = f"{MEMBER_DIR}/mllm_data.json"
NONMEMBERS_FULL_JSON = f"{NONMEMBER_DIR}/mllm_data.json"

OUTPUT_DIR = f"{DRIVE_ROOT}/results/text_only_zlib_mia_fashiongen_2b"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_SIZE   = 1000
SEED         = 42
DTYPE        = torch.bfloat16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# 1) Reconstruct Training Split
# ============================================================
def _load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("🔄 Reconstructing splits...")
all_members = _load_json(MEMBERS_FULL_JSON)
random.Random(SEED+123).shuffle(all_members)
m_data = all_members[:TRAIN_SIZE]

all_non = _load_json(NONMEMBERS_FULL_JSON)
random.Random(SEED+123).shuffle(all_non)
n_data = all_non[:TRAIN_SIZE]

print(f"✅ Member samples: {len(m_data)}")
print(f"✅ Non-member samples: {len(n_data)}")

# ============================================================
# 2) Load Model
# ============================================================
print(f"\n🚀 Loading Qwen2-VL + LoRA...")
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()
MODEL_DEVICE = next(model.parameters()).device
print(f"✅ Model loaded on {MODEL_DEVICE}")

# ============================================================
# 3) Core Zlib Logic
# ============================================================
def get_zlib_entropy(text: str) -> float:
    """
    计算文本的zlib压缩率
    返回值: compressed_size / original_size
    - 接近0: 高度可压缩（简单/重复文本）
    - 接近1: 难以压缩（复杂/随机文本）
    """
    if not text:
        return 1.0
    text_bytes = text.encode('utf-8')
    compressed_size = len(zlib.compress(text_bytes, level=9))
    original_size = len(text_bytes)
    return compressed_size / original_size

def load_img(root: str, rel_path: str) -> Optional[Image.Image]:
    try:
        p = os.path.join(root, rel_path)
        return Image.open(p).convert("RGB") if os.path.exists(p) else None
    except:
        return None

@torch.inference_mode()
def compute_text_only_scores(question: str, answer: str) -> Optional[Tuple[float, float, float]]:
    """
    计算text-only loss（不使用图像）
    Returns: (text_only_loss, zlib_ratio, calibrated_score) or None

    策略：
    1. 将question和answer拼接成纯文本
    2. 使用模型的language decoder计算loss
    3. 不传入任何图像信息
    """
    # 构建纯文本输入（移除图像标记）
    # 方法1: 完全去掉图像，只用文本
    msgs_full = [
        {"role":"user", "content":[{"type":"text", "text":question}]},
        {"role":"assistant", "content":[{"type":"text", "text":answer}]}
    ]
    msgs_u = [
        {"role":"user", "content":[{"type":"text", "text":question}]}
    ]

    text_full = processor.apply_chat_template(msgs_full, tokenize=False, add_generation_prompt=False)
    text_u    = processor.apply_chat_template(msgs_u,    tokenize=False, add_generation_prompt=True)

    # ⚠️ 关键：不传入images参数，纯文本encoding
    try:
        enc_full = processor(text=[text_full], images=None, return_tensors="pt").to(MODEL_DEVICE)
        enc_u    = processor(text=[text_u],    images=None, return_tensors="pt").to(MODEL_DEVICE)
    except Exception as e:
        # 如果processor要求必须有图像，则使用dummy blank image
        print(f"⚠️ Processor requires image, using blank image as fallback")
        blank_image = Image.new('RGB', (224, 224), color=(128, 128, 128))
        enc_full = processor(text=[text_full], images=[blank_image], return_tensors="pt").to(MODEL_DEVICE)
        enc_u    = processor(text=[text_u],    images=[blank_image], return_tensors="pt").to(MODEL_DEVICE)

    labels = enc_full["input_ids"].clone()
    prefix_len = enc_u["input_ids"].shape[1]
    labels[:, :prefix_len] = -100  # Mask prompt tokens

    # 计算loss
    with torch.cuda.amp.autocast(dtype=DTYPE):
        outputs = model(**enc_full, labels=labels)
        loss = outputs.loss.item()

    if not math.isfinite(loss):
        return None

    # Zlib calibration
    z_ratio = get_zlib_entropy(answer)
    calibrated_score = loss / max(z_ratio, 1e-6)

    return loss, z_ratio, calibrated_score

@torch.inference_mode()
def compute_multimodal_scores(image: Image.Image, question: str, answer: str) -> Optional[Tuple[float, float, float]]:
    """
    计算multimodal loss（使用图像）- 用于对比
    Returns: (mm_loss, zlib_ratio, calibrated_score) or None
    """
    msgs_full = [
        {"role":"user", "content":[{"type":"image"}, {"type":"text", "text":question}]},
        {"role":"assistant", "content":[{"type":"text", "text":answer}]}
    ]
    msgs_u = [
        {"role":"user", "content":[{"type":"image"}, {"type":"text", "text":question}]}
    ]

    text_full = processor.apply_chat_template(msgs_full, tokenize=False, add_generation_prompt=False)
    text_u    = processor.apply_chat_template(msgs_u,    tokenize=False, add_generation_prompt=True)

    enc_full = processor(text=[text_full], images=[image], return_tensors="pt").to(MODEL_DEVICE)
    enc_u    = processor(text=[text_u],    images=[image], return_tensors="pt").to(MODEL_DEVICE)

    labels = enc_full["input_ids"].clone()
    prefix_len = enc_u["input_ids"].shape[1]
    labels[:, :prefix_len] = -100

    with torch.cuda.amp.autocast(dtype=DTYPE):
        outputs = model(**enc_full, labels=labels)
        loss = outputs.loss.item()

    if not math.isfinite(loss):
        return None

    z_ratio = get_zlib_entropy(answer)
    calibrated_score = loss / max(z_ratio, 1e-6)

    return loss, z_ratio, calibrated_score

# ============================================================
# 4) Run Scoring Loop
# ============================================================
def run_scoring(data, root, label_name):
    """
    同时计算text-only和multimodal scores
    Returns: (text_only_results, multimodal_results)
    """
    # Text-only
    to_raw_losses = []
    to_zlib_ratios = []
    to_calib_scores = []

    # Multimodal
    mm_raw_losses = []
    mm_zlib_ratios = []
    mm_calib_scores = []

    for item in tqdm(data, desc=f"Scoring {label_name}"):
        # 提取文本
        q = item["messages"][0]["content"]
        if isinstance(q, list):
            q = "".join([x['text'] for x in q if x.get('type')=='text'])
        elif not isinstance(q, str):
            q = str(q)

        a = item["messages"][1]["content"]
        if isinstance(a, list):
            a = "".join([x['text'] for x in a if x.get('type')=='text'])
        elif not isinstance(a, str):
            a = str(a)

        # 1. Text-only score
        to_result = compute_text_only_scores(q, a)
        if to_result is not None:
            to_raw, to_zlib, to_calib = to_result
            to_raw_losses.append(to_raw)
            to_zlib_ratios.append(to_zlib)
            to_calib_scores.append(to_calib)

        # 2. Multimodal score (for comparison)
        img = load_img(root, item["images"][0])
        if img is not None:
            mm_result = compute_multimodal_scores(img, q, a)
            if mm_result is not None:
                mm_raw, mm_zlib, mm_calib = mm_result
                mm_raw_losses.append(mm_raw)
                mm_zlib_ratios.append(mm_zlib)
                mm_calib_scores.append(mm_calib)

    return {
        'text_only': (np.array(to_raw_losses), np.array(to_zlib_ratios), np.array(to_calib_scores)),
        'multimodal': (np.array(mm_raw_losses), np.array(mm_zlib_ratios), np.array(mm_calib_scores))
    }

print("\n⏳ Processing members...")
m_results = run_scoring(m_data, MEMBER_DIR, "Members")

print("⏳ Processing non-members...")
n_results = run_scoring(n_data, NONMEMBER_DIR, "Non-members")

# 解包结果
m_to_raw, m_to_zlib, m_to_calib = m_results['text_only']
m_mm_raw, m_mm_zlib, m_mm_calib = m_results['multimodal']

n_to_raw, n_to_zlib, n_to_calib = n_results['text_only']
n_mm_raw, n_mm_zlib, n_mm_calib = n_results['multimodal']

print(f"\n✅ Text-only scored: {len(m_to_raw)} members, {len(n_to_raw)} non-members")
print(f"✅ Multimodal scored: {len(m_mm_raw)} members, {len(n_mm_raw)} non-members")

# ============================================================
# 5) Evaluation Function
# ============================================================
def evaluate_mia(m_s, n_s, score_name="Score"):
    """
    评估MIA性能，自动判断方向
    """
    if len(m_s) == 0 or len(n_s) == 0:
        print(f"⚠️ {score_name}: Insufficient data")
        return None

    m_med, n_med = np.median(m_s), np.median(n_s)
    m_mean, n_mean = np.mean(m_s), np.mean(n_s)

    y_true = np.concatenate([np.ones(len(m_s)), np.zeros(len(n_s))])
    all_scores = np.concatenate([m_s, n_s])

    # Member score应该更低
    if m_med < n_med:
        direction = "Standard (M < N)"
        final_scores = -all_scores
    else:
        direction = "Inverted (M > N)"
        final_scores = all_scores

    auc = roc_auc_score(y_true, final_scores)
    fpr, tpr, thresholds = roc_curve(y_true, final_scores)

    idx_5fpr = np.where(fpr <= 0.05)[0]
    tpr_at_5fpr = tpr[idx_5fpr[-1]] if len(idx_5fpr) > 0 else 0.0

    print(f"\n{'='*60}")
    print(f"📊 {score_name}")
    print(f"{'='*60}")
    print(f"Direction:      {direction}")
    print(f"AUC:            {auc:.4f}")
    print(f"TPR @ 5% FPR:   {tpr_at_5fpr*100:.2f}%")
    print(f"Member   - Median: {m_med:.4f}, Mean: {m_mean:.4f}")
    print(f"NonMem   - Median: {n_med:.4f}, Mean: {n_mean:.4f}")
    print(f"{'='*60}")

    return {
        "auc": float(auc),
        "tpr_5fpr": float(tpr_at_5fpr),
        "m_median": float(m_med),
        "n_median": float(n_med),
        "m_mean": float(m_mean),
        "n_mean": float(n_mean),
        "direction": direction
    }

# ============================================================
# 6) Comprehensive Evaluation
# ============================================================
print("\n" + "="*70)
print("🔬 TEXT-ONLY MIA RESULTS".center(70))
print("="*70)

# Text-only raw loss
to_raw_results = evaluate_mia(m_to_raw, n_to_raw, "Text-Only Raw Loss")

# Text-only calibrated
to_calib_results = evaluate_mia(m_to_calib, n_to_calib, "Text-Only Zlib-Calibrated")

print("\n" + "="*70)
print("🔬 MULTIMODAL MIA RESULTS (For Comparison)".center(70))
print("="*70)

# Multimodal raw loss
mm_raw_results = evaluate_mia(m_mm_raw, n_mm_raw, "Multimodal Raw Loss")

# Multimodal calibrated
mm_calib_results = evaluate_mia(m_mm_calib, n_mm_calib, "Multimodal Zlib-Calibrated")

# ============================================================
# 7) Cross-Comparison Analysis
# ============================================================
print("\n" + "="*70)
print("🎯 COMPARISON ANALYSIS".center(70))
print("="*70)

if to_raw_results and mm_raw_results:
    print("\n📈 Text-Only vs Multimodal (Raw Loss)")
    print(f"Text-Only AUC:    {to_raw_results['auc']:.4f}")
    print(f"Multimodal AUC:   {mm_raw_results['auc']:.4f}")
    print(f"Difference:       {mm_raw_results['auc'] - to_raw_results['auc']:+.4f}")

    if mm_raw_results['auc'] > to_raw_results['auc'] + 0.05:
        print("✅ Vision significantly improves MIA (memorization is visual-dominant)")
    elif to_raw_results['auc'] > mm_raw_results['auc'] + 0.05:
        print("⚠️ Text-only performs better (unusual, check implementation)")
    else:
        print("➖ Similar performance (both modalities contribute)")

if to_raw_results and to_calib_results:
    print("\n📈 Text-Only: Raw vs Calibrated")
    auc_diff = to_calib_results['auc'] - to_raw_results['auc']
    tpr_diff = to_calib_results['tpr_5fpr'] - to_raw_results['tpr_5fpr']
    print(f"AUC Improvement:       {auc_diff:+.4f}")
    print(f"TPR@5%FPR Improvement: {tpr_diff:+.4f}")

    if auc_diff > 0.01:
        print("✅ Zlib calibration helps for text-only")
    elif auc_diff < -0.01:
        print("❌ Zlib calibration hurts text-only")
    else:
        print("➖ Minimal effect")

# Zlib ratio analysis
print("\n" + "="*70)
print("📈 Zlib Compression Ratio Statistics".center(70))
print("="*70)
print(f"Text-Only:")
print(f"  Member   - Median: {np.median(m_to_zlib):.4f}, Mean: {np.mean(m_to_zlib):.4f}")
print(f"  NonMem   - Median: {np.median(n_to_zlib):.4f}, Mean: {np.mean(n_to_zlib):.4f}")
print(f"Multimodal:")
print(f"  Member   - Median: {np.median(m_mm_zlib):.4f}, Mean: {np.mean(m_mm_zlib):.4f}")
print(f"  NonMem   - Median: {np.median(n_mm_zlib):.4f}, Mean: {np.mean(n_mm_zlib):.4f}")
print(f"Note: Lower ratio = more compressible (simpler text)")
print("="*70)

# ============================================================
# 8) Correlation Analysis
# ============================================================
from scipy.stats import spearmanr

print("\n" + "="*70)
print("🔗 Correlation Analysis".center(70))
print("="*70)

print("\nText-Only: Loss vs Zlib Ratio")
m_to_corr, m_to_pval = spearmanr(m_to_raw, m_to_zlib)
n_to_corr, n_to_pval = spearmanr(n_to_raw, n_to_zlib)
print(f"  Member:    ρ = {m_to_corr:.4f} (p={m_to_pval:.4e})")
print(f"  NonMember: ρ = {n_to_corr:.4f} (p={n_to_pval:.4e})")

print("\nMultimodal: Loss vs Zlib Ratio")
m_mm_corr, m_mm_pval = spearmanr(m_mm_raw, m_mm_zlib)
n_mm_corr, n_mm_pval = spearmanr(n_mm_raw, n_mm_zlib)
print(f"  Member:    ρ = {m_mm_corr:.4f} (p={m_mm_pval:.4e})")
print(f"  NonMember: ρ = {n_mm_corr:.4f} (p={n_mm_pval:.4e})")

print("\n💡 Interpretation:")
if abs(m_to_corr) > 0.3 or abs(n_to_corr) > 0.3:
    print("   Strong correlation in text-only → Zlib calibration theoretically valid")
if abs(m_mm_corr) < 0.1 and abs(n_mm_corr) < 0.1:
    print("   Weak correlation in multimodal → Vision dominates, text complexity irrelevant")

print("="*70)

# ============================================================
# 9) Save Results
# ============================================================
res_file = os.path.join(OUTPUT_DIR, "text_only_zlib_mia_results.json")
with open(res_file, "w") as f:
    json.dump({
        "text_only": {
            "raw": to_raw_results,
            "calibrated": to_calib_results,
            "scores": {
                "m_raw": m_to_raw.tolist(),
                "n_raw": n_to_raw.tolist(),
                "m_zlib": m_to_zlib.tolist(),
                "n_zlib": n_to_zlib.tolist(),
                "m_calib": m_to_calib.tolist(),
                "n_calib": n_to_calib.tolist()
            }
        },
        "multimodal": {
            "raw": mm_raw_results,
            "calibrated": mm_calib_results,
            "scores": {
                "m_raw": m_mm_raw.tolist(),
                "n_raw": n_mm_raw.tolist(),
                "m_zlib": m_mm_zlib.tolist(),
                "n_zlib": n_mm_zlib.tolist(),
                "m_calib": m_mm_calib.tolist(),
                "n_calib": n_mm_calib.tolist()
            }
        },
        "correlations": {
            "text_only": {
                "member": {"rho": float(m_to_corr), "pval": float(m_to_pval)},
                "nonmember": {"rho": float(n_to_corr), "pval": float(n_to_pval)}
            },
            "multimodal": {
                "member": {"rho": float(m_mm_corr), "pval": float(m_mm_pval)},
                "nonmember": {"rho": float(n_mm_corr), "pval": float(n_mm_pval)}
            }
        }
    }, f, indent=2)

print(f"\n💾 Results saved to {res_file}")
print("\n✨ All done!")

## Dc-Pdd

In [ ]:
import os, pickle, torch
import numpy as np
from collections import Counter
from transformers import AutoProcessor
from datasets import load_dataset
from tqdm import tqdm
from google.colab import drive

# 1. 初始化
drive.mount('/content/drive')

# Qwen2-VL 基座模型 ID
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
SAVE_PATH = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_qwen2.pkl"
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

# 加载 Qwen2-VL Processor
print(f"🚀 正在加载 {MODEL_ID} 的分词器...")
# 注意：Qwen2-VL 建议使用 AutoProcessor 以确保与模型预处理逻辑对齐
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = processor.tokenizer
vocab_size = len(tokenizer)

# 2. 加载 C4 数据集 (直接指向 JSON 文件避开脚本限制)
print("📥 正在加载 C4 英文语料 (Streaming)...")
dataset = load_dataset(
    "allenai/c4",
    data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
    split="train",
    streaming=True
)

# 3. 统计词频
counter = Counter()
MAX_SAMPLES = 50000  # 可以根据算力适当增加，5万样本足以捕捉通用分布

for i, ex in enumerate(tqdm(dataset, total=MAX_SAMPLES, desc="Qwen2 C4 词频统计")):
    if i >= MAX_SAMPLES: break
    # 使用 tokenizer.encode 提取 ID
    ids = tokenizer.encode(ex['text'], add_special_tokens=False)
    counter.update(ids)

# 4. 计算平滑词频 (Laplace Smoothing)
print("\n⚖️ 正在计算平滑词频...")
total_tokens = sum(counter.values())
# Qwen2 的词表很大，初始化为 float64 保证精度
freq_smo = np.ones(vocab_size, dtype=np.float64)

for tid, count in counter.items():
    if tid < vocab_size:
        freq_smo[tid] += count

# 归一化
freq_smo = freq_smo / (total_tokens + vocab_size)

# 5. 保存
with open(SAVE_PATH, "wb") as f:
    pickle.dump(freq_smo, f)

print(f"\n✅ Qwen2 原生频率表已保存至: {SAVE_PATH}")
print(f"统计词表大小: {vocab_size}")

In [ ]:
# ==============================================================================
# DC-PDD MIA (Qwen2-VL + LoRA) - FIX: always pass size + resample/interpolation
# Works even if processor.image_processor.do_resize=True by default.
# ==============================================================================

import os, json, math, random, pickle, warnings
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

warnings.filterwarnings("ignore")

# ============================================================
# 0) Paths & Config
# ============================================================
DRIVE_ROOT   = "/content/drive/MyDrive/FashionGen_MIA"
BASE_MODEL   = "/content/Qwen2-VL-2B-Instruct"
LORA_PATH    = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"
FREQ_PATH    = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_qwen2.pkl"

MEMBER_DIR        = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_DIR     = f"{DRIVE_ROOT}/nonmember_5k"
MEMBERS_FULL_JSON = f"{MEMBER_DIR}/mllm_data.json"
NONMEMBERS_FULL_JSON = f"{NONMEMBER_DIR}/mllm_data.json"

TRAIN_SIZE   = 1000
SEED         = 42

# DC-PDD hyperparams
ALPHA        = 0.01
DTYPE        = torch.bfloat16
USE_AMP      = True

# Qwen2-VL dynamic resize setup (must provide in your env)
IMG_SIZE = {"shortest_edge": 224, "longest_edge": 1344}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "Need GPU"

# ============================================================
# 1) Load freq table + processor + model
# ============================================================
print("🚀 Loading freq table / processor / model...")

with open(FREQ_PATH, "rb") as f:
    qwen_freq_smo = pickle.load(f)
VOCAB_SIZE_FREQ = len(qwen_freq_smo)

processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

MODEL_DEVICE = next(model.parameters()).device
print(f"✅ Model loaded. device={MODEL_DEVICE}, dtype={next(model.parameters()).dtype}")
print(f"✅ Freq vocab size = {VOCAB_SIZE_FREQ}")

# ============================================================
# 2) Helpers
# ============================================================
def _load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def robust_extract_text(c) -> str:
    if isinstance(c, str):
        return c.strip()
    if isinstance(c, list):
        parts = []
        for seg in c:
            if isinstance(seg, dict) and seg.get("type") == "text" and seg.get("text"):
                parts.append(seg["text"])
            elif isinstance(seg, str):
                parts.append(seg)
        return " ".join(parts).strip()
    return str(c).strip()

def load_img(root: str, rel_path: str):
    p = os.path.join(root, rel_path)
    if os.path.exists(p):
        return Image.open(p).convert("RGB")
    p2 = os.path.join(DRIVE_ROOT, rel_path)
    if os.path.exists(p2):
        return Image.open(p2).convert("RGB")
    return None

# ============================================================
# 3) CRITICAL FIX: safe processor call with size + resample/interpolation
# ============================================================
def processor_encode(text, image):
    """
    Your env throws:
      `size` and `resample/interpolation` must be specified if `do_resize` is `True`.
    So we ALWAYS pass size + (resample or interpolation).
    """
    # PIL resample enum compatibility
    try:
        RESAMPLE = Image.Resampling.BICUBIC  # PIL>=9
    except Exception:
        RESAMPLE = Image.BICUBIC

    base_kwargs = dict(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt",
        size=IMG_SIZE
    )

    # try 'resample' first, else fallback to 'interpolation'
    try:
        enc = processor(**base_kwargs, resample=RESAMPLE)
    except TypeError:
        enc = processor(**base_kwargs, interpolation="bicubic")

    return {k: v.to(MODEL_DEVICE) for k, v in enc.items()}

# ============================================================
# 4) Packing: full + user-only to get prefix_len, then score assistant tokens only
# ============================================================
def pack_for_dc_pdd(image: Image.Image, question: str, answer: str):
    msgs_full = [
        {"role": "user", "content": [{"type":"image"}, {"type":"text", "text": question}]},
        {"role": "assistant", "content": [{"type":"text", "text": answer}]},
    ]
    msgs_u = [
        {"role": "user", "content": [{"type":"image"}, {"type":"text", "text": question}]},
    ]

    text_full = processor.apply_chat_template(msgs_full, tokenize=False, add_generation_prompt=False)
    text_u    = processor.apply_chat_template(msgs_u,    tokenize=False, add_generation_prompt=True)

    enc_full = processor_encode(text_full, image)
    enc_u    = processor_encode(text_u,    image)

    prefix_len = int(enc_u["input_ids"].shape[1])
    return enc_full, prefix_len

# ============================================================
# 5) DC-PDD score
# ============================================================
@torch.inference_mode()
def compute_dc_pdd_score(image: Image.Image, question: str, answer: str) -> float:
    enc_full, prefix_len = pack_for_dc_pdd(image, question, answer)

    if USE_AMP and torch.cuda.is_available():
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            out = model(**enc_full)
    else:
        out = model(**enc_full)

    logits = out.logits                 # [1, T, V]
    input_ids = enc_full["input_ids"][0]  # [T]
    T = int(input_ids.shape[0])

    # assistant tokens start at prefix_len in input_ids
    if T <= prefix_len + 1:
        return float("nan")

    # logits[t] predicts token input_ids[t+1]
    start_pred = max(prefix_len - 1, 0)
    end_pred   = T - 1
    if start_pred >= end_pred:
        return float("nan")

    pred_logits = logits[0, start_pred:end_pred, :]        # [L, V]
    target_ids  = input_ids[start_pred+1:end_pred+1]       # [L]

    log_probs = torch.log_softmax(pred_logits, dim=-1)
    actual_lp = torch.gather(log_probs, 1, target_ids.unsqueeze(1)).squeeze(1)
    probs = torch.exp(actual_lp).float().cpu().numpy()

    t_ids_np = target_ids.detach().cpu().numpy()
    freqs = np.array([
        qwen_freq_smo[int(tid)] if (0 <= int(tid) < VOCAB_SIZE_FREQ) else 1e-10
        for tid in t_ids_np
    ], dtype=np.float64)

    deviation = probs * np.log(1.0 / (freqs + 1e-12))
    clipped = np.minimum(deviation, ALPHA)
    return float(np.mean(clipped))

# ============================================================
# 6) Run scoring
# ============================================================
def run_scoring(data, root, label_name):
    scores = []
    print(f"\n🔍 Scoring {label_name}...")
    for i, item in enumerate(tqdm(data, desc=label_name)):
        try:
            if not item.get("images"):
                continue
            img = load_img(root, item["images"][0])
            if img is None:
                continue

            q = robust_extract_text(item["messages"][0]["content"])
            a = robust_extract_text(item["messages"][1]["content"])
            if not a.strip():
                continue

            s = compute_dc_pdd_score(img, q, a)
            if (s is None) or (not np.isfinite(s)):
                continue

            scores.append(float(s))

        except Exception as e:
            if i < 2:
                print(f"Sample {i} Error: {e}")
            continue

    print(f"✅ {label_name} kept: {len(scores)}/{len(data)}")
    return np.array(scores, dtype=np.float64)

# ============================================================
# 7) Data + Metrics
# ============================================================
m_all = _load_json(MEMBERS_FULL_JSON)
n_all = _load_json(NONMEMBERS_FULL_JSON)

random.seed(SEED); random.shuffle(m_all); m_data = m_all[:TRAIN_SIZE]
random.seed(SEED); random.shuffle(n_all); n_data = n_all[:TRAIN_SIZE]

m_scores = run_scoring(m_data, MEMBER_DIR, "Member")
n_scores = run_scoring(n_data, NONMEMBER_DIR, "Non-Member")

if len(m_scores) > 0 and len(n_scores) > 0:
    y_true = np.concatenate([np.ones(len(m_scores)), np.zeros(len(n_scores))])
    all_s  = np.concatenate([m_scores, n_scores])
    auc = roc_auc_score(y_true, all_s)
    if auc < 0.5:
        auc = 1.0 - auc

    print("\n" + "="*60)
    print("📊 DC-PDD Results (assistant-only tokens; resize fixed)")
    print(f"Valid samples: M={len(m_scores)} N={len(n_scores)}")
    print(f"AUC: {auc:.4f}")
    print("="*60)
else:
    print("❌ No valid scores collected. (If kept=0, check the first Sample Error.)")


## M$^4$I

### data split

In [ ]:
# ===== 1) Strict split =====
def split_list(lst, n):
    return lst[:n], lst[n:]

rng = random.Random(SEED)

# shuffle copies (do not modify original)
members = members_all[:1000]
nonmembers = nonmembers_all[:1000]

rng.shuffle(members)
rng.shuffle(nonmembers)

# --- nonmember side: 200 FT + 200 train_nonmem + 100 dev_nonmem + 500 test_nonmem
ft200, rest = split_list(nonmembers, 200)
train_nonmem200, rest = split_list(rest, 200)
dev_nonmem100, rest = split_list(rest, 100)
test_nonmem500, rest = split_list(rest, 500)

assert len(ft200)==200 and len(train_nonmem200)==200 and len(dev_nonmem100)==100 and len(test_nonmem500)==500

# --- member side: dev_member100 + test_member500 (fixed from remaining 900)
dev_mem100, restm = split_list(members, 100)
# remaining 900
test_mem500, _ = split_list(restm, 500)

assert len(dev_mem100)==100 and len(test_mem500)==500

print("✅ Split summary:")
print("  FT (from nonmember)         :", len(ft200))
print("  Train nonmember (from nonm) :", len(train_nonmem200))
print("  Dev nonmember               :", len(dev_nonmem100))
print("  Test nonmember              :", len(test_nonmem500))
print("  Dev member                  :", len(dev_mem100))
print("  Test member                 :", len(test_mem500))


### finetune

In [ ]:
# =========================
# Module 2) Full FT on LoRA-baseline (FIXED)
# - NO pip -U (you already pinned versions)
# - force autograd ON
# - ensure requires_grad True
# - gradient checkpointing: enable_input_require_grads()
# - safer processor (use_fast=False)
# - move batch to the actual model device (works even if device_map is used)
# =========================

import os, re
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

# ✅ IMPORTANT: make sure training is NOT globally disabled by previous cells
torch.set_grad_enabled(True)

# ---- your paths ----
DRIVE_ROOT = "/content/drive/MyDrive/FashionGen_MIA"
BASE_MODEL = "/content/Qwen2-VL-2B-Instruct"
EXISTING_LORA_PATH = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"

MEMBER_DIR = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_DIR = f"{DRIVE_ROOT}/nonmember_5k"

DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float16
print("CUDA available:", torch.cuda.is_available(), "DTYPE:", DTYPE)
print("LoRA:", EXISTING_LORA_PATH)

# ---- load processor + base + lora ----
# ✅ use_fast=False to avoid fast/slow behavior drift
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True, use_fast=False)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    device_map="auto",           # keep your setting
    trust_remote_code=True
)

lora_model = PeftModel.from_pretrained(
    base_model,
    EXISTING_LORA_PATH,
    torch_dtype=DTYPE,
    device_map="auto"
)

lora_model.eval()
print("✅ Loaded (base + LoRA). Now merge LoRA for full fine-tune...")

# ---- merge LoRA into base weights to do true full fine-tune ----
ft_model = lora_model.merge_and_unload()

# ✅ ensure train mode + grad
ft_model.train()
ft_model.requires_grad_(True)

# ✅ for gradient checkpointing
ft_model.gradient_checkpointing_enable()
ft_model.enable_input_require_grads()

# ✅ must be False for training
ft_model.config.use_cache = False

# ---- sanity check: must have trainable params ----
num_trainable = sum(p.requires_grad for p in ft_model.parameters())
num_total = sum(1 for _ in ft_model.parameters())
print(f"Trainable params tensors: {num_trainable} / {num_total}")
assert num_trainable > 0, "No trainable params! grads are disabled somewhere."

# ✅ determine where the dispatched model lives (works with device_map too)
MODEL_DEVICE = next(ft_model.parameters()).device
print("MODEL_DEVICE:", MODEL_DEVICE)
print("✅ LoRA merged. Full fine-tune model ready.")

# ---- helpers: caption & image ----
IMAGE_TOKEN_RE = re.compile(r"<\s*image\s*>", flags=re.IGNORECASE)

def strip_image_token(s):
    if not isinstance(s, str):
        return ""
    s = IMAGE_TOKEN_RE.sub("", s)
    s = s.replace("\n", " ").strip()
    return re.sub(r"\s+", " ", s).strip()

def extract_text(content):
    if isinstance(content, str):
        return strip_image_token(content)
    if isinstance(content, list):
        parts = []
        for seg in content:
            if isinstance(seg, dict) and seg.get("type") == "text":
                parts.append(seg.get("text", ""))
            elif isinstance(seg, str):
                parts.append(seg)
        return strip_image_token(" ".join(parts))
    return strip_image_token(str(content))

def get_caption(item):
    msgs = item.get("messages", [])
    if len(msgs) < 2:
        return ""
    a = msgs[1]
    if a.get("role") != "assistant":
        return ""
    return extract_text(a.get("content", ""))

def load_image(root_dir, item):
    imgs = item.get("images", [])
    if not imgs:
        return None
    img_path = os.path.join(root_dir, imgs[0])
    if not os.path.exists(img_path):
        return None
    try:
        return Image.open(img_path).convert("RGB")
    except:
        return None

PROMPT_IMG = "Describe this fashion item."

def build_user_only_text(image):
    msgs = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": PROMPT_IMG}
        ]}
    ]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def build_full_sft_text(image, caption):
    # ✅ make assistant content explicit "text" segment (more robust)
    msgs = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": PROMPT_IMG}
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": caption}
        ]}
    ]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

class FT200Dataset(Dataset):
    def __init__(self, items, root_dir):
        self.samples = []
        for it in items:
            img = load_image(root_dir, it)
            cap = get_caption(it)
            if img is None or not cap:
                continue
            self.samples.append((img, cap))
        print(f"✅ FT dataset usable: {len(self.samples)} / {len(items)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_sft(batch):
    images = [x[0] for x in batch]
    caps   = [x[1] for x in batch]

    # full: user+assistant
    full_texts = [build_full_sft_text(im, cap) for im, cap in zip(images, caps)]
    full_inputs = processor(text=full_texts, images=images, return_tensors="pt", padding=True)

    # user-only (to mask user prompt tokens)
    user_texts = [build_user_only_text(im) for im in images]
    user_inputs = processor(text=user_texts, images=images, return_tensors="pt", padding=True)

    labels = full_inputs["input_ids"].clone()
    for i in range(labels.size(0)):
        user_len = int(user_inputs["attention_mask"][i].sum().item())
        labels[i, :user_len] = -100
    full_inputs["labels"] = labels

    # ✅ move to model device (not a hardcoded cuda:0)
    return {k: v.to(MODEL_DEVICE) for k, v in full_inputs.items()}

# =========
# IMPORTANT: you already have these splits in memory:
# ft200, train_nonmem200, dev_mem100, dev_nonmem100, test_mem500, test_nonmem500
# =========

train_ds = FT200Dataset(ft200, NONMEMBER_DIR)
train_dl = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_sft)

# ---- training hyperparams ----
lr = 1e-5
epochs = 1
grad_accum = 8

opt = torch.optim.AdamW(ft_model.parameters(), lr=lr)

step = 0
ft_model.train()

for ep in range(epochs):
    for batch in train_dl:
        # ✅ ensure autograd ON even if some other cell toggled it
        torch.set_grad_enabled(True)

        out = ft_model(**batch)
        loss = out.loss / grad_accum
        loss.backward()

        step += 1
        if step % grad_accum == 0:
            opt.step()
            opt.zero_grad(set_to_none=True)

        if step % 20 == 0:
            print(f"Epoch {ep+1}/{epochs} Step {step} Loss {loss.item():.4f}")

# ---- save to /content (NOT Drive) ----
FT_MODEL_DIR = "/content/qwen2vl_loraBaseline_fullFT_200"
os.makedirs(FT_MODEL_DIR, exist_ok=True)

ft_model.save_pretrained(FT_MODEL_DIR)
processor.save_pretrained(FT_MODEL_DIR)

print("✅ Saved full-FT model to:", FT_MODEL_DIR)


### get acts

In [ ]:
# =========================
# Module 3) Extract Activations
# Train activations: Full-FT model
# Dev/Test activations: LoRA-baseline model (base + adapter)
# Save: /content/acts/{train,dev,test}/layer_{L}_{start}.pt
# =========================

import os, re
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

# -------------------------
# 0) Paths / Config
# -------------------------
DRIVE_ROOT = "/content/drive/MyDrive/FashionGen_MIA"
BASE_MODEL = "/content/Qwen2-VL-2B-Instruct"
EXISTING_LORA_PATH = f"{DRIVE_ROOT}/adapters/fashiongen_1k_2b"

FT_MODEL_DIR = "/content/qwen2vl_loraBaseline_fullFT_200"   # ✅ 你 full-FT 保存目录

MEMBER_DIR    = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_DIR = f"{DRIVE_ROOT}/nonmember_5k"

ACTS_ROOT = "/content/acts"
TRAIN_DIR = os.path.join(ACTS_ROOT, "train")
DEV_DIR   = os.path.join(ACTS_ROOT, "dev")
TEST_DIR  = os.path.join(ACTS_ROOT, "test")
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(DEV_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float16

PROMPT_IMG = "Describe this fashion item."

print("CUDA:", torch.cuda.is_available(), "DTYPE:", DTYPE)
print("FT_MODEL_DIR:", FT_MODEL_DIR)
print("BASE_MODEL:", BASE_MODEL)
print("LoRA:", EXISTING_LORA_PATH)
print("ACTS_ROOT:", ACTS_ROOT)

torch.set_grad_enabled(False)

# -------------------------
# 1) Text/Image helpers
# -------------------------
IMAGE_TOKEN_RE = re.compile(r"<\s*image\s*>", flags=re.IGNORECASE)

def strip_image_token(s):
    if not isinstance(s, str):
        return ""
    s = IMAGE_TOKEN_RE.sub("", s)
    s = s.replace("\n", " ").strip()
    return re.sub(r"\s+", " ", s).strip()

def extract_text(content):
    if isinstance(content, str):
        return strip_image_token(content)
    if isinstance(content, list):
        parts = []
        for seg in content:
            if isinstance(seg, dict) and seg.get("type") == "text":
                parts.append(seg.get("text", ""))
            elif isinstance(seg, str):
                parts.append(seg)
        return strip_image_token(" ".join(parts))
    return strip_image_token(str(content))

def get_caption_from_item(item):
    msgs = item.get("messages", [])
    if len(msgs) < 2:
        return ""
    a = msgs[1]
    if a.get("role") != "assistant":
        return ""
    return extract_text(a.get("content", ""))

def load_image_from_item(root_dir, item):
    imgs = item.get("images", [])
    if not imgs:
        return None
    img_path = os.path.join(root_dir, imgs[0])
    if not os.path.exists(img_path):
        return None
    try:
        return Image.open(img_path).convert("RGB")
    except:
        return None

# -------------------------
# 2) Dataset (keeps label for later)
# -------------------------
def wrap(items, label, root_dir):
    return [{"item": it, "label": int(label), "root_dir": root_dir} for it in items]

# your splits (already prepared)
# Split summary:
#   FT (from nonmember)         : 200  -> member label=1
#   Train nonmember (from nonm) : 200  -> label=0
#   Dev member                  : 100  -> label=1
#   Dev nonmember               : 100  -> label=0
#   Test member                 : 500  -> label=1
#   Test nonmember              : 500  -> label=0

train_rows = wrap(ft200, 1, NONMEMBER_DIR) + wrap(train_nonmem200, 0, NONMEMBER_DIR)
dev_rows   = wrap(dev_mem100, 1, MEMBER_DIR) + wrap(dev_nonmem100, 0, NONMEMBER_DIR)
test_rows  = wrap(test_mem500, 1, MEMBER_DIR) + wrap(test_nonmem500, 0, NONMEMBER_DIR)

print("✅ sizes:",
      "train", len(train_rows),
      "dev", len(dev_rows),
      "test", len(test_rows))

class VLMRowsDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        item = r["item"]
        root = r["root_dir"]
        label = r["label"]

        img = load_image_from_item(root, item)
        cap = get_caption_from_item(item)

        if img is None:
            img = Image.new("RGB", (448, 448), (128, 128, 128))
        if not cap:
            cap = "N/A"

        return img, cap, label

# -------------------------
# 3) Build input text (image+caption as USER input)
# -------------------------
def build_user_text(processor, image, caption):
    msgs = [{
        "role": "user",
        "content": [
            {"type":"image", "image": image},
            {"type":"text",  "text": f"{PROMPT_IMG}\n\nCaption: {caption}"}
        ]
    }]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def collate_vlm(batch, processor, model_device):
    images, caps, labels = zip(*batch)
    texts = [build_user_text(processor, im, cap) for im, cap in zip(images, caps)]
    inputs = processor(
        text=texts,
        images=list(images),
        return_tensors="pt",
        padding=True
    )
    inputs = {k: v.to(model_device) for k, v in inputs.items()}
    labels = torch.tensor(labels, dtype=torch.long)
    return inputs, labels

# -------------------------
# 4) Core: extract & save last-token hidden states for all layers
# -------------------------
def extract_and_save(model, processor, rows, save_dir, batch_size=1):
    ds = VLMRowsDataset(rows)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                    collate_fn=lambda b: collate_vlm(b, processor, model.device))

    n_layers = model.config.num_hidden_layers
    hidden_size = model.config.hidden_size
    print(f"\n[Extract] -> {save_dir}")
    print("  layers:", n_layers, "hidden:", hidden_size, "batch:", batch_size, "model_device:", model.device)

    start = 0
    for (inputs, y) in tqdm(dl, desc=f"Extracting {os.path.basename(save_dir)}"):
        out = model(**inputs, output_hidden_states=True, return_dict=True)
        hstates = out.hidden_states  # tuple: len = n_layers+1

        attn = inputs.get("attention_mask", None)
        if attn is None:
            last_idx = torch.full((hstates[-1].size(0),), hstates[-1].size(1)-1, device=model.device)
        else:
            last_idx = attn.sum(dim=1) - 1  # [B]

        for L in range(n_layers):
            hs = hstates[L+1]  # [B,T,H]
            bsz = hs.size(0)
            gather_idx = last_idx.view(bsz, 1, 1).expand(bsz, 1, hs.size(-1))
            last_tok = hs.gather(dim=1, index=gather_idx).squeeze(1)  # [B,H]
            last_tok = last_tok.float().detach().cpu()
            torch.save(last_tok, os.path.join(save_dir, f"layer_{L}_{start}.pt"))

        # optional labels per batch (debug-friendly)
        torch.save(y.cpu(), os.path.join(save_dir, f"labels_{start}.pt"))

        start += len(y)

    print("✅ Saved:", save_dir)

# -------------------------
# 5) Load TWO models:
#    - FT model for TRAIN acts
#    - LoRA-baseline model for DEV/TEST acts
# -------------------------

# (A) Full-FT model
print("\n⏳ Loading Full-FT model for TRAIN activations...")
ft_processor = AutoProcessor.from_pretrained(FT_MODEL_DIR, trust_remote_code=True)
ft_model = Qwen2VLForConditionalGeneration.from_pretrained(
    FT_MODEL_DIR,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True
)
ft_model.eval()
ft_model.config.use_cache = False
print("✅ Full-FT loaded. device:", ft_model.device)

# (B) LoRA-baseline model (base + adapter, NOT merged, NOT FT)
print("\n⏳ Loading LoRA-baseline model for DEV/TEST activations...")
base_processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True
)
lora_baseline = PeftModel.from_pretrained(
    base_model,
    EXISTING_LORA_PATH,
    torch_dtype=DTYPE,
    device_map="auto"
)
lora_baseline.eval()
lora_baseline.config.use_cache = False
print("✅ LoRA-baseline loaded. device:", lora_baseline.device)

# -------------------------
# 6) Run extraction with your required design
# -------------------------
# 1) Train activations (FT model)
extract_and_save(ft_model, ft_processor, train_rows, TRAIN_DIR, batch_size=1)

# 2) Dev activations (LoRA-baseline model)
extract_and_save(lora_baseline, base_processor, dev_rows, DEV_DIR, batch_size=1)

# 3) Test activations (LoRA-baseline model)
extract_and_save(lora_baseline, base_processor, test_rows, TEST_DIR, batch_size=1)

print("\n✅ Done. Next: LRProbe module can load /content/acts/{train,dev,test}.")


### evaluate

In [ ]:
# =========================
# Module 4) MIA Evaluation (LRProbe)
# - Train probe on /content/acts/train (FT model activations)
# - Select best layer by DEV AUC on /content/acts/dev (LoRA-baseline activations)
# - Report TEST AUC + TPR@5%FPR on /content/acts/test (LoRA-baseline activations)
# =========================

import os
import re
import json
import random
import numpy as np
import torch
from glob import glob
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve, auc

# -------------------------
# 0) Config
# -------------------------
ACTS_ROOT = "/content/acts"
TRAIN_DIR = os.path.join(ACTS_ROOT, "train")
DEV_DIR   = os.path.join(ACTS_ROOT, "dev")
TEST_DIR  = os.path.join(ACTS_ROOT, "test")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Dirs:", TRAIN_DIR, DEV_DIR, TEST_DIR)

torch.set_grad_enabled(True)

# -------------------------
# 1) EXACT LRProbe (same as yours)
# -------------------------
class LRProbe(torch.nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(d_in, 1, bias=False),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def pred(self, x):
        return self(x).round()

    def score(self, x):
        return self(x)

    @staticmethod
    def from_data(acts, labels, lr=0.001, weight_decay=0.1, epochs=1000, device="cpu"):
        acts, labels = acts.to(device), labels.to(device)
        probe = LRProbe(acts.shape[-1]).to(device)

        opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=weight_decay)
        for _ in range(epochs):
            opt.zero_grad()
            loss = torch.nn.BCELoss()(probe(acts), labels)
            loss.backward()
            opt.step()

        return probe

# -------------------------
# 2) Utilities: load labels and activations
# -------------------------
def load_all_labels(split_dir):
    """
    Load labels saved as labels_{start}.pt and concatenate by start order.
    Returns: torch.FloatTensor [N]
    """
    files = sorted(glob(os.path.join(split_dir, "labels_*.pt")),
                   key=lambda p: int(re.findall(r"labels_(\d+)\.pt", p)[0]))
    if not files:
        raise ValueError(f"No label files found in {split_dir}")

    ys = []
    for p in files:
        y = torch.load(p, map_location="cpu")
        ys.append(y)
    y = torch.cat(ys, dim=0).float()
    return y

def load_layer_acts(split_dir, layer):
    """
    Load activations for a given layer from layer_{layer}_{start}.pt in start order.
    Returns: torch.FloatTensor [N, H]
    """
    pattern = os.path.join(split_dir, f"layer_{layer}_*.pt")
    files = sorted(glob(pattern),
                   key=lambda p: int(re.findall(rf"layer_{layer}_(\d+)\.pt", p)[0]))
    if not files:
        raise ValueError(f"No activation files found: {pattern}")

    acts = []
    for p in files:
        a = torch.load(p, map_location="cpu")  # [B,H]
        acts.append(a)
    acts = torch.cat(acts, dim=0).float()
    return acts

def center_scale(acts, center=True, scale=True):
    # EXACT same idea as your ActDataset: per-dimension normalization
    if center:
        acts = acts - torch.mean(acts, dim=0, keepdim=False)
    if scale:
        std = torch.std(acts, dim=0, keepdim=False)
        std[std == 0] = 1.0
        acts = acts / std
    return acts

# -------------------------
# 3) Metrics (AUC + TPR@5%FPR) — matches your logic
# -------------------------
def compute_metrics(scores, labels, print_result=True):
    """
    scores: numpy array, higher should mean "more member"
    labels: numpy array {0,1}
    """
    fpr, tpr, _ = roc_curve(labels.astype(bool), -scores)  # keep same sign convention as yours
    auc_val = auc(fpr, tpr)

    valid_idx = np.where(fpr < 0.05)[0]
    tpr_5_fpr = tpr[valid_idx[-1]] if len(valid_idx) else 0.0

    if print_result:
        print(f"AUC {auc_val:.4f}, TPR@5%FPR {tpr_5_fpr:.4f}")
    return auc_val, tpr_5_fpr

@torch.no_grad()
def eval_probe(probe, acts, labels_tensor):
    scores = probe.score(acts.to(device)).detach().cpu().numpy()
    # follow your convention: prediction list is negative of score
    pred = -scores
    lab = labels_tensor.detach().cpu().numpy()
    auc_val, tpr5 = compute_metrics(pred, lab, print_result=False)
    return auc_val, tpr5

# -------------------------
# 4) Main: train on train split; select best layer on dev; report test
# -------------------------
def run_mia(layer_num, lr=0.001, weight_decay=0.1, epochs=1000):
    # load labels once
    y_train = load_all_labels(TRAIN_DIR).to(device)
    y_dev   = load_all_labels(DEV_DIR).to(device)
    y_test  = load_all_labels(TEST_DIR).to(device)

    print("\nLabel counts:")
    print("  Train:", y_train.numel(), "members:", int((y_train==1).sum().item()), "nonmembers:", int((y_train==0).sum().item()))
    print("  Dev  :", y_dev.numel(),   "members:", int((y_dev==1).sum().item()),   "nonmembers:", int((y_dev==0).sum().item()))
    print("  Test :", y_test.numel(),  "members:", int((y_test==1).sum().item()),  "nonmembers:", int((y_test==0).sum().item()))

    dev_auc_list, dev_tpr5_list = [], []
    test_auc_list, test_tpr5_list = [], []

    best_layer = None
    best_dev_auc = -1

    for L in range(layer_num):
        # load acts
        X_train = load_layer_acts(TRAIN_DIR, L).to(device)
        X_dev   = load_layer_acts(DEV_DIR, L).to(device)
        X_test  = load_layer_acts(TEST_DIR, L).to(device)

        # normalize (center/scale) — do it separately per split to match your ActDataset behavior
        X_train = center_scale(X_train)
        X_dev   = center_scale(X_dev)
        X_test  = center_scale(X_test)

        # train probe on train split
        probe = LRProbe.from_data(
            X_train, y_train,
            lr=lr, weight_decay=weight_decay, epochs=epochs,
            device=device
        )

        # eval
        dev_auc, dev_tpr5 = eval_probe(probe, X_dev, y_dev)
        test_auc, test_tpr5 = eval_probe(probe, X_test, y_test)

        dev_auc_list.append(dev_auc); dev_tpr5_list.append(dev_tpr5)
        test_auc_list.append(test_auc); test_tpr5_list.append(test_tpr5)

        if dev_auc > best_dev_auc:
            best_dev_auc = dev_auc
            best_layer = L

        print(f"Layer {L:02d} | DEV AUC {dev_auc:.4f} TPR@5%FPR {dev_tpr5:.4f} | TEST AUC {test_auc:.4f} TPR@5%FPR {test_tpr5:.4f}")

    # final: use best dev layer's test metrics
    final_test_auc = test_auc_list[best_layer]
    final_test_tpr5 = test_tpr5_list[best_layer]

    print("\n==============================")
    print("✅ Selection by DEV AUC")
    print("==============================")
    print(f"Best DEV layer: {best_layer}")
    print(f"Best DEV AUC  : {best_dev_auc:.4f}")
    print(f"TEST AUC @best-dev-layer      : {final_test_auc:.4f}")
    print(f"TEST TPR@5%FPR @best-dev-layer: {final_test_tpr5:.4f}")

    # also report best test layer (optional, for reference)
    best_test_layer = int(np.argmax(test_auc_list))
    print("\n(Reference) Best TEST layer:", best_test_layer,
          "TEST AUC:", test_auc_list[best_test_layer],
          "DEV AUC:", dev_auc_list[best_test_layer])

    # save results
    out = {
        "layer_num": layer_num,
        "best_dev_layer": int(best_layer),
        "best_dev_auc": float(best_dev_auc),
        "test_auc_at_best_dev_layer": float(final_test_auc),
        "test_tpr5_at_best_dev_layer": float(final_test_tpr5),
        "per_layer": [
            {
                "layer": int(i),
                "dev_auc": float(dev_auc_list[i]),
                "dev_tpr5": float(dev_tpr5_list[i]),
                "test_auc": float(test_auc_list[i]),
                "test_tpr5": float(test_tpr5_list[i]),
            }
            for i in range(layer_num)
        ]
    }
    save_path = "/content/mia_lrprobe_results.json"
    with open(save_path, "w") as f:
        json.dump(out, f, indent=2)
    print("\n💾 Saved:", save_path)

# -------------------------
# 5) Run
# -------------------------
# Qwen2-VL-2B typically has 24 layers (but please set to your model's actual layer count)
# If you don't know, set layer_num=32 and it will error when missing files.
layer_num = 24   # <- 如果你 Module 3 输出到 layer_0..layer_23，就用 24
run_mia(layer_num=layer_num, lr=0.001, weight_decay=0.1, epochs=1000)


# Qwen-MedTrinity

In [ ]:
# 1. 挂载 Google Drive (每次重连 Colab 都必须做这一步)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. (可选) 验证文件是否存在，确保挂载成功
import os
target_file = "/content/drive/MyDrive/MedTrinity_MIA/member_5k/mllm_data.json"

if os.path.exists(target_file):
    print(f"✅ Drive 挂载成功，文件存在: {target_file}")
else:
    print(f"❌ 错误: 找不到文件。请检查 Drive 路径或账号是否正确。")
    print(f"   尝试访问的路径: {target_file}")

In [ ]:
#@title 1. LoRA Training
import os, json, re, shutil, subprocess
from pathlib import Path

# ========= 基本路径（保持不变）=========
LLAMA_REPO = "/content/LLaMA-Factory"
HF_BASE_ID = "Qwen/Qwen2-VL-2B-Instruct"
CACHE_DIR  = "/content/hf_cache"

# ========= 数据切换：只改用 1000 条评测集 JSON =========
MIA_ROOT    = "/content/drive/MyDrive/MedTrinity_MIA"
MEM_ROOT    = os.path.join(MIA_ROOT, "member_5k")                   # 图片仍在 member_5k 下
MEM_EVAL_JSON = os.path.join(MIA_ROOT, "member_eval_1000.json")     # 新数据：1000 条

# 如果 eval JSON 是 {"data":[...]} 结构，自动拍平到一个新文件（LLaMA-Factory 读起来更稳）
FLAT_JSON   = os.path.join(MEM_ROOT, "mllm_data_eval_1000_flat.json")

OUT_DIR    = "/content/Qwen2-VL-med-lora-A100"   # 不变
LOG_PATH   = "/content/train.log"
SH_PATH    = "/content/run_lora_train.sh"

# ========= 环境“止血贴”（不变）=========
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
os.environ["MPLBACKEND"] = "Agg"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ========= 基本检查（不变）=========
assert os.path.isdir(LLAMA_REPO),  "❌ /content/LLaMA-Factory 不存在"
assert os.path.isdir(MEM_ROOT),     "❌ 缺少 member_5k 图片目录"
assert os.path.isfile(MEM_EVAL_JSON), "❌ 缺少 member_eval_1000.json"
os.makedirs(CACHE_DIR, exist_ok=True)

# 拍平 JSON（若需要）
def ensure_flat_sharegpt(in_json: str, out_json: str):
    with open(in_json, "r", encoding="utf-8") as f:
        d = json.load(f)
    data = d["data"] if isinstance(d, dict) and "data" in d else d
    if not isinstance(data, list):
        raise ValueError("member_eval_1000.json 不是列表，也不含 'data' 列表。")
    # 简单校验每条至少有 images / messages 字段
    ok = 0
    for it in data:
        if isinstance(it, dict) and ("images" in it) and ("messages" in it):
            ok += 1
    if ok == 0:
        raise ValueError("member_eval_1000.json 中未找到包含 images/messages 的条目。")
    os.makedirs(os.path.dirname(out_json), exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    return out_json

FLAT_PATH = ensure_flat_sharegpt(MEM_EVAL_JSON, FLAT_JSON)
print(f"✅ 使用拍平后的 ShareGPT JSON: {FLAT_PATH}")

# ========= 注册数据到 LLaMA-Factory（仅更换 file_name）=========
os.chdir(LLAMA_REPO)
dataset_info_path = "data/dataset_info.json"
with open(dataset_info_path, "r", encoding="utf-8") as f:
    info = json.load(f)
# 用新的数据键，避免覆盖你原来的 5k 配置
info["mllm_member_eval_1000"] = {
    "file_name": FLAT_PATH,                # ← 只换这个
    "formatting": "sharegpt",
    "columns": {"messages":"messages","images":"images"},
    "tags": {"role_tag":"role","content_tag":"content","user_tag":"user","assistant_tag":"assistant"}
}
with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)
print("✅ dataset_info.json 已注册 mllm_member_eval_1000")

# ========= 自动续训（不变）=========
def latest_ckpt(root: str):
    cands = []
    p = Path(root)
    if not p.exists():
        return None
    for d in p.glob("checkpoint-*"):
        m = re.search(r"checkpoint-(\d+)$", d.name)
        if m:
            cands.append((int(m.group(1)), str(d)))
    if not cands:
        return None
    cands.sort(key=lambda x: x[0], reverse=True)
    return cands[0][1]

resume_dir = latest_ckpt(OUT_DIR)
if resume_dir:
    print(f"🔁 将从最近的 checkpoint 续训：{resume_dir}")
else:
    print("🔰 没有已存在的 checkpoint，本次从头训练。")
    if os.path.isdir(OUT_DIR):
        shutil.rmtree(OUT_DIR)

# ========= 写训练脚本（除数据名外，其它全部保持一致）=========
script = f"""#!/usr/bin/env bash
set -euo pipefail
cd "{LLAMA_REPO}"

python -m llamafactory.cli train \\
  --stage sft \\
  --do_train \\
  --model_name_or_path "{HF_BASE_ID}" \\
  --cache_dir "{CACHE_DIR}" \\
  --trust_remote_code \\
  --finetuning_type lora \\
  --template qwen2_vl \\
  --media_dir "{MEM_ROOT}" \\
  --dataset_dir data \\
  --dataset mllm_member_eval_1000 \\
  --cutoff_len 2048 \\
  --learning_rate 1e-4 \\
  --num_train_epochs 5.0 \\
  --max_samples 5000 \\
  --per_device_train_batch_size 4 \\
  --gradient_accumulation_steps 4 \\
  --lr_scheduler_type cosine \\
  --max_grad_norm 1.0 \\
  --logging_steps 5 \\
  --save_strategy steps \\
  --save_steps 100 \\
  --save_total_limit 30 \\
  --warmup_steps 0 \\
  --optim adamw_torch \\
  --report_to none \\
  --output_dir "{OUT_DIR}" \\
  --overwrite_output_dir \\
  --bf16 \\
  --plot_loss False \\
  --flash_attn sdpa \\
  --dataloader_num_workers 0 \\
  --no_dataloader_pin_memory \\
  --torch_empty_cache_steps 20 \\
  --lora_rank 64 \\
  --lora_alpha 128 \\
  --lora_dropout 0.0 \\
  --lora_target all \\
  --gradient_checkpointing \\
  {f'--resume_from_checkpoint {resume_dir}' if resume_dir else ''}
"""
Path(SH_PATH).write_text(script)
os.chmod(SH_PATH, 0o755)

# ========= 执行训练 + 记录日志（不变）=========
print("🚀 Launching LoRA training …")
with open(LOG_PATH, "wb") as logf:
    proc = subprocess.Popen(["bash", "-lc", SH_PATH], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    for line in iter(proc.stdout.readline, b""):
        logf.write(line)
        try:
            print(line.decode("utf-8"), end="")
        except UnicodeDecodeError:
            print(line.decode("latin-1"), end="")
    proc.wait()
    code = proc.returncode

print(f"\n🧾 Log saved to: {LOG_PATH}")
if code != 0:
    print("\n❌ 训练异常（可能是 segfault）。下面打印日志尾部 200 行：\n" + "="*80)
    with open(LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
        print("".join(f.readlines()[-200:]))
    raise SystemExit(code)

# ========= 核验目录 & 最优 Adapter（不变）=========
def find_adapters(root: str):
    hits = []
    for p in Path(root).rglob("adapter_config.json"):
        d = str(p.parent)
        m = re.search(r"checkpoint[-_]?(\d+)", d)
        step = int(m.group(1)) if m else -1
        try:
            mtime = os.path.getmtime(d)
        except Exception:
            mtime = 0.0
        hits.append((step, mtime, d))
    if not hits and (Path(root)/"adapter_config.json").is_file():
        hits.append((10**9, os.path.getmtime(root), root))
    return sorted(hits, key=lambda x: (x[0], x[1]), reverse=True)

def print_tree(root, max_items=300):
    print(f"\n📂 Tree under {root}:")
    cnt = 0
    for p in Path(root).rglob("*"):
        print(p.relative_to(root))
        cnt += 1
        if cnt >= max_items:
            print(f"... (showing first {max_items})")
            break

print_tree(OUT_DIR, max_items=300)
adapters = find_adapters(OUT_DIR)
if not adapters:
    print("\n⚠️ 训练完成但未发现 adapter_config.json，请将日志尾部 200 行发我定位。")
else:
    print("\n✅ Found LoRA adapter dirs:")
    for i, (step, mtime, d) in enumerate(adapters[:5]):
        print(f"  [{i}] {d} (step={step})")
    best = adapters[0][2]
    print(f"\n🎉 LoRA training finished. Best adapter: {best}")


In [ ]:
# @title 2. Loss Attack MIA
import os, json, random, math, subprocess, sys, csv
from pathlib import Path
from tqdm import tqdm

import torch
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import roc_auc_score
# ===== [NEW] 指标相关额外导入 =====
from sklearn.metrics import roc_curve, precision_recall_fscore_support, accuracy_score, confusion_matrix
# =================================

from transformers import AutoProcessor, AutoModelForVision2Seq

# ---------------- 路径配置（仅数据改成 LoRA 用的 eval-1000；其余保持不变） ----------------
LLAMA_REPO = "/content/LLaMA-Factory"
BASE_ID    = "Qwen/Qwen2-VL-2B-Instruct"
CACHE_DIR  = "/content/hf_cache"

# 你训练好的 LoRA 最佳 checkpoint（保持不变）
LORA_BEST  = "/content/Qwen2-VL-med-lora-A100/checkpoint-315"

# 合并后模型保存到 Drive（保持不变）
MERGED_DIR = "/content/drive/MyDrive/Qwen2-VL-med-merged-A100"

# ================= 仅此处数据切换：与 LoRA 训练一致 =================
MIA_ROOT       = "/content/drive/MyDrive/MedTrinity_MIA"
MEM_IMGROOT    = f"{MIA_ROOT}/member_5k"                 # 图像根目录与训练相同
NON_IMGROOT    = f"{MIA_ROOT}/nonmember_5k"             # 非成员图像根目录

# 用 LoRA 训练用的评测 1000 条 JSON
MEM_EVAL_JSON  = f"{MIA_ROOT}/member_eval_1000.json"     # 成员 = 训练集
NON_EVAL_JSON  = f"{MIA_ROOT}/nonmember_eval_1000.json"  # 非成员 = 未训练集

# 输出 CSV（保持不变）
OUT_CSV = f"{MIA_ROOT}/mia_losses_1000x1000.csv"

# ---------------- 0) 基本检查 ----------------
assert os.path.isdir(LLAMA_REPO), f"❌ LLaMA-Factory 不存在：{LLAMA_REPO}"
assert os.path.isdir(LORA_BEST),  f"❌ LoRA 目录不存在：{LORA_BEST}"
assert os.path.isfile(MEM_EVAL_JSON), f"❌ 缺少 {MEM_EVAL_JSON}"
assert os.path.isfile(NON_EVAL_JSON), f"❌ 缺少 {NON_EVAL_JSON}"
os.makedirs(CACHE_DIR, exist_ok=True)

# ---------------- 1) 合并 LoRA → 独立模型（保持不变） ----------------
print("🔁 Export (merge) base + LoRA → merged dir …")
os.chdir(LLAMA_REPO)
export_cmd = [
    sys.executable, "-m", "llamafactory.cli", "export",
    "--model_name_or_path", BASE_ID,
    "--adapter_name_or_path", LORA_BEST,
    "--template", "qwen2_vl",
    "--finetuning_type", "lora",
    "--export_dir", MERGED_DIR,
    "--export_size", "2",
    "--export_device", "cpu",
    "--cache_dir", CACHE_DIR,
    "--trust_remote_code"
]
ret = subprocess.call(export_cmd)
if ret != 0:
    raise RuntimeError(f"❌ 导出失败（返回码 {ret}），请确认 LoRA_BEST 路径与权限。")
print(f"✅ 合并完成 → {MERGED_DIR}")

# ---------------- 2) 准备 MIA 评测集（各 1000） ----------------
def load_eval_list(json_path, want_n=1000, seed=20251012):
    """支持两种结构：直接 list 或 {'data': [...]}"""
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    data = raw["data"] if isinstance(raw, dict) and "data" in raw else raw
    if not isinstance(data, list):
        raise ValueError(f"{json_path} 不是列表或不含 'data' 列表。")
    if len(data) < want_n:
        raise RuntimeError(f"{json_path} 样本不足 {want_n}（当前 {len(data)}）")
    # 保证和 LoRA 训练一致：优先使用前 1000；如需随机可改成 random.sample
    # random.seed(seed); data = random.sample(data, want_n)
    return data[:want_n]

print("📥 准备 member/nonmember 各 1000（与 LoRA 一致） …")
members_1k    = load_eval_list(MEM_EVAL_JSON, 1000)
nonmembers_1k = load_eval_list(NON_EVAL_JSON, 1000)
print(f"✅ member={len(members_1k)}  nonmember={len(nonmembers_1k)}")

# ---------------- 3) 加载合并后的模型（Vision2Seq） ----------------
print("🔁 加载合并后的模型（用于 MIA，Vision2Seq） …")
processor = AutoProcessor.from_pretrained(MERGED_DIR, trust_remote_code=True)
# A100 支持 bfloat16；如果你的环境不支持，可改成 torch.float16
model = AutoModelForVision2Seq.from_pretrained(
    MERGED_DIR, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True
).eval()
device = next(iter(model.parameters())).device
print(f"✅ device={device}  dtype={next(iter(model.parameters())).dtype}")

# ---------------- 4) Loss Attack（teacher-forced NLL） ----------------
@torch.no_grad()
def sample_loss(rec, img_root: str) -> float:
    """
    计算单样本的 teacher-forced 平均 token 损失。
    简洁稳妥：labels = input_ids（整序列），足够用于 MIA 的相对判别。
    """
    # 1) 读图
    rels = rec.get("images", [])
    if not rels:
        raise ValueError("记录缺少 images 字段")
    img_path = os.path.join(img_root, rels[0])
    image = Image.open(img_path).convert("RGB")

    # 2) 组装 messages：包含英文提示语 + 原问答
    msgs = rec.get("messages", None)
    if not msgs or not isinstance(msgs, list):
        # 兜底：如果没有 messages 字段
        prompt = ("As a medical vision-language model, based on the image, "
                  "generate a professional diagnostic result in English.\n"
                  "请根据这张医学图像，用中文给出专业的诊断结果。")
        answer = rec.get("caption", "")
        msgs = [
            {"role": "user", "content": [{"type":"image"}, {"type":"text","text": prompt}]},
            {"role": "assistant", "content": [{"type":"text","text": answer}]},
        ]
    else:
        # ShareGPT 文本：把 <image> 拆成独立片段，并加入英文指令
        u_content = msgs[0]["content"]
        if isinstance(u_content, str):
            chinese_q = u_content.replace("<image>", "").strip()
            prompt = (
                "As a medical vision-language model, based on the image, "
                "generate a professional diagnostic result in English.\n" + chinese_q
            )
            user_content = [{"type":"image"}, {"type":"text","text": prompt}]
        else:
            # 片段列表：确保包含一段 image；第一段 text 前拼英文提示
            user_content, saw_img = [], False
            for seg in u_content:
                if isinstance(seg, dict) and seg.get("type") == "image":
                    saw_img = True
                    user_content.append(seg)
                elif isinstance(seg, dict) and seg.get("type") == "text":
                    text_val = seg.get("text", "")
                    if not user_content:  # 第一段
                        text_val = (
                            "As a medical vision-language model, based on the image, "
                            "generate a professional diagnostic result in English.\n"
                        ) + text_val
                    user_content.append({"type":"text","text":text_val})
            if not saw_img:
                user_content = [{"type":"image"}] + user_content

        a_text = msgs[1]["content"]
        assist_content = [{"type":"text","text": a_text}] if isinstance(a_text, str) else a_text
        msgs = [{"role":"user","content":user_content},{"role":"assistant","content":assist_content}]

    # 3) 渲染模板，保留 assistant 内容以便 teacher forcing
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

    # 4) 编码并前向（单样本，节省显存）
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(device)
    labels = inputs["input_ids"].clone()
    out = model(**inputs, labels=labels)
    loss_val = float(out.loss.detach().float().item())

    # 释放中间显存
    del inputs, out
    torch.cuda.empty_cache()
    return loss_val

def batch_losses(records, img_root):
    vals = []
    for rec in tqdm(records, desc=f"loss@{os.path.basename(img_root)}"):
        try:
            vals.append(sample_loss(rec, img_root))
        except (UnidentifiedImageError, FileNotFoundError, OSError, ValueError) as e:
            # 单个样本异常不影响整体；记 NaN 并过滤
            vals.append(float("nan"))
        except Exception:
            vals.append(float("nan"))
    return [x for x in vals if not math.isnan(x)]

print("🧮 计算 losses（member 1000） …")
mem_losses = batch_losses(members_1k, MEM_IMGROOT)
print("🧮 计算 losses（nonmember 1000） …")
non_losses = batch_losses(nonmembers_1k, NON_IMGROOT)
print(f"✅ 有效样本：member={len(mem_losses)}  nonmember={len(non_losses)}")

# ---------------- 5) ROC-AUC（loss 越小越像 member → score = -loss） ----------------
if len(mem_losses)==0 or len(non_losses)==0:
    raise RuntimeError("有效样本为 0，无法计算 AUC。请检查 eval-1000 JSON 与图像相对路径是否匹配。")

y_true  = [1]*len(mem_losses) + [0]*len(non_losses)
y_score = [-x for x in mem_losses] + [-x for x in non_losses]
auc = roc_auc_score(y_true, y_score)
print(f"\n🏁 MIA ROC-AUC = {auc:.4f}")

# ---------------- [NEW] 选择 @5% FPR 的工作点并计算指标 ----------------
TARGET_FPR = 0.05

# 基于 y_score（分数越大越像 member）画 ROC，获得各阈值对应的 FPR/TPR
fpr, tpr, thresholds = roc_curve(y_true, y_score, drop_intermediate=False)

# 选择 fpr <= 0.05 中 TPR 最大的点；若不存在（全都 >0.05），则选与 0.05 最接近的点
candidates = [(i, tpr[i]) for i in range(len(fpr)) if fpr[i] <= TARGET_FPR]
if candidates:
    best_idx = max(candidates, key=lambda x: x[1])[0]
else:
    best_idx = min(range(len(fpr)), key=lambda i: abs(fpr[i] - TARGET_FPR))

thr = thresholds[best_idx]

# 用该阈值得到二分类预测（>=thr 判为 member=1）
y_pred = [1 if s >= thr else 0 for s in y_score]

# 统计指标
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)
acc = accuracy_score(y_true, y_pred)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("\n🎯 Operating point @5% FPR (target)")
print(f"  threshold = {thr:.6f}")
print(f"  FPR = {fpr[best_idx]:.4f}")
print(f"  TPR = {tpr[best_idx]:.4f}")
print(f"  Precision = {precision:.4f}")
print(f"  Recall    = {recall:.4f}")
print(f"  F1        = {f1:.4f}")
print(f"  Accuracy  = {acc:.4f}")
print(f"  Confusion Matrix: TP={tp}, FP={fp}, TN={tn}, FN={fn}")

# ---------------- 6) 保存 CSV 明细（保持不变） ----------------
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["split","loss","score(-loss)"])
    for x in mem_losses:
        w.writerow(["member", x, -x])
    for x in non_losses:
        w.writerow(["nonmember", x, -x])
print(f"💾 已保存明细：{OUT_CSV}")


In [ ]:
# @title 3. Entropy MIA
# ========================= MIA: Entropy (negative mean entropy) =========================
import os, json, random, math, csv, warnings
from typing import List, Dict, Any, Optional
from pathlib import Path
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import numpy as np

import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
# ===== [NEW] 额外导入，用于 Precision/Recall/F1 =====
from sklearn.metrics import precision_recall_fscore_support
# ====================================================
from transformers import AutoProcessor, AutoModelForVision2Seq

warnings.filterwarnings("ignore")

# ---------------- 路径（只把数据切到 eval-1000；其它保持不变） ----------------
MIA_ROOT       = "/content/drive/MyDrive/MedTrinity_MIA"
MEM_IMGROOT    = f"{MIA_ROOT}/member_5k"                 # 成员图像根目录（与训练一致）
NON_IMGROOT    = f"{MIA_ROOT}/nonmember_5k"             # 非成员图像根目录
MEM_EVAL_JSON  = f"{MIA_ROOT}/member_eval_1000.json"    # 成员=训练用 eval-1000
NON_EVAL_JSON  = f"{MIA_ROOT}/nonmember_eval_1000.json" # 非成员=未训练 eval-1000

# 合并后的模型目录（你已在 Loss Attack 中导出）
MERGED_DIR     = "/content/drive/MyDrive/Qwen2-VL-med-merged-A100"

OUT_CSV        = f"{MIA_ROOT}/mia_entropy_1000x1000.csv"
SEED           = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "需要 GPU"
device = torch.device("cuda")

# ---------------- 数据：严格使用 eval-1000（不再随机，确保与训练一致） ----------------
def load_eval_list(json_path: str, want_n=1000) -> list:
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    data = raw["data"] if isinstance(raw, dict) and "data" in raw else raw
    if not isinstance(data, list):
        raise ValueError(f"{json_path} 不是列表或不含 'data' 列表。")
    if len(data) < want_n:
        raise RuntimeError(f"{json_path} 样本不足 {want_n}（当前 {len(data)}）")
    # 与 LoRA 训练完全一致：使用前 1000 条（如需随机请改成 random.sample）
    return data[:want_n]

print("📥 准备 member/nonmember 各 1000（与 LoRA 一致） …")
members_1k    = load_eval_list(MEM_EVAL_JSON,  1000)
nonmembers_1k = load_eval_list(NON_EVAL_JSON, 1000)
print(f"✅ member={len(members_1k)}  nonmember={len(nonmembers_1k)}")

# ---------------- 模型 ----------------
print("🔁 加载合并后的模型（Vision2Seq） …")
processor = AutoProcessor.from_pretrained(MERGED_DIR, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MERGED_DIR, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True
).eval()
device = next(iter(model.parameters())).device
print(f"✅ device={device}  dtype={next(iter(model.parameters())).dtype}")

# ---------------- 打包 & 前缀对齐（只统计 assistant 段） ----------------
def _build_msgs(rec: Dict[str, Any]) -> List[Dict[str, Any]]:
    # 规范化成多模态片段：image + 英文提示 + 原问答
    rels = rec.get("images", [])
    assert rels and isinstance(rels, list), "记录缺少 images 字段或格式错误"
    msgs = rec.get("messages", None)

    if not msgs or not isinstance(msgs, list):
        prompt = ("As a medical vision-language model, based on the image, "
                  "generate a professional diagnostic result in English.\n"
                  "请根据这张医学图像，用中文给出专业的诊断结果。")
        answer = rec.get("caption", "")
        return [
            {"role": "user", "content": [{"type":"image"}, {"type":"text","text": prompt}]},
            {"role": "assistant", "content": [{"type":"text","text": answer}]},
        ]

    # ShareGPT 风格：把 <image> 拆片段，并在第一段 text 前加入英文提示
    u = msgs[0]["content"]
    if isinstance(u, str):
        chinese_q = u.replace("<image>", "").strip()
        prompt = (
            "As a medical vision-language model, based on the image, "
            "generate a professional diagnostic result in English.\n" + chinese_q
        )
        user_content = [{"type":"image"}, {"type":"text","text": prompt}]
    else:
        user_content, saw_img = [], False
        for seg in u:
            if isinstance(seg, dict) and seg.get("type") == "image":
                saw_img = True; user_content.append(seg)
            elif isinstance(seg, dict) and seg.get("type") == "text":
                text_val = seg.get("text","")
                if not user_content:
                    text_val = ("As a medical vision-language model, based on the image, "
                                "generate a professional diagnostic result in English.\n") + text_val
                user_content.append({"type":"text","text":text_val})
        if not saw_img:
            user_content = [{"type":"image"}] + user_content

    a = msgs[1]["content"]
    assist_content = [{"type":"text","text": a}] if isinstance(a, str) else a
    return [{"role":"user","content":user_content},{"role":"assistant","content":assist_content}]

def _pack(image: Image.Image, msgs: List[Dict[str,Any]]):
    # 两次 processor(text+image)：让 <image> 在 prefix/full 中展开一致
    user_only = [msgs[0]]
    user_text = processor.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(msgs,      tokenize=False, add_generation_prompt=False)

    enc_prefix = processor(text=[user_text], images=[image], return_tensors="pt").to(device)
    enc_full   = processor(text=[full_text], images=[image], return_tensors="pt").to(device)

    input_ids  = enc_full["input_ids"]
    labels     = input_ids.clone()
    prefix_len = int(enc_prefix["input_ids"].size(1))
    if input_ids.size(1) <= prefix_len:
        return None
    labels[:, :prefix_len] = -100         # 只在 assistant 段监督/计算
    enc_full["labels"] = labels
    return enc_full, prefix_len

@torch.no_grad()
def neg_entropy_one(rec: Dict[str,Any], img_root: str) -> Optional[float]:
    rels = rec.get("images", [])
    if not rels:
        return None
    img_path = os.path.join(img_root, rels[0])
    try:
        image = Image.open(img_path).convert("RGB")
    except (UnidentifiedImageError, FileNotFoundError, OSError):
        return None

    msgs   = _build_msgs(rec)
    packed = _pack(image, msgs)
    if packed is None:
        return None
    inputs, _ = packed
    out = model(**inputs)                      # logits
    logits = out.logits                        # [1, T, V]
    labels = inputs["labels"]                  # [1, T]

    # 只在 assistant token 上做熵
    shift_logits = logits[..., :-1, :]
    shift_labels = labels[...,  1: ]
    mask = shift_labels.ne(-100)[0]            # [T-1]
    if mask.sum() == 0:
        return None

    valid_logits = shift_logits[0][mask].float()   # [N, V]
    p = torch.softmax(valid_logits, dim=-1)
    ent = -(p * (p.add(1e-9).log())).sum(dim=-1)   # per-token entropy
    return float((-ent.mean()).item())             # 负熵（越大越像成员）

def run_side(recs, root):
    scores=[]
    for r in tqdm(recs, desc=f"entropy@{os.path.basename(root)}"):
        try:
            s = neg_entropy_one(r, root)
            if s is not None and math.isfinite(s):
                scores.append(s)
        except Exception:
            # 单样本错误不影响整体
            pass
    return scores

print("🧮 计算 entropy 分数 …")
mem_scores  = run_side(members_1k,    MEM_IMGROOT)
non_scores  = run_side(nonmembers_1k, NON_IMGROOT)
print(f"✅ 有效样本：member={len(mem_scores)}  nonmember={len(non_scores)}")

# ---------------- 评估 ----------------
if len(mem_scores)==0 or len(non_scores)==0:
    raise RuntimeError("有效样本为 0，无法计算 AUC。请检查 eval-1000 JSON 与图像相对路径。")

y  = np.array([1]*len(mem_scores)+[0]*len(non_scores))
sc = np.array(mem_scores+non_scores)

# AUC
auc = roc_auc_score(y, sc)

# 现有最佳阈值（Youden J）保持不变
fpr, tpr, thr = roc_curve(y, sc)
j = np.argmax(tpr - fpr)
acc = accuracy_score(y, (sc>=thr[j]).astype(int))
cm  = confusion_matrix(y, (sc>=thr[j]).astype(int))

print("\n====== Entropy-MIA 结果 ======")
print(f"AUC = {auc:.4f}")
print(f"Best thr = {thr[j]:.4f}")
print(f"Acc = {acc:.4f}")
print("Confusion [[TN FP][FN TP]]:\n", cm)

# ---------------- [NEW] @5% FPR 的 TPR/Precision/Recall/F1/Accuracy ----------------
TARGET_FPR = 0.05

# 在整条 ROC 上挑选 FPR<=5% 中 TPR 最大的点；若不存在则选离 5% 最近的点
candidates = [(i, tpr[i]) for i in range(len(fpr)) if fpr[i] <= TARGET_FPR]
if candidates:
    best_idx = max(candidates, key=lambda x: x[1])[0]
else:
    best_idx = min(range(len(fpr)), key=lambda i: abs(fpr[i] - TARGET_FPR))

thr_5 = thr[best_idx]
y_pred_5 = (sc >= thr_5).astype(int)

prec_5, rec_5, f1_5, _ = precision_recall_fscore_support(
    y, y_pred_5, average="binary", zero_division=0
)
acc_5 = accuracy_score(y, y_pred_5)
cm_5  = confusion_matrix(y, y_pred_5)

print("\n🎯 Operating point @5% FPR (target)")
print(f"  threshold = {thr_5:.6f}")
print(f"  FPR = {fpr[best_idx]:.4f}")
print(f"  TPR = {tpr[best_idx]:.4f}")
print(f"  Precision = {prec_5:.4f}")
print(f"  Recall    = {rec_5:.4f}")
print(f"  F1        = {f1_5:.4f}")
print(f"  Accuracy  = {acc_5:.4f}")
print("  Confusion [[TN FP][FN TP]]:\n", cm_5)

# ---------------- 保存 CSV ----------------
with open(OUT_CSV,"w",newline="",encoding="utf-8") as f:
    w=csv.writer(f); w.writerow(["split","score(neg-entropy)"])
    for s in mem_scores:  w.writerow(["member",s])
    for s in non_scores: w.writerow(["nonmember",s])
print(f"💾 已保存明细：{OUT_CSV}")


In [ ]:
# @title 4. Min-K
# ========================= MIA: Min-K (fixed K smallest NLL) =========================
import os, json, random, math, csv, warnings
from typing import List, Dict, Any, Optional
from pathlib import Path
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import numpy as np

import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
# ===== [NEW] 额外导入，用于 Precision/Recall/F1 =====
from sklearn.metrics import precision_recall_fscore_support
# ====================================================
from transformers import AutoProcessor, AutoModelForVision2Seq

warnings.filterwarnings("ignore")

# ---------------- 路径（把数据切到 eval-1000；其它保持不变） ----------------
MIA_ROOT       = "/content/drive/MyDrive/MedTrinity_MIA"
MEM_IMGROOT    = f"{MIA_ROOT}/member_5k"                 # 成员图像根目录（与训练一致）
NON_IMGROOT    = f"{MIA_ROOT}/nonmember_5k"             # 非成员图像根目录
MEM_EVAL_JSON  = f"{MIA_ROOT}/member_eval_1000.json"    # 成员=训练用 eval-1000
NON_EVAL_JSON  = f"{MIA_ROOT}/nonmember_eval_1000.json" # 非成员=未训练 eval-1000

# 合并后的模型目录（你已在 Loss Attack 中导出）
MERGED_DIR     = "/content/drive/MyDrive/Qwen2-VL-med-merged-A100"

OUT_CSV        = f"{MIA_ROOT}/mia_minkK_1000x1000.csv"

# 关键超参
K_MIN = 20   # 固定取最小的 K 个 NLL（可调：16~64 常用）

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "需要 GPU"
device = torch.device("cuda")

# ---------------- 数据：严格使用 eval-1000（不再随机，确保与训练一致） ----------------
def load_eval_list(json_path: str, want_n=1000) -> list:
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    data = raw["data"] if isinstance(raw, dict) and "data" in raw else raw
    if not isinstance(data, list):
        raise ValueError(f"{json_path} 不是列表或不含 'data' 列表。")
    if len(data) < want_n:
        raise RuntimeError(f"{json_path} 样本不足 {want_n}（当前 {len(data)}）")
    # 与 LoRA 训练完全一致：使用前 1000 条（如需随机请改成 random.sample）
    return data[:want_n]

print("📥 准备 member/nonmember 各 1000（与 LoRA 一致） …")
members_1k    = load_eval_list(MEM_EVAL_JSON,  1000)
nonmembers_1k = load_eval_list(NON_EVAL_JSON, 1000)
print(f"✅ member={len(members_1k)}  nonmember={len(nonmembers_1k)}")

# ---------------- 模型 ----------------
print("🔁 加载合并后的模型（Vision2Seq） …")
processor = AutoProcessor.from_pretrained(MERGED_DIR, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MERGED_DIR, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True
).eval()
device = next(iter(model.parameters())).device
print(f"✅ device={device}  dtype={next(iter(model.parameters())).dtype}")

# ---------------- 打包 & 前缀对齐（只统计 assistant 段） ----------------
def _build_msgs(rec: Dict[str, Any]) -> List[Dict[str, Any]]:
    # 规范化成多模态片段：image + 英文提示 + 原问答
    rels = rec.get("images", [])
    assert rels and isinstance(rels, list), "记录缺少 images 字段或格式错误"
    msgs = rec.get("messages", None)

    if not msgs or not isinstance(msgs, list):
        prompt = ("As a medical vision-language model, based on the image, "
                  "generate a professional diagnostic result in English.\n"
                  "请根据这张医学图像，用中文给出专业的诊断结果。")
        answer = rec.get("caption", "")
        return [
            {"role": "user", "content": [{"type":"image"}, {"type":"text","text": prompt}]},
            {"role": "assistant", "content": [{"type":"text","text": answer}]},
        ]

    # ShareGPT 风格：把 <image> 拆片段，并在第一段 text 前加入英文提示
    u = msgs[0]["content"]
    if isinstance(u, str):
        chinese_q = u.replace("<image>", "").strip()
        prompt = (
            "As a medical vision-language model, based on the image, "
            "generate a professional diagnostic result in English.\n" + chinese_q
        )
        user_content = [{"type":"image"}, {"type":"text","text": prompt}]
    else:
        user_content, saw_img = [], False
        for seg in u:
            if isinstance(seg, dict) and seg.get("type") == "image":
                saw_img = True; user_content.append(seg)
            elif isinstance(seg, dict) and seg.get("type") == "text":
                text_val = seg.get("text","")
                if not user_content:
                    text_val = ("As a medical vision-language model, based on the image, "
                                "generate a professional diagnostic result in English.\n") + text_val
                user_content.append({"type":"text","text":text_val})
        if not saw_img:
            user_content = [{"type":"image"}] + user_content

    a = msgs[1]["content"]
    assist_content = [{"type":"text","text": a}] if isinstance(a, str) else a
    return [{"role":"user","content":user_content},{"role":"assistant","content":assist_content}]

def _pack(image: Image.Image, msgs: List[Dict[str,Any]]):
    # 两次 processor(text+image)：让 <image> 在 prefix/full 中展开一致
    user_only = [msgs[0]]
    user_text = processor.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(msgs,      tokenize=False, add_generation_prompt=False)

    enc_prefix = processor(text=[user_text], images=[image], return_tensors="pt").to(device)
    enc_full   = processor(text=[full_text], images=[image], return_tensors="pt").to(device)

    input_ids  = enc_full["input_ids"]
    labels     = input_ids.clone()
    prefix_len = int(enc_prefix["input_ids"].size(1))
    if input_ids.size(1) <= prefix_len:
        return None
    labels[:, :prefix_len] = -100         # 只在 assistant 段监督/计算
    enc_full["labels"] = labels
    return enc_full, prefix_len

@torch.no_grad()
def mink_score_one(rec: Dict[str,Any], img_root: str, k: int) -> Optional[float]:
    rels = rec.get("images", [])
    if not rels:
        return None
    img_path = os.path.join(img_root, rels[0])
    try:
        image = Image.open(img_path).convert("RGB")
    except (UnidentifiedImageError, FileNotFoundError, OSError):
        return None

    msgs   = _build_msgs(rec)
    packed = _pack(image, msgs)
    if packed is None:
        return None
    inputs, _ = packed
    out = model(**inputs)
    logits = out.logits                        # [1, T, V]
    labels = inputs["labels"]                  # [1, T]

    # 只在 assistant token 上计算 per-token NLL
    shift_logits = logits[..., :-1, :]
    shift_labels = labels[...,  1: ]
    mask = shift_labels.ne(-100)[0]            # [T-1]
    if mask.sum() == 0:
        return None

    logp   = F.log_softmax(shift_logits[0].float(), dim=-1)  # [T-1, V]
    target = shift_labels[0].clamp(min=0)                    # [T-1]
    tok_lp = logp[torch.arange(logp.size(0)), target]        # [T-1]
    tok_nll = -tok_lp[mask]                                  # [N_valid]

    if tok_nll.numel() == 0:
        return None

    k_use = min(int(k), int(tok_nll.numel()))
    # 取最小的 K 个 NLL（= 对 -NLL 取 Top-K）
    vals, _ = torch.topk(-tok_nll, k_use)     # 最大的 -NLL ↔ 最小的 NLL
    score = float((-vals).mean().item())      # 负的(最小NLL均值) → 越大越像成员
    return score

def run_side(recs, root, k: int):
    out=[]
    for r in tqdm(recs, desc=f"minK@{os.path.basename(root)}"):
        try:
            s = mink_score_one(r, root, k)
            if s is not None and math.isfinite(s):
                out.append(s)
        except Exception:
            # 单样本错误不影响整体
            pass
    return out

print(f"🧮 计算 Min-K (K={K_MIN}) 分数 …")
mem_scores  = run_side(members_1k,    MEM_IMGROOT, K_MIN)
non_scores  = run_side(nonmembers_1k, NON_IMGROOT, K_MIN)
print(f"✅ 有效样本：member={len(mem_scores)}  nonmember={len(non_scores)}")

# ---------------- 评估 ----------------
if len(mem_scores)==0 or len(non_scores)==0:
    raise RuntimeError("有效样本为 0，无法计算 AUC。请检查 eval-1000 JSON 与图像相对路径。")

y  = np.array([1]*len(mem_scores)+[0]*len(non_scores))
sc = np.array(mem_scores+non_scores)

# AUC
auc = roc_auc_score(y, sc)

# 现有最佳阈值（Youden J）保持不变
fpr, tpr, thr = roc_curve(y, sc)
j = np.argmax(tpr - fpr)
acc = accuracy_score(y, (sc>=thr[j]).astype(int))
cm  = confusion_matrix(y, (sc>=thr[j]).astype(int))

print("\n====== Min-K MIA 结果 ======")
print(f"AUC = {auc:.4f}")
print(f"Best thr = {thr[j]:.4f}")
print(f"Acc = {acc:.4f}")
print("Confusion [[TN FP][FN TP]]:\n", cm)

# ---------------- [NEW] @5% FPR 的 TPR/Precision/Recall/F1/Accuracy ----------------
TARGET_FPR = 0.05

# 在整条 ROC 上挑选 FPR<=5% 中 TPR 最大的点；若不存在则选离 5% 最近的点
candidates = [(i, tpr[i]) for i in range(len(fpr)) if fpr[i] <= TARGET_FPR]
if candidates:
    best_idx = max(candidates, key=lambda x: x[1])[0]
else:
    best_idx = min(range(len(fpr)), key=lambda i: abs(fpr[i] - TARGET_FPR))

thr_5 = thr[best_idx]
y_pred_5 = (sc >= thr_5).astype(int)

prec_5, rec_5, f1_5, _ = precision_recall_fscore_support(
    y, y_pred_5, average="binary", zero_division=0
)
acc_5 = accuracy_score(y, y_pred_5)
cm_5  = confusion_matrix(y, y_pred_5)

print("\n🎯 Operating point @5% FPR (target)")
print(f"  threshold = {thr_5:.6f}")
print(f"  FPR = {fpr[best_idx]:.4f}")
print(f"  TPR = {tpr[best_idx]:.4f}")
print(f"  Precision = {prec_5:.4f}")
print(f"  Recall    = {rec_5:.4f}")
print(f"  F1        = {f1_5:.4f}")
print(f"  Accuracy  = {acc_5:.4f}")
print("  Confusion [[TN FP][FN TP]]:\n", cm_5)

# ---------------- 保存 CSV ----------------
with open(OUT_CSV,"w",newline="",encoding="utf-8") as f:
    w=csv.writer(f); w.writerow(["split","score(minK, -mean smallest K NLL)"])
    for s in mem_scores:  w.writerow(["member",s])
    for s in non_scores: w.writerow(["nonmember",s])
print(f"💾 已保存明细：{OUT_CSV}")


In [ ]:
# @title 5. Min-K++
# ========================= MIA: Min-K++ (adaptive, filtered, length-normalized) =========================
import os, json, random, math, csv, warnings, re
from pathlib import Path
from typing import List, Dict, Any, Optional
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import numpy as np

import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
# ===== [NEW] 额外导入，用于 Precision/Recall/F1 =====
from sklearn.metrics import precision_recall_fscore_support
# ====================================================
from transformers import AutoProcessor, AutoModelForVision2Seq

warnings.filterwarnings("ignore")

# ---------------- 路径（把数据切到 eval-1000；其它保持不变） ----------------
MIA_ROOT       = "/content/drive/MyDrive/MedTrinity_MIA"
MEM_IMGROOT    = f"{MIA_ROOT}/member_5k"                 # 成员图像根目录（与训练一致）
NON_IMGROOT    = f"{MIA_ROOT}/nonmember_5k"             # 非成员图像根目录
MEM_EVAL_JSON  = f"{MIA_ROOT}/member_eval_1000.json"    # 成员=训练用 eval-1000
NON_EVAL_JSON  = f"{MIA_ROOT}/nonmember_eval_1000.json" # 非成员=未训练 eval-1000
MERGED_DIR     = "/content/drive/MyDrive/Qwen2-VL-med-merged-A100"
OUT_CSV        = f"{MIA_ROOT}/mia_minkPP_1000x1000.csv"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "需要 GPU"
device = torch.device("cuda")

# ---------------- Min-K++ 超参 ----------------
ALPHA   = 0.20   # 底部比例（20%）
K_MIN   = 8      # 最小 k
K_MAX   = 64     # 最大 k
RE_PUNC = re.compile(r"^\W+$", re.UNICODE)  # 纯非字母数字（标点/符号）

# ---------------- 使用 eval-1000 固定列表（与 LoRA 训练保持一致） ----------------
def load_eval_list(json_path: str, want_n=1000) -> list:
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    data = raw["data"] if isinstance(raw, dict) and "data" in raw else raw
    if not isinstance(data, list):
        raise ValueError(f"{json_path} 不是列表或不含 'data' 列表。")
    if len(data) < want_n:
        raise RuntimeError(f"{json_path} 样本不足 {want_n}（当前 {len(data)}）")
    # 与 LoRA 训练完全一致：使用前 1000 条（如需随机可改为 random.sample）
    return data[:want_n]

print("📥 准备 member/nonmember 各 1000（与 LoRA 一致） …")
members_1k    = load_eval_list(MEM_EVAL_JSON,  1000)
nonmembers_1k = load_eval_list(NON_EVAL_JSON, 1000)
print(f"✅ member={len(members_1k)}  nonmember={len(nonmembers_1k)}")

# ---------------- 模型 ----------------
print("🔁 加载合并后的模型（Vision2Seq） …")
processor = AutoProcessor.from_pretrained(MERGED_DIR, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MERGED_DIR, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True
).eval()
device = next(iter(model.parameters())).device
print(f"✅ device={device}  dtype={next(iter(model.parameters())).dtype}")

# ---------------- 打包 & 前缀对齐（只统计 assistant 段） ----------------
def _build_msgs(rec: Dict[str, Any]) -> List[Dict[str, Any]]:
    rels = rec.get("images", [])
    assert rels and isinstance(rels, list), "记录缺少 images 字段或格式错误"
    msgs = rec.get("messages", None)

    if not msgs or not isinstance(msgs, list):
        prompt = ("As a medical vision-language model, based on the image, "
                  "generate a professional diagnostic result in English.\n"
                  "请根据这张医学图像，用中文给出专业的诊断结果。")
        answer = rec.get("caption", "")
        return [
            {"role": "user", "content": [{"type":"image"}, {"type":"text","text": prompt}]},
            {"role": "assistant", "content": [{"type":"text","text": answer}]},
        ]

    # ShareGPT 风格：把 <image> 拆片段，并在第一段 text 前加入英文提示
    u = msgs[0]["content"]
    if isinstance(u, str):
        chinese_q = u.replace("<image>", "").strip()
        prompt = (
            "As a medical vision-language model, based on the image, "
            "generate a professional diagnostic result in English.\n" + chinese_q
        )
        user_content = [{"type":"image"}, {"type":"text","text": prompt}]
    else:
        user_content, saw_img = [], False
        for seg in u:
            if isinstance(seg, dict) and seg.get("type") == "image":
                saw_img = True; user_content.append(seg)
            elif isinstance(seg, dict) and seg.get("type") == "text":
                text_val = seg.get("text","")
                if not user_content:
                    text_val = ("As a medical vision-language model, based on the image, "
                                "generate a professional diagnostic result in English.\n") + text_val
                user_content.append({"type":"text","text":text_val})
        if not saw_img:
            user_content = [{"type":"image"}] + user_content

    a = msgs[1]["content"]
    assist_content = [{"type":"text","text": a}] if isinstance(a, str) else a
    return [{"role":"user","content":user_content},{"role":"assistant","content":assist_content}]

def _pack(image: Image.Image, msgs: List[Dict[str,Any]]):
    # 两次 processor(text+image)：让 <image> 在 prefix/full 中展开一致
    user_only = [msgs[0]]
    user_text = processor.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(msgs,      tokenize=False, add_generation_prompt=False)

    enc_prefix = processor(text=[user_text], images=[image], return_tensors="pt").to(device)
    enc_full   = processor(text=[full_text], images=[image], return_tensors="pt").to(device)

    input_ids  = enc_full["input_ids"]
    labels     = input_ids.clone()
    prefix_len = int(enc_prefix["input_ids"].size(1))
    if input_ids.size(1) <= prefix_len:
        return None
    labels[:, :prefix_len] = -100         # 只在 assistant 段监督/计算
    enc_full["labels"] = labels
    return enc_full, prefix_len

def _is_informative(piece: str) -> bool:
    s = piece.strip()
    if not s: return False
    if RE_PUNC.match(s): return False
    return True

@torch.no_grad()
def mink_pp_score_one(rec: Dict[str,Any], img_root: str) -> Optional[float]:
    # 读图
    rels = rec.get("images", [])
    if not rels:
        return None
    img_path = os.path.join(img_root, rels[0])
    try:
        image = Image.open(img_path).convert("RGB")
    except (UnidentifiedImageError, FileNotFoundError, OSError):
        return None

    # 打包
    msgs   = _build_msgs(rec)
    packed = _pack(image, msgs)
    if packed is None:
        return None

    # 前向 & 取目标 token 的 NLL
    inputs, _ = packed
    out    = model(**inputs)
    logits = out.logits                        # [1, T, V]
    labels = inputs["labels"]                  # [1, T]

    shift_logits = logits[..., :-1, :]
    shift_labels = labels[...,  1: ]
    mask = shift_labels.ne(-100)[0]            # [T-1]
    if mask.sum() == 0:
        return None

    logp   = F.log_softmax(shift_logits[0].float(), dim=-1)  # [T-1, V]
    target = shift_labels[0].clamp(min=0)                    # [T-1]

    # ——— 过滤“无信息”token：空白/纯标点（根据 tokenizer 解码单 token）———
    idxs = torch.nonzero(mask, as_tuple=False).squeeze(1)    # indices in shift domain
    keep = []
    for i in idxs.tolist():
        # +1 因为 shift：当前 token 的 label 对应原 input_ids 的 i+1
        tok_id = int(inputs["input_ids"][0, i+1].item())
        piece  = processor.tokenizer.decode([tok_id], skip_special_tokens=True)
        if _is_informative(piece):
            keep.append(i)
    if not keep:
        return None
    keep = torch.tensor(keep, device=logp.device)

    tok_nll = -logp[keep, target[keep]]        # 仅保留“信息性”token 的 NLL
    L = tok_nll.numel()
    if L == 0:
        return None

    # ——— 自适应 k：k = clamp([ceil(ALPHA * L)], [K_MIN, K_MAX], [1, L]) ———
    k = int(math.ceil(ALPHA * L))
    k = max(K_MIN, min(K_MAX, k))
    k = min(k, L)

    # 取最小的 k 个 NLL（= 对 -NLL 取 Top-k）
    vals, _ = torch.topk(-tok_nll, k)         # 最大的 -NLL ↔ 最小的 NLL
    score = float((-vals).mean().item()) / math.sqrt(L)   # 长度归一化
    return score

def run_side(recs, root):
    out=[]
    for r in tqdm(recs, desc=f"minK++@{os.path.basename(root)}"):
        try:
            s = mink_pp_score_one(r, root)
            if s is not None and math.isfinite(s):
                out.append(s)
        except Exception:
            pass
    return out

print("🧮 计算 Min-K++ 分数 …")
mem_scores  = run_side(members_1k,    MEM_IMGROOT)
non_scores  = run_side(nonmembers_1k, NON_IMGROOT)
print(f"✅ 有效样本：member={len(mem_scores)}  nonmember={len(non_scores)}")

# ---------------- 评估 ----------------
if len(mem_scores)==0 or len(non_scores)==0:
    raise RuntimeError("有效样本为 0，无法计算 AUC。请检查 eval-1000 JSON 与图像相对路径。")

y  = np.array([1]*len(mem_scores)+[0]*len(non_scores))
sc = np.array(mem_scores+non_scores)

# AUC
auc = roc_auc_score(y, sc)

# 现有最佳阈值（Youden J）保持不变
fpr, tpr, thr = roc_curve(y, sc)
j = np.argmax(tpr - fpr)
acc = accuracy_score(y, (sc>=thr[j]).astype(int))
cm  = confusion_matrix(y, (sc>=thr[j]).astype(int))

print("\n====== Min-K++ MIA 结果 ======")
print(f"AUC = {auc:.4f}")
print(f"Best thr = {thr[j]:.4f}")
print(f"Acc = {acc:.4f}")
print("Confusion [[TN FP][FN TP]]:\n", cm)

# ---------------- [NEW] @5% FPR 的 TPR/Precision/Recall/F1/Accuracy ----------------
TARGET_FPR = 0.05

# 在整条 ROC 上挑选 FPR<=5% 中 TPR 最大的点；若不存在则选离 5% 最近的点
candidates = [(i, tpr[i]) for i in range(len(fpr)) if fpr[i] <= TARGET_FPR]
if candidates:
    best_idx = max(candidates, key=lambda x: x[1])[0]
else:
    best_idx = min(range(len(fpr)), key=lambda i: abs(fpr[i] - TARGET_FPR))

thr_5 = thr[best_idx]
y_pred_5 = (sc >= thr_5).astype(int)

prec_5, rec_5, f1_5, _ = precision_recall_fscore_support(
    y, y_pred_5, average="binary", zero_division=0
)
acc_5 = accuracy_score(y, y_pred_5)
cm_5  = confusion_matrix(y, y_pred_5)

print("\n🎯 Operating point @5% FPR (target)")
print(f"  threshold = {thr_5:.6f}")
print(f"  FPR = {fpr[best_idx]:.4f}")
print(f"  TPR = {tpr[best_idx]:.4f}")
print(f"  Precision = {prec_5:.4f}")
print(f"  Recall    = {rec_5:.4f}")
print(f"  F1        = {f1_5:.4f}")
print(f"  Accuracy  = {acc_5:.4f}")
print("  Confusion [[TN FP][FN TP]]:\n", cm_5)

# ---------------- 保存 CSV ----------------
with open(OUT_CSV,"w",newline="",encoding="utf-8") as f:
    w=csv.writer(f); w.writerow(["split","score(minK++, filtered & normed)"])
    for s in mem_scores:  w.writerow(["member",s])
    for s in non_scores: w.writerow(["nonmember",s])
print(f"💾 已保存明细：{OUT_CSV}")


In [ ]:
# @title 6. ModRényi
# ========================= MIA: ModRényi*（图像视图 + 文本视图；assistant 段 logits） =========================
import os, json, random, math, csv, warnings, re, io, subprocess, sys
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from PIL import Image, ImageFilter, UnidentifiedImageError
from tqdm import tqdm
import numpy as np

import torch
import torch.nn.functional as F
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)
from transformers import AutoProcessor, AutoModelForVision2Seq

# ---------------- 新增：与 Loss Attack 相同的 LoRA 合并与数据路径 ----------------
LLAMA_REPO = "/content/LLaMA-Factory"
BASE_ID    = "Qwen/Qwen2-VL-2B-Instruct"
CACHE_DIR  = "/content/hf_cache"

# 你训练好的 LoRA 最佳 checkpoint（保持不变）
LORA_BEST  = "/content/Qwen2-VL-med-lora-A100/checkpoint-315"

# 合并后模型保存到 Drive（保持不变）
MERGED_DIR = "/content/drive/MyDrive/Qwen2-VL-med-merged-A100"

# ================= 数据切换：与 LoRA 训练一致（eval-1000） =================
MIA_ROOT       = "/content/drive/MyDrive/MedTrinity_MIA"
MEM_IMGROOT    = f"{MIA_ROOT}/member_5k"                 # 图像根目录与训练相同
NON_IMGROOT    = f"{MIA_ROOT}/nonmember_5k"             # 非成员图像根目录
MEM_EVAL_JSON  = f"{MIA_ROOT}/member_eval_1000.json"     # 成员 = 训练集评测子集
NON_EVAL_JSON  = f"{MIA_ROOT}/nonmember_eval_1000.json"  # 非成员 = 未训练评测子集

# ---------------- 基本检查 + 合并 LoRA → 独立模型（保持与 Loss Attack 一致） ----------------
assert os.path.isdir(LLAMA_REPO), f"❌ LLaMA-Factory 不存在：{LLAMA_REPO}"
assert os.path.isdir(LORA_BEST),  f"❌ LoRA 目录不存在：{LORA_BEST}"
assert os.path.isfile(MEM_EVAL_JSON), f"❌ 缺少 {MEM_EVAL_JSON}"
assert os.path.isfile(NON_EVAL_JSON), f"❌ 缺少 {NON_EVAL_JSON}"
os.makedirs(CACHE_DIR, exist_ok=True)

print("🔁 Export (merge) base + LoRA → merged dir …")
cwd_backup = os.getcwd()
try:
    os.chdir(LLAMA_REPO)
    export_cmd = [
        sys.executable, "-m", "llamafactory.cli", "export",
        "--model_name_or_path", BASE_ID,
        "--adapter_name_or_path", LORA_BEST,
        "--template", "qwen2_vl",
        "--finetuning_type", "lora",
        "--export_dir", MERGED_DIR,
        "--export_size", "2",
        "--export_device", "cpu",
        "--cache_dir", CACHE_DIR,
        "--trust_remote_code"
    ]
    ret = subprocess.call(export_cmd)
    if ret != 0:
        raise RuntimeError(f"❌ 导出失败（返回码 {ret}），请确认 LoRA_BEST 路径与权限。")
finally:
    os.chdir(cwd_backup)
print(f"✅ 合并完成 → {MERGED_DIR}")

# ---------------- 数据（严格使用 eval-1000，保持 Loss Attack 的取前1000策略） ----------------
def load_eval_list(json_path, want_n=1000, seed=20251012):
    """支持两种结构：直接 list 或 {'data': [...]}；与 Loss Attack 一致取前 1000。"""
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    data = raw["data"] if isinstance(raw, dict) and "data" in raw else raw
    if not isinstance(data, list):
        raise ValueError(f"{json_path} 不是列表或不含 'data' 列表。")
    if len(data) < want_n:
        raise RuntimeError(f"{json_path} 样本不足 {want_n}（当前 {len(data)}）")
    return data[:want_n]

print("📥 准备 member/nonmember 各 1000（与 LoRA 一致） …")
members_1k    = load_eval_list(MEM_EVAL_JSON, 1000)
nonmembers_1k = load_eval_list(NON_EVAL_JSON, 1000)
print(f"✅ member={len(members_1k)}  nonmember={len(nonmembers_1k)}")

# ---------------- 模型加载（与 Loss/Entropy 脚本一致：merged Vision2Seq） ----------------
warnings.filterwarnings("ignore")
assert torch.cuda.is_available(), "需要 GPU"
device = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("🔁 加载合并后的模型（Vision2Seq） …")
processor = AutoProcessor.from_pretrained(MERGED_DIR, trust_remote_code=True)
# A100 支持 bfloat16；如果环境不支持，可改 torch.float16
model = AutoModelForVision2Seq.from_pretrained(
    MERGED_DIR, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True
).eval()
device = next(iter(model.parameters())).device
print(f"✅ device={device}  dtype={next(iter(model.parameters())).dtype}")

# ---------------- 以下为原 ModRényi* 方法与细节（保持不变，仅复用上面的模型/数据） ----------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------------- ModRényi*：视图配置 ----------------
IMG_VIEWS = 5     # 原图 + 4 种轻中度增广（与原方法一致）
TXT_VIEWS = 3     # 原文 + 2 个轻扰动
RENYI_ALPHAS = (0.5, 2.0)
LENGTH_NORM  = True

# 文本扰动：保持语义，大小写/空白
def normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def make_text_views(answer_text: str, want: int = TXT_VIEWS) -> List[str]:
    base = str(answer_text or "").strip()
    cand = [base, base.lower(), normalize_spaces(base)]
    out, seen = [], set()
    for c in cand:
        if c not in seen:
            out.append(c); seen.add(c)
    return out[:max(1, want)]

# 图像增广（PIL）——与原 ModRényi* 接近
from torchvision.transforms import (
    RandomResizedCrop, RandomRotation, RandomAffine, ColorJitter, InterpolationMode
)
def make_image_views(pil: Image.Image, want: int = IMG_VIEWS) -> List[Image.Image]:
    views = [pil]
    views.append(RandomResizedCrop(size=(256, 256), scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC)(pil))
    views.append(RandomRotation(degrees=30, interpolation=InterpolationMode.BICUBIC, expand=False)(pil))
    views.append(RandomAffine(degrees=20, translate=(0.08, 0.08), scale=(0.9, 1.1), interpolation=InterpolationMode.BICUBIC)(pil))
    views.append(ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25, hue=0.05)(pil))
    return views[:max(1, want)]

# ---------------- Qwen2-VL 的消息构造（沿用 Entropy 的容错风格） ----------------
PROMPT_PREFIX = (
    "As a medical vision-language model, based on the image, "
    "generate a professional diagnostic result in English.\n"
)

def _extract_answer_text(rec: Dict[str, Any]) -> str:
    msgs = rec.get("messages", None)
    if not msgs or not isinstance(msgs, list):
        return str(rec.get("caption", "")).strip()
    a = msgs[1]["content"]
    return a if isinstance(a, str) else str(a)

def _build_msgs_with_answer(rec: Dict[str,Any], answer_text: str) -> List[Dict[str,Any]]:
    # 与 Entropy 一致地构造 user；assistant 用传入的 answer_text（用于文本视图扰动）
    msgs = rec.get("messages", None)
    if not msgs or not isinstance(msgs, list):
        prompt = PROMPT_PREFIX + "请根据这张医学图像，用中文给出专业的诊断结果。"
        return [
            {"role": "user", "content": [{"type":"image"}, {"type":"text","text": prompt}]},
            {"role": "assistant", "content": [{"type":"text","text": answer_text}]},
        ]
    u = msgs[0]["content"]
    if isinstance(u, str):
        chinese_q = u.replace("<image>", "").strip()
        prompt = PROMPT_PREFIX + chinese_q
        user_content = [{"type":"image"}, {"type":"text","text": prompt}]
    else:
        user_content, saw_img = [], False
        for seg in u:
            if isinstance(seg, dict) and seg.get("type") == "image":
                saw_img = True; user_content.append(seg)
            elif isinstance(seg, dict) and seg.get("type") == "text":
                text_val = seg.get("text","")
                if not user_content:
                    text_val = PROMPT_PREFIX + text_val
                user_content.append({"type":"text","text":text_val})
        if not saw_img:
            user_content = [{"type":"image"}] + user_content
    assist_content = [{"type":"text","text": answer_text}]
    return [{"role":"user","content":user_content},{"role":"assistant","content":assist_content}]

def _pack_for_qwen(image: Image.Image, msgs: List[Dict[str,Any]]):
    # 两次打包以获得前缀长度；与 Entropy 一致，只在 assistant 段监督/取 logits
    user_only = [msgs[0]]
    user_text = processor.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(msgs,      tokenize=False, add_generation_prompt=False)

    enc_prefix = processor(text=[user_text], images=[image], return_tensors="pt").to(device)
    enc_full   = processor(text=[full_text], images=[image], return_tensors="pt").to(device)

    input_ids  = enc_full["input_ids"]
    labels     = input_ids.clone()
    prefix_len = int(enc_prefix["input_ids"].size(1))
    if input_ids.size(1) <= prefix_len:
        return None
    labels[:, :prefix_len] = -100
    enc_full["labels"] = labels
    return enc_full

# ---------------- 从 logits 取 assistant 段的 logP[T,V] ----------------
@torch.no_grad()
def assistant_logp_matrix_qwen(image: Image.Image, rec: Dict[str,Any], answer_text: str) -> Optional[np.ndarray]:
    msgs = _build_msgs_with_answer(rec, answer_text)
    enc  = _pack_for_qwen(image, msgs)
    if enc is None:
        return None
    out = model(**enc)               # logits
    logits = out.logits              # [1, L, V]
    labels = enc["labels"]           # [1, L]
    # next-token 对齐，仅取 assistant
    shift_logits = logits[..., :-1, :]
    shift_labels = labels[...,  1: ]
    mask = shift_labels.ne(-100)[0]
    if mask.sum().item() == 0:
        return None
    logp = F.log_softmax(shift_logits[0].float(), dim=-1)  # [L-1,V] → float32
    sel = logp[mask]                                       # [T, V]
    if sel.numel() == 0:
        return None
    return sel.detach().cpu().numpy()

# ---------------- Rényi 熵 + 稳健聚合 + 融合（与原 ModRényi* 一致） ----------------
def renyi_entropy_from_logp(logp_tok: np.ndarray, alpha: float) -> float:
    a = alpha * logp_tok
    m = np.max(a)
    return float((1.0/(1.0-alpha)) * (m + np.log(np.exp(a - m).sum())))

def renyi_seq_from_logp(logp_seq: np.ndarray, alpha: float) -> float:
    vals = [renyi_entropy_from_logp(logp_seq[t], alpha) for t in range(logp_seq.shape[0])]
    return float(np.mean(vals)) if vals else float("nan")

def robust_aggregate(values: List[float], trim: float = 0.1) -> float:
    arr = np.array([v for v in values if np.isfinite(v)], dtype=float)
    if arr.size == 0: return float("nan")
    arr.sort()
    k = int(math.floor(trim * arr.size))
    if k * 2 < arr.size:
        arr = arr[k: arr.size - k]
    return float(np.mean(arr))

def fused_modrenyi_score(image_views_logp: List[Optional[np.ndarray]],
                         text_views_logp:  List[Optional[np.ndarray]],
                         alphas=RENYI_ALPHAS,
                         length_norm=LENGTH_NORM) -> float:
    parts = []
    for views in [image_views_logp, text_views_logp]:
        vv = [x for x in views if x is not None]
        if not vv:
            parts.extend([float("nan")] * len(alphas))
            continue
        T = vv[0].shape[0]
        for a in alphas:
            per_view = []
            for lp in vv:
                if lp is None or not np.isfinite(lp).all(): continue
                H = renyi_seq_from_logp(lp, a)
                per_view.append(H)
            if not per_view:
                parts.append(float("nan")); continue
            H_bar = robust_aggregate(per_view, trim=0.1)
            if length_norm and T > 0:
                H_bar = H_bar / math.sqrt(T)
            parts.append(-H_bar)  # 负熵：越大越像成员
    valid = np.array([x for x in parts if np.isfinite(x)], dtype=float)
    if valid.size == 0:
        return float("nan")
    if valid.size >= 2 and np.std(valid) > 1e-12:
        z = (valid - np.mean(valid)) / np.std(valid)
        return float(np.mean(z))
    else:
        return float(np.mean(valid))

# ---------------- 视图适配器（与原方法相同思想） ----------------
def image_views_logp_list_qwen(image: Image.Image, rec: Dict[str,Any], answer_text: str) -> List[Optional[np.ndarray]]:
    views = make_image_views(image, IMG_VIEWS)
    out = []
    for v in views:
        try:
            out.append(assistant_logp_matrix_qwen(v, rec, answer_text))
        except Exception:
            out.append(None)
    return out

def text_views_logp_list_qwen(image: Image.Image, rec: Dict[str,Any], base_answer: str) -> List[Optional[np.ndarray]]:
    texts = make_text_views(base_answer, TXT_VIEWS)
    out = []
    for t in texts:
        try:
            out.append(assistant_logp_matrix_qwen(image, rec, t))
        except Exception:
            out.append(None)
    return out

# ---------------- I/O ----------------
def _safe_open_image(path: str) -> Optional[Image.Image]:
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

def _img_root_for_source(source: str) -> str:
    return MEM_IMGROOT if source == "member" else NON_IMGROOT

def fused_score_for_record(rec: Dict[str,Any], source: str) -> Optional[float]:
    # 1) 取图像
    rel = rec["images"][0] if "images" in rec and rec["images"] else None
    if not rel: return None
    img_path = os.path.join(_img_root_for_source(source), rel)
    image = _safe_open_image(img_path)
    if image is None: return None

    # 2) 取原始答案文本（assistant 段）
    base_answer = _extract_answer_text(rec)
    if not str(base_answer).strip():
        return None

    # 3) 多视图 logits → ModRényi* 融合
    try:
        img_views = image_views_logp_list_qwen(image, rec, base_answer)
        txt_views = text_views_logp_list_qwen(image, rec, base_answer)
        return fused_modrenyi_score(img_views, txt_views, alphas=RENYI_ALPHAS, length_norm=LENGTH_NORM)
    except Exception:
        return None

# ---------------- 主跑 & 评估 ----------------
print("\n⏳ 计算 ModRényi*（fused）分数（Qwen2-VL 数据/模型）…")
mem_scores, non_scores = [], []
for r in tqdm(members_1k, desc="members"):
    s = fused_score_for_record(r, "member")
    if s is not None and math.isfinite(s): mem_scores.append(s)
for r in tqdm(nonmembers_1k, desc="nonmembers"):
    s = fused_score_for_record(r, "nonmember")
    if s is not None and math.isfinite(s): non_scores.append(s)

print(f"\n📊 有效样本：members={len(mem_scores)} / {len(members_1k)} | non-members={len(non_scores)} / {len(nonmembers_1k)}")
if len(mem_scores)==0 or len(non_scores)==0:
    raise RuntimeError("有效样本为 0，无法计算 MIA 指标。")

y_true = np.array([1]*len(mem_scores) + [0]*len(non_scores))
scores = np.array(mem_scores + non_scores)

# ---- @5% FPR 的完整指标（保持原 ModRényi* 的计算方式与输出格式）----
def metrics_at_target_fpr(y_true: np.ndarray, scores: np.ndarray, target_fpr: float = 0.05):
    fpr, tpr, thr = roc_curve(y_true, scores)
    idx = np.searchsorted(fpr, target_fpr, side="right")
    if idx == 0:
        thr_star = thr[0]; tpr_star = tpr[0]
    elif idx >= len(thr):
        thr_star = thr[-1]; tpr_star = tpr[-1]
    else:
        x0, x1 = fpr[idx-1], fpr[idx]
        y0, y1 = tpr[idx-1], tpr[idx]
        t0, t1 = thr[idx-1], thr[idx]
        w = (target_fpr - x0) / (x1 - x0 + 1e-12)
        tpr_star = y0 + w*(y1 - y0)
        thr_star = t0 + w*(t1 - t0)

    y_pred = (scores >= thr_star).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    acc = accuracy_score(y_true, y_pred)  # ← 新增 Accuracy
    return {
        "thr@5%FPR": float(thr_star),
        "TPR@5%FPR": float(tpr_star),
        "Precision": float(prec),
        "Recall": float(rec),
        "F1": float(f1),
        "Accuracy": float(acc),             # ← 新增 Accuracy
        "ROC-AUC": float(roc_auc_score(y_true, scores)),
    }

m = metrics_at_target_fpr(y_true, scores, target_fpr=0.05)

print("\n====== ModRényi* (fused) MIA（Qwen2-VL + 评测 1000×1000） ======")
print(f"AUC           = {m['ROC-AUC']:.4f}")
print(f"TPR @5% FPR   = {m['TPR@5%FPR']:.4f}")
print(f"Precision     = {m['Precision']:.4f}")
print(f"Recall        = {m['Recall']:.4f}")
print(f"F1            = {m['F1']:.4f}")
print(f"Accuracy      = {m['Accuracy']:.4f}")  # ← 新增打印
print(f"thr @5% FPR   = {m['thr@5%FPR']:.6f}")


In [ ]:
# @title 7. Similarity
# Similarity-MIA (Image + Text) with Fusion — robust (fixed)
# - 方法与细节保持不变（图像扰动→OpenCLIP相似度；文本生成→句嵌入相似度；z-score融合；FPR=5% 阈值）
# - 仅替换为：LoRA 合并后的 Qwen2-VL 作为文本生成模型；数据改为 eval-1000 + 本地图像根目录
# - 新增：AUC + @5%FPR 的 TPR/Precision/Recall/F1/Accuracy（并打印混淆矩阵）
# - 修复：净化 GenerationConfig + 设置 pad_token，消除所有 generate() 的告警输出
# ===============================================
import os, io, json, math, random, shutil, warnings, traceback, subprocess, sys
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import torch
from PIL import Image, ImageFilter, UnidentifiedImageError
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, precision_recall_fscore_support

warnings.filterwarnings("ignore")

# ------- 额外：降低 Transformers 日志级别，避免非关键提示刷屏 -------
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

# ---------------- 路径配置（与 Loss Attack 一致） ----------------
LLAMA_REPO = "/content/LLaMA-Factory"
BASE_ID    = "Qwen/Qwen2-VL-2B-Instruct"
CACHE_DIR  = "/content/hf_cache"

# 你训练好的 LoRA 最佳 checkpoint（保持不变）
LORA_BEST  = "/content/Qwen2-VL-med-lora-A100/checkpoint-315"

# 合并后模型保存到 Drive（保持不变）
MERGED_DIR = "/content/drive/MyDrive/Qwen2-VL-med-merged-A100"

# ================= 仅此处数据切换：与 LoRA 训练一致 =================
MIA_ROOT       = "/content/drive/MyDrive/MedTrinity_MIA"
MEM_IMGROOT    = f"{MIA_ROOT}/member_5k"                 # 图像根目录与训练相同
NON_IMGROOT    = f"{MIA_ROOT}/nonmember_5k"             # 非成员图像根目录

# 用 LoRA 训练用的评测 1000 条 JSON
MEM_EVAL_JSON  = f"{MIA_ROOT}/member_eval_1000.json"     # 成员 = 训练集
NON_EVAL_JSON  = f"{MIA_ROOT}/nonmember_eval_1000.json"  # 非成员 = 未训练集

# 输出目录/文件
OUT_DIR  = os.path.join(MIA_ROOT, "sim_mia_outputs")
OUT_CSV  = os.path.join(OUT_DIR, "mia_similarity_fused_1000x1000.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- 基本参数（方法保持不变） ----------------
NUM_MEMBERS, NUM_NONMEMBERS = 1000, 1000
SEED = 42

# 图像扰动
GAUSS_RADIUS = 1.0
JPEG_QUALITY = 60

# 文本生成（降载）
MAX_NEW_TOKENS = 24
USER_PROMPT    = "Describe the medical image."
TEXT_FAIL_TOL  = 20      # 连续失败达到阈值后禁用文本通道

# 融合权重
W_IMG, W_TEXT = 0.60, 0.40

# ---------------- 设备 & 模型（Qwen2-VL merged for text generation） ----------------
assert torch.cuda.is_available(), "需要 GPU"
device = "cuda:0"
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32  = True
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print(f"✅ Using {device}, dtype={dtype}")

# 0) 基本检查 + 合并 LoRA → 独立模型（与 Loss Attack 一致）
assert os.path.isdir(LLAMA_REPO), f"❌ LLaMA-Factory 不存在：{LLAMA_REPO}"
assert os.path.isdir(LORA_BEST),  f"❌ LoRA 目录不存在：{LORA_BEST}"
assert os.path.isfile(MEM_EVAL_JSON), f"❌ 缺少 {MEM_EVAL_JSON}"
assert os.path.isfile(NON_EVAL_JSON), f"❌ 缺少 {NON_EVAL_JSON}"
os.makedirs(CACHE_DIR, exist_ok=True)

print("🔁 Export (merge) base + LoRA → merged dir …")
cwd_backup = os.getcwd()
try:
    os.chdir(LLAMA_REPO)
    export_cmd = [
        sys.executable, "-m", "llamafactory.cli", "export",
        "--model_name_or_path", BASE_ID,
        "--adapter_name_or_path", LORA_BEST,
        "--template", "qwen2_vl",
        "--finetuning_type", "lora",
        "--export_dir", MERGED_DIR,
        "--export_size", "2",
        "--export_device", "cpu",
        "--cache_dir", CACHE_DIR,
        "--trust_remote_code"
    ]
    ret = subprocess.call(export_cmd)
    if ret != 0:
        raise RuntimeError(f"❌ 导出失败（返回码 {ret}），请确认 LoRA_BEST 路径与权限。")
finally:
    os.chdir(cwd_backup)
print(f"✅ 合并完成 → {MERGED_DIR}")

# 1) 加载 Qwen2-VL merged 作为文本生成模型
from transformers import AutoProcessor, AutoModelForVision2Seq, GenerationConfig
processor = AutoProcessor.from_pretrained(MERGED_DIR, trust_remote_code=True)
vlm = AutoModelForVision2Seq.from_pretrained(
    MERGED_DIR,
    torch_dtype=dtype,          # 关键：使用 torch_dtype
    device_map=None,
    trust_remote_code=True,
    low_cpu_mem_usage=True
).to(device)
if hasattr(vlm, "config"):
    vlm.config.use_cache = False  # 降低 KV cache 压力

# —— 关键修复 A：为分词器/模型设置 pad_token，避免 generate() 打印 “Setting pad_token_id=...”
tok = processor.tokenizer
# 若没有 pad_token，则用 eos 充当 pad
if getattr(tok, "pad_token_id", None) is None or tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
# 同步到模型配置与生成配置
eos_id = int(tok.eos_token_id)
pad_id = int(tok.pad_token_id)
vlm.config.eos_token_id = eos_id
vlm.config.pad_token_id = pad_id

clean_gc = GenerationConfig.from_model_config(vlm.config)
clean_gc.max_new_tokens = MAX_NEW_TOKENS
clean_gc.do_sample     = False
clean_gc.num_beams     = 1
clean_gc.use_cache     = False
clean_gc.eos_token_id  = eos_id
clean_gc.pad_token_id  = pad_id
vlm.generation_config  = clean_gc

vlm.eval()
print("✅ Qwen2-VL (merged) 就绪")

# 2) 预加载 OpenCLIP（图像通道）
try:
    import open_clip
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch>=2.24.0", "timm"])
    import open_clip
OPENCLIP_MODEL, _, OPENCLIP_TRANS = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai", device=device)
OPENCLIP_MODEL.eval()
print("✅ OpenCLIP (ViT-L/14) 就绪")

# 3) 句向量模型（文本通道）
try:
    from sentence_transformers import SentenceTransformer
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer
txt_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
txt_embedder.eval()

# ---------------- 数据：严格使用 eval-1000（与 Loss Attack 一致） ----------------
def load_eval_list(json_path, want_n=1000, seed=20251012):
    """支持两种结构：直接 list 或 {'data': [...]}；取前 1000。"""
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    data = raw["data"] if isinstance(raw, dict) and "data" in raw else raw
    if not isinstance(data, list):
        raise ValueError(f"{json_path} 不是列表或不含 'data' 列表。")
    if len(data) < want_n:
        raise RuntimeError(f"{json_path} 样本不足 {want_n}（当前 {len(data)}）")
    return data[:want_n]

random.seed(SEED)
members    = load_eval_list(MEM_EVAL_JSON,  NUM_MEMBERS)
nonmembers = load_eval_list(NON_EVAL_JSON, NUM_NONMEMBERS)
print(f"\n📦 样本就绪：members={len(members)} | non-members={len(nonmembers)}")

# ---------------- I/O（本地图像路径，与 Loss Attack 对齐） ----------------
def _join_img_path(root: str, rec: Dict[str, Any]) -> Optional[str]:
    rels = rec.get("images", [])
    if not rels or not isinstance(rels, list):
        return None
    return os.path.join(root, rels[0])

def load_member_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = _join_img_path(MEM_IMGROOT, sample)
    if not p or not os.path.exists(p): return None
    return Image.open(p).convert("RGB")

def load_nonmember_image(sample: Dict[str, Any]) -> Optional[Image.Image]:
    p = _join_img_path(NON_IMGROOT, sample)
    if not p or not os.path.exists(p): return None
    return Image.open(p).convert("RGB")

# ---------------- 扰动与生成（方法保持不变；仅更换到 Qwen2-VL 的生成接口） ----------------
def corrupt_image(img: Image.Image) -> Image.Image:
    x = img.filter(ImageFilter.GaussianBlur(radius=GAUSS_RADIUS))
    buf = io.BytesIO(); x.save(buf, format="JPEG", quality=JPEG_QUALITY, optimize=True)
    buf.seek(0); x = Image.open(buf).convert("RGB")
    return x

@torch.inference_mode()
def generate_text(image: Image.Image) -> str:
    # 对 Qwen2-VL：使用图像片段 + 英文提示作为 user
    msgs = [{"role": "user", "content": [{"type":"image"}, {"type":"text","text": USER_PROMPT}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = processor(text=[prompt], images=[image], return_tensors="pt").to(device)
    out_ids = vlm.generate(**enc)  # generation_config / pad/eos 已全局设好
    text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    torch.cuda.empty_cache()
    return text.strip()

# ---------------- 两通道得分（与原实现一致） ----------------
first_err = {"img": None, "txt": None}

@torch.inference_mode()
def image_similarity_score(image: Image.Image) -> Optional[float]:
    try:
        img2 = corrupt_image(image)
        t1 = OPENCLIP_TRANS(image).unsqueeze(0).to(device)
        t2 = OPENCLIP_TRANS(img2).unsqueeze(0).to(device)
        z1 = OPENCLIP_MODEL.encode_image(t1).float(); z1 = z1 / (z1.norm(dim=-1, keepdim=True)+1e-12)
        z2 = OPENCLIP_MODEL.encode_image(t2).float(); z2 = z2 / (z2.norm(dim=-1, keepdim=True)+1e-12)
        sim = torch.cosine_similarity(z1[0], z2[0], dim=-1).clamp(-1, 1).item()
        return float(sim)
    except Exception as e:
        if first_err["img"] is None:
            first_err["img"] = "".join(traceback.format_exception_only(type(e), e)).strip()
        return None

@torch.inference_mode()
def text_similarity_score(image: Image.Image) -> Optional[float]:
    try:
        img2 = corrupt_image(image)
        t1 = generate_text(image)
        t2 = generate_text(img2)
        e1 = txt_embedder.encode([t1], convert_to_tensor=True, normalize_embeddings=True, device=device)[0]
        e2 = txt_embedder.encode([t2], convert_to_tensor=True, normalize_embeddings=True, device=device)[0]
        sim = torch.cosine_similarity(e1, e2, dim=-1).clamp(-1, 1).item()
        return float(sim)
    except Exception as e:
        if first_err["txt"] is None:
            first_err["txt"] = "".join(traceback.format_exception_only(type(e), e)).strip()
        return None

# ---------------- 批量收集（容错与逐步禁用文本通道保持不变） ----------------
FAIL = {"image":0, "img_only_fail":0, "txt_only_fail":0}
def collect_scores(samples, source: str, enable_text: bool = True):
    sims_img, sims_txt = [], []
    loader = load_member_image if source == "member" else load_nonmember_image
    text_fail_streak = 0
    for i in tqdm(range(len(samples)), desc=f"{source}"):
        s = samples[i]
        # 读图
        try:
            img = loader(s)
            if img is None:
                FAIL["image"] += 1
                sims_img.append(None); sims_txt.append(None); continue
        except Exception:
            FAIL["image"] += 1
            sims_img.append(None); sims_txt.append(None); continue

        # 图像通道
        si = image_similarity_score(img)
        if si is None: FAIL["img_only_fail"] += 1
        sims_img.append(si)

        # 文本通道（可禁用）
        st = None
        if enable_text:
            st = text_similarity_score(img)
            if st is None:
                text_fail_streak += 1
            else:
                text_fail_streak = 0
            # 连续失败过多 → 自动禁用文本通道，后续只跑图像
            if text_fail_streak >= TEXT_FAIL_TOL:
                enable_text = False
                print(f"⚠️ 文本通道连续失败 {TEXT_FAIL_TOL} 次，后续将禁用文本通道，只保留图像通道。")
        else:
            st = None

        if st is None: FAIL["txt_only_fail"] += 1
        sims_txt.append(st)

        if (i+1) % 50 == 0:
            torch.cuda.empty_cache()

    return sims_img, sims_txt, enable_text

print("\n⏳ 计算相似度分数（members） …")
mem_img_raw, mem_txt_raw, txt_enabled = collect_scores(members, "member", enable_text=True)
print("⏳ 计算相似度分数（nonmembers） …")
non_img_raw, non_txt_raw, _          = collect_scores(nonmembers, "nonmember", enable_text=txt_enabled)

# 过滤 None
def _nan_filter(xs): return np.array([x for x in xs if (x is not None and np.isfinite(x))], dtype=np.float32)
mem_img = _nan_filter(mem_img_raw);  non_img = _nan_filter(non_img_raw)
mem_txt = _nan_filter(mem_txt_raw);  non_txt = _nan_filter(non_txt_raw)

print(f"\n📊 有效样本（按通道 after filter）：")
print(f"image: mem={len(mem_img)} non={len(non_img)}")
print(f"text : mem={len(mem_txt)} non={len(non_txt)}")
if first_err["img"]: print(f"🧪 首个图像通道异常：{first_err['img']}")
if first_err["txt"]: print(f"🧪 首个文本通道异常：{first_err['txt']}")
print(f"❗ 失败统计：{FAIL}")

# ---------------- 融合与指标（方法保持不变；容忍单通道） ----------------
def zscore(arr):
    mu, sd = arr.mean(), arr.std() + 1e-8
    return (arr - mu) / (sd + 1e-8), mu, sd

have_img = (len(non_img) > 10) and (len(mem_img) > 10)
have_txt = (len(non_txt) > 10) and (len(mem_txt) > 10)
if not have_img and not have_txt:
    raise RuntimeError("两条通道都没有有效分数，请先根据上面的首个异常信息排查（通常是文本生成 OOM / 接口不兼容）。")

# 仅保留可用通道
parts = []
if have_img: parts.append(("img", mem_img, non_img, W_IMG))
if have_txt: parts.append(("txt", mem_txt, non_txt, W_TEXT))
if len(parts) == 1:
    # 单通道时将权重设为 1
    parts[0] = (parts[0][0], parts[0][1], parts[0][2], 1.0)

# z-score on non-members per-part，然后融合
fused_mem, fused_non = None, None
details = {}
for name, mem_arr, non_arr, w in parts:
    non_z, mu, sd = zscore(non_arr)
    mem_z = (mem_arr - mu) / (sd + 1e-8)
    details[name] = (mem_arr, non_arr, mem_z, non_z, w)
    if fused_mem is None:
        fused_mem = w * mem_z
        fused_non = w * non_z
    else:
        fused_mem = fused_mem + w * mem_z
        fused_non = fused_non + w * non_z

# AUC
y_true = np.array([1]*len(fused_mem) + [0]*len(fused_non))
scores = np.concatenate([fused_mem, fused_non], axis=0)
auc = roc_auc_score(y_true, scores)

# FPR=5% 阈值（基于 non-members 的分位数）
thr_5 = float(np.quantile(fused_non, 0.95))
y_pred = (scores >= thr_5).astype(int)

cm  = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn + 1e-12)
tpr = tp / (tp + fn + 1e-12)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
acc = accuracy_score(y_true, y_pred)

print("\n====== Similarity-MIA（Image+Text 融合，稳健版 | Qwen2-VL merged） ======")
print(f"AUC                = {auc:.4f}")
print(f"Threshold @FPR=5%  = {thr_5:.4f}  (实际 FPR={fpr*100:.2f}%)")
print(f"TPR (Recall)       = {tpr*100:.2f}%")
print(f"Precision          = {prec*100:.2f}%")
print(f"F1                 = {f1:.4f}")
print(f"Accuracy           = {acc*100:.2f}%")
print("Confusion [[TN FP][FN TP]]：")
print(cm)

# ---------------- 保存明细（与原方法一致） ----------------
import csv
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    hdr = ["split"]
    if have_img: hdr += ["s_img", "z_img(non-z)"]
    if have_txt: hdr += ["s_txt", "z_txt(non-z)"]
    hdr += ["s_fused"]
    w.writerow(hdr)

    def write_side(tag, mem: bool):
        if mem:
            zs = fused_mem; base = {k:v for k,v in details.items()}
        else:
            zs = fused_non
        n = len(zs)
        for i in range(n):
            row = [ "member" if mem else "nonmember" ]
            if have_img:
                mi, ni, miz, niz, wi = details["img"]
                arr = mi if mem else ni
                z   = miz if mem else niz
                row += [ float(arr[i%len(arr)]), float(z[i%len(z)]) ]
            if have_txt:
                mt, nt, mtz, ntz, wt = details["txt"]
                arr = mt if mem else nt
                z   = mtz if mem else ntz
                row += [ float(arr[i%len(arr)]), float(z[i%len(z)]) ]
            row += [ float(zs[i]) ]
            w.writerow(row)

    write_side("member", True)
    write_side("nonmember", False)

print(f"\n💾 明细已保存：{OUT_CSV}")


In [ ]:
# ==================================================================================
# @title 8. GradAudit P/S/N
# GradAudit / GradSafe P-S-N experiment (Qwen2-VL-2B + PEFT LoRA)
# ==================================================================================
# - Model: Qwen2-VL-2B-Instruct base + your PEFT LoRA (preferred)
# - Grads: LoRA params only, restricted to Vision last3 + Text last3 + (Connector/MM projector)
# - Loss: supervised (labels masked to only assistant segment)
# - P/S/N:
#     P = member paired (original member eval)
#     S = member images but shuffled assistant texts within member pool (j!=i)
#     N = nonmember eval
# - Sensitive mask rule (tau-or-topQ):
#     use gap>tau; if empty -> use topQ (q=0.80 => top20%)
# ==================================================================================

import os, json, gc, random, re, tarfile
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, confusion_matrix

from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

# ---------------------------
# 0) Paths & hyperparams
# ---------------------------
DRIVE_ROOT = "/content/drive/MyDrive/MedTrinity_MIA"
MEMBER_JSON    = os.path.join(DRIVE_ROOT, "member_eval_1000.json")
NONMEMBER_JSON = os.path.join(DRIVE_ROOT, "nonmember_eval_1000.json")
MEMBER_ROOT    = os.path.join(DRIVE_ROOT, "member_5k")
NONMEMBER_ROOT = os.path.join(DRIVE_ROOT, "nonmember_5k")

BASE_ID    = "Qwen/Qwen2-VL-2B-Instruct"
CACHE_DIR  = "/content/hf_cache"

# ✅ your packed adapter on Drive
LORA_TAR = os.path.join(DRIVE_ROOT, "lora_adapters", "qwen2vl_medtrinity_eval1000_lora.tar.gz")
# local unpack dir (avoid Drive IO during runtime)
LORA_LOCAL_DIR = "/content/qwen2vl_lora_local"

SEED   = 20251012
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# Split sizes
CALIB_P = 200
CALIB_N = 200
PROBE_P = 800
PROBE_S = 800
PROBE_N = 800

# Sensitive mask rule
SENS_TAU = 0.10
Q_TOP    = 0.80          # q=0.8 => top 20%

# Last layers selection
VISION_LAST_N = 3
TEXT_LAST_M   = 3

PROMPT_EN = (
    "As a medical vision-language model, describe this medical image and "
    "provide a professional diagnostic impression."
)

OUT_CSV = os.path.join(DRIVE_ROOT, "gradaudit_qwen2vl_lora_PSN_eval1000.csv")

# ---------------------------
# 1) Utilities
# ---------------------------
def set_all_seeds(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    return d["data"] if isinstance(d, dict) and "data" in d else d

def ensure_image(path: str) -> Optional[Image.Image]:
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, FileNotFoundError, OSError, KeyError):
        return None

def build_messages_with_image(prompt_text: str) -> List[Dict[str, Any]]:
    return [{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt_text}]}]

def extract_assistant_text(rec: Dict[str, Any]) -> str:
    msgs = rec.get("messages", [])
    if not msgs:
        return str(rec.get("caption", "")).strip()
    last = msgs[-1]
    cont = last.get("content", "")
    if isinstance(cont, str):
        return cont.strip()
    if isinstance(cont, list):
        out = []
        for seg in cont:
            if isinstance(seg, dict) and seg.get("type") == "text":
                out.append(seg.get("text", ""))
        return " ".join(out).strip()
    if isinstance(cont, dict):
        return str(cont.get("text", "")).strip()
    return str(cont).strip()

def set_assistant_text(rec: Dict[str,Any], new_text: str) -> Dict[str,Any]:
    x = dict(rec)
    msgs = x.get("messages", None)
    if not isinstance(msgs, list) or len(msgs) == 0:
        x["caption"] = str(new_text)
        return x
    msgs2 = [dict(m) for m in msgs]
    last = dict(msgs2[-1])
    last["role"] = last.get("role", "assistant")
    last["content"] = [{"type":"text", "text": str(new_text)}]
    msgs2[-1] = last
    x["messages"] = msgs2
    return x

def get_rel_image_path(rec: Dict[str,Any]) -> Optional[str]:
    imgs = rec.get("images", None)
    if isinstance(imgs, list) and len(imgs) > 0:
        return imgs[0]
    if "image" in rec and isinstance(rec["image"], str):
        return rec["image"]
    return None

def _autocast_ctx():
    if DEVICE == "cuda":
        if DTYPE == torch.bfloat16:
            return torch.autocast("cuda", dtype=torch.bfloat16)
        if DTYPE == torch.float16:
            return torch.autocast("cuda", dtype=torch.float16)
    return torch.autocast("cpu", enabled=False)

def safe_cuda_cleanup():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# ---------------------------
# 2) Prepare adapter locally (unpack tar.gz if needed)
# ---------------------------
def ensure_local_adapter(tar_path: str, local_dir: str) -> str:
    os.makedirs(local_dir, exist_ok=True)
    cfg = os.path.join(local_dir, "adapter_config.json")
    if os.path.isfile(cfg):
        return local_dir
    if not os.path.isfile(tar_path):
        raise FileNotFoundError(f"LoRA tar not found: {tar_path}")
    print(f"📦 Unpacking adapter: {tar_path} -> {local_dir}")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=local_dir)
    if not os.path.isfile(cfg):
        raise RuntimeError("Unpacked adapter but adapter_config.json not found. Check tar contents.")
    return local_dir

# ---------------------------
# 3) Load data & build P/S/N
# ---------------------------
set_all_seeds(SEED)

members_all = load_json_list(MEMBER_JSON)
nonmem_all  = load_json_list(NONMEMBER_JSON)
assert len(members_all) >= 1000 and len(nonmem_all) >= 1000

members_all = members_all[:1000]
nonmem_all  = nonmem_all[:1000]

# P: paired members
P_all = members_all

# S: same images, shuffled assistant texts among member pool (force j!=i)
caps = [extract_assistant_text(x) for x in members_all]
perm = list(range(len(caps)))
random.Random(SEED + 999).shuffle(perm)
for i in range(len(perm)):
    if perm[i] == i:
        j = (i + 1) % len(perm)
        perm[i], perm[j] = perm[j], perm[i]
S_all = [set_assistant_text(members_all[i], caps[perm[i]]) for i in range(len(members_all))]

# N: nonmembers
N_all = nonmem_all

print(f"✅ Built sets: P={len(P_all)} | S={len(S_all)} | N={len(N_all)}")
diff_cnt = sum(extract_assistant_text(P_all[i]) != extract_assistant_text(S_all[i]) for i in range(50))
print(f"Sanity (first 50): P vs S assistant-text different = {diff_cnt}/50")

# shuffle deterministically then split
rnd = random.Random(SEED)
rnd.shuffle(P_all); rnd.shuffle(S_all); rnd.shuffle(N_all)

P_calib = P_all[:CALIB_P]
N_calib = N_all[:CALIB_N]

P_probe = P_all[CALIB_P:CALIB_P+PROBE_P]
S_probe = S_all[CALIB_P:CALIB_P+PROBE_S]
N_probe = N_all[CALIB_N:CALIB_N+PROBE_N]

print(f"Calib: P={len(P_calib)}, N={len(N_calib)}")
print(f"Probe: P={len(P_probe)}, S={len(S_probe)}, N={len(N_probe)}")

# ---------------------------
# 4) Load model (base + LoRA)
# ---------------------------
def load_base_plus_lora(base_id: str, adapter_dir: str):
    processor = AutoProcessor.from_pretrained(base_id, trust_remote_code=True, cache_dir=CACHE_DIR)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        base_id,
        device_map="auto",
        torch_dtype=DTYPE,
        trust_remote_code=True,
        cache_dir=CACHE_DIR
    )
    print(f"🔗 Using LoRA adapter: {adapter_dir}")
    model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=False)
    model.eval()
    return processor, model

adapter_dir = ensure_local_adapter(LORA_TAR, LORA_LOCAL_DIR)
print("🔁 Loading base+LoRA (preferred)…")
processor, model = load_base_plus_lora(BASE_ID, adapter_dir)
print(f"✅ Device={DEVICE}  dtype={DTYPE}")

# ---------------------------
# 5) Enable grads: LoRA only + restrict to last layers + connector/projector
# ---------------------------
def _infer_layer_ids(names: List[str], pattern: str) -> Tuple[List[int], Optional[re.Pattern]]:
    rgx = re.compile(pattern)
    ids = []
    for n in names:
        m = rgx.search(n)
        if m:
            try:
                ids.append(int(m.group(1)))
            except Exception:
                pass
    ids = sorted(set(ids))
    if ids:
        return ids, rgx
    return [], None

def mark_trainable_lora_lastlayers(m) -> Tuple[List[str], Dict[str, Any]]:
    for _, p in m.named_parameters():
        p.requires_grad_(False)

    all_names = [n for n, _ in m.named_parameters()]

    vision_patterns = [
        r"visual\.blocks\.(\d+)\.",
        r"vision_tower\.blocks\.(\d+)\.",
        r"vision_model\.encoder\.layers\.(\d+)\.",
        r"visual\.model\.layers\.(\d+)\.",
    ]
    vision_ids, vision_rgx, vision_pat = [], None, None
    for pat in vision_patterns:
        ids, rgx = _infer_layer_ids(all_names, pat)
        if ids:
            vision_ids, vision_rgx, vision_pat = ids, rgx, pat
            break

    text_patterns = [
        r"model\.layers\.(\d+)\.",
        r"language_model\.model\.layers\.(\d+)\.",
        r"language_model\.layers\.(\d+)\.",
    ]
    text_ids, text_rgx, text_pat = [], None, None
    for pat in text_patterns:
        ids, rgx = _infer_layer_ids(all_names, pat)
        if ids:
            text_ids, text_rgx, text_pat = ids, rgx, pat
            break

    vmax = max(vision_ids) if vision_ids else None
    tmax = max(text_ids) if text_ids else None
    v_start = (vmax - (VISION_LAST_N - 1)) if vmax is not None else None
    t_start = (tmax - (TEXT_LAST_M   - 1)) if tmax is not None else None

    selected = []
    for n, p in m.named_parameters():
        if p.ndim < 2:
            continue
        nl = n.lower()
        if "lora" not in nl:
            continue

        keep = False

        if any(k in nl for k in ["mm_projector", "multi_modal_projector", "vision_proj", "projector", "connector"]):
            keep = True

        if (not keep) and (vision_rgx is not None) and (v_start is not None):
            m1 = vision_rgx.search(n)
            if m1 and int(m1.group(1)) >= v_start:
                keep = True

        if (not keep) and (text_rgx is not None) and (t_start is not None):
            m2 = text_rgx.search(n)
            if m2 and int(m2.group(1)) >= t_start:
                keep = True

        if keep:
            p.requires_grad_(True)
            selected.append(n)

    meta = dict(
        vision_pat=vision_pat, vmax=vmax, v_start=v_start,
        text_pat=text_pat, tmax=tmax, t_start=t_start
    )
    return selected, meta

selected_params, meta = mark_trainable_lora_lastlayers(model)
print(f"🔧 Trainable LoRA params selected = {len(selected_params)}")
print(f"  vision_pat={meta['vision_pat']}  v_range={meta['v_start']}..{meta['vmax']}")
print(f"  text_pat={meta['text_pat']}      t_range={meta['t_start']}..{meta['tmax']}")
if len(selected_params) == 0:
    raise RuntimeError("No LoRA params selected. Check param name patterns for this model.")

# ---------------------------
# 6) Loss: only assistant segment contributes
# ---------------------------
def _prefix_len_tokens(image: Image.Image, prompt_text: str) -> int:
    msgs = build_messages_with_image(prompt_text)
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    batch = processor(images=[image], text=[text], return_tensors="pt")
    return int(batch["input_ids"].shape[1])

def backward_and_collect(img_path: str, target_text: str) -> Dict[str, torch.Tensor]:
    image = ensure_image(img_path)
    if image is None:
        return {}

    try:
        pl = _prefix_len_tokens(image, PROMPT_EN)

        msgs_full = build_messages_with_image(PROMPT_EN) + [
            {"role":"assistant","content":[{"type":"text","text":target_text}]}
        ]
        text_full = processor.apply_chat_template(msgs_full, tokenize=False, add_generation_prompt=False)
        batch = processor(images=[image], text=[text_full], return_tensors="pt").to(model.device)

        labels = batch["input_ids"].clone()
        labels[:, :pl] = -100

        model.train()
        model.zero_grad(set_to_none=True)

        with _autocast_ctx():
            out = model(**batch, labels=labels)
            loss = out.loss

        if (loss is None) or (not torch.isfinite(loss)):
            model.zero_grad(set_to_none=True)
            safe_cuda_cleanup()
            model.eval()
            return {}

        loss.backward()

        gd = {}
        for n, p in model.named_parameters():
            if not p.requires_grad:
                continue
            g = p.grad
            if g is None or g.ndim < 2:
                continue
            gd[n] = g.detach().to(torch.float32).cpu()

        model.zero_grad(set_to_none=True)
        safe_cuda_cleanup()
        model.eval()
        return gd
    except Exception:
        model.zero_grad(set_to_none=True)
        safe_cuda_cleanup()
        model.eval()
        return {}

def row_col_cos(a: torch.Tensor, b: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if a.ndim < 2:
        s = F.cosine_similarity(a.flatten().unsqueeze(0), b.flatten().unsqueeze(0), dim=1)
        s = torch.nan_to_num(s, 0.0)
        return s, s
    r = F.cosine_similarity(a, b, dim=1)
    c = F.cosine_similarity(a.T, b.T, dim=1)
    return torch.nan_to_num(r, 0.0), torch.nan_to_num(c, 0.0)

# ---------------------------
# 7) Build reference grads from P_calib
# ---------------------------
def build_reference_grad(samples: List[Dict[str, Any]], root_dir: str) -> Dict[str, torch.Tensor]:
    ref: Dict[str, torch.Tensor] = {}
    ok = 0
    for ex in tqdm(samples, desc="RefGrad(P_calib)", dynamic_ncols=True):
        rel = get_rel_image_path(ex)
        if not rel:
            continue
        img_path = os.path.join(root_dir, rel)
        target = extract_assistant_text(ex)
        if not target:
            continue

        gd = backward_and_collect(img_path, target)
        if not gd:
            continue

        ok += 1
        if not ref:
            ref = {k: v.clone() for k, v in gd.items()}
        else:
            for k in list(ref.keys()):
                if (k in gd) and (ref[k].shape == gd[k].shape):
                    ref[k] += gd[k]
                else:
                    ref.pop(k, None)

    if ok == 0:
        raise RuntimeError("Reference gradient build failed: 0 valid grads.")
    for k in ref:
        ref[k] /= ok
    print(f"✅ RefGrad built with ok={ok}/{len(samples)}, params={len(ref)}")
    return ref

# ---------------------------
# 8) Avg sims on calib P and calib N
# ---------------------------
def avg_rowcol_sims(samples: List[Dict[str, Any]],
                    ref: Dict[str, torch.Tensor],
                    root_dir: str,
                    tag: str) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor], int]:
    acc_r: Dict[str, torch.Tensor] = {}
    acc_c: Dict[str, torch.Tensor] = {}
    cnt = 0
    for ex in tqdm(samples, desc=f"Sims({tag})", dynamic_ncols=True):
        rel = get_rel_image_path(ex)
        if not rel:
            continue
        img_path = os.path.join(root_dir, rel)
        target = extract_assistant_text(ex)
        if not target:
            continue

        gd = backward_and_collect(img_path, target)
        if not gd:
            continue

        cnt += 1
        for name, g in gd.items():
            if (name not in ref) or (ref[name].shape != g.shape):
                continue
            rs, cs = row_col_cos(g, ref[name])
            if name not in acc_r:
                acc_r[name], acc_c[name] = rs.clone(), cs.clone()
            else:
                if acc_r[name].shape == rs.shape:
                    acc_r[name] += rs
                if acc_c[name].shape == cs.shape:
                    acc_c[name] += cs

    if cnt == 0:
        return {}, {}, 0
    for k in acc_r:
        acc_r[k] /= cnt
        acc_c[k] /= cnt
    print(f"✅ Sims({tag}) computed with cnt={cnt}/{len(samples)}")
    return acc_r, acc_c, cnt

# ---------------------------
# 9) Sensitive mask rule: tau-or-topQ (NO MIN_KEEP_RATIO, NO top-k fallback)
# ---------------------------
def _mask_tau_or_topq(gap_1d: torch.Tensor, tau: float, q: float) -> torch.Tensor:
    gap = torch.nan_to_num(gap_1d.detach().to(torch.float32).cpu(), nan=0.0, posinf=0.0, neginf=0.0)
    if gap.numel() == 0:
        return torch.zeros_like(gap, dtype=torch.bool)

    tau_mask = (gap > float(tau))
    if tau_mask.any():
        return tau_mask

    # fallback only when tau_mask is empty
    qthr = torch.quantile(gap, float(q))
    return (gap >= qthr)

def build_sensitive_masks(P_rc, N_rc, tau: float = SENS_TAU, q: float = Q_TOP):
    # ✅ avg_rowcol_sims returns (row_dict, col_dict, cnt)
    pr, pc, p_cnt = P_rc
    nr, nc, n_cnt = N_rc

    row_masks, col_masks = {}, {}
    total = 0
    layers_used = 0

    for name in pr:
        if name not in nr or name not in pc or name not in nc:
            continue

        rgap = pr[name] - nr[name]
        cgap = pc[name] - nc[name]

        rm = _mask_tau_or_topq(rgap, tau=tau, q=q)
        cm = _mask_tau_or_topq(cgap, tau=tau, q=q)

        if rm.any() or cm.any():
            row_masks[name] = rm
            col_masks[name] = cm
            total += int(rm.sum().item() + cm.sum().item())
            layers_used += 1

    print(f"🧾 Mask build: P_cnt={p_cnt}, N_cnt={n_cnt}, layers_used={layers_used}")
    return row_masks, col_masks, total

# ---------------------------
# 10) Score function (GradAudit-style masked grad-cos)
# ---------------------------
def gradsafe_score(ex: Dict[str, Any],
                   ref: Dict[str, torch.Tensor],
                   row_masks: Dict[str, torch.Tensor],
                   col_masks: Dict[str, torch.Tensor],
                   root_dir: str) -> float:
    rel = get_rel_image_path(ex)
    if not rel:
        return 0.0
    img_path = os.path.join(root_dir, rel)
    target = extract_assistant_text(ex)
    if not target:
        return 0.0

    gd = backward_and_collect(img_path, target)
    if not gd:
        return 0.0

    sims: List[float] = []
    for name, g in gd.items():
        if (name in ref) and (name in row_masks) and (name in col_masks) and (ref[name].shape == g.shape):
            rs, cs = row_col_cos(g, ref[name])
            rm, cm = row_masks[name], col_masks[name]
            if rm.any():
                sims.extend(rs[rm].cpu().tolist())
            if cm.any():
                sims.extend(cs[cm].cpu().tolist())

    return float(np.mean(sims)) if sims else 0.0

def score_set(samples: List[Dict[str,Any]], root_dir: str,
              ref, row_masks, col_masks, tag: str) -> np.ndarray:
    out = []
    for ex in tqdm(samples, desc=f"Score({tag})", dynamic_ncols=True):
        out.append(gradsafe_score(ex, ref, row_masks, col_masks, root_dir=root_dir))
    return np.array(out, dtype=np.float32)

# ---------------------------
# 11) Run pipeline
# ---------------------------
print("\n" + "="*90)
print("Run GradAudit P/S/N pipeline (LoRA grads only, last layers only)")
print("="*90)

print("Step A) Build reference gradient from P_calib")
ref = build_reference_grad(P_calib, root_dir=MEMBER_ROOT)

print("\nStep B) Compute avg sims for P_calib vs N_calib")
P_rc = avg_rowcol_sims(P_calib, ref, root_dir=MEMBER_ROOT, tag="P_calib")
N_rc = avg_rowcol_sims(N_calib, ref, root_dir=NONMEMBER_ROOT, tag="N_calib")
if (not P_rc[0]) or (not N_rc[0]):
    raise RuntimeError("Empty calib sims -> cannot build masks. Check data paths or reduce calib sizes.")

print("\nStep C) Build sensitive masks (tau-or-topQ)")
row_masks, col_masks, total_sens = build_sensitive_masks(P_rc, N_rc, tau=SENS_TAU, q=Q_TOP)
print(f"✅ Sensitive dims total = {total_sens}")
if total_sens == 0:
    print("⚠️ total_sens=0 -> scores may collapse. Consider lowering tau (e.g., 0.05) or increasing CALIB sizes.")

print("\nStep D) Score probes: P_probe / S_probe / N_probe")
P_scores = score_set(P_probe, MEMBER_ROOT, ref, row_masks, col_masks, tag="P")
S_scores = score_set(S_probe, MEMBER_ROOT, ref, row_masks, col_masks, tag="S")
N_scores = score_set(N_probe, NONMEMBER_ROOT, ref, row_masks, col_masks, tag="N")

# ---------------------------
# 12) Evaluations
# ---------------------------
print("\n" + "="*90)
print("📊 Evaluations")
print("="*90)

def auc_pair(pos: np.ndarray, neg: np.ndarray) -> float:
    y = np.array([1]*len(pos) + [0]*len(neg), dtype=np.int32)
    s = np.concatenate([pos, neg], axis=0)
    if np.all(s == s[0]):
        return 0.5
    return float(roc_auc_score(y, s))

auc_P_S = auc_pair(P_scores, S_scores)
auc_P_N = auc_pair(P_scores, N_scores)
auc_S_N = auc_pair(S_scores, N_scores)

print(f"🏆 Experiment-1 AUC (P vs S) = {auc_P_S:.4f}")
print(f"Sanity AUC (P vs N) = {auc_P_N:.4f}")
print(f"Sanity AUC (S vs N) = {auc_S_N:.4f}")

# Experiment 2: threshold from P quantile to fix TPR(P)=80%, report FPR(S)
target_tpr = 0.80
thr = float(np.quantile(P_scores, 1.0 - target_tpr))
TPR_on_P = float(np.mean(P_scores >= thr))
FPR_on_S = float(np.mean(S_scores >= thr))

print("\n✅ Experiment-2 (Fix TPR on P=80%, report FPR on S)")
print(f"Threshold (from P quantile) = {thr:.6f}")
print(f"TPR on P (should be ~0.80)  = {TPR_on_P*100:.2f}%")
print(f"FPR on S (mis-kill rate)    = {FPR_on_S*100:.2f}%")

# Confusion matrix for P vs S under thr
y_ps = np.array([1]*len(P_scores) + [0]*len(S_scores), dtype=np.int32)
s_ps = np.concatenate([P_scores, S_scores], axis=0)
pred_ps = (s_ps >= thr).astype(int)
cm_ps = confusion_matrix(y_ps, pred_ps)
print("\nConfusion Matrix (P=pos, S=neg) under TPR≈80% threshold:")
print(cm_ps)

# Save CSV
import pandas as pd
df = pd.DataFrame({
    "P_score": P_scores,
    "S_score": S_scores,
    "N_score": N_scores
})
df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"\n💾 Saved: {OUT_CSV}")
print("\n✅ Done.")


In [ ]:
# ==================================================================================
# @title 9. NA-PDD
# NA-PDD for Vision-Language Models (Qwen2-VL + LoRA) - Structure Analysis
# Step 1: Print model structure and identify LoRA parameters
# ==================================================================================

import os, json, gc, random, re, tarfile
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional
from collections import Counter

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve, auc, accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

import warnings
warnings.filterwarnings("ignore")

# ---------------------------
# 0) Paths & Hyperparameters
# ---------------------------
DRIVE_ROOT = "/content/drive/MyDrive/MedTrinity_MIA"
MEMBER_JSON    = os.path.join(DRIVE_ROOT, "member_eval_1000.json")
NONMEMBER_JSON = os.path.join(DRIVE_ROOT, "nonmember_eval_1000.json")
MEMBER_ROOT    = os.path.join(DRIVE_ROOT, "member_5k")
NONMEMBER_ROOT = os.path.join(DRIVE_ROOT, "nonmember_5k")

BASE_ID    = "Qwen/Qwen2-VL-2B-Instruct"
CACHE_DIR  = "/content/hf_cache"

LORA_TAR = os.path.join(DRIVE_ROOT, "lora_adapters", "qwen2vl_medtrinity_eval1000_lora.tar.gz")
LORA_LOCAL_DIR = "/content/qwen2vl_lora_local"

SEED   = 20251012
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# Split sizes
CALIB_P = 200
CALIB_N = 200
PROBE_P = 800
PROBE_S = 800
PROBE_N = 800

# NA-PDD specific parameters
ACTIVATION_THRESHOLD = 2.0
TOP_N_LAYERS = 15
RELATIVE_RATIO_THRESHOLD = 1.0

PROMPT_EN = (
    "As a medical vision-language model, describe this medical image and "
    "provide a professional diagnostic impression."
)

OUT_CSV = os.path.join(DRIVE_ROOT, "napdd_qwen2vl_lora_multimodal_PSN_eval1000.csv")

# ---------------------------
# Utilities
# ---------------------------
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    return d["data"] if isinstance(d, dict) and "data" in d else d

def ensure_image(path: str) -> Optional[Image.Image]:
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, FileNotFoundError, OSError, KeyError):
        return None

def build_messages_with_image(prompt_text: str) -> List[Dict[str, Any]]:
    return [{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt_text}]}]

def extract_assistant_text(rec: Dict[str, Any]) -> str:
    msgs = rec.get("messages", [])
    if not msgs:
        return str(rec.get("caption", "")).strip()
    last = msgs[-1]
    cont = last.get("content", "")
    if isinstance(cont, str):
        return cont.strip()
    if isinstance(cont, list):
        out = []
        for seg in cont:
            if isinstance(seg, dict) and seg.get("type") == "text":
                out.append(seg.get("text", ""))
        return " ".join(out).strip()
    if isinstance(cont, dict):
        return str(cont.get("text", "")).strip()
    return str(cont).strip()

def set_assistant_text(rec: Dict[str,Any], new_text: str) -> Dict[str,Any]:
    x = dict(rec)
    msgs = x.get("messages", None)
    if not isinstance(msgs, list) or len(msgs) == 0:
        x["caption"] = str(new_text)
        return x
    msgs2 = [dict(m) for m in msgs]
    last = dict(msgs2[-1])
    last["role"] = last.get("role", "assistant")
    last["content"] = [{"type":"text", "text": str(new_text)}]
    msgs2[-1] = last
    x["messages"] = msgs2
    return x

def get_rel_image_path(rec: Dict[str,Any]) -> Optional[str]:
    imgs = rec.get("images", None)
    if isinstance(imgs, list) and len(imgs) > 0:
        return imgs[0]
    if "image" in rec and isinstance(rec["image"], str):
        return rec["image"]
    return None

def safe_cuda_cleanup():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

def ensure_local_adapter(tar_path: str, local_dir: str) -> str:
    os.makedirs(local_dir, exist_ok=True)
    cfg = os.path.join(local_dir, "adapter_config.json")
    if os.path.isfile(cfg):
        return local_dir
    if not os.path.isfile(tar_path):
        raise FileNotFoundError(f"LoRA tar not found: {tar_path}")
    print(f"📦 Unpacking adapter: {tar_path} -> {local_dir}")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=local_dir)
    if not os.path.isfile(cfg):
        raise RuntimeError("Unpacked adapter but adapter_config.json not found.")
    return local_dir

# ---------------------------
# Load data & build P/S/N
# ---------------------------
set_all_seeds(SEED)

members_all = load_json_list(MEMBER_JSON)
nonmem_all  = load_json_list(NONMEMBER_JSON)
assert len(members_all) >= 1000 and len(nonmem_all) >= 1000

members_all = members_all[:1000]
nonmem_all  = nonmem_all[:1000]

P_all = members_all

caps = [extract_assistant_text(x) for x in members_all]
perm = list(range(len(caps)))
random.Random(SEED + 999).shuffle(perm)
for i in range(len(perm)):
    if perm[i] == i:
        j = (i + 1) % len(perm)
        perm[i], perm[j] = perm[j], perm[i]
S_all = [set_assistant_text(members_all[i], caps[perm[i]]) for i in range(len(members_all))]

N_all = nonmem_all

print(f"✅ Built sets: P={len(P_all)} | S={len(S_all)} | N={len(N_all)}")
diff_cnt = sum(extract_assistant_text(P_all[i]) != extract_assistant_text(S_all[i]) for i in range(50))
print(f"Sanity (first 50): P vs S assistant-text different = {diff_cnt}/50")

rnd = random.Random(SEED)
rnd.shuffle(P_all); rnd.shuffle(S_all); rnd.shuffle(N_all)

P_calib = P_all[:CALIB_P]
N_calib = N_all[:CALIB_N]

P_probe = P_all[CALIB_P:CALIB_P+PROBE_P]
S_probe = S_all[CALIB_P:CALIB_P+PROBE_S]
N_probe = N_all[CALIB_N:CALIB_N+PROBE_N]

print(f"Calib: P={len(P_calib)}, N={len(N_calib)}")
print(f"Probe: P={len(P_probe)}, S={len(S_probe)}, N={len(N_probe)}")

# ---------------------------
# Load model (base + LoRA)
# ---------------------------
def load_base_plus_lora(base_id: str, adapter_dir: str):
    processor = AutoProcessor.from_pretrained(base_id, trust_remote_code=True, cache_dir=CACHE_DIR)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        base_id,
        device_map="auto",
        torch_dtype=DTYPE,
        trust_remote_code=True,
        cache_dir=CACHE_DIR
    )
    print(f"🔗 Using LoRA adapter: {adapter_dir}")
    model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=False)
    model.eval()
    return processor, model

adapter_dir = ensure_local_adapter(LORA_TAR, LORA_LOCAL_DIR)
print("🔁 Loading base+LoRA...")
processor, model = load_base_plus_lora(BASE_ID, adapter_dir)
print(f"✅ Device={DEVICE}  dtype={DTYPE}")

# ---------------------------
# STEP 1: Analyze Model Structure and LoRA Parameters
# ---------------------------
print("\n" + "="*90)
print("📊 STEP 1: Analyzing Model Structure and LoRA Parameters")
print("="*90)

if hasattr(model, 'base_model'):
    base = model.base_model.model
else:
    base = model

# Collect all parameter names and categorize them
all_params = {}
lora_params = {}
vision_params = {}
connector_params = {}
language_params = {}

print("\n🔍 Scanning all parameters...")
for name, param in model.named_parameters():
    all_params[name] = param.shape

    # Categorize LoRA parameters
    if 'lora' in name.lower():
        lora_params[name] = param.shape

        # Further categorize LoRA params by component
        if 'visual' in name or 'vision' in name:
            vision_params[name] = param.shape
        elif any(x in name.lower() for x in ['merger', 'projector', 'connector', 'mm_projector']):
            connector_params[name] = param.shape
        elif 'language' in name or 'model.layers' in name:
            language_params[name] = param.shape

print(f"\n📈 Parameter Statistics:")
print(f"  Total parameters: {len(all_params)}")
print(f"  LoRA parameters: {len(lora_params)}")
print(f"    - Vision LoRA: {len(vision_params)}")
print(f"    - Connector LoRA: {len(connector_params)}")
print(f"    - Language LoRA: {len(language_params)}")

# Print sample LoRA parameters from each component
print(f"\n📋 Sample LoRA Parameters by Component:")

if vision_params:
    print(f"\n  Vision Component ({len(vision_params)} params):")
    for i, (name, shape) in enumerate(list(vision_params.items())[:5]):
        print(f"    {name}: {shape}")
    if len(vision_params) > 5:
        print(f"    ... and {len(vision_params) - 5} more")

if connector_params:
    print(f"\n  Connector Component ({len(connector_params)} params):")
    for i, (name, shape) in enumerate(list(connector_params.items())[:5]):
        print(f"    {name}: {shape}")
    if len(connector_params) > 5:
        print(f"    ... and {len(connector_params) - 5} more")

if language_params:
    print(f"\n  Language Component ({len(language_params)} params):")
    for i, (name, shape) in enumerate(list(language_params.items())[:5]):
        print(f"    {name}: {shape}")
    if len(language_params) > 5:
        print(f"    ... and {len(language_params) - 5} more")

# Identify layer patterns
print(f"\n🔍 Identifying Layer Patterns:")

vision_layers = set()
connector_layers = set()
language_layers = set()

for name in lora_params.keys():
    # Extract layer numbers
    if 'visual.blocks' in name:
        match = re.search(r'visual\.blocks\.(\d+)', name)
        if match:
            vision_layers.add(int(match.group(1)))
    elif 'language_model.layers' in name or 'model.layers' in name:
        match = re.search(r'layers\.(\d+)', name)
        if match:
            language_layers.add(int(match.group(1)))

print(f"  Vision layers with LoRA: {sorted(vision_layers) if vision_layers else 'None'}")
print(f"  Language layers with LoRA: {sorted(language_layers) if language_layers else 'None'}")

# ---------------------------
# STEP 2: Identify Module Structure
# ---------------------------
print(f"\n" + "="*90)
print("📊 STEP 2: Identifying Module Structure")
print("="*90)

def find_all_modules_with_lora(model, prefix=""):
    """Recursively find all modules that contain LoRA parameters"""
    modules_with_lora = []

    for name, module in model.named_modules():
        full_name = f"{prefix}.{name}" if prefix else name

        # Check if this module has any LoRA parameters
        has_lora = False
        for param_name, _ in module.named_parameters(recurse=False):
            if 'lora' in param_name.lower():
                has_lora = True
                break

        if has_lora:
            modules_with_lora.append((full_name, type(module).__name__, module))

    return modules_with_lora

lora_modules = find_all_modules_with_lora(model)

print(f"\n📋 Modules containing LoRA parameters ({len(lora_modules)} total):")

# Group by component
vision_modules = [m for m in lora_modules if 'visual' in m[0] or 'vision' in m[0]]
connector_modules = [m for m in lora_modules if any(x in m[0].lower() for x in ['merger', 'projector', 'connector'])]
language_modules = [m for m in lora_modules if 'language' in m[0] or 'model.layers' in m[0]]

print(f"\n  Vision Modules ({len(vision_modules)}):")
for name, type_name, _ in vision_modules[:10]:
    print(f"    {name} ({type_name})")
if len(vision_modules) > 10:
    print(f"    ... and {len(vision_modules) - 10} more")

print(f"\n  Connector Modules ({len(connector_modules)}):")
for name, type_name, _ in connector_modules[:10]:
    print(f"    {name} ({type_name})")
if len(connector_modules) > 10:
    print(f"    ... and {len(connector_modules) - 10} more")

print(f"\n  Language Modules ({len(language_modules)}):")
for name, type_name, _ in language_modules[:10]:
    print(f"    {name} ({type_name})")
if len(language_modules) > 10:
    print(f"    ... and {len(language_modules) - 10} more")

# ---------------------------
# STEP 3: Smart Hook Registration focusing on LoRA modules
# ---------------------------
print(f"\n" + "="*90)
print("📊 STEP 3: Registering Hooks on LoRA-Enhanced Modules")
print("="*90)

activations = {}
activated_neurons = {}

def get_activation_hook(name, hook_type="threshold"):
    """Generic hook function to capture activations"""
    def hook(module, input, output):
        if isinstance(output, tuple):
            output = output[0]

        if output is None:
            return

        if not isinstance(output, torch.Tensor):
            return

        if output.dim() >= 2:
            if hook_type == "threshold":
                activation_mask = (output > ACTIVATION_THRESHOLD).detach().cpu()
            else:
                activation_mask = (output.abs() > 1e-6).detach().cpu()

            activated_neurons[name] = activation_mask
    return hook

hooks = []

# Strategy: Register hooks on modules that have LoRA parameters
# This ensures we capture activations from LoRA-enhanced parts

print("\n🔧 Registering hooks on LoRA-enhanced activation functions...")

vision_hooks = 0
connector_hooks = 0
language_hooks = 0

# For each module with LoRA, try to find and hook its activation function
for module_name, module_type, module in lora_modules:
    # Determine component
    component = "unknown"
    if 'visual' in module_name or 'vision' in module_name:
        component = "vision"
    elif any(x in module_name.lower() for x in ['merger', 'projector', 'connector']):
        component = "connector"
    elif 'language' in module_name or 'model.layers' in module_name:
        component = "language"

    # Try to find activation function in this module
    hook_registered = False

    # Common activation function patterns
    for act_attr in ['act_fn', 'act', 'activation_fn', 'activation', 'gelu', 'silu', 'relu']:
        if hasattr(module, act_attr):
            try:
                act_fn = getattr(module, act_attr)
                if callable(act_fn) or isinstance(act_fn, torch.nn.Module):
                    hook = act_fn.register_forward_hook(
                        get_activation_hook(f'{component}_{module_name}_act', 'threshold')
                    )
                    hooks.append(hook)
                    hook_registered = True

                    if component == "vision":
                        vision_hooks += 1
                    elif component == "connector":
                        connector_hooks += 1
                    elif component == "language":
                        language_hooks += 1

                    break
            except:
                pass

    # If module is itself an activation function
    if not hook_registered and module_type in ['SiLU', 'GELU', 'ReLU', 'Tanh']:
        try:
            hook = module.register_forward_hook(
                get_activation_hook(f'{component}_{module_name}', 'threshold')
            )
            hooks.append(hook)

            if component == "vision":
                vision_hooks += 1
            elif component == "connector":
                connector_hooks += 1
            elif component == "language":
                language_hooks += 1
        except:
            pass

print(f"\n📊 Hook Registration Summary:")
print(f"  Vision Component:       {vision_hooks} hooks")
print(f"  Connector Component:    {connector_hooks} hooks")
print(f"  Language Component:     {language_hooks} hooks")
print(f"  Total:                  {len(hooks)} hooks")

# If we still don't have enough hooks, fall back to standard layer detection
if len(hooks) < 10:
    print(f"\n⚠️  Few hooks registered. Falling back to standard layer detection...")

    # Vision encoder
    if hasattr(base, 'visual') and hasattr(base.visual, 'blocks'):
        for i, block in enumerate(base.visual.blocks):
            if hasattr(block, 'mlp'):
                for act_attr in ['act', 'act_fn']:
                    if hasattr(block.mlp, act_attr):
                        try:
                            act_fn = getattr(block.mlp, act_attr)
                            hook = act_fn.register_forward_hook(
                                get_activation_hook(f'vision_block_{i}_mlp_act', 'threshold')
                            )
                            hooks.append(hook)
                            vision_hooks += 1
                            break
                        except:
                            pass

    # Language model
    if hasattr(base, 'language_model') and hasattr(base.language_model, 'layers'):
        for i, layer in enumerate(base.language_model.layers):
            if hasattr(layer, 'mlp'):
                for act_attr in ['act_fn', 'activation_fn']:
                    if hasattr(layer.mlp, act_attr):
                        try:
                            act_fn = getattr(layer.mlp, act_attr)
                            hook = act_fn.register_forward_hook(
                                get_activation_hook(f'language_layer_{i}_mlp_act', 'threshold')
                            )
                            hooks.append(hook)
                            language_hooks += 1
                            break
                        except:
                            pass

    print(f"\n📊 Updated Hook Registration Summary:")
    print(f"  Vision Component:       {vision_hooks} hooks")
    print(f"  Connector Component:    {connector_hooks} hooks")
    print(f"  Language Component:     {language_hooks} hooks")
    print(f"  Total:                  {len(hooks)} hooks")

if len(hooks) == 0:
    raise RuntimeError("No hooks could be registered. Cannot proceed.")

# ---------------------------
# Rest of the pipeline (same as before)
# ---------------------------
def process_sample_activation(img_path: str, target_text: str, sample_id: int) -> Dict[str, Any]:
    activations.clear()
    activated_neurons.clear()

    image = ensure_image(img_path)
    if image is None:
        return {'sample_id': sample_id, 'neural_signature': {}}

    try:
        msgs_full = build_messages_with_image(PROMPT_EN) + [
            {"role":"assistant","content":[{"type":"text","text":target_text}]}
        ]
        text_full = processor.apply_chat_template(msgs_full, tokenize=False, add_generation_prompt=False)
        batch = processor(images=[image], text=[text_full], return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model(**batch)

        sample_neural_signature = {}
        for key, value in activated_neurons.items():
            try:
                if len(value.shape) >= 2:
                    mask = value.squeeze(0).numpy()

                    if len(mask.shape) == 2:
                        active_neurons = set()
                        for pos in range(mask.shape[0]):
                            active_indices = np.where(mask[pos])[0]
                            active_neurons.update(active_indices)

                        if active_neurons:
                            sample_neural_signature[key] = list(active_neurons)
                    elif len(mask.shape) == 1:
                        active_indices = np.where(mask)[0]
                        if len(active_indices) > 0:
                            sample_neural_signature[key] = active_indices.tolist()
            except Exception as e:
                pass

        safe_cuda_cleanup()
        return {
            'sample_id': sample_id,
            'neural_signature': sample_neural_signature
        }

    except Exception as e:
        safe_cuda_cleanup()
        return {'sample_id': sample_id, 'neural_signature': {}}

def collect_activations(samples: List[Dict[str,Any]], root_dir: str, tag: str) -> List[Dict[str,Any]]:
    results = []
    for i, sample in enumerate(tqdm(samples, desc=f"Collect({tag})", dynamic_ncols=True)):
        rel = get_rel_image_path(sample)
        if not rel:
            continue
        img_path = os.path.join(root_dir, rel)
        target = extract_assistant_text(sample)
        if not target:
            continue

        result = process_sample_activation(img_path, target, i)
        results.append(result)

        if (i + 1) % 50 == 0:
            safe_cuda_cleanup()

    return results

def analyze_neuron_activation_patterns(member_samples, nonmember_samples):
    results = {}
    layer_names = set()
    for sample in member_samples + nonmember_samples:
        layer_names.update(sample['neural_signature'].keys())

    for layer_name in layer_names:
        member_neuron_counts = Counter()
        nonmember_neuron_counts = Counter()

        for sample in member_samples:
            if layer_name in sample['neural_signature']:
                member_neuron_counts.update(sample['neural_signature'][layer_name])

        for sample in nonmember_samples:
            if layer_name in sample['neural_signature']:
                nonmember_neuron_counts.update(sample['neural_signature'][layer_name])

        member_freq = {n: count / len(member_samples) for n, count in member_neuron_counts.items()}
        nonmember_freq = {n: count / len(nonmember_samples) for n, count in nonmember_neuron_counts.items()}

        member_dominant = {}
        for neuron, freq in member_freq.items():
            if neuron not in nonmember_freq or freq > nonmember_freq[neuron] * 1.5:
                member_dominant[neuron] = freq

        nonmember_dominant = {}
        for neuron, freq in nonmember_freq.items():
            if neuron not in member_freq or freq > member_freq[neuron] * 1.5:
                nonmember_dominant[neuron] = freq

        common_neurons = {}
        for neuron in set(member_freq.keys()) & set(nonmember_freq.keys()):
            if neuron not in member_dominant and neuron not in nonmember_dominant:
                common_neurons[neuron] = (member_freq[neuron], nonmember_freq[neuron])

        results[layer_name] = {
            'member_dominant': member_dominant,
            'nonmember_dominant': nonmember_dominant,
            'common_neurons': common_neurons,
            'member_freq': member_freq,
            'nonmember_freq': nonmember_freq
        }

    return results

def build_reference_patterns(P_samples, N_samples):
    print(f"Building reference patterns: P={len(P_samples)}, N={len(N_samples)}")
    return analyze_neuron_activation_patterns(P_samples, N_samples)

def calculate_layer_discrimination_scores(reference_patterns):
    layer_scores = {}
    for layer_name, data in reference_patterns.items():
        member_dominant_count = len(data['member_dominant'])
        nonmember_dominant_count = len(data['nonmember_dominant'])
        discrimination_score = member_dominant_count - nonmember_dominant_count
        layer_scores[layer_name] = discrimination_score
    return layer_scores

def select_discriminative_layers(layer_scores, top_n=TOP_N_LAYERS):
    sorted_scores = sorted(layer_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    selected_layers = [layer for layer, score in sorted_scores[:top_n]]

    print(f"\n📋 Selected {len(selected_layers)} most discriminative layers:")
    vision_count = sum(1 for l in selected_layers if 'vision' in l)
    connector_count = sum(1 for l in selected_layers if 'connector' in l)
    language_count = sum(1 for l in selected_layers if 'language' in l)
    print(f"  Vision:     {vision_count}")
    print(f"  Connector:  {connector_count}")
    print(f"  Language:   {language_count}")

    return selected_layers

def predict_membership_by_relative_ratio(sample, reference_patterns, discriminative_layers, threshold=RELATIVE_RATIO_THRESHOLD):
    if not sample['neural_signature']:
        return 0, 0, 0

    layers_counted = 0
    total_member_ratio = 0
    total_nonmember_ratio = 0

    for layer_name in discriminative_layers:
        if layer_name not in sample['neural_signature'] or layer_name not in reference_patterns:
            continue

        layer_data = reference_patterns[layer_name]
        sample_neurons = set(sample['neural_signature'][layer_name])

        if not sample_neurons:
            continue

        member_dominant_set = set(layer_data['member_dominant'].keys())
        member_overlap = len(sample_neurons.intersection(member_dominant_set))

        nonmember_dominant_set = set(layer_data['nonmember_dominant'].keys())
        nonmember_overlap = len(sample_neurons.intersection(nonmember_dominant_set))

        member_ratio = member_overlap / len(member_dominant_set) if len(member_dominant_set) > 0 else 0
        nonmember_ratio = nonmember_overlap / len(nonmember_dominant_set) if len(nonmember_dominant_set) > 0 else 0

        total_member_ratio += member_ratio
        total_nonmember_ratio += nonmember_ratio
        layers_counted += 1

    if layers_counted == 0:
        return 0, 0, 0

    avg_member_ratio = total_member_ratio / layers_counted
    avg_nonmember_ratio = total_nonmember_ratio / layers_counted

    if avg_nonmember_ratio == 0:
        ratio = float('inf')
    else:
        ratio = avg_member_ratio / avg_nonmember_ratio

    prediction = 1 if ratio >= threshold else 0
    return prediction, avg_member_ratio, avg_nonmember_ratio

def find_best_relative_ratio_threshold(val_samples, reference_patterns, discriminative_layers):
    ratios = []
    for sample in tqdm(val_samples, desc="Find threshold", dynamic_ncols=True):
        _, member_ratio, nonmember_ratio = predict_membership_by_relative_ratio(
            sample, reference_patterns, discriminative_layers, threshold=1.0
        )
        ratio = float('inf') if nonmember_ratio == 0 else member_ratio / nonmember_ratio
        ratios.append(ratio)

    filtered_ratios = [r for r in ratios if r != float('inf') and not np.isnan(r)]
    if not filtered_ratios:
        print("Warning: No valid ratios, using default threshold 1.0")
        return 1.0

    best_threshold = float(np.median(filtered_ratios))
    print(f"✅ Best threshold = {best_threshold:.4f}")
    return best_threshold

def score_set(samples: List[Dict[str,Any]], reference_patterns, discriminative_layers, tag: str) -> np.ndarray:
    scores = []
    for sample in tqdm(samples, desc=f"Score({tag})", dynamic_ncols=True):
        _, member_ratio, nonmember_ratio = predict_membership_by_relative_ratio(
            sample, reference_patterns, discriminative_layers
        )
        ratio = float('inf') if nonmember_ratio == 0 else member_ratio / nonmember_ratio
        if ratio == float('inf'):
            ratio = 1000.0
        if np.isnan(ratio):
            ratio = 0.0
        scores.append(ratio)
    return np.array(scores, dtype=np.float32)

# ---------------------------
# Run Pipeline
# ---------------------------
print("\n" + "="*90)
print("Running NA-PDD for LoRA-Enhanced Multi-Modal Model (Qwen2-VL)")
print("="*90)

print("\nStep A) Collect neuron activations for calibration sets")
P_calib_acts = collect_activations(P_calib, MEMBER_ROOT, tag="P_calib")
N_calib_acts = collect_activations(N_calib, NONMEMBER_ROOT, tag="N_calib")

print("\nStep B) Build reference activation patterns")
reference_patterns = build_reference_patterns(P_calib_acts, N_calib_acts)

print("\nStep C) Calculate layer discrimination scores")
layer_scores = calculate_layer_discrimination_scores(reference_patterns)

print("\nStep D) Select discriminative layers")
discriminative_layers = select_discriminative_layers(layer_scores, top_n=TOP_N_LAYERS)

print("\nStep E) Collect activations for probe sets")
P_probe_acts = collect_activations(P_probe, MEMBER_ROOT, tag="P_probe")
S_probe_acts = collect_activations(S_probe, MEMBER_ROOT, tag="S_probe")
N_probe_acts = collect_activations(N_probe, NONMEMBER_ROOT, tag="N_probe")

print("\nStep F) Find optimal threshold")
val_size = min(100, len(P_probe_acts) // 4)
val_samples = P_probe_acts[:val_size]
best_threshold = find_best_relative_ratio_threshold(val_samples, reference_patterns, discriminative_layers)

print("\nStep G) Score all probe sets")
P_scores = score_set(P_probe_acts, reference_patterns, discriminative_layers, tag="P")
S_scores = score_set(S_probe_acts, reference_patterns, discriminative_layers, tag="S")
N_scores = score_set(N_probe_acts, reference_patterns, discriminative_layers, tag="N")

# ---------------------------
# Evaluations
# ---------------------------
print("\n" + "="*90)
print("📊 Evaluation Results")
print("="*90)

def auc_pair(pos: np.ndarray, neg: np.ndarray) -> float:
    y = np.array([1]*len(pos) + [0]*len(neg), dtype=np.int32)
    s = np.concatenate([pos, neg], axis=0)
    if np.all(s == s[0]):
        return 0.5
    return float(roc_auc_score(y, s))

auc_P_S = auc_pair(P_scores, S_scores)
auc_P_N = auc_pair(P_scores, N_scores)
auc_S_N = auc_pair(S_scores, N_scores)

print(f"\n🏆 Experiment-1: AUC Metrics")
print(f"AUC (P vs S) = {auc_P_S:.4f}  [Main result: paired vs shuffled]")
print(f"AUC (P vs N) = {auc_P_N:.4f}  [Sanity check: paired vs nonmember]")
print(f"AUC (S vs N) = {auc_S_N:.4f}  [Sanity check: shuffled vs nonmember]")

target_tpr = 0.80
thr = float(np.quantile(P_scores, 1.0 - target_tpr))
TPR_on_P = float(np.mean(P_scores >= thr))
FPR_on_S = float(np.mean(S_scores >= thr))

print(f"\n✅ Experiment-2: Fixed TPR Analysis")
print(f"Threshold (from P quantile) = {thr:.6f}")
print(f"TPR on P (target ~80%)      = {TPR_on_P*100:.2f}%")
print(f"FPR on S (mis-kill rate)    = {FPR_on_S*100:.2f}%")

y_ps = np.array([1]*len(P_scores) + [0]*len(S_scores), dtype=np.int32)
s_ps = np.concatenate([P_scores, S_scores], axis=0)
pred_ps = (s_ps >= thr).astype(int)
cm_ps = confusion_matrix(y_ps, pred_ps)
print("\nConfusion Matrix (P=positive, S=negative):")
print(cm_ps)

print(f"\n📈 Additional Metrics (P vs N):")
y_pn = np.array([1]*len(P_scores) + [0]*len(N_scores), dtype=np.int32)
s_pn = np.concatenate([P_scores, N_scores], axis=0)
pred_pn = (s_pn >= best_threshold).astype(int)
acc = accuracy_score(y_pn, pred_pn)
prec, rec, f1, _ = precision_recall_fscore_support(y_pn, pred_pn, average='binary')
print(f"Accuracy  = {acc:.4f}")
print(f"Precision = {prec:.4f}")
print(f"Recall    = {rec:.4f}")
print(f"F1-Score  = {f1:.4f}")

import pandas as pd
df = pd.DataFrame({
    "P_score": P_scores,
    "S_score": S_scores,
    "N_score": N_scores
})
df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"\n💾 Results saved to: {OUT_CSV}")

for hook in hooks:
    hook.remove()

print("\n✅ LoRA-Enhanced Multi-Modal NA-PDD pipeline completed successfully!")

In [ ]:
# ==================================================================================
# @title 10. GradNorm
# GradNorm MIA (member vs nonmember) for Qwen2-VL-2B + PEFT LoRA
# ==================================================================================
# - Data: member_eval_1000.json vs nonmember_eval_1000.json
# - Score: GradNorm = sqrt(sum ||grad||^2 over selected trainable params)
# - Grads: LoRA only, restricted to Vision lastN + Text lastM + (Connector/MM projector)
# - Loss: supervised, labels masked to only assistant segment (same as your previous pipeline)
# - Eval: AUC, TPR@5%FPR
# ==================================================================================

import os, json, gc, random, re, tarfile, time
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

import torch
from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

from sklearn.metrics import roc_auc_score, roc_curve

# ---------------------------
# 0) Paths & hyperparams
# ---------------------------
DRIVE_ROOT = "/content/drive/MyDrive/MedTrinity_MIA"

MEMBER_JSON    = os.path.join(DRIVE_ROOT, "member_eval_1000.json")
NONMEMBER_JSON = os.path.join(DRIVE_ROOT, "nonmember_eval_1000.json")

MEMBER_ROOT    = os.path.join(DRIVE_ROOT, "member_5k")
NONMEMBER_ROOT = os.path.join(DRIVE_ROOT, "nonmember_5k")

BASE_ID    = "Qwen/Qwen2-VL-2B-Instruct"
CACHE_DIR  = "/content/hf_cache"

LORA_TAR = os.path.join(DRIVE_ROOT, "lora_adapters", "qwen2vl_medtrinity_eval1000_lora.tar.gz")
LORA_LOCAL_DIR = "/content/qwen2vl_lora_local"

SEED   = 20251012
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# last layers
VISION_LAST_N = 3
TEXT_LAST_M   = 3

# prompt
PROMPT_EN = (
    "As a medical vision-language model, describe this medical image and "
    "provide a professional diagnostic impression."
)

# speed/safety knobs
PRINT_EVERY = 25   # update postfix every N samples
EPS = 1e-12

# ---------------------------
# 1) Utilities
# ---------------------------
def set_all_seeds(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    return d["data"] if isinstance(d, dict) and "data" in d else d

def ensure_image(path: str) -> Optional[Image.Image]:
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, FileNotFoundError, OSError, KeyError):
        return None

def build_messages_with_image(prompt_text: str) -> List[Dict[str, Any]]:
    return [{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt_text}]}]

def extract_assistant_text(rec: Dict[str, Any]) -> str:
    msgs = rec.get("messages", [])
    if not msgs:
        return str(rec.get("caption", "")).strip()
    last = msgs[-1]
    cont = last.get("content", "")
    if isinstance(cont, str):
        return cont.strip()
    if isinstance(cont, list):
        out = []
        for seg in cont:
            if isinstance(seg, dict) and seg.get("type") == "text":
                out.append(seg.get("text", ""))
        return " ".join(out).strip()
    if isinstance(cont, dict):
        return str(cont.get("text", "")).strip()
    return str(cont).strip()

def get_rel_image_path(rec: Dict[str,Any]) -> Optional[str]:
    imgs = rec.get("images", None)
    if isinstance(imgs, list) and len(imgs) > 0:
        return imgs[0]
    if "image" in rec and isinstance(rec["image"], str):
        return rec["image"]
    return None

def _autocast_ctx():
    if DEVICE == "cuda":
        if DTYPE == torch.bfloat16:
            return torch.autocast("cuda", dtype=torch.bfloat16)
        if DTYPE == torch.float16:
            return torch.autocast("cuda", dtype=torch.float16)
    return torch.autocast("cpu", enabled=False)

def safe_cuda_cleanup():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# ---------------------------
# 2) Prepare adapter locally
# ---------------------------
def ensure_local_adapter(tar_path: str, local_dir: str) -> str:
    os.makedirs(local_dir, exist_ok=True)
    cfg = os.path.join(local_dir, "adapter_config.json")
    if os.path.isfile(cfg):
        return local_dir
    if not os.path.isfile(tar_path):
        raise FileNotFoundError(f"LoRA tar not found: {tar_path}")
    print(f"📦 Unpacking adapter: {tar_path} -> {local_dir}")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=local_dir)
    if not os.path.isfile(cfg):
        raise RuntimeError("Unpacked adapter but adapter_config.json not found. Check tar contents.")
    return local_dir

# ---------------------------
# 3) Load model (base + LoRA)
# ---------------------------
def load_base_plus_lora(base_id: str, adapter_dir: str):
    processor = AutoProcessor.from_pretrained(base_id, trust_remote_code=True, cache_dir=CACHE_DIR)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        base_id,
        device_map="auto",
        torch_dtype=DTYPE,
        trust_remote_code=True,
        cache_dir=CACHE_DIR
    )
    print(f"🔗 Using LoRA adapter: {adapter_dir}")
    model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=False)
    model.eval()
    return processor, model

# ---------------------------
# 4) Select trainable params: LoRA only + last layers + projector/connector
# ---------------------------
def _infer_layer_ids(names: List[str], pattern: str) -> Tuple[List[int], Optional[re.Pattern]]:
    rgx = re.compile(pattern)
    ids = []
    for n in names:
        m = rgx.search(n)
        if m:
            try:
                ids.append(int(m.group(1)))
            except Exception:
                pass
    ids = sorted(set(ids))
    return (ids, rgx) if ids else ([], None)

def mark_trainable_lora_lastlayers(m) -> Tuple[List[str], Dict[str, Any]]:
    for _, p in m.named_parameters():
        p.requires_grad_(False)

    all_names = [n for n, _ in m.named_parameters()]

    vision_patterns = [
        r"visual\.blocks\.(\d+)\.",
        r"vision_tower\.blocks\.(\d+)\.",
        r"vision_model\.encoder\.layers\.(\d+)\.",
        r"visual\.model\.layers\.(\d+)\.",
    ]
    vision_ids, vision_rgx, vision_pat = [], None, None
    for pat in vision_patterns:
        ids, rgx = _infer_layer_ids(all_names, pat)
        if ids:
            vision_ids, vision_rgx, vision_pat = ids, rgx, pat
            break

    text_patterns = [
        r"model\.layers\.(\d+)\.",
        r"language_model\.model\.layers\.(\d+)\.",
        r"language_model\.layers\.(\d+)\.",
    ]
    text_ids, text_rgx, text_pat = [], None, None
    for pat in text_patterns:
        ids, rgx = _infer_layer_ids(all_names, pat)
        if ids:
            text_ids, text_rgx, text_pat = ids, rgx, pat
            break

    vmax = max(vision_ids) if vision_ids else None
    tmax = max(text_ids) if text_ids else None
    v_start = (vmax - (VISION_LAST_N - 1)) if vmax is not None else None
    t_start = (tmax - (TEXT_LAST_M   - 1)) if tmax is not None else None

    selected = []
    for n, p in m.named_parameters():
        if p.ndim < 2:
            continue
        nl = n.lower()
        if "lora" not in nl:
            continue

        keep = False
        if any(k in nl for k in ["mm_projector", "multi_modal_projector", "vision_proj", "projector", "connector"]):
            keep = True

        if (not keep) and (vision_rgx is not None) and (v_start is not None):
            m1 = vision_rgx.search(n)
            if m1 and int(m1.group(1)) >= v_start:
                keep = True

        if (not keep) and (text_rgx is not None) and (t_start is not None):
            m2 = text_rgx.search(n)
            if m2 and int(m2.group(1)) >= t_start:
                keep = True

        if keep:
            p.requires_grad_(True)
            selected.append(n)

    meta = dict(
        vision_pat=vision_pat, vmax=vmax, v_start=v_start,
        text_pat=text_pat, tmax=tmax, t_start=t_start
    )
    return selected, meta

# ---------------------------
# 5) Prefix len cache (assistant-only loss)
# ---------------------------
def compute_prefix_len_once(processor, prompt_text: str, image_path_candidates: List[str]) -> int:
    for p in image_path_candidates:
        im = ensure_image(p)
        if im is None:
            continue
        msgs = build_messages_with_image(prompt_text)
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        batch = processor(images=[im], text=[text], return_tensors="pt")
        return int(batch["input_ids"].shape[1])
    return 0

# ---------------------------
# 6) GradNorm per sample
# ---------------------------
def gradnorm_one(processor, model, img_path: str, target_text: str, prefix_len_cached: int) -> Optional[float]:
    image = ensure_image(img_path)
    if image is None:
        return None
    if not target_text:
        return None

    try:
        pl = int(prefix_len_cached or 0)

        msgs_full = build_messages_with_image(PROMPT_EN) + [
            {"role":"assistant","content":[{"type":"text","text":target_text}]}
        ]
        text_full = processor.apply_chat_template(msgs_full, tokenize=False, add_generation_prompt=False)
        batch = processor(images=[image], text=[text_full], return_tensors="pt").to(model.device)

        # fallback if cached pl bad
        if pl <= 0 or pl > int(batch["input_ids"].shape[1]):
            msgs = build_messages_with_image(PROMPT_EN)
            text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            b2 = processor(images=[image], text=[text], return_tensors="pt")
            pl = int(b2["input_ids"].shape[1])

        labels = batch["input_ids"].clone()
        labels[:, :pl] = -100

        model.train()
        model.zero_grad(set_to_none=True)

        with _autocast_ctx():
            out = model(**batch, labels=labels)
            loss = out.loss

        if (loss is None) or (not torch.isfinite(loss)):
            model.zero_grad(set_to_none=True)
            model.eval()
            safe_cuda_cleanup()
            return None

        loss.backward()

        sq_sum = None
        for n, p in model.named_parameters():
            if not p.requires_grad:
                continue
            g = p.grad
            if g is None:
                continue
            gg = (g.detach().to(torch.float32) ** 2).sum()
            sq_sum = gg if sq_sum is None else (sq_sum + gg)

        model.zero_grad(set_to_none=True)
        model.eval()
        safe_cuda_cleanup()

        if sq_sum is None:
            return None
        return float(torch.sqrt(sq_sum + EPS).item())

    except Exception:
        try:
            model.zero_grad(set_to_none=True)
            model.eval()
        except Exception:
            pass
        safe_cuda_cleanup()
        return None

def score_dataset(processor, model, records: List[Dict[str,Any]], root_dir: str, label: int,
                  prefix_len_cached: int, tag: str) -> Tuple[np.ndarray, np.ndarray]:
    scores = []
    y = []
    ok = 0
    t0 = time.time()

    pbar = tqdm(records, desc=f"GradNorm({tag})", dynamic_ncols=True)
    for i, ex in enumerate(pbar):
        rel = get_rel_image_path(ex)
        if not rel:
            scores.append(np.nan); y.append(label); continue
        img_path = os.path.join(root_dir, rel)
        target = extract_assistant_text(ex)

        gn = gradnorm_one(processor, model, img_path, target, prefix_len_cached)
        if gn is None:
            scores.append(np.nan)
        else:
            scores.append(float(gn)); ok += 1
        y.append(label)

        if (i + 1) % max(1, PRINT_EVERY) == 0:
            elapsed = time.time() - t0
            pbar.set_postfix_str(f"ok={ok}/{i+1} elapsed={elapsed:.1f}s")

    return np.array(scores, dtype=np.float32), np.array(y, dtype=np.int32)

# ---------------------------
# 7) Metrics: AUC + TPR@5%FPR
# ---------------------------
def drop_nan(scores: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    m = np.isfinite(scores)
    return scores[m], y[m]

def tpr_at_fpr(scores: np.ndarray, y: np.ndarray, target_fpr: float = 0.05) -> float:
    fpr, tpr, thr = roc_curve(y, scores)
    idx = np.where(fpr <= target_fpr)[0]
    if len(idx) == 0:
        return 0.0
    return float(tpr[idx[-1]])

# ---------------------------
# 8) Main
# ---------------------------
set_all_seeds(SEED)
assert torch.cuda.is_available(), "❌ Need CUDA to run GradNorm efficiently."

adapter_dir = ensure_local_adapter(LORA_TAR, LORA_LOCAL_DIR)
processor, model = load_base_plus_lora(BASE_ID, adapter_dir)

selected_params, meta = mark_trainable_lora_lastlayers(model)
print(f"🔧 Trainable LoRA params selected = {len(selected_params)}")
print(f"  vision_pat={meta['vision_pat']}  v_range={meta['v_start']}..{meta['vmax']}")
print(f"  text_pat={meta['text_pat']}      t_range={meta['t_start']}..{meta['tmax']}")
if len(selected_params) == 0:
    raise RuntimeError("No LoRA params selected. Check name patterns for this model.")

# load data (1000 / 1000)
members = load_json_list(MEMBER_JSON)[:1000]
nonmem  = load_json_list(NONMEMBER_JSON)[:1000]
print(f"✅ Loaded: members={len(members)} nonmembers={len(nonmem)}")

# cache prefix_len using one member image (fallback to nonmember if needed)
cand_paths = []
for ex in members[:50]:
    rel = get_rel_image_path(ex)
    if rel:
        cand_paths.append(os.path.join(MEMBER_ROOT, rel))
for ex in nonmem[:50]:
    rel = get_rel_image_path(ex)
    if rel:
        cand_paths.append(os.path.join(NONMEMBER_ROOT, rel))

PREFIX_LEN = compute_prefix_len_once(processor, PROMPT_EN, cand_paths)
print(f"✅ Cached PREFIX_LEN = {PREFIX_LEN}")

print("\n" + "="*90)
print("Step 1) Compute GradNorm scores for members (1000)")
print("="*90)
m_scores, m_y = score_dataset(processor, model, members, MEMBER_ROOT, label=1,
                              prefix_len_cached=PREFIX_LEN, tag="members")

print("\n" + "="*90)
print("Step 2) Compute GradNorm scores for nonmembers (1000)")
print("="*90)
n_scores, n_y = score_dataset(processor, model, nonmem, NONMEMBER_ROOT, label=0,
                              prefix_len_cached=PREFIX_LEN, tag="nonmembers")

scores = np.concatenate([m_scores, n_scores], axis=0)
y      = np.concatenate([m_y, n_y], axis=0)

scores, y = drop_nan(scores, y)
print(f"\n✅ Valid scored samples: {len(scores)}/{2000}  (nan dropped)")

if len(np.unique(y)) < 2:
    raise RuntimeError("Need both classes after NaN drop, but one class missing.")

# raw AUC
auc_raw = float(roc_auc_score(y, scores))

# auto direction: if auc<0.5, flip scores so that larger=>more member-like
if auc_raw < 0.5:
    scores = -scores
    auc = 1.0 - auc_raw
    flipped = True
else:
    auc = auc_raw
    flipped = False

tpr5 = tpr_at_fpr(scores, y, target_fpr=0.05)

print("\n" + "="*90)
print("📊 GradNorm Results (member vs nonmember, 1000/1000)")
print("="*90)
print(f"AUC        = {auc:.4f}   (raw={auc_raw:.4f}, flipped={flipped})")
print(f"TPR@5%FPR  = {tpr5*100:.2f}%")
print(f"n_valid_M  = {int((y==1).sum())}")
print(f"n_valid_N  = {int((y==0).sum())}")

# Optional: summary stats
m_sc = scores[y==1]
n_sc = scores[y==0]
print("\nScore stats (after direction fix if applied):")
print(f"  mean(M)={m_sc.mean():.6f}  std(M)={m_sc.std():.6f}")
print(f"  mean(N)={n_sc.mean():.6f}  std(N)={n_sc.std():.6f}")

# Optional: save
import pandas as pd
out_csv = os.path.join(DRIVE_ROOT, "gradnorm_member_vs_nonmember_eval1000.csv")
pd.DataFrame({
    "label": y.astype(int),
    "score": scores.astype(np.float32),
}).to_csv(out_csv, index=False, encoding="utf-8")
print(f"\n💾 Saved: {out_csv}")

print("\n✅ Done.")


## zlib

In [ ]:
# ==================================================================================
# Zlib-Calibrated MIA for Qwen2-VL-2B (Medical LoRA)
# 整合功能：自动解压、基座+LoRA合并、数据格式兼容、双重攻击对比
# ==================================================================================

import os, json, gc, random, re, zlib, math, time, tarfile
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, roc_curve

from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel

# ---------------------------
# 0) 路径与配置 (基于你的环境)
# ---------------------------
DRIVE_ROOT = "/content/drive/MyDrive/MedTrinity_MIA"
LORA_TAR   = os.path.join(DRIVE_ROOT, "lora_adapters", "qwen2vl_medtrinity_eval1000_lora.tar.gz")
LORA_LOCAL_DIR = "/content/qwen2vl_lora_local"

MEMBER_JSON    = os.path.join(DRIVE_ROOT, "member_eval_1000.json")
NONMEMBER_JSON = os.path.join(DRIVE_ROOT, "nonmember_eval_1000.json")
MEMBER_ROOT    = os.path.join(DRIVE_ROOT, "member_5k")
NONMEMBER_ROOT = os.path.join(DRIVE_ROOT, "nonmember_5k")

BASE_ID    = "Qwen/Qwen2-VL-2B-Instruct"
SEED       = 20251012
DTYPE      = torch.bfloat16
DEVICE     = "cuda"

PROMPT_EN = (
    "As a medical vision-language model, describe this medical image and "
    "provide a professional diagnostic impression."
)

# ---------------------------
# 1) 辅助函数
# ---------------------------
def ensure_local_adapter(tar_path: str, local_dir: str) -> str:
    """确保 LoRA 权重已解压到本地"""
    os.makedirs(local_dir, exist_ok=True)
    if os.path.isfile(os.path.join(local_dir, "adapter_config.json")):
        return local_dir
    print(f"📦 Unpacking adapter: {tar_path} -> {local_dir}")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=local_dir)
    return local_dir

def get_zlib_ratio(text: str) -> float:
    """计算文本 Zlib 压缩率 (压缩后长度 / 原始长度)"""
    if not text: return 1.0
    text_bytes = text.encode('utf-8')
    return len(zlib.compress(text_bytes, level=9)) / len(text_bytes)

def load_json_data(path: str):
    """兼容性加载 JSON 数据 (处理 list 或 dict 结构)"""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["data"] if isinstance(data, dict) and "data" in data else data

# ---------------------------
# 2) 核心评分逻辑
# ---------------------------
@torch.inference_mode()
def compute_scores_one(processor, model, img_path, target_text, prefix_len):
    try:
        if not os.path.exists(img_path): return None
        image = Image.open(img_path).convert("RGB")

        # 模拟训练时的对话结构
        msgs_full = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT_EN}]},
            {"role": "assistant", "content": [{"type": "text", "text": target_text}]}
        ]
        text_full = processor.apply_chat_template(msgs_full, tokenize=False, add_generation_prompt=False)
        batch = processor(images=[image], text=[text_full], return_tensors="pt").to(model.device)

        labels = batch["input_ids"].clone()
        labels[:, :prefix_len] = -100 # 仅计算 Assistant 部分的 Loss

        with torch.autocast("cuda", dtype=DTYPE):
            outputs = model(**batch, labels=labels)
            raw_loss = outputs.loss.item()

        if not math.isfinite(raw_loss): return None

        # Zlib 校准
        z_ratio = get_zlib_ratio(target_text)
        zlib_score = raw_loss / max(z_ratio, 1e-6)

        return raw_loss, zlib_score
    except Exception as e:
        return None

def run_scoring_loop(records, root_dir, tag, prefix_len):
    raw_list, zlib_list = [], []
    for ex in tqdm(records, desc=f"Scoring {tag}"):
        rel = ex.get("images", [None])[0] or ex.get("image")
        if not rel: continue

        # 正确提取助手文本 (Assistant)
        target = ""
        msgs = ex.get("messages", [])
        if msgs:
            last_msg = msgs[-1]
            content = last_msg.get("content", "")
            if isinstance(content, list):
                target = " ".join([s['text'] for s in content if s.get('type')=='text'])
            else:
                target = content

        res = compute_scores_one(processor, model, os.path.join(root_dir, rel), target, prefix_len)
        if res:
            raw_list.append(res[0])
            zlib_list.append(res[1])

    return np.array(raw_list), np.array(zlib_list)

# ---------------------------
# 3) 主流程：加载与运行
# ---------------------------
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# A. 加载模型 (基座 + LoRA 合并)
print("🚀 Loading Base Model and Merging LoRA...")
adapter_path = ensure_local_adapter(LORA_TAR, LORA_LOCAL_DIR)
processor = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True)
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_ID, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True
)
# 这里是关键：将 LoRA 挂载到基座模型上
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print(f"✅ Model combined and ready on {model.device}")

# B. 数据准备 (使用兼容模式读取)
print("\n🔄 Loading data...")
members = load_json_data(MEMBER_JSON)[:1000]
nonmembers = load_json_data(NONMEMBER_JSON)[:1000]
print(f"✅ Samples: Member={len(members)}, Non-Member={len(nonmembers)}")

# C. 预计算 Prefix Length (Masking)
test_prompt = processor.apply_chat_template(
    [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT_EN}]}],
    tokenize=False, add_generation_prompt=True
)
PREFIX_LEN = processor(text=[test_prompt], images=[Image.new('RGB', (224, 224))], return_tensors="pt")["input_ids"].shape[1]
print(f"✅ Assistant Prefix Length: {PREFIX_LEN}")

# D. 开始评分
print("\n⏳ Running evaluations (this may take a while)...")
m_raw, m_zlib = run_scoring_loop(members, MEMBER_ROOT, "Member", PREFIX_LEN)
n_raw, n_zlib = run_scoring_loop(nonmembers, NONMEMBER_ROOT, "Non-Member", PREFIX_LEN)

# ---------------------------
# 4) 指标计算与对比
# ---------------------------
def evaluate_metrics(m_scores, n_scores, name):
    y_true = np.concatenate([np.ones(len(m_scores)), np.zeros(len(n_scores))])
    all_s = np.concatenate([m_scores, n_scores])

    # MIA 方向判定：Member 理论上 Loss 更低，所以取负值让 Member 分数更高从而计算正确 AUC
    auc_val = roc_auc_score(y_true, -all_s)
    if auc_val < 0.5:
        auc_val = 1 - auc_val
        final_s = all_s
    else:
        final_s = -all_s

    fpr, tpr, _ = roc_curve(y_true, final_s)
    tpr5 = tpr[np.where(fpr <= 0.05)[0][-1]] if any(fpr <= 0.05) else 0.0

    print(f"\n📊 {name} 结果:")
    print(f"  AUC:          {auc_val:.4f}")
    print(f"  TPR @ 5% FPR: {tpr5*100:.2f}%")
    print(f"  Member 中位数: {np.median(m_scores):.4f}")
    print(f"  NonMem 中位数: {np.median(n_scores):.4f}")
    return auc_val, tpr5

print("\n" + "="*60)
print("🏁 FINAL MIA COMPARISON REPORT")
print("="*60)
raw_auc, raw_tpr = evaluate_metrics(m_raw, n_raw, "Raw Loss MIA (Baseline)")
zlib_auc, zlib_tpr = evaluate_metrics(m_zlib, n_zlib, "Zlib-Calibrated MIA (Refined)")
print("\n" + "="*60)

# E. 结果导出
res_path = os.path.join(DRIVE_ROOT, "final_mia_zlib_report.json")
with open(res_path, "w") as f:
    json.dump({
        "raw_loss": {"auc": raw_auc, "tpr5": raw_tpr},
        "zlib_mia": {"auc": zlib_auc, "tpr5": zlib_tpr},
        "stats": {
            "m_count": len(m_raw),
            "n_count": len(n_raw),
            "prefix_len": PREFIX_LEN
        }
    }, f, indent=4)
print(f"💾 Report saved to: {res_path}")

## dc-pdd

In [ ]:
# ==============================================================================
# DC-PDD MIA (MedTrinity Qwen2-VL) - COMPLETE FIX
# Features:
# 1. Auto-extract LoRA .tar.gz (Fixes "Can't find adapter_config.json")
# 2. Fix image processor crash (Manually pass size + resample)
# 3. Assistant-Only DC-PDD Scoring
# ==============================================================================

import os, json, math, random, pickle, warnings, tarfile
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel

warnings.filterwarnings("ignore")

# ============================================================
# 0) Paths & Config (MedTrinity Environment)
# ============================================================
DRIVE_ROOT   = "/content/drive/MyDrive/MedTrinity_MIA"
BASE_MODEL   = "/content/Qwen2-VL-2B-Instruct"

# --- 关键路径修复：指向 tar.gz 并定义解压目录 ---
LORA_TAR       = os.path.join(DRIVE_ROOT, "lora_adapters", "qwen2vl_medtrinity_eval1000_lora.tar.gz")
LORA_LOCAL_DIR = "/content/qwen2vl_lora_local"  # 解压到的本地临时目录

# 频率表路径
FREQ_PATH    = "/content/drive/MyDrive/LLM_MIA/data/fre_dis_c4_qwen2.pkl"

# 数据集路径
MEMBER_DIR        = f"{DRIVE_ROOT}/member_5k"
NONMEMBER_DIR     = f"{DRIVE_ROOT}/nonmember_5k"
MEMBERS_FULL_JSON = f"{MEMBER_DIR}/mllm_data.json"
NONMEMBERS_FULL_JSON = f"{NONMEMBER_DIR}/mllm_data.json"

TRAIN_SIZE   = 1000
SEED         = 42

# DC-PDD hyperparams
ALPHA        = 0.01
DTYPE        = torch.bfloat16
USE_AMP      = True

# Qwen2-VL Image Params (Required to fix processor crash)
IMG_SIZE = {"shortest_edge": 224, "longest_edge": 1344}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# 1) Helper: Auto-Extract LoRA (Copied from Zlib code)
# ============================================================
def ensure_local_adapter(tar_path: str, local_dir: str) -> str:
    """如果本地没有解压后的 adapter_config.json，则解压 tar.gz"""
    os.makedirs(local_dir, exist_ok=True)
    config_path = os.path.join(local_dir, "adapter_config.json")

    if os.path.isfile(config_path):
        print(f"✅ Found extracted adapter at: {local_dir}")
        return local_dir

    if not os.path.exists(tar_path):
        raise FileNotFoundError(f"❌ LoRA tarball not found at: {tar_path}")

    print(f"📦 Unpacking adapter: {tar_path} -> {local_dir}")
    try:
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=local_dir)
        return local_dir
    except Exception as e:
        raise RuntimeError(f"Failed to extract LoRA: {e}")

# ============================================================
# 2) Load Models & Frequencies
# ============================================================
print("🚀 Loading Med-Qwen2-VL Model & Frequency Table...")

# 2.1 Load Frequencies
if not os.path.exists(FREQ_PATH):
    raise FileNotFoundError(f"❌ Frequency file missing: {FREQ_PATH}")
with open(FREQ_PATH, "rb") as f:
    qwen_freq_smo = pickle.load(f)
VOCAB_SIZE_FREQ = len(qwen_freq_smo)

# 2.2 Prepare LoRA Path
try:
    adapter_path = ensure_local_adapter(LORA_TAR, LORA_LOCAL_DIR)
except Exception as e:
    print(f"⚠️ Warning: Auto-extract failed ({e}), trying raw path...")
    adapter_path = os.path.join(DRIVE_ROOT, "lora_adapters", "qwen2vl_medtrinity_eval1000_lora")

# 2.3 Load Processor & Base Model
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True
)

# 2.4 Load Peft Model
print(f"🔗 Loading LoRA from: {adapter_path}")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

MODEL_DEVICE = next(model.parameters()).device
print(f"✅ Model Loaded Successfully on {MODEL_DEVICE}")

# ============================================================
# 3) Helpers: Robust Data Loading
# ============================================================
def _load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def robust_extract_text(c) -> str:
    """Compatible with list/dict/str content formats"""
    if isinstance(c, str):
        return c.strip()
    if isinstance(c, list):
        parts = []
        for seg in c:
            if isinstance(seg, dict) and seg.get("type") == "text" and seg.get("text"):
                parts.append(seg["text"])
            elif isinstance(seg, str):
                parts.append(seg)
        return " ".join(parts).strip()
    return str(c).strip()

def load_img(root: str, rel_path: str):
    """Try multiple paths to find the image"""
    paths_to_try = [
        os.path.join(root, rel_path),
        os.path.join(DRIVE_ROOT, rel_path),
        rel_path
    ]
    for p in paths_to_try:
        if os.path.exists(p):
            return Image.open(p).convert("RGB")
    return None

# ============================================================
# 4) CORE FIX: Processor Encode (Fixes 'do_resize' error)
# ============================================================
def processor_encode(text, image):
    """
    Manually passes 'size' and 'resample' to bypass the
    internal configuration conflict in Qwen2-VL processor.
    """
    try:
        RESAMPLE = Image.Resampling.BICUBIC
    except Exception:
        RESAMPLE = Image.BICUBIC

    base_kwargs = dict(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt",
        size=IMG_SIZE  # {"shortest_edge": 224, "longest_edge": 1344}
    )

    try:
        enc = processor(**base_kwargs, resample=RESAMPLE)
    except TypeError:
        # Fallback for older transformers versions
        enc = processor(**base_kwargs, interpolation="bicubic")

    return {k: v.to(MODEL_DEVICE) for k, v in enc.items()}

# ============================================================
# 5) Packing Logic (Assistant Only)
# ============================================================
def pack_for_dc_pdd(image: Image.Image, question: str, answer: str):
    msgs_full = [
        {"role": "user", "content": [{"type":"image"}, {"type":"text", "text": question}]},
        {"role": "assistant", "content": [{"type":"text", "text": answer}]},
    ]
    msgs_u = [
        {"role": "user", "content": [{"type":"image"}, {"type":"text", "text": question}]},
    ]

    text_full = processor.apply_chat_template(msgs_full, tokenize=False, add_generation_prompt=False)
    text_u    = processor.apply_chat_template(msgs_u,    tokenize=False, add_generation_prompt=True)

    enc_full = processor_encode(text_full, image)
    enc_u    = processor_encode(text_u,    image)

    prefix_len = int(enc_u["input_ids"].shape[1])
    return enc_full, prefix_len

# ============================================================
# 6) DC-PDD Score Calculation
# ============================================================
@torch.inference_mode()
def compute_dc_pdd_score(image: Image.Image, question: str, answer: str) -> float:
    # 1. Prepare inputs
    enc_full, prefix_len = pack_for_dc_pdd(image, question, answer)

    # 2. Forward pass
    if USE_AMP and torch.cuda.is_available():
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            out = model(**enc_full)
    else:
        out = model(**enc_full)

    logits = out.logits                 # [1, T, V]
    input_ids = enc_full["input_ids"][0]  # [T]
    T = int(input_ids.shape[0])

    if T <= prefix_len + 1:
        return float("nan")

    # 3. Align Logits (Predicting next token)
    # We only care about tokens AFTER the User prompt (the Assistant's answer)
    start_pred = max(prefix_len - 1, 0)
    end_pred   = T - 1

    if start_pred >= end_pred:
        return float("nan")

    pred_logits = logits[0, start_pred:end_pred, :]        # [L, V]
    target_ids  = input_ids[start_pred+1:end_pred+1]       # [L]

    # 4. Calculate Probabilities
    log_probs = torch.log_softmax(pred_logits, dim=-1)
    actual_lp = torch.gather(log_probs, 1, target_ids.unsqueeze(1)).squeeze(1)
    probs = torch.exp(actual_lp).float().cpu().numpy()

    # 5. Frequency Calibration
    t_ids_np = target_ids.detach().cpu().numpy()
    freqs = np.array([
        qwen_freq_smo[int(tid)] if (0 <= int(tid) < VOCAB_SIZE_FREQ) else 1e-10
        for tid in t_ids_np
    ], dtype=np.float64)

    # 6. DC-PDD Formula: P * log(1/freq) with truncation
    deviation = probs * np.log(1.0 / (freqs + 1e-12))
    clipped = np.minimum(deviation, ALPHA)

    return float(np.mean(clipped))

# ============================================================
# 7) Execution Loop
# ============================================================
def run_scoring(data, root, label_name):
    scores = []
    print(f"\n🔍 Scoring {label_name}...")

    # Limit for quick testing? Remove [:TRAIN_SIZE] inside loop if passing full data
    for i, item in enumerate(tqdm(data, desc=label_name)):
        try:
            # Check image
            if not item.get("images"): continue
            img = load_img(root, item["images"][0])
            if img is None: continue

            # Extract text
            q = robust_extract_text(item["messages"][0]["content"])
            a = robust_extract_text(item["messages"][1]["content"])
            if not a.strip(): continue

            # Compute Score
            s = compute_dc_pdd_score(img, q, a)

            if (s is None) or (not np.isfinite(s)): continue
            scores.append(float(s))

        except Exception as e:
            if i < 3: print(f"Sample {i} Error: {e}")
            continue

    print(f"✅ {label_name} kept: {len(scores)}/{len(data)}")
    return np.array(scores, dtype=np.float64)

# --- Load Data ---
try:
    m_all = _load_json(MEMBERS_FULL_JSON)
    n_all = _load_json(NONMEMBERS_FULL_JSON)
except FileNotFoundError as e:
    print(f"❌ Data file not found: {e}")
    m_all, n_all = [], []

if m_all and n_all:
    random.seed(SEED); random.shuffle(m_all); m_data = m_all[:TRAIN_SIZE]
    random.seed(SEED); random.shuffle(n_all); n_data = n_all[:TRAIN_SIZE]

    # --- Run Scoring ---
    m_scores = run_scoring(m_data, MEMBER_DIR, "Member")
    n_scores = run_scoring(n_data, NONMEMBER_DIR, "Non-Member")

    # --- Metrics ---
    if len(m_scores) > 0 and len(n_scores) > 0:
        y_true = np.concatenate([np.ones(len(m_scores)), np.zeros(len(n_scores))])
        all_s  = np.concatenate([m_scores, n_scores])

        auc = roc_auc_score(y_true, all_s)
        # Flip if AUC < 0.5
        if auc < 0.5:
            auc = 1.0 - auc

        print("\n" + "="*60)
        print("📊 MedTrinity DC-PDD Results")
        print(f"Valid samples: M={len(m_scores)} N={len(n_scores)}")
        print(f"AUC: {auc:.4f}")
        print("="*60)
    else:
        print("❌ No valid scores collected. Please check paths and image existence.")
else:
    print("❌ Script stopped due to missing data.")

## M$^4$I

### data split

In [ ]:
# =========================
# Module 1) Split data - 固定FT + 多组Reference设计
# 实验设计：
#   - FT (固定): 200 samples from nonmember [0, 200)
#   - Reference pool: 200 samples from nonmember [200, 400)
#     * ref100: [200, 300)
#     * ref150: [200, 350)
#     * ref200: [200, 400)
#   - Dev nonmember: 100 samples [400, 500)
#   - Test nonmember: 500 samples [500, 1000)
#   - Dev member: 100 samples [0, 100)
#   - Test member: 500 samples [100, 600)
#
# Output: /content/splits_medlora.json (保持原文件名兼容)
# =========================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- paths ----
DRIVE_ROOT = "/content/drive/MyDrive/MedTrinity_MIA"
MEMBER_JSON    = os.path.join(DRIVE_ROOT, "member_eval_1000.json")
NONMEMBER_JSON = os.path.join(DRIVE_ROOT, "nonmember_eval_1000.json")

def load_json_list(path):
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    return d["data"] if isinstance(d, dict) and "data" in d else d

members = load_json_list(MEMBER_JSON)[:1000]
nonmem  = load_json_list(NONMEMBER_JSON)[:1000]

print("Loaded:", len(members), len(nonmem))

# ========================================
# 🔥 关键配置：固定 FT + 多组 Reference
# ========================================
FT_SIZE = 200
REFERENCE_SIZES = [100, 150, 200]
DEV_SIZE = 100
TEST_SIZE = 500

# 计算总需求
TOTAL_NONMEMBER_NEEDED = FT_SIZE + max(REFERENCE_SIZES) + DEV_SIZE + TEST_SIZE
TOTAL_MEMBER_NEEDED = DEV_SIZE + TEST_SIZE

print("\n" + "="*70)
print("📊 数据需求分析 (无重叠设计)")
print("="*70)
print(f"Nonmember 总需求: {TOTAL_NONMEMBER_NEEDED}")
print(f"  - FT (固定)           : [0, {FT_SIZE})           = {FT_SIZE}")
print(f"  - Reference pool      : [{FT_SIZE}, {FT_SIZE + max(REFERENCE_SIZES)})      = {max(REFERENCE_SIZES)}")
print(f"  - Dev nonmember       : [{FT_SIZE + max(REFERENCE_SIZES)}, {FT_SIZE + max(REFERENCE_SIZES) + DEV_SIZE})      = {DEV_SIZE}")
print(f"  - Test nonmember      : [{FT_SIZE + max(REFERENCE_SIZES) + DEV_SIZE}, {TOTAL_NONMEMBER_NEEDED})     = {TEST_SIZE}")
print(f"\nMember 总需求: {TOTAL_MEMBER_NEEDED}")
print(f"  - Dev member          : [0, {DEV_SIZE})         = {DEV_SIZE}")
print(f"  - Test member         : [{DEV_SIZE}, {TOTAL_MEMBER_NEEDED})        = {TEST_SIZE}")
print("="*70 + "\n")

# ========================================
# 数据划分（严格无重叠）
# ========================================
rng = random.Random(SEED)
rng.shuffle(members)
rng.shuffle(nonmem)

# --- Nonmember 划分 (1000个) ---
ft200 = nonmem[0:FT_SIZE]  # [0, 200) - 固定FT数据

ref_pool_start = FT_SIZE
ref_pool_end = FT_SIZE + max(REFERENCE_SIZES)
ref_pool = nonmem[ref_pool_start:ref_pool_end]  # [200, 400) - Reference池

dev_start = ref_pool_end
dev_end = dev_start + DEV_SIZE
dev_nonmem100 = nonmem[dev_start:dev_end]  # [400, 500)

test_start = dev_end
test_end = test_start + TEST_SIZE
test_nonmem500 = nonmem[test_start:test_end]  # [500, 1000)

# --- Member 划分 (1000个) ---
dev_mem100 = members[0:DEV_SIZE]  # [0, 100)
test_mem500 = members[DEV_SIZE:DEV_SIZE + TEST_SIZE]  # [100, 600)
unused_mem = members[DEV_SIZE + TEST_SIZE:]  # [600, 1000)

# ========================================
# 打印划分摘要
# ========================================
print("\n" + "="*70)
print("✅ 数据划分完成（严格无重叠）")
print("="*70)
print("\n【Nonmember 划分】")
print(f"  FT (固定)            : [0, {FT_SIZE})         = {len(ft200)} samples")
print(f"  Reference pool       : [{ref_pool_start}, {ref_pool_end})       = {len(ref_pool)} samples")
for ref_size in REFERENCE_SIZES:
    print(f"    - ref{ref_size:<3}           : [{ref_pool_start}, {ref_pool_start + ref_size})       = {ref_size} samples")
print(f"  Dev nonmember        : [{dev_start}, {dev_end})       = {len(dev_nonmem100)} samples")
print(f"  Test nonmember       : [{test_start}, {test_end})      = {len(test_nonmem500)} samples")

print("\n【Member 划分】")
print(f"  Dev member           : [0, {DEV_SIZE})         = {len(dev_mem100)} samples")
print(f"  Test member          : [{DEV_SIZE}, {DEV_SIZE + TEST_SIZE})        = {len(test_mem500)} samples")
print(f"  Unused member        : [{DEV_SIZE + TEST_SIZE}, 1000)       = {len(unused_mem)} samples")
print("="*70 + "\n")

# ========================================
# 🔥 关键：为兼容现有代码，创建多个配置
# ========================================
# 策略：保存多个配置到同一个文件，Module 4 可以选择加载哪个
all_splits = {}

for ref_size in REFERENCE_SIZES:
    # 从同一个 ref_pool 取前缀（确保小ref是大ref的子集）
    train_nonmem = ref_pool[:ref_size]

    config_name = f"ref{ref_size}"
    all_splits[config_name] = {
        "seed": SEED,
        "ft_size": FT_SIZE,
        "ref_size": ref_size,
        "ft200": ft200,  # 所有实验共享同一个 FT 数据
        f"train_nonmem{ref_size}": train_nonmem,
        "dev_mem100": dev_mem100,
        "dev_nonmem100": dev_nonmem100,
        "test_mem500": test_mem500,
        "test_nonmem500": test_nonmem500,
    }

    print(f"✅ Configuration '{config_name}':")
    print(f"  FT (固定, 所有实验共享) : {len(ft200)}")
    print(f"  Reference               : {len(train_nonmem)} (from pool [{ref_pool_start}, {ref_pool_start + ref_size}))")
    print(f"  Dev (member/nonmember)  : {len(dev_mem100)} / {len(dev_nonmem100)}")
    print(f"  Test (member/nonmember) : {len(test_mem500)} / {len(test_nonmem500)}")
    print()

# ========================================
# 🔥 关键：添加默认配置（兼容现有 Module 2/3）
# ========================================
# Module 2/3 可能直接读取顶层字段，所以添加默认配置（使用 ref200）
default_ref_size = 200
all_splits.update({
    "seed": SEED,
    "ft200": ft200,
    "train_nonmem200": ref_pool[:default_ref_size],  # 默认使用 ref200
    "dev_mem100": dev_mem100,
    "dev_nonmem100": dev_nonmem100,
    "test_mem500": test_mem500,
    "test_nonmem500": test_nonmem500,
})

print("\n" + "="*70)
print("ℹ️  添加了顶层默认配置 (使用 ref200)，确保 Module 2/3 兼容")
print("="*70)

# ========================================
# 数据完整性验证
# ========================================
print("\n" + "="*70)
print("🔍 数据完整性验证")
print("="*70)

def verify_no_overlap(split_a, split_b, name_a, name_b):
    """验证两个 split 之间没有重叠"""
    def get_ids(split):
        ids = set()
        for item in split:
            uid = (item.get("image") or
                   item.get("images", [None])[0] if isinstance(item.get("images"), list) else None or
                   item.get("id") or
                   str(item))
            ids.add(uid)
        return ids

    ids_a = get_ids(split_a)
    ids_b = get_ids(split_b)
    overlap = ids_a & ids_b

    if overlap:
        print(f"  ❌ {name_a} 和 {name_b} 有 {len(overlap)} 个重叠样本！")
        return False
    else:
        print(f"  ✅ {name_a} ⊥ {name_b}")
        return True

all_ok = True

# 验证 nonmember splits 无重叠
print("\n【Nonmember 无重叠验证】")
all_ok &= verify_no_overlap(ft200, ref_pool, "FT", "Reference Pool")
all_ok &= verify_no_overlap(ft200, dev_nonmem100, "FT", "Dev Nonmember")
all_ok &= verify_no_overlap(ft200, test_nonmem500, "FT", "Test Nonmember")
all_ok &= verify_no_overlap(ref_pool, dev_nonmem100, "Reference Pool", "Dev Nonmember")
all_ok &= verify_no_overlap(ref_pool, test_nonmem500, "Reference Pool", "Test Nonmember")
all_ok &= verify_no_overlap(dev_nonmem100, test_nonmem500, "Dev Nonmember", "Test Nonmember")

# 验证 member splits 无重叠
print("\n【Member 无重叠验证】")
all_ok &= verify_no_overlap(dev_mem100, test_mem500, "Dev Member", "Test Member")

# 验证 reference 子集关系
print("\n【Reference 子集关系验证】")
for i, size_i in enumerate(REFERENCE_SIZES[:-1]):
    for size_j in REFERENCE_SIZES[i+1:]:
        ref_i_ids = {item.get("image") or item.get("images", [None])[0] for item in ref_pool[:size_i]}
        ref_j_ids = {item.get("image") or item.get("images", [None])[0] for item in ref_pool[:size_j]}
        if ref_i_ids.issubset(ref_j_ids):
            print(f"  ✅ ref{size_i} ⊂ ref{size_j} (子集关系正确)")
        else:
            print(f"  ❌ ref{size_i} 不是 ref{size_j} 的子集！")
            all_ok = False

if all_ok:
    print("\n" + "="*70)
    print("✅✅✅ 所有数据划分验证通过！无重叠，子集关系正确。")
    print("="*70)
else:
    print("\n" + "="*70)
    print("❌ 数据划分验证失败，请检查数据。")
    print("="*70)

# ========================================
# 保存配置（兼容原文件名）
# ========================================
SPLIT_PATH = "/content/splits_medlora.json"
with open(SPLIT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_splits, f, ensure_ascii=False, indent=2)

print(f"\n✅ Saved splits to: {SPLIT_PATH}")


### finetune

In [ ]:
# =========================
# Module 2) Full FT on LoRA-baseline (FT200)
# - Base model: Qwen/Qwen2-VL-2B-Instruct (HF)
# - Baseline: base + medical LoRA (tar.gz -> unpack -> load)
# - Full fine-tune on ft200 (from nonmember)
# - Save full-FT model to local Colab: /content/qwen2vl_med_loraBaseline_fullFT_200
# =========================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, re, json, tarfile, gc
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

# ---------------------------
# 0) Paths & Hyperparameters (YOUR SETTING)
# ---------------------------
DRIVE_ROOT = "/content/drive/MyDrive/MedTrinity_MIA"
MEMBER_JSON    = os.path.join(DRIVE_ROOT, "member_eval_1000.json")
NONMEMBER_JSON = os.path.join(DRIVE_ROOT, "nonmember_eval_1000.json")
MEMBER_ROOT    = os.path.join(DRIVE_ROOT, "member_5k")
NONMEMBER_ROOT = os.path.join(DRIVE_ROOT, "nonmember_5k")

BASE_ID    = "Qwen/Qwen2-VL-2B-Instruct"
CACHE_DIR  = "/content/hf_cache"

LORA_TAR = os.path.join(DRIVE_ROOT, "lora_adapters", "qwen2vl_medtrinity_eval1000_lora.tar.gz")
LORA_LOCAL_DIR = "/content/qwen2vl_lora_local"

SEED   = 20251012
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# FT hyperparams
LR = 1e-5
EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8

print("CUDA available:", torch.cuda.is_available(), "DEVICE:", DEVICE, "DTYPE:", DTYPE)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------------------------
# 1) Ensure local adapter (tar.gz -> dir)
# ---------------------------
def ensure_local_adapter(tar_path: str, local_dir: str) -> str:
    os.makedirs(local_dir, exist_ok=True)
    cfg = os.path.join(local_dir, "adapter_config.json")
    if os.path.isfile(cfg):
        print("✅ Adapter already unpacked:", local_dir)
        return local_dir
    if not os.path.isfile(tar_path):
        raise FileNotFoundError(f"LoRA tar not found: {tar_path}")

    print(f"📦 Unpacking LoRA: {tar_path} -> {local_dir}")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=local_dir)

    if not os.path.isfile(cfg):
        # 有些 tar 会多包一层目录：尝试自动找到
        for root, dirs, files in os.walk(local_dir):
            if "adapter_config.json" in files:
                found = root
                print("✅ Found adapter_config.json under:", found)
                return found
        raise RuntimeError("Unpacked but adapter_config.json not found. Check tar contents.")
    return local_dir

ADAPTER_DIR = ensure_local_adapter(LORA_TAR, LORA_LOCAL_DIR)
print("Adapter dir:", ADAPTER_DIR)

# ---------------------------
# 2) Load FT200 split (must exist)
# ---------------------------
SPLIT_PATH = "/content/splits_medlora.json"  # 你 Module1 保存的 split 文件
if not os.path.exists(SPLIT_PATH):
    raise FileNotFoundError(
        f"Split file not found: {SPLIT_PATH}\n"
        f"Please run Module 1 first to create ft200."
    )

with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

ft200 = splits["ft200"]
print("✅ Loaded ft200:", len(ft200))

# ---------------------------
# 3) Load processor + base + LoRA (baseline)
# ---------------------------
processor = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True, cache_dir=CACHE_DIR)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_ID,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
    cache_dir=CACHE_DIR
)

lora_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    torch_dtype=DTYPE,
    device_map="auto",
    is_trainable=False
)
lora_model.eval()
print("✅ Loaded baseline (base + LoRA). Now merge LoRA for full fine-tune...")

# ---------------------------
# 4) Merge LoRA -> get plain model -> enable full FT
# ---------------------------
ft_model = lora_model.merge_and_unload()

# 保险：确保所有参数可训练
for p in ft_model.parameters():
    p.requires_grad_(True)

ft_model.train()
ft_model.gradient_checkpointing_enable()
ft_model.config.use_cache = False

# quick sanity
n_trainable = sum(p.requires_grad for p in ft_model.parameters())
print("✅ LoRA merged. Full fine-tune model ready.")
print("Trainable params tensors:", n_trainable, "/", sum(1 for _ in ft_model.parameters()))
print("MODEL first param device:", next(ft_model.parameters()).device)

# ---------------------------
# 5) Helpers: image/caption extraction
# ---------------------------
IMAGE_TOKEN_RE = re.compile(r"<\s*image\s*>", flags=re.IGNORECASE)

def strip_image_token(s):
    if not isinstance(s, str):
        return ""
    s = IMAGE_TOKEN_RE.sub("", s)
    s = s.replace("\n", " ").strip()
    return re.sub(r"\s+", " ", s).strip()

def extract_text(content):
    if isinstance(content, str):
        return strip_image_token(content)
    if isinstance(content, list):
        parts = []
        for seg in content:
            if isinstance(seg, dict) and seg.get("type") == "text":
                parts.append(seg.get("text", ""))
            elif isinstance(seg, str):
                parts.append(seg)
        return strip_image_token(" ".join(parts))
    if isinstance(content, dict):
        return strip_image_token(str(content.get("text", "")))
    return strip_image_token(str(content))

def get_caption(item):
    msgs = item.get("messages", [])
    if not isinstance(msgs, list) or len(msgs) == 0:
        return str(item.get("caption", "")).strip()
    # prefer last assistant
    last = msgs[-1]
    if isinstance(last, dict) and last.get("role","") == "assistant":
        return extract_text(last.get("content",""))
    return extract_text(last.get("content",""))

def get_rel_image_path(item):
    imgs = item.get("images", None)
    if isinstance(imgs, list) and len(imgs) > 0:
        return imgs[0]
    if isinstance(item.get("image", None), str):
        return item["image"]
    return None

def load_image(root_dir, item):
    rel = get_rel_image_path(item)
    if not rel:
        return None
    img_path = os.path.join(root_dir, rel)
    if not os.path.exists(img_path):
        return None
    try:
        return Image.open(img_path).convert("RGB")
    except (UnidentifiedImageError, OSError):
        return None

PROMPT_IMG = (
    "As a medical vision-language model, describe this medical image and "
    "provide a professional diagnostic impression."
)

def build_user_only_text(image):
    msgs = [{"role":"user","content":[{"type":"image","image":image},{"type":"text","text":PROMPT_IMG}]}]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def build_full_sft_text(image, caption):
    msgs = [
        {"role":"user","content":[{"type":"image","image":image},{"type":"text","text":PROMPT_IMG}]},
        {"role":"assistant","content": caption}
    ]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

class FT200Dataset(Dataset):
    def __init__(self, items, root_dir):
        self.samples = []
        for it in items:
            img = load_image(root_dir, it)
            cap = get_caption(it)
            if img is None or not cap:
                continue
            self.samples.append((img, cap))
        print(f"✅ FT dataset usable: {len(self.samples)} / {len(items)}")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate_sft(batch):
    images = [x[0] for x in batch]
    caps   = [x[1] for x in batch]

    full_texts  = [build_full_sft_text(im, cap) for im, cap in zip(images, caps)]
    full_inputs = processor(text=full_texts, images=images, return_tensors="pt", padding=True)

    user_texts  = [build_user_only_text(im) for im in images]
    user_inputs = processor(text=user_texts, images=images, return_tensors="pt", padding=True)

    labels = full_inputs["input_ids"].clone()
    for i in range(labels.size(0)):
        user_len = int(user_inputs["attention_mask"][i].sum().item())
        labels[i, :user_len] = -100
    full_inputs["labels"] = labels

    return {k: v.to(next(ft_model.parameters()).device) for k, v in full_inputs.items()}

# ---------------------------
# 6) Train full FT
# ---------------------------
train_ds = FT200Dataset(ft200, NONMEMBER_ROOT)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_sft)

opt = torch.optim.AdamW(ft_model.parameters(), lr=LR)

step = 0
ft_model.train()

for ep in range(EPOCHS):
    for batch in train_dl:
        out = ft_model(**batch)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        step += 1

        if step % GRAD_ACCUM == 0:
            opt.step()
            opt.zero_grad(set_to_none=True)

        if step % 20 == 0:
            print(f"Epoch {ep+1}/{EPOCHS} Step {step} Loss {loss.item():.4f}")

        # optional small cleanup
        if torch.cuda.is_available() and step % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()

# ---------------------------
# 7) Save full-FT model locally
# ---------------------------
FT_MODEL_DIR = "/content/qwen2vl_med_loraBaseline_fullFT_200"
os.makedirs(FT_MODEL_DIR, exist_ok=True)
ft_model.save_pretrained(FT_MODEL_DIR)
processor.save_pretrained(FT_MODEL_DIR)
print("✅ Saved full-FT model to:", FT_MODEL_DIR)


### get acts

In [ ]:
# =========================
# Module 3) Extract Activations (FIXED)
# - Train acts: full-FT model
# - Dev/Test acts: LoRA-baseline model
# - Feature: last-token hidden state for each layer (including embedding)
# =========================

import os, json, tarfile, gc, shutil
import torch
from tqdm.auto import tqdm
from PIL import Image, UnidentifiedImageError

from transformers import AutoProcessor
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration
from peft import PeftModel

# ---------------------------
# 0) Paths
# ---------------------------
DRIVE_ROOT = "/content/drive/MyDrive/MedTrinity_MIA"
MEMBER_ROOT    = os.path.join(DRIVE_ROOT, "member_5k")
NONMEMBER_ROOT = os.path.join(DRIVE_ROOT, "nonmember_5k")

# Use your real base path or HF id
# BASE_ID = "Qwen/Qwen2-VL-2B-Instruct"
BASE_ID = "/content/Qwen2-VL-2B-Instruct"

CACHE_DIR  = "/content/hf_cache"

LORA_TAR = os.path.join(DRIVE_ROOT, "lora_adapters", "qwen2vl_medtrinity_eval1000_lora.tar.gz")
LORA_LOCAL_DIR = "/content/qwen2vl_lora_local"

FT_MODEL_DIR = "/content/qwen2vl_med_loraBaseline_fullFT_200"
SPLIT_PATH   = "/content/splits_medlora.json"

SEED   = 20251012
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print("DEVICE:", DEVICE, "DTYPE:", DTYPE)

PROMPT_IMG = (
    "As a medical vision-language model, describe this medical image and "
    "provide a professional diagnostic impression."
)

# If you want SFT-sequence (include assistant caption) set True
USE_SFT_SEQUENCE = False

# ---------------------------
# 1) Adapter unpack
# ---------------------------
def ensure_local_adapter(tar_path: str, local_dir: str) -> str:
    os.makedirs(local_dir, exist_ok=True)
    cfg = os.path.join(local_dir, "adapter_config.json")
    if os.path.isfile(cfg):
        return local_dir
    if not os.path.isfile(tar_path):
        raise FileNotFoundError(f"LoRA tar not found: {tar_path}")
    print(f"📦 Unpacking LoRA: {tar_path} -> {local_dir}")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=local_dir)

    # adapter may be nested
    if os.path.isfile(cfg):
        return local_dir
    for root, _, files in os.walk(local_dir):
        if "adapter_config.json" in files:
            return root
    raise RuntimeError("adapter_config.json not found after unpack.")

ADAPTER_DIR = ensure_local_adapter(LORA_TAR, LORA_LOCAL_DIR)
print("Adapter dir:", ADAPTER_DIR)

# ---------------------------
# 2) Load splits
# ---------------------------
with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

ft200           = splits["ft200"]
train_nonmem200 = splits["train_nonmem200"]
dev_mem100      = splits["dev_mem100"]
dev_nonmem100   = splits["dev_nonmem100"]
test_mem500     = splits["test_mem500"]
test_nonmem500  = splits["test_nonmem500"]

print("✅ Loaded splits:", len(ft200), len(train_nonmem200),
      len(dev_mem100), len(dev_nonmem100),
      len(test_mem500), len(test_nonmem500))

# ---------------------------
# 3) Helpers
# ---------------------------
def get_rel_image_path(item):
    imgs = item.get("images", None)
    if isinstance(imgs, list) and len(imgs) > 0:
        return imgs[0]
    if isinstance(item.get("image", None), str):
        return item["image"]
    return None

def load_image(root_dir, item):
    rel = get_rel_image_path(item)
    if not rel:
        return None
    path = os.path.join(root_dir, rel)
    if not os.path.exists(path):
        return None
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError):
        return None

def get_caption(item):
    msgs = item.get("messages", [])
    if isinstance(msgs, list) and len(msgs) > 0:
        last = msgs[-1]
        if isinstance(last, dict) and last.get("role","") == "assistant":
            cont = last.get("content", "")
            if isinstance(cont, str):
                return cont.strip()
            if isinstance(cont, list):
                parts = []
                for seg in cont:
                    if isinstance(seg, dict) and seg.get("type") == "text":
                        parts.append(seg.get("text",""))
                return " ".join(parts).strip()
    return str(item.get("caption","")).strip()

def build_user_only_text(processor, image):
    msgs = [{"role":"user","content":[{"type":"image","image":image},{"type":"text","text":PROMPT_IMG}]}]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def build_sft_text(processor, image, caption):
    msgs = [
        {"role":"user","content":[{"type":"image","image":image},{"type":"text","text":PROMPT_IMG}]},
        {"role":"assistant","content": caption}
    ]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

# ---------------------------
# 4) Model loaders
# ---------------------------
def load_full_ft_model(model_dir: str):
    processor = AutoProcessor.from_pretrained(model_dir, trust_remote_code=True)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_dir, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True
    )
    model.eval()
    return processor, model

def load_lora_baseline(base_id: str, adapter_dir: str):
    processor = AutoProcessor.from_pretrained(base_id, trust_remote_code=True, cache_dir=CACHE_DIR)
    base = Qwen2VLForConditionalGeneration.from_pretrained(
        base_id, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True, cache_dir=CACHE_DIR
    )
    model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)
    model.eval()
    return processor, model

# ---------------------------
# 5) Extract & save activations
# ---------------------------
@torch.no_grad()
def get_last_token_hiddens(model, processor, image, caption=None):
    if USE_SFT_SEQUENCE:
        assert caption is not None and len(caption) > 0
        text = build_sft_text(processor, image, caption)
        inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)
    else:
        text = build_user_only_text(processor, image)
        inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    out = model(**inputs, output_hidden_states=True, return_dict=True)
    hstates = out.hidden_states  # (emb, layer1..layerN)

    vecs = []
    for h in hstates:
        vecs.append(h[0, -1, :].detach().to(torch.float32).cpu())  # [dim]
    return vecs

def ensure_dir(p): os.makedirs(p, exist_ok=True)

def detect_num_layers(model, processor):
    dummy = Image.new("RGB", (448, 448), color=(128,128,128))
    vecs = get_last_token_hiddens(model, processor, dummy, caption="dummy" if USE_SFT_SEQUENCE else None)
    return len(vecs)

def generate_split_acts(processor, model, dataset, root_dir, save_dir, idx_offset=0):
    ensure_dir(save_dir)
    num_layers = detect_num_layers(model, processor)
    print(f"✅ hidden_states length = {num_layers} (includes embedding).")

    for i in tqdm(range(len(dataset)), desc=f"Acts -> {save_dir}"):
        item = dataset[i]
        img = load_image(root_dir, item)
        if img is None:
            continue
        cap = get_caption(item) if USE_SFT_SEQUENCE else None
        vecs = get_last_token_hiddens(model, processor, img, caption=cap)

        out_idx = i + idx_offset
        for layer_id, v in enumerate(vecs):
            torch.save(v.unsqueeze(0), os.path.join(save_dir, f"layer_{layer_id}_{out_idx}.pt"))

        if torch.cuda.is_available() and (out_idx % 50 == 0):
            torch.cuda.empty_cache()
            gc.collect()

# ---------------------------
# 6) RUN
# ---------------------------
ACTS_ROOT = "/content/acts"
TRAIN_DIR = os.path.join(ACTS_ROOT, "train")
DEV_DIR   = os.path.join(ACTS_ROOT, "dev")
TEST_DIR  = os.path.join(ACTS_ROOT, "test")

# Clean all (recommended)
shutil.rmtree(TRAIN_DIR, ignore_errors=True)
shutil.rmtree(DEV_DIR, ignore_errors=True)
shutil.rmtree(TEST_DIR, ignore_errors=True)
ensure_dir(TRAIN_DIR); ensure_dir(DEV_DIR); ensure_dir(TEST_DIR)

print("\n=== Load FT model (TRAIN acts) ===")
ft_proc, ft_model = load_full_ft_model(FT_MODEL_DIR)
print("✅ FT model loaded:", FT_MODEL_DIR)

print("\n=== Load LoRA-baseline model (DEV/TEST acts) ===")
base_proc, base_lora_model = load_lora_baseline(BASE_ID, ADAPTER_DIR)
print("✅ Baseline loaded.")

# 1) TRAIN acts: FT model on ft200 (member, from nonmember pool)
print("\n[1/3] TRAIN acts (FT model on ft200 -> label=1)")
generate_split_acts(ft_proc, ft_model, ft200, NONMEMBER_ROOT, TRAIN_DIR, idx_offset=0)

# 1b) TRAIN acts: FT model on train_nonmem200 (nonmember -> label=0, offset=200)
print("\n[1b/3] TRAIN acts (FT model on train_nonmem200 -> label=0, offset=200)")
generate_split_acts(ft_proc, ft_model, train_nonmem200, NONMEMBER_ROOT, TRAIN_DIR, idx_offset=len(ft200))

# 2) DEV acts: baseline on dev_mem100 + dev_nonmem100
print("\n[2/3] DEV acts (baseline on dev)")
print("  - dev member")
generate_split_acts(base_proc, base_lora_model, dev_mem100, MEMBER_ROOT, DEV_DIR, idx_offset=0)
print("  - dev nonmember (offset=100)")
generate_split_acts(base_proc, base_lora_model, dev_nonmem100, NONMEMBER_ROOT, DEV_DIR, idx_offset=len(dev_mem100))

# 3) TEST acts: baseline on test_mem500 + test_nonmem500
print("\n[3/3] TEST acts (baseline on test)")
print("  - test member")
generate_split_acts(base_proc, base_lora_model, test_mem500, MEMBER_ROOT, TEST_DIR, idx_offset=0)
print("  - test nonmember (offset=500)")
generate_split_acts(base_proc, base_lora_model, test_nonmem500, NONMEMBER_ROOT, TEST_DIR, idx_offset=len(test_mem500))

print("\n✅ Done. Acts saved under:", ACTS_ROOT)
print("train:", TRAIN_DIR)
print("dev  :", DEV_DIR)
print("test :", TEST_DIR)



### evaluate

In [ ]:
# =========================
# Module 4) LRProbe MIA - Reference Data Sensitivity Analysis
# - Train probe: /content/acts/train  (FT-model acts)
# - Dev/Test:    /content/acts/dev & /content/acts/test (LoRA-baseline acts)
# - Test multiple reference sizes: 100, 150, 200
# - Select best layer by Dev AUC, report AUC + TPR@5%FPR on Dev & Test
# =========================

import os, json, random
import numpy as np
import torch
from glob import glob
from tqdm import tqdm
from sklearn.metrics import auc, roc_curve

# -------------------------
# 0) Reproducibility
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# -------------------------
# 1) Load splits configuration
# -------------------------
SPLIT_PATH = "/content/splits_medlora.json"
with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    all_splits = json.load(f)

print("✅ Loaded splits configuration from:", SPLIT_PATH)
print("Available configurations:", [k for k in all_splits.keys() if k.startswith("ref")])

# -------------------------
# 2) Helper: make labeled dataset
# -------------------------
def make_labeled_list(member_list, nonmember_list):
    """Create dataset with labels: member=1, nonmember=0"""
    ds = []
    for x in member_list:
        ds.append({"label": 1})
    for x in nonmember_list:
        ds.append({"label": 0})
    return ds

# -------------------------
# 3) LRProbe (identical to original)
# -------------------------
class LRProbe(torch.nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(d_in, 1, bias=False),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def pred(self, x):
        return self(x).round()

    def score(self, x):
        return self(x)

    @staticmethod
    def from_data(acts, labels, lr=0.001, weight_decay=0.1, epochs=1000, device="cpu"):
        acts, labels = acts.to(device), labels.to(device)
        probe = LRProbe(acts.shape[-1]).to(device)

        opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=weight_decay)
        for _ in range(epochs):
            opt.zero_grad()
            loss = torch.nn.BCELoss()(probe(acts), labels)
            loss.backward()
            opt.step()

        return probe

# -------------------------
# 4) ActDataset (identical to original)
# -------------------------
class ActDataset:
    def __init__(self, dataset, dataset_name, layer_num, device):
        """
        dataset: list with {"label":0/1}
        dataset_name: 'train'/'dev'/'test' (folder under /content/acts)
        layer_num: number of layers to evaluate
        """
        self.data = {}
        self.device = device
        self.layer_num = layer_num
        self.dataset_size = len(dataset)

        labels_list = [dataset[i]["label"] for i in range(self.dataset_size)]
        self.labels_tensor = torch.tensor(labels_list, dtype=torch.float32).to(device)

        for layer in range(layer_num):
            layer_acts = self.collect_acts(dataset_name, layer)
            layer_acts = layer_acts[:self.dataset_size]
            last_token_acts = self.get_last_token(layer_acts)
            self.data[layer] = (last_token_acts, self.labels_tensor)

    def get_last_token(self, acts):
        if len(acts.shape) == 3:
            return acts[:, -1, :]
        else:
            return acts

    def collect_acts(self, dataset_name, layer, center=True, scale=True):
        directory = os.path.join("/content/acts", dataset_name)
        pattern = os.path.join(directory, f"layer_{layer}_*.pt")
        activation_files = sorted(
            glob(pattern),
            key=lambda x: int(x.split("_")[-1].split(".")[0])
        )
        if len(activation_files) == 0:
            raise ValueError(f"No activation files found for {pattern}.")

        loaded_acts = []
        for fpath in activation_files:
            batch_acts = torch.load(fpath, map_location=self.device)
            loaded_acts.append(batch_acts)
        acts = torch.cat(loaded_acts, dim=0).to(self.device)

        if center:
            acts = acts - torch.mean(acts, dim=0)
        if scale:
            std_vals = torch.std(acts, dim=0)
            std_vals[std_vals == 0] = 1.0
            acts = acts / std_vals

        return acts.float().to(self.device)

    def get(self, layer):
        return self.data[layer]

# -------------------------
# 5) Metrics (AUC + TPR@5%FPR)
# -------------------------
def compute_metrics(prediction, answers, print_result=True):
    fpr, tpr, _ = roc_curve(np.array(answers, dtype=bool), -np.array(prediction))
    auc_val = auc(fpr, tpr)

    valid_idx = np.where(fpr < 0.05)[0]
    tpr_5_fpr = tpr[valid_idx[-1]] if len(valid_idx) else 0.0

    if print_result:
        print(f" AUC {auc_val:.4f}, TPR@5%FPR {tpr_5_fpr:.4f}")
    return fpr, tpr, auc_val, tpr_5_fpr

def evaluate(probe, test_acts, test_data):
    scores = probe.score(test_acts)
    predictions = []
    labels = []
    for i, ex in tqdm(enumerate(test_data), total=len(test_data), leave=False):
        predictions.append(-scores[i].item())
        labels.append(ex["label"])
    _, _, auc_val, tpr_5_fpr = compute_metrics(predictions, labels, print_result=False)
    return auc_val, tpr_5_fpr

# -------------------------
# 6) Infer layer number
# -------------------------
def infer_layer_num_from_disk(split_name="dev"):
    d = os.path.join("/content/acts", split_name)
    files = glob(os.path.join(d, "layer_*_*.pt"))
    if not files:
        raise RuntimeError(f"No act files found under {d}. Run Module 3 first.")
    layer_ids = []
    for fp in files:
        base = os.path.basename(fp)
        try:
            L = int(base.split("_")[1])
            layer_ids.append(L)
        except:
            pass
    return max(layer_ids) + 1

# -------------------------
# 7) Run probe for single reference size
# -------------------------
def run_probe_single(ds_train, ds_dev, ds_test, layer_num, device):
    """Run probe for a single configuration"""
    print(f"\n📊 Building ActDatasets (layer_num={layer_num})...")

    train_act_dataset = ActDataset(ds_train, "train", layer_num, device)
    dev_act_dataset   = ActDataset(ds_dev,   "dev",   layer_num, device)
    test_act_dataset  = ActDataset(ds_test,  "test",  layer_num, device)

    dev_auc_list, dev_tpr5_list = [], []
    test_auc_list, test_tpr5_list = [], []
    layers = []

    print("\n🔍 Scanning layers...")
    for layer in tqdm(range(layer_num), desc="Layers"):
        train_acts, train_labels = train_act_dataset.get(layer)
        probe = LRProbe.from_data(train_acts, train_labels, device=device)

        dev_acts, _ = dev_act_dataset.get(layer)
        dev_auc, dev_tpr5 = evaluate(probe, dev_acts, ds_dev)

        test_acts, _ = test_act_dataset.get(layer)
        test_auc, test_tpr5 = evaluate(probe, test_acts, ds_test)

        dev_auc_list.append(dev_auc)
        dev_tpr5_list.append(dev_tpr5)
        test_auc_list.append(test_auc)
        test_tpr5_list.append(test_tpr5)
        layers.append(layer)

        if layer % 5 == 0:  # Print every 5 layers
            print(f"  Layer {layer:02d}: dev_auc={dev_auc:.4f}, test_auc={test_auc:.4f}")

    best_idx = int(np.argmax(dev_auc_list))
    best_layer = layers[best_idx]

    return {
        "best_layer": best_layer,
        "dev_auc": float(dev_auc_list[best_idx]),
        "dev_tpr5": float(dev_tpr5_list[best_idx]),
        "test_auc": float(test_auc_list[best_idx]),
        "test_tpr5": float(test_tpr5_list[best_idx]),
        "all_layers": {
            "dev_auc": [float(x) for x in dev_auc_list],
            "dev_tpr5": [float(x) for x in dev_tpr5_list],
            "test_auc": [float(x) for x in test_auc_list],
            "test_tpr5": [float(x) for x in test_tpr5_list],
        }
    }

# -------------------------
# 8) Main: Loop through reference sizes
# -------------------------
print("\n" + "="*70)
print("🚀 Starting Reference Data Sensitivity Analysis")
print("="*70)

# Infer layer number once
layer_num = infer_layer_num_from_disk("dev")
print(f"✅ Detected {layer_num} layers from activation files")

# Store all results
all_results = {}
REFERENCE_SIZES = [100, 150, 200]

for ref_size in REFERENCE_SIZES:
    ref_key = f"ref{ref_size}"

    print("\n" + "="*70)
    print(f"🔬 Testing Reference Size = {ref_size}")
    print("="*70)

    # Load configuration for this reference size
    if ref_key not in all_splits:
        print(f"❌ Configuration '{ref_key}' not found in splits file!")
        continue

    splits = all_splits[ref_key]

    # Extract data
    ft200 = splits["ft200"]
    train_nonmem = splits[f"train_nonmem{ref_size}"]
    dev_mem100 = splits["dev_mem100"]
    dev_nonmem100 = splits["dev_nonmem100"]
    test_mem500 = splits["test_mem500"]
    test_nonmem500 = splits["test_nonmem500"]

    print(f"\n📋 Data sizes:")
    print(f"  FT (member, fixed)      : {len(ft200)}")
    print(f"  Reference (nonmember)   : {len(train_nonmem)}")
    print(f"  Dev (member/nonmember)  : {len(dev_mem100)} / {len(dev_nonmem100)}")
    print(f"  Test (member/nonmember) : {len(test_mem500)} / {len(test_nonmem500)}")

    # Construct datasets
    ds_train = make_labeled_list(ft200, train_nonmem)
    ds_dev   = make_labeled_list(dev_mem100, dev_nonmem100)
    ds_test  = make_labeled_list(test_mem500, test_nonmem500)

    print(f"  Total train size: {len(ds_train)} (FT={len(ft200)} + Ref={len(train_nonmem)})")

    # Run probe
    results = run_probe_single(ds_train, ds_dev, ds_test, layer_num, device)
    results["ref_size"] = ref_size
    results["ft_size"] = len(ft200)
    results["train_size"] = len(ds_train)

    all_results[ref_key] = results

    # Print summary
    print("\n" + "-"*70)
    print(f"📊 Results for ref_size={ref_size}:")
    print(f"  Best Layer: {results['best_layer']}")
    print(f"  DEV  : AUC={results['dev_auc']:.4f}, TPR@5%FPR={results['dev_tpr5']:.4f}")
    print(f"  TEST : AUC={results['test_auc']:.4f}, TPR@5%FPR={results['test_tpr5']:.4f}")
    print("-"*70)

# -------------------------
# 9) Save all results
# -------------------------
out_path = "/content/mia_lrprobe_results_ref_sensitivity.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)

print("\n" + "="*70)
print("✅ All experiments completed!")
print("="*70)
print(f"Results saved to: {out_path}")

# -------------------------
# 10) Summary table
# -------------------------
print("\n" + "="*70)
print("📊 REFERENCE DATA SENSITIVITY SUMMARY")
print("="*70)
print(f"{'Ref Size':<10} {'Best Layer':<12} {'Dev AUC':<10} {'Dev TPR@5%':<12} {'Test AUC':<10} {'Test TPR@5%':<12}")
print("-"*70)

for ref_size in REFERENCE_SIZES:
    ref_key = f"ref{ref_size}"
    if ref_key in all_results:
        r = all_results[ref_key]
        print(f"{ref_size:<10} {r['best_layer']:<12} {r['dev_auc']:<10.4f} {r['dev_tpr5']:<12.4f} {r['test_auc']:<10.4f} {r['test_tpr5']:<12.4f}")

print("="*70)

# Mingpt4-V2

In [ ]:
# ============================================================
# MiniGPT-v2 LOCAL SETUP (Colab /content, NO Drive)
# Output: variable `model` loaded on GPU/CPU
# ============================================================

import os, sys, subprocess, types, re
from pathlib import Path

def run(cmd, quiet=False):
    cmd = [str(c) for c in cmd]
    print("▶", " ".join(cmd))
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL if quiet else None)

def banner(t):
    print("\n" + "="*80)
    print(t)
    print("="*80)

# -----------------------------
# 0) Paths
# -----------------------------
ROOT       = Path("/content")
REPO_DIR   = ROOT / "MiniGPT-4"
ASSETS_DIR = ROOT / "minigptv2_assets"
CKPT_DIR   = ASSETS_DIR / "checkpoints"
LLM_DIR    = ASSETS_DIR / "llm"
HF_CACHE   = ASSETS_DIR / "hf_cache"

for d in [ASSETS_DIR, CKPT_DIR, LLM_DIR, HF_CACHE]:
    d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")

banner("MiniGPT-v2 LOCAL SETUP (MODEL ONLY)")
print("REPO_DIR  :", REPO_DIR)
print("ASSETS_DIR:", ASSETS_DIR)
print("HF_CACHE  :", HF_CACHE)

# -----------------------------
# 1) Install deps (locked)
# -----------------------------
banner("Step 1: Install dependencies (locked)")
run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
     "transformers", "peft", "accelerate", "bitsandbytes"], quiet=True)

run([sys.executable, "-m", "pip", "install", "-q",
     "huggingface_hub", "omegaconf", "pyyaml",
     "transformers==4.37.2",
     "accelerate==0.27.2",
     "peft==0.13.2",
     "bitsandbytes",
     "safetensors",
     "timm", "einops", "sentencepiece",
     "opencv-python-headless",
     "iopath", "fvcore", "yacs", "portalocker",
     "webdataset", "braceexpand",
     "decord"
], quiet=True)

import transformers, peft, accelerate
print("✅ transformers:", transformers.__version__)
print("✅ peft       :", peft.__version__)
print("✅ accelerate :", accelerate.__version__)

# -----------------------------
# 2) Fresh clone repo
# -----------------------------
banner("Step 2: Fresh clone MiniGPT-4 repo")
if REPO_DIR.exists():
    run(["bash", "-lc", f"rm -rf '{REPO_DIR}'"], quiet=True)

run(["git", "clone", "--quiet", "https://github.com/Vision-CAIR/MiniGPT-4.git", str(REPO_DIR)])
print("✅ cloned:", REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# -----------------------------
# 3) Stub optional module: visual_genome
# -----------------------------
banner("Step 3: Stub optional modules (visual_genome)")
vg = types.ModuleType("visual_genome")
vg.local = types.SimpleNamespace()
sys.modules["visual_genome"] = vg
print("✅ visual_genome stub injected")

# -----------------------------
# 4) Patch base_model.py safely
#   - remove prepare_model_for_int8_training ONLY from peft import block
#   - prepend shim that defines it via prepare_model_for_kbit_training
# -----------------------------
banner("Step 4: Patch base_model.py (PEFT fix + shim)")

base_model_path = REPO_DIR / "minigpt4" / "models" / "base_model.py"
assert base_model_path.exists(), f"❌ base_model.py not found: {base_model_path}"
txt = base_model_path.read_text(encoding="utf-8")

# remove token only inside the "from peft import (...)" block if present
pattern_block = r"(from\s+peft\s+import\s*\(\s*[\s\S]*?\))"
m = re.search(pattern_block, txt)
if m:
    block = m.group(1)
    block2 = re.sub(r"^\s*prepare_model_for_int8_training\s*,?\s*$", "", block, flags=re.MULTILINE)
    block2 = re.sub(r",\s*prepare_model_for_int8_training\s*(,)?", lambda mm: "," if mm.group(1) else "", block2)
    block2 = re.sub(r",\s*,", ",", block2)
    txt = txt[:m.start()] + block2 + txt[m.end():]
else:
    # fallback for single-line import
    txt = re.sub(r"(from\s+peft\s+import\s+.*)\bprepare_model_for_int8_training\b\s*,?\s*", r"\1", txt)

shim = """# ===== PEFT compatibility shim (AUTO-INJECTED) =====
# MiniGPT expects `prepare_model_for_int8_training` (old PEFT API).
# PEFT>=0.13 uses `prepare_model_for_kbit_training`.
try:
    from peft import prepare_model_for_kbit_training as prepare_model_for_int8_training
except Exception:
    def prepare_model_for_int8_training(model, *args, **kwargs):
        return model
# =========================================================

"""
if "PEFT compatibility shim (AUTO-INJECTED)" not in txt:
    txt = shim + txt

base_model_path.write_text(txt, encoding="utf-8")
print("✅ base_model.py patched")

# -----------------------------
# 5) Overwrite modeling_llama.py for transformers==4.37.2
# -----------------------------
banner("Step 5: Overwrite modeling_llama.py (safe wrapper)")

llama_path = REPO_DIR / "minigpt4" / "models" / "modeling_llama.py"
assert llama_path.exists(), f"❌ modeling_llama.py not found: {llama_path}"

llama_wrapper = r'''"""
MiniGPT-v2 compatible LLaMA wrapper (safe for transformers==4.37.2)
- Adds `reduction` (per-sample loss)
- Ignores unknown kwargs
"""
import torch
from typing import Optional, List, Union, Tuple
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import CausalLMOutputWithPast

try:
    from transformers.models.llama.modeling_llama import LlamaForCausalLM as LlamaBase
except Exception:
    from transformers import LlamaForCausalLM as LlamaBase


class LlamaForCausalLM(LlamaBase):
    def forward(
        self,
        input_ids: torch.LongTensor = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[List[torch.FloatTensor]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        reduction: str = "mean",
        **kwargs
    ) -> Union[Tuple, CausalLMOutputWithPast]:

        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        hidden_states = outputs[0]
        logits = self.lm_head(hidden_states).float()

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fct = CrossEntropyLoss(reduction=reduction)
            loss = loss_fct(
                shift_logits.view(-1, self.config.vocab_size),
                shift_labels.view(-1).to(shift_logits.device),
            )
            if reduction == "none":
                loss = loss.view(logits.size(0), -1).mean(1)

        if not return_dict:
            out = (logits,) + outputs[1:]
            return (loss,) + out if loss is not None else out

        return CausalLMOutputWithPast(
            loss=loss,
            logits=logits,
            past_key_values=outputs.past_key_values,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )
'''
llama_path.write_text(llama_wrapper, encoding="utf-8")
print("✅ modeling_llama.py overwritten")

# -----------------------------
# 6) Clear cache + import registration modules
# -----------------------------
banner("Step 6: Import minigpt4 modules (register models/tasks)")
for k in list(sys.modules.keys()):
    if k.startswith("minigpt4"):
        sys.modules.pop(k, None)

import minigpt4.models
import minigpt4.processors
import minigpt4.tasks
import minigpt4.datasets.builders
print("✅ minigpt4 imports OK")

# -----------------------------
# 7) Download v2 checkpoint (HF Space)
# -----------------------------
banner("Step 7: Download MiniGPT-v2 checkpoint (HF Space)")
from huggingface_hub import hf_hub_download

SPACE_ID = "Vision-CAIR/MiniGPT-v2"
CKPT_FILENAME = "minigptv2_checkpoint.pth"

ckpt_local_path = Path(
    hf_hub_download(
        repo_id=SPACE_ID,
        filename=CKPT_FILENAME,
        repo_type="space",
        cache_dir=str(HF_CACHE),
        local_dir=str(CKPT_DIR),
        local_dir_use_symlinks=False,
    )
)
print("✅ ckpt:", ckpt_local_path, "| MB:", ckpt_local_path.stat().st_size/1024/1024)

# -----------------------------
# 8) Ensure local LLM exists (download if missing)
# -----------------------------
banner("Step 8: Ensure local LLM (Llama-2-7b-chat-hf)")
from huggingface_hub import snapshot_download

LLM_REPO = "meta-llama/Llama-2-7b-chat-hf"
llm_local_dir = LLM_DIR / "Llama-2-7b-chat-hf"

def llm_ok(p: Path) -> bool:
    return (p / "config.json").exists() and ((p / "tokenizer.model").exists() or (p / "tokenizer.json").exists())

if not llm_ok(llm_local_dir):
    print("⚠️ Llama-2 is gated. Ensure your HF account has access, then login in this runtime.")
    try:
        from huggingface_hub import notebook_login
        notebook_login()
    except Exception as e:
        print("⚠️ notebook_login failed/skipped:", e)

    snapshot_download(
        repo_id=LLM_REPO,
        local_dir=str(llm_local_dir),
        local_dir_use_symlinks=False,
        resume_download=True,
        cache_dir=str(HF_CACHE),
        ignore_patterns=["*.msgpack", "*.h5", "*.ot", "*.md", ".gitattributes"],
    )

if not llm_ok(llm_local_dir):
    raise RuntimeError(f"❌ LLM not ready at: {llm_local_dir}")
print("✅ LLM ready:", llm_local_dir)

# -----------------------------
# 9) Patch eval config + model config (ckpt + local llm + disable 8bit)
# -----------------------------
banner("Step 9: Patch configs (ckpt + llm + disable 8bit)")
import yaml

EVAL_CFG  = REPO_DIR / "eval_configs" / "minigptv2_eval.yaml"
MODEL_CFG = REPO_DIR / "minigpt4" / "configs" / "models" / "minigpt_v2.yaml"

eval_data = yaml.safe_load(EVAL_CFG.read_text())
eval_data.setdefault("model", {})
eval_data["model"]["ckpt"] = str(ckpt_local_path)

# force disable 8bit routes
eval_data["model"]["low_resource"] = False
eval_data["model"]["device_8bit"] = 0
eval_data["model"]["load_in_8bit"] = False
eval_data["model"]["load_8bit"] = False

EVAL_CFG.write_text(yaml.safe_dump(eval_data, sort_keys=False))
print("✅ Patched eval cfg model.ckpt =", ckpt_local_path)

model_data = yaml.safe_load(MODEL_CFG.read_text())
assert isinstance(model_data, dict) and "model" in model_data and isinstance(model_data["model"], dict), "❌ model cfg format unexpected"

def patch_llm_subtree(obj):
    if isinstance(obj, dict):
        return {k: patch_llm_subtree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [patch_llm_subtree(v) for v in obj]
    if isinstance(obj, str):
        s = obj.lower()
        if any(t in s for t in ["meta-llama", "llama-2", "vicuna", "lmsys", "llama"]):
            return str(llm_local_dir)
        return obj
    return obj

model_data["model"] = patch_llm_subtree(model_data["model"])
MODEL_CFG.write_text(yaml.safe_dump(model_data, sort_keys=False))
print("✅ Patched model cfg llm path(s) ->", llm_local_dir)

# -----------------------------
# 10) Build model & load checkpoint => `model`
# -----------------------------
banner("Step 10: Build MiniGPT-v2 model & load checkpoint")
import torch
from argparse import Namespace
from minigpt4.common.config import Config
from minigpt4.common.registry import registry

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM(GB):", torch.cuda.get_device_properties(0).total_memory/1024**3)

args = Namespace(cfg_path=str(EVAL_CFG), options=[])
cfg = Config(args)
model_cfg = cfg.model_cfg
print("Model arch:", model_cfg.arch)

model_cls = registry.get_model_class(model_cfg.arch)
assert model_cls is not None, f"❌ Model not registered: {model_cfg.arch}"

model = model_cls.from_config(model_cfg)

ckpt = torch.load(str(ckpt_local_path), map_location="cpu")
state = ckpt.get("model", ckpt)
msg = model.load_state_dict(state, strict=False)

print("✅ load_state_dict done")
print("  missing keys   :", len(msg.missing_keys))
print("  unexpected keys:", len(msg.unexpected_keys))

model = model.to(device)
model.eval()
torch.cuda.empty_cache()

banner("DONE ✅ MiniGPT-v2 model is ready")
print("Variable: model")
print("ckpt:", ckpt_local_path)
print("llm :", llm_local_dir)


## data

In [ ]:
# ============================================================
# HF Streaming: AbdoTW/COCO_2014
# - train: sample 1000 as MEMBER
# - val  : sample 1000 as NON-MEMBER
# - streaming=True (no full download)
# - returns PyTorch DataLoaders
# ============================================================
import os, random
from pathlib import Path
from typing import Dict, Any, List

import torch
from torch.utils.data import IterableDataset, DataLoader

from datasets import load_dataset
from PIL import Image
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode

# -------------------------
# Config
# -------------------------
DATASET_ID = "AbdoTW/COCO_2014"
CACHE_DIR  = "/content/hf_cache_coco2014"   # streaming cache (small)
SEED       = 42
N_MEMBER   = 1000
N_NONMEM   = 1000
BATCH_SIZE = 8
SHUFFLE_BUFFER = 10_000   # bigger => more random but more RAM

os.makedirs(CACHE_DIR, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)

# -------------------------
# Preprocess (MiniGPT-v2 commonly uses 448 w/ CLIP norm)
# If your model expects 224, change image_size to 224.
# -------------------------
image_size = 448
preprocess = transforms.Compose([
    transforms.Resize((image_size, image_size), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.48145466, 0.4578275, 0.40821073),
        std=(0.26862954, 0.26130258, 0.27577711),
    ),
])

# -------------------------
# Helpers: robust image/caption extraction
# -------------------------
def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")
    if isinstance(x, dict) and "bytes" in x:
        from io import BytesIO
        return Image.open(BytesIO(x["bytes"])).convert("RGB")
    if isinstance(x, (str, Path)):
        return Image.open(str(x)).convert("RGB")
    raise TypeError(f"Unsupported image type: {type(x)}")

def pick_caption(ex: Dict[str, Any]) -> str:
    # common names
    for k in ["caption", "text", "sentence", "raw_caption"]:
        if k in ex and ex[k] is not None:
            v = ex[k]
            if isinstance(v, str):
                return v
            if isinstance(v, list) and len(v) > 0:
                return str(v[0])
            return str(v)

    # nested annotations
    if "annotations" in ex and ex["annotations"] is not None:
        ann = ex["annotations"]
        if isinstance(ann, list) and len(ann) > 0:
            a0 = ann[0]
            if isinstance(a0, dict):
                for kk in ["caption", "text"]:
                    if kk in a0:
                        return str(a0[kk])
            return str(a0)

    return ""

def list_splits(dataset_id: str):
    # non-streaming builder metadata only (tiny)
    ds_builder = load_dataset(dataset_id, split=None, cache_dir=CACHE_DIR)
    return list(ds_builder.keys())

def resolve_val_split(dataset_id: str):
    splits = list_splits(dataset_id)
    # prefer val, else validation
    if "val" in splits:
        return "val"
    if "validation" in splits:
        return "validation"
    raise RuntimeError(f"❌ This dataset has no val/validation split. Available splits: {splits}")

def build_stream(dataset_id: str, split: str, seed: int, buffer_size: int):
    ds = load_dataset(dataset_id, split=split, streaming=True, cache_dir=CACHE_DIR)
    ds = ds.shuffle(seed=seed, buffer_size=buffer_size)  # approximate shuffle
    return ds

def take_n(ds, n: int):
    out = []
    it = iter(ds)
    for _ in range(n):
        out.append(next(it))
    return out

# -------------------------
# Resolve split names
# -------------------------
print(f"Dataset: {DATASET_ID}")
splits = list_splits(DATASET_ID)
print("Available splits:", splits)

VAL_SPLIT = resolve_val_split(DATASET_ID)
print("Using VAL split as:", VAL_SPLIT)

# -------------------------
# Streaming sample (ONLY 1000 each)
# -------------------------
print(f"\nLoading STREAMING dataset: {DATASET_ID}")
train_stream = build_stream(DATASET_ID, "train", SEED, SHUFFLE_BUFFER)
val_stream   = build_stream(DATASET_ID, VAL_SPLIT, SEED + 1, SHUFFLE_BUFFER)

print(f"Sampling member={N_MEMBER} from train ...")
member_samples = take_n(train_stream, N_MEMBER)

print(f"Sampling nonmember={N_NONMEM} from {VAL_SPLIT} ...")
nonmember_samples = take_n(val_stream, N_NONMEM)

print("✅ sampled done",
      f"member={len(member_samples)}",
      f"nonmember={len(nonmember_samples)}")

# -------------------------
# Torch IterableDataset over the sampled list
# -------------------------
class SampleListIterable(IterableDataset):
    def __init__(self, samples: List[Dict[str, Any]], transform=None):
        self.samples = samples
        self.transform = transform

        # infer image key
        self.image_key = "image" if "image" in samples[0] else None
        if self.image_key is None:
            for k, v in samples[0].items():
                if isinstance(v, Image.Image) or (isinstance(v, dict) and "bytes" in v):
                    self.image_key = k
                    break
        if self.image_key is None:
            raise KeyError(f"Could not find image field in sample keys: {list(samples[0].keys())}")

    def __iter__(self):
        for ex in self.samples:
            img = to_pil(ex[self.image_key])
            if self.transform:
                img = self.transform(img)

            cap = pick_caption(ex)
            yield {
                "image": img,        # Tensor [3,H,W]
                "caption": cap,      # str
                "raw": ex            # keep raw fields if you need id/url, etc.
            }

def collate_fn(batch: List[Dict[str, Any]]):
    images = torch.stack([b["image"] for b in batch], dim=0)
    captions = [b["caption"] for b in batch]
    raws = [b["raw"] for b in batch]
    return {"image": images, "caption": captions, "raw": raws}

member_ds = SampleListIterable(member_samples, transform=preprocess)
nonmem_ds = SampleListIterable(nonmember_samples, transform=preprocess)

# NOTE: For IterableDataset, keep num_workers=0 to avoid worker-side issues
member_loader = DataLoader(
    member_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True, collate_fn=collate_fn
)

nonmem_loader = DataLoader(
    nonmem_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True, collate_fn=collate_fn
)

# -------------------------
# Sanity check
# -------------------------
b1 = next(iter(member_loader))
b2 = next(iter(nonmem_loader))
print("\n✅ MEMBER batch image:", b1["image"].shape, "| caption[0]:", b1["caption"][0][:120])
print("✅ NONMEM batch image:", b2["image"].shape, f"| caption[0]:", b2["caption"][0][:120])

print("\nReady variables:")
print(" - member_loader")
print(" - nonmem_loader")
print(" - member_samples, nonmember_samples")
print(f" - VAL_SPLIT = {VAL_SPLIT}")


## loss attack

In [ ]:
# ============================================================
# Loss-based MIA for MiniGPT-v2 (ROBUST, NaN-safe)
# REQUIRED metrics:
#   - AUC
#   - TPR@5% FPR
#
# Assumes already exist:
#   - model
#   - member_loader  (1000 from train)
#   - nonmem_loader  (1000 from validation)
# ============================================================

import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve

# -------------------------
# 0) Device + AMP
# -------------------------
device = next(model.parameters()).device
USE_AMP_DEFAULT = (device.type == "cuda")
print("✅ MIA device:", device, "| AMP(default):", USE_AMP_DEFAULT)

# -------------------------
# 1) Build samples for MiniGPT-v2 forward
# -------------------------
PROMPT = "Describe the image in detail."

def batch_to_samples(batch):
    images = batch["image"].to(device, non_blocking=True)
    caps = batch["caption"]
    caps = [c if isinstance(c, str) and len(c.strip()) > 0 else " " for c in caps]
    return {
        "image": images,
        "text_input": [PROMPT] * len(caps),
        "answer": caps,
    }

# -------------------------
# 2) Forward helper (optionally AMP)
# -------------------------
@torch.no_grad()
def forward_loss(samples, use_amp: bool):
    if use_amp and device.type == "cuda":
        # newer API; avoids your deprecation warning
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out = model.forward(samples, reduction="none")
    else:
        out = model.forward(samples, reduction="none")

    if isinstance(out, dict) and "loss" in out:
        loss = out["loss"]
    else:
        loss = getattr(out, "loss", None)

    if loss is None:
        raise RuntimeError("❌ model.forward did not return loss.")
    if not torch.is_tensor(loss):
        raise RuntimeError("❌ loss is not a torch tensor.")
    if loss.ndim == 0:
        raise RuntimeError(
            "❌ loss is scalar. reduction='none' not working. "
            "Need per-sample loss tensor shape [B]."
        )
    return loss

# -------------------------
# 3) Collect per-sample losses with NaN/Inf handling
# -------------------------
@torch.no_grad()
def collect_losses(dataloader, desc="", use_amp_default=True):
    losses_all = []
    bad_count_total = 0
    total_seen = 0

    for batch in tqdm(dataloader, desc=desc):
        samples = batch_to_samples(batch)
        bsz = samples["image"].shape[0]
        total_seen += bsz

        # 3.1 try with default AMP setting
        loss = forward_loss(samples, use_amp=use_amp_default)
        finite_mask = torch.isfinite(loss)

        # 3.2 if any non-finite and AMP was on, retry same batch with AMP off
        if (not finite_mask.all()) and use_amp_default and device.type == "cuda":
            loss_retry = forward_loss(samples, use_amp=False)
            finite_mask_retry = torch.isfinite(loss_retry)
            if finite_mask_retry.any():
                loss = loss_retry
                finite_mask = finite_mask_retry

        # 3.3 keep only finite losses; drop bad ones
        if not finite_mask.all():
            bad = int((~finite_mask).sum().item())
            bad_count_total += bad

        good_losses = loss[finite_mask].detach().float().cpu().numpy()
        if good_losses.size > 0:
            losses_all.append(good_losses)

    if len(losses_all) == 0:
        raise RuntimeError("❌ No valid (finite) losses collected. Something is wrong with forward/loss.")

    losses_np = np.concatenate(losses_all, axis=0)

    print(f"[INFO] {desc}: total_seen={total_seen}, kept={len(losses_np)}, dropped(non-finite)={bad_count_total}")
    return losses_np

print("\nCollecting MEMBER losses (train=member=1)...")
member_losses = collect_losses(member_loader, desc="member", use_amp_default=USE_AMP_DEFAULT)

print("\nCollecting NON-MEMBER losses (val=nonmember=0)...")
nonmem_losses = collect_losses(nonmem_loader, desc="nonmember", use_amp_default=USE_AMP_DEFAULT)

print("\n✅ Loss shapes after filtering:", "member", member_losses.shape, "| nonmember", nonmem_losses.shape)

# -------------------------
# 4) Build MIA score + REQUIRED metrics (NaN-safe)
# -------------------------
member_scores = -member_losses
nonmem_scores = -nonmem_losses

y_true = np.concatenate([
    np.ones(len(member_scores), dtype=np.int32),
    np.zeros(len(nonmem_scores), dtype=np.int32)
], axis=0)
y_score = np.concatenate([member_scores, nonmem_scores], axis=0)

# final safety filter (should be all finite now)
finite = np.isfinite(y_score)
if not np.all(finite):
    dropped = int((~finite).sum())
    print(f"⚠️ Dropping {dropped} non-finite scores at final stage (unexpected).")
    y_true = y_true[finite]
    y_score = y_score[finite]

if len(np.unique(y_true)) < 2:
    raise RuntimeError("❌ Only one class remains after filtering; cannot compute AUC/ROC.")

auc = roc_auc_score(y_true, y_score)
fpr, tpr, thresholds = roc_curve(y_true, y_score)

def tpr_at_fpr(fpr_arr, tpr_arr, thr_arr, target_fpr=0.05):
    idx = np.searchsorted(fpr_arr, target_fpr, side="right") - 1
    idx = int(np.clip(idx, 0, len(tpr_arr) - 1))
    return float(tpr_arr[idx]), float(fpr_arr[idx]), float(thr_arr[idx])

tpr_at_5, realized_fpr, thr_at_5 = tpr_at_fpr(fpr, tpr, thresholds, target_fpr=0.05)

print("\n" + "="*80)
print("📌 Loss-MIA REQUIRED METRICS (MiniGPT-v2) — NaN-safe")
print("="*80)
print(f"AUC            : {auc:.6f}")
print(f"TPR@5%FPR      : {tpr_at_5:.6f}   (realized FPR={realized_fpr:.6f}, thr={thr_at_5:.6f})")
print("="*80)

print("\nSanity stats (loss, after filtering):")
print(f"  member mean={member_losses.mean():.6f} std={member_losses.std():.6f} min={member_losses.min():.6f} max={member_losses.max():.6f}")
print(f"  nonmem mean={nonmem_losses.mean():.6f} std={nonmem_losses.std():.6f} min={nonmem_losses.min():.6f} max={nonmem_losses.max():.6f}")

loss_mia = {
    "member_losses": member_losses,
    "nonmem_losses": nonmem_losses,
    "member_scores": member_scores,
    "nonmem_scores": nonmem_scores,
    "auc": float(auc),
    "tpr_at_5fpr": float(tpr_at_5),
    "thr_at_5fpr": float(thr_at_5),
    "roc": {"fpr": fpr, "tpr": tpr, "thresholds": thresholds},
}
print("\n✅ Exported dict: loss_mia (auc + tpr_at_5fpr)")


## Entropy

In [ ]:
# ============================================================
# ✅ MiniGPT-v2 TRUE Entropy MIA (HOOK logits, NaN-safe, AMP->FP32 fallback)
#
# 你现在得到 neg-loss 的根因：model.forward(...) 没有把 logits 返回出来。
# 解决：用 forward hook 在 LLaMA 的 lm_head 处“抓 logits”，然后计算 token-level entropy。
#
# Score:
#   score = - mean_token_entropy   (entropy 越低越像 member)
#
# NaN-safe:
#   (1) 默认 AMP；若该 batch 出现 NaN/Inf，则该 batch 重新用 FP32 计算
#   (2) 仍不 finite 的样本丢弃
#
# Metrics: AUC + TPR@5%FPR
# CSV: member/nonmember scores
#
# Assumes:
#   - model (MiniGPTv2)
#   - member_loader, nonmem_loader
# ============================================================

import os, csv, warnings, traceback
import numpy as np
from tqdm import tqdm

import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings("ignore")

# -------------------------
# 0) Config
# -------------------------
OUT_DIR = "/content/minigptv2_entropy_mia_results"
os.makedirs(OUT_DIR, exist_ok=True)

MAX_PER_SPLIT = 1000
PROMPT = "Describe the image in detail."
USE_AMP_DEFAULT = True  # cuda 上默认 True
DEBUG = False           # True 会打印每次失败的真实异常

# -------------------------
# 1) Device / model state
# -------------------------
device = next(model.parameters()).device
model.eval()
print("[INFO] device:", device)
USE_AMP_DEFAULT = (device.type == "cuda")
print("[INFO] AMP default:", USE_AMP_DEFAULT)

# -------------------------
# 2) Build samples for MiniGPT-v2 forward
# -------------------------
def batch_to_samples(batch):
    images = batch["image"].to(device, non_blocking=True)
    caps = batch["caption"]
    caps = [c if isinstance(c, str) and len(c.strip()) > 0 else " " for c in caps]
    return {
        "image": images,                         # [B,3,H,W]
        "text_input": [PROMPT] * len(caps),      # list[str]
        "answer": caps,                          # list[str]
    }

# -------------------------
# 3) Find lm_head + hook to capture logits
# -------------------------
def find_lm_head_module(root: torch.nn.Module):
    """
    在 MiniGPT/LLAMA 堆里找 lm_head。
    返回 (module, qualified_name)；找不到则报错。
    """
    # 优先常见路径
    candidates = [
        "llama_model.lm_head",
        "llama_model.model.lm_head",
        "llama_model.base_model.model.lm_head",
        "model.lm_head",
        "lm_head",
    ]
    for path in candidates:
        cur = root
        ok = True
        for part in path.split("."):
            if hasattr(cur, part):
                cur = getattr(cur, part)
            else:
                ok = False
                break
        if ok and isinstance(cur, torch.nn.Module):
            return cur, path

    # 兜底：遍历 named_modules
    for name, mod in root.named_modules():
        if name.endswith("lm_head"):
            return mod, name

    raise RuntimeError("❌ Cannot find lm_head in this MiniGPTv2 model. Please inspect model structure.")

_LM_HEAD, _LM_HEAD_NAME = find_lm_head_module(model)
print("[INFO] lm_head found at:", _LM_HEAD_NAME)

# -------------------------
# 4) Forward once + capture logits via hook, then compute per-sample entropy
# -------------------------
def compute_entropy_from_logits(logits: torch.Tensor, labels: torch.Tensor = None) -> torch.Tensor:
    """
    logits: [B, T, V]
    labels(optional): [B, T] with -100 as ignore
    returns: entropy_per_sample [B] (fp32)
    """
    logits = logits.float()
    logp = torch.log_softmax(logits, dim=-1)          # [B,T,V]
    p = torch.exp(logp)                               # [B,T,V]
    ent = -(p * logp).sum(dim=-1)                     # [B,T]

    if labels is not None and torch.is_tensor(labels) and labels.ndim == 2:
        labels = labels.to(ent.device)
        # 常见：labels 中 -100 表示忽略（prompt 部分）
        mask = (labels != -100)
        if mask.any():
            ent_sum = (ent * mask).sum(dim=1)         # [B]
            cnt = mask.sum(dim=1).clamp_min(1)        # [B]
            return ent_sum / cnt
        # 如果 mask 全空：退化为全 token 平均
        return ent.mean(dim=1)

    # 没 labels：就对所有 token 平均（仍是 entropy attack，只是包含 prompt）
    return ent.mean(dim=1)

@torch.no_grad()
def forward_entropy_score(samples: dict, use_amp: bool):
    """
    returns:
      scores_np: np.ndarray [B]   (score = -entropy)
      mode: str
    """
    captured = {"logits": None}

    def _hook(mod, inp, out):
        # out: [B,T,V]
        captured["logits"] = out

    h = _LM_HEAD.register_forward_hook(_hook)
    try:
        # forward（让模型走一遍内部 teacher forcing / loss 计算）
        if use_amp and device.type == "cuda":
            with torch.amp.autocast("cuda", dtype=torch.float16):
                out = model.forward(samples, reduction="none")
        else:
            out = model.forward(samples, reduction="none")

        logits = captured["logits"]
        if logits is None:
            raise RuntimeError("❌ Hook did not capture logits. lm_head might not be used in this forward path.")

        labels = None
        if isinstance(out, dict):
            labels = out.get("labels", None)

        ent = compute_entropy_from_logits(logits, labels=labels)  # [B]
        score = -ent                                              # entropy 越低越大 => 越像 member
        return score.detach().float().cpu().numpy(), "neg-entropy(hook)"
    finally:
        h.remove()

# -------------------------
# 5) Dataloader runner (NaN-safe + AMP->FP32 fallback)
# -------------------------
def run_scores(dataloader, max_n=1000, desc=""):
    scores = []
    modes = []
    seen = 0
    n_fallback_fp32_batches = 0
    n_drop_nonfinite = 0

    for batch in tqdm(dataloader, desc=desc):
        if seen >= max_n:
            break

        bs = batch["image"].size(0)
        if seen + bs > max_n:
            keep = max_n - seen
            batch = {
                "image": batch["image"][:keep],
                "caption": batch["caption"][:keep],
            }
            bs = keep

        samples = batch_to_samples(batch)

        # 1) AMP first
        try:
            s_np, mode = forward_entropy_score(samples, use_amp=USE_AMP_DEFAULT)
        except Exception as e:
            if DEBUG:
                print("\n[DEBUG] AMP forward failed:", repr(e))
                traceback.print_exc()
            # 如果 AMP 直接报错，也改 FP32 重试
            n_fallback_fp32_batches += 1
            s_np, mode = forward_entropy_score(samples, use_amp=False)

        # 2) AMP 结果含 NaN/Inf -> FP32 重算该 batch
        if not np.isfinite(s_np).all():
            n_fallback_fp32_batches += 1
            s_np, mode = forward_entropy_score(samples, use_amp=False)

        # 3) drop non-finite
        finite_mask = np.isfinite(s_np)
        if not finite_mask.all():
            n_drop_nonfinite += int((~finite_mask).sum())
            s_np = s_np[finite_mask]

        scores.extend(list(map(float, s_np)))
        modes.append(mode)
        seen += bs

    mode_main = modes[0] if len(modes) else "unknown"
    mode_set = set(modes)
    if len(mode_set) > 1:
        print(f"[INFO] multiple scoring modes: {mode_set} | main={mode_main}")
    else:
        print(f"[INFO] scoring mode: {mode_main}")

    print(f"[INFO] fp32 fallback batches: {n_fallback_fp32_batches}")
    print(f"[INFO] dropped non-finite scores: {n_drop_nonfinite}")
    return scores, mode_main

print("\n[RUN] member entropy scores ...")
mem_scores, mem_mode = run_scores(member_loader, max_n=MAX_PER_SPLIT, desc="member")

print("\n[RUN] nonmember entropy scores ...")
non_scores, non_mode = run_scores(nonmem_loader, max_n=MAX_PER_SPLIT, desc="nonmember")

print(f"\n[INFO] member: N={len(mem_scores)} mode={mem_mode}")
print(f"[INFO] nonmem: N={len(non_scores)} mode={non_mode}")

if len(mem_scores) == 0 or len(non_scores) == 0:
    raise RuntimeError("❌ empty scores after filtering; indicates severe instability or hook failure.")

# -------------------------
# 6) Save CSV
# -------------------------
MEM_CSV = os.path.join(OUT_DIR, "member_scores_entropy.csv")
NON_CSV = os.path.join(OUT_DIR, "nonmember_scores_entropy.csv")

with open(MEM_CSV, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["score"]); w.writerows([[x] for x in mem_scores])

with open(NON_CSV, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["score"]); w.writerows([[x] for x in non_scores])

print("[CSV]", MEM_CSV, "|", NON_CSV)

# -------------------------
# 7) Metrics: AUC + TPR@5%FPR
# -------------------------
y_true  = np.array([1]*len(mem_scores) + [0]*len(non_scores), dtype=np.int32)
y_score = np.array(mem_scores + non_scores, dtype=np.float64)

finite_all = np.isfinite(y_score)
if not finite_all.all():
    bad = int((~finite_all).sum())
    print(f"[WARN] still found {bad} non-finite in y_score; filtering again.")
    y_true = y_true[finite_all]
    y_score = y_score[finite_all]

auc = roc_auc_score(y_true, y_score)
fpr, tpr, thr = roc_curve(y_true, y_score)

def tpr_at_fpr(fpr_arr, tpr_arr, target_fpr=0.05):
    idx = np.searchsorted(fpr_arr, target_fpr, side="right") - 1
    idx = int(np.clip(idx, 0, len(tpr_arr) - 1))
    return float(tpr_arr[idx]), float(fpr_arr[idx]), float(thr[idx])

tpr_5, fpr_real, thr_5 = tpr_at_fpr(fpr, tpr, 0.05)

print("\n" + "="*80)
print("✅ TRUE Entropy MIA for MiniGPT-v2 (hook logits, NaN-safe)")
print("="*80)
print(f"Score mode (member):     {mem_mode}")
print(f"Score mode (nonmember):  {non_mode}")
print(f"AUC:                     {auc:.6f}")
print(f"TPR@5%FPR:               {tpr_5:.6f}   (realized FPR={fpr_real:.6f}, thr={thr_5:.6f})")
print("="*80)

entropy_mia = {
    "mem_scores": np.array(mem_scores, dtype=np.float32),
    "non_scores": np.array(non_scores, dtype=np.float32),
    "auc": float(auc),
    "tpr_at_5fpr": float(tpr_5),
    "thr_at_5fpr": float(thr_5),
    "roc": {"fpr": fpr, "tpr": tpr, "thresholds": thr},
    "modes": {"member": mem_mode, "nonmember": non_mode},
    "csv": {"member": MEM_CSV, "nonmember": NON_CSV},
    "lm_head": _LM_HEAD_NAME,
}
print("✅ Exported dict: entropy_mia")


## Min-K

In [ ]:
# ============================================================
# ✅ MiniGPT-v2 Min-K MIA (DataLoader version, NaN-safe)
# Data:
#   - member_loader   (train split, 1000)
#   - nonmem_loader   (val split,   1000)
# Metrics:
#   - AUC
#   - TPR@5%FPR
# ============================================================

import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve

# -------------------------
# 0) Device / model
# -------------------------
device = next(model.parameters()).device
model.eval()
USE_AMP = (device.type == "cuda")

print("✅ Device:", device, "| AMP:", USE_AMP)

tok = getattr(model, "llama_tokenizer", None)
if tok is None:
    raise RuntimeError("❌ model.llama_tokenizer not found")

# -------------------------
# 1) Min-K parameters
# -------------------------
PROMPT = "Describe the image briefly."
WINDOW_TOKENS = 16
MAX_WINDOWS   = 64
K_FRAC        = 0.05      # Min-K %
MAX_PER_SPLIT = 1000

# -------------------------
# 2) Caption → token windows
# -------------------------
def split_caption_to_windows(caption: str,
                             window_tokens=16,
                             max_windows=64):
    ids = tok(caption, add_special_tokens=False).input_ids
    if not ids:
        return []
    pieces = []
    for i in range(0, len(ids), window_tokens):
        if len(pieces) >= max_windows:
            break
        sub = tok.decode(
            ids[i:i+window_tokens],
            skip_special_tokens=True
        ).strip()
        if sub:
            pieces.append(sub)
    return pieces if pieces else [caption]

# -------------------------
# 3) One-window loss (NaN-safe)
# -------------------------
@torch.no_grad()
def window_loss(image_tensor, answer_text: str):
    """
    image_tensor: [1, 3, H, W]
    """
    samples = {
        "image": image_tensor,
        "text_input": [PROMPT],
        "answer": [answer_text],
    }

    # AMP first
    try:
        if USE_AMP:
            with torch.cuda.amp.autocast(dtype=torch.float16):
                out = model.forward(samples, reduction="none")
        else:
            out = model.forward(samples, reduction="none")
    except Exception:
        return None

    loss = out.get("loss", None)
    if loss is None:
        return None

    val = float(loss.detach().float().cpu().item())
    if np.isfinite(val):
        return val

    # FP32 fallback
    try:
        out = model.forward(samples, reduction="none")
        val = float(out["loss"].detach().float().cpu().item())
        return val if np.isfinite(val) else None
    except Exception:
        return None

# -------------------------
# 4) Min-K score for ONE sample
# -------------------------
def mink_score_single(image_tensor, caption: str):
    pieces = split_caption_to_windows(
        caption,
        window_tokens=WINDOW_TOKENS,
        max_windows=MAX_WINDOWS
    )

    losses = []
    for p in pieces:
        l = window_loss(image_tensor, p)
        if l is not None:
            losses.append(l)

    if len(losses) == 0:
        return None  # 丢弃该样本

    losses = np.asarray(losses)
    k = max(1, int(np.ceil(len(losses) * K_FRAC)))
    idx = np.argpartition(losses, k-1)[:k]
    score = -losses[idx].mean()

    return float(score) if np.isfinite(score) else None

# -------------------------
# 5) Run over DataLoader
# -------------------------
def run_mink_loader(dataloader, desc=""):
    scores = []
    dropped = 0
    seen = 0

    for batch in tqdm(dataloader, desc=desc):
        images = batch["image"].to(device, non_blocking=True)
        captions = batch["caption"]

        for i in range(images.size(0)):
            if seen >= MAX_PER_SPLIT:
                break

            cap = captions[i]
            if not isinstance(cap, str) or len(cap.strip()) == 0:
                dropped += 1
                continue

            img = images[i:i+1]  # keep batch dim
            s = mink_score_single(img, cap)

            if s is None:
                dropped += 1
            else:
                scores.append(s)

            seen += 1

        if seen >= MAX_PER_SPLIT:
            break

    print(f"[INFO] {desc}: kept={len(scores)} dropped={dropped}")
    return scores

# -------------------------
# 6) Run MIA
# -------------------------
print("\n[RUN] Min-K on MEMBER ...")
mem_scores = run_mink_loader(member_loader, desc="member")

print("\n[RUN] Min-K on NON-MEMBER ...")
non_scores = run_mink_loader(nonmem_loader, desc="nonmember")

if len(mem_scores) == 0 or len(non_scores) == 0:
    raise RuntimeError("❌ No valid Min-K scores")

# -------------------------
# 7) Metrics (AUC + TPR@5%FPR)
# -------------------------
y_true = np.array(
    [1]*len(mem_scores) + [0]*len(non_scores),
    dtype=np.int32
)
y_score = np.array(mem_scores + non_scores, dtype=np.float64)

auc = roc_auc_score(y_true, y_score)
fpr, tpr, _ = roc_curve(y_true, y_score)

def tpr_at_fpr(fpr_arr, tpr_arr, target=0.05):
    idx = np.searchsorted(fpr_arr, target, side="right") - 1
    idx = int(np.clip(idx, 0, len(tpr_arr)-1))
    return float(tpr_arr[idx]), float(fpr_arr[idx])

tpr5, fpr_real = tpr_at_fpr(fpr, tpr, 0.05)

print("\n" + "="*80)
print("📌 MiniGPT-v2 Min-K MIA (COCO 2014 streaming)")
print("="*80)
print(f"AUC           : {auc:.6f}")
print(f"TPR@5%FPR     : {tpr5:.6f}   (real FPR={fpr_real:.6f})")
print("="*80)

# export
mink_mia = {
    "member_scores": np.array(mem_scores, dtype=np.float32),
    "nonmember_scores": np.array(non_scores, dtype=np.float32),
    "auc": float(auc),
    "tpr_at_5fpr": float(tpr5),
}
print("✅ Exported dict: mink_mia")


## Min-k++

In [ ]:
# ============================================================
# Min-K++ MIA for MiniGPT-v2 (完整适配版)
# 核心方法：
#   ✅ 窗口化 caption → 独立前向 → NLL 列表
#   ✅ Z-norm + 多 K 值 (0.01, 0.02, 0.05, 0.10)
#   ✅ Max 聚合作为最终分数
#   ✅ 黑盒方法（无需梯度）
#   ✅ 评估指标：AUC + TPR/Precision/Recall/F1/Accuracy @5%FPR
#
# 假设已存在：
#   - model           (MiniGPT-v2)
#   - member_loader   (DataLoader)
#   - nonmem_loader   (DataLoader)
# ============================================================

import os, gc, random, csv
import numpy as np
from typing import List, Optional, Dict
import warnings
warnings.filterwarnings("ignore")

import torch
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)

# ============================================================
# 0. 基础配置
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert "model" in globals(), "❌ 未发现全局 `model`。"
assert "member_loader" in globals(), "❌ 未发现 `member_loader`。"
assert "nonmem_loader" in globals(), "❌ 未发现 `nonmem_loader`。"

device = next(model.parameters()).device
model.eval()

print(f"✅ Device: {device}")
print(f"✅ 模型类型: {type(model).__name__}")

# ============================================================
# 1. 获取 tokenizer
# ============================================================
# MiniGPT-v2 的 tokenizer 通常在 model.llama_tokenizer 或 model.tokenizer
tokenizer = None

# 尝试多种可能的属性名
for attr_name in ['llama_tokenizer', 'tokenizer', 'text_tokenizer', 'lm_tokenizer']:
    if hasattr(model, attr_name):
        tokenizer = getattr(model, attr_name)
        print(f"✅ 找到 tokenizer: model.{attr_name}")
        break

if tokenizer is None:
    raise RuntimeError("❌ 无法找到 tokenizer，Min-K++ 需要 tokenizer 进行窗口切分。")

# ============================================================
# 2. 超参数配置
# ============================================================
WINDOW_TOKENS = 16      # 每个窗口的 token 数
MAX_WINDOWS = 64        # 最多处理的窗口数
K_LIST = (0.01, 0.02, 0.05, 0.10)  # 多个 K 值
MAX_PER_SPLIT = 1000    # 每个数据集最多处理样本数

print(f"\n📋 Min-K++ 配置:")
print(f"  WINDOW_TOKENS: {WINDOW_TOKENS}")
print(f"  MAX_WINDOWS: {MAX_WINDOWS}")
print(f"  K_LIST: {K_LIST}")
print(f"  MAX_PER_SPLIT: {MAX_PER_SPLIT}")

# ============================================================
# 3. MiniGPT-v2 输入构造
# ============================================================
PROMPT = "Describe the image in detail."

def _make_samples(image_tensor, caption: str):
    """构造 MiniGPT-v2 的输入格式"""
    cap = caption if isinstance(caption, str) and caption.strip() else " "
    return {
        "image": image_tensor,      # [1, 3, H, W]
        "text_input": [PROMPT],     # 问题/指令
        "answer": [cap],            # 答案
    }

def _forward_loss(samples):
    """获取 loss"""
    try:
        out = model.forward(samples, reduction="mean")
    except TypeError:
        out = model(samples, reduction="mean")

    if isinstance(out, dict) and "loss" in out:
        return out["loss"]
    if hasattr(out, "loss"):
        return out.loss
    if torch.is_tensor(out):
        return out
    raise RuntimeError("❌ Cannot find loss in model output.")

# ============================================================
# 4. 窗口切分函数
# ============================================================
def split_caption_to_windows(
    caption: str,
    window_tokens: int = WINDOW_TOKENS,
    max_windows: int = MAX_WINDOWS
) -> List[str]:
    """
    将 caption 切分为固定 token 长度的窗口

    返回: List[str]，每个元素是一个窗口的文本
    """
    # Tokenize
    try:
        ids = tokenizer(caption, add_special_tokens=False).input_ids
    except:
        # 备用方案：如果 tokenizer 调用失败，按字符切分
        words = caption.split()
        return [' '.join(words[i:i+window_tokens])
                for i in range(0, len(words), window_tokens)][:max_windows]

    if not ids:
        return []

    # 按窗口切分
    pieces = []
    for i in range(0, len(ids), window_tokens):
        if len(pieces) >= max_windows:
            break

        sub_ids = ids[i:i+window_tokens]
        try:
            sub_txt = tokenizer.decode(sub_ids, skip_special_tokens=True).strip()
        except:
            continue

        if sub_txt:
            pieces.append(sub_txt)

    # 如果切分失败，返回完整 caption
    if not pieces:
        pieces = [caption]

    return pieces

# ============================================================
# 5. 窗口 NLL 计算
# ============================================================
@torch.inference_mode()
def window_nll_list(
    image_tensor,
    caption: str,
    window_tokens: int = WINDOW_TOKENS,
    max_windows: int = MAX_WINDOWS
) -> List[float]:
    """
    对每个窗口独立前向，获取 loss 作为 NLL proxy

    返回: List[float]，每个窗口的 loss（过滤 NaN/Inf）
    """
    losses = []

    windows = split_caption_to_windows(caption, window_tokens, max_windows)

    for window_text in windows:
        try:
            # 构造输入
            samples = _make_samples(image_tensor, window_text)

            # 前向获取 loss
            loss = _forward_loss(samples)

            if torch.is_tensor(loss):
                loss_val = float(loss.item())
            else:
                loss_val = float(loss)

            # 过滤 NaN/Inf
            if np.isfinite(loss_val):
                losses.append(loss_val)

        except Exception as e:
            # 静默跳过失败的窗口
            continue

    return losses

# ============================================================
# 6. Min-K++ 核心算法
# ============================================================
def min_kpp_from_losses(
    losses: List[float],
    k_list: tuple = K_LIST
) -> Optional[float]:
    """
    从 loss 列表计算 Min-K++ 分数

    步骤:
    1. Z-normalize losses
    2. 对每个 K，取 top-K 最小的 z-score
    3. 返回所有 K 的最大值

    返回: float 或 None（如果无有效 loss）
    """
    # 二次过滤（保险）
    vals = [v for v in losses if np.isfinite(v)]
    if not vals:
        return None

    # 转 tensor
    x = torch.tensor(vals, dtype=torch.float32)

    # Z-normalize
    mean = x.mean()
    std = x.std()
    z = (x - mean) / (std + 1e-6)

    # 对每个 K 计算分数
    scores = []
    for frac in k_list:
        k = max(1, int(np.ceil(len(z) * frac)))

        # top-K 最小的 z-score（即 -z 最大）
        topk_vals, _ = torch.topk(-z, k)
        scores.append(topk_vals.mean())

    # Max 聚合
    final_score = torch.stack(scores).max()

    return float(final_score.item())

# ============================================================
# 7. 单样本评分函数
# ============================================================
@torch.inference_mode()
def min_kpp_score(image_tensor, caption: str) -> Optional[float]:
    """
    对单个样本计算 Min-K++ 分数

    返回: float 或 None
    """
    # 计算所有窗口的 NLL
    losses = window_nll_list(image_tensor, caption)

    # Min-K++ 算法
    return min_kpp_from_losses(losses, k_list=K_LIST)

# ============================================================
# 8. 批量评分函数
# ============================================================
def run_split_scores(dataloader, max_samples: int = MAX_PER_SPLIT, tag: str = "") -> List[float]:
    """
    对 dataloader 中的样本进行 Min-K++ 评分

    返回: List[float]，有效分数列表
    """
    scores = []
    seen = 0

    for batch in tqdm(dataloader, desc=f"Min-K++ {tag}"):
        imgs = batch["image"].to(device, non_blocking=True)
        caps = batch["caption"]
        bsz = imgs.size(0)

        for i in range(bsz):
            if seen >= max_samples:
                break

            img_1 = imgs[i:i+1]  # [1, 3, H, W]
            cap_1 = caps[i]

            # 计算分数
            score = min_kpp_score(img_1, cap_1)

            if score is not None and np.isfinite(score):
                scores.append(score)

            seen += 1

            # 清理
            del img_1
            if seen % 10 == 0:
                gc.collect()
                torch.cuda.empty_cache()

        del imgs
        if seen >= max_samples:
            break

    return scores

# ============================================================
# 9. 评估指标计算
# ============================================================
def compute_metrics(mem_scores: List[float], non_scores: List[float]) -> Dict:
    """
    计算完整的评估指标
    """
    # 全局过滤 NaN/Inf
    mem_scores = [v for v in mem_scores if np.isfinite(v)]
    non_scores = [v for v in non_scores if np.isfinite(v)]

    if len(mem_scores) == 0 or len(non_scores) == 0:
        raise RuntimeError("❌ 过滤后无有效分数")

    # 构造标签和分数
    y_true = np.array([1]*len(mem_scores) + [0]*len(non_scores))
    y_score = np.array(mem_scores + non_scores, dtype=float)

    results = {}

    # AUC
    try:
        auc = roc_auc_score(y_true, y_score)
        results["auc"] = auc
    except:
        results["auc"] = 0.5

    # ROC curve
    fpr, tpr, thresholds = roc_curve(y_true, y_score)

    # 最佳阈值 (Youden's J)
    if len(thresholds) > 0:
        j_idx = np.argmax(tpr - fpr)
        best_thr = float(thresholds[j_idx])
        results["best_threshold"] = best_thr

        y_pred = (y_score >= best_thr).astype(int)
        results["accuracy"] = accuracy_score(y_true, y_pred)
        results["confusion_matrix"] = confusion_matrix(y_true, y_pred).tolist()
    else:
        results["best_threshold"] = float("inf")
        results["accuracy"] = 0.5
        results["confusion_matrix"] = [[len(non_scores), 0], [len(mem_scores), 0]]

    # @5%FPR 指标
    non_np = np.array(non_scores, dtype=float)
    if len(non_np) > 0:
        thr_5 = float(np.quantile(non_np, 0.95))
        y_pred_5 = (y_score >= thr_5).astype(int)

        cm_5 = confusion_matrix(y_true, y_pred_5)
        if cm_5.size == 4:
            tn, fp, fn, tp = cm_5.ravel()
            fpr_achieved = fp / max(1, (fp + tn))
            tpr_achieved = tp / max(1, (tp + fn))
        else:
            fpr_achieved = float("nan")
            tpr_achieved = float("nan")

        prec, rec, f1, _ = precision_recall_fscore_support(
            y_true, y_pred_5, average="binary", zero_division=0
        )
        acc_5 = accuracy_score(y_true, y_pred_5)

        results["threshold_5fpr"] = thr_5
        results["actual_fpr"] = fpr_achieved
        results["tpr_at_5fpr"] = tpr_achieved
        results["precision_at_5fpr"] = prec
        results["recall_at_5fpr"] = rec
        results["f1_at_5fpr"] = f1
        results["accuracy_at_5fpr"] = acc_5
        results["confusion_matrix_5fpr"] = cm_5.tolist()

    return results

# ============================================================
# 10. 主流程
# ============================================================
print("\n" + "="*80)
print("🚀 Min-K++ MIA for MiniGPT-v2")
print("="*80)

# 运行评分
print("\n⏳ 对成员样本评分 (Min-K++) ...")
mem_scores = run_split_scores(member_loader, max_samples=MAX_PER_SPLIT, tag="Member")

print("\n⏳ 对非成员样本评分 (Min-K++) ...")
non_scores = run_split_scores(nonmem_loader, max_samples=MAX_PER_SPLIT, tag="NonMember")

print(f"\n✅ 有效样本数: member={len(mem_scores)} | nonmember={len(non_scores)}")

# 计算指标
print("\n📊 计算评估指标 ...")
results = compute_metrics(mem_scores, non_scores)

# ============================================================
# 11. 结果展示
# ============================================================
print("\n" + "="*80)
print("📊 Min-K++ MIA 结果")
print("="*80)
print(f"AUC:                     {results['auc']:.6f}")
print(f"最佳阈值 (Youden J):      {results['best_threshold']:.6f}")
print(f"准确率 @ best_thr:       {results['accuracy']:.6f}")
print("\nConfusion Matrix @ best_thr [[TN FP][FN TP]]:")
print(np.array(results['confusion_matrix']))

if 'tpr_at_5fpr' in results:
    print("\n" + "-"*80)
    print("📊 Metrics @ 5% FPR (target)")
    print("-"*80)
    print(f"阈值 (95th percentile):  {results['threshold_5fpr']:.6f}")
    print(f"实际 FPR:                {results['actual_fpr']*100:.2f}%")
    print(f"TPR (Recall):           {results['tpr_at_5fpr']*100:.2f}%")
    print(f"Precision:              {results['precision_at_5fpr']*100:.2f}%")
    print(f"F1:                     {results['f1_at_5fpr']:.6f}")
    print(f"Accuracy:               {results['accuracy_at_5fpr']*100:.2f}%")
    print("\nConfusion Matrix @ 5%FPR:")
    print(np.array(results['confusion_matrix_5fpr']))

# ============================================================
# 12. 保存结果
# ============================================================
OUT_DIR = "/content/minigptv2_minkpp_results"
os.makedirs(OUT_DIR, exist_ok=True)

# 保存分数 CSV
mem_csv = os.path.join(OUT_DIR, "member_scores_minkpp.csv")
non_csv = os.path.join(OUT_DIR, "nonmember_scores_minkpp.csv")

with open(mem_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["score"])
    w.writerows([[s] for s in mem_scores])

with open(non_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["score"])
    w.writerows([[s] for s in non_scores])

# 保存完整结果
import json
results_json = {
    "config": {
        "model": type(model).__name__,
        "WINDOW_TOKENS": WINDOW_TOKENS,
        "MAX_WINDOWS": MAX_WINDOWS,
        "K_LIST": K_LIST,
        "MAX_PER_SPLIT": MAX_PER_SPLIT,
        "member_samples": len(mem_scores),
        "nonmember_samples": len(non_scores),
    },
    "results": results
}

with open(os.path.join(OUT_DIR, "minkpp_results.json"), "w") as f:
    json.dump(results_json, f, indent=2)

print(f"\n💾 结果已保存:")
print(f"   {mem_csv}")
print(f"   {non_csv}")
print(f"   {OUT_DIR}/minkpp_results.json")

print("\n✅ Min-K++ MIA 完成!")
print("="*80)

## ModRényi*

In [ ]:
# ============================================================
# ModRényi* (Fused) MIA for MiniGPT-v2 (完整适配版)
# 核心方法：
#   ✅ 图像视图 + 文本视图的轻扰动
#   ✅ 优先提取 logits/labels 计算 Rényi 熵
#   ✅ 自动回退到 loss 作为负熵近似
#   ✅ Fused 融合策略：trimmed-mean + z-norm
#   ✅ 评估指标：AUC + TPR/Precision/Recall/F1 @5%FPR
#
# 假设已存在：
#   - model           (MiniGPT-v2)
#   - member_loader   (DataLoader)
#   - nonmem_loader   (DataLoader)
# ============================================================

import os, gc, math, random, re, csv
import numpy as np
from typing import List, Optional, Dict, Any
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from torchvision.transforms import (
    RandomResizedCrop, RandomRotation,
    RandomAffine, ColorJitter
)

from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score,
    confusion_matrix, precision_recall_fscore_support
)

# ============================================================
# 0. 基础配置
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert "model" in globals(), "❌ 未发现全局 `model`。"
assert "member_loader" in globals(), "❌ 未发现 `member_loader`。"
assert "nonmem_loader" in globals(), "❌ 未发现 `nonmem_loader`。"

device = next(model.parameters()).device
model.eval()

print(f"✅ Device: {device}")
print(f"✅ 模型类型: {type(model).__name__}")

# ============================================================
# 1. 获取 tokenizer（用于文本扰动）
# ============================================================
tokenizer = None
for attr_name in ['llama_tokenizer', 'tokenizer', 'text_tokenizer', 'lm_tokenizer']:
    if hasattr(model, attr_name):
        tokenizer = getattr(model, attr_name)
        print(f"✅ 找到 tokenizer: model.{attr_name}")
        break

if tokenizer is None:
    print("⚠️ 未找到 tokenizer，文本视图将受限")

# ============================================================
# 2. 超参数配置
# ============================================================
IMG_VIEWS = 5           # 图像视图数量
TXT_VIEWS = 3           # 文本视图数量
ALPHAS = (0.5, 2.0)     # Rényi 熵的 alpha 参数
LENGTH_NORM = True      # 是否做长度归一化
MAX_PER_SPLIT = 1000    # 每个数据集最多处理样本数

print(f"\n📋 ModRényi* 配置:")
print(f"  图像视图数: {IMG_VIEWS}")
print(f"  文本视图数: {TXT_VIEWS}")
print(f"  Rényi alphas: {ALPHAS}")
print(f"  长度归一化: {LENGTH_NORM}")
print(f"  MAX_PER_SPLIT: {MAX_PER_SPLIT}")

# ============================================================
# 3. 图像预处理基础
# ============================================================
IMG_SIZE = 224

img_base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.48145466, 0.4578275, 0.40821073),
        std=(0.26862954, 0.26130258, 0.27577711)
    ),
])

# ============================================================
# 4. 视图生成函数
# ============================================================
def make_image_views(pil_img: Image.Image, n_views: int = IMG_VIEWS) -> List[Image.Image]:
    """
    生成图像的轻扰动视图

    返回: List[PIL.Image]
    """
    views = [pil_img]  # 原始图像

    # 视图 1: Random Resized Crop
    if n_views > 1:
        views.append(
            RandomResizedCrop(
                size=(256, 256),
                scale=(0.8, 1.0),
                interpolation=InterpolationMode.BICUBIC
            )(pil_img)
        )

    # 视图 2: Random Rotation
    if n_views > 2:
        views.append(
            RandomRotation(
                degrees=30,
                interpolation=InterpolationMode.BICUBIC,
                expand=False
            )(pil_img)
        )

    # 视图 3: Random Affine
    if n_views > 3:
        views.append(
            RandomAffine(
                degrees=20,
                translate=(0.08, 0.08),
                scale=(0.9, 1.1),
                interpolation=InterpolationMode.BICUBIC
            )(pil_img)
        )

    # 视图 4: Color Jitter
    if n_views > 4:
        views.append(
            ColorJitter(
                brightness=0.3,
                contrast=0.3,
                saturation=0.25,
                hue=0.05
            )(pil_img)
        )

    return views[:max(1, n_views)]

def normalize_spaces(s: str) -> str:
    """标准化空白字符"""
    return re.sub(r"\s+", " ", s).strip()

def make_text_views(caption: str, n_views: int = TXT_VIEWS) -> List[str]:
    """
    生成文本的轻扰动视图

    返回: List[str]
    """
    base = str(caption or "").strip()

    candidates = [
        base,                    # 原始
        base.lower(),           # 小写
        normalize_spaces(base), # 空白标准化
    ]

    # 去重
    out, seen = [], set()
    for c in candidates:
        if c and c not in seen:
            out.append(c)
            seen.add(c)

    return out[:max(1, n_views)]

# ============================================================
# 5. MiniGPT-v2 输入构造
# ============================================================
PROMPT = "Describe the image in detail."

def _make_samples(image_tensor, caption: str):
    """构造 MiniGPT-v2 的输入格式"""
    cap = caption if isinstance(caption, str) and caption.strip() else " "
    return {
        "image": image_tensor,
        "text_input": [PROMPT],
        "answer": [cap],
    }

# ============================================================
# 6. 前向推理：优先获取 logits/labels，回退到 loss
# ============================================================
@torch.inference_mode()
def forward_any(image_tensor, caption: str):
    """
    尝试获取 logits/labels 或 loss

    返回: (logits, labels, loss, mode)
        - mode = "logits": 成功获取 logits 和 labels
        - mode = "loss": 只获取到 loss
        - mode = "none": 都失败
    """
    samples = _make_samples(image_tensor, caption)

    try:
        out = model.forward(samples)
    except:
        try:
            out = model(samples)
        except:
            return None, None, None, "none"

    logits = labels = loss = None

    # 尝试从输出中提取
    if isinstance(out, dict):
        logits = out.get("logits", None)
        labels = out.get("labels", None)
        loss = out.get("loss", None)
    else:
        logits = getattr(out, "logits", None)
        labels = getattr(out, "labels", None)
        loss = getattr(out, "loss", None)

    # 判断模式
    if logits is not None and labels is not None:
        return logits, labels, loss, "logits"

    if loss is not None:
        # 处理 loss 可能是 tuple 的情况
        if not torch.is_tensor(loss):
            try:
                loss = loss[0]
            except:
                pass
        return None, None, loss, "loss"

    return None, None, None, "none"

# ============================================================
# 7. 从 logits/labels 提取 assistant 段的 log-probs
# ============================================================
def assistant_logp_from_logits(
    logits: torch.Tensor,
    labels: torch.Tensor
) -> Optional[np.ndarray]:
    """
    提取 assistant 段的 log-probabilities

    输入:
        logits: [1, seq_len, vocab_size]
        labels: [1, seq_len]  (-100 表示非 assistant 位置)

    返回:
        np.ndarray [T, V]，其中 T 是 assistant 段长度
    """
    if logits.ndim != 3 or labels.ndim != 2:
        return None

    # Shift for next-token prediction
    shift_logits = logits[..., :-1, :]  # [1, L-1, V]
    shift_labels = labels[..., 1:]      # [1, L-1]

    # 找到 assistant 段（labels != -100）
    mask = shift_labels.ne(-100)[0]     # [L-1]

    if mask.sum().item() == 0:
        return None

    # 计算 log-softmax
    logp = F.log_softmax(shift_logits[0].float(), dim=-1)  # [L-1, V]

    # 提取 assistant 段
    sel = logp[mask]  # [T, V]

    if sel.numel() == 0:
        return None

    return sel.detach().cpu().numpy()

# ============================================================
# 8. Rényi 熵计算
# ============================================================
def renyi_entropy_from_logp(logp_tok: np.ndarray, alpha: float) -> float:
    """
    单个 token 位置的 Rényi 熵

    输入:
        logp_tok: [V]，log-probabilities
        alpha: Rényi 参数

    返回: float
    """
    a = alpha * logp_tok
    m = np.max(a)
    return float((1.0 / (1.0 - alpha)) * (m + np.log(np.exp(a - m).sum())))

def renyi_seq_from_logp(logp_seq: np.ndarray, alpha: float) -> float:
    """
    序列的平均 Rényi 熵

    输入:
        logp_seq: [T, V]
        alpha: Rényi 参数

    返回: float
    """
    if logp_seq.shape[0] == 0:
        return float("nan")

    vals = [
        renyi_entropy_from_logp(logp_seq[t], alpha)
        for t in range(logp_seq.shape[0])
    ]

    return float(np.mean(vals)) if vals else float("nan")

# ============================================================
# 9. 稳健聚合函数
# ============================================================
def robust_aggregate(values: List[float], trim: float = 0.1) -> float:
    """
    Trimmed mean：去除两端各 trim% 的极值后取平均

    输入:
        values: List[float]
        trim: 修剪比例（0.1 = 去除两端各 10%）

    返回: float
    """
    arr = np.array([v for v in values if np.isfinite(v)], dtype=float)

    if arr.size == 0:
        return float("nan")

    arr.sort()
    k = int(math.floor(trim * arr.size))

    if k * 2 < arr.size:
        arr = arr[k: arr.size - k]

    return float(np.mean(arr))

# ============================================================
# 10. Fused ModRényi* 核心算法
# ============================================================
def fused_modrenyi_from_logp_views(
    image_views_logp: List[Optional[np.ndarray]],
    text_views_logp: List[Optional[np.ndarray]],
    alphas: tuple = ALPHAS,
    length_norm: bool = LENGTH_NORM
) -> float:
    """
    从图像和文本视图的 log-probs 计算融合 ModRényi* 分数

    步骤:
    1. 对每组视图（图像/文本），计算每个 alpha 的 Rényi 熵
    2. 用 trimmed-mean 稳健聚合同组视图
    3. 长度归一化（可选）
    4. 负熵（-H）作为成员性分数
    5. Z-norm 融合所有分数

    返回: float（融合后的分数）
    """
    parts = []

    for views in [image_views_logp, text_views_logp]:
        # 过滤有效视图
        valid_views = [x for x in views if x is not None]

        if not valid_views:
            # 该组视图全无效，填充 NaN
            parts.extend([float("nan")] * len(alphas))
            continue

        # 获取序列长度（用于归一化）
        T = valid_views[0].shape[0]

        # 对每个 alpha
        for alpha in alphas:
            per_view_scores = []

            for logp in valid_views:
                if logp is None or not np.isfinite(logp).all():
                    continue

                # 计算该视图的 Rényi 熵
                H = renyi_seq_from_logp(logp, alpha)
                per_view_scores.append(H)

            if not per_view_scores:
                parts.append(float("nan"))
                continue

            # Trimmed-mean 聚合
            H_bar = robust_aggregate(per_view_scores, trim=0.1)

            # 长度归一化
            if length_norm and T > 0:
                H_bar = H_bar / math.sqrt(T)

            # 负熵：越大越像成员
            parts.append(-H_bar)

    # Z-norm 融合
    valid = np.array([x for x in parts if np.isfinite(x)], dtype=float)

    if valid.size == 0:
        return float("nan")

    if valid.size >= 2 and np.std(valid) > 1e-12:
        # Z-normalize
        z = (valid - np.mean(valid)) / np.std(valid)
        return float(np.mean(z))
    else:
        return float(np.mean(valid))

# ============================================================
# 11. 回退方案：基于 loss 的融合
# ============================================================
def fused_from_loss_views(
    image_view_losses: List[Optional[float]],
    text_view_losses: List[Optional[float]]
) -> float:
    """
    当无法获取 logits 时，使用 -loss 作为负熵近似

    返回: float（融合后的分数）
    """
    parts = []

    for views in [image_view_losses, text_view_losses]:
        valid_views = [v for v in views if (v is not None and np.isfinite(v))]

        if not valid_views:
            # 填充两个 NaN（对应两个 alpha）
            parts.append(float("nan"))
            parts.append(float("nan"))
            continue

        # -loss 作为负熵代理
        s = robust_aggregate([-float(v) for v in valid_views], trim=0.1)

        # 重复两次（保持与 alphas 数量一致）
        parts.append(s)
        parts.append(s)

    # Z-norm 融合
    valid = np.array([x for x in parts if np.isfinite(x)], dtype=float)

    if valid.size == 0:
        return float("nan")

    if valid.size >= 2 and np.std(valid) > 1e-12:
        z = (valid - np.mean(valid)) / np.std(valid)
        return float(np.mean(z))
    else:
        return float(np.mean(valid))

# ============================================================
# 12. 图像/文本视图评分
# ============================================================
def image_views_scores(image_tensor, caption: str, n_views: int = IMG_VIEWS):
    """
    对图像视图进行评分

    返回: (logp_list, loss_list, has_logits)
    """
    # 先转换为 PIL（用于扰动）
    # 从 tensor 恢复 PIL（反归一化）
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)

    img_denorm = image_tensor[0].cpu() * std + mean
    img_denorm = torch.clamp(img_denorm, 0, 1)

    pil_img = transforms.ToPILImage()(img_denorm)

    # 生成视图
    views = make_image_views(pil_img, n_views)

    logp_list = []
    loss_list = []
    has_logits = False

    for view_pil in views:
        # 转回 tensor
        view_tensor = img_base_transform(view_pil).unsqueeze(0).to(device)

        # 前向
        logits, labels, loss, mode = forward_any(view_tensor, caption)

        if mode == "logits":
            has_logits = True
            logp_list.append(assistant_logp_from_logits(logits, labels))
            loss_list.append(None)
        elif mode == "loss":
            logp_list.append(None)
            if torch.is_tensor(loss):
                loss_list.append(float(loss.detach().cpu().item()))
            else:
                loss_list.append(float(loss))
        else:
            logp_list.append(None)
            loss_list.append(None)

    return logp_list, loss_list, has_logits

def text_views_scores(image_tensor, caption: str, n_views: int = TXT_VIEWS):
    """
    对文本视图进行评分

    返回: (logp_list, loss_list, has_logits)
    """
    views = make_text_views(caption, n_views)

    logp_list = []
    loss_list = []
    has_logits = False

    for text_view in views:
        logits, labels, loss, mode = forward_any(image_tensor, text_view)

        if mode == "logits":
            has_logits = True
            logp_list.append(assistant_logp_from_logits(logits, labels))
            loss_list.append(None)
        elif mode == "loss":
            logp_list.append(None)
            if torch.is_tensor(loss):
                loss_list.append(float(loss.detach().cpu().item()))
            else:
                loss_list.append(float(loss))
        else:
            logp_list.append(None)
            loss_list.append(None)

    return logp_list, loss_list, has_logits

# ============================================================
# 13. 单样本融合评分
# ============================================================
def fused_score_for_sample(
    image_tensor,
    caption: str,
    img_views: int = IMG_VIEWS,
    txt_views: int = TXT_VIEWS
) -> Optional[float]:
    """
    对单个样本计算融合 ModRényi* 分数

    返回: float 或 None
    """
    if not caption or not isinstance(caption, str):
        return None

    # 图像视图
    img_logp, img_loss, img_has_log = image_views_scores(
        image_tensor, caption, img_views
    )

    # 文本视图
    txt_logp, txt_loss, txt_has_log = text_views_scores(
        image_tensor, caption, txt_views
    )

    # 优先使用 logits
    if img_has_log or txt_has_log:
        score = fused_modrenyi_from_logp_views(
            img_logp, txt_logp,
            alphas=ALPHAS,
            length_norm=LENGTH_NORM
        )
        if np.isfinite(score):
            return float(score)

    # 回退：使用 loss
    score_fb = fused_from_loss_views(img_loss, txt_loss)
    if np.isfinite(score_fb):
        return float(score_fb)

    return None

# ============================================================
# 14. 批量评分
# ============================================================
def run_split_scores(
    dataloader,
    max_samples: int = MAX_PER_SPLIT,
    tag: str = ""
) -> List[float]:
    """
    对 dataloader 中的样本进行 ModRényi* 评分

    返回: List[float]
    """
    scores = []
    seen = 0

    for batch in tqdm(dataloader, desc=f"ModRényi* {tag}"):
        imgs = batch["image"].to(device, non_blocking=True)
        caps = batch["caption"]
        bsz = imgs.size(0)

        for i in range(bsz):
            if seen >= max_samples:
                break

            img_1 = imgs[i:i+1]
            cap_1 = caps[i]

            try:
                score = fused_score_for_sample(img_1, cap_1)
                if score is not None and np.isfinite(score):
                    scores.append(score)
            except Exception as e:
                continue

            seen += 1

            # 清理
            del img_1
            if seen % 10 == 0:
                gc.collect()
                torch.cuda.empty_cache()

        del imgs
        if seen >= max_samples:
            break

    return scores

# ============================================================
# 15. 评估指标 @5%FPR
# ============================================================
def metrics_at_target_fpr(
    y_true: np.ndarray,
    scores: np.ndarray,
    target_fpr: float = 0.05
) -> Dict[str, float]:
    """
    计算 @5%FPR 的各项指标

    返回: Dict
    """
    fpr, tpr, thr = roc_curve(y_true, scores)

    # 插值找到 @5%FPR 的阈值
    idx = np.searchsorted(fpr, target_fpr, side="right")

    if idx == 0:
        thr_star = thr[0]
        tpr_star = tpr[0]
    elif idx >= len(thr):
        thr_star = thr[-1]
        tpr_star = tpr[-1]
    else:
        x0, x1 = fpr[idx-1], fpr[idx]
        y0, y1 = tpr[idx-1], tpr[idx]
        t0, t1 = thr[idx-1], thr[idx]
        w = (target_fpr - x0) / (x1 - x0 + 1e-12)
        tpr_star = y0 + w * (y1 - y0)
        thr_star = t0 + w * (t1 - t0)

    # 计算其他指标
    y_pred = (scores >= thr_star).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    return {
        "ROC-AUC": float(roc_auc_score(y_true, scores)),
        "thr@5%FPR": float(thr_star),
        "TPR@5%FPR": float(tpr_star),
        "Precision": float(prec),
        "Recall": float(rec),
        "F1": float(f1),
    }

# ============================================================
# 16. 主流程
# ============================================================
print("\n" + "="*80)
print("🚀 ModRényi* (Fused) MIA for MiniGPT-v2")
print("="*80)

# 运行评分
print("\n⏳ 对成员样本评分 ...")
mem_scores = run_split_scores(member_loader, max_samples=MAX_PER_SPLIT, tag="Member")

print("\n⏳ 对非成员样本评分 ...")
non_scores = run_split_scores(nonmem_loader, max_samples=MAX_PER_SPLIT, tag="NonMember")

print(f"\n✅ 有效样本数: member={len(mem_scores)} | nonmember={len(non_scores)}")

if len(mem_scores) == 0 or len(non_scores) == 0:
    raise RuntimeError("❌ 有效样本为 0，请检查模型输出")

# 计算指标
y_true = np.array([1]*len(mem_scores) + [0]*len(non_scores), dtype=int)
scores_all = np.array(mem_scores + non_scores, dtype=float)

metrics = metrics_at_target_fpr(y_true, scores_all, target_fpr=0.05)

# ============================================================
# 17. 结果展示
# ============================================================
print("\n" + "="*80)
print("📊 ModRényi* (Fused) MIA 结果")
print("="*80)
print(f"AUC:           {metrics['ROC-AUC']:.4f}")
print(f"TPR @5% FPR:   {metrics['TPR@5%FPR']:.4f}")
print(f"Precision:     {metrics['Precision']:.4f}")
print(f"Recall:        {metrics['Recall']:.4f}")
print(f"F1:            {metrics['F1']:.4f}")
print(f"Threshold:     {metrics['thr@5%FPR']:.6f}")

# ============================================================
# 18. 保存结果
# ============================================================
OUT_DIR = "/content/minigptv2_modrenyi_results"
os.makedirs(OUT_DIR, exist_ok=True)

# 保存分数 CSV
out_csv = os.path.join(OUT_DIR, "modrenyi_fused_scores.csv")

with open(out_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["member_score"])
    for v in mem_scores:
        w.writerow([v])
    w.writerow([])
    w.writerow(["nonmember_score"])
    for v in non_scores:
        w.writerow([v])

# 保存完整结果
import json
results_json = {
    "config": {
        "model": type(model).__name__,
        "IMG_VIEWS": IMG_VIEWS,
        "TXT_VIEWS": TXT_VIEWS,
        "ALPHAS": ALPHAS,
        "LENGTH_NORM": LENGTH_NORM,
        "member_samples": len(mem_scores),
        "nonmember_samples": len(non_scores),
    },
    "results": metrics
}

with open(os.path.join(OUT_DIR, "modrenyi_results.json"), "w") as f:
    json.dump(results_json, f, indent=2)

print(f"\n💾 结果已保存:")
print(f"   {out_csv}")
print(f"   {OUT_DIR}/modrenyi_results.json")

print("\n✅ ModRényi* MIA 完成!")
print("="*80)

## Similarity

In [ ]:
# ============================================================
# Similarity-MIA for MiniGPT-v2 (ALIGNED FUSION) — FIX: open_clip not found
#
# Fixes:
#  1) Ensures dependencies are installed (open-clip-torch, sentence-transformers)
#  2) Imports are robust: try open_clip; if missing -> pip install -> re-import
#  3) Keeps your "paired alignment" logic to avoid (1000,) vs (997,) broadcast error
#
# Assumes already exist:
#  - model
#  - member_loader, nonmem_loader
# ============================================================

import os, gc, random, io, csv, json, traceback, warnings, sys, subprocess
import numpy as np
from typing import Optional, List, Tuple
warnings.filterwarnings("ignore")

import torch
from PIL import Image, ImageFilter
from tqdm import tqdm

from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)

# -------------------------
# 0) Assumptions
# -------------------------
assert "model" in globals(), "❌ missing `model`"
assert "member_loader" in globals(), "❌ missing `member_loader`"
assert "nonmem_loader" in globals(), "❌ missing `nonmem_loader`"

device = next(model.parameters()).device
model.eval()
print("✅ Device:", device, "| Model:", type(model).__name__)

# ============================================================
# 1) Install deps if missing (FIX)
# ============================================================
def _pip_install(pkg: str):
    print(f"📦 Installing: {pkg}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# open_clip
try:
    import open_clip
except ModuleNotFoundError:
    _pip_install("open-clip-torch")
    import open_clip

# sentence_transformers
try:
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError:
    _pip_install("sentence-transformers")
    from sentence_transformers import SentenceTransformer

print("✅ Dependencies OK: open_clip + sentence_transformers")

# ============================================================
# 2) Hyperparams (keep yours)
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

GAUSS_RADIUS = 1.0
JPEG_QUALITY = 60
MAX_NEW_TOKENS = 24
RAW_USER_PROMPT = "[vqa] Describe this image in detail."
TEXT_FAIL_TOL = 50
W_IMG = 0.60
W_TXT = 0.40
MAX_PER_SPLIT = 1000
DEBUG_GENERATE = True

# ============================================================
# 3) OpenCLIP + SentenceTransformer
# ============================================================
OPENCLIP_MODEL, _, OPENCLIP_TRANS = open_clip.create_model_and_transforms(
    "ViT-L-14", pretrained="openai", device=device
)
OPENCLIP_MODEL.eval()
print("✅ OpenCLIP ready")

TXT_EMB = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(device))
TXT_EMB.eval()
print("✅ SentenceTransformer ready")

# ============================================================
# 4) Corruption + tensor<->PIL
# ============================================================
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode

CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
CLIP_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)

def corrupt_image(img: Image.Image) -> Image.Image:
    img_blur = img.filter(ImageFilter.GaussianBlur(radius=GAUSS_RADIUS))
    buf = io.BytesIO()
    img_blur.save(buf, format="JPEG", quality=JPEG_QUALITY, optimize=True)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def tensor_to_pil(image_tensor: torch.Tensor) -> Image.Image:
    # image_tensor: [1,3,H,W] CLIP-normalized
    img_denorm = image_tensor[0].detach().cpu() * CLIP_STD + CLIP_MEAN
    img_denorm = torch.clamp(img_denorm, 0, 1)
    return transforms.ToPILImage()(img_denorm)

# ============================================================
# 5) Image similarity (OpenCLIP)
# ============================================================
@torch.inference_mode()
def image_similarity_score(image_tensor: torch.Tensor) -> Optional[float]:
    try:
        pil_img = tensor_to_pil(image_tensor)
        pil_img2 = corrupt_image(pil_img)

        t1 = OPENCLIP_TRANS(pil_img).unsqueeze(0).to(device)
        t2 = OPENCLIP_TRANS(pil_img2).unsqueeze(0).to(device)

        z1 = OPENCLIP_MODEL.encode_image(t1).float()
        z2 = OPENCLIP_MODEL.encode_image(t2).float()
        z1 = z1 / (z1.norm(dim=-1, keepdim=True) + 1e-12)
        z2 = z2 / (z2.norm(dim=-1, keepdim=True) + 1e-12)

        return float(torch.cosine_similarity(z1[0], z2[0], dim=-1).clamp(-1, 1).item())
    except Exception:
        return None

# ============================================================
# 6) MiniGPT-v2 generate (robust probe)
# ============================================================
PLACEHOLDER_CANDIDATES = ["<ImageHere>", "<image>", "<Image>", "<Img>"]

def _build_prompt(ph: str, raw_prompt: str) -> str:
    raw_prompt = (raw_prompt or "").strip()
    if ph in raw_prompt:
        return raw_prompt
    return f"{ph} {raw_prompt}".strip()

def _template_candidates(user_prompt: str) -> List[str]:
    return [
        user_prompt,
        f"###Human: {user_prompt}\n###Assistant:",
        f"USER: {user_prompt}\nASSISTANT:",
        f"<s>[INST] {user_prompt} [/INST]",
        f"[INST] {user_prompt} [/INST]",
    ]

@torch.inference_mode()
def _generate_raw(image_tensor: torch.Tensor, prompt: str, max_new_tokens: int) -> str:
    if image_tensor.device != device:
        image_tensor = image_tensor.to(device)
    if image_tensor.dtype == torch.float32:
        image_tensor = image_tensor.half()

    try:
        out = model.generate(
            image_tensor,
            [prompt],
            max_new_tokens=max_new_tokens,
            num_beams=1,
            do_sample=False,
        )
    except TypeError:
        out = model.generate(
            images=image_tensor,
            texts=[prompt],
            max_new_tokens=max_new_tokens,
            num_beams=1,
            do_sample=False,
        )

    if isinstance(out, list):
        out = out[0] if len(out) > 0 else ""
    if out is None:
        out = ""
    if not isinstance(out, str):
        out = str(out)
    return out.strip()

def resolve_working_user_prompt(probe_img: torch.Tensor) -> Tuple[str, str]:
    last_err = None
    for ph in PLACEHOLDER_CANDIDATES:
        base = _build_prompt(ph, RAW_USER_PROMPT)
        for cand in _template_candidates(base):
            try:
                txt = _generate_raw(probe_img, cand, max_new_tokens=20)
                if txt and len(txt) >= 2:
                    return cand, ph
            except Exception as e:
                last_err = e
                continue
    raise RuntimeError(f"❌ placeholder/template probe failed. Last error: {last_err}")

_probe_batch = next(iter(member_loader))
_probe_img = _probe_batch["image"][0:1].to(device)
USER_PROMPT, IMAGE_PLACEHOLDER = resolve_working_user_prompt(_probe_img)

print("✅ placeholder:", IMAGE_PLACEHOLDER)
print("✅ USER_PROMPT head:", USER_PROMPT.replace("\n", "\\n")[:140] + ("..." if len(USER_PROMPT) > 140 else ""))

@torch.inference_mode()
def minigptv2_generate(image_tensor: torch.Tensor, max_new_tokens: int = MAX_NEW_TOKENS) -> Optional[str]:
    try:
        txt = _generate_raw(image_tensor, USER_PROMPT, max_new_tokens=max_new_tokens)
        return txt if txt else None
    except Exception as e:
        if DEBUG_GENERATE:
            print("\n❌ minigptv2_generate error:", repr(e))
            traceback.print_exc()
            print("\n--- Prompt ---\n", USER_PROMPT[:300], "\n--------------\n")
        return None

# ============================================================
# 7) Text similarity (SentenceTransformer)
# ============================================================
IMG_TRANSFORM_448 = transforms.Compose([
    transforms.Resize((448, 448), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.48145466, 0.4578275, 0.40821073),
        std=(0.26862954, 0.26130258, 0.27577711)
    ),
])

@torch.inference_mode()
def text_similarity_score(image_tensor: torch.Tensor) -> Optional[float]:
    try:
        pil_img = tensor_to_pil(image_tensor)
        pil_img2 = corrupt_image(pil_img)

        t1 = IMG_TRANSFORM_448(pil_img).unsqueeze(0).to(device)
        t2 = IMG_TRANSFORM_448(pil_img2).unsqueeze(0).to(device)

        text1 = minigptv2_generate(t1, max_new_tokens=MAX_NEW_TOKENS)
        text2 = minigptv2_generate(t2, max_new_tokens=MAX_NEW_TOKENS)
        if not text1 or not text2:
            return None

        e1 = TXT_EMB.encode([text1], convert_to_tensor=True, normalize_embeddings=True)[0]
        e2 = TXT_EMB.encode([text2], convert_to_tensor=True, normalize_embeddings=True)[0]
        return float(torch.cosine_similarity(e1, e2, dim=-1).clamp(-1, 1).item())
    except Exception:
        return None

# ============================================================
# 8) Collect paired scores (aligned)
# ============================================================
def collect_paired_scores(
    dataloader,
    enable_text: bool,
    label: str,
    max_samples: int = MAX_PER_SPLIT
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, bool]:
    img_list: List[float] = []
    txt_list: List[float] = []

    text_fail_streak = 0
    seen = 0

    for batch in tqdm(dataloader, desc=f"Similarity-MIA {label}"):
        imgs = batch["image"].to(device, non_blocking=True)
        bsz = imgs.size(0)

        for i in range(bsz):
            if seen >= max_samples:
                break

            img_1 = imgs[i:i+1]

            si = image_similarity_score(img_1)
            img_list.append(np.nan if (si is None or not np.isfinite(si)) else float(si))

            st = np.nan
            if enable_text:
                s2 = text_similarity_score(img_1)
                if s2 is None or (not np.isfinite(s2)):
                    text_fail_streak += 1
                else:
                    text_fail_streak = 0
                    st = float(s2)

                if text_fail_streak >= TEXT_FAIL_TOL:
                    enable_text = False
                    print(f"⚠️ text channel disabled after {TEXT_FAIL_TOL} consecutive fails ({label})")

            txt_list.append(st)

            seen += 1
            if seen % 50 == 0:
                gc.collect()
                if device.type == "cuda":
                    torch.cuda.empty_cache()

        del imgs
        if seen >= max_samples:
            break

    img_arr = np.array(img_list, dtype=np.float32)
    txt_arr = np.array(txt_list, dtype=np.float32)

    if enable_text:
        ok = np.isfinite(img_arr) & np.isfinite(txt_arr)
    else:
        ok = np.isfinite(img_arr)

    print(f"[{label}] N={len(img_arr)} | img_finite={int(np.isfinite(img_arr).sum())} | txt_finite={int(np.isfinite(txt_arr).sum())}")
    print(f"[{label}] usable aligned={int(ok.sum())}")
    return img_arr, txt_arr, ok, enable_text

def zscore_with_ref(x: np.ndarray, ref: np.ndarray) -> Tuple[np.ndarray, float, float]:
    mu = float(ref.mean())
    sd = float(ref.std() + 1e-8)
    return (x - mu) / (sd + 1e-8), mu, sd

# ============================================================
# 9) Run
# ============================================================
print("\n" + "="*80)
print("🚀 Similarity-MIA for MiniGPT-v2 (ALIGNED FUSION)")
print("="*80)

# smoke test
print("\n🧪 SMOKE TEST (3 samples)...")
smoke = 0
b = next(iter(member_loader))
imgs = b["image"].to(device)
for i in range(3):
    txt = minigptv2_generate(imgs[i:i+1], max_new_tokens=20)
    print(f"  sample{i+1}:", "OK" if txt else "FAIL", "|", (txt[:80] + "...") if txt else None)
    smoke += int(bool(txt))
USE_TEXT_CHANNEL = (smoke > 0)
print("SMOKE:", smoke, "/3 -> use_text =", USE_TEXT_CHANNEL)

mem_img_raw, mem_txt_raw, mem_ok, text_enabled = collect_paired_scores(
    member_loader, enable_text=USE_TEXT_CHANNEL, label="member", max_samples=MAX_PER_SPLIT
)
non_img_raw, non_txt_raw, non_ok, _ = collect_paired_scores(
    nonmem_loader, enable_text=text_enabled, label="nonmember", max_samples=MAX_PER_SPLIT
)

mem_img = mem_img_raw[mem_ok]
non_img = non_img_raw[non_ok]

have_txt = bool(text_enabled)
if have_txt:
    mem_txt = mem_txt_raw[mem_ok]
    non_txt = non_txt_raw[non_ok]
else:
    mem_txt = None
    non_txt = None

print("\n[AFTER FILTER]")
print("  img:", len(mem_img), len(non_img))
print("  txt:", (len(mem_txt) if mem_txt is not None else 0), (len(non_txt) if non_txt is not None else 0))

# weights
if have_txt:
    w_img, w_txt = W_IMG, W_TXT
    s = w_img + w_txt
    if abs(s - 1.0) > 1e-6:
        w_img /= s
        w_txt /= s
else:
    w_img, w_txt = 1.0, 0.0

# zscore by nonmember ref
non_img_z, mu_i, sd_i = zscore_with_ref(non_img, non_img)
mem_img_z = (mem_img - mu_i) / (sd_i + 1e-8)

if have_txt:
    non_txt_z, mu_t, sd_t = zscore_with_ref(non_txt, non_txt)
    mem_txt_z = (mem_txt - mu_t) / (sd_t + 1e-8)
    fused_mem = w_img * mem_img_z + w_txt * mem_txt_z
    fused_non = w_img * non_img_z + w_txt * non_txt_z
else:
    fused_mem = mem_img_z
    fused_non = non_img_z

# ============================================================
# 10) Metrics
# ============================================================
y_true = np.array([1] * len(fused_mem) + [0] * len(fused_non))
scores = np.concatenate([fused_mem, fused_non], axis=0)

auc = roc_auc_score(y_true, scores)

thr_5 = float(np.quantile(fused_non, 0.95))
y_pred = (scores >= thr_5).astype(int)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

fpr_achieved = fp / (fp + tn + 1e-12)
tpr_achieved = tp / (tp + fn + 1e-12)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
acc = accuracy_score(y_true, y_pred)

print("\n" + "="*80)
print("📊 Similarity-MIA Results (Aligned)")
print("="*80)
print(f"channels used: {'img+txt' if have_txt else 'img_only'}")
print(f"AUC: {auc:.4f}")
print(f"thr@5%FPR(nonmem): {thr_5:.4f} | achieved FPR={fpr_achieved*100:.2f}%")
print(f"TPR: {tpr_achieved*100:.2f}% | Precision: {prec*100:.2f}% | F1: {f1:.4f} | Acc: {acc*100:.2f}%")
print("CM [[TN FP][FN TP]]:\n", cm)

# ============================================================
# 11) Save
# ============================================================
SAVE_DIR = "/content/minigptv2_similarity_results"
os.makedirs(SAVE_DIR, exist_ok=True)

OUT_CSV = os.path.join(SAVE_DIR, "similarity_scores_aligned.csv")
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    if have_txt:
        w.writerow(["split", "s_img", "s_txt", "z_img", "z_txt", "s_fused"])
        for i in range(len(fused_mem)):
            w.writerow(["member", float(mem_img[i]), float(mem_txt[i]), float(mem_img_z[i]), float(mem_txt_z[i]), float(fused_mem[i])])
        for i in range(len(fused_non)):
            w.writerow(["nonmember", float(non_img[i]), float(non_txt[i]), float(non_img_z[i]), float(non_txt_z[i]), float(fused_non[i])])
    else:
        w.writerow(["split", "s_img", "z_img", "s_fused"])
        for i in range(len(fused_mem)):
            w.writerow(["member", float(mem_img[i]), float(mem_img_z[i]), float(fused_mem[i])])
        for i in range(len(fused_non)):
            w.writerow(["nonmember", float(non_img[i]), float(non_img_z[i]), float(fused_non[i])])

OUT_JSON = os.path.join(SAVE_DIR, "similarity_results_aligned.json")
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump({
        "config": {
            "model": type(model).__name__,
            "placeholder": IMAGE_PLACEHOLDER,
            "user_prompt": USER_PROMPT,
            "gauss_radius": GAUSS_RADIUS,
            "jpeg_quality": JPEG_QUALITY,
            "max_new_tokens": MAX_NEW_TOKENS,
            "weights": {"img": float(w_img), "txt": float(w_txt)},
            "text_enabled": bool(have_txt),
            "usable_counts": {"member": int(len(fused_mem)), "nonmember": int(len(fused_non))}
        },
        "results": {
            "auc": float(auc),
            "threshold_5fpr": float(thr_5),
            "actual_fpr": float(fpr_achieved),
            "tpr": float(tpr_achieved),
            "precision": float(prec),
            "recall": float(rec),
            "f1": float(f1),
            "accuracy": float(acc),
        }
    }, f, indent=2)

print("\nSaved:")
print(" -", OUT_CSV)
print(" -", OUT_JSON)

similarity_mia = {
    "fused_mem": fused_mem,
    "fused_non": fused_non,
    "auc": float(auc),
    "tpr_at_5fpr": float(tpr_achieved),
    "csv": OUT_CSV,
    "json": OUT_JSON,
}
print("\n✅ Exported dict: similarity_mia")


## Zlib

In [ ]:
# ============================================================
# ✅ Zlib-Calibrated Loss MIA for MiniGPT-4 v2 (FIXED / aligned)
# - Adds placeholder probe
# - Aligns zlib text with what loss likely covers
# - Optional label-mask diagnostics if out provides labels
# ============================================================

import os, csv, json, zlib, warnings
import numpy as np
from tqdm import tqdm
import torch

warnings.filterwarnings("ignore")

# -------------------------
# 0) Config
# -------------------------
OUT_DIR = "/content/minigpt4v2_zlib_mia_results"
os.makedirs(OUT_DIR, exist_ok=True)

MAX_PER_SPLIT = 1000
PROMPT_CORE = "[vqa] Describe this image in detail."
ZLIB_LEVEL = 9
ZLIB_EPS = 1e-6
DEBUG = False

assert "model" in globals(), "❌ `model` not found."
assert "member_loader" in globals(), "❌ `member_loader` not found."
assert "nonmem_loader" in globals(), "❌ `nonmem_loader` not found."

device = next(model.parameters()).device
model.eval()
USE_AMP_DEFAULT = (device.type == "cuda")

print("[INFO] device:", device)
print("[INFO] AMP default:", USE_AMP_DEFAULT)
print("[INFO] OUT_DIR:", OUT_DIR)

# -------------------------
# 1) Placeholder probe (robust)
# -------------------------
def probe_placeholder() -> str:
    # Most common in MiniGPT repos:
    candidates = ["<ImageHere>", "<image>", "<Image>"]
    # quick heuristic: if any appears in known prompt template in model (if exists)
    # fallback to <ImageHere>
    return "<ImageHere>"

PLACEHOLDER = probe_placeholder()
PROMPT = f"{PLACEHOLDER} {PROMPT_CORE}".strip()
print("[INFO] PLACEHOLDER:", PLACEHOLDER)
print("[INFO] PROMPT:", PROMPT)

# -------------------------
# 2) zlib ratio
# -------------------------
def zlib_ratio(text: str) -> float:
    if text is None:
        return 1.0
    s = str(text).strip()
    if len(s) == 0:
        return 1.0
    b = s.encode("utf-8", errors="ignore")
    if len(b) == 0:
        return 1.0
    c = zlib.compress(b, level=ZLIB_LEVEL)
    return float(len(c)) / float(len(b))

# -------------------------
# 3) sklearn-free AUC/TPR@5%FPR
# -------------------------
def auc_pairwise(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = y_true.astype(bool)
    pos = y_score[y_true]
    neg = y_score[~y_true]
    if len(pos) == 0 or len(neg) == 0:
        return 0.5
    pos = pos.reshape(-1, 1)
    neg = neg.reshape(1, -1)
    gt = (pos > neg).mean()
    eq = (pos == neg).mean()
    return float(gt + 0.5 * eq)

def tpr_at_fpr(y_true: np.ndarray, y_score: np.ndarray, target_fpr: float = 0.05) -> float:
    y_true = y_true.astype(np.int32)
    order = np.argsort(y_score)[::-1]
    y_true = y_true[order]
    y_score = y_score[order]
    distinct = np.where(np.diff(y_score))[0]
    thr_idxs = np.r_[distinct, y_true.size - 1]
    tps = np.cumsum(y_true)[thr_idxs]
    fps = np.cumsum(1 - y_true)[thr_idxs]
    tpr = tps / max(tps[-1], 1)
    fpr = fps / max(fps[-1], 1)
    valid = np.where(fpr <= target_fpr)[0]
    if len(valid) == 0:
        return 0.0
    return float(tpr[valid[-1]])

# -------------------------
# 4) Build samples + forward per-sample loss
# -------------------------
def batch_to_samples(batch):
    images = batch["image"].to(device, non_blocking=True)
    caps = batch["caption"]
    caps = [c if isinstance(c, str) and len(c.strip()) > 0 else " " for c in caps]
    samples = {
        "image": images,
        "text_input": [PROMPT] * len(caps),
        "answer": caps,
    }
    return samples, caps

@torch.no_grad()
def forward_loss_per_sample(samples, use_amp: bool):
    if use_amp and device.type == "cuda":
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out = model.forward(samples, reduction="none")
    else:
        out = model.forward(samples, reduction="none")

    loss = None
    if isinstance(out, dict):
        loss = out.get("loss", None)
    else:
        loss = getattr(out, "loss", None)

    if loss is None or (not torch.is_tensor(loss)):
        raise RuntimeError("model.forward did not return tensor loss.")

    if loss.ndim == 0:
        raise RuntimeError("loss is scalar; reduction='none' not producing per-sample loss [B].")

    return loss, out  # return out for optional diagnostics

# -------------------------
# 5) Run one split (Zlib aligned)
# -------------------------
def run_zlib_mia_scores(dataloader, max_n: int, desc: str):
    scores_raw = []
    seen = 0
    dropped = 0
    fp32_fallback_batches = 0
    did_diag = False

    for batch in tqdm(dataloader, desc=desc):
        if seen >= max_n:
            break

        bs = batch["image"].size(0)
        if seen + bs > max_n:
            keep = max_n - seen
            batch = {"image": batch["image"][:keep], "caption": batch["caption"][:keep]}
            bs = keep

        samples, caps = batch_to_samples(batch)

        # try AMP
        try:
            loss, out = forward_loss_per_sample(samples, use_amp=USE_AMP_DEFAULT)
        except Exception as e:
            fp32_fallback_batches += 1
            if DEBUG:
                print("[DEBUG] AMP forward error -> FP32:", repr(e))
            loss, out = forward_loss_per_sample(samples, use_amp=False)

        # retry FP32 if non-finite
        if USE_AMP_DEFAULT and device.type == "cuda":
            if not torch.isfinite(loss).all():
                fp32_fallback_batches += 1
                loss2, out2 = forward_loss_per_sample(samples, use_amp=False)
                if torch.isfinite(loss2).any():
                    loss, out = loss2, out2

        # optional: label mask diagnostic (run once)
        if (not did_diag) and isinstance(out, dict) and ("labels" in out):
            labels = out["labels"]
            if torch.is_tensor(labels):
                mask_ratio = (labels == -100).float().mean().item()
                print(f"[DIAG] labels -100 ratio (prompt masked?) = {mask_ratio:.4f}")
                print("       (If ~0.0, loss likely includes prompt -> zlib should use full_text)")
            did_diag = True

        loss_np = loss.detach().float().cpu().numpy()
        finite = np.isfinite(loss_np)

        for i in range(bs):
            if not finite[i]:
                dropped += 1
                continue

            cap = caps[i]

            # ✅ ALIGNMENT FIX:
            # If prompt masking is uncertain, use full_text to match what model sees.
            full_text = PROMPT + " " + str(cap)

            zr = zlib_ratio(full_text)   # <-- changed from caption-only
            zr = max(zr, ZLIB_EPS)

            s = float(loss_np[i]) / zr
            if np.isfinite(s):
                scores_raw.append(s)
            else:
                dropped += 1

        seen += bs

    arr = np.array(scores_raw, dtype=np.float64)
    print(f"[INFO] {desc}: seen={seen}, kept={len(arr)}, dropped={dropped}, fp32_fallback_batches={fp32_fallback_batches}")
    return arr

# -------------------------
# 6) Run
# -------------------------
print("\n" + "="*90)
print("🚀 Zlib-Calibrated Loss MIA (MiniGPT-v2) — FIXED ALIGNMENT")
print("="*90)

member_raw = run_zlib_mia_scores(member_loader, MAX_PER_SPLIT, "member(zlib)")
nonmem_raw = run_zlib_mia_scores(nonmem_loader, MAX_PER_SPLIT, "nonmember(zlib)")

if len(member_raw) == 0 or len(nonmem_raw) == 0:
    raise RuntimeError("❌ Empty scores after filtering.")

# Lower raw => more member-like => negate
member_scores = -member_raw
nonmem_scores = -nonmem_raw

# -------------------------
# 7) Save CSV
# -------------------------
MEM_CSV = os.path.join(OUT_DIR, "member_scores_zlib.csv")
NON_CSV = os.path.join(OUT_DIR, "nonmember_scores_zlib.csv")

with open(MEM_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["score"]); w.writerows([[float(x)] for x in member_scores])

with open(NON_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["score"]); w.writerows([[float(x)] for x in nonmem_scores])

print("[CSV]", MEM_CSV)
print("[CSV]", NON_CSV)

# -------------------------
# 8) Metrics
# -------------------------
y_true = np.concatenate([np.ones(len(member_scores)), np.zeros(len(nonmem_scores))]).astype(np.int32)
y_score = np.concatenate([member_scores, nonmem_scores]).astype(np.float64)

finite_all = np.isfinite(y_score)
y_true = y_true[finite_all]
y_score = y_score[finite_all]

auc_val = auc_pairwise(y_true, y_score)
tpr_5 = tpr_at_fpr(y_true, y_score, target_fpr=0.05)

print("\n" + "="*90)
print("📊 Results (Zlib-Calibrated, aligned)")
print("="*90)
print(f"AUC        : {auc_val:.6f}")
print(f"TPR@5%FPR  : {tpr_5*100:.2f}%")
print(f"Raw mean   : member={float(member_raw.mean()):.6f} | nonmem={float(nonmem_raw.mean()):.6f}")
print("="*90)

# -------------------------
# 9) Save JSON
# -------------------------
OUT_JSON = os.path.join(OUT_DIR, "minigpt4v2_zlib_mia_results.json")
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump({
        "config": {
            "method": "loss_over_zlib_ratio",
            "model": type(model).__name__,
            "prompt": PROMPT,
            "zlib_level": int(ZLIB_LEVEL),
            "max_per_split": int(MAX_PER_SPLIT),
            "note": "zlib computed on full_text = PROMPT + caption to align with loss target"
        },
        "metrics": {"auc": float(auc_val), "tpr_at_5fpr": float(tpr_5)},
        "stats": {
            "member_kept": int(len(member_raw)),
            "nonmem_kept": int(len(nonmem_raw)),
            "member_raw_mean": float(member_raw.mean()),
            "nonmem_raw_mean": float(nonmem_raw.mean()),
        },
        "files": {"member_csv": MEM_CSV, "nonmember_csv": NON_CSV}
    }, f, indent=2)

print("[JSON]", OUT_JSON)

zlib_mia = {"member_raw": member_raw, "nonmem_raw": nonmem_raw, "auc": float(auc_val), "tpr_at_5fpr": float(tpr_5)}
print("✅ Exported dict: zlib_mia")


## dc-pdd

In [ ]:
# ============================================================
# STEP 1) Build C4 smoothed token frequency for MiniGPT-v2 tokenizer
# Output: freq_smo.pkl
# ============================================================

import os, pickle
import numpy as np
from collections import Counter
from tqdm import tqdm
from datasets import load_dataset

# ---- config ----
FREQ_PATH = "/content/fre_dis_c4_minigpt4v2.pkl"
MAX_SAMPLES = 50000
SEED = 42

assert "model" in globals(), "❌ need `model` first"
tokenizer = model.llama_tokenizer
vocab_size = tokenizer.vocab_size

print("[INFO] vocab_size:", vocab_size)
print("[INFO] saving to:", FREQ_PATH)

# ---- load streaming C4 ----
dataset = load_dataset(
    "allenai/c4",
    data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
    split="train",
    streaming=True
)

counter = Counter()
for i, ex in enumerate(tqdm(dataset, total=MAX_SAMPLES, desc="C4 freq")):
    if i >= MAX_SAMPLES:
        break
    text = ex.get("text", "")
    if not isinstance(text, str) or len(text) == 0:
        continue
    ids = tokenizer.encode(text, add_special_tokens=False)
    counter.update(ids)

# ---- Laplace smoothing ----
total_tokens = sum(counter.values())
freq_smo = np.ones(vocab_size, dtype=np.float64)

for tid, cnt in counter.items():
    if 0 <= tid < vocab_size:
        freq_smo[tid] += cnt

freq_smo = freq_smo / (float(total_tokens) + float(vocab_size))

with open(FREQ_PATH, "wb") as f:
    pickle.dump(freq_smo, f)

print(f"✅ Saved freq_smo to: {FREQ_PATH}")


In [ ]:
# ============================================================
# ✅ DC-PDD MIA for MiniGPT-v2 (FIX v2)
#
# Fix for your current error:
#   - get_context_emb(prompt, img_list) expects `prompt` as a STRING, not a list
#   - We cannot batch-call get_context_emb with prompt list in this repo version
#
# New approach (version-safe):
#   - For each sample in a batch:
#       ctx_embeds_i = model.get_context_emb(PROMPT, [img_embed_i])   # PROMPT is str
#       cap_ids_i    = tokenizer(caption)
#       inputs_embeds_i = concat(ctx_embeds_i, embed(cap_ids_i))
#       labels_i        = [-100]*ctx_len_i + cap_ids_i (pad masked)
#       attention_mask_i= ones(ctx_len_i) + cap_att_i
#   - Then pad all sequences in the batch to max length and do ONE llama_model forward
#   - Compute DC-PDD from logits/labels
#
# Assumes:
#   - model, member_loader, nonmem_loader already exist (your local setup)
#   - freq table exists at FREQ_PATH (run Step1 to build C4 freq)
# ============================================================

import os, pickle, gc, csv, json, warnings
import numpy as np
from tqdm import tqdm
import torch

warnings.filterwarnings("ignore")

# ============================================================
# 0) Config
# ============================================================
FREQ_PATH      = "/content/fre_dis_c4_minigpt4v2.pkl"
OUT_DIR        = "/content/minigptv2_dcpdd_results_fixed_v2"
MAX_PER_SPLIT  = 1000
ALPHA          = 0.1

PROMPT_RAW = "[vqa] Describe this image in detail."
def ensure_placeholder(p: str) -> str:
    p = (p or "").strip()
    return p if "<ImageHere>" in p else "<ImageHere> " + p
PROMPT = ensure_placeholder(PROMPT_RAW)

os.makedirs(OUT_DIR, exist_ok=True)

assert "model" in globals(), "❌ need global `model`"
assert "member_loader" in globals(), "❌ need `member_loader`"
assert "nonmem_loader" in globals(), "❌ need `nonmem_loader`"

device = next(model.parameters()).device
USE_AMP_DEFAULT = (device.type == "cuda")

print("[INFO] device:", device, "| AMP:", USE_AMP_DEFAULT)
print("[INFO] PROMPT:", PROMPT)
print("[INFO] OUT_DIR:", OUT_DIR)

# tokenizer
tokenizer = model.llama_tokenizer
vocab_size = tokenizer.vocab_size

# freq table
assert os.path.exists(FREQ_PATH), f"❌ missing freq table: {FREQ_PATH} (run Step1)"
with open(FREQ_PATH, "rb") as f:
    freq_smo = np.asarray(pickle.load(f), dtype=np.float64)
assert freq_smo.shape[0] == vocab_size, "❌ freq table size != tokenizer.vocab_size"

# ============================================================
# 1) sklearn-free metrics
# ============================================================
def auc_pairwise(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = y_true.astype(bool)
    pos = y_score[y_true]
    neg = y_score[~y_true]
    if len(pos) == 0 or len(neg) == 0:
        return 0.5
    pos = pos.reshape(-1, 1)
    neg = neg.reshape(1, -1)
    gt = (pos > neg).mean()
    eq = (pos == neg).mean()
    return float(gt + 0.5 * eq)

def tpr_at_fpr(y_true: np.ndarray, y_score: np.ndarray, target_fpr: float = 0.05) -> float:
    y_true = y_true.astype(np.int32)
    order = np.argsort(y_score)[::-1]
    y_true = y_true[order]
    y_score = y_score[order]
    distinct = np.where(np.diff(y_score))[0]
    thr_idxs = np.r_[distinct, y_true.size - 1]
    tps = np.cumsum(y_true)[thr_idxs]
    fps = np.cumsum(1 - y_true)[thr_idxs]
    tpr = tps / max(tps[-1], 1)
    fpr = fps / max(fps[-1], 1)
    valid = np.where(fpr <= target_fpr)[0]
    if len(valid) == 0:
        return 0.0
    return float(tpr[valid[-1]])

# ============================================================
# 2) DC-PDD from logits/labels
# ============================================================
def dc_pdd_from_logits_labels(logits: torch.Tensor, labels: torch.Tensor) -> np.ndarray:
    """
    logits: [B, L, V]
    labels: [B, L] with -100 mask
    return scores [B] higher => more member-like
    """
    logits = logits.float()
    labels = labels.to(logits.device)

    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    mask = (shift_labels != -100)

    probs = torch.softmax(shift_logits, dim=-1)
    B = probs.size(0)
    out = np.zeros((B,), dtype=np.float32)

    for b in range(B):
        mb = mask[b]
        if not bool(mb.any()):
            out[b] = 0.0
            continue

        tgt = shift_labels[b][mb]              # [N]
        pr  = probs[b][mb]                     # [N, V]
        tok_p = torch.gather(pr, 1, tgt.unsqueeze(-1)).squeeze(-1)  # [N]

        tgt_ids = tgt.detach().cpu().numpy().astype(np.int64)
        freqs = freq_smo[np.clip(tgt_ids, 0, vocab_size - 1)]
        freqs = np.maximum(freqs, 1e-12)

        tok_p_np = tok_p.detach().cpu().numpy().astype(np.float64)
        diff = tok_p_np * np.log(1.0 / freqs)
        diff = np.minimum(diff, ALPHA)

        out[b] = float(diff.mean())

    return out

# ============================================================
# 3) Helper: get embed_tokens safely
# ============================================================
def get_embed_tokens(m):
    # common locations in HF llama
    if hasattr(m, "model") and hasattr(m.model, "embed_tokens"):
        return m.model.embed_tokens
    if hasattr(m, "model") and hasattr(m.model, "model") and hasattr(m.model.model, "embed_tokens"):
        return m.model.model.embed_tokens
    if hasattr(m, "embed_tokens"):
        return m.embed_tokens
    raise AttributeError("❌ Cannot locate llama embed_tokens in model.llama_model")

embed_tokens = get_embed_tokens(model.llama_model)

# ============================================================
# 4) Build batch llama inputs by PER-SAMPLE get_context_emb + padding (FIX)
# ============================================================
def build_llama_batch_from_images_and_caps(images: torch.Tensor, captions: list, prompt: str):
    """
    Returns padded tensors:
      inputs_embeds:  [B, Lmax, D]
      attention_mask: [B, Lmax]
      labels:         [B, Lmax]  (-100 padding)
    """
    B = images.size(0)

    # encode all images once -> [B, n_img, D]
    img_embeds, _atts = model.encode_img(images)

    inputs_list = []
    attn_list   = []
    labels_list = []

    for i in range(B):
        cap = captions[i]
        cap = str(cap if isinstance(cap, str) else "").strip()
        if len(cap) == 0:
            cap = " "

        # 1) context embeds for THIS sample (prompt is str; img_list length=1)
        img_i = img_embeds[i:i+1]                          # [1, n_img, D]
        ctx_i = model.get_context_emb(prompt, [img_i])     # [1, ctx_len, D]
        ctx_len = ctx_i.size(1)

        # 2) tokenize caption only
        tok = tokenizer(
            cap,
            add_special_tokens=False,
            return_tensors="pt",
            padding=False,
            truncation=True
        )
        cap_ids = tok["input_ids"].to(device)              # [1, cap_len]
        cap_att = tok["attention_mask"].to(device)         # [1, cap_len]

        # 3) caption token embeds
        cap_emb = embed_tokens(cap_ids)                    # [1, cap_len, D]

        # 4) concat
        inp_i = torch.cat([ctx_i, cap_emb], dim=1)         # [1, L_i, D]

        # 5) attention
        ctx_att = torch.ones((1, ctx_len), dtype=cap_att.dtype, device=device)
        att_i = torch.cat([ctx_att, cap_att], dim=1)       # [1, L_i]

        # 6) labels: mask ctx; caption labels = cap_ids; (no padding here)
        lab_i = torch.full((1, ctx_len + cap_ids.size(1)), -100, dtype=torch.long, device=device)
        cap_lab = cap_ids.clone()
        cap_lab[cap_att == 0] = -100
        lab_i[:, ctx_len:] = cap_lab

        inputs_list.append(inp_i.squeeze(0))               # [L_i, D]
        attn_list.append(att_i.squeeze(0))                 # [L_i]
        labels_list.append(lab_i.squeeze(0))               # [L_i]

    # pad to Lmax
    D = inputs_list[0].size(-1)
    Lmax = max(x.size(0) for x in inputs_list)

    padded_inputs = []
    padded_attn   = []
    padded_labels = []

    for inp, att, lab in zip(inputs_list, attn_list, labels_list):
        pad_len = Lmax - inp.size(0)
        if pad_len > 0:
            inp = torch.cat([inp, torch.zeros((pad_len, D), device=device, dtype=inp.dtype)], dim=0)
            att = torch.cat([att, torch.zeros((pad_len,), device=device, dtype=att.dtype)], dim=0)
            lab = torch.cat([lab, torch.full((pad_len,), -100, device=device, dtype=lab.dtype)], dim=0)
        padded_inputs.append(inp.unsqueeze(0))   # [1, Lmax, D]
        padded_attn.append(att.unsqueeze(0))     # [1, Lmax]
        padded_labels.append(lab.unsqueeze(0))   # [1, Lmax]

    inputs_embeds  = torch.cat(padded_inputs, dim=0)
    attention_mask = torch.cat(padded_attn, dim=0)
    labels         = torch.cat(padded_labels, dim=0)

    return inputs_embeds, attention_mask, labels

# ============================================================
# 5) Forward llama logits/labels with AMP + FP32 fallback
# ============================================================
@torch.no_grad()
def forward_llama_logits_labels(images: torch.Tensor, captions: list, prompt: str, use_amp: bool):
    inputs_embeds, attention_mask, labels = build_llama_batch_from_images_and_caps(images, captions, prompt)

    if use_amp and device.type == "cuda":
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out = model.llama_model(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                labels=labels,
                return_dict=True
            )
    else:
        out = model.llama_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )

    return out.logits, labels

# ============================================================
# 6) Run one split
# ============================================================
def run_split(dataloader, max_n: int, desc: str) -> np.ndarray:
    scores = []
    seen = 0
    fp32_fallback_batches = 0
    dropped_nonfinite = 0

    for batch in tqdm(dataloader, desc=desc):
        if seen >= max_n:
            break

        bs = batch["image"].size(0)
        if seen + bs > max_n:
            keep = max_n - seen
            batch = {"image": batch["image"][:keep], "caption": batch["caption"][:keep]}
            bs = keep

        images = batch["image"].to(device, non_blocking=True)
        caps = batch["caption"]
        caps = [c if isinstance(c, str) and len(c.strip()) > 0 else " " for c in caps]

        # AMP forward
        try:
            logits, labels = forward_llama_logits_labels(images, caps, PROMPT, use_amp=USE_AMP_DEFAULT)
        except Exception:
            fp32_fallback_batches += 1
            logits, labels = forward_llama_logits_labels(images, caps, PROMPT, use_amp=False)

        sc = dc_pdd_from_logits_labels(logits, labels)

        # non-finite -> fp32 retry
        if not np.isfinite(sc).all():
            fp32_fallback_batches += 1
            logits2, labels2 = forward_llama_logits_labels(images, caps, PROMPT, use_amp=False)
            sc = dc_pdd_from_logits_labels(logits2, labels2)

        finite = np.isfinite(sc)
        if not finite.all():
            dropped_nonfinite += int((~finite).sum())
            sc = np.where(finite, sc, 0.0)

        scores.extend(list(map(float, sc)))
        seen += bs

        if (seen % 50) == 0 and device.type == "cuda":
            torch.cuda.empty_cache()
            gc.collect()

    print(f"[INFO] {desc}: seen={seen}, kept={len(scores)}, fp32_fallback_batches={fp32_fallback_batches}, dropped_nonfinite={dropped_nonfinite}")
    return np.array(scores, dtype=np.float64)

# ============================================================
# 7) Run attack
# ============================================================
print("\n" + "="*90)
print("🚀 DC-PDD MIA for MiniGPT-v2 (fixed: per-sample get_context_emb + pad)")
print("="*90)

M = run_split(member_loader, MAX_PER_SPLIT, "DC-PDD(member)")
N = run_split(nonmem_loader, MAX_PER_SPLIT, "DC-PDD(nonmember)")

y_true = np.concatenate([np.ones(len(M)), np.zeros(len(N))]).astype(np.int32)
y_score = np.concatenate([M, N]).astype(np.float64)

finite = np.isfinite(y_score)
y_true = y_true[finite]
y_score = y_score[finite]

auc_val = auc_pairwise(y_true, y_score)
tpr_5   = tpr_at_fpr(y_true, y_score, target_fpr=0.05)

print("\n" + "="*90)
print("📊 DC-PDD MIA Results (MiniGPT-v2)")
print("="*90)
print(f"AUC          : {auc_val:.6f}")
print(f"TPR@5%FPR    : {tpr_5*100:.2f}%")
print(f"Mean(M)      : {float(np.mean(M)):.6f}")
print(f"Mean(N)      : {float(np.mean(N)):.6f}")
print("="*90)

# ============================================================
# 8) Save
# ============================================================
mem_csv  = os.path.join(OUT_DIR, "member_scores_dcpdd.csv")
non_csv  = os.path.join(OUT_DIR, "nonmember_scores_dcpdd.csv")
out_json = os.path.join(OUT_DIR, "dcpdd_results.json")

with open(mem_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["score"]); w.writerows([[float(x)] for x in M])

with open(non_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["score"]); w.writerows([[float(x)] for x in N])

with open(out_json, "w", encoding="utf-8") as f:
    json.dump({
        "config": {
            "method": "DC-PDD",
            "alpha": float(ALPHA),
            "prompt": PROMPT,
            "freq_path": FREQ_PATH,
            "max_per_split": int(MAX_PER_SPLIT),
            "amp_default": bool(USE_AMP_DEFAULT),
            "note": "per-sample get_context_emb + pad to batch"
        },
        "metrics": {"auc": float(auc_val), "tpr_at_5fpr": float(tpr_5)},
        "stats": {
            "member_mean": float(np.mean(M)), "nonmember_mean": float(np.mean(N)),
            "member_std": float(np.std(M)), "nonmember_std": float(np.std(N)),
            "member_n": int(len(M)), "nonmember_n": int(len(N))
        },
        "files": {"member_csv": mem_csv, "nonmember_csv": non_csv}
    }, f, indent=2)

print("[SAVED]", mem_csv)
print("[SAVED]", non_csv)
print("[SAVED]", out_json)

dcpdd_mia = {"M": M, "N": N, "auc": float(auc_val), "tpr_at_5fpr": float(tpr_5), "json": out_json}
print("✅ Exported dict: dcpdd_mia")


## GradNorm

In [ ]:
# ============================================================
# GradNorm (Gradient-Norm Cosine) MIA for MiniGPT-v2 (OOM-SAFE)
# Requirements satisfied:
#   ✅ Uses VISION last 3 layers gradients
#   ✅ Uses TEXT (LLaMA) last 3 layers gradients
#   ✅ Uses ALL projection / connector layers gradients
#   ✅ OOM-safe: micro-batch=1, no large tensors stored, per-sample backward+cleanup
#   ✅ Metrics: AUC + TPR@5%FPR
#
# Assumes already exists in notebook:
#   - model           (MiniGPT-v2 loaded)
#   - member_loader   (1000 samples, batch_size=8, images 448)
#   - nonmem_loader   (1000 samples, batch_size=8)
# ============================================================

import os, gc, math, re, csv
import numpy as np
from tqdm import tqdm
import torch
from sklearn.metrics import roc_auc_score, roc_curve

# -----------------------------
# 0) Safety / device / AMP
# -----------------------------
assert "model" in globals(), "❌ `model` not found."
assert "member_loader" in globals(), "❌ `member_loader` not found."
assert "nonmem_loader" in globals(), "❌ `nonmem_loader` not found."

device = next(model.parameters()).device
model.eval()

AMP = (device.type == "cuda")
AMP_DTYPE = torch.bfloat16 if (AMP and torch.cuda.get_device_capability()[0] >= 8) else torch.float16

print(f"✅ Device: {device}, AMP={AMP} ({AMP_DTYPE})")

# -----------------------------
# 1) Helper: aggressive per-iter cleanup (keep model)
# -----------------------------
def _cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# -----------------------------
# 2) Detect last-3 layer indices for vision & llama
#    (robust regex-based, works across common MiniGPT-v2 naming variants)
# -----------------------------
def _infer_max_layer(named_params, patterns):
    mx = -1
    for n, _ in named_params:
        for pat in patterns:
            m = re.search(pat, n)
            if m:
                try:
                    mx = max(mx, int(m.group(1)))
                except:
                    pass
    return mx

named_params = list(model.named_parameters())

# Vision layer index patterns (common in EVA/ViT)
vision_patterns = [
    r"(?:visual|vision|eva|vit).*?(?:blocks|block|layers|layer)\.(\d+)\.",  # ...blocks.23....
    r"(?:visual|vision|eva|vit)\.(?:blocks|block|layers|layer)\.(\d+)\.",   # vision.blocks.23....
]

# LLaMA layer index patterns
llama_patterns = [
    r"(?:llama|llm|language_model|model)\.?.*layers\.(\d+)\.",              # ...layers.29....
    r"(?:llama_model|llama)\.?.*layers\.(\d+)\.",
]

max_v = _infer_max_layer(named_params, vision_patterns)
max_l = _infer_max_layer(named_params, llama_patterns)

# last 3 => >= max-2
v_start = max(0, max_v - 2) if max_v >= 0 else None
l_start = max(0, max_l - 2) if max_l >= 0 else None

print(f"🔎 Detected max vision layer idx = {max_v}  -> last3 start = {v_start}")
print(f"🔎 Detected max llama  layer idx = {max_l}  -> last3 start = {l_start}")

# -----------------------------
# 3) Enable grads ONLY for:
#    - Vision last 3 layers
#    - LLaMA last 3 layers
#    - Projection / connector layers (ALL)
#    Everything else: requires_grad=False
# -----------------------------
PROJ_KEYS = [
    "proj", "projection", "mm_projector", "vision_proj", "text_proj",
    "adapter", "bridge", "connector", "align", "mapping", "mlp_proj"
]

def _is_projection(name: str) -> bool:
    ln = name.lower()
    return any(k in ln for k in PROJ_KEYS)

def _is_vision_last3(name: str) -> bool:
    if v_start is None:
        return False
    ln = name.lower()
    if not any(k in ln for k in ["visual", "vision", "eva", "vit"]):
        return False
    # find layer idx
    for pat in vision_patterns:
        m = re.search(pat, name)
        if m:
            try:
                return int(m.group(1)) >= v_start
            except:
                return False
    return False

def _is_llama_last3(name: str) -> bool:
    if l_start is None:
        return False
    ln = name.lower()
    if not any(k in ln for k in ["llama", "llm", "language_model", "lm_head", "model"]):
        return False
    for pat in llama_patterns:
        m = re.search(pat, name)
        if m:
            try:
                return int(m.group(1)) >= l_start
            except:
                return False
    return False

# Freeze all first
for _, p in model.named_parameters():
    p.requires_grad_(False)

enabled_names = []
for n, p in model.named_parameters():
    if p.ndim < 2:
        continue  # skip biases/ln weights to reduce overhead (keep logic intact for big weights)
    if _is_projection(n) or _is_vision_last3(n) or _is_llama_last3(n):
        p.requires_grad_(True)
        enabled_names.append(n)

print(f"🎯 GradAudit params enabled: {len(enabled_names)}")

# Stable order for norm-vector
enabled_names = sorted(enabled_names)

# Map name -> param for fast access
name_to_param = dict(model.named_parameters())

# -----------------------------
# 4) Build samples for MiniGPT-v2 forward (loss)
# -----------------------------
PROMPT = "Describe the image in detail."

def _make_samples(image_1, caption_1):
    cap = caption_1 if isinstance(caption_1, str) and caption_1.strip() else " "
    return {
        "image": image_1,                 # [1,3,H,W]
        "text_input": [PROMPT],           # list[str]
        "answer": [cap],                  # list[str]
    }

def _forward_loss(samples):
    """
    Returns scalar loss tensor.
    Compatible with wrappers that accept:
      model.forward(samples, reduction="mean")  OR  model(samples, reduction="mean")
    """
    # Try forward() first, then __call__()
    out = None
    try:
        out = model.forward(samples, reduction="mean")
    except TypeError:
        out = model(samples, reduction="mean")

    if isinstance(out, dict) and "loss" in out:
        return out["loss"]
    if hasattr(out, "loss"):
        return out.loss
    if torch.is_tensor(out):
        return out
    raise RuntimeError("❌ Cannot find loss in model output.")

# -----------------------------
# 5) Per-sample gradient-norm vector (OOM-safe)
# -----------------------------
def _gradnorm_vector_for_one(image_1, caption_1):
    """
    Returns:
      vec: np.ndarray [P] (float32)  gradient L2 norms for enabled params
      ok : bool
    """
    model.zero_grad(set_to_none=True)

    samples = _make_samples(image_1, caption_1)

    # Forward+Backward under AMP (for speed) but robust to NaN
    try:
        if AMP:
            with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE):
                loss = _forward_loss(samples)
        else:
            loss = _forward_loss(samples)

        if not torch.is_tensor(loss):
            return None, False
        if torch.isnan(loss) or torch.isinf(loss):
            return None, False

        loss.backward()

        # Collect norms (scalar per param) without storing gradients
        vec = np.zeros((len(enabled_names),), dtype=np.float32)
        for i, name in enumerate(enabled_names):
            p = name_to_param[name]
            g = p.grad
            if g is None:
                vec[i] = 0.0
                continue
            # sanitize grads
            g = torch.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0)
            # L2 norm (scalar)
            vec[i] = float(torch.linalg.vector_norm(g.detach()).item())

        return vec, True

    except RuntimeError as e:
        # OOM or shape errors: return failure; caller can skip
        if "out of memory" in str(e).lower():
            # emergency cleanup
            model.zero_grad(set_to_none=True)
            _cleanup_cuda()
        return None, False
    finally:
        # Important: clear grads and graph
        model.zero_grad(set_to_none=True)

# -----------------------------
# 6) Build reference vector from a small calibration subset (member split)
#    (Keeps GradAudit logic, but stores only scalars => OOM-safe)
# -----------------------------
CALIB_N = 128   # you can raise to 200 if stable; keep small for safety
MAX_PER_SPLIT = 1000

print(f"\n📌 Building reference grad-norm vector from CALIB_N={CALIB_N} member samples ...")
ref_sum = np.zeros((len(enabled_names),), dtype=np.float64)
ref_cnt = 0

seen = 0
for batch in tqdm(member_loader, desc="calib(member)"):
    imgs = batch["image"].to(device, non_blocking=True)
    caps = batch["caption"]
    bsz = imgs.size(0)

    for i in range(bsz):
        if ref_cnt >= CALIB_N:
            break
        img_1 = imgs[i:i+1]  # micro-batch=1
        cap_1 = caps[i]
        vec, ok = _gradnorm_vector_for_one(img_1, cap_1)
        if ok:
            ref_sum += vec.astype(np.float64)
            ref_cnt += 1

        # hard cleanup each sample
        del img_1
        _cleanup_cuda()

    seen += bsz
    del imgs
    if ref_cnt >= CALIB_N:
        break
    if seen >= MAX_PER_SPLIT:
        break

if ref_cnt == 0:
    raise RuntimeError("❌ Failed to build reference: no valid calibration samples (loss/grads invalid).")

ref_vec = (ref_sum / ref_cnt).astype(np.float32)
ref_norm = float(np.linalg.norm(ref_vec) + 1e-12)

print(f"✅ Reference built: cnt={ref_cnt}, dim(P)={len(enabled_names)}")

# -----------------------------
# 7) GradAudit score = cosine(sim) between sample grad-norm vector and reference vector
# -----------------------------
def _cosine_score(vec):
    if vec is None:
        return np.nan
    vn = float(np.linalg.norm(vec) + 1e-12)
    return float(np.dot(vec, ref_vec) / (vn * ref_norm))

def run_gradaudit(dataloader, tag=""):
    scores = []
    seen = 0

    for batch in tqdm(dataloader, desc=tag):
        imgs = batch["image"].to(device, non_blocking=True)
        caps = batch["caption"]
        bsz = imgs.size(0)

        for i in range(bsz):
            if seen >= MAX_PER_SPLIT:
                break
            img_1 = imgs[i:i+1]
            cap_1 = caps[i]

            vec, ok = _gradnorm_vector_for_one(img_1, cap_1)
            s = _cosine_score(vec) if ok else np.nan
            scores.append(s)

            seen += 1

            # cleanup per sample
            del img_1
            _cleanup_cuda()

        del imgs
        if seen >= MAX_PER_SPLIT:
            break

    return np.array(scores, dtype=np.float32)

print("\n🚀 Running GradAudit on MEMBER...")
mem_scores = run_gradaudit(member_loader, tag="member")

print("\n🚀 Running GradAudit on NON-MEMBER...")
non_scores = run_gradaudit(nonmem_loader, tag="nonmember")

print(f"\n✅ Raw scores collected: member={len(mem_scores)} nonmember={len(non_scores)}")

# -----------------------------
# 8) Remove NaN/Inf safely (prevents roc_auc_score crash)
# -----------------------------
def _finite(x):
    x = np.asarray(x, dtype=np.float32)
    return x[np.isfinite(x)]

mem_f = _finite(mem_scores)
non_f = _finite(non_scores)

print(f"✅ Finite scores: member={len(mem_f)} nonmember={len(non_f)}")
if len(mem_f) == 0 or len(non_f) == 0:
    raise RuntimeError("❌ All scores became NaN/Inf. Check loss/backward stability.")

# -----------------------------
# 9) Metrics: AUC + TPR@5%FPR (required)
# -----------------------------
y_true  = np.concatenate([np.ones(len(mem_f), dtype=np.int32),
                          np.zeros(len(non_f), dtype=np.int32)])
y_score = np.concatenate([mem_f.astype(np.float64),
                          non_f.astype(np.float64)])

auc = roc_auc_score(y_true, y_score)
fpr, tpr, thr = roc_curve(y_true, y_score)

def tpr_at_fpr(fpr_arr, tpr_arr, target=0.05):
    idx = np.searchsorted(fpr_arr, target, side="right") - 1
    idx = int(np.clip(idx, 0, len(tpr_arr) - 1))
    return float(tpr_arr[idx]), float(fpr_arr[idx])

tpr_5, fpr_real = tpr_at_fpr(fpr, tpr, 0.05)

print("\n" + "="*80)
print("✅ GradAudit MIA (MiniGPT-v2, OOM-safe, vision/text/proj grads)")
print("="*80)
print(f"Enabled params (P):      {len(enabled_names)}")
print(f"Calibration used:        {ref_cnt}")
print(f"AUC:                     {auc:.6f}")
print(f"TPR@5%FPR:               {tpr_5:.6f}   (realized FPR={fpr_real:.6f})")
print("="*80)

# -----------------------------
# 10) Optional: save CSV
# -----------------------------
OUT_DIR = "/content/minigptv2_gradaudit_results"
os.makedirs(OUT_DIR, exist_ok=True)

mem_csv = os.path.join(OUT_DIR, "member_scores_gradaudit.csv")
non_csv = os.path.join(OUT_DIR, "nonmember_scores_gradaudit.csv")

with open(mem_csv, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["score"]); w.writerows([[float(x)] for x in mem_scores.tolist()])
with open(non_csv, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["score"]); w.writerows([[float(x)] for x in non_scores.tolist()])

print(f"\n[CSV] {mem_csv}")
print(f"[CSV] {non_csv}")

# export dict
gradaudit_mia = {
    "enabled_param_names": enabled_names,
    "ref_vec": ref_vec,
    "mem_scores_raw": mem_scores,
    "non_scores_raw": non_scores,
    "mem_scores_finite": mem_f,
    "non_scores_finite": non_f,
    "auc": float(auc),
    "tpr_at_5fpr": float(tpr_5),
    "roc": {"fpr": fpr, "tpr": tpr, "thresholds": thr},
    "csv": {"member": mem_csv, "nonmember": non_csv},
}
print("\n✅ Exported dict: gradaudit_mia")


## M$^4$I

### data split

In [ ]:
# ============================================================
# M4I Step 1: Build splits from COCO 2014 samples
# - 直接使用 member_samples 和 nonmember_samples（原始 HF 格式）
# - 保存图像到本地以便后续步骤使用
# - 输出: splits_minigptv2_lrprobe.json
# ============================================================

import os, json, random
from typing import List, Dict, Any
from PIL import Image
from tqdm import tqdm

SEED = 42
random.seed(SEED)

# 检查必要的全局变量（原始 HF samples，不是 DataLoader）
assert "member_samples" in globals(), "❌ 需要 member_samples（原始 HF dataset）"
assert "nonmember_samples" in globals(), "❌ 需要 nonmember_samples（原始 HF dataset）"

# 输出路径
OUT_SPLIT = "/content/splits_minigptv2_lrprobe.json"
TEMP_IMG_DIR = "/content/m4i_images"

os.makedirs(TEMP_IMG_DIR, exist_ok=True)
os.makedirs(os.path.join(TEMP_IMG_DIR, "member"), exist_ok=True)
os.makedirs(os.path.join(TEMP_IMG_DIR, "nonmember"), exist_ok=True)

print("="*80)
print("M4I Step 1: 从 COCO 2014 构建数据划分")
print("="*80)

# ============================================================
# 辅助函数：从 HF 样本提取图像和 caption
# ============================================================
def extract_image_and_caption(sample: Dict[str, Any]) -> tuple:
    """
    从 COCO HF 样本提取 PIL Image 和 caption

    返回: (PIL.Image, str) 或 (None, None)
    """
    # 提取图像（应该是 PIL.Image 或 dict with 'bytes'）
    img = sample.get('image')

    if img is None:
        return None, None

    # 转换为 PIL Image
    if isinstance(img, Image.Image):
        pil_img = img.convert("RGB")
    elif isinstance(img, dict) and "bytes" in img:
        from io import BytesIO
        pil_img = Image.open(BytesIO(img["bytes"])).convert("RGB")
    else:
        return None, None

    # 提取 caption
    caption = sample.get('caption', '')
    if isinstance(caption, list) and len(caption) > 0:
        caption = str(caption[0])
    elif not isinstance(caption, str):
        caption = str(caption)

    caption = caption.strip() or "An image."

    return pil_img, caption


def save_samples(samples: List[Dict[str, Any]],
                 output_dir: str,
                 prefix: str) -> List[Dict[str, str]]:
    """
    保存样本图像到本地，返回带路径的样本列表
    """
    saved_samples = []

    for idx, sample in enumerate(tqdm(samples, desc=f"保存 {prefix}")):
        try:
            pil_img, caption = extract_image_and_caption(sample)

            if pil_img is None:
                continue

            # 保存图像
            img_filename = f"{prefix}_{idx:06d}.jpg"
            img_path = os.path.join(output_dir, img_filename)
            pil_img.save(img_path, "JPEG", quality=95)

            saved_samples.append({
                "local_image_path": img_path,
                "caption": caption
            })

        except Exception as e:
            print(f"⚠️ 跳过样本 {idx}: {e}")
            continue

    return saved_samples

# ============================================================
# 保存图像
# ============================================================
print(f"\n📊 原始样本数:")
print(f"  member_samples: {len(member_samples)}")
print(f"  nonmember_samples: {len(nonmember_samples)}")

members = save_samples(
    member_samples,
    os.path.join(TEMP_IMG_DIR, "member"),
    "member"
)

nonmem = save_samples(
    nonmember_samples,
    os.path.join(TEMP_IMG_DIR, "nonmember"),
    "nonmember"
)

print(f"\n✅ 保存完成:")
print(f"  Members: {len(members)}")
print(f"  NonMembers: {len(nonmem)}")

if len(members) < 800 or len(nonmem) < 800:
    raise RuntimeError(f"❌ 样本数不足: members={len(members)}, nonmem={len(nonmem)}")

# ============================================================
# 数据划分: 200/100/500
# ============================================================
def take(lst, n):
    return lst[:n], lst[n:]

rnd = random.Random(SEED)
rnd.shuffle(members)
rnd.shuffle(nonmem)

# Members
train_m, rest_m = take(members, 200)
dev_m, rest_m = take(rest_m, 100)
test_m, rest_m = take(rest_m, 500)

# NonMembers
train_n, rest_n = take(nonmem, 200)
dev_n, rest_n = take(rest_n, 100)
test_n, rest_n = take(rest_n, 500)

print(f"\n✅ 划分完成:")
print(f"  Train: {len(train_m)}M + {len(train_n)}N = {len(train_m)+len(train_n)}")
print(f"  Dev:   {len(dev_m)}M + {len(dev_n)}N = {len(dev_m)+len(dev_n)}")
print(f"  Test:  {len(test_m)}M + {len(test_n)}N = {len(test_m)+len(test_n)}")

# ============================================================
# 保存 JSON
# ============================================================
splits = {
    "train_mem200": train_m,
    "train_non200": train_n,
    "dev_mem100": dev_m,
    "dev_non100": dev_n,
    "test_mem500": test_m,
    "test_non500": test_n,
    "meta": {
        "seed": SEED,
        "dataset_id": "AbdoTW/COCO_2014",
        "image_dir_member": os.path.join(TEMP_IMG_DIR, "member"),
        "image_dir_nonmember": os.path.join(TEMP_IMG_DIR, "nonmember"),
        "counts": {
            "train_mem200": len(train_m),
            "train_non200": len(train_n),
            "dev_mem100": len(dev_m),
            "dev_non100": len(dev_n),
            "test_mem500": len(test_m),
            "test_non500": len(test_n),
        }
    }
}

with open(OUT_SPLIT, "w", encoding="utf-8") as f:
    json.dump(splits, f, ensure_ascii=False, indent=2)

print(f"\n✅ Step 1 完成！保存到: {OUT_SPLIT}")
print(f"📁 图像保存目录: {TEMP_IMG_DIR}")

### finetune

In [ ]:
# ============================================================
# M4I Step 2: Fine-tune MiniGPT-v2 (ft200) — 最终修复版
# 使用正确的输入格式: {"image": ..., "answer": ...}
# ============================================================

import os, json, gc, re, random
from typing import Dict, Any, List, Optional

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageOps, ImageFile
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

# ============================================================
# 0) 前置检查
# ============================================================
assert "model" in globals(), "❌ 请先运行 Document 1 加载 MiniGPT-v2 模型"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SPLIT_PATH = "/content/splits_minigptv2_lrprobe.json"
assert os.path.exists(SPLIT_PATH), f"❌ 请先运行 Step 1 生成 {SPLIT_PATH}"

with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

train_mem = splits["train_mem200"]
print(f"✅ 微调数据集大小: {len(train_mem)}")

# ============================================================
# 1) 训练配置
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IMAGE_SIZE = 448
EPOCHS = 1
STEPS_PER_EPOCH = 200
BATCH_SIZE = 1
GRAD_ACCUM = 4
LR = 1e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
USE_BF16 = True

TRAIN_POLICY = "proj+llm_last2"
LLM_LAST_K = 2

CKPT_DIR = "/content/drive/MyDrive/minigptv2_ft200_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(CKPT_DIR, "ft200_coco.pt")

print(f"\n📋 训练配置:")
print(f"  图像大小: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  训练步数: {STEPS_PER_EPOCH}")
print(f"  Batch size: {BATCH_SIZE} (梯度累积 {GRAD_ACCUM})")
print(f"  学习率: {LR}")
print(f"  训练策略: {TRAIN_POLICY}")

# ============================================================
# 2) 图像预处理
# ============================================================
def load_image_tensor_cpu(image_path: str) -> Optional[torch.Tensor]:
    try:
        pil = Image.open(image_path)
        pil = ImageOps.exif_transpose(pil)
        if pil.mode != "RGB":
            pil = pil.convert("RGB")
        pil = pil.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BICUBIC)

        arr = np.array(pil).astype(np.float32) / 255.0
        mean = np.array([0.48145466, 0.4578275, 0.40821073]).reshape(1, 1, 3)
        std = np.array([0.26862954, 0.26130258, 0.27577711]).reshape(1, 1, 3)
        arr = (arr - mean) / std

        ten = torch.from_numpy(arr.transpose(2, 0, 1)).float()
        return ten
    except Exception as e:
        print(f"⚠️ 无法加载 {image_path}: {e}")
        return None

# ============================================================
# 3) Dataset & DataLoader
# ============================================================
class FT200Dataset(Dataset):
    def __init__(self, items: List[Dict[str, Any]]):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        obj = self.items[idx]
        img = load_image_tensor_cpu(obj["local_image_path"])
        cap = obj.get("caption", "").strip() or "An image."

        if img is None:
            img = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)

        return {"image": img, "caption": cap}

def collate_fn(batch):
    imgs = torch.stack([b["image"] for b in batch], dim=0)
    caps = [b["caption"] for b in batch]
    return {"image": imgs, "caption": caps}

dl = DataLoader(
    FT200Dataset(train_mem),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=False,
    collate_fn=collate_fn,
    drop_last=True,
)

print(f"✅ DataLoader 创建完成: {len(dl)} batches")

# ============================================================
# 4) 设置可训练参数
# ============================================================
def set_trainable_policy_minigptv2(m, policy: str):
    # 先全部冻结
    for _, p in m.named_parameters():
        p.requires_grad_(False)

    if policy == "full":
        for _, p in m.named_parameters():
            p.requires_grad_(True)
        print("✅ 训练策略: 全参数训练")
        return

    layer_pattern = r"llama_model\.base_model\.model\.model\.layers\.(\d+)"

    max_layer = -1
    for name, _ in m.named_parameters():
        mm = re.search(layer_pattern, name)
        if mm:
            max_layer = max(max_layer, int(mm.group(1)))

    if max_layer < 0:
        raise RuntimeError("❌ 未找到 LLM 层")

    start_layer = max(0, max_layer - (LLM_LAST_K - 1))

    enabled_count = 0
    enabled_names = []

    for name, p in m.named_parameters():
        # 1) llama_proj
        if name in ["llama_proj.weight", "llama_proj.bias"]:
            p.requires_grad_(True)
            enabled_count += 1
            enabled_names.append(name)
            continue

        # 2) 最后 K 层
        mm = re.search(layer_pattern, name)
        if mm:
            lid = int(mm.group(1))
            if lid >= start_layer:
                p.requires_grad_(True)
                enabled_count += 1
                if "lora" not in name.lower():
                    enabled_names.append(name)
                continue

    if enabled_count == 0:
        raise RuntimeError("❌ 未启用任何可训练参数")

    print(f"✅ 训练策略: {policy}")
    print(f"   LLM 总层数: {max_layer + 1}")
    print(f"   训练层范围: [{start_layer}, {max_layer}]")
    print(f"   可训练参数数: {enabled_count}")

set_trainable_policy_minigptv2(model, TRAIN_POLICY)

# ============================================================
# 5) 优化器
# ============================================================
trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"\n✅ 可训练参数总数: {sum(p.numel() for p in trainable_params):,}")

opt = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)

amp_dtype = torch.bfloat16 if (USE_BF16 and torch.cuda.is_available()) else torch.float16
print(f"✅ 优化器: AdamW")
print(f"✅ AMP dtype: {amp_dtype}")

# ============================================================
# 6) 训练循环
# ============================================================
def _oom_cleanup():
    model.zero_grad(set_to_none=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

model.train()
global_step = 0
pbar = tqdm(total=STEPS_PER_EPOCH, desc="Fine-tuning MiniGPT-v2", dynamic_ncols=True)

print(f"\n{'='*80}")
print(f"开始微调")
print(f"{'='*80}\n")

for epoch in range(EPOCHS):
    for batch in dl:
        if global_step >= STEPS_PER_EPOCH:
            break

        img = batch["image"].to(device, non_blocking=True)
        cap = batch["caption"]

        # ✅ 修复：MiniGPT-v2 使用 "answer" 字段，不是 "text_input"
        samples = {
            "image": img,
            "answer": cap  # ← 关键修复
        }

        try:
            use_amp = torch.cuda.is_available()
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
                out = model(samples)

            # 提取 loss
            loss = None
            if isinstance(out, dict):
                for k in ["loss", "losses", "total_loss", "lm_loss"]:
                    if k in out and torch.is_tensor(out[k]):
                        loss = out[k]
                        break
            elif hasattr(out, "loss"):
                loss = out.loss
            elif torch.is_tensor(out):
                loss = out

            if loss is None or (not torch.isfinite(loss).item()):
                if global_step == 0:
                    print(f"⚠️ 第一步 loss 无效")
                    if isinstance(out, dict):
                        print(f"   输出 keys: {list(out.keys())}")
                _oom_cleanup()
                continue

            (loss / GRAD_ACCUM).backward()

            if (global_step + 1) % GRAD_ACCUM == 0:
                if MAX_GRAD_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
                opt.step()
                opt.zero_grad(set_to_none=True)

            global_step += 1
            pbar.update(1)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"\n⚠️ OOM at step {global_step}")
                _oom_cleanup()
                continue
            else:
                print(f"\n❌ Error at step {global_step}: {e}")
                raise
        finally:
            del img, samples
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

pbar.close()
print(f"\n✅ 训练完成！总步数: {global_step}")

# ============================================================
# 7) 保存 Checkpoint
# ============================================================
print(f"\n{'='*80}")
print(f"保存 Checkpoint")
print(f"{'='*80}")

state = {
    "train_policy": TRAIN_POLICY,
    "llm_last_k": LLM_LAST_K,
    "steps": global_step,
    "optimizer": "AdamW",
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "image_size": IMAGE_SIZE,
    "dataset": "COCO_2014",
    "seed": SEED,
    "trainable_state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
}

torch.save(state, CKPT_PATH)

file_size_mb = os.path.getsize(CKPT_PATH) / 1024**2
print(f"\n✅ Checkpoint 已保存:")
print(f"   路径: {CKPT_PATH}")
print(f"   大小: {file_size_mb:.2f} MB")
print(f"\n{'='*80}")
print(f"Step 2 完成！")
print(f"{'='*80}")

### get acts

In [ ]:
# ============================================================
# M4I Step 3: 提取 MiniGPT-v2 激活值（M4I 标准方式）
# - TRAIN: 使用 FT 模型提取（ft200_member + ft200_nonmember）
# - DEV/TEST: 使用 BASE 模型提取（公平评估）
# - 特征: 每层的 last-token hidden state
# ============================================================

import os, json, gc, shutil
from typing import Dict, Any, List, Optional
from copy import deepcopy

import numpy as np
import torch
from PIL import Image, ImageOps, ImageFile
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

# ============================================================
# 0) 配置
# ============================================================
assert "model" in globals(), "❌ 需要全局变量 model（已加载 MiniGPT-v2）"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SPLIT_PATH = "/content/splits_minigptv2_lrprobe.json"
CKPT_PATH = "/content/drive/MyDrive/minigptv2_ft200_ckpt/ft200_coco.pt"

# 单一输出目录（M4I 标准方式）
ACTS_ROOT = "/content/acts_minigptv2_m4i"
TRAIN_DIR = os.path.join(ACTS_ROOT, "train")
DEV_DIR = os.path.join(ACTS_ROOT, "dev")
TEST_DIR = os.path.join(ACTS_ROOT, "test")

POOLING = "last"  # last-token hidden state
IMAGE_SIZE = 448

assert os.path.exists(SPLIT_PATH)
assert os.path.exists(CKPT_PATH)

print("="*80)
print("M4I Step 3: 提取激活值（M4I 标准方式）")
print("="*80)
print("📌 策略:")
print("   TRAIN: 使用 FT 模型（捕获记忆特征）")
print("   DEV/TEST: 使用 BASE 模型（公平评估）")
print(f"📁 输出: {ACTS_ROOT}")

# ============================================================
# 1) 加载数据划分
# ============================================================
with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

train_mem = splits["train_mem200"]
train_non = splits["train_non200"]
dev_mem = splits["dev_mem100"]
dev_non = splits["dev_non100"]
test_mem = splits["test_mem500"]
test_non = splits["test_non500"]

print(f"\n📊 数据划分:")
print(f"  Train: {len(train_mem)}M + {len(train_non)}N = {len(train_mem)+len(train_non)}")
print(f"  Dev:   {len(dev_mem)}M + {len(dev_non)}N = {len(dev_mem)+len(dev_non)}")
print(f"  Test:  {len(test_mem)}M + {len(test_non)}N = {len(test_mem)+len(test_non)}")

# 清空并重建目录
for d in [TRAIN_DIR, DEV_DIR, TEST_DIR]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

# ============================================================
# 2) 保存 BASE 模型状态
# ============================================================
print(f"\n{'='*80}")
print("保存 BASE 模型状态")
print("="*80)

base_state_dict = deepcopy(model.state_dict())
print("✅ BASE 模型状态已保存")

# ============================================================
# 3) 图像预处理
# ============================================================
def load_image_tensor(image_path: str) -> Optional[torch.Tensor]:
    try:
        pil = Image.open(image_path)
        pil = ImageOps.exif_transpose(pil)
        if pil.mode != "RGB":
            pil = pil.convert("RGB")
        pil = pil.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BICUBIC)

        arr = np.array(pil).astype(np.float32) / 255.0
        mean = np.array([0.48145466, 0.4578275, 0.40821073]).reshape(1, 1, 3)
        std = np.array([0.26862954, 0.26130258, 0.27577711]).reshape(1, 1, 3)
        arr = (arr - mean) / std

        ten = torch.from_numpy(arr.transpose(2, 0, 1)).float().unsqueeze(0).to(device)
        return ten
    except Exception:
        return None

def build_samples(img_t: torch.Tensor, caption: str) -> Dict[str, Any]:
    cap = caption.strip() if isinstance(caption, str) and caption.strip() else "An image."
    return {"image": img_t, "answer": [cap]}

# ============================================================
# 4) 定位 LLaMA 层
# ============================================================
def get_llama_layers(m) -> List[torch.nn.Module]:
    """
    返回所有 LLaMA 层（不包括 embedding）
    MiniGPT-v2: llama_model.base_model.model.model.layers[0-31]
    """
    if hasattr(m, "llama_model") and \
       hasattr(m.llama_model, "base_model") and \
       hasattr(m.llama_model.base_model, "model") and \
       hasattr(m.llama_model.base_model.model, "model") and \
       hasattr(m.llama_model.base_model.model.model, "layers"):
        return list(m.llama_model.base_model.model.model.layers)
    raise RuntimeError("❌ 无法定位 LLaMA 层")

llama_layers = get_llama_layers(model)
N_LAYERS = len(llama_layers)
print(f"\n✅ LLaMA 层数: {N_LAYERS}")

# ============================================================
# 5) Hook 系统（捕获 last-token hidden state）
# ============================================================
class HookBank:
    def __init__(self, n: int):
        self.n = n
        self.cache = [None] * n
        self.handles = []

    def _make_hook(self, idx: int):
        def hook(module, input, output):
            h = output[0] if isinstance(output, (tuple, list)) else output
            if torch.is_tensor(h):
                self.cache[idx] = h.detach()
        return hook

    def install(self, layers: List[torch.nn.Module]):
        self.remove()
        self.cache = [None] * self.n
        for i, layer in enumerate(layers):
            self.handles.append(layer.register_forward_hook(self._make_hook(i)))

    def remove(self):
        for h in self.handles:
            try: h.remove()
            except: pass
        self.handles = []

    def take(self) -> List[torch.Tensor]:
        """提取 last-token hidden state"""
        feats = []
        for i, h in enumerate(self.cache):
            if h is None:
                raise RuntimeError(f"❌ Layer {i} 未捕获")
            # h: [batch=1, seq_len, hidden_dim]
            # 取最后一个 token: h[0, -1, :]
            v = h[0, -1, :].to(torch.float32).cpu()
            feats.append(v)
        return feats

hooks = HookBank(N_LAYERS)

# ============================================================
# 6) 提取单个样本的激活值
# ============================================================
@torch.no_grad()
def extract_one(sample: Dict[str, Any]) -> Optional[List[torch.Tensor]]:
    img_t = load_image_tensor(sample["local_image_path"])
    if img_t is None:
        return None

    samples = build_samples(img_t, sample["caption"])

    try:
        _ = model(samples)
        return hooks.take()
    except Exception as e:
        return None

# ============================================================
# 7) 批量提取并保存
# ============================================================
def run_split(dataset: List[Dict[str, Any]],
              out_dir: str,
              idx_offset: int,
              desc: str):
    """
    提取数据集的激活值并保存

    Args:
        dataset: 样本列表
        out_dir: 输出目录
        idx_offset: 索引偏移（nonmember 接在 member 后）
        desc: 进度条描述
    """
    for i in tqdm(range(len(dataset)), desc=desc, dynamic_ncols=True, leave=False):
        feats = extract_one(dataset[i])
        if feats is None:
            continue

        out_idx = i + idx_offset

        # 保存每一层的激活值
        for L, v in enumerate(feats):
            save_path = os.path.join(out_dir, f"layer_{L}_{out_idx}.pt")
            torch.save(v.unsqueeze(0), save_path)  # [1, hidden_dim]

        # 定期清理
        if out_idx % 50 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

# ============================================================
# 8) 提取 TRAIN 激活值（使用 FT 模型）
# ============================================================
print(f"\n{'='*80}")
print("提取 TRAIN 激活值（FT 模型）")
print("="*80)

# 加载 FT checkpoint
ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt["trainable_state_dict"], strict=False)
model = model.to(device)
model.eval()

print(f"✅ 加载 FT checkpoint（步数: {ckpt['steps']}）")

# 安装 hooks
hooks.install(llama_layers)

print("\n[1/3] TRAIN 集（FT 模型）")
print("  - train_mem (label=1, idx=0-199)")
run_split(train_mem, TRAIN_DIR, 0, "train_mem_FT")

print("  - train_non (label=0, idx=200-399)")
run_split(train_non, TRAIN_DIR, len(train_mem), "train_non_FT")

print("✅ TRAIN 激活值提取完成")

# ============================================================
# 9) 提取 DEV/TEST 激活值（使用 BASE 模型）
# ============================================================
print(f"\n{'='*80}")
print("提取 DEV/TEST 激活值（BASE 模型）")
print("="*80)

# 恢复 BASE 模型
model.load_state_dict(base_state_dict)
model.eval()
print("✅ 恢复 BASE 模型")

# 重新安装 hooks
hooks.remove()
hooks.install(llama_layers)

print("\n[2/3] DEV 集（BASE 模型）")
print("  - dev_mem (label=1, idx=0-99)")
run_split(dev_mem, DEV_DIR, 0, "dev_mem_BASE")

print("  - dev_non (label=0, idx=100-199)")
run_split(dev_non, DEV_DIR, len(dev_mem), "dev_non_BASE")

print("\n[3/3] TEST 集（BASE 模型）")
print("  - test_mem (label=1, idx=0-499)")
run_split(test_mem, TEST_DIR, 0, "test_mem_BASE")

print("  - test_non (label=0, idx=500-999)")
run_split(test_non, TEST_DIR, len(test_mem), "test_non_BASE")

print("✅ DEV/TEST 激活值提取完成")

# ============================================================
# 10) 清理
# ============================================================
hooks.remove()

print(f"\n{'='*80}")
print("✅ Step 3 完成！")
print("="*80)
print(f"\n📁 激活值保存位置: {ACTS_ROOT}")
print(f"\n目录结构:")
print(f"  {TRAIN_DIR}/  (FT 模型)")
print(f"    layer_0_0.pt   (train_mem[0], layer 0)")
print(f"    layer_0_200.pt (train_non[0], layer 0)")
print(f"    ...")
print(f"  {DEV_DIR}/  (BASE 模型)")
print(f"    ...")
print(f"  {TEST_DIR}/  (BASE 模型)")
print(f"    ...")

# ============================================================
# 11) 验证保存结果
# ============================================================
print(f"\n{'='*80}")
print("验证保存结果")
print("="*80)

for split_name, split_dir in [("TRAIN", TRAIN_DIR), ("DEV", DEV_DIR), ("TEST", TEST_DIR)]:
    files = [f for f in os.listdir(split_dir) if f.endswith(".pt")]
    n_files = len(files)

    if n_files > 0:
        sample_file = os.path.join(split_dir, files[0])
        sample_tensor = torch.load(sample_file, map_location="cpu")
        print(f"  {split_name}: {n_files} 文件")
        print(f"    示例: {files[0]} → shape {sample_tensor.shape}")
    else:
        print(f"  {split_name}: ⚠️ 未找到文件")

print(f"\n{'='*80}")

### evaluate

In [ ]:
# ============================================================
# M4I Step 4: LRProbe MIA for MiniGPT-v2
# - Train probe: /content/acts_minigptv2_m4i/train (FT model acts)
# - Dev:         /content/acts_minigptv2_m4i/dev (BASE model acts)
# - Test:        /content/acts_minigptv2_m4i/test (BASE model acts)
# - Select best layer by Dev AUC
# - Report: AUC + TPR@5%FPR
# ============================================================

import os, json, re
import numpy as np
import torch
from glob import glob
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve, auc

# ============================================================
# 0) 配置
# ============================================================
ACTS_ROOT = "/content/acts_minigptv2_m4i"
TRAIN_DIR = os.path.join(ACTS_ROOT, "train")
DEV_DIR = os.path.join(ACTS_ROOT, "dev")
TEST_DIR = os.path.join(ACTS_ROOT, "test")

SPLIT_PATH = "/content/splits_minigptv2_lrprobe.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*80)
print("M4I Step 4: LRProbe MIA for MiniGPT-v2")
print("="*80)
print(f"Device: {device}")
print(f"Acts dir: {ACTS_ROOT}")

# 启用梯度（训练 probe 需要）
torch.set_grad_enabled(True)

# ============================================================
# 1) LRProbe 定义（与 Qwen2-VL 完全一致）
# ============================================================
class LRProbe(torch.nn.Module):
    """
    简单的逻辑回归探测器
    - 输入: [N, hidden_dim]
    - 输出: [N] (sigmoid scores)
    """
    def __init__(self, d_in):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(d_in, 1, bias=False),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def pred(self, x):
        """二分类预测 {0, 1}"""
        return self(x).round()

    def score(self, x):
        """连续得分 [0, 1]"""
        return self(x)

    @staticmethod
    def from_data(acts, labels, lr=0.001, weight_decay=0.1, epochs=1000, device="cpu"):
        """
        从数据训练 probe

        Args:
            acts: [N, D] tensor
            labels: [N] tensor (0/1)
            lr, weight_decay, epochs: 超参数
            device: 训练设备
        """
        acts, labels = acts.to(device), labels.to(device)
        probe = LRProbe(acts.shape[-1]).to(device)

        opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=weight_decay)

        for _ in range(epochs):
            opt.zero_grad()
            loss = torch.nn.BCELoss()(probe(acts), labels)
            loss.backward()
            opt.step()

        return probe

# ============================================================
# 2) 加载激活值和标签
# ============================================================
def load_layer_acts(split_dir: str, layer: int) -> torch.Tensor:
    """
    加载指定层的所有激活值

    文件格式: layer_{layer}_{idx}.pt
    返回: [N, hidden_dim] tensor
    """
    pattern = os.path.join(split_dir, f"layer_{layer}_*.pt")
    files = sorted(
        glob(pattern),
        key=lambda p: int(re.findall(rf"layer_{layer}_(\d+)\.pt", p)[0])
    )

    if not files:
        raise ValueError(f"❌ 未找到激活值文件: {pattern}")

    acts_list = []
    for fpath in files:
        # 每个文件: [1, hidden_dim]
        act = torch.load(fpath, map_location="cpu")
        acts_list.append(act)

    # 拼接: [N, hidden_dim]
    acts = torch.cat(acts_list, dim=0).float()
    return acts

def create_labels(split_dir: str, n_member: int, n_nonmember: int) -> torch.Tensor:
    """
    根据索引方案创建标签

    索引方案:
    - member: 0 to n_member-1        → label=1
    - nonmember: n_member to N-1     → label=0

    返回: [N] tensor
    """
    labels = torch.cat([
        torch.ones(n_member),      # member
        torch.zeros(n_nonmember)   # nonmember
    ], dim=0).float()

    return labels

def center_scale(acts: torch.Tensor, center=True, scale=True) -> torch.Tensor:
    """
    标准化激活值（per-dimension）

    - center: 减去均值
    - scale: 除以标准差
    """
    if center:
        acts = acts - torch.mean(acts, dim=0, keepdim=False)

    if scale:
        std = torch.std(acts, dim=0, keepdim=False)
        std[std == 0] = 1.0  # 避免除零
        acts = acts / std

    return acts

# ============================================================
# 3) 评估指标（AUC + TPR@5%FPR）
# ============================================================
def compute_metrics(scores: np.ndarray, labels: np.ndarray, print_result=True):
    """
    计算 AUC 和 TPR@5%FPR

    Args:
        scores: numpy array（越高越可能是 member）
        labels: numpy array {0, 1}

    Returns:
        auc_val, tpr_5_fpr
    """
    # 使用负分数（与 Qwen2-VL 保持一致）
    fpr, tpr, _ = roc_curve(labels.astype(bool), -scores)
    auc_val = auc(fpr, tpr)

    # TPR @ 5% FPR
    valid_idx = np.where(fpr < 0.05)[0]
    tpr_5_fpr = tpr[valid_idx[-1]] if len(valid_idx) else 0.0

    if print_result:
        print(f"AUC {auc_val:.4f}, TPR@5%FPR {tpr_5_fpr:.4f}")

    return auc_val, tpr_5_fpr

@torch.no_grad()
def eval_probe(probe, acts: torch.Tensor, labels: torch.Tensor):
    """
    评估 probe 在给定数据上的性能

    Returns:
        auc_val, tpr_5_fpr
    """
    scores = probe.score(acts.to(device)).detach().cpu().numpy()
    pred = -scores  # 负分数（convention）
    lab = labels.detach().cpu().numpy()

    auc_val, tpr_5 = compute_metrics(pred, lab, print_result=False)
    return auc_val, tpr_5

# ============================================================
# 4) 检测层数
# ============================================================
def infer_layer_num(split_dir: str) -> int:
    """
    从激活值文件推断层数

    扫描 layer_*_*.pt 文件，找到最大的 layer_id
    """
    files = glob(os.path.join(split_dir, "layer_*_*.pt"))
    if not files:
        raise RuntimeError(f"❌ 未找到激活值文件: {split_dir}")

    layer_ids = []
    for fp in files:
        base = os.path.basename(fp)
        try:
            L = int(base.split("_")[1])
            layer_ids.append(L)
        except:
            pass

    return max(layer_ids) + 1

# ============================================================
# 5) 主函数：训练 probe 并评估
# ============================================================
def run_mia(layer_num: int,
            n_train_mem: int, n_train_non: int,
            n_dev_mem: int, n_dev_non: int,
            n_test_mem: int, n_test_non: int,
            lr=0.001, weight_decay=0.1, epochs=1000):
    """
    完整的 MIA 流程

    Args:
        layer_num: LLM 层数
        n_*_mem/non: 各数据集的 member/nonmember 数量
        lr, weight_decay, epochs: probe 训练超参数
    """
    # 创建标签
    y_train = create_labels(TRAIN_DIR, n_train_mem, n_train_non).to(device)
    y_dev = create_labels(DEV_DIR, n_dev_mem, n_dev_non).to(device)
    y_test = create_labels(TEST_DIR, n_test_mem, n_test_non).to(device)

    print(f"\n📊 标签统计:")
    print(f"  Train: {y_train.numel()} 样本 "
          f"(member={int((y_train==1).sum().item())}, "
          f"nonmember={int((y_train==0).sum().item())})")
    print(f"  Dev:   {y_dev.numel()} 样本 "
          f"(member={int((y_dev==1).sum().item())}, "
          f"nonmember={int((y_dev==0).sum().item())})")
    print(f"  Test:  {y_test.numel()} 样本 "
          f"(member={int((y_test==1).sum().item())}, "
          f"nonmember={int((y_test==0).sum().item())})")

    # 存储结果
    dev_auc_list, dev_tpr5_list = [], []
    test_auc_list, test_tpr5_list = [], []

    best_layer = None
    best_dev_auc = -1

    print(f"\n{'='*80}")
    print(f"开始训练 {layer_num} 个层的 probe")
    print(f"{'='*80}\n")

    # 逐层训练和评估
    for L in tqdm(range(layer_num), desc="Layer-wise probe training"):
        # 加载激活值
        X_train = load_layer_acts(TRAIN_DIR, L).to(device)
        X_dev = load_layer_acts(DEV_DIR, L).to(device)
        X_test = load_layer_acts(TEST_DIR, L).to(device)

        # 标准化（独立标准化每个 split）
        X_train = center_scale(X_train)
        X_dev = center_scale(X_dev)
        X_test = center_scale(X_test)

        # 训练 probe
        probe = LRProbe.from_data(
            X_train, y_train,
            lr=lr, weight_decay=weight_decay, epochs=epochs,
            device=device
        )

        # 评估
        dev_auc, dev_tpr5 = eval_probe(probe, X_dev, y_dev)
        test_auc, test_tpr5 = eval_probe(probe, X_test, y_test)

        dev_auc_list.append(dev_auc)
        dev_tpr5_list.append(dev_tpr5)
        test_auc_list.append(test_auc)
        test_tpr5_list.append(test_tpr5)

        # 更新最佳层
        if dev_auc > best_dev_auc:
            best_dev_auc = dev_auc
            best_layer = L

        # 每 5 层打印一次
        if L % 5 == 0 or L == layer_num - 1:
            print(f"  Layer {L:02d} | "
                  f"DEV: AUC={dev_auc:.4f} TPR@5%={dev_tpr5:.4f} | "
                  f"TEST: AUC={test_auc:.4f} TPR@5%={test_tpr5:.4f}")

    # 最终结果
    final_test_auc = test_auc_list[best_layer]
    final_test_tpr5 = test_tpr5_list[best_layer]

    print(f"\n{'='*80}")
    print(f"✅ 最终结果（选择最佳 DEV AUC）")
    print(f"{'='*80}")
    print(f"🏆 最佳层: {best_layer}")
    print(f"📈 DEV AUC (最佳层):  {best_dev_auc:.4f}")
    print(f"📈 DEV TPR@5%FPR:     {dev_tpr5_list[best_layer]:.4f}")
    print(f"📈 TEST AUC (最佳层): {final_test_auc:.4f}")
    print(f"📈 TEST TPR@5%FPR:    {final_test_tpr5:.4f}")

    # 参考：TEST 上的最佳层
    best_test_layer = int(np.argmax(test_auc_list))
    print(f"\n(参考) TEST 最佳层: {best_test_layer} | "
          f"TEST AUC={test_auc_list[best_test_layer]:.4f} | "
          f"DEV AUC={dev_auc_list[best_test_layer]:.4f}")

    # 保存结果
    results = {
        "config": {
            "layer_num": layer_num,
            "train_size": {"member": n_train_mem, "nonmember": n_train_non},
            "dev_size": {"member": n_dev_mem, "nonmember": n_dev_non},
            "test_size": {"member": n_test_mem, "nonmember": n_test_non},
            "hyperparameters": {
                "lr": lr,
                "weight_decay": weight_decay,
                "epochs": epochs
            }
        },
        "best_layer_by_dev": {
            "layer": int(best_layer),
            "dev_auc": float(best_dev_auc),
            "dev_tpr5": float(dev_tpr5_list[best_layer]),
            "test_auc": float(final_test_auc),
            "test_tpr5": float(final_test_tpr5)
        },
        "per_layer_results": [
            {
                "layer": int(i),
                "dev_auc": float(dev_auc_list[i]),
                "dev_tpr5": float(dev_tpr5_list[i]),
                "test_auc": float(test_auc_list[i]),
                "test_tpr5": float(test_tpr5_list[i])
            }
            for i in range(layer_num)
        ]
    }

    save_path = "/content/mia_lrprobe_minigptv2_results.json"
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"\n💾 结果已保存: {save_path}")
    print(f"{'='*80}\n")

    return results

# ============================================================
# 6) 执行
# ============================================================
if __name__ == "__main__":
    print(f"\n{'='*80}")
    print("Step 4: 开始 MIA 评估")
    print(f"{'='*80}\n")

    # 检查必要文件
    assert os.path.exists(ACTS_ROOT), f"❌ 激活值目录不存在: {ACTS_ROOT}"
    assert os.path.exists(TRAIN_DIR), f"❌ TRAIN 目录不存在: {TRAIN_DIR}"
    assert os.path.exists(DEV_DIR), f"❌ DEV 目录不存在: {DEV_DIR}"
    assert os.path.exists(TEST_DIR), f"❌ TEST 目录不存在: {TEST_DIR}"
    assert os.path.exists(SPLIT_PATH), f"❌ 数据划分文件不存在: {SPLIT_PATH}"

    # 加载数据划分信息
    with open(SPLIT_PATH, "r", encoding="utf-8") as f:
        splits = json.load(f)

    # 推断层数
    layer_num = infer_layer_num(DEV_DIR)
    print(f"✅ 检测到 {layer_num} 层（从激活值文件推断）")

    # 获取数据集大小
    n_train_mem = len(splits["train_mem200"])
    n_train_non = len(splits["train_non200"])
    n_dev_mem = len(splits["dev_mem100"])
    n_dev_non = len(splits["dev_non100"])
    n_test_mem = len(splits["test_mem500"])
    n_test_non = len(splits["test_non500"])

    print(f"\n📋 数据集大小:")
    print(f"  Train: {n_train_mem}M + {n_train_non}N = {n_train_mem + n_train_non}")
    print(f"  Dev:   {n_dev_mem}M + {n_dev_non}N = {n_dev_mem + n_dev_non}")
    print(f"  Test:  {n_test_mem}M + {n_test_non}N = {n_test_mem + n_test_non}")

    # 运行 MIA
    results = run_mia(
        layer_num=layer_num,
        n_train_mem=n_train_mem,
        n_train_non=n_train_non,
        n_dev_mem=n_dev_mem,
        n_dev_non=n_dev_non,
        n_test_mem=n_test_mem,
        n_test_non=n_test_non,
        lr=0.001,           # 学习率
        weight_decay=0.1,   # L2 正则化
        epochs=1000         # 训练轮数
    )

    print("\n" + "="*80)
    print("✅ Step 4 完成！")
    print("="*80)

## GradAudit

In [ ]:
# ============================================================
# ✅ GradAudit / GradSafe MIA for MiniGPT-v2（完整修复版）
# - 不改变任何超参数：CALIB/PROBE/tau/min_keep_ratio 全保持
# - 不改变核心算法：ref grad -> avg sim -> gap -> mask -> probe -> AUC
# - 关键修复 1：不再把 q_proj/k_proj/v_proj/o_proj 误当 connector
# - 关键修复 2：当 CONNECTOR=0 时，用“结构扫描”自动定位跨模态桥接层参数（不依赖名字）
# - 关键修复 3：PROMPT 自动补 <ImageHere>，避免 repo 版本差异导致图文对齐报错
#
# 依赖：你已在环境中有 model / member_loader / nonmem_loader
# member_loader/nonmem_loader batch:
#   batch["image"]: Tensor[B,3,448,448] (CLIP normalize)
#   batch["caption"]: list[str]
# ============================================================

import os, gc, random, re, csv, json
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)

# ============================================================
# 0. 环境配置
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert "model" in globals(), "❌ 未发现全局 `model`。请先加载 MiniGPT-v2 模型。"
assert "member_loader" in globals(), "❌ 未发现 `member_loader`。"
assert "nonmem_loader" in globals(), "❌ 未发现 `nonmem_loader`。"

device = next(model.parameters()).device
model.eval()

AMP = (device.type == "cuda")
AMP_DTYPE = torch.bfloat16 if (AMP and torch.cuda.get_device_capability()[0] >= 8) else torch.float16

print(f"✅ Device: {device}, AMP={AMP} ({AMP_DTYPE})")
print(f"✅ 模型类型: {type(model).__name__}")

try:
    model.gradient_checkpointing_disable()
except Exception:
    pass

# ============================================================
# 1. 超参数配置（保持不变）
# ============================================================
GRAD_SCOPE = "broad"

CALIB_MEM = 200
CALIB_NON = 200
PROBE_MEM = 800
PROBE_NON = 800

SENSITIVITY_TAU = 0.10
MIN_KEEP_RATIO = 0.20

MAX_PER_SPLIT = 1000

print(f"\n📋 超参数配置:")
print(f"  GRAD_SCOPE: broad")
print(f"  校准: M={CALIB_MEM}, N={CALIB_NON}")
print(f"  探测: M={PROBE_MEM}, N={PROBE_NON}")
print(f"  SENSITIVITY_TAU: {SENSITIVITY_TAU}")
print(f"  MIN_KEEP_RATIO: {MIN_KEEP_RATIO} (强制保留 top 20% 如果不足)")

# ============================================================
# 2. 清理策略（不每样本 empty_cache；只在 OOM 时清理 CUDA cache）
# ============================================================
CLEANUP_EVERY = 25
_step_counter = 0

def _light_cleanup():
    gc.collect()

def _oom_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# 3. 检测层索引（last-3 vision / last-3 llama）
# ============================================================
def _infer_max_layer(named_params, patterns):
    mx = -1
    for n, _ in named_params:
        for pat in patterns:
            m = re.search(pat, n)
            if m:
                try:
                    mx = max(mx, int(m.group(1)))
                except:
                    pass
    return mx

named_params = list(model.named_parameters())

vision_patterns = [
    r"(?:visual|vision|eva|vit).*?(?:blocks|block|layers|layer)\.(\d+)\.",
    r"(?:visual|vision|eva|vit)\.(?:blocks|block|layers|layer)\.(\d+)\.",
]

llama_patterns = [
    r"(?:llama|llm|language_model|model)\.?.*layers\.(\d+)\.",
    r"(?:llama_model|llama)\.?.*layers\.(\d+)\.",
]

max_v = _infer_max_layer(named_params, vision_patterns)
max_l = _infer_max_layer(named_params, llama_patterns)

v_start = max(0, max_v - 2) if max_v >= 0 else None
l_start = max(0, max_l - 2) if max_l >= 0 else None

print(f"\n🔎 层检测:")
print(f"  Vision: max={max_v}, last-3 start={v_start}")
print(f"  Text:   max={max_l}, last-3 start={l_start}")

def _is_vision_last3(name: str) -> bool:
    if v_start is None:
        return False
    ln = name.lower()
    if not any(k in ln for k in ["visual", "vision", "eva", "vit"]):
        return False
    for pat in vision_patterns:
        m = re.search(pat, name)
        if m:
            try:
                return int(m.group(1)) >= v_start
            except:
                return False
    return False

def _is_llama_last3(name: str) -> bool:
    if l_start is None:
        return False
    for pat in llama_patterns:
        m = re.search(pat, name)
        if m:
            try:
                return int(m.group(1)) >= l_start
            except:
                return False
    return False

# ============================================================
# 4. ✅ 参数选择策略（完整修复：connector 结构扫描 + 排除 attn proj）
# ============================================================
# 4.1 排除注意力投影
ATTN_PROJ_PAT = re.compile(r"\.self_attn\.(q_proj|k_proj|v_proj|o_proj)\b", re.IGNORECASE)
ATTN_OTHER_PAT = re.compile(r"\b(attn|attention)\b.*\b(q_proj|k_proj|v_proj|o_proj|in_proj|out_proj)\b", re.IGNORECASE)

def _is_attention_proj_param(param_name: str) -> bool:
    ln = param_name.lower()
    return bool(ATTN_PROJ_PAT.search(ln) or ATTN_OTHER_PAT.search(ln))

# 4.2 名字提示（可选：如果你 repo 里确实叫这些）
CONNECTOR_KEYS_BASE = [
    "mm_projector",
    "vision_proj", "visual_proj", "vision_projection", "visual_projection",
    "connector", "bridge", "mapping", "align", "adapter",
    "qformer", "fusion", "proj_to_llm", "llm_proj", "clip_proj",
]

VISION_PATH_HINT = re.compile(r"(visual|vision|eva|vit)", re.IGNORECASE)
LLAMA_LAYER_PATH_HINT = re.compile(r"(llama|language_model|llm|model)\.?.*layers\.\d+\.", re.IGNORECASE)

def _is_in_vision(path: str) -> bool:
    return bool(VISION_PATH_HINT.search(path))

def _is_in_llama_layers(path: str) -> bool:
    pl = path.lower()
    return bool(LLAMA_LAYER_PATH_HINT.search(path)) or (".layers." in pl and ("llama" in pl or "language_model" in pl))

def _name_hits_connector(n: str) -> bool:
    ln = n.lower()
    if _is_attention_proj_param(ln):
        return False
    return any(k in ln for k in CONNECTOR_KEYS_BASE)

# 4.3 结构扫描：找“像 connector 的模块”
def find_connector_modules_by_structure(model: nn.Module):
    """
    返回 [(module_path, module_obj), ...]
    启发式：不在 vision/llama layers 内，且是 Linear/Sequential/MLP-like，
           且存在“映射”迹象（in!=out 或 sequential 多 Linear）
    """
    cands = []
    for path, m in model.named_modules():
        if not path:
            continue
        pl = path.lower()

        # 排除 vision 主干 & llama 层堆栈内部
        if _is_in_vision(pl) or _is_in_llama_layers(pl):
            continue

        is_linear = isinstance(m, nn.Linear)
        is_seq = isinstance(m, nn.Sequential)
        is_mlp_like = any(k in pl for k in ["projector", "projection", "connector", "bridge", "mapping", "align", "adapter", "qformer", "fusion", "mm"])

        if not (is_linear or is_seq or is_mlp_like):
            continue

        score = 0
        if is_linear:
            score += 3 if (m.in_features != m.out_features) else 1
        if is_seq:
            linears = [x for x in m.modules() if isinstance(x, nn.Linear)]
            if len(linears) >= 2:
                score += 3
            elif len(linears) == 1:
                score += 1
            for lin in linears:
                if lin.in_features != lin.out_features:
                    score += 1
        if is_mlp_like:
            score += 2

        if score >= 3:
            cands.append((path, m, score))

    cands.sort(key=lambda x: x[2], reverse=True)
    return [(p, m) for p, m, _ in cands]

def connector_param_names(model: nn.Module):
    """
    返回 connector 参数名 set：
      - 先用名字关键词命中
      - 再用结构扫描补充
    """
    # 名字命中
    by_name = set([n for n, _ in model.named_parameters() if _name_hits_connector(n)])

    # 结构扫描命中
    mods = find_connector_modules_by_structure(model)
    mod_paths = set([p for p, _ in mods])
    by_struct = set()
    for n, _ in model.named_parameters():
        for p in mod_paths:
            if n.startswith(p + "."):
                if _is_attention_proj_param(n):
                    continue
                by_struct.add(n)
                break

    return by_name, by_struct, mods

# 4.4 启用梯度分组（connector / vision last-3 / llama last-3）
for _, p in model.named_parameters():
    p.requires_grad_(False)

enabled_names = []
param_groups = {'visual': [], 'text': [], 'connector': []}
name_to_param = dict(model.named_parameters())

conn_by_name, conn_by_struct, conn_modules = connector_param_names(model)
conn_names = set(conn_by_name) | set(conn_by_struct)

print("\n🔎 Connector 诊断：")
print(f"  by-name:   {len(conn_by_name)}")
print(f"  by-struct: {len(conn_by_struct)}")
print(f"  modules(top10):")
for i, (p, m) in enumerate(conn_modules[:10]):
    print(f"    [{i:02d}] {p} | {type(m).__name__}")

if len(conn_names) == 0:
    # 给你一个“最像 connector”的参数名候选（用于你手动加关键词）
    candidates = []
    for n, p in model.named_parameters():
        if p.ndim < 2:
            continue
        if _is_attention_proj_param(n):
            continue
        ln = n.lower()
        # 打分：含 mm/proj/bridge 等 + 不在纯 llama 层 + 不在纯 vision block
        score = 0
        score += 2 if any(k in ln for k in ["mm", "project", "proj", "bridge", "map", "align", "adapter", "fusion", "qformer"]) else 0
        score += 1 if ("vision" in ln or "visual" in ln) else 0
        score += 1 if ("llama" in ln or "language_model" in ln) else 0
        # connector 往往同时沾边，但不应是 self_attn proj
        if score >= 2:
            candidates.append((score, n, tuple(p.shape)))
    candidates.sort(key=lambda x: x[0], reverse=True)

    print("\n⚠️ 诊断：connector 参数=0。下面列出最像 connector 的参数名候选（排除 attn 投影后）：")
    for s, n, sh in candidates[:30]:
        print(f"  score={s} | {n} | {sh}")
    print("👉 如果这些名字与你 repo 一致，把对应关键字加入 CONNECTOR_KEYS_BASE 即可。\n")

for n, p in model.named_parameters():
    if p.ndim < 2:
        continue

    group = None
    if n in conn_names:
        group = "connector"
    elif _is_vision_last3(n):
        group = "visual"
    elif _is_llama_last3(n):
        group = "text"

    if group is not None:
        p.requires_grad_(True)
        enabled_names.append(n)
        param_groups[group].append((n, tuple(p.shape)))

enabled_names = sorted(enabled_names)

print(f"\n🎯 启用梯度的参数: {len(enabled_names)}")
for group_name, items in param_groups.items():
    print(f"  {group_name.upper():<10}: {len(items)}")
    for name, shape in items[:3]:
        print(f"    - {name}: {shape}")
    if len(items) > 3:
        print(f"    ... 还有 {len(items)-3} 个")

# ============================================================
# 5. MiniGPT-v2 输入构造（PROMPT 自动补 <ImageHere>）
# ============================================================
PROMPT_RAW = "Describe the image in detail."

def ensure_placeholder(p: str) -> str:
    p = (p or "").strip()
    return p if "<ImageHere>" in p else "<ImageHere> " + p

PROMPT = ensure_placeholder(PROMPT_RAW)
PROMPT_LIST = [PROMPT]
print("\n[INFO] PROMPT:", PROMPT)

def _make_samples(image_1, caption_1):
    cap = caption_1 if isinstance(caption_1, str) and caption_1.strip() else " "
    return {"image": image_1, "text_input": PROMPT_LIST, "answer": [cap]}

def _forward_loss(samples):
    try:
        out = model.forward(samples, reduction="mean")
    except TypeError:
        out = model(samples, reduction="mean")

    if isinstance(out, dict) and "loss" in out:
        return out["loss"]
    if hasattr(out, "loss"):
        return out.loss
    if torch.is_tensor(out):
        return out
    raise RuntimeError("❌ Cannot find loss in model output.")

# ============================================================
# 6. 单样本梯度计算（grad_dict 为空则返回 None）
# ============================================================
def compute_grad_dict_gpu(image_1, caption_1):
    global _step_counter
    model.zero_grad(set_to_none=True)
    samples = _make_samples(image_1, caption_1)

    try:
        if AMP:
            with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE):
                loss = _forward_loss(samples)
        else:
            loss = _forward_loss(samples)

        if (not torch.is_tensor(loss)) or torch.isnan(loss) or torch.isinf(loss):
            return None

        loss.backward()

        grad_dict = {}
        for name in enabled_names:
            p = name_to_param[name]
            g = p.grad
            if g is None:
                continue
            g = torch.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0)
            grad_dict[name] = g.detach().half()

        if len(grad_dict) == 0:
            return None

        _step_counter += 1
        if (_step_counter % CLEANUP_EVERY) == 0:
            _light_cleanup()

        return grad_dict

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            model.zero_grad(set_to_none=True)
            _oom_cleanup()
        return None
    finally:
        model.zero_grad(set_to_none=True)

# ============================================================
# 7. 行列余弦相似度（GPU）
# ============================================================
def row_col_cosine_gpu(grad_gpu, ref_gpu):
    if grad_gpu.ndim < 2:
        flat_sim = F.cosine_similarity(
            grad_gpu.flatten().unsqueeze(0),
            ref_gpu.flatten().unsqueeze(0),
            dim=1
        )
        flat_sim = torch.nan_to_num(flat_sim, nan=0.0)
        return flat_sim, flat_sim

    row_sim = F.cosine_similarity(grad_gpu, ref_gpu, dim=1)
    row_sim = torch.nan_to_num(row_sim, nan=0.0)

    col_sim = F.cosine_similarity(grad_gpu.T, ref_gpu.T, dim=1)
    col_sim = torch.nan_to_num(col_sim, nan=0.0)

    return row_sim, col_sim

# ============================================================
# 8. 构建参考梯度（CPU float32 累加 -> GPU half）
# ============================================================
def build_reference_gradient(loader, tag="member", take=200):
    ref_sum_cpu = {}
    ref_count = 0
    seen = 0

    print(f"\n📌 构建参考梯度 ({tag}, target={take}) ...")

    for batch in tqdm(loader, desc=f"ref-{tag}"):
        imgs = batch["image"].to(device, non_blocking=True)
        caps = batch["caption"]
        bsz = imgs.size(0)

        for i in range(bsz):
            if ref_count >= take:
                break
            img_1 = imgs[i:i+1]
            cap_1 = caps[i]

            grad_dict_gpu = compute_grad_dict_gpu(img_1, cap_1)
            if grad_dict_gpu is None:
                continue

            ref_count += 1

            if not ref_sum_cpu:
                ref_sum_cpu = {k: v.float().cpu().clone() for k, v in grad_dict_gpu.items()}
            else:
                for k in list(ref_sum_cpu.keys()):
                    if k in grad_dict_gpu and ref_sum_cpu[k].shape == grad_dict_gpu[k].shape:
                        ref_sum_cpu[k].add_(grad_dict_gpu[k].float().cpu())
                    else:
                        ref_sum_cpu.pop(k, None)

        seen += bsz
        if ref_count >= take:
            break
        if seen >= MAX_PER_SPLIT:
            break

    if ref_count == 0:
        raise RuntimeError(f"❌ 参考梯度构建失败: {tag}")

    ref_avg_gpu = {k: (v / ref_count).half().to(device) for k, v in ref_sum_cpu.items()}
    print(f"✅ 参考梯度完成: {tag}, 有效样本={ref_count}, 参数数={len(ref_avg_gpu)}")
    return ref_avg_gpu

# ============================================================
# 9. 平均行列相似度（加诊断：ok 成功样本数）
# ============================================================
def compute_avg_rowcol_similarity(loader, ref_grads_gpu, tag="member", take=200):
    acc_row_gpu = {}
    acc_col_gpu = {}
    ok = 0
    seen = 0

    print(f"\n📌 计算平均相似度 ({tag}, target={take}) ...")

    for batch in tqdm(loader, desc=f"sim-{tag}"):
        imgs = batch["image"].to(device, non_blocking=True)
        caps = batch["caption"]
        bsz = imgs.size(0)

        for i in range(bsz):
            if ok >= take:
                break

            img_1 = imgs[i:i+1]
            cap_1 = caps[i]

            grad_dict_gpu = compute_grad_dict_gpu(img_1, cap_1)
            if grad_dict_gpu is None:
                continue

            ok += 1

            for name, g in grad_dict_gpu.items():
                ref = ref_grads_gpu.get(name, None)
                if ref is None or ref.shape != g.shape:
                    continue

                rs, cs = row_col_cosine_gpu(g, ref)

                if name not in acc_row_gpu:
                    acc_row_gpu[name] = rs.float()
                    acc_col_gpu[name] = cs.float()
                else:
                    if acc_row_gpu[name].shape == rs.shape:
                        acc_row_gpu[name].add_(rs.float())
                    if acc_col_gpu[name].shape == cs.shape:
                        acc_col_gpu[name].add_(cs.float())

        seen += bsz
        if ok >= take:
            break
        if seen >= MAX_PER_SPLIT:
            break

    if ok == 0 or len(acc_row_gpu) == 0:
        print(f"⚠️ {tag}: ok={ok}, acc_row_gpu={len(acc_row_gpu)} -> 返回空（说明梯度/匹配失败）")
        return {}, {}

    avg_row_gpu = {k: v / ok for k, v in acc_row_gpu.items()}
    avg_col_gpu = {k: v / ok for k, v in acc_col_gpu.items()}

    print(f"✅ 平均相似度完成: {tag}, ok样本={ok}, 参数数={len(avg_row_gpu)}")
    return avg_row_gpu, avg_col_gpu

# ============================================================
# 10. 构建敏感维度掩码（保持不变）
# ============================================================
def build_adaptive_masks(mem_row_gpu, mem_col_gpu, non_row_gpu, non_col_gpu,
                         tau=SENSITIVITY_TAU, min_keep_ratio=MIN_KEEP_RATIO):
    row_masks_gpu = {}
    col_masks_gpu = {}
    total_dims = 0
    total_selected = 0

    print(f"\n📌 构建敏感维度掩码 (tau={tau}, min_keep_ratio={min_keep_ratio}) ...")

    for name in mem_row_gpu:
        if name not in non_row_gpu or name not in mem_col_gpu or name not in non_col_gpu:
            continue

        row_gap = mem_row_gpu[name] - non_row_gpu[name]
        col_gap = mem_col_gpu[name] - non_col_gpu[name]

        # row
        row_size = row_gap.numel()
        total_dims += row_size
        row_tau_mask = (row_gap > tau)
        row_selected = row_tau_mask.sum().item()

        if row_selected < row_size * min_keep_ratio:
            k = max(1, int(row_size * min_keep_ratio))
            _, topk_idx = torch.topk(row_gap.flatten(), k)
            row_mask = torch.zeros_like(row_gap, dtype=torch.bool)
            row_mask.view(-1)[topk_idx] = True
        else:
            row_mask = row_tau_mask

        # col
        col_size = col_gap.numel()
        total_dims += col_size
        col_tau_mask = (col_gap > tau)
        col_selected = col_tau_mask.sum().item()

        if col_selected < col_size * min_keep_ratio:
            k = max(1, int(col_size * min_keep_ratio))
            _, topk_idx = torch.topk(col_gap.flatten(), k)
            col_mask = torch.zeros_like(col_gap, dtype=torch.bool)
            col_mask.view(-1)[topk_idx] = True
        else:
            col_mask = col_tau_mask

        row_masks_gpu[name] = row_mask
        col_masks_gpu[name] = col_mask
        total_selected += int(row_mask.sum().item() + col_mask.sum().item())

    ratio = total_selected / total_dims if total_dims > 0 else 0.0
    print(f"✅ 敏感维度总数: {total_selected}/{total_dims} ({ratio*100:.1f}%)")
    return row_masks_gpu, col_masks_gpu, total_selected

# ============================================================
# 11. GradSafe 评分（GPU sum/count 等价 mean）
# ============================================================
def gradsafe_score_single(image_1, caption_1, ref_grads_gpu, row_masks_gpu, col_masks_gpu):
    grad_dict_gpu = compute_grad_dict_gpu(image_1, caption_1)
    if grad_dict_gpu is None:
        return 0.0

    total_sum = None
    total_cnt = 0

    for name, g in grad_dict_gpu.items():
        ref = ref_grads_gpu.get(name, None)
        rmask = row_masks_gpu.get(name, None)
        cmask = col_masks_gpu.get(name, None)
        if ref is None or rmask is None or cmask is None:
            continue
        if ref.shape != g.shape:
            continue

        rs, cs = row_col_cosine_gpu(g, ref)

        if rmask.any():
            vals = rs[rmask]
            total_cnt += int(vals.numel())
            total_sum = vals.sum() if total_sum is None else (total_sum + vals.sum())

        if cmask.any():
            vals = cs[cmask]
            total_cnt += int(vals.numel())
            total_sum = vals.sum() if total_sum is None else (total_sum + vals.sum())

    if total_cnt == 0 or total_sum is None:
        return 0.0

    return float((total_sum / total_cnt).item())

def probe_dataset(loader, ref_grads_gpu, row_masks_gpu, col_masks_gpu, tag="member", take=800):
    scores = []
    seen = 0

    print(f"\n🚀 探测 {tag} (target={take}) ...")

    for batch in tqdm(loader, desc=f"probe-{tag}"):
        imgs = batch["image"].to(device, non_blocking=True)
        caps = batch["caption"]
        bsz = imgs.size(0)

        for i in range(bsz):
            if seen >= take:
                break
            img_1 = imgs[i:i+1]
            cap_1 = caps[i]

            scores.append(gradsafe_score_single(img_1, cap_1, ref_grads_gpu, row_masks_gpu, col_masks_gpu))
            seen += 1

            if (seen % CLEANUP_EVERY) == 0:
                _light_cleanup()

        if seen >= take:
            break
        if seen >= MAX_PER_SPLIT:
            break

    print(f"✅ 探测完成: {tag}, 样本数={len(scores)}")
    return scores

# ============================================================
# 12. 评估指标（保持不变）
# ============================================================
def evaluate_metrics(scores, labels):
    results = {}
    try:
        auc = roc_auc_score(labels, scores)
        results["auc"] = float(auc)
    except:
        results["auc"] = float("nan")
        return results

    fpr, tpr, thresholds = roc_curve(labels, scores)
    j_idx = np.argmax(tpr - fpr)
    best_thr = float(thresholds[j_idx])
    results["best_threshold"] = best_thr

    pred = (scores >= best_thr).astype(int)
    results["accuracy"] = float(accuracy_score(labels, pred))
    results["confusion_matrix"] = confusion_matrix(labels, pred).tolist()

    nonmem_mask = (labels == 0)
    if nonmem_mask.sum() > 0:
        thr_5 = float(np.quantile(scores[nonmem_mask], 0.95))
        pred_5 = (scores >= thr_5).astype(int)
        cm5 = confusion_matrix(labels, pred_5)

        if cm5.size == 4:
            tn, fp, fn, tp = cm5.ravel()
            fpr_achieved = fp / (fp + tn + 1e-12)
            tpr_achieved = tp / (tp + fn + 1e-12)
        else:
            fpr_achieved = float("nan")
            tpr_achieved = float("nan")

        prec, rec, f1, _ = precision_recall_fscore_support(labels, pred_5, average="binary", zero_division=0)

        results["threshold_5fpr"] = thr_5
        results["actual_fpr"] = float(fpr_achieved)
        results["tpr_at_5fpr"] = float(tpr_achieved)
        results["precision_at_5fpr"] = float(prec)
        results["recall_at_5fpr"] = float(rec)
        results["f1_at_5fpr"] = float(f1)
        results["accuracy_at_5fpr"] = float(accuracy_score(labels, pred_5))
        results["confusion_matrix_5fpr"] = cm5.tolist()

    return results

# ============================================================
# 13. 主流程
# ============================================================
print("\n" + "="*80)
print("🚀 GradSafe MIA for MiniGPT-v2 (Connector auto-detect + no-attn-proj)")
print("="*80)

print("\n【阶段1：校准】")
ref_grads_gpu = build_reference_gradient(member_loader, tag="member", take=CALIB_MEM)

mem_row_gpu, mem_col_gpu = compute_avg_rowcol_similarity(member_loader, ref_grads_gpu, tag="member", take=CALIB_MEM)
non_row_gpu, non_col_gpu = compute_avg_rowcol_similarity(nonmem_loader, ref_grads_gpu, tag="nonmember", take=CALIB_NON)

row_masks_gpu, col_masks_gpu, total_critical = build_adaptive_masks(
    mem_row_gpu, mem_col_gpu, non_row_gpu, non_col_gpu,
    tau=SENSITIVITY_TAU, min_keep_ratio=MIN_KEEP_RATIO
)

if total_critical == 0:
    raise RuntimeError("❌ 未找到敏感维度（请看上面 ok样本/参数数 是否为 0，以定位是梯度失败还是匹配失败）")

print("\n【阶段2：探测】")
mem_scores = probe_dataset(member_loader, ref_grads_gpu, row_masks_gpu, col_masks_gpu, tag="member", take=PROBE_MEM)
non_scores = probe_dataset(nonmem_loader, ref_grads_gpu, row_masks_gpu, col_masks_gpu, tag="nonmember", take=PROBE_NON)

print("\n【阶段3：评估】")
scores = np.array(mem_scores + non_scores, dtype=np.float64)
labels = np.array([1]*len(mem_scores) + [0]*len(non_scores), dtype=np.int64)
results = evaluate_metrics(scores, labels)

print("\n" + "="*80)
print("📊 GradSafe MIA 结果")
print("="*80)
print(f"AUC:             {results['auc']:.4f}")
print(f"最佳阈值:         {results['best_threshold']:.4f}")
print(f"准确率:           {results['accuracy']:.4f}")
print("\nConfusion Matrix [[TN FP][FN TP]]:")
print(np.array(results["confusion_matrix"]))

if "tpr_at_5fpr" in results:
    print("\n" + "-"*80)
    print("📊 Metrics @ 5% FPR")
    print("-"*80)
    print(f"阈值:             {results['threshold_5fpr']:.6f}")
    print(f"实际 FPR:         {results['actual_fpr']*100:.2f}%")
    print(f"TPR:              {results['tpr_at_5fpr']*100:.2f}%")
    print(f"Precision:        {results['precision_at_5fpr']*100:.2f}%")
    print(f"F1:               {results['f1_at_5fpr']:.4f}")
    print(f"Accuracy:         {results['accuracy_at_5fpr']*100:.2f}%")
    print("\nConfusion Matrix:")
    print(np.array(results["confusion_matrix_5fpr"]))

OUT_DIR = "/content/minigptv2_gradsafe_results"
os.makedirs(OUT_DIR, exist_ok=True)

mem_csv = os.path.join(OUT_DIR, "member_scores.csv")
non_csv = os.path.join(OUT_DIR, "nonmember_scores.csv")

with open(mem_csv, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["score"]); w.writerows([[s] for s in mem_scores])

with open(non_csv, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["score"]); w.writerows([[s] for s in non_scores])

with open(os.path.join(OUT_DIR, "gradsafe_results.json"), "w") as f:
    json.dump({"config": {
        "model": type(model).__name__,
        "CALIB_MEM": CALIB_MEM, "CALIB_NON": CALIB_NON,
        "PROBE_MEM": PROBE_MEM, "PROBE_NON": PROBE_NON,
        "SENSITIVITY_TAU": SENSITIVITY_TAU, "MIN_KEEP_RATIO": MIN_KEEP_RATIO,
        "enabled_params": len(enabled_names),
        "enabled_connector_params": int(len(param_groups["connector"])),
        "connector_by_name": int(len(conn_by_name)),
        "connector_by_struct": int(len(conn_by_struct)),
        "critical_dims": int(total_critical),
        "CLEANUP_EVERY": CLEANUP_EVERY,
        "PROMPT": PROMPT,
    }, "results": results}, f, indent=2)

print(f"\n💾 结果已保存:\n   {mem_csv}\n   {non_csv}\n   {OUT_DIR}/gradsafe_results.json")
print("\n✅ 实验完成!")
print("="*80)
